# 10 Audience Sentiment Analysis

This GitHub copy has been cleaned for readability: duplicate/empty cells were removed and saved cell outputs were cleared. Code and analysis workflow are retained.

**Execution note:** Original project paths may reference Google Drive/Colab. Update paths and provide required source files/API credentials in your own environment before rerunning.

### Objective
Analyze audience responses to selected Indian movies using "
YouTube comments, with careful attention to comment relevance, "
language, sentiment, source context and validation.


In [ ]:
# ============================================================
# NOTEBOOK 10 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/gdrive2')

print("\nGoogle Drive mounted successfully.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — PROJECT PATH AND ORIGINAL DATA VERIFICATION
# ============================================================

from pathlib import Path

DRIVE_ROOT = Path("/content/gdrive2")
PROJECT_ROOT = DRIVE_ROOT / "MyDrive" / "IndianCinemaTrendAnalytics"

print("=" * 75)
print("PROJECT PATH VERIFICATION")
print("=" * 75)

print("Drive mounted:", DRIVE_ROOT.exists())
print("MyDrive exists:", (DRIVE_ROOT / "MyDrive").exists())
print("Project exists:", PROJECT_ROOT.exists())

if PROJECT_ROOT.exists():

    print("\nPROJECT CONTENTS")
    for item in sorted(PROJECT_ROOT.iterdir()):
        print(("DIR  " if item.is_dir() else "FILE "), item.name)

    print("\nORIGINAL YOUTUBE FILES")

    target_files = [
        "ICMB_130_YOUTUBE_COMMENT_PILOT.csv",
        "ICMB_130_YOUTUBE_DISCOVERY_PILOT.csv",
        "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_STATUS_PILOT.csv"
    ]

    for filename in target_files:
        matches = list(PROJECT_ROOT.rglob(filename))

        print(f"\n{filename}")
        if matches:
            for path in matches:
                print("FOUND:", path)
                print("SIZE (KB):", round(path.stat().st_size / 1024, 2))
        else:
            print("NOT FOUND")

else:
    print("\nProject directory not found at the expected location.")

print("\nVERIFICATION COMPLETE — NO FILES MODIFIED.")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

DATA_ROOT = PROJECT_ROOT / "01_data"

print("=" * 75)
print("ACTUAL DATA DIRECTORY STRUCTURE")
print("=" * 75)

for folder in [DATA_ROOT, DATA_ROOT / "raw", DATA_ROOT / "processed"]:

    print(f"\n{folder}")
    print("Exists:", folder.exists())

    if folder.exists():
        for item in sorted(folder.iterdir()):
            print(
                ("DIR  " if item.is_dir() else "FILE "),
                item.name
            )

print("\nSEARCHING FOR ORIGINAL COMMENT FILE")

for path in DATA_ROOT.rglob("ICMB_130_YOUTUBE_COMMENT_PILOT.csv"):
    print("FOUND:", path)
    print("SIZE (KB):", round(path.stat().st_size / 1024, 2))

print("\nREAD-ONLY INSPECTION COMPLETE.")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

AUDIENCE_RAW_DIR = PROJECT_ROOT / "01_data/raw/audience_sentiment"
AUDIENCE_PROCESSED_DIR = PROJECT_ROOT / "01_data/processed/audience_sentiment"

for label, folder in [
    ("RAW AUDIENCE DATA", AUDIENCE_RAW_DIR),
    ("PROCESSED AUDIENCE DATA", AUDIENCE_PROCESSED_DIR)
]:
    print("\n" + "=" * 75)
    print(label)
    print("=" * 75)

    if folder.exists():
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                print(
                    path.relative_to(folder),
                    "|",
                    round(path.stat().st_size / 1024, 2),
                    "KB"
                )
    else:
        print("Directory not found.")

print("\nINSPECTION COMPLETE — NO FILES MODIFIED.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — LOAD ORIGINAL AUDIENCE SENTIMENT DATA
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = PROJECT_ROOT / "01_data/raw/audience_sentiment"
PROCESSED_DIR = PROJECT_ROOT / "01_data/processed/audience_sentiment"

# Source files
comments_path = (
    RAW_DIR / "youtube/comments/ICMB_130_YOUTUBE_COMMENT_PILOT.csv"
)

reference_path = (
    PROCESSED_DIR / "ICMB_130_AUDIENCE_MOVIE_REFERENCE_VALIDATED.csv"
)

screening_path = (
    PROCESSED_DIR / "ICMB_130_YOUTUBE_SCREENING_V3_PILOT.csv"
)

# Load datasets
comments_df = pd.read_csv(comments_path)
movie_reference_df = pd.read_csv(reference_path)
video_screening_df = pd.read_csv(screening_path)

# Display dataset information
datasets = {
    "YOUTUBE COMMENTS": comments_df,
    "VALIDATED MOVIE REFERENCE": movie_reference_df,
    "YOUTUBE VIDEO SCREENING V3": video_screening_df
}

for name, df in datasets.items():
    print("\n" + "=" * 75)
    print(name)
    print("=" * 75)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nFirst 3 records:")
    display(df.head(3))

print("\nALL THREE DATASETS LOADED SUCCESSFULLY.")
print("Original source files remain unchanged.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — COMMENT, VIDEO AND MOVIE RELATIONSHIP CHECK
# ============================================================

# 1. Check unique identifiers
print("=" * 75)
print("IDENTIFIER CHECK")
print("=" * 75)

print("Comment records:", len(comments_df))
print("Unique comment IDs:", comments_df["comment_id"].nunique())
print("Unique video IDs in comments:", comments_df["video_id"].nunique())
print("Unique movie IDs in comments:", comments_df["movie_record_id"].nunique())

print("\nUnique video IDs in screening:", video_screening_df["video_id"].nunique())
print("Unique movie IDs in reference:", movie_reference_df["movie_record_id"].nunique())

# 2. Check whether comment video IDs exist in screening
screening_video_ids = set(video_screening_df["video_id"].dropna())
comment_video_ids = set(comments_df["video_id"].dropna())

matched_videos = comment_video_ids.intersection(screening_video_ids)
unmatched_videos = comment_video_ids - screening_video_ids

print("\n" + "=" * 75)
print("COMMENT-TO-VIDEO MATCHING")
print("=" * 75)

print("Matched video IDs:", len(matched_videos))
print("Unmatched video IDs:", len(unmatched_videos))

if unmatched_videos:
    print("Unmatched IDs:", sorted(unmatched_videos))

# 3. Check movie ID consistency
reference_movie_ids = set(movie_reference_df["movie_record_id"].dropna())
comment_movie_ids = set(comments_df["movie_record_id"].dropna())

print("\n" + "=" * 75)
print("COMMENT-TO-MOVIE REFERENCE MATCHING")
print("=" * 75)

print("Movie IDs found in reference:", len(comment_movie_ids.intersection(reference_movie_ids)))
print("Movie IDs missing from reference:", len(comment_movie_ids - reference_movie_ids))

# 4. Summarise comments by movie and video
print("\n" + "=" * 75)
print("COMMENT DISTRIBUTION BY MOVIE")
print("=" * 75)

movie_comment_summary = (
    comments_df.groupby(
        ["movie_record_id", "movie_title", "movie_year"],
        dropna=False
    )
    .agg(
        comment_count=("comment_id", "count"),
        unique_videos=("video_id", "nunique"),
        total_likes=("like_count", "sum")
    )
    .reset_index()
    .sort_values("comment_count", ascending=False)
)

display(movie_comment_summary)

print("\n" + "=" * 75)
print("COMMENT DISTRIBUTION BY VIDEO")
print("=" * 75)

video_comment_summary = (
    comments_df.groupby(
        ["movie_record_id", "movie_title", "video_id", "video_title"],
        dropna=False
    )
    .agg(
        comment_count=("comment_id", "count"),
        total_likes=("like_count", "sum")
    )
    .reset_index()
    .sort_values("comment_count", ascending=False)
)

display(video_comment_summary)

print("\nRELATIONSHIP CHECK COMPLETED — NO SOURCE FILES MODIFIED.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — INTEGRATED COMMENT ANALYSIS DATASET
# ============================================================

# Select video-level context from V3 screening
video_context_columns = [
    "movie_record_id",
    "video_id",
    "video_type_v3",
    "franchise_status_v3",
    "video_language_v3",
    "screening_decision_v3",
    "video_title",
    "channel_title",
    "published_at"
]

video_context = video_screening_df[
    video_context_columns
].copy()

# Select validated movie metadata
movie_context_columns = [
    "movie_record_id",
    "language",
    "industry",
    "india_gross_cr",
    "tmdb_id"
]

movie_context = movie_reference_df[
    movie_context_columns
].copy()

# Merge comments with video screening context
comment_analysis_df = comments_df.merge(
    video_context,
    on=["movie_record_id", "video_id"],
    how="left",
    suffixes=("", "_screening"),
    validate="many_to_one",
    indicator="video_merge_status"
)

# Merge validated movie metadata
comment_analysis_df = comment_analysis_df.merge(
    movie_context,
    on="movie_record_id",
    how="left",
    validate="many_to_one",
    indicator="movie_merge_status"
)

# Display integration results
print("=" * 75)
print("INTEGRATED COMMENT ANALYSIS DATASET")
print("=" * 75)

print("Final shape:", comment_analysis_df.shape)
print("Original comment records:", len(comments_df))

print("\nVideo merge:")
print(comment_analysis_df["video_merge_status"].value_counts())

print("\nMovie merge:")
print(comment_analysis_df["movie_merge_status"].value_counts())

print("\nVideo type distribution:")
display(
    comment_analysis_df["video_type_v3"]
    .value_counts(dropna=False)
    .rename_axis("video_type")
    .reset_index(name="comment_count")
)

print("\nScreening decision distribution:")
display(
    comment_analysis_df["screening_decision_v3"]
    .value_counts(dropna=False)
    .rename_axis("screening_decision")
    .reset_index(name="comment_count")
)

print("\nIntegrated sample:")
display(
    comment_analysis_df[
        [
            "movie_title",
            "video_id",
            "comment_text",
            "video_type_v3",
            "screening_decision_v3",
            "language",
            "industry"
        ]
    ].head(5)
)

print("\nINTEGRATION COMPLETED — SOURCE FILES UNCHANGED.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — COMMENT QUALITY AND TEXT PROFILING
# ============================================================

import re
import numpy as np

df = comment_analysis_df.copy()

# Text preparation for profiling only
df["comment_text_clean"] = (
    df["comment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["character_count"] = df["comment_text_clean"].str.len()

df["word_count"] = df["comment_text_clean"].apply(
    lambda x: len(x.split()) if x else 0
)

df["contains_url"] = df["comment_text_clean"].str.contains(
    r"https?://|www\.",
    case=False,
    regex=True
)

df["is_empty"] = df["comment_text_clean"].eq("")

df["is_duplicate_text"] = df.duplicated(
    subset=["comment_text_clean"],
    keep=False
) & ~df["is_empty"]

# Overall quality profile
print("=" * 75)
print("COMMENT QUALITY PROFILE")
print("=" * 75)

quality_summary = pd.DataFrame({
    "Metric": [
        "Total comments",
        "Empty comments",
        "Non-empty comments",
        "Duplicate-text records",
        "Comments containing URLs",
        "Comments with fewer than 3 words",
        "Average characters per comment",
        "Median characters per comment",
        "Average words per comment",
        "Median words per comment"
    ],
    "Value": [
        len(df),
        int(df["is_empty"].sum()),
        int((~df["is_empty"]).sum()),
        int(df["is_duplicate_text"].sum()),
        int(df["contains_url"].sum()),
        int((df["word_count"] < 3).sum()),
        round(df["character_count"].mean(), 2),
        round(df["character_count"].median(), 2),
        round(df["word_count"].mean(), 2),
        round(df["word_count"].median(), 2)
    ]
})

display(quality_summary)

# Comment length distribution
print("\nCOMMENT LENGTH DISTRIBUTION")

df["length_category"] = pd.cut(
    df["character_count"],
    bins=[-1, 0, 20, 50, 100, 250, np.inf],
    labels=[
        "Empty",
        "1–20 characters",
        "21–50 characters",
        "51–100 characters",
        "101–250 characters",
        "Above 250 characters"
    ]
)

display(
    df["length_category"]
    .value_counts(sort=False)
    .rename_axis("length_category")
    .reset_index(name="comment_count")
)

# Sample comments for manual inspection
print("\nSAMPLE COMMENTS — FIRST 15 NON-EMPTY RECORDS")

display(
    df.loc[
        ~df["is_empty"],
        [
            "movie_title",
            "video_title",
            "comment_text_clean",
            "character_count",
            "word_count",
            "contains_url"
        ]
    ].head(15)
)

print("\nPROFILING COMPLETE — NO SOURCE FILES MODIFIED.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — PRELIMINARY COMMENT RELEVANCE TRIAGE
# ============================================================

import re

# Recalculate required indicators directly in the analysis dataset
comment_analysis_df["comment_text_clean"] = (
    comment_analysis_df["comment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

comment_analysis_df["word_count"] = (
    comment_analysis_df["comment_text_clean"]
    .str.split()
    .str.len()
)

comment_analysis_df["contains_url"] = (
    comment_analysis_df["comment_text_clean"]
    .str.contains(r"https?://|www\.", case=False, regex=True)
)

# Preliminary triage
comment_analysis_df["relevance_status"] = "Pending Relevance Review"

comment_analysis_df.loc[
    comment_analysis_df["contains_url"],
    "relevance_status"
] = "Needs Review — URL/Promotional Content"

comment_analysis_df.loc[
    (comment_analysis_df["word_count"] < 3) &
    (~comment_analysis_df["contains_url"]),
    "relevance_status"
] = "Needs Review — Very Short Text"

# Results
print("=" * 75)
print("PRELIMINARY RELEVANCE TRIAGE")
print("=" * 75)

display(
    comment_analysis_df["relevance_status"]
    .value_counts()
    .rename_axis("relevance_status")
    .reset_index(name="comment_count")
)

print("\nFLAGGED COMMENTS")

display(
    comment_analysis_df.loc[
        comment_analysis_df["relevance_status"] != "Pending Relevance Review",
        [
            "movie_title",
            "comment_text_clean",
            "word_count",
            "contains_url",
            "relevance_status"
        ]
    ]
)

print("\nTotal records retained:", len(comment_analysis_df))
print("Original source files unchanged.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — SCRIPT IDENTIFICATION
# ============================================================

import unicodedata
import pandas as pd

def detect_script(text):
    if not isinstance(text, str) or not text.strip():
        return "Empty"

    counts = {
        "Devanagari": 0,
        "Latin": 0,
        "Telugu": 0,
        "Tamil": 0,
        "Kannada": 0,
        "Other": 0
    }

    for char in text:
        if not char.isalpha():
            continue

        code = ord(char)

        if 0x0900 <= code <= 0x097F:
            counts["Devanagari"] += 1
        elif 0x0041 <= code <= 0x024F:
            counts["Latin"] += 1
        elif 0x0C00 <= code <= 0x0C7F:
            counts["Telugu"] += 1
        elif 0x0B80 <= code <= 0x0BFF:
            counts["Tamil"] += 1
        elif 0x0C80 <= code <= 0x0CFF:
            counts["Kannada"] += 1
        else:
            counts["Other"] += 1

    active = [name for name, count in counts.items() if count > 0]

    if not active:
        return "No identifiable script"

    if len(active) == 1:
        return active[0]

    return "Mixed: " + " + ".join(active)

comment_analysis_df["script_category"] = (
    comment_analysis_df["comment_text_clean"].apply(detect_script)
)

print("=" * 75)
print("COMMENT WRITING-SCRIPT DISTRIBUTION")
print("=" * 75)

script_summary = (
    comment_analysis_df["script_category"]
    .value_counts()
    .rename_axis("script_category")
    .reset_index(name="comment_count")
)

script_summary["percentage"] = (
    script_summary["comment_count"] / len(comment_analysis_df) * 100
).round(2)

display(script_summary)

print("\nSAMPLE COMMENTS BY SCRIPT")

for script in comment_analysis_df["script_category"].unique():
    print(f"\n{script}")
    display(
        comment_analysis_df.loc[
            comment_analysis_df["script_category"] == script,
            ["movie_title", "comment_text_clean"]
        ].head(3)
    )

print("\nSCRIPT IDENTIFICATION COMPLETE.")
print("Note: Script identification does not establish language.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — PROVISIONAL LANGUAGE CLASSIFICATION
# ============================================================

def classify_language_provisionally(script):
    mapping = {
        "Devanagari": "Hindi (script-based candidate)",
        "Telugu": "Telugu (script-based candidate)",
        "Tamil": "Tamil (script-based candidate)",
        "Kannada": "Kannada (script-based candidate)",
        "Latin": "Undetermined — Latin script",
        "Mixed: Devanagari + Latin": "Mixed Hindi/Latin — review",
        "Mixed: Latin + Telugu": "Mixed Telugu/Latin — review",
        "Other": "Other — review",
        "No identifiable script": "Undetermined"
    }

    return mapping.get(script, "Undetermined — review")

comment_analysis_df["language_category"] = (
    comment_analysis_df["script_category"]
    .map(classify_language_provisionally)
)

language_summary = (
    comment_analysis_df["language_category"]
    .value_counts()
    .rename_axis("language_category")
    .reset_index(name="comment_count")
)

language_summary["percentage"] = (
    language_summary["comment_count"] /
    len(comment_analysis_df) * 100
).round(2)

print("=" * 75)
print("PROVISIONAL LANGUAGE CLASSIFICATION")
print("=" * 75)

display(language_summary)

print("\nLATIN-SCRIPT COMMENT SAMPLES — LANGUAGE UNDETERMINED")

display(
    comment_analysis_df.loc[
        comment_analysis_df["script_category"] == "Latin",
        ["movie_title", "comment_text_clean"]
    ].sample(
        min(15, (comment_analysis_df["script_category"] == "Latin").sum()),
        random_state=42
    )
)

print("\nNo forced language identification performed.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — YOUTUBE ACQUISITION MANIFEST REVIEW
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_ACQUISITION_MANIFEST.csv"
)

manifest_df = pd.read_csv(MANIFEST_PATH)

print("=" * 75)
print("YOUTUBE ACQUISITION MANIFEST")
print("=" * 75)

print("Shape:", manifest_df.shape)
print("\nColumns:")
print(manifest_df.columns.tolist())

print("\nFirst 10 records:")
display(manifest_df.head(10))

print("\nAcquisition status distribution:")

if "acquisition_status" in manifest_df.columns:
    display(
        manifest_df["acquisition_status"]
        .value_counts(dropna=False)
        .rename_axis("acquisition_status")
        .reset_index(name="movie_count")
    )
else:
    print("No acquisition_status column found.")

print("\nUnique movie records:",
      manifest_df["movie_record_id"].nunique())

print("\nManifest loaded — original file unchanged.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — DISCOVERY PILOT & API CONFIGURATION CHECK
# Read-only inspection; no API calls and no file modifications
# ============================================================

import os
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_YOUTUBE_DIR = (
    PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"
)

DISCOVERY_PATH = (
    RAW_YOUTUBE_DIR / "ICMB_130_YOUTUBE_DISCOVERY_PILOT.csv"
)

print("=" * 75)
print("1. EXISTING YOUTUBE DISCOVERY PILOT")
print("=" * 75)

if DISCOVERY_PATH.exists():
    discovery_df = pd.read_csv(DISCOVERY_PATH)

    print("Shape:", discovery_df.shape)
    print("Columns:", discovery_df.columns.tolist())

    for col in ["movie_record_id", "video_id", "discovery_source"]:
        if col in discovery_df.columns:
            print(f"\n{col} — unique values:",
                  discovery_df[col].nunique(dropna=True))

    if "movie_record_id" in discovery_df.columns:
        print("\nCandidates per movie:")
        display(
            discovery_df.groupby(
                ["movie_record_id", "movie_title"],
                dropna=False
            ).size().reset_index(name="candidate_count")
            if "movie_title" in discovery_df.columns
            else discovery_df.groupby(
                "movie_record_id", dropna=False
            ).size().reset_index(name="candidate_count")
        )

    display(discovery_df.head(5))
else:
    print("Discovery pilot file not found:", DISCOVERY_PATH)

print("\n" + "=" * 75)
print("2. YOUTUBE API CONFIGURATION — KEY NOT DISPLAYED")
print("=" * 75)

possible_key_names = [
    "YOUTUBE_API_KEY",
    "YOUTUBE_DATA_API_KEY",
    "GOOGLE_API_KEY"
]

for key_name in possible_key_names:
    print(f"{key_name} configured in environment:",
          bool(os.environ.get(key_name)))

print("\n" + "=" * 75)
print("3. YOUTUBE RAW FOLDER CONTENTS")
print("=" * 75)

for path in sorted(RAW_YOUTUBE_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT),
              f"({path.stat().st_size / 1024:.2f} KB)")

print("\nRead-only inspection completed.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — CHECK COLAB SECRET CONFIGURATION
# No API calls; the key is never displayed.
# ============================================================

from google.colab import userdata

secret_names = [
    "YOUTUBE_API_KEY",
    "YOUTUBE_DATA_API_KEY",
    "GOOGLE_API_KEY"
]

available_secrets = []

for name in secret_names:
    try:
        value = userdata.get(name)
        if value and value.strip():
            available_secrets.append(name)
    except Exception:
        pass

print("=" * 65)
print("YOUTUBE API SECRET CHECK")
print("=" * 65)

if available_secrets:
    print("Available secret name(s):", available_secrets)
    print("API key is accessible through Colab Secrets.")
else:
    print("No YouTube API key found under the checked secret names.")
    print("No key values were displayed.")

print("\nNo API requests were made.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — YOUTUBE API CONNECTION CHECK
# Uses Colab Secret; never displays the API key.
# ============================================================

from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

API_KEY = userdata.get("YOUTUBE_API_KEY")

try:
    youtube = build(
        "youtube",
        "v3",
        developerKey=API_KEY,
        cache_discovery=False
    )

    # Lightweight API request: retrieve the authenticated
    # public API service's own metadata.
    response = youtube.i18nLanguages().list(
        part="snippet",
        hl="en"
    ).execute()

    print("YouTube API connection: SUCCESS")
    print("API response status: SUCCESS")
    print("Languages returned:", len(response.get("items", [])))
    print("API key was not displayed.")

except HttpError as e:
    print("YouTube API HTTP error:", e.resp.status)
    print("Error details:", e.content.decode("utf-8")[:1000])

except Exception as e:
    print("Connection check failed:", str(e)[:1000])

In [ ]:
# ============================================================
# NOTEBOOK 10 — INSPECT FIRST DISCOVERY BATCH
# Read-only; no API calls and no file modifications
# ============================================================

print("Master dataset columns:")
print(master_df.columns.tolist())

print("\nFirst batch columns:")
print(batch_df.columns.tolist())

# Display available identification fields
preferred_columns = [
    "movie_record_id",
    "movie_title",
    "year",
    "rank",
    "language",
    "tmdb_id"
]

display_columns = [
    col for col in preferred_columns
    if col in batch_df.columns
]

print("\nSelected first-batch movies:")
display(batch_df[display_columns])

print("\nBatch size:", len(batch_df))
print("Estimated search.list quota:", len(batch_df) * 100)
print("No API calls made. No files modified.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — PREPARE DISCOVERY BATCH USING VALIDATED IDS
# Read-only; no API calls and no file modifications
# ============================================================

import pandas as pd
from pathlib import Path

REFERENCE_PATH = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_AUDIENCE_MOVIE_REFERENCE_VALIDATED.csv"
)

reference_df = pd.read_csv(REFERENCE_PATH)

reference_df["movie_record_id"] = (
    reference_df["movie_record_id"].astype(str)
)

explored_ids = set(
    pilot_df["movie_record_id"].dropna().astype(str)
)

remaining_reference_df = (
    reference_df[
        ~reference_df["movie_record_id"].isin(explored_ids)
    ]
    .copy()
    .sort_values(["year", "rank"])
    .reset_index(drop=True)
)

batch_df = remaining_reference_df.head(10).copy()

print("Validated reference shape:", reference_df.shape)
print("Movies with TMDB IDs:", reference_df["tmdb_id"].notna().sum())
print("Movies with IMDb IDs:", reference_df["imdb_id"].notna().sum())
print("Remaining movies:", len(remaining_reference_df))
print("First batch size:", len(batch_df))

display(
    batch_df[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "language",
            "tmdb_id",
            "imdb_id"
        ]
    ]
)

print("\nBoth IDs will be retained as identity metadata.")
print("No API calls made. No files modified.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — YOUTUBE DISCOVERY: BATCH 01
# 10 movies | Up to 10 candidates per movie
# Saves incrementally; does not overwrite the pilot
# ============================================================

import pandas as pd
import time
from pathlib import Path
from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

OUTPUT_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = (
    OUTPUT_DIR / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_01.csv"
)

if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"Output already exists: {OUTPUT_PATH}\n"
        "To avoid overwriting existing discovery results, "
        "inspect that file before continuing."
    )

API_KEY = userdata.get("YOUTUBE_API_KEY")

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY,
    cache_discovery=False
)

# ------------------------------------------------------------
# 2. Prepare batch
# ------------------------------------------------------------

required_columns = [
    "movie_record_id",
    "movie_title",
    "year",
    "language",
    "tmdb_id",
    "imdb_id"
]

missing = [
    col for col in required_columns
    if col not in batch_df.columns
]

if missing:
    raise KeyError(f"Missing required columns: {missing}")

discovery_results = []
completed_movies = []
failed_movies = []

print("=" * 75)
print("YOUTUBE DISCOVERY — BATCH 01")
print("=" * 75)
print("Movies to search:", len(batch_df))
print("Maximum candidates:", len(batch_df) * 10)
print("Estimated search quota:", len(batch_df) * 100)

# ------------------------------------------------------------
# 3. Search one movie at a time and checkpoint results
# ------------------------------------------------------------

for index, movie in batch_df.iterrows():

    movie_id = str(movie["movie_record_id"])
    title = str(movie["movie_title"])
    year = int(movie["year"])
    language = str(movie["language"])

    query = f'"{title}" {year} Indian movie review reaction'

    print(f"\n[{len(completed_movies) + len(failed_movies) + 1}/"
          f"{len(batch_df)}] Searching: {title} ({year})")

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=10,
            order="relevance",
            regionCode="IN"
        ).execute()

        items = response.get("items", [])

        for item in items:
            snippet = item.get("snippet", {})
            video_id = item.get("id", {}).get("videoId")

            if not video_id:
                continue

            discovery_results.append({
                "movie_record_id": movie_id,
                "movie_title": title,
                "movie_year": year,
                "language": language,
                "tmdb_id": movie["tmdb_id"],
                "imdb_id": movie["imdb_id"],
                "search_query": query,
                "video_id": video_id,
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "discovery_source": "YouTube_Data_API_v3",
                "discovery_batch": "Batch_01"
            })

        completed_movies.append(movie_id)

        # Checkpoint after each successful movie
        checkpoint_df = pd.DataFrame(discovery_results)

        if not checkpoint_df.empty:
            checkpoint_df = checkpoint_df.drop_duplicates(
                subset=["video_id"]
            )
            checkpoint_df.to_csv(OUTPUT_PATH, index=False)

        print("Candidates returned:", len(items))
        print("Unique candidates collected:",
              len(checkpoint_df))

    except HttpError as e:
        error_text = e.content.decode(
            "utf-8", errors="replace"
        )[:500]

        failed_movies.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "error": f"HTTP {e.resp.status}: {error_text}"
        })

        print("API request failed:", e.resp.status)
        print(error_text)

        # Stop if quota is exhausted
        if b"quotaExceeded" in e.content:
            print("\nQuota exhausted. Stopping safely.")
            break

    except Exception as e:
        failed_movies.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "error": str(e)[:500]
        })

        print("Error:", str(e)[:500])

    time.sleep(0.5)

# ------------------------------------------------------------
# 4. Final batch summary
# ------------------------------------------------------------

if discovery_results:
    final_batch_df = pd.DataFrame(
        discovery_results
    ).drop_duplicates(subset=["video_id"])

    final_batch_df.to_csv(
        OUTPUT_PATH,
        index=False
    )

    print("\nSaved file:", OUTPUT_PATH)
    print("Final candidate records:", len(final_batch_df))
    print("Unique video IDs:",
          final_batch_df["video_id"].nunique())
else:
    final_batch_df = pd.DataFrame()
    print("\nNo candidate records were returned.")

print("\nMovies successfully searched:", len(completed_movies))
print("Movies with errors:", len(failed_movies))

if failed_movies:
    display(pd.DataFrame(failed_movies))

print("\nOriginal pilot discovery file remains unchanged.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 01 DISCOVERY QUALITY REVIEW
# Read-only; no API calls and no file modifications
# ============================================================

import pandas as pd

print("Batch 01 shape:", final_batch_df.shape)
print("Unique videos:", final_batch_df["video_id"].nunique())
print("Movies represented:",
      final_batch_df["movie_record_id"].nunique())

print("\nCandidates per movie:")
display(
    final_batch_df.groupby(
        ["movie_record_id", "movie_title"]
    ).size().reset_index(name="candidate_count")
)

print("\nVideo title review — all 100 candidates:")
for _, row in final_batch_df.iterrows():
    print(
        f"\nMovie: {row['movie_title']} ({row['movie_year']})"
        f"\nVideo: {row['video_title']}"
        f"\nChannel: {row['channel_title']}"
        f"\nVideo ID: {row['video_id']}"
    )

print("\nDiscovery quality review prepared.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 01 AUTOMATED VIDEO SCREENING
# Rule-based first pass; identity verification remains separate
# ============================================================

import pandas as pd
import re
from pathlib import Path

screening_df = final_batch_df.copy()

def classify_video(row):

    title = str(row.get("video_title", "")).lower()
    description = str(row.get("description", "")).lower()
    text = title + " " + description

    # Identify content type
    if re.search(r"\b(full movie|full film|complete movie|entire movie)\b", text):
        video_type = "Full Movie"

    elif re.search(r"\b(trailer|teaser|official promo)\b", title):
        video_type = "Trailer"

    elif re.search(r"\b(song|lyrical|music video|jukebox|bumbro|apun bola)\b", title):
        video_type = "Song"

    elif re.search(r"\b(scene|best scene|fight scene|dialogue scene|clip)\b", title):
        video_type = "Scene or Clip"

    elif re.search(r"\b(movie reaction|film reaction|first time watching|reacts to|reaction)\b", title):
        video_type = "Audience Reaction"

    elif re.search(r"\b(movie review|film review|review|roast|must watch|cringe|marvel)\b", title):
        video_type = "Movie Review"

    elif re.search(r"\b(explained|story explained|ending explained)\b", title):
        video_type = "Movie Explanation"

    elif re.search(r"\b(hit or flop|box office|budget|collection|verdict)\b", title):
        video_type = "Box Office Discussion"

    else:
        video_type = "Other / Unclear"

    # Identify likely relevance to audience feedback
    if video_type in ["Audience Reaction", "Movie Review"]:
        relevance = "High — Audience Feedback Candidate"

    elif video_type in [
        "Movie Explanation",
        "Box Office Discussion"
    ]:
        relevance = "Medium — Contextual Discussion"

    elif video_type in ["Scene or Clip", "Song"]:
        relevance = "Low — Scene/Song Reaction Context"

    elif video_type == "Trailer":
        relevance = "Low — Trailer Feedback"

    elif video_type == "Full Movie":
        relevance = "Exclude from Review Sentiment"

    else:
        relevance = "Needs Manual Review"

    # Ambiguous movie title requires extra identity checking
    ambiguous = str(row.get("movie_title", "")).strip().lower() == "indian"

    if ambiguous:
        identity_status = "Manual Identity Verification Required"
    else:
        identity_status = "Candidate — Identity Not Yet Verified"

    return pd.Series({
        "screened_video_type": video_type,
        "screening_relevance": relevance,
        "identity_status": identity_status
    })


screening_df[
    [
        "screened_video_type",
        "screening_relevance",
        "identity_status"
    ]
] = screening_df.apply(classify_video, axis=1)

print("=" * 70)
print("BATCH 01 — SCREENING SUMMARY")
print("=" * 70)

print("\nVideo type distribution:")
display(
    screening_df["screened_video_type"]
    .value_counts()
    .rename_axis("video_type")
    .reset_index(name="candidate_count")
)

print("\nRelevance distribution:")
display(
    screening_df["screening_relevance"]
    .value_counts()
    .rename_axis("relevance")
    .reset_index(name="candidate_count")
)

print("\nCandidates for audience-feedback analysis:")
feedback_candidates = screening_df[
    screening_df["screening_relevance"]
    == "High — Audience Feedback Candidate"
]

print("Total:", len(feedback_candidates))

display(
    feedback_candidates[
        [
            "movie_title",
            "video_title",
            "channel_title",
            "screened_video_type",
            "identity_status",
            "video_id"
        ]
    ]
)

# Save to a new file; never overwrite discovery or pilot files
OUTPUT_PATH = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_SCREENING_BATCH_01.csv"
)

if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"Output already exists; not overwriting: {OUTPUT_PATH}"
    )

screening_df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved screening file:", OUTPUT_PATH)
print("Discovery source file remains unchanged.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 01 VIDEO STATISTICS & COMMENT AVAILABILITY
# Read-only API retrieval; saves a separate statistics file
# ============================================================

import pandas as pd
from pathlib import Path
from googleapiclient.errors import HttpError

# Refine provisional candidates: exclude song/scene terms
candidate_mask = (
    screening_df["screening_relevance"].eq(
        "High — Audience Feedback Candidate"
    )
    & ~screening_df["video_title"].str.contains(
        r"\b(song|lyrical|music video|jukebox)\b",
        case=False,
        na=False
    )
)

stats_candidates = screening_df.loc[candidate_mask].copy()

# Remove duplicate video IDs
stats_candidates = stats_candidates.drop_duplicates(
    subset=["video_id"]
).reset_index(drop=True)

print("Videos selected for statistics:", len(stats_candidates))

video_ids = stats_candidates["video_id"].tolist()
statistics_records = []

# YouTube videos.list accepts up to 50 IDs per request.
for start in range(0, len(video_ids), 50):

    id_batch = video_ids[start:start + 50]

    try:
        response = youtube.videos().list(
            part="statistics,status,contentDetails",
            id=",".join(id_batch)
        ).execute()

        returned = {
            item["id"]: item
            for item in response.get("items", [])
        }

        for video_id in id_batch:

            item = returned.get(video_id)

            if item is None:
                statistics_records.append({
                    "video_id": video_id,
                    "video_available": False,
                    "view_count": None,
                    "like_count": None,
                    "comment_count": None,
                    "comments_disabled": None,
                    "video_duration": None,
                    "privacy_status": None
                })
                continue

            stats = item.get("statistics", {})
            status = item.get("status", {})
            details = item.get("contentDetails", {})

            statistics_records.append({
                "video_id": video_id,
                "video_available": True,
                "view_count": int(stats.get("viewCount", 0)),
                "like_count": int(stats.get("likeCount", 0)),
                "comment_count": (
                    int(stats["commentCount"])
                    if "commentCount" in stats else None
                ),
                "comments_disabled": (
                    "commentCount" not in stats
                ),
                "video_duration": details.get("duration"),
                "privacy_status": status.get("privacyStatus")
            })

    except HttpError as e:
        print("Statistics API error:", e.resp.status)
        print(e.content.decode("utf-8", errors="replace")[:500])
        raise

statistics_df = pd.DataFrame(statistics_records)

# Attach statistics to candidate metadata
stats_review_df = stats_candidates.merge(
    statistics_df,
    on="video_id",
    how="left",
    validate="one_to_one"
)

print("\nVIDEO STATISTICS SUMMARY")
print("=" * 70)

print("Videos checked:", len(stats_review_df))
print("Available videos:", stats_review_df["video_available"].sum())
print("Videos with comments:",
      stats_review_df["comment_count"].notna().sum())
print("Comments disabled or unavailable:",
      stats_review_df["comments_disabled"].sum())

print("\nMovie-wise comment availability:")
display(
    stats_review_df.groupby("movie_title").agg(
        candidate_videos=("video_id", "nunique"),
        available_videos=("video_available", "sum"),
        total_views=("view_count", "sum"),
        total_likes=("like_count", "sum"),
        available_comment_counts=("comment_count", "count")
    ).reset_index()
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_VIDEO_STATISTICS_BATCH_01.csv"
)

if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"File already exists; refusing to overwrite: {OUTPUT_PATH}"
    )

stats_review_df.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)
print("No comments downloaded. Existing pilot files unchanged.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 01 COMMENT VOLUME PRIORITIZATION
# Uses existing statistics; no API calls
# ============================================================

# Sort by movie and available comment count
comment_priority_df = stats_review_df.copy()

comment_priority_df["comment_count"] = pd.to_numeric(
    comment_priority_df["comment_count"],
    errors="coerce"
)

print("=" * 70)
print("BATCH 01 — COMMENT VOLUME SUMMARY")
print("=" * 70)

print(
    "Total available comments across candidate videos:",
    int(comment_priority_df["comment_count"].fillna(0).sum())
)

print("\nMovie-wise comment availability:")
movie_comment_summary = (
    comment_priority_df.groupby("movie_title")
    .agg(
        candidate_videos=("video_id", "nunique"),
        total_available_comments=("comment_count", "sum"),
        median_comments_per_video=("comment_count", "median"),
        total_views=("view_count", "sum")
    )
    .reset_index()
    .sort_values("total_available_comments", ascending=False)
)

display(movie_comment_summary)

print("\nTop 20 videos by available comment count:")
display(
    comment_priority_df[
        [
            "movie_title",
            "video_title",
            "channel_title",
            "video_id",
            "view_count",
            "comment_count",
            "screened_video_type",
            "identity_status"
        ]
    ]
    .sort_values("comment_count", ascending=False)
    .head(20)
)

print("\nNo API calls made. No files modified.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — PREPARE BATCH 01 COMMENT ACQUISITION CANDIDATES
# ============================================================

eligible_df = stats_review_df.copy()

# Exclude ambiguous movie identity
eligible_df = eligible_df[
    eligible_df["movie_title"].str.strip().str.lower() != "indian"
].copy()

# Exclude full movies, songs, trailers and scene/clip reactions
excluded_types = [
    "Full Movie",
    "Song",
    "Trailer",
    "Scene or Clip"
]

eligible_df = eligible_df[
    ~eligible_df["screened_video_type"].isin(excluded_types)
].copy()

# Keep reviews and audience reactions only
eligible_df = eligible_df[
    eligible_df["screened_video_type"].isin([
        "Audience Reaction",
        "Movie Review"
    ])
].copy()

eligible_df = eligible_df.drop_duplicates(
    subset=["video_id"]
).reset_index(drop=True)

eligible_df["proposed_comment_limit"] = 100
eligible_df["verification_status"] = "Pending Manual Verification"

print("=" * 70)
print("BATCH 01 — COMMENT ACQUISITION CANDIDATES")
print("=" * 70)

print("Eligible candidates:", len(eligible_df))
print("Movies represented:",
      eligible_df["movie_record_id"].nunique())

print("\nCandidates per movie:")
display(
    eligible_df.groupby("movie_title")
    .size()
    .reset_index(name="candidate_videos")
)

print("\nCandidate verification sheet:")
display(
    eligible_df[
        [
            "movie_record_id",
            "movie_title",
            "video_id",
            "video_title",
            "channel_title",
            "screened_video_type",
            "comment_count",
            "proposed_comment_limit",
            "verification_status"
        ]
    ]
)

print("\nNo comments downloaded. No files modified.")

In [ ]:
from pathlib import Path
import pandas as pd

# Keep only candidates currently shortlisted
verification_df = eligible_df.copy()

# Add fields for manual verification
verification_df["film_identity_verified"] = ""
verification_df["content_relevance"] = ""
verification_df["reaction_part_or_series"] = ""
verification_df["final_acquisition_decision"] = ""
verification_df["verification_notes"] = ""

# Add direct YouTube links for convenient checking
verification_df["video_url"] = (
    "https://www.youtube.com/watch?v=" + verification_df["video_id"].astype(str)
)

# Keep useful columns together
preferred_columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "video_title",
    "channel_title",
    "screened_video_type",
    "comment_count",
    "proposed_comment_limit",
    "video_url",
    "film_identity_verified",
    "content_relevance",
    "reaction_part_or_series",
    "final_acquisition_decision",
    "verification_notes"
]

verification_df = verification_df[preferred_columns]

output_path = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/audience_sentiment"
    / "ICMB_130_YOUTUBE_MANUAL_VERIFICATION_BATCH_01.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
verification_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("=" * 70)
print("MANUAL VIDEO VERIFICATION SHEET CREATED")
print("=" * 70)
print("Candidates:", len(verification_df))
print("Movies:", verification_df["movie_record_id"].nunique())
print("Saved:", output_path)
print("\nVerification fields:")
print("film_identity_verified: Yes / No / Uncertain")
print("content_relevance: Relevant / Partially Relevant / Irrelevant")
print("final_acquisition_decision: Include / Exclude / Hold")

In [ ]:
from pathlib import Path
import pandas as pd

# Manually supplied and user-verified full-movie uploads
full_movie_sources = [
    {
        "movie_record_id": "2001_2",
        "movie_title": "Kabhi Khushi Kabhie Gham...",
        "movie_year": 2001,
        "video_id": "uFIy9vgicKs"
    },
    {
        "movie_record_id": "2001_3",
        "movie_title": "Lagaan",
        "movie_year": 2001,
        "video_id": "9rH_iEeSta0"
    },
    {
        "movie_record_id": "2002_1",
        "movie_title": "Devdas",
        "movie_year": 2002,
        "video_id": "XG9U_elK3Cw"
    }
]

# Retrieve current YouTube metadata
video_ids = [x["video_id"] for x in full_movie_sources]

response = youtube.videos().list(
    part="snippet,statistics,status,contentDetails",
    id=",".join(video_ids)
).execute()

api_records = {item["id"]: item for item in response.get("items", [])}

rows = []

for source in full_movie_sources:
    video_id = source["video_id"]
    item = api_records.get(video_id)

    row = {
        **source,
        "video_url": f"https://www.youtube.com/watch?v={video_id}",
        "source_type": "Full Movie Upload",
        "film_identity_verified": "Yes",
        "content_relevance": "Relevant",
        "verification_basis": "Manually verified by project researcher",
        "final_acquisition_decision": "Include",
        "metadata_retrieval_status": "Found" if item else "Not Found"
    }

    if item:
        snippet = item.get("snippet", {})
        statistics = item.get("statistics", {})
        status = item.get("status", {})
        details = item.get("contentDetails", {})

        row.update({
            "youtube_upload_date": snippet.get("publishedAt"),
            "youtube_video_title": snippet.get("title"),
            "channel_title": snippet.get("channelTitle"),
            "view_count": statistics.get("viewCount"),
            "like_count": statistics.get("likeCount"),
            "comment_count": statistics.get("commentCount"),
            "comments_disabled": (
                "Yes" if statistics.get("commentCount") is None else "No"
            ),
            "privacy_status": status.get("privacyStatus"),
            "video_duration": details.get("duration")
        })

    rows.append(row)

full_movie_df = pd.DataFrame(rows)

output_path = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/audience_sentiment"
    / "ICMB_130_YOUTUBE_SUPPLEMENTAL_FULL_MOVIE_CANDIDATES.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
full_movie_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("=" * 70)
print("SUPPLEMENTAL FULL-MOVIE SOURCES")
print("=" * 70)
print(full_movie_df[
    [
        "movie_title", "video_id", "metadata_retrieval_status",
        "youtube_upload_date", "view_count", "like_count", "comment_count"
    ]
].to_string(index=False))
print("\nSaved:", output_path)

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
PROCESSED = BASE / "01_data/processed/audience_sentiment"

# Read manually verified file using its original encoding
manual_df = pd.read_csv(
    PROCESSED / "ICMB_130_YOUTUBE_MANUAL_VERIFICATION_BATCH_01.csv",
    encoding="utf-8-sig"
)

# Read supplemental full-movie sources
full_df = pd.read_csv(
    PROCESSED / "ICMB_130_YOUTUBE_SUPPLEMENTAL_FULL_MOVIE_CANDIDATES.csv",
    encoding="utf-8-sig"
)

# Validate required columns
required_columns = [
    "video_id",
    "movie_title",
    "final_acquisition_decision",
    "screened_video_type"
]

missing = [c for c in required_columns if c not in manual_df.columns]

if missing:
    raise ValueError(f"Missing columns in manual verification file: {missing}")

# Select only manually approved videos
approved_manual = manual_df[
    manual_df["final_acquisition_decision"]
    .astype(str).str.strip().str.lower().eq("include")
].copy()

approved_manual["source_type"] = approved_manual["screened_video_type"]
approved_manual["source_verification"] = "Manual Verification"

# Standardize supplemental full-movie sources
approved_full = full_df.copy()
approved_full["source_verification"] = "Manually Supplied Full Movie"

# Combine both source groups
acquisition_df = pd.concat(
    [approved_manual, approved_full],
    ignore_index=True,
    sort=False
)

# Remove duplicate video IDs
acquisition_df = acquisition_df.drop_duplicates(
    subset=["video_id"],
    keep="first"
).reset_index(drop=True)

# Save as a new file
output_path = (
    PROCESSED /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_01.csv"
)

acquisition_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("APPROVED COMMENT ACQUISITION SOURCES — BATCH 01")
print("=" * 70)

print("Manually approved candidates:", len(approved_manual))
print("Supplemental full-movie candidates:", len(approved_full))
print("Total unique approved videos:", acquisition_df["video_id"].nunique())
print("Movies represented:", acquisition_df["movie_title"].nunique())

print("\nSources by type:")
print(acquisition_df["source_type"].value_counts(dropna=False))

print("\nMovie-wise sources:")
print(
    acquisition_df.groupby("movie_title")["video_id"]
    .nunique()
    .to_string()
)

print("\nSaved:", output_path)

In [ ]:
from pathlib import Path
import pandas as pd

file_path = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/audience_sentiment"
    / "ICMB_130_YOUTUBE_MANUAL_VERIFICATION_BATCH_01.csv"
)

manual_check = pd.read_csv(file_path, encoding="utf-8-sig")

print("Decision values exactly as stored:")
for value, count in manual_check["final_acquisition_decision"].value_counts(dropna=False).items():
    print(repr(value), ":", count)

decisions = (
    manual_check["final_acquisition_decision"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

approved_check = manual_check[decisions == "include"].copy()

print("\nApproved rows found:", len(approved_check))

print(
    approved_check[
        ["movie_title", "video_id", "video_title",
         "final_acquisition_decision"]
    ].to_string(index=False)
)

In [ ]:
import pandas as pd
from pathlib import Path

# Read the file already uploaded into Colab
filename = "ICMB_130_YOUTUBE_MANUAL_VERIFICATION_BATCH_01.csv"

manual_df = pd.read_csv(
    filename,
    encoding="latin1"
)

# Normalize decision values
decisions = (
    manual_df["final_acquisition_decision"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Select approved records
approved_manual = manual_df[decisions == "include"].copy()

# Validate the records before saving
assert len(manual_df) == 45, f"Expected 45 rows, found {len(manual_df)}"
assert len(approved_manual) == 5, (
    f"Expected 5 approved records, found {len(approved_manual)}"
)

# Save corrected file in Google Drive
PROCESSED = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/audience_sentiment"
)

drive_path = (
    PROCESSED /
    "ICMB_130_YOUTUBE_MANUAL_VERIFICATION_BATCH_01_CORRECTED.csv"
)

manual_df.to_csv(
    drive_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 65)
print("MANUAL VERIFICATION FILE — CORRECTED")
print("=" * 65)

print("Total records:", len(manual_df))
print("Include:", (decisions == "include").sum())
print("Hold:", (decisions == "hold").sum())
print("Exclude:", (decisions == "exclude").sum())

print("\nApproved videos:")
print(
    approved_manual[
        ["movie_title", "video_id", "video_title"]
    ].to_string(index=False)
)

print("\nSaved to:", drive_path)

In [ ]:
from pathlib import Path
import pandas as pd

PROCESSED = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/audience_sentiment"
)

# Load corrected manual verification
manual_df = pd.read_csv(
    PROCESSED / "ICMB_130_YOUTUBE_MANUAL_VERIFICATION_BATCH_01_CORRECTED.csv",
    encoding="utf-8-sig"
)

# Load supplemental full-movie sources
full_df = pd.read_csv(
    PROCESSED / "ICMB_130_YOUTUBE_SUPPLEMENTAL_FULL_MOVIE_CANDIDATES.csv",
    encoding="utf-8-sig"
)

# Retain only manually approved candidates
approved_manual = manual_df[
    manual_df["final_acquisition_decision"]
    .astype(str).str.strip().str.casefold().eq("include")
].copy()

approved_manual["source_type"] = approved_manual["screened_video_type"]
approved_manual["source_verification"] = "Manual Verification"

# Standardize full-movie sources
approved_full = full_df.copy()
approved_full["source_verification"] = "Manually Supplied Full Movie"

# Combine without modifying either original file
acquisition_df = pd.concat(
    [approved_manual, approved_full],
    ignore_index=True,
    sort=False
)

# Ensure unique video IDs
acquisition_df = acquisition_df.drop_duplicates(
    subset=["video_id"],
    keep="first"
).reset_index(drop=True)

# Validate
assert acquisition_df["video_id"].nunique() == len(acquisition_df)
assert len(acquisition_df) == 8, (
    f"Expected 8 unique sources, found {len(acquisition_df)}"
)

# Save new consolidated file
output_path = (
    PROCESSED /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_01.csv"
)

acquisition_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("APPROVED COMMENT ACQUISITION SOURCES — BATCH 01")
print("=" * 70)

print("Total unique videos:", acquisition_df["video_id"].nunique())
print("Movies represented:", acquisition_df["movie_title"].nunique())

print("\nMovie-wise source counts:")
print(
    acquisition_df.groupby("movie_title")["video_id"]
    .nunique().to_string()
)

print("\nSource types:")
print(acquisition_df["source_type"].value_counts(dropna=False))

print("\nSaved:", output_path)

In [ ]:
from pathlib import Path
import pandas as pd

PROCESSED = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/audience_sentiment"
)

SOURCE_FILE = (
    PROCESSED /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_01.csv"
)

sources_df = pd.read_csv(SOURCE_FILE, encoding="utf-8-sig")

required = ["movie_title", "video_id", "source_type", "comment_count"]
missing = [c for c in required if c not in sources_df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

sources_df["comment_count"] = pd.to_numeric(
    sources_df["comment_count"], errors="coerce"
)

# Allocate a maximum of 100 comments per movie
movie_video_counts = sources_df.groupby("movie_title")["video_id"].transform("nunique")
sources_df["proposed_comment_limit"] = (
    100 / movie_video_counts
).astype(int)

# Readiness summary
print("=" * 70)
print("BATCH 01 — COMMENT ACQUISITION READINESS")
print("=" * 70)

print("Unique videos:", sources_df["video_id"].nunique())
print("Movies:", sources_df["movie_title"].nunique())
print("Missing video IDs:", sources_df["video_id"].isna().sum())
print("Missing comment counts:", sources_df["comment_count"].isna().sum())

print("\nPer-video sampling plan:")
print(
    sources_df[
        ["movie_title", "video_id", "source_type",
         "comment_count", "proposed_comment_limit"]
    ].to_string(index=False)
)

print("\nMaximum planned sample:",
      sources_df["proposed_comment_limit"].sum())

In [ ]:
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone
from googleapiclient.errors import HttpError
import time

PROCESSED = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/processed/audience_sentiment"
)

RAW = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/raw/audience_sentiment/youtube/comments"
)

SOURCE_FILE = (
    PROCESSED /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_01.csv"
)

sources_df = pd.read_csv(SOURCE_FILE, encoding="utf-8-sig")

sources_df["comment_count"] = pd.to_numeric(
    sources_df["comment_count"], errors="coerce"
)

movie_video_counts = sources_df.groupby("movie_title")["video_id"].transform("nunique")
sources_df["proposed_comment_limit"] = (
    100 / movie_video_counts
).astype(int)

all_comments = []
acquisition_log = []

retrieval_timestamp = datetime.now(timezone.utc).isoformat()

for _, source in sources_df.iterrows():

    video_id = str(source["video_id"])
    limit = int(source["proposed_comment_limit"])

    print(f"\nCollecting: {source['movie_title']} | {video_id} | Limit: {limit}")

    retrieved = 0
    status = "Success"
    error_message = ""

    try:
        response = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=min(limit, 100),
            order="time",
            textFormat="plainText"
        ).execute()

        for item in response.get("items", []):

            comment = item["snippet"]["topLevelComment"]
            snippet = comment["snippet"]

            all_comments.append({
                "movie_record_id": source.get("movie_record_id", ""),
                "movie_title": source["movie_title"],
                "movie_year": source.get("movie_year", ""),
                "video_id": video_id,
                "video_title": source.get(
                    "video_title",
                    source.get("youtube_video_title", "")
                ),
                "source_type": source["source_type"],
                "comment_id": comment["id"],
                "comment_text": snippet.get("textOriginal", ""),
                "author_channel_id": (
                    snippet.get("authorChannelId", {}).get("value", "")
                ),
                "published_at": snippet.get("publishedAt", ""),
                "updated_at": snippet.get("updatedAt", ""),
                "like_count": snippet.get("likeCount", 0),
                "comment_reply_count": item["snippet"].get(
                    "totalReplyCount", 0
                ),
                "retrieval_order": "Most Recent",
                "retrieved_at": retrieval_timestamp
            })

        retrieved = len(response.get("items", []))

        if retrieved == 0:
            status = "No Comments Returned"

    except HttpError as e:
        status = "API Error"
        error_message = str(e)[:500]

    except Exception as e:
        status = "Retrieval Error"
        error_message = str(e)[:500]

    acquisition_log.append({
        "movie_title": source["movie_title"],
        "video_id": video_id,
        "source_type": source["source_type"],
        "reported_comment_count": source["comment_count"],
        "proposed_comment_limit": limit,
        "comments_retrieved": retrieved,
        "acquisition_status": status,
        "error_message": error_message,
        "retrieved_at": retrieval_timestamp
    })

    print("Retrieved:", retrieved, "| Status:", status)

# Create separate raw output
comments_df_batch01 = pd.DataFrame(all_comments)
acquisition_log_df = pd.DataFrame(acquisition_log)

RAW.mkdir(parents=True, exist_ok=True)

comments_path = RAW / "ICMB_130_YOUTUBE_COMMENT_BATCH_01.csv"
log_path = PROCESSED / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_01.csv"

comments_df_batch01.to_csv(
    comments_path, index=False, encoding="utf-8-sig"
)

acquisition_log_df.to_csv(
    log_path, index=False, encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("BATCH 01 — COMMENT ACQUISITION SUMMARY")
print("=" * 70)
print("Videos processed:", len(acquisition_log_df))
print("Comments retrieved:", len(comments_df_batch01))
print("Unique comment IDs:", comments_df_batch01["comment_id"].nunique())
print("Movies represented:", comments_df_batch01["movie_title"].nunique())
print("Successful videos:", (acquisition_log_df["acquisition_status"] == "Success").sum())
print("Comments file:", comments_path)
print("Acquisition log:", log_path)

print("\nMovie-wise comment counts:")
if not comments_df_batch01.empty:
    print(
        comments_df_batch01.groupby("movie_title")["comment_id"]
        .nunique().to_string()
    )

print("\nAcquisition status:")
print(acquisition_log_df[
    ["movie_title", "video_id", "proposed_comment_limit",
     "comments_retrieved", "acquisition_status"]
].to_string(index=False))

In [ ]:
import pandas as pd
from pathlib import Path

RAW = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/raw/audience_sentiment/youtube/comments"
)

comments_path = RAW / "ICMB_130_YOUTUBE_COMMENT_BATCH_01.csv"

comments_df_batch01 = pd.read_csv(
    comments_path,
    encoding="utf-8-sig"
)

# Basic text-quality profiling
text = comments_df_batch01["comment_text"].fillna("").astype(str)

comments_df_batch01["comment_char_count"] = text.str.len()
comments_df_batch01["comment_word_count"] = text.str.split().str.len()
comments_df_batch01["is_empty"] = text.str.strip().eq("")
comments_df_batch01["contains_url"] = text.str.contains(
    r"https?://|www\.",
    case=False,
    regex=True
)

print("=" * 70)
print("BATCH 01 — COMMENT QUALITY PROFILE")
print("=" * 70)

print("Total comments:", len(comments_df_batch01))
print("Unique comment IDs:", comments_df_batch01["comment_id"].nunique())
print("Empty comments:", comments_df_batch01["is_empty"].sum())
print("Duplicate comment texts:", text.duplicated().sum())
print("Comments containing URLs:", comments_df_batch01["contains_url"].sum())

print("\nComment length statistics:")
print(
    comments_df_batch01[
        ["comment_char_count", "comment_word_count"]
    ].describe().round(2)
)

print("\nMovie-wise comment counts:")
print(
    comments_df_batch01.groupby("movie_title")["comment_id"]
    .nunique().to_string()
)

print("\nSource-type distribution:")
print(
    comments_df_batch01["source_type"]
    .value_counts(dropna=False).to_string()
)

print("\nSample comments:")
print(
    comments_df_batch01[
        ["movie_title", "source_type", "comment_text"]
    ].head(10).to_string(index=False)
)

In [ ]:
import pandas as pd
import re
from pathlib import Path

RAW = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data/raw/audience_sentiment/youtube/comments"
)

comments_path = RAW / "ICMB_130_YOUTUBE_COMMENT_BATCH_01.csv"

comments_df_batch01 = pd.read_csv(
    comments_path,
    encoding="utf-8-sig"
)

# Recalculate text features because the original CSV is unchanged
text = comments_df_batch01["comment_text"].fillna("").astype(str)

comments_df_batch01["comment_char_count"] = text.str.len()
comments_df_batch01["comment_word_count"] = text.str.split().str.len()

# Duplicate investigation
duplicate_mask = text.duplicated(keep=False)

duplicate_df = comments_df_batch01.loc[
    duplicate_mask,
    ["movie_title", "video_id", "comment_id", "comment_text"]
].copy()

# Script profiling
def detect_script(value):
    value = str(value)

    scripts = []

    if re.search(r"[\u0900-\u097F]", value):
        scripts.append("Devanagari")
    if re.search(r"[\u0B80-\u0BFF]", value):
        scripts.append("Tamil")
    if re.search(r"[\u0C00-\u0C7F]", value):
        scripts.append("Telugu")
    if re.search(r"[\u0C80-\u0CFF]", value):
        scripts.append("Kannada")
    if re.search(r"[A-Za-z]", value):
        scripts.append("Latin")

    if not scripts:
        return "Other / Emoji / Symbols"

    if len(scripts) > 1:
        return "Mixed Script"

    return scripts[0]

comments_df_batch01["script_profile"] = text.apply(detect_script)

# Comment length categories
comments_df_batch01["length_category"] = pd.cut(
    comments_df_batch01["comment_word_count"],
    bins=[-1, 2, 5, 15, float("inf")],
    labels=[
        "Very Short (0-2 words)",
        "Short (3-5 words)",
        "Medium (6-15 words)",
        "Long (16+ words)"
    ]
)

print("=" * 70)
print("BATCH 01 — DUPLICATE AND SCRIPT PROFILING")
print("=" * 70)

print("Total comments:", len(comments_df_batch01))
print("Duplicate text records:", duplicate_mask.sum())
print("Unique duplicated text values:", text[duplicate_mask].nunique())

print("\nScript distribution:")
print(comments_df_batch01["script_profile"].value_counts().to_string())

print("\nScript distribution by movie:")
print(pd.crosstab(
    comments_df_batch01["movie_title"],
    comments_df_batch01["script_profile"]
).to_string())

print("\nComment length distribution:")
print(comments_df_batch01["length_category"].value_counts(sort=False).to_string())

print("\nDuplicate examples:")
if not duplicate_df.empty:
    print(duplicate_df.head(15).to_string(index=False))

In [ ]:
import pandas as pd
import re
from pathlib import Path

BASE = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW = BASE / "01_data/raw/audience_sentiment/youtube/comments"
PROCESSED = BASE / "01_data/processed/audience_sentiment"

comments_path = RAW / "ICMB_130_YOUTUBE_COMMENT_BATCH_01.csv"

df = pd.read_csv(comments_path, encoding="utf-8-sig")

# Preserve original comment text
text = df["comment_text"].fillna("").astype(str)
normalized = text.str.strip().str.casefold()

# Text length
df["comment_char_count"] = text.str.len()
df["comment_word_count"] = text.str.split().str.len()

# Corrected script detection
def detect_script(value):
    value = str(value)

    script_patterns = {
        "Devanagari": r"[\u0900-\u097F]",
        "Bengali": r"[\u0980-\u09FF]",
        "Tamil": r"[\u0B80-\u0BFF]",
        "Telugu": r"[\u0C00-\u0C7F]",
        "Kannada": r"[\u0C80-\u0CFF]",
        "Latin": r"[A-Za-z]"
    }

    detected = [
        name for name, pattern in script_patterns.items()
        if re.search(pattern, value)
    ]

    if len(detected) == 0:
        return "Other / Emoji / Symbols"

    if len(detected) > 1:
        return "Mixed Script"

    return detected[0]

df["script_profile"] = text.apply(detect_script)

# Quality flags
df["is_empty"] = normalized.eq("")
df["is_very_short"] = df["comment_word_count"] <= 2
df["contains_url"] = text.str.contains(
    r"https?://|www\.",
    case=False,
    regex=True
)

# Exact duplicate identification
df["normalized_comment_text"] = normalized

df["duplicate_text_flag"] = (
    normalized.ne("") &
    normalized.duplicated(keep=False)
)

df["duplicate_within_video_flag"] = (
    df.duplicated(
        subset=["video_id", "normalized_comment_text"],
        keep=False
    ) &
    normalized.ne("")
)

df["duplicate_across_videos_flag"] = (
    df.groupby("normalized_comment_text")["video_id"]
    .transform("nunique").gt(1) &
    normalized.ne("") &
    df["duplicate_text_flag"]
)

# Comment length category
df["length_category"] = pd.cut(
    df["comment_word_count"],
    bins=[-1, 2, 5, 15, float("inf")],
    labels=[
        "Very Short (0-2 words)",
        "Short (3-5 words)",
        "Medium (6-15 words)",
        "Long (16+ words)"
    ]
)

# Save processed copy without deleting any records
output_path = (
    PROCESSED /
    "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_01.csv"
)

PROCESSED.mkdir(parents=True, exist_ok=True)

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("BATCH 01 — CORRECTED COMMENT QUALITY PROFILE")
print("=" * 70)

print("Total records:", len(df))
print("Unique comment IDs:", df["comment_id"].nunique())
print("Empty comments:", df["is_empty"].sum())
print("Very short comments:", df["is_very_short"].sum())
print("Comments containing URLs:", df["contains_url"].sum())
print("Duplicate-group records:", df["duplicate_text_flag"].sum())
print("Duplicates within same video:", df["duplicate_within_video_flag"].sum())
print("Duplicates across videos:", df["duplicate_across_videos_flag"].sum())

print("\nCorrected script distribution:")
print(df["script_profile"].value_counts().to_string())

print("\nScript distribution by movie:")
print(pd.crosstab(
    df["movie_title"],
    df["script_profile"]
).to_string())

print("\nLength distribution:")
print(df["length_category"].value_counts(sort=False).to_string())

print("\nSaved:", output_path)

In [ ]:
import importlib.util

packages = [
    "langdetect",
    "langid",
    "lingua",
    "fasttext",
    "transformers",
    "nltk",
    "spacy",
    "pycld3"
]

print("=" * 65)
print("LANGUAGE IDENTIFICATION — ENVIRONMENT CHECK")
print("=" * 65)

for package in packages:
    available = importlib.util.find_spec(package) is not None
    print(f"{package:<18} : {'Available' if available else 'Not installed'}")

In [ ]:
%pip install -q lingua-language-detector

In [ ]:
import lingua
from lingua import Language

print("=" * 70)
print("LINGUA — VERSION AND SUPPORTED LANGUAGE DIAGNOSTICS")
print("=" * 70)

print("Lingua module:", lingua.__file__)
print("Language class:", Language)
print("Language class type:", type(Language))

print("\nAvailable language attributes:")

available_languages = [
    name for name in dir(Language)
    if name.isupper() and not name.startswith("_")
]

for name in available_languages:
    print(name)

print("\nTotal available language attributes:", len(available_languages))

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

QUALITY_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_01.csv"
)

df_quality = pd.read_csv(QUALITY_FILE, encoding="utf-8")

print("=" * 70)
print("YOUTUBE COMMENT QUALITY PROFILE — SCHEMA CHECK")
print("=" * 70)

print("Dataset shape:", df_quality.shape)

print("\nColumns:")
for col in df_quality.columns:
    print(f"- {col}")

print("\nData types:")
print(df_quality.dtypes)

print("\nFirst 3 records:")
display(df_quality.head(3))

In [ ]:

import pandas as pd
from pathlib import Path
from lingua import Language, LanguageDetectorBuilder

# ------------------------------------------------------------
# 1. LOAD EXISTING QUALITY PROFILE
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

QUALITY_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_01.csv"
)

df_quality = pd.read_csv(QUALITY_FILE, encoding="utf-8")

# ------------------------------------------------------------
# 2. BUILD DETECTOR USING SUPPORTED LANGUAGES
# ------------------------------------------------------------

language_names = [
    name for name in dir(Language)
    if name.isupper() and not name.startswith("_")
]

supported_languages = [
    getattr(Language, name)
    for name in language_names
]

detector = (
    LanguageDetectorBuilder
    .from_languages(*supported_languages)
    .build()
)

# ------------------------------------------------------------
# 3. SELECT 20 RANDOM COMMENTS
# ------------------------------------------------------------

pilot = (
    df_quality
    .sample(n=20, random_state=42)
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. LANGUAGE IDENTIFICATION
# ------------------------------------------------------------

def identify_language(row):

    comment = str(row["comment_text"]).strip()
    script = row["script_profile"]

    result = {
        "language_candidate": None,
        "language_confidence": None,
        "second_language_candidate": None,
        "second_language_confidence": None,
        "confidence_margin": None,
        "language_status": None
    }

    # Empty or very short comments
    if len(comment) < 5:
        result["language_status"] = "Undetermined - short text"
        return pd.Series(result)

    # Emoji/symbol-only comments
    if script == "Other / Emoji / Symbols":
        result["language_status"] = "Undetermined - emoji/symbol"
        return pd.Series(result)

    # Calculate language confidence
    confidence_values = (
        detector.compute_language_confidence_values(comment)
    )

    if not confidence_values:
        result["language_status"] = "Undetermined - no prediction"
        return pd.Series(result)

    first = confidence_values[0]

    result["language_candidate"] = first.language.name
    result["language_confidence"] = round(first.value, 4)

    if len(confidence_values) > 1:
        second = confidence_values[1]

        result["second_language_candidate"] = second.language.name
        result["second_language_confidence"] = round(
            second.value, 4
        )

        result["confidence_margin"] = round(
            first.value - second.value, 4
        )
    else:
        result["confidence_margin"] = first.value

    # Provisional decision rules
    if script == "Latin":
        result["language_status"] = (
            "Manual review - Latin script ambiguity"
        )

    elif (
        first.value >= 0.70
        and result["confidence_margin"] >= 0.20
    ):
        result["language_status"] = "High-confidence candidate"

    else:
        result["language_status"] = "Manual review - low confidence"

    return pd.Series(result)


# ------------------------------------------------------------
# 5. APPLY TO PILOT SAMPLE
# ------------------------------------------------------------

language_results = pilot.apply(
    identify_language,
    axis=1
)

pilot = pd.concat(
    [pilot, language_results],
    axis=1
)

# ------------------------------------------------------------
# 6. DISPLAY PILOT RESULTS
# ------------------------------------------------------------

display_columns = [
    "movie_title",
    "source_type",
    "comment_text",
    "script_profile",
    "language_candidate",
    "language_confidence",
    "second_language_candidate",
    "confidence_margin",
    "language_status"
]

print("=" * 80)
print("LINGUA — 20 COMMENT LANGUAGE IDENTIFICATION PILOT")
print("=" * 80)

print("Pilot records:", len(pilot))
print("Supported detector languages:", len(supported_languages))

display(pilot[display_columns])

print("\nLANGUAGE STATUS DISTRIBUTION")
display(
    pilot["language_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

print("\nLANGUAGE CANDIDATE DISTRIBUTION")
display(
    pilot["language_candidate"]
    .value_counts(dropna=False)
    .rename_axis("language")
    .reset_index(name="count")
)


In [ ]:
print("=" * 70)
print("ICMB-130 — SCRIPT-WISE LANGUAGE IDENTIFICATION FEASIBILITY")
print("=" * 70)

script_summary = (
    df_quality
    .groupby("script_profile")
    .agg(
        comment_count=("comment_id", "count"),
        unique_movies=("movie_title", "nunique"),
        average_words=("comment_word_count", "mean")
    )
    .reset_index()
)

script_summary["percentage"] = (
    script_summary["comment_count"]
    / len(df_quality) * 100
).round(2)

script_summary["average_words"] = (
    script_summary["average_words"].round(2)
)

display(
    script_summary.sort_values(
        "comment_count", ascending=False
    )
)

print("\nTOTAL COMMENTS:", len(df_quality))

In [ ]:
import importlib.util
import torch

print("=" * 70)
print("MULTILINGUAL LANGUAGE IDENTIFICATION ENVIRONMENT CHECK")
print("=" * 70)

packages = [
    "transformers",
    "torch",
    "sentencepiece",
    "tokenizers",
    "huggingface_hub",
    "fasttext"
]

for package in packages:
    print(
        f"{package:20s}:",
        "AVAILABLE" if importlib.util.find_spec(package)
        else "NOT INSTALLED"
    )

print("\nPyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

REFERENCE_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_AUDIENCE_MOVIE_REFERENCE_VALIDATED.csv"
)

df_reference = pd.read_csv(REFERENCE_FILE)

BATCH_02_IDS = [
    "2005_2",
    "2005_3",
    "2005_4",
    "2005_5",
    "2006_1"
]

batch_02_movies = (
    df_reference[
        df_reference["movie_record_id"].isin(BATCH_02_IDS)
    ]
    .copy()
)

batch_02_movies["batch_order"] = (
    batch_02_movies["movie_record_id"]
    .map({movie_id: i for i, movie_id in enumerate(BATCH_02_IDS, 1)})
)

batch_02_movies = (
    batch_02_movies
    .sort_values("batch_order")
    .drop(columns="batch_order")
)

print("=" * 75)
print("ICMB-130 — YOUTUBE DISCOVERY BATCH 02 PREFLIGHT")
print("=" * 75)

print("Expected movies:", len(BATCH_02_IDS))
print("Verified movies:", len(batch_02_movies))
print("Unique movie IDs:", batch_02_movies["movie_record_id"].nunique())

display(
    batch_02_movies[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "language",
            "industry",
            "tmdb_id",
            "imdb_id"
        ]
    ]
)

assert len(batch_02_movies) == 5, (
    "Expected exactly five Batch 02 movies. Check the reference file."
)

assert batch_02_movies["movie_record_id"].tolist() == BATCH_02_IDS

print("\nBatch 02 reference verification: PASSED")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

DISCOVERY_DIR = (
    PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"
)

files_to_inspect = [
    DISCOVERY_DIR / "ICMB_130_YOUTUBE_DISCOVERY_PILOT.csv",
    DISCOVERY_DIR / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_01.csv"
]

for file_path in files_to_inspect:

    print("\n" + "=" * 75)
    print(file_path.name)
    print("=" * 75)

    if file_path.exists():
        temp_df = pd.read_csv(file_path)

        print("Shape:", temp_df.shape)
        print("Columns:", temp_df.columns.tolist())

        if "search_query" in temp_df.columns:
            print("\nSearch queries:")
            display(
                temp_df["search_query"]
                .dropna()
                .drop_duplicates()
                .to_frame()
            )
    else:
        print("FILE NOT FOUND")

In [ ]:

import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. PATHS AND CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

DISCOVERY_DIR = (
    PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"
)

OUTPUT_FILE = (
    DISCOVERY_DIR / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_02.csv"
)

DISCOVERY_DIR.mkdir(parents=True, exist_ok=True)

# Safety: never overwrite an existing Batch 02 file
if OUTPUT_FILE.exists():
    raise FileExistsError(
        f"Batch 02 discovery file already exists: {OUTPUT_FILE}\n"
        "Inspect the existing file before deciding whether to rerun."
    )

# ------------------------------------------------------------
# 2. VERIFIED BATCH 02 MOVIES
# ------------------------------------------------------------

batch_movies = [
    {
        "movie_record_id": "2005_2",
        "movie_title": "Anniyan",
        "movie_year": 2005,
        "language": "Tamil",
        "industry": "Kollywood",
        "tmdb_id": 19978,
        "imdb_id": "tt0376127",
        "search_language": "ta"
    },
    {
        "movie_record_id": "2005_3",
        "movie_title": "No Entry",
        "movie_year": 2005,
        "language": "Hindi",
        "industry": "Bollywood",
        "tmdb_id": 20390,
        "imdb_id": "tt0451833",
        "search_language": "hi"
    },
    {
        "movie_record_id": "2005_4",
        "movie_title": "Bunty Aur Babli",
        "movie_year": 2005,
        "language": "Hindi",
        "industry": "Bollywood",
        "tmdb_id": 11763,
        "imdb_id": "tt0448206",
        "search_language": "hi"
    },
    {
        "movie_record_id": "2005_5",
        "movie_title": "Garam Masala",
        "movie_year": 2005,
        "language": "Hindi",
        "industry": "Bollywood",
        "tmdb_id": 19636,
        "imdb_id": "tt0453671",
        "search_language": "hi"
    },
    {
        "movie_record_id": "2006_1",
        "movie_title": "Dhoom 2",
        "movie_year": 2006,
        "language": "Hindi",
        "industry": "Bollywood",
        "tmdb_id": 14194,
        "imdb_id": "tt0441048",
        "search_language": "hi"
    }
]

# ------------------------------------------------------------
# 3. YOUTUBE API CLIENT CHECK
# ------------------------------------------------------------

if "youtube" not in globals():
    raise RuntimeError(
        "YouTube API client not found. Run the existing API "
        "initialization cell before continuing."
    )

# ------------------------------------------------------------
# 4. SEARCH ONE TIME PER MOVIE
# ------------------------------------------------------------

discovery_records = []

for movie in batch_movies:

    query = (
        f'"{movie["movie_title"]}" '
        f'{movie["movie_year"]} '
        f'{movie["language"]} movie review reaction'
    )

    print("\n" + "=" * 75)
    print("Searching:", movie["movie_title"])
    print("Query:", query)
    print("=" * 75)

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=10,
            regionCode="IN",
            relevanceLanguage=movie["search_language"],
            order="relevance"
        ).execute()

        items = response.get("items", [])

        print("Candidates returned:", len(items))

        for item in items:

            snippet = item.get("snippet", {})
            video_id = item.get("id", {}).get("videoId")

            if not video_id:
                continue

            discovery_records.append({
                "movie_record_id": movie["movie_record_id"],
                "movie_title": movie["movie_title"],
                "movie_year": movie["movie_year"],
                "language": movie["language"],
                "tmdb_id": movie["tmdb_id"],
                "imdb_id": movie["imdb_id"],
                "search_query": query,
                "video_id": video_id,
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "discovery_source": "YouTube Data API v3",
                "discovery_batch": "Batch 02"
            })

    except Exception as error:
        print("Search failed:", type(error).__name__, str(error))
        raise

# ------------------------------------------------------------
# 5. CREATE AND VALIDATE DISCOVERY DATASET
# ------------------------------------------------------------

df_batch_02 = pd.DataFrame(discovery_records)

if df_batch_02.empty:
    raise ValueError("No YouTube candidates were returned.")

before_deduplication = len(df_batch_02)

df_batch_02 = df_batch_02.drop_duplicates(
    subset=["video_id"],
    keep="first"
).reset_index(drop=True)

print("\n" + "=" * 75)
print("BATCH 02 DISCOVERY SUMMARY")
print("=" * 75)

print("Records returned:", before_deduplication)
print("Unique video candidates:", len(df_batch_02))
print(
    "Movies represented:",
    df_batch_02["movie_record_id"].nunique()
)
print(
    "Unique video IDs:",
    df_batch_02["video_id"].nunique()
)

print("\nCandidates by movie:")
display(
    df_batch_02.groupby(
        ["movie_record_id", "movie_title"]
    ).size().reset_index(name="candidate_count")
)

# ------------------------------------------------------------
# 6. SAVE SEPARATE RAW DISCOVERY FILE
# ------------------------------------------------------------

df_batch_02.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved successfully:", OUTPUT_FILE.name)
print("Saved shape:", df_batch_02.shape)
print(
    "Unique video IDs in saved file:",
    df_batch_02["video_id"].nunique()
)


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

SCREENING_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_SCREENING_BATCH_01.csv"
)

df_screening = pd.read_csv(SCREENING_FILE)

print("=" * 75)
print("BATCH 01 — AUTOMATED SCREENING IMPLEMENTATION REVIEW")
print("=" * 75)

print("Shape:", df_screening.shape)

print("\nColumns:")
for col in df_screening.columns:
    print("-", col)

print("\nDecision distribution:")
for col in df_screening.columns:
    if any(term in col.lower() for term in ["decision", "status", "confidence", "classification"]):
        print(f"\n{col}")
        display(df_screening[col].value_counts(dropna=False))

print("\nSample records:")
display(df_screening.head(10))

In [ ]:
print("=" * 70)
print("BATCH 01 — SCREENING CATEGORY DISTRIBUTION")
print("=" * 70)

for column in [
    "screened_video_type",
    "screening_relevance",
    "identity_status"
]:
    print(f"\n{column.upper()}")
    display(
        df_screening[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="count")
    )

print("\nVIDEO TYPE × RELEVANCE")
display(
    pd.crosstab(
        df_screening["screened_video_type"],
        df_screening["screening_relevance"]
    )
)

In [ ]:

import pandas as pd
import re
from pathlib import Path

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = (
    PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

INPUT_FILE = (
    RAW_DIR / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_02.csv"
)

OUTPUT_FILE = (
    PROCESSED_DIR / "ICMB_130_YOUTUBE_SCREENING_BATCH_02.csv"
)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        "Batch 02 screening file already exists. "
        "Inspect it before rerunning."
    )

df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")

# ------------------------------------------------------------
# 2. KEYWORD GROUPS
# ------------------------------------------------------------

VIDEO_PATTERNS = {
    "Trailer": [
        r"\bofficial trailer\b", r"\btrailer\b", r"\bteaser\b"
    ],
    "Song": [
        r"\bsong\b", r"\blyrics\b", r"\bjukebox\b",
        r"\baudio jukebox\b", r"\bmusic video\b"
    ],
    "Full Movie": [
        r"\bfull movie\b", r"\bcomplete movie\b",
        r"\bfull film\b", r"\bwatch full\b",
        r"\bfull hindi movie\b", r"\bfull tamil movie\b"
    ],
    "Audience Reaction": [
        r"\breaction\b", r"\breacting\b", r"\bwatchalong\b",
        r"\bwatch along\b", r"\bfirst time watching\b"
    ],
    "Movie Review": [
        r"\breview\b", r"\broast\b", r"\brevisited\b",
        r"\bretrospective\b", r"\bopinion\b"
    ],
    "Movie Explanation": [
        r"\bexplained\b", r"\bmovie explanation\b",
        r"\bstory explained\b", r"\bending explained\b"
    ],
    "Box Office Discussion": [
        r"\bbox office\b", r"\bcollection\b",
        r"\bcollections\b", r"\bbudget\b", r"\bverdict\b"
    ],
    "Scene or Clip": [
        r"\bscene\b", r"\bclip\b", r"\bfight scene\b",
        r"\bcomedy scene\b", r"\bbest scenes\b"
    ]
}

# ------------------------------------------------------------
# 3. CLASSIFICATION FUNCTION
# ------------------------------------------------------------

def classify_candidate(row):

    title = str(row.get("video_title", ""))
    description = str(row.get("description", ""))

    movie = str(row.get("movie_title", ""))
    year = str(row.get("movie_year", ""))

    title_clean = title.lower()
    description_clean = description.lower()

    combined_text = title_clean + " " + description_clean

    result = {
        "screened_video_type": "Other / Unclear",
        "screening_relevance": "Needs Review",
        "identity_evidence": "",
        "identity_score": 0,
        "identity_status": "Possible Candidate",
        "screening_reason": ""
    }

    # --------------------------------------------------------
    # Movie identity evidence
    # --------------------------------------------------------

    movie_pattern = re.escape(movie.lower())

    title_match = bool(
        re.search(movie_pattern, title_clean)
    )

    description_match = bool(
        re.search(movie_pattern, description_clean)
    )

    year_match = bool(
        re.search(r"\b" + re.escape(year) + r"\b", combined_text)
    ) if year.isdigit() else False

    identity_score = (
        (50 if title_match else 0)
        + (25 if description_match else 0)
        + (15 if year_match else 0)
    )

    evidence = []

    if title_match:
        evidence.append("Movie title in video title")

    if description_match:
        evidence.append("Movie title in description")

    if year_match:
        evidence.append("Release year mentioned")

    result["identity_score"] = identity_score
    result["identity_evidence"] = (
        "; ".join(evidence) if evidence else "No direct metadata match"
    )

    # --------------------------------------------------------
    # Video type classification
    # --------------------------------------------------------

    detected_type = "Other / Unclear"

    for video_type, patterns in VIDEO_PATTERNS.items():
        if any(
            re.search(pattern, combined_text)
            for pattern in patterns
        ):
            detected_type = video_type
            break

    result["screened_video_type"] = detected_type

    # --------------------------------------------------------
    # Identity status
    # --------------------------------------------------------

    if title_match and (description_match or year_match):
        result["identity_status"] = (
            "Strong Metadata Match — Candidate"
        )

    elif title_match or description_match:
        result["identity_status"] = (
            "Possible Candidate — Identity Evidence Incomplete"
        )

    else:
        result["identity_status"] = (
            "Weak Metadata Match — Review Required"
        )

    # --------------------------------------------------------
    # Relevance classification
    # --------------------------------------------------------

    if detected_type in ["Movie Review", "Audience Reaction"]:
        if title_match:
            result["screening_relevance"] = (
                "High — Audience Feedback Candidate"
            )
        else:
            result["screening_relevance"] = (
                "Medium — Identity Evidence Incomplete"
            )

    elif detected_type == "Full Movie":
        result["screening_relevance"] = (
            "Separate — Full Movie Audience Comments"
        )

    elif detected_type in [
        "Movie Explanation", "Box Office Discussion"
    ]:
        result["screening_relevance"] = (
            "Medium — Contextual Discussion"
        )

    elif detected_type in [
        "Trailer", "Song", "Scene or Clip"
    ]:
        result["screening_relevance"] = (
            "Low — Exclude from Primary Review Corpus"
        )

    else:
        result["screening_relevance"] = (
            "Needs Review"
        )

    # --------------------------------------------------------
    # Screening explanation
    # --------------------------------------------------------

    result["screening_reason"] = (
        f"Type: {detected_type}; "
        f"Identity score: {identity_score}/90; "
        f"Title match: {title_match}; "
        f"Description match: {description_match}; "
        f"Year match: {year_match}"
    )

    return pd.Series(result)


# ------------------------------------------------------------
# 4. APPLY SCREENING
# ------------------------------------------------------------

screening_results = df.apply(
    classify_candidate,
    axis=1
)

df_screened = pd.concat(
    [df, screening_results],
    axis=1
)

# ------------------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------------------

assert len(df_screened) == len(df)
assert df_screened["video_id"].nunique() == len(df_screened)
assert df_screened["screened_video_type"].notna().all()

# ------------------------------------------------------------
# 6. SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("ICMB-130 — AUTOMATED YOUTUBE SCREENING BATCH 02")
print("=" * 75)

print("Total candidates:", len(df_screened))
print("Unique videos:", df_screened["video_id"].nunique())
print(
    "Movies represented:",
    df_screened["movie_record_id"].nunique()
)

print("\nVIDEO TYPE DISTRIBUTION")
display(
    df_screened["screened_video_type"]
    .value_counts()
    .rename_axis("video_type")
    .reset_index(name="count")
)

print("\nRELEVANCE DISTRIBUTION")
display(
    df_screened["screening_relevance"]
    .value_counts()
    .rename_axis("relevance")
    .reset_index(name="count")
)

print("\nIDENTITY STATUS DISTRIBUTION")
display(
    df_screened["identity_status"]
    .value_counts()
    .rename_axis("identity_status")
    .reset_index(name="count")
)

print("\nMOVIE-WISE SCREENING")
display(
    df_screened.groupby(
        ["movie_record_id", "movie_title", "screening_relevance"]
    ).size().reset_index(name="candidate_count")
)

# ------------------------------------------------------------
# 7. SAVE SCREENING OUTPUT
# ------------------------------------------------------------

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_screened.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", OUTPUT_FILE.name)
print("Output shape:", df_screened.shape)


In [ ]:
print("=" * 75)
print("BATCH 02 — TARGETED UNCERTAINTY REVIEW")
print("=" * 75)

uncertain = df_screened[
    (df_screened["screening_relevance"] == "Needs Review")
    | (df_screened["screened_video_type"] == "Other / Unclear")
].copy()

print("Candidates requiring targeted inspection:", len(uncertain))

display(
    uncertain[
        [
            "movie_title",
            "video_title",
            "channel_title",
            "published_at",
            "identity_score",
            "identity_evidence",
            "screened_video_type",
            "screening_relevance",
            "screening_reason"
        ]
    ]
)

In [ ]:
# ------------------------------------------------------------
# BATCH 02 — TARGETED CLASSIFICATION REFINEMENT
# ------------------------------------------------------------

df_refined = df_screened.copy()

# Gucci React series: retain as audience reaction candidates
gucci_mask = (
    (df_refined["movie_title"] == "No Entry")
    & (df_refined["channel_title"] == "Gucci React")
)

df_refined.loc[
    gucci_mask, "screened_video_type"
] = "Audience Reaction"

df_refined.loc[
    gucci_mask, "screening_relevance"
] = "High — Audience Reaction Candidate"

df_refined.loc[
    gucci_mask, "identity_status"
] = "Possible — Reaction Series Candidate"

df_refined.loc[
    gucci_mask, "screening_reason"
] = (
    "Channel and title indicate a multi-part reaction series; "
    "movie identity remains a candidate pending content confirmation."
)

# Film City HD: separate full-movie source
film_mask = (
    (df_refined["movie_title"] == "No Entry")
    & (df_refined["channel_title"] == "Film City HD")
)

df_refined.loc[
    film_mask, "screened_video_type"
] = "Full Movie"

df_refined.loc[
    film_mask, "screening_relevance"
] = "Separate — Full Movie Audience Comments"

df_refined.loc[
    film_mask, "identity_status"
] = "Strong Metadata Match — Full Movie Candidate"

df_refined.loc[
    film_mask, "screening_reason"
] = (
    "Title identifies No Entry (2005) and describes a Hindi movie; "
    "retained separately as a full-movie upload candidate."
)

# Feature Friday: retrospective discussion candidate
feature_mask = (
    (df_refined["movie_title"] == "Dhoom 2")
    & (df_refined["channel_title"] == "Feature Friday Movies (INDIA)")
)

df_refined.loc[
    feature_mask, "screened_video_type"
] = "Movie Discussion"

df_refined.loc[
    feature_mask, "screening_relevance"
] = "Medium — Retrospective Discussion Candidate"

df_refined.loc[
    feature_mask, "identity_status"
] = "Possible — Movie Discussion Candidate"

df_refined.loc[
    feature_mask, "screening_reason"
] = (
    "Video title explicitly references Dhoom 2; "
    "discussion/review context requires content confirmation."
)

# JHALLU BHAI: retain as uncertain
jhallubhai_mask = (
    (df_refined["movie_title"] == "Dhoom 2")
    & (df_refined["channel_title"] == "JHALLU BHAI KI CLASS")
)

df_refined.loc[
    jhallubhai_mask, "identity_status"
] = "Hold — Insufficient Evidence"

df_refined.loc[
    jhallubhai_mask, "screening_reason"
] = (
    "Title contains Dhoom 2 but does not establish whether "
    "the video is a review, reaction or unrelated reference."
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("BATCH 02 — REFINED SCREENING RESULTS")
print("=" * 75)

print("Total candidates:", len(df_refined))

print("\nVideo types:")
display(
    df_refined["screened_video_type"]
    .value_counts()
    .rename_axis("video_type")
    .reset_index(name="count")
)

print("\nRelevance:")
display(
    df_refined["screening_relevance"]
    .value_counts()
    .rename_axis("relevance")
    .reset_index(name="count")
)

print("\nRefined uncertain records:")
display(
    df_refined.loc[
        gucci_mask | film_mask | feature_mask | jhallubhai_mask,
        [
            "movie_title",
            "video_title",
            "channel_title",
            "screened_video_type",
            "screening_relevance",
            "identity_status"
        ]
    ]
)

# ------------------------------------------------------------
# SAVE REFINED OUTPUT SEPARATELY
# ------------------------------------------------------------

REFINED_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_SCREENING_BATCH_02_REFINED.csv"
)

if REFINED_FILE.exists():
    raise FileExistsError(
        "Refined screening file already exists. Inspect before rerunning."
    )

df_refined.to_csv(
    REFINED_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", REFINED_FILE.name)
print("Shape:", df_refined.shape)

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 02
# YOUTUBE VIDEO METADATA VALIDATION
# ============================================================

import os
import pandas as pd
from googleapiclient.discovery import build
from google.colab import userdata

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "audience_sentiment"
)

INPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_SCREENING_BATCH_02_REFINED.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_VIDEO_METADATA_BATCH_02.csv"
)

# ------------------------------------------------------------
# 2. LOAD REFINED SCREENING
# ------------------------------------------------------------

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Refined screening file not found: {INPUT_FILE}"
    )

df_screened = pd.read_csv(INPUT_FILE)

required_columns = [
    "video_id",
    "movie_record_id",
    "movie_title",
    "screened_video_type",
    "screening_relevance",
    "identity_status"
]

missing = [
    col for col in required_columns
    if col not in df_screened.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

video_ids = (
    df_screened["video_id"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

print("Candidates:", len(df_screened))
print("Unique video IDs:", len(video_ids))

# ------------------------------------------------------------
# 3. CONNECT TO YOUTUBE API
# ------------------------------------------------------------

API_KEY = userdata.get("YOUTUBE_API_KEY")

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY,
    cache_discovery=False
)

# ------------------------------------------------------------
# 4. FETCH VIDEO METADATA
# ------------------------------------------------------------

metadata_records = []

for start in range(0, len(video_ids), 50):

    batch_ids = video_ids[start:start + 50]

    response = youtube.videos().list(
        part="snippet,contentDetails,statistics",
        id=",".join(batch_ids)
    ).execute()

    for item in response.get("items", []):

        snippet = item.get("snippet", {})
        statistics = item.get("statistics", {})
        content = item.get("contentDetails", {})

        metadata_records.append({
            "video_id": item.get("id"),
            "video_title_actual": snippet.get("title"),
            "channel_id_actual": snippet.get("channelId"),
            "channel_title_actual": snippet.get("channelTitle"),
            "published_at_actual": snippet.get("publishedAt"),
            "category_id": snippet.get("categoryId"),
            "duration_iso8601": content.get("duration"),
            "definition": content.get("definition"),
            "view_count": statistics.get("viewCount"),
            "like_count": statistics.get("likeCount"),
            "comment_count": statistics.get("commentCount"),
            "metadata_status": "Retrieved"
        })

metadata_df = pd.DataFrame(metadata_records)

# ------------------------------------------------------------
# 5. MERGE WITH SCREENING RECORDS
# ------------------------------------------------------------

df_metadata = df_screened.merge(
    metadata_df,
    on="video_id",
    how="left",
    validate="many_to_one"
)

df_metadata["metadata_status"] = (
    df_metadata["metadata_status"]
    .fillna("Unavailable or Video Not Public")
)

# ------------------------------------------------------------
# 6. SAVE WITHOUT OVERWRITING
# ------------------------------------------------------------

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        "Output already exists. Existing file preserved:\n"
        + OUTPUT_FILE
    )

df_metadata.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. VALIDATION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BATCH 02 — VIDEO METADATA VALIDATION")
print("=" * 65)

print("Screening records:", len(df_screened))
print("Unique requested videos:", len(video_ids))
print("Metadata records retrieved:", len(metadata_df))
print(
    "Metadata unavailable:",
    len(video_ids) - metadata_df["video_id"].nunique()
)

print("\nMetadata status:")
print(df_metadata["metadata_status"].value_counts())

print("\nScreening type distribution:")
print(df_metadata["screened_video_type"].value_counts())

print("\nSaved:", os.path.basename(OUTPUT_FILE))
print("Final shape:", df_metadata.shape)

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 02
# METADATA-BASED ACQUISITION PRIORITIZATION
# ============================================================

import os
import re
import pandas as pd

INPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_VIDEO_METADATA_BATCH_02.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_METADATA_PRIORITY_BATCH_02.csv"
)

df_meta = pd.read_csv(INPUT_FILE)

# ------------------------------------------------------------
# 1. CONVERT ISO 8601 DURATION TO SECONDS
# ------------------------------------------------------------

def duration_to_seconds(value):

    if pd.isna(value):
        return None

    match = re.fullmatch(
        r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?",
        str(value)
    )

    if not match:
        return None

    hours, minutes, seconds = [
        int(x) if x else 0 for x in match.groups()
    ]

    return hours * 3600 + minutes * 60 + seconds


df_meta["duration_seconds"] = (
    df_meta["duration_iso8601"]
    .apply(duration_to_seconds)
)

df_meta["duration_minutes"] = (
    df_meta["duration_seconds"] / 60
).round(2)

# ------------------------------------------------------------
# 2. CONVERT ENGAGEMENT METRICS
# ------------------------------------------------------------

for col in ["view_count", "like_count", "comment_count"]:
    df_meta[col] = pd.to_numeric(
        df_meta[col],
        errors="coerce"
    )

# ------------------------------------------------------------
# 3. ADD METADATA-BASED REVIEW FLAGS
# ------------------------------------------------------------

def metadata_flag(row):

    video_type = str(row["screened_video_type"])
    duration = row["duration_minutes"]

    if pd.isna(duration):
        return "Duration Unavailable"

    if video_type == "Full Movie":

        if duration >= 90:
            return "Duration Consistent with Full Movie"

        return "Full Movie Label — Duration Check Required"

    if video_type == "Trailer":

        return "Trailer Candidate"

    if video_type in [
        "Audience Reaction",
        "Movie Review",
        "Movie Discussion",
        "Box Office Discussion"
    ]:

        if duration <= 90:
            return "Short-form / Standard Discussion Candidate"

        return "Long-form Discussion — Inspect Duration"

    return "Other — Inspect Metadata"


df_meta["metadata_review_flag"] = df_meta.apply(
    metadata_flag,
    axis=1
)

# ------------------------------------------------------------
# 4. IDENTIFY POSSIBLE MULTI-PART SERIES
# ------------------------------------------------------------

def extract_part_number(title):

    if pd.isna(title):
        return None

    patterns = [
        r"PART\s*(\d+)",
        r"PART\s*(\d+)\s*/",
        r"\((\d+)\s*/\s*\d+\)"
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            str(title),
            flags=re.IGNORECASE
        )

        if match:
            return int(match.group(1))

    return None


df_meta["part_number"] = (
    df_meta["video_title_actual"]
    .apply(extract_part_number)
)

# ------------------------------------------------------------
# 5. CREATE A REVIEW TABLE
# ------------------------------------------------------------

review_columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "video_title_actual",
    "channel_title_actual",
    "screened_video_type",
    "screening_relevance",
    "identity_status",
    "duration_minutes",
    "view_count",
    "like_count",
    "comment_count",
    "part_number",
    "metadata_review_flag"
]

review_df = df_meta[review_columns].copy()

review_df = review_df.sort_values(
    ["movie_title", "screened_video_type", "view_count"],
    ascending=[True, True, False]
)

# ------------------------------------------------------------
# 6. SAVE OUTPUT SAFELY
# ------------------------------------------------------------

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        f"Output already exists; not overwritten: {OUTPUT_FILE}"
    )

review_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("BATCH 02 — METADATA-BASED PRIORITIZATION")
print("=" * 70)

print("\nDuration review flags:")
print(df_meta["metadata_review_flag"].value_counts())

print("\nVideo type vs metadata review flag:")
print(pd.crosstab(
    df_meta["screened_video_type"],
    df_meta["metadata_review_flag"]
))

print("\nMulti-part candidates:")
print(
    review_df[
        review_df["part_number"].notna()
    ][
        [
            "movie_title",
            "video_title_actual",
            "channel_title_actual",
            "part_number"
        ]
    ].to_string(index=False)
)

print("\nSaved:", os.path.basename(OUTPUT_FILE))
print("Output shape:", review_df.shape)

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 02
# FINAL SOURCE SHORTLIST
# ============================================================

import os
import pandas as pd
import re

INPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_METADATA_PRIORITY_BATCH_02.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_02.csv"
)

df_shortlist = pd.read_csv(INPUT_FILE)

# ------------------------------------------------------------
# 1. ASSIGN ACQUISITION CATEGORY
# ------------------------------------------------------------

def assign_category(row):

    video_type = str(row["screened_video_type"])
    relevance = str(row["screening_relevance"])
    flag = str(row["metadata_review_flag"])

    if video_type in ["Audience Reaction", "Movie Review"]:
        return "Primary — Review / Reaction"

    if video_type == "Full Movie":
        return "Separate — Full Movie Comments"

    if video_type in ["Movie Discussion", "Box Office Discussion"]:
        return "Secondary — Contextual Discussion"

    if video_type == "Trailer":
        return "Exclude — Trailer"

    if relevance.startswith("Low"):
        return "Exclude — Low Relevance"

    return "Hold — Manual Review"


df_shortlist["acquisition_category"] = df_shortlist.apply(
    assign_category,
    axis=1
)

# ------------------------------------------------------------
# 2. FLAG MULTI-PART SERIES
# ------------------------------------------------------------

def get_series_flag(row):

    title = str(row["video_title_actual"])
    channel = str(row["channel_title_actual"])

    if pd.notna(row["part_number"]):
        return "Multi-part candidate — consolidate by channel"

    return "No part number detected"


df_shortlist["series_review"] = df_shortlist.apply(
    get_series_flag,
    axis=1
)

# ------------------------------------------------------------
# 3. ASSIGN ACQUISITION PRIORITY
# ------------------------------------------------------------

def assign_priority(row):

    category = row["acquisition_category"]

    if category == "Primary — Review / Reaction":
        return "High"

    if category == "Separate — Full Movie Comments":
        return "Separate"

    if category == "Secondary — Contextual Discussion":
        return "Medium"

    if category == "Hold — Manual Review":
        return "Review"

    return "Exclude"


df_shortlist["acquisition_priority"] = df_shortlist.apply(
    assign_priority,
    axis=1
)

# ------------------------------------------------------------
# 4. SAVE SAFELY
# ------------------------------------------------------------

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        f"Existing file preserved: {OUTPUT_FILE}"
    )

df_shortlist.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 5. SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("BATCH 02 — SOURCE SHORTLIST")
print("=" * 70)

print("\nAcquisition category:")
print(
    df_shortlist["acquisition_category"]
    .value_counts()
    .to_string()
)

print("\nPriority distribution:")
print(
    df_shortlist["acquisition_priority"]
    .value_counts()
    .to_string()
)

print("\nMovie-wise primary candidates:")
print(
    df_shortlist[
        df_shortlist["acquisition_category"]
        == "Primary — Review / Reaction"
    ]
    .groupby("movie_title")
    .size()
    .to_string()
)

print("\nMulti-part candidates:")
print(
    df_shortlist["series_review"]
    .value_counts()
    .to_string()
)

print("\nSaved:", os.path.basename(OUTPUT_FILE))
print("Shape:", df_shortlist.shape)

In [ ]:
# ============================================================
# BATCH 02 — REACTION SERIES CONSOLIDATION REVIEW
# DISPLAY ONLY — NO FILES MODIFIED
# ============================================================

import os
import pandas as pd

INPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_02.csv"
)

df_review = pd.read_csv(INPUT_FILE)

primary = df_review[
    df_review["acquisition_category"]
    == "Primary — Review / Reaction"
].copy()

display_columns = [
    "movie_title",
    "video_id",
    "video_title_actual",
    "channel_title_actual",
    "screened_video_type",
    "screening_relevance",
    "duration_minutes",
    "view_count",
    "comment_count",
    "part_number"
]

print("=" * 75)
print("BATCH 02 — PRIMARY SOURCE REVIEW BY MOVIE AND CHANNEL")
print("=" * 75)

for movie, group in primary.groupby("movie_title", sort=True):

    print(f"\n{'=' * 65}")
    print(f"MOVIE: {movie}")
    print(f"Primary candidates: {len(group)}")
    print("=" * 65)

    for channel, channel_group in group.groupby(
        "channel_title_actual",
        dropna=False,
        sort=True
    ):

        print(f"\nCHANNEL: {channel}")
        display(
            channel_group[display_columns]
            .sort_values(
                ["part_number", "view_count"],
                ascending=[True, False],
                na_position="last"
            )
            .reset_index(drop=True)
        )

In [ ]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# BATCH 02 — PROPOSED YOUTUBE SOURCE MANIFEST
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

SOURCE_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_02.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment"
)

PROPOSED_FILE = (
    OUTPUT_DIR
    / "ICMB_130_YOUTUBE_PROPOSED_SOURCES_BATCH_02.csv"
)

# Exact candidate selections
selected_sources = {
    "1hwFSpGpiVw": {
        "selection_rationale": (
            "Anniyan general movie reaction; Part 1 of a 4-part series."
        ),
        "source_group": "CineDesi",
        "series_part": "Part 1/4"
    },
    "Asz7_nYTpVI": {
        "selection_rationale": (
            "Explicitly identifies the original 2005 Bunty Aur Babli movie."
        ),
        "source_group": "Pratik's Film Theories",
        "series_part": "Standalone review"
    },
    "NYufKMVec-Q": {
        "selection_rationale": (
            "Dhoom 2 general movie reaction; Part 1 of a 3-part series."
        ),
        "source_group": "ChaatNChat",
        "series_part": "Part 1/3"
    },
    "aczizi0yX0M": {
        "selection_rationale": (
            "Garam Masala general movie reaction; Part 1 of the "
            "selected CineDesi 3-part series."
        ),
        "source_group": "CineDesi",
        "series_part": "Part 1/3"
    },
    "d3MJpWR_nfE": {
        "selection_rationale": (
            "No Entry general movie reaction; Part 3 of a 3-part series."
        ),
        "source_group": "CineDesi",
        "series_part": "Part 3/3"
    }
}

# Load original shortlist
shortlist = pd.read_csv(SOURCE_FILE)

if "video_id" not in shortlist.columns:
    raise ValueError(
        "Required column 'video_id' is missing. "
        f"Available columns: {shortlist.columns.tolist()}"
    )

# Verify all five selected IDs exist
available_ids = set(shortlist["video_id"].astype(str))
missing_ids = set(selected_sources) - available_ids

if missing_ids:
    raise ValueError(
        f"Selected video IDs not found in shortlist: {missing_ids}"
    )

# Preserve every original candidate and its existing columns
manifest = shortlist.copy()
manifest["video_id"] = manifest["video_id"].astype(str)

manifest["selection_status"] = manifest["video_id"].apply(
    lambda x: "Proposed — Pending Approval"
    if x in selected_sources else "Not Selected"
)

manifest["selection_rationale"] = manifest["video_id"].apply(
    lambda x: selected_sources[x]["selection_rationale"]
    if x in selected_sources else ""
)

manifest["source_group"] = manifest["video_id"].apply(
    lambda x: selected_sources[x]["source_group"]
    if x in selected_sources else ""
)

manifest["series_part"] = manifest["video_id"].apply(
    lambda x: selected_sources[x]["series_part"]
    if x in selected_sources else ""
)

manifest["identity_verification_status"] = manifest["video_id"].apply(
    lambda x: "Metadata/title match — not independently verified"
    if x in selected_sources else "Not selected"
)

# Save as a NEW file; original shortlist remains untouched
if PROPOSED_FILE.exists():
    raise FileExistsError(
        f"Output already exists. No file overwritten: {PROPOSED_FILE}"
    )

manifest.to_csv(PROPOSED_FILE, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# VERIFICATION
# ------------------------------------------------------------

print("=" * 65)
print("BATCH 02 — PROPOSED SOURCE MANIFEST")
print("=" * 65)

print("Original shortlist records:", len(shortlist))
print("Manifest records:", len(manifest))
print(
    "Proposed selections:",
    (manifest["selection_status"] == "Proposed — Pending Approval").sum()
)
print(
    "Unselected candidates:",
    (manifest["selection_status"] == "Not Selected").sum()
)
print("Unique video IDs:", manifest["video_id"].nunique())
print("Saved:", PROPOSED_FILE.exists())

print("\nPROPOSED SOURCES")
display(
    manifest.loc[
        manifest["selection_status"] == "Proposed — Pending Approval",
        [
            "movie_title",
            "video_id",
            "source_group",
            "series_part",
            "selection_rationale",
            "identity_verification_status"
        ]
    ].reset_index(drop=True)
)

print("\nFinal shape:", manifest.shape)

In [ ]:
# ------------------------------------------------------------
# BATCH 02 — CORRECTED METADATA VERIFICATION DISPLAY
# ------------------------------------------------------------

display_columns = [
    "movie_title_selection",
    "video_id",
    "source_group",
    "series_part",
    "video_title_actual",
    "channel_title_actual",
    "published_at_actual",
    "duration_iso8601",
    "view_count",
    "like_count",
    "comment_count",
    "metadata_status",
    "_merge"
]

# Display only columns that actually exist
available_columns = [
    col for col in display_columns
    if col in verification.columns
]

print("=" * 75)
print("BATCH 02 — FIVE PROPOSED SOURCES: METADATA REVIEW")
print("=" * 75)

display(
    verification[available_columns].reset_index(drop=True)
)

In [ ]:
# Display actual video titles and engagement metrics

review_columns = [
    "movie_title_selection",
    "video_id",
    "video_title_actual",
    "channel_title_actual",
    "view_count",
    "like_count",
    "comment_count"
]

display(
    verification[
        [c for c in review_columns if c in verification.columns]
    ].reset_index(drop=True)
)

In [ ]:
print("VERIFICATION DATAFRAME COLUMNS:")
for col in verification.columns:
    print(col)

print("\nMETADATA DATAFRAME COLUMNS:")
for col in metadata.columns:
    print(col)

print("\nVERIFICATION SHAPE:", verification.shape)

In [ ]:
# ------------------------------------------------------------
# BATCH 02 — FINAL PROPOSED SOURCE METADATA REVIEW
# ------------------------------------------------------------

review_columns = [
    "movie_title_selection",
    "video_id",
    "video_title_actual_selection",
    "channel_title_actual_selection",
    "duration_minutes",
    "view_count_selection",
    "like_count_selection",
    "comment_count_selection",
    "published_at_actual",
    "metadata_status"
]

review_table = verification[review_columns].copy()

print("=" * 80)
print("BATCH 02 — FINAL PROPOSED SOURCE REVIEW")
print("=" * 80)

print("Total proposed sources:", len(review_table))
print("Unique video IDs:", review_table["video_id"].nunique())

display(review_table.reset_index(drop=True))

In [ ]:
# ------------------------------------------------------------
# BATCH 02 — VERIFY SAVED APPROVED ACQUISITION MANIFEST
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

APPROVED_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_02.csv"
)

approved_manifest = pd.read_csv(APPROVED_FILE)

approved = approved_manifest[
    approved_manifest["acquisition_approval_status"]
    == "Approved for Comment Acquisition"
].copy()

# Identify the correct movie title column
title_col = (
    "movie_title_selection"
    if "movie_title_selection" in approved.columns
    else "movie_title"
)

display_columns = [
    title_col,
    "video_id",
    "source_group",
    "series_part",
    "acquisition_approval_status",
    "approval_rationale",
    "identity_verification_status"
]

print("=" * 70)
print("BATCH 02 — SAVED APPROVED SOURCES VERIFICATION")
print("=" * 70)

print("Total records:", len(approved_manifest))
print("Approved sources:", len(approved))
print("Unique video IDs:", approved_manifest["video_id"].nunique())
print("Approved movies:", approved[title_col].nunique())

display(approved[display_columns].reset_index(drop=True))

In [ ]:
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# BATCH 02 — COMMENT ACQUISITION PREFLIGHT
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment"
)

APPROVED_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_02.csv"
)

BATCH01_RAW = (
    RAW_DIR
    / "ICMB_130_YOUTUBE_COMMENT_BATCH_01.csv"
)

BATCH01_LOG = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_01.csv"
)

BATCH02_RAW = (
    RAW_DIR
    / "ICMB_130_YOUTUBE_COMMENT_BATCH_02.csv"
)

BATCH02_LOG = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_02.csv"
)

print("=" * 70)
print("BATCH 02 — COMMENT ACQUISITION PREFLIGHT")
print("=" * 70)

# Check approved manifest
assert APPROVED_FILE.exists(), "Approved manifest is missing."

approved_df = pd.read_csv(APPROVED_FILE)

approved_df = approved_df[
    approved_df["acquisition_approval_status"]
    == "Approved for Comment Acquisition"
].copy()

print("\nAPPROVED SOURCES")
print("Approved videos:", len(approved_df))
print("Unique video IDs:", approved_df["video_id"].nunique())
display(
    approved_df[["movie_title", "video_id", "source_group", "series_part"]]
    .reset_index(drop=True)
)

# Check Batch 01 reference files
print("\nBATCH 01 REFERENCE FILES")

for file_path in [BATCH01_RAW, BATCH01_LOG]:
    print(file_path.name, "Exists:", file_path.exists())

    if file_path.exists():
        df = pd.read_csv(file_path, nrows=5)
        print("Columns:", df.columns.tolist())

# Check Batch 02 output paths
print("\nBATCH 02 OUTPUT PATHS")
print("Raw output already exists:", BATCH02_RAW.exists())
print("Log output already exists:", BATCH02_LOG.exists())

# Check API client without exposing credentials
print("\nYOUTUBE API CLIENT")
print("Variable 'youtube' available:", "youtube" in globals())

print("\nPREFLIGHT COMPLETE")

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display

# ------------------------------------------------------------
# BATCH 02 — YOUTUBE COMMENT ACQUISITION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment"
)

APPROVED_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_02.csv"
)

RAW_OUTPUT = (
    RAW_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_02.csv"
)

LOG_OUTPUT = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_02.csv"
)

COMMENT_LIMIT = 100

# ------------------------------------------------------------
# SAFETY CHECKS
# ------------------------------------------------------------

if RAW_OUTPUT.exists() or LOG_OUTPUT.exists():
    raise FileExistsError(
        "A Batch 02 output already exists. "
        "No existing file will be overwritten."
    )

if "youtube" not in globals():
    raise NameError("YouTube API client 'youtube' is unavailable.")

approved_df = pd.read_csv(APPROVED_FILE)

approved_df = approved_df[
    approved_df["acquisition_approval_status"]
    == "Approved for Comment Acquisition"
].copy()

if len(approved_df) != 5:
    raise ValueError(
        f"Expected 5 approved sources, found {len(approved_df)}."
    )

if approved_df["video_id"].nunique() != 5:
    raise ValueError("Duplicate or missing approved video IDs detected.")

# Resolve available metadata columns safely
def get_value(row, possible_columns, default=None):
    for column in possible_columns:
        if column in row.index and pd.notna(row[column]):
            return row[column]
    return default

comment_records = []
acquisition_logs = []

print("=" * 70)
print("BATCH 02 — YOUTUBE COMMENT ACQUISITION")
print("=" * 70)

# ------------------------------------------------------------
# ACQUIRE UP TO 100 TOP-LEVEL COMMENTS PER VIDEO
# ------------------------------------------------------------

for _, source in approved_df.iterrows():

    video_id = str(source["video_id"])
    movie_title = get_value(
        source, ["movie_title", "movie_title_selection"], "Unknown"
    )
    movie_record_id = get_value(
        source, ["movie_record_id", "movie_record_id_selection"], ""
    )
    movie_year = get_value(
        source, ["movie_year", "year"], ""
    )
    source_type = get_value(
        source,
        ["screened_video_type", "screened_video_type_selection",
         "acquisition_category"],
        "Review / Reaction"
    )
    reported_comment_count = get_value(
        source,
        ["comment_count", "comment_count_selection"],
        None
    )

    retrieved_at = datetime.now(timezone.utc).isoformat()
    video_comments = []
    error_message = ""

    print(f"\nMovie: {movie_title}")
    print(f"Video ID: {video_id}")

    try:
        next_page_token = None

        while len(video_comments) < COMMENT_LIMIT:

            remaining = COMMENT_LIMIT - len(video_comments)

            request_args = {
                "part": "snippet",
                "videoId": video_id,
                "maxResults": min(100, remaining),
                "order": "time",
                "textFormat": "plainText"
            }

            if next_page_token:
                request_args["pageToken"] = next_page_token

            response = (
                youtube.commentThreads()
                .list(**request_args)
                .execute()
            )

            items = response.get("items", [])

            for item in items:

                thread_snippet = item.get("snippet", {})
                top_comment = thread_snippet.get(
                    "topLevelComment", {}
                )
                snippet = top_comment.get("snippet", {})

                video_comments.append({
                    "movie_record_id": movie_record_id,
                    "movie_title": movie_title,
                    "movie_year": movie_year,
                    "video_id": video_id,
                    "video_title": get_value(
                        source,
                        ["video_title_actual", "video_title",
                         "video_title_actual_selection"],
                        ""
                    ),
                    "source_type": source_type,
                    "comment_id": top_comment.get("id", ""),
                    "comment_text": snippet.get("textDisplay", ""),
                    "author_channel_id": snippet.get(
                        "authorChannelId", {}
                    ).get("value", ""),
                    "published_at": snippet.get("publishedAt", ""),
                    "updated_at": snippet.get("updatedAt", ""),
                    "like_count": snippet.get("likeCount", 0),
                    "comment_reply_count": thread_snippet.get(
                        "totalReplyCount", 0
                    ),
                    "retrieval_order": len(video_comments) + 1,
                    "retrieved_at": retrieved_at
                })

                if len(video_comments) >= COMMENT_LIMIT:
                    break

            next_page_token = response.get("nextPageToken")

            if not next_page_token or not items:
                break

        acquisition_status = (
            "Success" if video_comments else "No comments available"
        )

    except Exception as e:
        error_message = str(e)
        acquisition_status = "API Error"

    comment_records.extend(video_comments)

    acquisition_logs.append({
        "movie_title": movie_title,
        "video_id": video_id,
        "source_type": source_type,
        "reported_comment_count": reported_comment_count,
        "proposed_comment_limit": COMMENT_LIMIT,
        "comments_retrieved": len(video_comments),
        "acquisition_status": acquisition_status,
        "error_message": error_message,
        "retrieved_at": retrieved_at
    })

    print("Comments retrieved:", len(video_comments))
    print("Status:", acquisition_status)

# ------------------------------------------------------------
# CREATE DATAFRAMES USING BATCH 01 SCHEMAS
# ------------------------------------------------------------

comment_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "video_id",
    "video_title",
    "source_type",
    "comment_id",
    "comment_text",
    "author_channel_id",
    "published_at",
    "updated_at",
    "like_count",
    "comment_reply_count",
    "retrieval_order",
    "retrieved_at"
]

log_columns = [
    "movie_title",
    "video_id",
    "source_type",
    "reported_comment_count",
    "proposed_comment_limit",
    "comments_retrieved",
    "acquisition_status",
    "error_message",
    "retrieved_at"
]

comments_df = pd.DataFrame(
    comment_records, columns=comment_columns
)

acquisition_log_df = pd.DataFrame(
    acquisition_logs, columns=log_columns
)

# ------------------------------------------------------------
# VALIDATE AND SAVE — NEW FILES ONLY
# ------------------------------------------------------------

if RAW_OUTPUT.exists() or LOG_OUTPUT.exists():
    raise FileExistsError(
        "An output appeared during acquisition. "
        "Saving cancelled to prevent overwriting."
    )

comments_df.to_csv(
    RAW_OUTPUT, index=False, encoding="utf-8-sig"
)

acquisition_log_df.to_csv(
    LOG_OUTPUT, index=False, encoding="utf-8-sig"
)

# ------------------------------------------------------------
# FINAL ACQUISITION REPORT
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BATCH 02 — ACQUISITION SUMMARY")
print("=" * 70)

print("Approved videos:", len(approved_df))
print("Successful video requests:",
      (acquisition_log_df["acquisition_status"] == "Success").sum())
print("Total comments retrieved:", len(comments_df))
print("Unique comment IDs:", comments_df["comment_id"].nunique())
print("Duplicate comment IDs:",
      comments_df["comment_id"].duplicated().sum())
print("Empty comments:",
      comments_df["comment_text"].fillna("").str.strip().eq("").sum())

print("\nCOMMENTS BY MOVIE:")
if not comments_df.empty:
    display(
        comments_df.groupby("movie_title")
        .agg(
            comments_retrieved=("comment_id", "count"),
            unique_comments=("comment_id", "nunique")
        )
        .reset_index()
    )

print("\nACQUISITION LOG:")
display(acquisition_log_df)

print("\nRAW COMMENTS SAVED:", RAW_OUTPUT.exists())
print("ACQUISITION LOG SAVED:", LOG_OUTPUT.exists())
print("Raw shape:", comments_df.shape)
print("Log shape:", acquisition_log_df.shape)

In [ ]:
import pandas as pd
import re
import unicodedata
from pathlib import Path
from collections import Counter

# ------------------------------------------------------------
# BATCH 02 — COMMENT QUALITY PROFILING
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_FILE = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments/"
    / "ICMB_130_YOUTUBE_COMMENT_BATCH_02.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_02.csv"
)

if not RAW_FILE.exists():
    raise FileNotFoundError(RAW_FILE)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        "Quality profile already exists. No overwrite performed."
    )

df = pd.read_csv(RAW_FILE, encoding="utf-8-sig")

required = [
    "movie_title", "video_id", "comment_id", "comment_text"
]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ------------------------------------------------------------
# TEXT NORMALIZATION
# ------------------------------------------------------------

def normalize_text(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = text.lower().strip()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["comment_text"] = df["comment_text"].fillna("").astype(str)

df["comment_char_count"] = df["comment_text"].str.len()
df["comment_word_count"] = df["comment_text"].str.split().str.len()

df["normalized_comment_text"] = df["comment_text"].apply(
    normalize_text
)

df["is_empty"] = df["comment_text"].str.strip().eq("")
df["is_very_short"] = df["comment_word_count"] <= 2
df["contains_url"] = df["comment_text"].str.contains(
    r"https?://|www\.", case=False, regex=True, na=False
)

# ------------------------------------------------------------
# SCRIPT PROFILING
# ------------------------------------------------------------

def detect_script(text):
    if not text or not text.strip():
        return "Empty"

    scripts = set()

    for char in text:
        code = ord(char)

        if not char.isalpha():
            continue

        if 0x0000 <= code <= 0x024F:
            scripts.add("Latin")
        elif 0x0900 <= code <= 0x097F:
            scripts.add("Devanagari")
        elif 0x0980 <= code <= 0x09FF:
            scripts.add("Bengali")
        elif 0x0B80 <= code <= 0x0BFF:
            scripts.add("Tamil")
        elif 0x0C00 <= code <= 0x0C7F:
            scripts.add("Telugu")
        elif 0x0C80 <= code <= 0x0CFF:
            scripts.add("Kannada")
        elif 0x0A80 <= code <= 0x0AFF:
            scripts.add("Gujarati")
        elif 0x0A00 <= code <= 0x0A7F:
            scripts.add("Gurmukhi")
        elif 0x0600 <= code <= 0x06FF:
            scripts.add("Arabic")
        else:
            scripts.add("Other")

    if not scripts:
        return "Other / Emoji / Symbols"

    if len(scripts) > 1:
        return "Mixed Script"

    return list(scripts)[0]

df["script_profile"] = df["comment_text"].apply(detect_script)

# ------------------------------------------------------------
# DUPLICATE FLAGS
# ------------------------------------------------------------

valid_text = df["normalized_comment_text"].ne("")

df["duplicate_text_flag"] = (
    valid_text
    & df["normalized_comment_text"].duplicated(keep=False)
)

df["duplicate_within_video_flag"] = (
    valid_text
    & df.duplicated(
        subset=["video_id", "normalized_comment_text"],
        keep=False
    )
)

df["duplicate_across_videos_flag"] = (
    valid_text
    & df.duplicated(
        subset=["normalized_comment_text", "video_id"],
        keep=False
    )
    & df["normalized_comment_text"].isin(
        df.loc[
            valid_text,
            "normalized_comment_text"
        ].value_counts().loc[lambda x: x > 1].index
    )
    & ~df["duplicate_within_video_flag"]
)

df["length_category"] = pd.cut(
    df["comment_word_count"],
    bins=[-1, 2, 5, 20, float("inf")],
    labels=["Very Short", "Short", "Medium", "Long"]
).astype(str)

# ------------------------------------------------------------
# SAVE PROFILE
# ------------------------------------------------------------

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("BATCH 02 — COMMENT QUALITY PROFILE")
print("=" * 70)

print("Total comments:", len(df))
print("Unique comment IDs:", df["comment_id"].nunique())
print("Empty comments:", df["is_empty"].sum())
print("Very short comments:", df["is_very_short"].sum())
print("Comments containing URLs:", df["contains_url"].sum())

print("\nDUPLICATE CHECKS")
print("Duplicate text records:", df["duplicate_text_flag"].sum())
print("Within-video duplicate records:",
      df["duplicate_within_video_flag"].sum())
print("Across-video duplicate records:",
      df["duplicate_across_videos_flag"].sum())

print("\nSCRIPT DISTRIBUTION")
display(
    df["script_profile"].value_counts()
    .rename_axis("script_profile")
    .reset_index(name="comments")
)

print("\nLENGTH DISTRIBUTION")
display(
    df["length_category"].value_counts()
    .rename_axis("length_category")
    .reset_index(name="comments")
)

print("\nMOVIE-WISE PROFILE")
display(
    df.groupby("movie_title").agg(
        comments=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        very_short=("is_very_short", "sum"),
        urls=("contains_url", "sum")
    ).reset_index()
)

print("\nProfile saved:", OUTPUT_FILE.exists())
print("Profile shape:", df.shape)

In [ ]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# BATCH 03 — NEXT MOVIE ALLOCATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

REFERENCE_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_AUDIENCE_MOVIE_REFERENCE_VALIDATED.csv"
)

COMMENTS_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

reference = pd.read_csv(REFERENCE_FILE)

# Collect movie IDs already represented in raw comment files
acquired_ids = set()
file_report = []

for file_path in COMMENTS_DIR.glob("*.csv"):
    if "COMMENT" not in file_path.name.upper():
        continue

    try:
        existing = pd.read_csv(file_path, usecols=lambda c: c == "movie_record_id")

        if "movie_record_id" in existing.columns:
            ids = set(existing["movie_record_id"].dropna().astype(str))
            acquired_ids.update(ids)
            file_report.append((file_path.name, len(ids)))

    except Exception as e:
        print("Could not inspect:", file_path.name, "|", str(e))

reference["movie_record_id"] = (
    reference["movie_record_id"].astype(str)
)

remaining = reference[
    ~reference["movie_record_id"].isin(acquired_ids)
].copy()

remaining = remaining.sort_values(
    ["year", "rank"]
)

batch03 = remaining.head(5).copy()

print("=" * 70)
print("BATCH 03 — MOVIE ALLOCATION")
print("=" * 70)

print("Reference movies:", len(reference))
print("Previously acquired movie IDs:", len(acquired_ids))
print("Remaining movies:", len(remaining))
print("Batch 03 allocation:", len(batch03))

print("\nPREVIOUS COMMENT FILES INSPECTED:")
for filename, count in file_report:
    print(filename, "—", count, "movie IDs")

print("\nNEXT FIVE MOVIES:")
display(
    batch03[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "rank",
            "language",
            "industry",
            "tmdb_id",
            "imdb_id"
        ]
    ].reset_index(drop=True)
)

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------
# BATCH 03 — YOUTUBE VIDEO DISCOVERY
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

DATA_DIR = (
    PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"
)

REFERENCE_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_AUDIENCE_MOVIE_REFERENCE_VALIDATED.csv"
)

OUTPUT_FILE = (
    DATA_DIR / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_03.csv"
)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        "Batch 03 discovery file already exists. No overwrite performed."
    )

if "youtube" not in globals():
    raise NameError("YouTube API client is unavailable.")

# Load reference and select the five allocated movies
reference = pd.read_csv(REFERENCE_FILE)
reference["movie_record_id"] = reference["movie_record_id"].astype(str)

batch03_ids = ["2000_3", "2000_4", "2000_5", "2001_4", "2001_5"]

batch03_movies = reference[
    reference["movie_record_id"].isin(batch03_ids)
].copy()

if len(batch03_movies) != 5:
    raise ValueError(
        f"Expected five allocated movies; found {len(batch03_movies)}."
    )

# Keep the intended order
batch03_movies["allocation_order"] = (
    batch03_movies["movie_record_id"].map(
        {movie_id: i for i, movie_id in enumerate(batch03_ids)}
    )
)
batch03_movies = batch03_movies.sort_values("allocation_order")

discovery_records = []
discovery_errors = []

print("=" * 70)
print("BATCH 03 — YOUTUBE DISCOVERY")
print("=" * 70)

for _, movie in batch03_movies.iterrows():

    movie_id = str(movie["movie_record_id"])
    title = str(movie["movie_title"])
    year = int(movie["year"])
    language = str(movie["language"])

    # Special query for the previously ambiguous title
    if movie_id == "2001_4":
        query = (
            '"Indian" 2001 Hindi film Sunny Deol '
            "movie review reaction"
        )
    else:
        query = (
            f'"{title}" {year} {language} '
            "movie review reaction"
        )

    print(f"\nSearching: {title} ({year})")
    print("Query:", query)

    try:
        response = (
            youtube.search()
            .list(
                part="snippet",
                q=query,
                type="video",
                maxResults=10,
                regionCode="IN",
                relevanceLanguage="hi"
            )
            .execute()
        )

        items = response.get("items", [])

        for item in items:
            snippet = item.get("snippet", {})

            discovery_records.append({
                "movie_record_id": movie_id,
                "movie_title": title,
                "movie_year": year,
                "industry": movie["industry"],
                "search_query": query,
                "video_id": item["id"].get("videoId", ""),
                "video_title": snippet.get("title", ""),
                "channel_id": snippet.get("channelId", ""),
                "channel_title": snippet.get("channelTitle", ""),
                "published_at": snippet.get("publishedAt", ""),
                "description": snippet.get("description", ""),
                "discovery_source": "YouTube Data API v3"
            })

        print("Candidates retrieved:", len(items))

    except Exception as e:
        discovery_errors.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "error": str(e)
        })
        print("API ERROR:", str(e))

# ------------------------------------------------------------
# BUILD AND SAVE DISCOVERY DATA
# ------------------------------------------------------------

columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "industry",
    "search_query",
    "video_id",
    "video_title",
    "channel_id",
    "channel_title",
    "published_at",
    "description",
    "discovery_source"
]

discovery_df = pd.DataFrame(
    discovery_records,
    columns=columns
)

# Check duplicate video IDs within this discovery batch
discovery_df["video_id"] = discovery_df["video_id"].astype(str)

if not discovery_df.empty:
    discovery_df = discovery_df.drop_duplicates(
        subset=["video_id"],
        keep="first"
    )

discovery_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BATCH 03 — DISCOVERY SUMMARY")
print("=" * 70)

print("Allocated movies:", len(batch03_movies))
print("Candidates retrieved:", len(discovery_df))
print("Unique video IDs:", discovery_df["video_id"].nunique())
print("Movies represented:", discovery_df["movie_record_id"].nunique())
print("API errors:", len(discovery_errors))

print("\nCANDIDATES BY MOVIE:")
if not discovery_df.empty:
    display(
        discovery_df.groupby(
            ["movie_record_id", "movie_title"]
        ).agg(
            candidates=("video_id", "count"),
            unique_videos=("video_id", "nunique")
        ).reset_index()
    )

print("\nINDIAN TITLE — IDENTITY REVIEW:")
display(
    discovery_df[
        discovery_df["movie_record_id"] == "2001_4"
    ][["video_id", "video_title", "channel_title"]]
)

if discovery_errors:
    print("\nAPI ERROR DETAILS:")
    display(pd.DataFrame(discovery_errors))

print("\nSaved:", OUTPUT_FILE.exists())
print("Output shape:", discovery_df.shape)

In [ ]:
import pandas as pd
import re
from pathlib import Path

# ------------------------------------------------------------
# BATCH 03 — AUTOMATED YOUTUBE CANDIDATE SCREENING
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = (
    PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

INPUT_FILE = RAW_DIR / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_03.csv"

OUTPUT_FILE = (
    PROCESSED_DIR / "ICMB_130_YOUTUBE_SCREENING_BATCH_03.csv"
)

if OUTPUT_FILE.exists():
    raise FileExistsError("Screening output already exists.")

df = pd.read_csv(INPUT_FILE).fillna("")

# ------------------------------------------------------------
# SCREENING RULES
# ------------------------------------------------------------

def screen_candidate(row):

    title = str(row["video_title"]).lower()
    description = str(row["description"]).lower()
    channel = str(row["channel_title"]).lower()
    movie = str(row["movie_title"]).lower()
    record_id = str(row["movie_record_id"])

    text = f"{title} {description} {channel}"

    # Identity evidence
    if record_id == "2001_4":
        identity_match = (
            "indian" in title
            and (
                "sunny deol" in text
                or "sunny deol" in movie
                or "2001" in title
            )
        )
        identity_evidence = (
            "Indian title with Sunny Deol / 2001 evidence"
            if identity_match else "Ambiguous Indian title"
        )
        identity_score = 90 if identity_match else 20

    else:
        title_terms = [
            term for term in re.findall(r"[a-z0-9]+", movie)
            if len(term) > 2
        ]
        matched_terms = [
            term for term in title_terms if term in title
        ]

        identity_match = bool(matched_terms)
        identity_score = (
            min(95, 65 + 15 * len(matched_terms))
            if identity_match else 15
        )
        identity_evidence = (
            "Movie title terms matched"
            if identity_match else "Movie title not clearly matched"
        )

    if identity_score >= 80:
        identity_status = "Strong Metadata Match — Candidate"
    elif identity_score >= 50:
        identity_status = "Possible Candidate — Identity Evidence Incomplete"
    else:
        identity_status = "Manual Identity Verification Required"

    # Video type
    if re.search(r"\b(trailer|teaser)\b", text):
        video_type = "Trailer"
        relevance = "Low — Trailer Feedback"
    elif re.search(r"\b(song|lyrical|jukebox|music video)\b", title):
        video_type = "Song"
        relevance = "Low — Scene/Song Context"
    elif re.search(r"\b(full movie|complete movie|entire movie)\b", title):
        video_type = "Full Movie"
        relevance = "Separate — Full Movie Audience Comments"
    elif re.search(r"\b(reaction|reacts|reacting)\b", text):
        video_type = "Audience Reaction"
        relevance = "High — Audience Feedback Candidate"
    elif re.search(r"\b(movie review|film review|review)\b", title):
        video_type = "Movie Review"
        relevance = "High — Audience Feedback Candidate"
    elif re.search(r"\b(scene|fight|climax|entry|dialogue|clip)\b", title):
        video_type = "Scene/Clip"
        relevance = "Low — Scene/Song Context"
    elif re.search(r"\b(explained|ending explained|analysis|discussion|facts|unknown facts)\b", text):
        video_type = "Movie Discussion"
        relevance = "Medium — Contextual Discussion"
    else:
        video_type = "Other / Unclear"
        relevance = "Needs Manual Review"

    # Special stricter identity handling for Indian (2001)
    if record_id == "2001_4" and not identity_match:
        relevance = "Needs Manual Review"

    if not identity_match:
        screening_reason = "Movie identity evidence is insufficient."
    else:
        screening_reason = (
            f"{identity_evidence}. Content classified as {video_type}."
        )

    return pd.Series({
        "screened_video_type": video_type,
        "screening_relevance": relevance,
        "identity_evidence": identity_evidence,
        "identity_score": identity_score,
        "identity_status": identity_status,
        "screening_reason": screening_reason
    })

screened = df.join(
    df.apply(screen_candidate, axis=1)
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

screened.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("BATCH 03 — AUTOMATED SCREENING RESULTS")
print("=" * 70)

print("Total candidates:", len(screened))
print("Unique video IDs:", screened["video_id"].nunique())

print("\nVIDEO TYPE DISTRIBUTION:")
display(
    screened["screened_video_type"]
    .value_counts()
    .rename_axis("video_type")
    .reset_index(name="candidates")
)

print("\nRELEVANCE DISTRIBUTION:")
display(
    screened["screening_relevance"]
    .value_counts()
    .rename_axis("relevance")
    .reset_index(name="candidates")
)

print("\nIDENTITY STATUS:")
display(
    screened["identity_status"]
    .value_counts()
    .rename_axis("identity_status")
    .reset_index(name="candidates")
)

print("\nMOVIE-WISE SCREENING:")
display(
    screened.groupby(
        ["movie_record_id", "movie_title", "screened_video_type"]
    ).size().unstack(fill_value=0).reset_index()
)

print("\nINDIAN (2001) — STRICT IDENTITY REVIEW:")
display(
    screened[
        screened["movie_record_id"] == "2001_4"
    ][[
        "video_id",
        "video_title",
        "channel_title",
        "screened_video_type",
        "identity_score",
        "identity_status",
        "screening_reason"
    ]]
)

print("\nSaved:", OUTPUT_FILE.exists())
print("Output shape:", screened.shape)

In [ ]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# BATCH 03 — YOUTUBE VIDEO METADATA VALIDATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"

OUTPUT_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

INPUT_FILE = RAW_DIR / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_03.csv"

OUTPUT_FILE = (
    OUTPUT_DIR / "ICMB_130_YOUTUBE_VIDEO_METADATA_BATCH_03.csv"
)

if OUTPUT_FILE.exists():
    raise FileExistsError("Batch 03 metadata file already exists.")

discovery = pd.read_csv(INPUT_FILE)

video_ids = discovery["video_id"].dropna().astype(str).unique().tolist()

metadata_records = []
errors = []

print("Video IDs to validate:", len(video_ids))

# YouTube videos.list accepts up to 50 IDs per request
for start in range(0, len(video_ids), 50):

    batch_ids = video_ids[start:start + 50]

    try:
        response = (
            youtube.videos()
            .list(
                part="snippet,contentDetails,statistics",
                id=",".join(batch_ids)
            )
            .execute()
        )

        for item in response.get("items", []):

            snippet = item.get("snippet", {})
            statistics = item.get("statistics", {})
            details = item.get("contentDetails", {})

            metadata_records.append({
                "video_id": item.get("id"),
                "video_title_actual": snippet.get("title", ""),
                "channel_id_actual": snippet.get("channelId", ""),
                "channel_title_actual": snippet.get("channelTitle", ""),
                "published_at_actual": snippet.get("publishedAt", ""),
                "category_id": snippet.get("categoryId", ""),
                "duration_iso8601": details.get("duration", ""),
                "definition": details.get("definition", ""),
                "view_count": statistics.get("viewCount", 0),
                "like_count": statistics.get("likeCount", 0),
                "comment_count": statistics.get("commentCount", 0),
                "metadata_status": "Retrieved"
            })

        print("Metadata retrieved:", len(metadata_records))

    except Exception as e:
        errors.append(str(e))
        print("API ERROR:", str(e))

metadata_df = pd.DataFrame(metadata_records)

# Preserve discovery associations
metadata_df["video_id"] = metadata_df["video_id"].astype(str)

metadata_df = discovery.merge(
    metadata_df,
    on="video_id",
    how="left"
)

metadata_df["metadata_status"] = metadata_df[
    "metadata_status"
].fillna("Unavailable")

metadata_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("BATCH 03 — METADATA VALIDATION SUMMARY")
print("=" * 70)

print("Discovery candidates:", len(discovery))
print("Metadata records retrieved:",
      (metadata_df["metadata_status"] == "Retrieved").sum())
print("Unavailable records:",
      (metadata_df["metadata_status"] == "Unavailable").sum())
print("Unique video IDs:", metadata_df["video_id"].nunique())
print("API errors:", len(errors))

print("\nMETADATA BY MOVIE:")
display(
    metadata_df.groupby("movie_title").agg(
        candidates=("video_id", "count"),
        retrieved=("metadata_status",
                   lambda x: (x == "Retrieved").sum())
    ).reset_index()
)

print("\nSaved:", OUTPUT_FILE.exists())
print("Output shape:", metadata_df.shape)

In [ ]:
import pandas as pd
import re
from pathlib import Path

# ------------------------------------------------------------
# BATCH 03 — SOURCE SHORTLIST
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

SCREENING_FILE = (
    PROCESSED_DIR / "ICMB_130_YOUTUBE_SCREENING_BATCH_03.csv"
)

METADATA_FILE = (
    PROCESSED_DIR / "ICMB_130_YOUTUBE_VIDEO_METADATA_BATCH_03.csv"
)

OUTPUT_FILE = (
    PROCESSED_DIR / "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_03.csv"
)

if OUTPUT_FILE.exists():
    raise FileExistsError("Batch 03 shortlist already exists.")

screening = pd.read_csv(SCREENING_FILE)
metadata = pd.read_csv(METADATA_FILE)

# Use metadata as the base and attach screening classifications
screen_cols = [
    "video_id",
    "screened_video_type",
    "screening_relevance",
    "identity_evidence",
    "identity_score",
    "identity_status",
    "screening_reason"
]

shortlist = metadata.merge(
    screening[screen_cols],
    on="video_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# DURATION PARSING
# ------------------------------------------------------------

def duration_to_seconds(value):
    if pd.isna(value):
        return None

    match = re.fullmatch(
        r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?",
        str(value)
    )

    if not match:
        return None

    hours, minutes, seconds = match.groups()

    return (
        int(hours or 0) * 3600
        + int(minutes or 0) * 60
        + int(seconds or 0)
    )

shortlist["duration_seconds"] = (
    shortlist["duration_iso8601"].apply(duration_to_seconds)
)

shortlist["duration_minutes"] = (
    shortlist["duration_seconds"] / 60
).round(2)

# ------------------------------------------------------------
# SOURCE CATEGORY
# ------------------------------------------------------------

def assign_category(row):

    relevance = str(row["screening_relevance"])
    video_type = str(row["screened_video_type"])
    identity = str(row["identity_status"])
    record_id = str(row["movie_record_id"])

    # Special protection for ambiguous movie identity
    if identity == "Manual Identity Verification Required":
        return "Hold — Identity Review"

    if record_id == "2001_4" and identity != "Strong Metadata Match — Candidate":
        return "Hold — Identity Review"

    if video_type == "Full Movie":
        return "Separate — Full Movie Comments"

    if video_type in ["Trailer", "Song"]:
        return "Exclude — Trailer/Song"

    if video_type == "Scene/Clip":
        return "Secondary — Scene-Specific Reaction"

    if video_type == "Movie Discussion":
        return "Secondary — Contextual Discussion"

    if video_type in ["Audience Reaction", "Movie Review"]:
        if "High" in relevance:
            return "Primary — Review / Reaction"

    return "Hold — Manual Review"

shortlist["source_category"] = shortlist.apply(
    assign_category, axis=1
)

# ------------------------------------------------------------
# METADATA PRIORITY
# ------------------------------------------------------------

def assign_priority(row):

    category = row["source_category"]

    if category.startswith("Exclude"):
        return "Exclude"

    if category.startswith("Hold"):
        return "Manual Review"

    comments = pd.to_numeric(
        row["comment_count"], errors="coerce"
    )

    if pd.isna(comments):
        return "Metadata Check"

    if comments >= 100:
        return "High"
    elif comments >= 30:
        return "Medium"
    else:
        return "Low"

shortlist["acquisition_priority"] = shortlist.apply(
    assign_priority, axis=1
)

# Metadata/title matching is not independent video verification
shortlist["identity_verification_status"] = (
    "Metadata screening only — not independently verified"
)

shortlist.loc[
    shortlist["source_category"] == "Hold — Identity Review",
    "identity_verification_status"
] = "Identity review required"

# ------------------------------------------------------------
# SAVE ALL 50 CANDIDATES
# ------------------------------------------------------------

shortlist.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("BATCH 03 — SOURCE SHORTLIST SUMMARY")
print("=" * 70)

print("Total candidates:", len(shortlist))
print("Unique video IDs:", shortlist["video_id"].nunique())

print("\nSOURCE CATEGORY DISTRIBUTION:")
display(
    shortlist["source_category"]
    .value_counts()
    .rename_axis("source_category")
    .reset_index(name="candidates")
)

print("\nPRIORITY DISTRIBUTION:")
display(
    shortlist["acquisition_priority"]
    .value_counts()
    .rename_axis("priority")
    .reset_index(name="candidates")
)

print("\nMOVIE-WISE SOURCE DISTRIBUTION:")
display(
    shortlist.groupby(
        ["movie_record_id", "movie_title", "source_category"]
    ).size().unstack(fill_value=0).reset_index()
)

print("\nINDIAN (2001) — SOURCE REVIEW:")
display(
    shortlist[
        shortlist["movie_record_id"] == "2001_4"
    ][[
        "video_id",
        "video_title_actual",
        "channel_title_actual",
        "screened_video_type",
        "source_category",
        "comment_count",
        "identity_verification_status"
    ]]
)

print("\nPRIMARY CANDIDATES:")
display(
    shortlist[
        shortlist["source_category"] == "Primary — Review / Reaction"
    ][[
        "movie_title",
        "video_id",
        "video_title_actual",
        "channel_title_actual",
        "duration_minutes",
        "view_count",
        "comment_count",
        "acquisition_priority"
    ]].sort_values(
        ["movie_title", "comment_count"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

print("\nSaved:", OUTPUT_FILE.exists())
print("Output shape:", shortlist.shape)

In [ ]:
# ------------------------------------------------------------
# BATCH 03 — TARGETED CANDIDATE REVIEW (CORRECTED)
# ------------------------------------------------------------

print("Available shortlist columns:")
print(shortlist.columns.tolist())

review_cols = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "video_title_actual",
    "channel_title_actual",
    "screened_video_type",
    "source_category",
    "comment_count",
    "duration_minutes"
]

# Keep only columns that actually exist
review_cols = [
    c for c in review_cols if c in shortlist.columns
]

unresolved = shortlist[
    shortlist["source_category"].isin([
        "Hold — Manual Review",
        "Hold — Identity Review"
    ])
].copy()

print("\n" + "=" * 70)
print("UNRESOLVED CANDIDATES")
print("=" * 70)
print("Candidates requiring review:", len(unresolved))

display(
    unresolved[review_cols]
    .sort_values(
        ["movie_title", "comment_count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("CURRENT PRIMARY CANDIDATES")
print("=" * 70)

primary = shortlist[
    shortlist["source_category"] == "Primary — Review / Reaction"
].copy()

display(
    primary[review_cols]
    .sort_values(
        ["movie_title", "comment_count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

In [ ]:
# ============================================================
# BATCH 03 — SAVE CORRECTED SOURCE SHORTLIST
# ============================================================

from pathlib import Path
import pandas as pd

# Correct project directory
PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_AUDIENCE_DIR = (
    PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"
)

PROCESSED_AUDIENCE_DIR.mkdir(parents=True, exist_ok=True)

# Confirm that the in-memory shortlist exists
assert "corrected" in globals(), (
    "Corrected shortlist not found. Rerun the previous correction cell first."
)

output_path = (
    PROCESSED_AUDIENCE_DIR /
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_03_CORRECTED.csv"
)

corrected.to_csv(output_path, index=False, encoding="utf-8-sig")

# Verify saved file
saved = pd.read_csv(output_path)

print("=" * 70)
print("BATCH 03 — CORRECTED SHORTLIST SAVED")
print("=" * 70)
print("File exists:", output_path.exists())
print("Rows:", len(saved))
print("Unique video IDs:", saved["video_id"].nunique())

print("\nSource category distribution:")
display(saved["source_category"].value_counts().to_frame("count"))

print("\nRemaining primary candidates:")
display(
    saved[
        saved["source_category"] == "Primary — Review / Reaction"
    ][[
        "movie_record_id",
        "movie_title",
        "video_id",
        "video_title_actual",
        "channel_title_actual",
        "comment_count",
        "duration_minutes"
    ]].sort_values(
        ["movie_title", "comment_count"],
        ascending=[True, False]
    )
)

print("\nSaved path:", output_path)

In [ ]:
# ============================================================
# BATCH 03 — TARGETED DISCOVERY FOR MISSING REVIEW SOURCES
# ============================================================

import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

target_movies = [
    {
        "movie_record_id": "2000_4",
        "movie_title": "Dulhan Hum Le Jayenge",
        "year": 2000,
        "query": '"Dulhan Hum Le Jayenge" 2000 full movie review reaction',
        "language": "hi"
    },
    {
        "movie_record_id": "2001_4",
        "movie_title": "Indian",
        "year": 2001,
        "query": '"Indian" "Sunny Deol" 2001 movie review reaction',
        "language": "hi"
    },
    {
        "movie_record_id": "2000_5",
        "movie_title": "Josh",
        "year": 2000,
        "query": '"Josh" "Shah Rukh Khan" 2000 movie review reaction',
        "language": "hi"
    }
]

new_records = []

for movie in target_movies:
    print(f"Searching: {movie['movie_title']}")

    response = youtube.search().list(
        part="snippet",
        q=movie["query"],
        type="video",
        maxResults=10,
        order="relevance",
        regionCode="IN",
        relevanceLanguage=movie["language"]
    ).execute()

    for item in response.get("items", []):
        snippet = item["snippet"]

        new_records.append({
            "movie_record_id": movie["movie_record_id"],
            "movie_title": movie["movie_title"],
            "movie_year": movie["year"],
            "search_query": movie["query"],
            "video_id": item["id"]["videoId"],
            "video_title": snippet.get("title"),
            "channel_id": snippet.get("channelId"),
            "channel_title": snippet.get("channelTitle"),
            "published_at": snippet.get("publishedAt"),
            "description": snippet.get("description"),
            "discovery_source": "Batch 03 Targeted Discovery"
        })

targeted_df = pd.DataFrame(new_records)

# Preserve all original discovery records
target_path = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "01_data" / "raw" / "audience_sentiment" / "youtube"
    / "ICMB_130_YOUTUBE_TARGETED_DISCOVERY_BATCH_03.csv"
)

targeted_df.to_csv(target_path, index=False, encoding="utf-8-sig")

print("\n" + "=" * 65)
print("TARGETED DISCOVERY SUMMARY")
print("=" * 65)
print("Candidates:", len(targeted_df))
print("Unique video IDs:", targeted_df["video_id"].nunique())
print("\nCandidates per movie:")
display(
    targeted_df.groupby("movie_title")
    .agg(
        candidates=("video_id", "count"),
        unique_videos=("video_id", "nunique")
    )
)

display(
    targeted_df[
        ["movie_title", "video_id", "video_title", "channel_title"]
    ]
)

print("\nSaved:", target_path)

In [ ]:
# ============================================================
# BATCH 03 — TARGETED CANDIDATE METADATA INSPECTION
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_YOUTUBE_DIR = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube"
)

target_path = (
    RAW_YOUTUBE_DIR /
    "ICMB_130_YOUTUBE_TARGETED_DISCOVERY_BATCH_03.csv"
)

targeted = pd.read_csv(target_path)

# Remove candidates already present in the original Batch 03 discovery
original_path = (
    RAW_YOUTUBE_DIR /
    "ICMB_130_YOUTUBE_DISCOVERY_BATCH_03.csv"
)

original = pd.read_csv(original_path)

existing_ids = set(original["video_id"].astype(str))

new_candidates = targeted[
    ~targeted["video_id"].astype(str).isin(existing_ids)
].copy()

print("Targeted candidates:", len(targeted))
print("Already present in original discovery:", len(targeted) - len(new_candidates))
print("New unique candidates:", new_candidates["video_id"].nunique())

# Fetch metadata only for new candidates
metadata_records = []

for _, row in new_candidates.iterrows():
    try:
        response = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=str(row["video_id"])
        ).execute()

        items = response.get("items", [])

        if not items:
            metadata_records.append({
                "video_id": row["video_id"],
                "metadata_status": "Unavailable"
            })
            continue

        item = items[0]
        snippet = item.get("snippet", {})
        stats = item.get("statistics", {})
        details = item.get("contentDetails", {})

        metadata_records.append({
            "movie_record_id": row["movie_record_id"],
            "movie_title": row["movie_title"],
            "video_id": row["video_id"],
            "video_title_actual": snippet.get("title"),
            "channel_title_actual": snippet.get("channelTitle"),
            "published_at_actual": snippet.get("publishedAt"),
            "duration_iso8601": details.get("duration"),
            "view_count": stats.get("viewCount"),
            "like_count": stats.get("likeCount"),
            "comment_count": stats.get("commentCount"),
            "description": snippet.get("description"),
            "default_language": snippet.get("defaultLanguage"),
            "metadata_status": "Retrieved"
        })

    except Exception as e:
        metadata_records.append({
            "video_id": row["video_id"],
            "metadata_status": "Error",
            "error": str(e)
        })

metadata_df = pd.DataFrame(metadata_records)

metadata_path = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment" /
    "ICMB_130_YOUTUBE_TARGETED_METADATA_BATCH_03.csv"
)

metadata_df.to_csv(metadata_path, index=False, encoding="utf-8-sig")

print("\nMetadata rows:", len(metadata_df))
print("Retrieved:", (metadata_df["metadata_status"] == "Retrieved").sum())

display(
    metadata_df[[
        "movie_title",
        "video_id",
        "video_title_actual",
        "channel_title_actual",
        "duration_iso8601",
        "view_count",
        "comment_count",
        "metadata_status"
    ]]
)

print("\nSaved:", metadata_path)

In [ ]:
# ============================================================
# BATCH 03 — FINAL TARGETED CLASSIFICATION UPDATE
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

input_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_03_CORRECTED.csv"
)

updated = pd.read_csv(input_path)

updates = {
    # Dulhan Hum Le Jayenge
    "DrG850kBr4U": (
        "Secondary — Contextual Discussion",
        "Unknown facts and budget video; not a dedicated movie review"
    ),
    "BhMdsChbAaI": (
        "Exclude — Short Cast Video",
        "48-second cast video; not audience feedback"
    ),
    "ce1bejqT-jc": (
        "Separate — Full Movie Comments",
        "Full-movie upload; comments must remain separate from review/reaction sources"
    ),
    "m4jdA8y4VYg": (
        "Exclude — Music Upload",
        "Music upload, not audience feedback"
    ),

    # Indian (2001)
    "rcbW5U0i7fQ": (
        "Exclude — Scene-Specific Reaction",
        "Reaction on Sunny Deol movie scene"
    ),
    "GNtdHb01su8": (
        "Exclude — Scene-Specific Reaction",
        "Reaction on selected scenes/dialogues"
    ),
    "Q7ou4yAKjhM": (
        "Secondary — Contextual Discussion",
        "Sunny Deol career discussion, not a dedicated Indian (2001) review"
    ),
    "vizQWSaPlqI": (
        "Hold — Identity and Content Verification",
        "Long video; film identity and actual content not independently confirmed"
    ),
    "WANphr8ey0I": (
        "Exclude — Scene-Specific Reaction",
        "Climax fight-scene reaction"
    ),

    # Josh
    "twECqLow1iM": (
        "Hold — Review Content Verification",
        "Potential movie review/revisit; content not independently verified"
    ),
    "YpaINyUyNWk": (
        "Exclude — Song-Specific Reaction",
        "Sailaru Sailare song reaction"
    ),
    "IQSNY5dOjFw": (
        "Secondary — Contextual Discussion",
        "Aamir Khan casting story; not a dedicated movie review"
    ),
    "PMRcXKfG3ag": (
        "Exclude — Unrelated Short",
        "Location-related short; not movie audience feedback"
    ),
    "5ipQ2j9dyGQ": (
        "Exclude — Short Facts Content",
        "Short facts video; not audience feedback"
    )
}

for video_id, (category, reason) in updates.items():
    mask = updated["video_id"].astype(str).eq(video_id)
    updated.loc[mask, "source_category"] = category
    updated.loc[mask, "screening_reason"] = reason

output_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_03_UPDATED.csv"
)

updated.to_csv(output_path, index=False, encoding="utf-8-sig")

print("=" * 70)
print("BATCH 03 — UPDATED CLASSIFICATION")
print("=" * 70)
print("Rows:", len(updated))
print("Unique videos:", updated["video_id"].nunique())

print("\nCategory distribution:")
display(updated["source_category"].value_counts().to_frame("count"))

print("\nCandidates retained for further verification:")
display(
    updated[
        updated["source_category"].isin([
            "Primary — Review / Reaction",
            "Hold — Review Content Verification",
            "Hold — Identity and Content Verification"
        ])
    ][[
        "movie_title",
        "video_id",
        "video_title_actual",
        "channel_title_actual",
        "comment_count",
        "source_category",
        "screening_reason"
    ]]
)

print("\nSaved:", output_path)

In [ ]:
# ============================================================
# BATCH 03 — VERIFY UNRESOLVED CANDIDATES
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

updated_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_03_UPDATED.csv"
)

updated = pd.read_csv(updated_path)

check_ids = [
    "twECqLow1iM",
    "vizQWSaPlqI",
    "LvgN1K8NeVs",
    "otw6m4U6IKw",
    "k2RbtGoxg4A"
]

check = updated[
    updated["video_id"].astype(str).isin(check_ids)
].copy()

columns = [
    "movie_title",
    "video_id",
    "video_title_actual",
    "channel_title_actual",
    "duration_minutes",
    "view_count",
    "comment_count",
    "source_category",
    "screening_reason"
]

print("Candidate verification:")
display(check[columns].reset_index(drop=True))

print("\nChecking whether targeted metadata candidates exist in updated shortlist:")

targeted_metadata_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_TARGETED_METADATA_BATCH_03.csv"
)

targeted_metadata = pd.read_csv(targeted_metadata_path)

target_ids = set(targeted_metadata["video_id"].astype(str))
updated_ids = set(updated["video_id"].astype(str))

print("Targeted metadata candidates:", len(target_ids))
print("Already in updated shortlist:", len(target_ids & updated_ids))
print("Not yet merged:", len(target_ids - updated_ids))

print("\nCandidates not yet merged:")
display(
    targeted_metadata[
        targeted_metadata["video_id"].astype(str).isin(target_ids - updated_ids)
    ][[
        "movie_title",
        "video_id",
        "video_title_actual",
        "channel_title_actual",
        "duration_iso8601",
        "comment_count"
    ]]
)

In [ ]:
# ============================================================
# BATCH 03 — MERGE TARGETED CANDIDATES INTO CORRECTED SHORTLIST
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

original_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_03_CORRECTED.csv"
)

targeted_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_TARGETED_METADATA_BATCH_03.csv"
)

original = pd.read_csv(original_path)
targeted_meta = pd.read_csv(targeted_path)

# Add classification fields to the targeted metadata
targeted = targeted_meta.copy()

targeted["source_category"] = "Hold — Manual Review"
targeted["acquisition_priority"] = "Manual Review"
targeted["identity_verification_status"] = "Not Independently Verified"
targeted["screening_reason"] = "Targeted candidate; classification based on available title and metadata"

decisions = {
    "ce1bejqT-jc": (
        "Separate — Full Movie Comments",
        "Separate",
        "Full-movie upload; keep audience comments separate from review/reaction sources"
    ),
    "DrG850kBr4U": (
        "Secondary — Contextual Discussion",
        "Medium",
        "Unknown facts and budget video; not a dedicated movie review"
    ),
    "BhMdsChbAaI": (
        "Exclude — Short Cast Video",
        "Exclude",
        "Short cast video; not audience feedback"
    ),
    "m4jdA8y4VYg": (
        "Exclude — Music Upload",
        "Exclude",
        "Music upload; not audience feedback"
    ),
    "rcbW5U0i7fQ": (
        "Exclude — Scene-Specific Reaction",
        "Exclude",
        "Reaction on selected Sunny Deol movie scene"
    ),
    "GNtdHb01su8": (
        "Exclude — Scene-Specific Reaction",
        "Exclude",
        "Reaction on selected movie scenes/dialogues"
    ),
    "Q7ou4yAKjhM": (
        "Secondary — Contextual Discussion",
        "Medium",
        "Sunny Deol career discussion; not a dedicated Indian (2001) review"
    ),
    "vizQWSaPlqI": (
        "Hold — Identity and Content Verification",
        "Manual Review",
        "Long video; identity and actual content not independently confirmed"
    ),
    "WANphr8ey0I": (
        "Exclude — Scene-Specific Reaction",
        "Exclude",
        "Climax fight-scene reaction"
    ),
    "IQSNY5dOjFw": (
        "Secondary — Contextual Discussion",
        "Medium",
        "Aamir Khan casting story; not a dedicated Josh review"
    ),
    "PMRcXKfG3ag": (
        "Exclude — Unrelated Short",
        "Exclude",
        "Location-related short; not movie audience feedback"
    ),
    "YpaINyUyNWk": (
        "Exclude — Song-Specific Reaction",
        "Exclude",
        "Song reaction; not general movie feedback"
    ),
    "twECqLow1iM": (
        "Hold — Review Content Verification",
        "Manual Review",
        "Potential Josh movie review/revisit; content not independently verified"
    ),
    "5ipQ2j9dyGQ": (
        "Exclude — Short Facts Content",
        "Exclude",
        "Short facts video; not audience feedback"
    )
}

for video_id, (category, priority, reason) in decisions.items():
    mask = targeted["video_id"].astype(str).eq(video_id)
    targeted.loc[mask, "source_category"] = category
    targeted.loc[mask, "acquisition_priority"] = priority
    targeted.loc[mask, "screening_reason"] = reason

# Align targeted metadata fields to original shortlist fields
for col in original.columns:
    if col not in targeted.columns:
        targeted[col] = pd.NA

targeted = targeted[original.columns]

# Preserve original 50; append only new unique IDs
existing_ids = set(original["video_id"].astype(str))

targeted_new = targeted[
    ~targeted["video_id"].astype(str).isin(existing_ids)
].copy()

combined = pd.concat(
    [original, targeted_new],
    ignore_index=True
)

# Save a new combined version
output_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_03_COMBINED.csv"
)

combined.to_csv(output_path, index=False, encoding="utf-8-sig")

# Verify
saved = pd.read_csv(output_path)

print("=" * 70)
print("BATCH 03 — COMBINED SHORTLIST VERIFICATION")
print("=" * 70)
print("Original rows:", len(original))
print("New targeted rows appended:", len(targeted_new))
print("Combined rows:", len(saved))
print("Unique video IDs:", saved["video_id"].nunique())

print("\nCategory distribution:")
display(saved["source_category"].value_counts().to_frame("count"))

print("\nCandidates retained for further verification:")
display(
    saved[
        saved["source_category"].isin([
            "Primary — Review / Reaction",
            "Hold — Review Content Verification",
            "Hold — Identity and Content Verification"
        ])
    ][[
        "movie_title",
        "video_id",
        "video_title_actual",
        "channel_title_actual",
        "comment_count",
        "source_category",
        "screening_reason"
    ]].reset_index(drop=True)
)

print("\nSaved:", output_path)

In [ ]:
# ============================================================
# BATCH 03 — FINAL CANDIDATE METADATA REVIEW
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

shortlist_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_03_COMBINED.csv"
)

shortlist = pd.read_csv(shortlist_path)

review_ids = [
    "LvgN1K8NeVs",   # Mission Kashmir
    "otw6m4U6IKw",   # Dil Chahta Hai
    "twECqLow1iM",   # Josh
    "vizQWSaPlqI"    # Indian
]

review = shortlist[
    shortlist["video_id"].astype(str).isin(review_ids)
].copy()

for col in ["description", "published_at_actual", "duration_iso8601"]:
    if col not in review.columns:
        review[col] = pd.NA

print("=" * 70)
print("BATCH 03 — FINAL CANDIDATE DETAILS")
print("=" * 70)

for _, row in review.iterrows():
    print("\n" + "-" * 70)
    print("Movie:", row["movie_title"])
    print("Video ID:", row["video_id"])
    print("Title:", row["video_title_actual"])
    print("Channel:", row["channel_title_actual"])
    print("Duration:", row["duration_minutes"], "minutes")
    print("Views:", row["view_count"])
    print("Comments:", row["comment_count"])
    print("Published:", row["published_at_actual"])
    print("Current category:", row["source_category"])
    print("\nDescription:")
    print(str(row["description"])[:2500])

print("\n" + "=" * 70)
print("Review completed.")

In [ ]:
# ============================================================
# BATCH 03 — FINAL TARGETED DISCOVERY
# Movies: Dulhan Hum Le Jayenge and Josh
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_YOUTUBE_DIR = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube"
)

target_movies = [
    {
        "movie_record_id": "2000_4",
        "movie_title": "Dulhan Hum Le Jayenge",
        "year": 2000,
        "query": '"Dulhan Hum Le Jayenge" movie review reaction full movie analysis',
        "language": "hi"
    },
    {
        "movie_record_id": "2000_5",
        "movie_title": "Josh",
        "year": 2000,
        "query": '"Josh" "Shah Rukh Khan" 2000 full movie review reaction analysis',
        "language": "hi"
    }
]

records = []

for movie in target_movies:
    print(f"Searching: {movie['movie_title']}")

    response = youtube.search().list(
        part="snippet",
        q=movie["query"],
        type="video",
        maxResults=10,
        order="relevance",
        regionCode="IN",
        relevanceLanguage=movie["language"]
    ).execute()

    for item in response.get("items", []):
        snippet = item["snippet"]

        records.append({
            "movie_record_id": movie["movie_record_id"],
            "movie_title": movie["movie_title"],
            "movie_year": movie["year"],
            "search_query": movie["query"],
            "video_id": item["id"]["videoId"],
            "video_title": snippet.get("title"),
            "channel_id": snippet.get("channelId"),
            "channel_title": snippet.get("channelTitle"),
            "published_at": snippet.get("publishedAt"),
            "description": snippet.get("description"),
            "discovery_source": "Batch 03 Final Targeted Discovery"
        })

final_targeted = pd.DataFrame(records)

output_path = (
    RAW_YOUTUBE_DIR /
    "ICMB_130_YOUTUBE_FINAL_TARGETED_DISCOVERY_BATCH_03.csv"
)

final_targeted.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 70)
print("FINAL TARGETED DISCOVERY SUMMARY")
print("=" * 70)
print("Total candidates:", len(final_targeted))
print("Unique video IDs:", final_targeted["video_id"].nunique())

display(
    final_targeted[
        ["movie_title", "video_id", "video_title", "channel_title"]
    ]
)

print("\nSaved:", output_path)

In [ ]:
# ============================================================
# BATCH 03 — FULL-MOVIE SOURCE METADATA VERIFICATION
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

candidates = [
    {
        "movie_record_id": "2001_4",
        "movie_title": "Indian",
        "year": 2001,
        "video_id": "vizQWSaPlqI"
    },
    {
        "movie_record_id": "2000_4",
        "movie_title": "Dulhan Hum Le Jayenge",
        "year": 2000,
        "video_id": "65Sfbl1m2K4"
    },
    {
        "movie_record_id": "2000_5",
        "movie_title": "Josh",
        "year": 2000,
        "video_id": "z0dciRZCp98"
    }
]

records = []

for candidate in candidates:
    response = youtube.videos().list(
        part="snippet,contentDetails,statistics,status",
        id=candidate["video_id"]
    ).execute()

    items = response.get("items", [])

    if not items:
        records.append({
            **candidate,
            "metadata_status": "Unavailable"
        })
        continue

    item = items[0]
    snippet = item.get("snippet", {})
    details = item.get("contentDetails", {})
    stats = item.get("statistics", {})
    status = item.get("status", {})

    records.append({
        **candidate,
        "video_title": snippet.get("title"),
        "channel_title": snippet.get("channelTitle"),
        "published_at": snippet.get("publishedAt"),
        "description": snippet.get("description"),
        "duration": details.get("duration"),
        "view_count": stats.get("viewCount"),
        "like_count": stats.get("likeCount"),
        "comment_count": stats.get("commentCount"),
        "privacy_status": status.get("privacyStatus"),
        "upload_status": status.get("uploadStatus"),
        "metadata_status": "Retrieved"
    })

full_movie_metadata = pd.DataFrame(records)

output_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_FULL_MOVIE_METADATA_BATCH_03.csv"
)

full_movie_metadata.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("BATCH 03 — FULL-MOVIE METADATA VERIFICATION")
print("=" * 70)

for _, row in full_movie_metadata.iterrows():
    print("\n" + "-" * 65)
    print("Movie:", row["movie_title"])
    print("Video ID:", row["video_id"])
    print("Title:", row.get("video_title"))
    print("Channel:", row.get("channel_title"))
    print("Duration:", row.get("duration"))
    print("Published:", row.get("published_at"))
    print("Views:", row.get("view_count"))
    print("Likes:", row.get("like_count"))
    print("Comments:", row.get("comment_count"))
    print("Privacy:", row.get("privacy_status"))
    print("Metadata:", row.get("metadata_status"))
    print("Description:", str(row.get("description", ""))[:1200])

print("\nSaved:", output_path)

In [ ]:
# ============================================================
# BATCH 03 — APPROVED YOUTUBE ACQUISITION MANIFEST
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

SHORTLIST_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_03_COMBINED.csv"
)

FULL_METADATA_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_FULL_MOVIE_METADATA_BATCH_03.csv"
)

shortlist = pd.read_csv(SHORTLIST_PATH)
full_metadata = pd.read_csv(FULL_METADATA_PATH)

approved_specs = [
    {
        "movie_record_id": "2000_3",
        "movie_title": "Mission Kashmir",
        "video_id": "LvgN1K8NeVs",
        "source_type": "General Movie Reaction",
        "series_part": "Part 2/3",
        "approval_rationale": (
            "General movie reaction. Part 2 of a three-part series; "
            "this sample represents only the available selected part."
        )
    },
    {
        "movie_record_id": "2001_5",
        "movie_title": "Dil Chahta Hai",
        "video_id": "otw6m4U6IKw",
        "source_type": "General Movie Reaction",
        "series_part": "Part 1",
        "approval_rationale": (
            "General movie reaction covering the film."
        )
    },
    {
        "movie_record_id": "2001_4",
        "movie_title": "Indian",
        "video_id": "vizQWSaPlqI",
        "source_type": "Full Movie Upload",
        "series_part": "Standalone Full Movie",
        "approval_rationale": (
            "Full-movie upload; verified as the 2001 Hindi film "
            "starring Sunny Deol. Comments kept separate."
        )
    },
    {
        "movie_record_id": "2000_4",
        "movie_title": "Dulhan Hum Le Jayenge",
        "video_id": "65Sfbl1m2K4",
        "source_type": "Full Movie Upload",
        "series_part": "Standalone Full Movie",
        "approval_rationale": (
            "Full-movie upload; title, cast, director and release "
            "details support film identity. Comments kept separate."
        )
    },
    {
        "movie_record_id": "2000_5",
        "movie_title": "Josh",
        "video_id": "z0dciRZCp98",
        "source_type": "Full Movie Upload",
        "series_part": "Standalone Full Movie",
        "approval_rationale": (
            "Full-movie upload; title, cast, director and synopsis "
            "support film identity. Comments kept separate."
        )
    }
]

manifest_records = []

for spec in approved_specs:

    video_id = spec["video_id"]

    # Locate source metadata
    if spec["source_type"] == "Full Movie Upload":
        match = full_metadata[
            full_metadata["video_id"].astype(str) == video_id
        ]
    else:
        match = shortlist[
            shortlist["video_id"].astype(str) == video_id
        ]

    if len(match) != 1:
        raise ValueError(
            f"Expected exactly one metadata record for {video_id}; "
            f"found {len(match)}"
        )

    source = match.iloc[0]

    manifest_records.append({
        **spec,
        "video_title": source.get(
            "video_title_actual",
            source.get("video_title")
        ),
        "channel_title": source.get(
            "channel_title_actual",
            source.get("channel_title")
        ),
        "published_at": source.get(
            "published_at_actual",
            source.get("published_at")
        ),
        "duration": source.get(
            "duration_iso8601",
            source.get("duration")
        ),
        "view_count": source.get("view_count"),
        "like_count": source.get("like_count"),
        "comment_count": source.get("comment_count"),
        "acquisition_approval_status": "Approved",
        "identity_verification_status": "Metadata and description verified",
        "planned_comment_sample": 100,
        "comment_order": "time",
        "comment_scope": "Top-level comments only",
        "replies_included": False
    })

manifest = pd.DataFrame(manifest_records)

# Save as a new file
output_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_03.csv"
)

manifest.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# Reload and verify
verified = pd.read_csv(output_path)

print("=" * 70)
print("BATCH 03 — APPROVED ACQUISITION MANIFEST")
print("=" * 70)

print("Approved sources:", len(verified))
print("Unique video IDs:", verified["video_id"].nunique())
print(
    "General reaction sources:",
    (verified["source_type"] == "General Movie Reaction").sum()
)
print(
    "Full-movie sources:",
    (verified["source_type"] == "Full Movie Upload").sum()
)
print(
    "Total planned comments:",
    verified["planned_comment_sample"].sum()
)

display(
    verified[[
        "movie_record_id",
        "movie_title",
        "video_id",
        "source_type",
        "series_part",
        "comment_count",
        "acquisition_approval_status"
    ]]
)

print("\nFile exists:", output_path.exists())
print("Saved:", output_path)

In [ ]:
# ============================================================
# BATCH 03 — YOUTUBE COMMENT ACQUISITION
# ============================================================

from pathlib import Path
import pandas as pd
from datetime import datetime, timezone
from googleapiclient.errors import HttpError
import time

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

COMMENTS_DIR = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

COMMENTS_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_03.csv"
)

manifest = pd.read_csv(manifest_path)

assert len(manifest) == 5
assert manifest["video_id"].nunique() == 5
assert (manifest["acquisition_approval_status"] == "Approved").all()

all_comments = []
acquisition_log = []

for _, source in manifest.iterrows():

    video_id = str(source["video_id"])
    movie_title = source["movie_title"]

    print(f"\nAcquiring comments: {movie_title} | {video_id}")

    video_comments = []
    next_page_token = None
    pages = 0
    error_message = None

    try:
        while len(video_comments) < 100:

            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, 100 - len(video_comments)),
                order="time",
                textFormat="plainText",
                pageToken=next_page_token
            ).execute()

            pages += 1

            for item in response.get("items", []):

                top_comment = item["snippet"]["topLevelComment"]
                snippet = top_comment["snippet"]

                video_comments.append({
                    "batch_id": "BATCH_03",
                    "movie_record_id": source["movie_record_id"],
                    "movie_title": movie_title,
                    "video_id": video_id,
                    "source_type": source["source_type"],
                    "series_part": source["series_part"],
                    "channel_title": source["channel_title"],
                    "comment_id": top_comment["id"],
                    "comment_text": snippet.get("textDisplay", ""),
                    "comment_author": snippet.get("authorDisplayName"),
                    "published_at": snippet.get("publishedAt"),
                    "updated_at": snippet.get("updatedAt"),
                    "like_count": snippet.get("likeCount", 0),
                    "reply_count": item["snippet"].get(
                        "totalReplyCount", 0
                    ),
                    "acquired_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat()
                })

                if len(video_comments) >= 100:
                    break

            next_page_token = response.get("nextPageToken")

            if not next_page_token or not response.get("items"):
                break

    except HttpError as e:
        error_message = str(e)

    all_comments.extend(video_comments)

    acquisition_log.append({
        "movie_record_id": source["movie_record_id"],
        "movie_title": movie_title,
        "video_id": video_id,
        "source_type": source["source_type"],
        "requested_comments": 100,
        "comments_acquired": len(video_comments),
        "pages_retrieved": pages,
        "acquisition_status": (
            "Success" if error_message is None else "API Error"
        ),
        "error_message": error_message,
        "acquired_at_utc": datetime.now(
            timezone.utc
        ).isoformat()
    })

    print("Comments acquired:", len(video_comments))

# Create dataframes
comments_df = pd.DataFrame(all_comments)
log_df = pd.DataFrame(acquisition_log)

# Save raw comments and acquisition log separately
comments_path = (
    COMMENTS_DIR /
    "ICMB_130_YOUTUBE_COMMENT_BATCH_03.csv"
)

log_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_03.csv"
)

comments_df.to_csv(
    comments_path,
    index=False,
    encoding="utf-8-sig"
)

log_df.to_csv(
    log_path,
    index=False,
    encoding="utf-8-sig"
)

# Verification
saved_comments = pd.read_csv(comments_path)
saved_log = pd.read_csv(log_path)

print("\n" + "=" * 70)
print("BATCH 03 — COMMENT ACQUISITION SUMMARY")
print("=" * 70)

print("Total comments:", len(saved_comments))
print("Unique comment IDs:", saved_comments["comment_id"].nunique())
print("Unique videos:", saved_comments["video_id"].nunique())
print("Empty comments:", saved_comments["comment_text"].fillna("").str.strip().eq("").sum())

print("\nComments by movie:")
display(
    saved_comments.groupby(
        ["movie_title", "source_type"]
    ).agg(
        comments=("comment_id", "count"),
        unique_comments=("comment_id", "nunique")
    )
)

print("\nAcquisition log:")
display(saved_log)

print("\nRaw comments saved:", comments_path)
print("Acquisition log saved:", log_path)

In [ ]:
# ============================================================
# BATCH 03 — COMMENT QUALITY PROFILING
# ============================================================

from pathlib import Path
import pandas as pd
import re
import unicodedata

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_DIR = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

input_path = (
    COMMENTS_DIR /
    "ICMB_130_YOUTUBE_COMMENT_BATCH_03.csv"
)

df = pd.read_csv(input_path)

# Preserve original comment text
df["comment_text"] = df["comment_text"].fillna("").astype(str)

# Basic text features
df["comment_char_count"] = df["comment_text"].str.len()

df["comment_word_count"] = df["comment_text"].apply(
    lambda x: len(x.split())
)

df["is_empty"] = df["comment_text"].str.strip().eq("")

df["is_very_short"] = (
    (df["comment_char_count"] <= 10) |
    (df["comment_word_count"] <= 2)
) & (~df["is_empty"])

df["contains_url"] = df["comment_text"].str.contains(
    r"https?://|www\.",
    case=False,
    regex=True
)

# Script profile
def detect_script(text):
    counts = {
        "Latin": 0,
        "Devanagari": 0,
        "Bengali": 0,
        "Tamil": 0,
        "Telugu": 0,
        "Kannada": 0,
        "Other": 0
    }

    for char in text:
        if not char.isalpha():
            continue

        code = ord(char)

        if 0x0041 <= code <= 0x024F:
            counts["Latin"] += 1
        elif 0x0900 <= code <= 0x097F:
            counts["Devanagari"] += 1
        elif 0x0980 <= code <= 0x09FF:
            counts["Bengali"] += 1
        elif 0x0B80 <= code <= 0x0BFF:
            counts["Tamil"] += 1
        elif 0x0C00 <= code <= 0x0C7F:
            counts["Telugu"] += 1
        elif 0x0C80 <= code <= 0x0CFF:
            counts["Kannada"] += 1
        else:
            counts["Other"] += 1

    active = [k for k, v in counts.items() if v > 0]

    if not active:
        return "Other / Emoji / Symbols"
    if len(active) > 1:
        return "Mixed Script"

    return active[0]

df["script_profile"] = df["comment_text"].apply(detect_script)

# Normalize for duplicate detection
def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["normalized_comment_text"] = (
    df["comment_text"].apply(normalize_text)
)

valid = df["normalized_comment_text"].ne("")

# Duplicate flags across all Batch 03 videos
df["duplicate_text_flag"] = False
df["duplicate_within_video_flag"] = False
df["duplicate_across_videos_flag"] = False

df.loc[valid, "duplicate_text_flag"] = (
    df.loc[valid, "normalized_comment_text"].duplicated(
        keep=False
    )
)

video_counts = (
    df.loc[valid]
    .groupby("normalized_comment_text")["video_id"]
    .nunique()
)

cross_video_texts = set(
    video_counts[video_counts > 1].index
)

df["duplicate_across_videos_flag"] = (
    df["normalized_comment_text"].isin(cross_video_texts)
)

within_video_counts = (
    df.loc[valid]
    .groupby(
        ["video_id", "normalized_comment_text"]
    )
    .size()
)

within_video_pairs = set(
    within_video_counts[within_video_counts > 1].index
)

df["duplicate_within_video_flag"] = [
    (video_id, text) in within_video_pairs
    for video_id, text in zip(
        df["video_id"],
        df["normalized_comment_text"]
    )
]

# Comment length categories
def classify_length(words):
    if words <= 2:
        return "Very Short"
    elif words <= 10:
        return "Short"
    elif words <= 40:
        return "Medium"
    return "Long"

df["length_category"] = df["comment_word_count"].apply(
    classify_length
)

# Save enriched profile
output_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_03.csv"
)

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# Summary
print("=" * 70)
print("BATCH 03 — COMMENT QUALITY PROFILE")
print("=" * 70)

print("Total comments:", len(df))
print("Unique comment IDs:", df["comment_id"].nunique())
print("Empty comments:", df["is_empty"].sum())
print("Very short comments:", df["is_very_short"].sum())
print("Comments containing URLs:", df["contains_url"].sum())

print("\nDuplicate indicators:")
print("Duplicate text records:", df["duplicate_text_flag"].sum())
print("Within-video duplicate records:", df["duplicate_within_video_flag"].sum())
print("Across-video duplicate records:", df["duplicate_across_videos_flag"].sum())

print("\nScript distribution:")
display(df["script_profile"].value_counts().to_frame("count"))

print("\nLength distribution:")
display(df["length_category"].value_counts().to_frame("count"))

print("\nMovie-wise quality summary:")
display(
    df.groupby(["movie_title", "source_type"]).agg(
        comments=("comment_id", "count"),
        average_words=("comment_word_count", "mean"),
        very_short=("is_very_short", "sum"),
        urls=("contains_url", "sum"),
        duplicate_text=("duplicate_text_flag", "sum")
    ).round(2)
)

print("\nSaved:", output_path)

In [ ]:
# ============================================================
# BATCH 04 — NEXT 10 MOVIE ALLOCATION
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

COMMENTS_DIR = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

REFERENCE_PATH = (
    PROCESSED_DIR /
    "ICMB_130_AUDIENCE_MOVIE_REFERENCE_VALIDATED.csv"
)

reference = pd.read_csv(REFERENCE_PATH)

# Identify movies already acquired from existing comment files
comment_files = sorted(
    COMMENTS_DIR.glob("ICMB_130_YOUTUBE_COMMENT_BATCH_*.csv")
)

acquired_movie_ids = set()

for file_path in comment_files:
    batch_df = pd.read_csv(file_path, usecols=["movie_record_id"])
    acquired_movie_ids.update(
        batch_df["movie_record_id"].dropna().astype(str).unique()
    )

# Also inspect the existing pilot file
pilot_path = (
    COMMENTS_DIR /
    "ICMB_130_YOUTUBE_COMMENT_PILOT.csv"
)

if pilot_path.exists():
    pilot_df = pd.read_csv(
        pilot_path,
        usecols=["movie_record_id"]
    )
    acquired_movie_ids.update(
        pilot_df["movie_record_id"].dropna().astype(str).unique()
    )

# Identify next 10 movies in reference order
reference["movie_record_id"] = (
    reference["movie_record_id"].astype(str)
)

remaining = reference[
    ~reference["movie_record_id"].isin(acquired_movie_ids)
].copy()

remaining = remaining.sort_values(
    ["year", "rank"]
).reset_index(drop=True)

batch04 = remaining.head(10).copy()

if len(batch04) != 10:
    raise ValueError(
        f"Only {len(batch04)} unacquired movies remain; "
        "check the reference and comment files."
    )

# Add batch information
batch04["batch_id"] = "BATCH_04"
batch04["planned_comment_sample"] = 100
batch04["allocation_status"] = "Allocated"

# Save allocation
output_path = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_BATCH_04_MOVIE_ALLOCATION.csv"
)

batch04.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

# Verification
saved = pd.read_csv(output_path)

print("=" * 70)
print("BATCH 04 — MOVIE ALLOCATION VERIFICATION")
print("=" * 70)

print("Movies allocated:", len(saved))
print("Unique movie IDs:", saved["movie_record_id"].nunique())
print("Already acquired movies excluded:", len(acquired_movie_ids))
print("Planned comments:", saved["planned_comment_sample"].sum())

display(
    saved[[
        "movie_record_id",
        "movie_title",
        "year",
        "rank",
        "language",
        "industry",
        "tmdb_id",
        "planned_comment_sample"
    ]]
)

print("\nSaved:", output_path)

In [ ]:
# ============================================================
# BATCH 04 — YOUTUBE DISCOVERY (10 MOVIES)
# ============================================================

import pandas as pd
import time
from pathlib import Path
from googleapiclient.discovery import build
from google.colab import userdata

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

RAW_DIR = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube"
)

ALLOCATION_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_BATCH_04_MOVIE_ALLOCATION.csv"
)

OUTPUT_PATH = (
    RAW_DIR /
    "ICMB_130_YOUTUBE_DISCOVERY_BATCH_04.csv"
)

allocation = pd.read_csv(ALLOCATION_PATH)

api_key = userdata.get("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=api_key)

# Language-specific search hints
language_hints = {
    "Hindi": "Hindi",
    "Tamil": "Tamil",
    "Telugu": "Telugu",
    "Kannada": "Kannada"
}

all_candidates = []
api_errors = []

for _, movie in allocation.iterrows():

    title = movie["movie_title"]
    year = int(movie["year"])
    language = movie["language"]
    record_id = movie["movie_record_id"]

    query = (
        f'"{title}" {year} {language_hints.get(language, language)} '
        f'movie review reaction audience'
    )

    print(f"Searching: {title} ({year})")

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=10,
            order="relevance",
            regionCode="IN",
            relevanceLanguage="hi" if language == "Hindi" else None
        ).execute()

        for item in response.get("items", []):

            snippet = item["snippet"]

            all_candidates.append({
                "movie_record_id": record_id,
                "movie_title": title,
                "movie_year": year,
                "language": language,
                "industry": movie["industry"],
                "tmdb_id": movie["tmdb_id"],
                "search_query": query,
                "video_id": item["id"]["videoId"],
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "discovery_source": "YouTube Data API v3",
                "discovery_batch": "BATCH_04"
            })

        time.sleep(0.2)

    except Exception as e:
        api_errors.append({
            "movie_record_id": record_id,
            "movie_title": title,
            "error": str(e)
        })
        print("API error:", str(e))

discovery = pd.DataFrame(all_candidates)

if not discovery.empty:
    discovery = discovery.drop_duplicates(
        subset=["video_id"]
    )

    discovery.to_csv(
        OUTPUT_PATH,
        index=False,
        encoding="utf-8-sig"
    )

print("\n" + "=" * 70)
print("BATCH 04 — DISCOVERY SUMMARY")
print("=" * 70)

print("Movies allocated:", allocation["movie_record_id"].nunique())
print("Candidate records:", len(all_candidates))
print("Unique video IDs:", discovery["video_id"].nunique() if not discovery.empty else 0)
print("Movies represented:", discovery["movie_record_id"].nunique() if not discovery.empty else 0)
print("API errors:", len(api_errors))

if not discovery.empty:
    print("\nCandidates per movie:")
    display(
        discovery.groupby(
            ["movie_record_id", "movie_title"]
        ).size().reset_index(name="candidate_count")
    )

    print("\nDiscovery preview:")
    display(
        discovery[
            ["movie_title", "video_id", "video_title", "channel_title"]
        ].head(20)
    )

print("\nSaved:", OUTPUT_PATH)

if api_errors:
    display(pd.DataFrame(api_errors))

In [ ]:
# ============================================================
# BATCH 04 — YOUTUBE VIDEO METADATA ENRICHMENT
# ============================================================

import pandas as pd
from pathlib import Path
from googleapiclient.discovery import build
from google.colab import userdata

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

DISCOVERY_PATH = (
    RAW_DIR / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_04.csv"
)

OUTPUT_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_METADATA_BATCH_04.csv"
)

discovery = pd.read_csv(DISCOVERY_PATH)

if discovery.empty:
    raise ValueError("Discovery file is empty.")

if discovery["video_id"].nunique() != len(discovery):
    raise ValueError("Duplicate video IDs found. Resolve before metadata retrieval.")

youtube = build(
    "youtube",
    "v3",
    developerKey=userdata.get("YOUTUBE_API_KEY")
)

metadata_records = []
api_errors = []

video_ids = discovery["video_id"].dropna().astype(str).tolist()

for start in range(0, len(video_ids), 50):
    batch_ids = video_ids[start:start + 50]

    try:
        response = youtube.videos().list(
            part="snippet,contentDetails,statistics,status",
            id=",".join(batch_ids),
            maxResults=50
        ).execute()

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            stats = item.get("statistics", {})
            details = item.get("contentDetails", {})
            status = item.get("status", {})

            metadata_records.append({
                "video_id": item["id"],
                "metadata_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "duration": details.get("duration"),
                "definition": details.get("definition"),
                "view_count": stats.get("viewCount"),
                "like_count": stats.get("likeCount"),
                "comment_count": stats.get("commentCount"),
                "privacy_status": status.get("privacyStatus"),
                "upload_status": status.get("uploadStatus")
            })

    except Exception as e:
        api_errors.append({
            "batch_start": start,
            "error": str(e)
        })

metadata = pd.DataFrame(metadata_records)

enriched = discovery.merge(
    metadata,
    on="video_id",
    how="left",
    suffixes=("", "_metadata")
)

enriched.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("BATCH 04 — METADATA ENRICHMENT SUMMARY")
print("=" * 70)
print("Discovered candidates:", len(discovery))
print("Unique discovered IDs:", discovery["video_id"].nunique())
print("Metadata records retrieved:", len(metadata))
print("Candidates with metadata:", enriched["duration"].notna().sum())
print("Candidates without metadata:", enriched["duration"].isna().sum())
print("API errors:", len(api_errors))

print("\nMetadata coverage by movie:")
display(
    enriched.groupby(["movie_record_id", "movie_title"])
    .agg(
        candidates=("video_id", "nunique"),
        metadata_retrieved=("duration", "count")
    )
    .reset_index()
)

print("\nCandidate metadata preview:")
display(
    enriched[[
        "movie_title", "video_id", "metadata_title",
        "channel_title", "duration", "view_count",
        "like_count", "comment_count"
    ]].head(20)
)

print("\nSaved:", OUTPUT_PATH)

if api_errors:
    display(pd.DataFrame(api_errors))

In [ ]:
# ============================================================
# BATCH 04 — AUTOMATED SOURCE SCREENING
# ============================================================

import pandas as pd
import re
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

INPUT_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_METADATA_BATCH_04.csv"
)

OUTPUT_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_04.csv"
)

df = pd.read_csv(INPUT_PATH)

def classify_video(row):

    title = str(row.get("metadata_title", "")).lower()
    description = str(row.get("description", "")).lower()
    text = title + " " + description

    duration = str(row.get("duration", ""))

    # Parse ISO 8601 YouTube duration
    match = re.fullmatch(
        r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?",
        duration
    )

    seconds = 0
    if match:
        h, m, s = match.groups()
        seconds = (
            int(h or 0) * 3600 +
            int(m or 0) * 60 +
            int(s or 0)
        )

    # Full movie candidate
    if (
        seconds >= 5400
        and any(term in text for term in [
            "full movie", "full film", "complete movie",
            "watch full", "hindi movie"
        ])
    ):
        return "Separate Full Movie", "High", "Long duration with full-movie wording"

    # Clearly irrelevant categories
    if any(term in title for term in [
        "song", "video song", "jukebox", "lyrics",
        "trailer", "teaser", "status", "4k edit",
        "scene", "best scene", "comedy scene",
        "dialogue", "shorts", "facts", "trivia",
        "unknown facts", "making of"
    ]):
        return "Exclude", "High", "Likely song, trailer, clip, edit or trivia"

    # Primary reviews and reactions
    if any(term in text for term in [
        "movie review", "film review", "review of",
        "reaction", "reacts to", "reacting to",
        "movie revisit", "rewatch", "first time watching",
        "watching for the first time",
        "my thoughts on", "movie explained"
    ]):
        return "Primary Review/Reaction", "Medium", "Review or reaction wording detected"

    # Contextual discussion
    if any(term in text for term in [
        "analysis", "explained", "ending explained",
        "character analysis", "retrospective",
        "discussion", "why it failed", "why it worked"
    ]):
        return "Secondary Contextual Discussion", "Medium", "Relevant discussion wording detected"

    # Short-duration and ambiguous candidates
    if seconds < 90:
        return "Exclude", "Medium", "Very short video; likely clip or promotional content"

    return "Hold Manual Review", "Low", "Content type not established by metadata"

results = df.apply(classify_video, axis=1, result_type="expand")

df["screening_category"] = results[0]
df["screening_confidence"] = results[1]
df["screening_reason"] = results[2]

df["duration_seconds"] = df["duration"].apply(
    lambda x: (
        int(re.search(r"(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", str(x)).group(1) or 0) * 3600
        + int(re.search(r"(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", str(x)).group(2) or 0) * 60
        + int(re.search(r"(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", str(x)).group(3) or 0)
    )
    if pd.notna(x) else 0
)

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("BATCH 04 — SOURCE SCREENING SUMMARY")
print("=" * 70)

print("Total candidates:", len(df))
print("Unique video IDs:", df["video_id"].nunique())

print("\nCategory distribution:")
display(
    df["screening_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="candidate_count")
)

print("\nResults by movie:")
display(
    pd.crosstab(
        df["movie_title"],
        df["screening_category"]
    ).reset_index()
)

print("\nPrimary review/reaction candidates:")
display(
    df.loc[
        df["screening_category"] == "Primary Review/Reaction",
        ["movie_title", "video_id", "metadata_title",
         "channel_title", "duration", "view_count",
         "comment_count", "screening_confidence"]
    ]
)

print("\nFull-movie candidates:")
display(
    df.loc[
        df["screening_category"] == "Separate Full Movie",
        ["movie_title", "video_id", "metadata_title",
         "channel_title", "duration", "view_count",
         "comment_count"]
    ]
)

print("\nSaved:", OUTPUT_PATH)

In [ ]:
# ============================================================
# BATCH 04 — REVIEW UNCERTAIN AND FULL-MOVIE CANDIDATES
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_PATH = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment" /
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_04.csv"
)

df = pd.read_csv(INPUT_PATH)

review_categories = [
    "Hold Manual Review",
    "Secondary Contextual Discussion",
    "Separate Full Movie"
]

for category in review_categories:

    subset = df[df["screening_category"] == category].copy()

    print("\n" + "=" * 90)
    print(category.upper(), "| Candidates:", len(subset))
    print("=" * 90)

    display(
        subset[[
            "movie_record_id",
            "movie_title",
            "video_id",
            "metadata_title",
            "channel_title",
            "duration",
            "view_count",
            "like_count",
            "comment_count",
            "description",
            "screening_reason"
        ]].reset_index(drop=True)
    )

print("\n" + "=" * 90)
print("PRIMARY CANDIDATE COVERAGE BY MOVIE")
print("=" * 90)

primary = df[
    df["screening_category"] == "Primary Review/Reaction"
]

display(
    primary.groupby(
        ["movie_record_id", "movie_title"]
    ).agg(
        candidate_count=("video_id", "nunique"),
        distinct_channels=("channel_id", "nunique")
    ).reset_index()
)

In [ ]:
# ============================================================
# BATCH 04 — MANUAL DECISION SHEET
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

INPUT_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_04.csv"
)

OUTPUT_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_04.csv"
)

df = pd.read_csv(INPUT_PATH)

# Preserve the automated classification.
df["original_screening_category"] = df["screening_category"]

# Start with the automated recommendation, but do not treat it as approval.
df["manual_decision"] = "Pending"
df["manual_verification_status"] = "Not verified"
df["manual_notes"] = ""

# Apply only the clearly supported metadata-based decisions.
decisions = {
    "OSsupclsnV4": ("Exclude", "Malini Sharma biography"),
    "v8QLP-AVWBk": ("Secondary Contextual Discussion", "Raaz retrospective"),
    "N26nB6Zpo8s": ("Primary Review/Reaction", "Film revisit and personal thoughts"),
    "vFZOmk4984Y": ("Exclude", "Music launch"),
    "0b-gtdsJIsI": ("Primary Review/Reaction", "Kaante retrospective review"),
    "7UzmGhuK68I": ("Primary Review/Reaction", "Film discussion: has it aged well"),
    "bGQw7K8v070": ("Primary Review/Reaction", "Foreign audience reaction"),
    "WuKud_a8Rag": ("Exclude", "Song-related content"),
    "fLgW5KB769c": ("Exclude", "Unrelated comedy content"),
    "mM5AHGlKhBo": ("Secondary Contextual Discussion", "Film success retrospective"),
    "Sk5xqz-r1e4": ("Primary Review/Reaction", "Film-focused review/reaction"),
    "aNAR31Ak4Aw": ("Secondary Contextual Discussion", "Director/cast discussion"),
    "kl92u4fXhUY": ("Hold Manual Review", "Re-release coverage requires content confirmation"),
    "B2yTWV0Yevo": ("Exclude", "Interview covering multiple films")
}

for video_id, (decision, note) in decisions.items():
    mask = df["video_id"].astype(str) == video_id
    df.loc[mask, "manual_decision"] = decision
    df.loc[mask, "manual_notes"] = note
    df.loc[mask, "manual_verification_status"] = "Metadata-based recommendation"

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Total candidates:", len(df))
print("Decisions populated:", (df["manual_decision"] != "Pending").sum())
print("Still pending:", (df["manual_decision"] == "Pending").sum())

print("\nDecision distribution:")
display(
    df["manual_decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="count")
)

print("\nSaved:", OUTPUT_PATH)

In [ ]:
# ============================================================
# BATCH 04 — PENDING CANDIDATE REVIEW QUEUE
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_PATH = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment" /
    "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_04.csv"
)

df = pd.read_csv(INPUT_PATH)

pending = df[df["manual_decision"] == "Pending"].copy()

# Prioritize likely review/reaction candidates, then full movies,
# then remaining uncertain candidates.
priority = {
    "Primary Review/Reaction": 1,
    "Separate Full Movie": 2,
    "Hold Manual Review": 3,
    "Secondary Contextual Discussion": 4,
    "Exclude": 5
}

pending["review_priority"] = (
    pending["original_screening_category"]
    .map(priority)
    .fillna(6)
)

pending = pending.sort_values(
    ["movie_year", "movie_title", "review_priority", "view_count"],
    ascending=[True, True, True, False]
)

print("=" * 75)
print("BATCH 04 — PENDING REVIEW QUEUE")
print("=" * 75)

print("Pending candidates:", len(pending))
print("Movies represented:", pending["movie_record_id"].nunique())

print("\nPending candidates by original category:")
display(
    pending["original_screening_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="count")
)

for movie, group in pending.groupby("movie_title", sort=False):

    print(f"\n{'=' * 75}\n{movie} — {len(group)} pending candidates\n{'=' * 75}")

    display(
        group[[
            "video_id",
            "metadata_title",
            "channel_title",
            "duration",
            "view_count",
            "comment_count",
            "original_screening_category"
        ]]
    )

In [ ]:
# ============================================================
# BATCH 04 — PROPOSED SOURCE DECISIONS
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

INPUT_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_04.csv"
)

OUTPUT_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_PROPOSED_DECISIONS_BATCH_04.csv"
)

df = pd.read_csv(INPUT_PATH)

# Keep original decisions unchanged.
df["proposed_decision"] = df["manual_decision"]
df["proposed_source_type"] = ""
df["proposal_reason"] = ""

# Proposed primary sources.
primary_sources = {
    "nL8c1ZVMg9Q": ("Primary Review/Reaction", "Movie Review"),
    "5poXL2KsT-M": ("Primary Review/Reaction", "Revisit Reaction"),
    "xJ19W4fTjrE": ("Primary Review/Reaction", "Movie Review"),
    "meBbNyI7258": ("Primary Review/Reaction", "Movie Reaction"),
    "_3S1aVYJxuY": ("Primary Review/Reaction", "Movie Review"),
    "5pqLlrKOLGo": ("Primary Review/Reaction", "Movie Reaction"),
    "NXPeYKMqy_w": ("Primary Review/Reaction", "Revisit Reaction"),
    "v5VTctTyw1s": ("Primary Review/Reaction", "Movie Reaction Part 1/4"),
    "iI6qwt3R1WY": ("Primary Review/Reaction", "Movie Reaction Part 1/3")
}

for video_id, (decision, source_type) in primary_sources.items():
    mask = df["video_id"].astype(str) == video_id
    df.loc[mask, "proposed_decision"] = decision
    df.loc[mask, "proposed_source_type"] = source_type
    df.loc[mask, "proposal_reason"] = "Proposed primary source; pending final approval"

# Proposed full-movie sources.
full_movie_sources = {
    "btYIMkogNWo": "Full Movie Upload",
    "MlI-uKrYqeA": "Full Movie Upload",
    "M6mcuySVazA": "Full Movie Upload"
}

for video_id, source_type in full_movie_sources.items():
    mask = df["video_id"].astype(str) == video_id
    df.loc[mask, "proposed_decision"] = "Separate Full Movie"
    df.loc[mask, "proposed_source_type"] = source_type
    df.loc[mask, "proposal_reason"] = "Proposed full-movie source; identity and availability checks required"

# Mark The Hero as having no suitable candidate in this discovery batch.
hero_mask = df["movie_title"].eq("The Hero: Love Story of a Spy")
df.loc[hero_mask, "proposal_reason"] = (
    "No suitable review/reaction candidate identified in this batch"
)

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("BATCH 04 — PROPOSED DECISION SUMMARY")
print("=" * 70)

print("Total candidates:", len(df))
print("Proposed primary sources:", len(primary_sources))
print("Proposed full-movie sources:", len(full_movie_sources))
print("Original manual decisions preserved:", "manual_decision" in df.columns)

print("\nProposed selections:")
display(
    df[df["proposed_source_type"] != ""][
        [
            "movie_title",
            "video_id",
            "metadata_title",
            "channel_title",
            "proposed_decision",
            "proposed_source_type",
            "proposal_reason"
        ]
    ].reset_index(drop=True)
)

print("\nSaved:", OUTPUT_PATH)

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_FILE = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_PROPOSED_DECISIONS_BATCH_04.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_FINAL_SOURCE_MANIFEST_BATCH_04.csv"
)

df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")

# Explicitly approved primary selections
primary_ids = {
    "nL8c1ZVMg9Q",
    "5poXL2KsT-M",
    "xJ19W4fTjrE",
    "meBbNyI7258",
    "_3S1aVYJxuY",
    "5pqLlrKOLGo",
    "NXPeYKMqy_w",
    "v5VTctTyw1s",
    "iI6qwt3R1WY"
}

# Supplementary full-movie selections
full_movie_ids = {
    "btYIMkogNWo",
    "MlI-uKrYqeA",
    "M6mcuySVazA"
}

selected = df[
    df["video_id"].isin(primary_ids | full_movie_ids)
].copy()

selected["acquisition_role"] = selected["video_id"].map(
    lambda x: (
        "Primary Audience Discussion"
        if x in primary_ids
        else "Supplementary Full-Movie Discussion"
    )
)

selected["acquisition_status"] = selected["video_id"].map(
    lambda x: (
        "Approved for primary comment acquisition"
        if x in primary_ids
        else "Supplementary source — excluded from primary acquisition"
    )
)

selected["verification_status"] = (
    "Metadata-based selection; video not independently watched"
)

selected["sampling_target"] = selected["video_id"].map(
    lambda x: 100 if x in primary_ids else 0
)

selected["sampling_unit"] = "Top-level YouTube comments"
selected["sampling_order"] = "time"
selected["replies_included"] = False

primary = selected[selected["video_id"].isin(primary_ids)].copy()
supplementary = selected[
    selected["video_id"].isin(full_movie_ids)
].copy()

# Validation before saving
assert len(primary) == 9, f"Expected 9 primary sources; found {len(primary)}"
assert len(supplementary) == 3, (
    f"Expected 3 supplementary sources; found {len(supplementary)}"
)
assert selected["video_id"].nunique() == 12
assert primary["movie_title"].nunique() == 9
assert primary["sampling_target"].sum() == 900

selected.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("=" * 70)
print("BATCH 04 — CORRECTED SOURCE MANIFEST")
print("=" * 70)

print("Primary sources:", len(primary))
print("Supplementary full-movie sources:", len(supplementary))
print("Unique selected video IDs:", selected["video_id"].nunique())
print("Primary movies covered:", primary["movie_title"].nunique())
print("Primary comment target:", primary["sampling_target"].sum())

print("\nPRIMARY SOURCES")
print(
    primary[
        ["movie_title", "video_id", "channel_title"]
    ].to_string(index=False)
)

print("\nSUPPLEMENTARY SOURCES")
print(
    supplementary[
        ["movie_title", "video_id", "channel_title"]
    ].to_string(index=False)
)

print("\nSaved:", OUTPUT_FILE)
print("VALIDATION PASSED")

In [ ]:
import os
import time
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import userdata

# ============================================================
# BATCH 04 — YOUTUBE COMMENT ACQUISITION
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

MANIFEST_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_FINAL_SOURCE_MANIFEST_BATCH_04.csv"
)

RAW_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment"
)

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

COMMENTS_FILE = (
    RAW_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_04.csv"
)

LOG_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_04.csv"
)

# ------------------------------------------------------------
# 1. SAFETY CHECKS
# ------------------------------------------------------------

if not MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"Source manifest not found: {MANIFEST_FILE}"
    )

if COMMENTS_FILE.exists() or LOG_FILE.exists():
    raise FileExistsError(
        "Batch 04 acquisition output already exists. "
        "Stopping to prevent accidental overwrite."
    )

manifest = pd.read_csv(
    MANIFEST_FILE,
    encoding="utf-8-sig"
)

required_columns = [
    "movie_title",
    "video_id",
    "channel_title",
    "proposed_source_type",
    "acquisition_role"
]

missing = [
    col for col in required_columns
    if col not in manifest.columns
]

if missing:
    raise ValueError(
        f"Manifest missing required columns: {missing}"
    )

primary = manifest[
    manifest["acquisition_role"]
    == "Primary Audience Discussion"
].copy()

if len(primary) != 9:
    raise ValueError(
        f"Expected 9 primary sources, found {len(primary)}"
    )

if primary["video_id"].nunique() != 9:
    raise ValueError("Duplicate video IDs found.")

# ------------------------------------------------------------
# 2. YOUTUBE API CONNECTION
# ------------------------------------------------------------

api_key = userdata.get("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError(
        "YOUTUBE_API_KEY not found in Colab Secrets."
    )

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key
)

print("YouTube API connection initialized.")
print("Primary videos:", len(primary))
print("Target comments per video: 100")

# ------------------------------------------------------------
# 3. COMMENT ACQUISITION
# ------------------------------------------------------------

all_comments = []
acquisition_log = []

for idx, row in primary.reset_index(drop=True).iterrows():

    video_id = str(row["video_id"])
    movie_title = str(row["movie_title"])
    channel_title = str(row["channel_title"])

    print("\n" + "=" * 65)
    print(f"Movie {idx + 1}/9: {movie_title}")
    print("Video ID:", video_id)
    print("Channel:", channel_title)

    video_comments = []
    next_page_token = None
    request_count = 0
    api_status = "SUCCESS"
    error_message = ""

    try:
        while len(video_comments) < 100:

            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(
                    100,
                    100 - len(video_comments)
                ),
                order="time",
                textFormat="plainText",
                pageToken=next_page_token
            ).execute()

            request_count += 1

            for item in response.get("items", []):

                top_comment = (
                    item["snippet"]["topLevelComment"]
                )

                snippet = top_comment["snippet"]

                video_comments.append({
                    "movie_record_id": row.get(
                        "movie_record_id", ""
                    ),
                    "movie_title": movie_title,
                    "video_id": video_id,
                    "channel_title": channel_title,
                    "source_type": row[
                        "proposed_source_type"
                    ],
                    "comment_id": top_comment["id"],
                    "comment_text": snippet.get(
                        "textDisplay", ""
                    ),
                    "author_name": snippet.get(
                        "authorDisplayName", ""
                    ),
                    "published_at": snippet.get(
                        "publishedAt", ""
                    ),
                    "updated_at": snippet.get(
                        "updatedAt", ""
                    ),
                    "like_count": snippet.get(
                        "likeCount", 0
                    ),
                    "reply_count": item[
                        "snippet"
                    ].get("totalReplyCount", 0),
                    "comment_position": len(
                        video_comments
                    ) + 1,
                    "sampling_order": "time",
                    "sampling_unit": "top_level_comment"
                })

                if len(video_comments) >= 100:
                    break

            next_page_token = response.get(
                "nextPageToken"
            )

            if not next_page_token:
                break

        if len(video_comments) == 0:
            api_status = "SUCCESS_NO_COMMENTS"

    except HttpError as e:

        api_status = "API_ERROR"
        error_message = str(e)

    except Exception as e:

        api_status = "ERROR"
        error_message = str(e)

    all_comments.extend(video_comments)

    acquisition_log.append({
        "movie_title": movie_title,
        "video_id": video_id,
        "channel_title": channel_title,
        "target_comments": 100,
        "comments_acquired": len(video_comments),
        "unique_comment_ids": len({
            x["comment_id"] for x in video_comments
        }),
        "api_requests": request_count,
        "api_status": api_status,
        "error_message": error_message,
        "acquisition_timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "sampling_order": "time",
        "replies_included": False
    })

    print("Comments acquired:", len(video_comments))
    print("API status:", api_status)

    if error_message:
        print("Error:", error_message[:500])

    time.sleep(0.2)

# ------------------------------------------------------------
# 4. SAVE RAW COMMENTS AND ACQUISITION LOG
# ------------------------------------------------------------

comments_df = pd.DataFrame(all_comments)
log_df = pd.DataFrame(acquisition_log)

comments_df.to_csv(
    COMMENTS_FILE,
    index=False,
    encoding="utf-8-sig"
)

log_df.to_csv(
    LOG_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 5. VALIDATION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BATCH 04 — COMMENT ACQUISITION SUMMARY")
print("=" * 70)

print("Videos attempted:", len(log_df))
print("Successful videos:", (
    log_df["api_status"] == "SUCCESS"
).sum())

print("Videos with no comments:", (
    log_df["api_status"] == "SUCCESS_NO_COMMENTS"
).sum())

print("API errors:", (
    log_df["api_status"].isin(
        ["API_ERROR", "ERROR"]
    )
).sum())

print("Total comments acquired:", len(comments_df))

if not comments_df.empty:
    print(
        "Unique comment IDs:",
        comments_df["comment_id"].nunique()
    )
    print(
        "Empty comments:",
        comments_df["comment_text"]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )
    print("\nComments by movie:")
    print(
        comments_df.groupby("movie_title")
        .size()
        .to_string()
    )

print("\nAcquisition log:")
print(
    log_df[
        [
            "movie_title",
            "comments_acquired",
            "api_status",
            "api_requests"
        ]
    ].to_string(index=False)
)

print("\nRaw comments saved:", COMMENTS_FILE)
print("Acquisition log saved:", LOG_FILE)

print("\nACQUISITION CELL COMPLETED")

In [ ]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

# ============================================================
# BATCH 04 — YOUTUBE COMMENT QUALITY PROFILING
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_FILE = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments/"
    / "ICMB_130_YOUTUBE_COMMENT_BATCH_04.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment"
)

PROFILE_FILE = (
    OUTPUT_DIR
    / "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_04.csv"
)

MOVIE_FILE = (
    OUTPUT_DIR
    / "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_04.csv"
)

if not RAW_FILE.exists():
    raise FileNotFoundError(f"Raw comments file not found: {RAW_FILE}")

if PROFILE_FILE.exists() or MOVIE_FILE.exists():
    raise FileExistsError(
        "Batch 04 quality profile already exists. "
        "Stopping to avoid overwriting."
    )

df = pd.read_csv(RAW_FILE, encoding="utf-8-sig")

required = [
    "movie_title",
    "video_id",
    "comment_id",
    "comment_text"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# ------------------------------------------------------------
# 1. BASIC TEXT NORMALIZATION
# ------------------------------------------------------------

df["comment_text"] = df["comment_text"].fillna("").astype(str)

df["normalized_text"] = (
    df["comment_text"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df["character_count"] = df["comment_text"].str.len()

df["word_count"] = df["comment_text"].apply(
    lambda x: len(x.split())
)

df["is_empty"] = df["normalized_text"].eq("")

df["is_very_short"] = (
    (df["character_count"] <= 10)
    | (df["word_count"] <= 2)
)

# Length category based on word count
def classify_length(words):
    if words <= 2:
        return "Very Short"
    elif words <= 5:
        return "Short"
    elif words <= 20:
        return "Medium"
    else:
        return "Long"

df["length_category"] = df["word_count"].apply(
    classify_length
)

# ------------------------------------------------------------
# 2. SCRIPT COMPOSITION
# ------------------------------------------------------------

def detect_script(text):
    scripts = set()

    for char in text:
        if not char.isalpha():
            continue

        name = unicodedata.name(char, "")

        if "LATIN" in name:
            scripts.add("Latin")
        elif "DEVANAGARI" in name:
            scripts.add("Devanagari")
        elif "BENGALI" in name:
            scripts.add("Bengali")
        elif "TAMIL" in name:
            scripts.add("Tamil")
        elif "TELUGU" in name:
            scripts.add("Telugu")
        elif "GUJARATI" in name:
            scripts.add("Gujarati")
        elif "GURMUKHI" in name:
            scripts.add("Gurmukhi")
        elif "MALAYALAM" in name:
            scripts.add("Malayalam")
        elif "KANNADA" in name:
            scripts.add("Kannada")
        elif "ARABIC" in name:
            scripts.add("Arabic")
        elif "ORIYA" in name or "ODIA" in name:
            scripts.add("Odia")
        else:
            scripts.add("Other")

    if not scripts:
        return "Other / Emoji / Symbols"
    elif len(scripts) > 1:
        return "Mixed Script"
    else:
        return list(scripts)[0]

df["script_category"] = df["comment_text"].apply(
    detect_script
)

# ------------------------------------------------------------
# 3. URL FLAGS
# ------------------------------------------------------------

df["contains_url"] = df["comment_text"].str.contains(
    r"https?://|www\.",
    case=False,
    regex=True,
    na=False
)

# ------------------------------------------------------------
# 4. DUPLICATE TEXT FLAGS
# ------------------------------------------------------------

# Ignore empty comments for duplicate-text analysis.
valid_text = df["normalized_text"].ne("")

text_video_counts = (
    df[valid_text]
    .groupby("normalized_text")["video_id"]
    .nunique()
)

text_record_counts = (
    df[valid_text]
    .groupby("normalized_text")["comment_id"]
    .count()
)

df["duplicate_text_flag"] = (
    valid_text
    & df["normalized_text"].map(text_record_counts).fillna(0).gt(1)
)

df["duplicate_within_video_flag"] = False
df["duplicate_across_videos_flag"] = False

for text_value, group in df[valid_text].groupby("normalized_text"):
    if len(group) > 1:
        duplicated_rows = group.index

        if group["video_id"].nunique() == 1:
            df.loc[
                duplicated_rows,
                "duplicate_within_video_flag"
            ] = True

        if group["video_id"].nunique() > 1:
            df.loc[
                duplicated_rows,
                "duplicate_across_videos_flag"
            ] = True

# ------------------------------------------------------------
# 5. OVERALL QUALITY PROFILE
# ------------------------------------------------------------

profile_rows = [
    ("total_comments", len(df)),
    ("unique_comment_ids", df["comment_id"].nunique()),
    ("unique_videos", df["video_id"].nunique()),
    ("empty_comments", int(df["is_empty"].sum())),
    ("very_short_comments", int(df["is_very_short"].sum())),
    ("comments_containing_urls", int(df["contains_url"].sum())),
    ("duplicate_text_records", int(df["duplicate_text_flag"].sum())),
    ("duplicate_within_video_records", int(
        df["duplicate_within_video_flag"].sum()
    )),
    ("duplicate_across_video_records", int(
        df["duplicate_across_videos_flag"].sum()
    )),
    ("average_word_count", round(df["word_count"].mean(), 2)),
    ("median_word_count", round(df["word_count"].median(), 2)),
    ("average_character_count", round(
        df["character_count"].mean(), 2
    ))
]

for category, count in df["script_category"].value_counts().items():
    profile_rows.append((f"script_{category}", int(count)))

for category, count in df["length_category"].value_counts().items():
    profile_rows.append((f"length_{category}", int(count)))

profile_df = pd.DataFrame(
    profile_rows,
    columns=["metric", "value"]
)

# ------------------------------------------------------------
# 6. MOVIE-WISE QUALITY PROFILE
# ------------------------------------------------------------

movie_profile = (
    df.groupby("movie_title")
    .agg(
        comments=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        average_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        very_short_comments=("is_very_short", "sum"),
        empty_comments=("is_empty", "sum"),
        url_comments=("contains_url", "sum"),
        duplicate_text_records=("duplicate_text_flag", "sum"),
        duplicate_within_video=("duplicate_within_video_flag", "sum"),
        duplicate_across_videos=("duplicate_across_videos_flag", "sum")
    )
    .reset_index()
)

movie_profile["average_words"] = (
    movie_profile["average_words"].round(2)
)

movie_profile["median_words"] = (
    movie_profile["median_words"].round(2)
)

# ------------------------------------------------------------
# 7. SAVE PROFILES
# ------------------------------------------------------------

profile_df.to_csv(
    PROFILE_FILE,
    index=False,
    encoding="utf-8-sig"
)

movie_profile.to_csv(
    MOVIE_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 8. VALIDATION AND DISPLAY
# ------------------------------------------------------------

print("=" * 70)
print("BATCH 04 — COMMENT QUALITY PROFILE")
print("=" * 70)

print("\nOVERALL METRICS")
print(profile_df.to_string(index=False))

print("\nMOVIE-WISE PROFILE")
print(movie_profile.to_string(index=False))

print("\nSCRIPT DISTRIBUTION")
print(df["script_category"].value_counts().to_string())

print("\nLENGTH DISTRIBUTION")
print(df["length_category"].value_counts().to_string())

print("\nVALIDATION")
print("Raw comments:", len(df))
print("Unique comment IDs:", df["comment_id"].nunique())
print("Unique videos:", df["video_id"].nunique())
print("Empty comments:", int(df["is_empty"].sum()))
print("Raw file preserved:", RAW_FILE.exists())

print("\nSaved overall profile:", PROFILE_FILE)
print("Saved movie profile:", MOVIE_FILE)

assert len(df) == 846
assert df["comment_id"].nunique() == 846
assert df["video_id"].nunique() == 9
assert df["is_empty"].sum() == 0

print("\nQUALITY PROFILING COMPLETED")

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# BATCH 05 — MOVIE ALLOCATION
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

REFERENCE_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_AUDIENCE_MOVIE_REFERENCE_VALIDATED.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_BATCH_05_MOVIE_ALLOCATION.csv"
)

COMMENT_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment"
)

# ------------------------------------------------------------
# 1. LOAD VALIDATED 130-MOVIE REFERENCE
# ------------------------------------------------------------

if not REFERENCE_FILE.exists():
    raise FileNotFoundError(
        f"Validated movie reference not found: {REFERENCE_FILE}"
    )

reference = pd.read_csv(
    REFERENCE_FILE,
    encoding="utf-8-sig"
)

required_columns = [
    "movie_record_id",
    "movie_title",
    "year",
    "rank",
    "language",
    "tmdb_id",
    "imdb_id"
]

missing = [
    col for col in required_columns
    if col not in reference.columns
]

if missing:
    raise ValueError(f"Missing reference columns: {missing}")

# ------------------------------------------------------------
# 2. IDENTIFY ALREADY ACQUIRED MOVIES
# ------------------------------------------------------------

acquired_ids = set()

# Pilot and Batches 01–04 raw comment files
batch_files = [
    "ICMB_130_YOUTUBE_COMMENT_PILOT.csv",
    "ICMB_130_YOUTUBE_COMMENT_BATCH_01.csv",
    "ICMB_130_YOUTUBE_COMMENT_BATCH_02.csv",
    "ICMB_130_YOUTUBE_COMMENT_BATCH_03.csv",
    "ICMB_130_YOUTUBE_COMMENT_BATCH_04.csv"
]

for filename in batch_files:
    file_path = COMMENT_DIR / filename

    if not file_path.exists():
        print("File not found; checking alternative:", filename)
        continue

    existing = pd.read_csv(
        file_path,
        usecols=["movie_record_id"],
        encoding="utf-8-sig"
    )

    ids = (
        existing["movie_record_id"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    acquired_ids.update(ids)

# ------------------------------------------------------------
# 3. EXCLUDE PREVIOUSLY COVERED MOVIES
# ------------------------------------------------------------

reference["movie_record_id"] = (
    reference["movie_record_id"].astype(str)
)

remaining = reference[
    ~reference["movie_record_id"].isin(acquired_ids)
].copy()

# Preserve original reference order.
remaining = remaining.reset_index(drop=True)

# ------------------------------------------------------------
# 4. SELECT NEXT 10 MOVIES
# ------------------------------------------------------------

batch_05 = remaining.head(10).copy()

if len(batch_05) != 10:
    raise ValueError(
        f"Only {len(batch_05)} movies remain; expected 10."
    )

batch_05["batch_number"] = 5
batch_05["acquisition_status"] = "Not yet acquired"
batch_05["planned_comment_target"] = 100

# ------------------------------------------------------------
# 5. VALIDATION BEFORE SAVING
# ------------------------------------------------------------

if OUTPUT_FILE.exists():
    raise FileExistsError(
        "Batch 05 allocation file already exists. "
        "Stopping to prevent accidental overwrite."
    )

assert batch_05["movie_record_id"].nunique() == 10
assert not batch_05["movie_record_id"].isin(acquired_ids).any()

batch_05.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 70)
print("BATCH 05 — MOVIE ALLOCATION SUMMARY")
print("=" * 70)

print("Total movies in reference:", len(reference))
print("Previously covered movie IDs:", len(acquired_ids))
print("Remaining movies:", len(remaining))
print("Batch 05 movies allocated:", len(batch_05))
print(
    "Planned maximum comments:",
    batch_05["planned_comment_target"].sum()
)

print("\nBATCH 05 MOVIES")
print(
    batch_05[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "rank",
            "language",
            "tmdb_id",
            "imdb_id"
        ]
    ].to_string(index=False)
)

print("\nDuplicate movie IDs:", batch_05["movie_record_id"].duplicated().sum())
print(
    "Previously acquired movies included:",
    batch_05["movie_record_id"].isin(acquired_ids).sum()
)

print("\nSaved allocation:")
print(OUTPUT_FILE)

print("\nBATCH 05 ALLOCATION COMPLETED")

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from googleapiclient.discovery import build
from google.colab import userdata

# ============================================================
# BATCH 05 — YOUTUBE VIDEO DISCOVERY
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

ALLOCATION_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_BATCH_05_MOVIE_ALLOCATION.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/"
    / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_05.csv"
)

# ------------------------------------------------------------
# 1. CHECK INPUT AND OUTPUT
# ------------------------------------------------------------

if not ALLOCATION_FILE.exists():
    raise FileNotFoundError(
        f"Allocation file not found: {ALLOCATION_FILE}"
    )

if OUTPUT_FILE.exists():
    raise FileExistsError(
        "Batch 05 discovery file already exists. "
        "Stopping to prevent accidental overwrite."
    )

allocation = pd.read_csv(
    ALLOCATION_FILE,
    encoding="utf-8-sig"
)

if len(allocation) != 10:
    raise ValueError(
        f"Expected 10 movies, found {len(allocation)}"
    )

# ------------------------------------------------------------
# 2. CONNECT TO YOUTUBE API
# ------------------------------------------------------------

api_key = userdata.get("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError(
        "YOUTUBE_API_KEY not found in Colab Secrets."
    )

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key
)

print("YouTube API connection initialized.")
print("Movies to search:", len(allocation))
print("Maximum candidates per movie: 10")

# ------------------------------------------------------------
# 3. DISCOVER VIDEOS
# ------------------------------------------------------------

all_candidates = []
search_log = []

for index, movie in allocation.reset_index(drop=True).iterrows():

    movie_title = str(movie["movie_title"])
    movie_year = int(movie["year"])
    movie_id = str(movie["movie_record_id"])
    language = str(movie["language"])
    industry = str(movie.get("industry", ""))

    query = f"{movie_title} {movie_year} movie review reaction"

    print("\n" + "=" * 65)
    print(f"Movie {index + 1}/10: {movie_title}")
    print("Search query:", query)

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=10,
            order="relevance"
        ).execute()

        items = response.get("items", [])

        for rank, item in enumerate(items, start=1):

            snippet = item.get("snippet", {})

            all_candidates.append({
                "movie_record_id": movie_id,
                "movie_title": movie_title,
                "movie_year": movie_year,
                "language": language,
                "industry": industry,
                "search_query": query,
                "candidate_rank": rank,
                "video_id": item["id"]["videoId"],
                "video_title": snippet.get("title", ""),
                "channel_id": snippet.get("channelId", ""),
                "channel_title": snippet.get("channelTitle", ""),
                "published_at": snippet.get("publishedAt", ""),
                "description": snippet.get("description", ""),
                "discovery_source": "YouTube Data API v3",
                "discovery_timestamp_utc": datetime.now(
                    timezone.utc
                ).isoformat()
            })

        search_log.append({
            "movie_record_id": movie_id,
            "movie_title": movie_title,
            "search_query": query,
            "candidates_found": len(items),
            "api_status": "SUCCESS",
            "error_message": ""
        })

        print("Candidates retrieved:", len(items))

    except Exception as e:

        search_log.append({
            "movie_record_id": movie_id,
            "movie_title": movie_title,
            "search_query": query,
            "candidates_found": 0,
            "api_status": "ERROR",
            "error_message": str(e)
        })

        print("API error:", str(e)[:300])

# ------------------------------------------------------------
# 4. CREATE DISCOVERY DATAFRAME
# ------------------------------------------------------------

discovery_df = pd.DataFrame(all_candidates)
search_log_df = pd.DataFrame(search_log)

if discovery_df.empty:
    raise RuntimeError(
        "No candidates retrieved. Discovery file not saved."
    )

# ------------------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BATCH 05 — DISCOVERY SUMMARY")
print("=" * 70)

print("Total candidates:", len(discovery_df))
print("Unique video IDs:", discovery_df["video_id"].nunique())
print("Movies represented:", discovery_df["movie_record_id"].nunique())
print("Successful searches:", (
    search_log_df["api_status"] == "SUCCESS"
).sum())
print("Search errors:", (
    search_log_df["api_status"] == "ERROR"
).sum())

print("\nCandidates by movie:")
print(
    discovery_df.groupby("movie_title")
    .size()
    .to_string()
)

print("\nSearch log:")
print(
    search_log_df[
        [
            "movie_title",
            "candidates_found",
            "api_status"
        ]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 6. SAVE DISCOVERY FILE
# ------------------------------------------------------------

discovery_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nDiscovery file saved:")
print(OUTPUT_FILE)

print("\nBATCH 05 DISCOVERY COMPLETED")

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from googleapiclient.discovery import build
from google.colab import userdata

# ============================================================
# BATCH 05 — YOUTUBE VIDEO METADATA
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

DISCOVERY_FILE = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/"
    / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_05.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_METADATA_BATCH_05.csv"
)

if not DISCOVERY_FILE.exists():
    raise FileNotFoundError(DISCOVERY_FILE)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        "Metadata output already exists. "
        "Stopping to prevent overwrite."
    )

discovery = pd.read_csv(
    DISCOVERY_FILE,
    encoding="utf-8-sig"
)

video_ids = (
    discovery["video_id"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

if len(video_ids) != 100:
    raise ValueError(
        f"Expected 100 unique video IDs; found {len(video_ids)}"
    )

# ------------------------------------------------------------
# YOUTUBE API CONNECTION
# ------------------------------------------------------------

api_key = userdata.get("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError("YOUTUBE_API_KEY missing from Colab Secrets.")

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key
)

print("YouTube API connection initialized.")
print("Unique videos to retrieve:", len(video_ids))

# ------------------------------------------------------------
# FETCH METADATA IN BATCHES OF 50
# ------------------------------------------------------------

metadata_records = []
api_errors = []

for start in range(0, len(video_ids), 50):

    batch_ids = video_ids[start:start + 50]

    print(
        f"\nRetrieving metadata for videos "
        f"{start + 1}–{start + len(batch_ids)}"
    )

    try:
        response = youtube.videos().list(
            part="snippet,contentDetails,statistics,status",
            id=",".join(batch_ids),
            maxResults=50
        ).execute()

        for item in response.get("items", []):

            snippet = item.get("snippet", {})
            statistics = item.get("statistics", {})
            details = item.get("contentDetails", {})
            status = item.get("status", {})

            metadata_records.append({
                "video_id": item["id"],
                "metadata_title": snippet.get("title", ""),
                "channel_id": snippet.get("channelId", ""),
                "channel_title": snippet.get("channelTitle", ""),
                "published_at": snippet.get("publishedAt", ""),
                "description": snippet.get("description", ""),
                "duration": details.get("duration", ""),
                "definition": details.get("definition", ""),
                "view_count": statistics.get("viewCount", 0),
                "like_count": statistics.get("likeCount", 0),
                "comment_count": statistics.get("commentCount", 0),
                "privacy_status": status.get("privacyStatus", ""),
                "embeddable": status.get("embeddable", None),
                "metadata_retrieval_status": "SUCCESS"
            })

    except Exception as e:

        api_errors.append({
            "batch_start": start,
            "video_ids": ";".join(batch_ids),
            "error": str(e)
        })

# ------------------------------------------------------------
# MERGE WITH DISCOVERY RECORDS
# ------------------------------------------------------------

metadata_df = pd.DataFrame(metadata_records)

if metadata_df.empty:
    raise RuntimeError("No metadata retrieved. Output not saved.")

metadata_df = discovery.merge(
    metadata_df,
    on="video_id",
    how="left",
    validate="one_to_one"
)

metadata_df["metadata_retrieval_status"] = (
    metadata_df["metadata_retrieval_status"]
    .fillna("NOT_RETRIEVED")
)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

metadata_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BATCH 05 — METADATA RETRIEVAL SUMMARY")
print("=" * 70)

print("Discovery candidates:", len(discovery))
print("Metadata records retrieved:", len(metadata_records))
print("Unique metadata video IDs:", metadata_df["video_id"].nunique())

print(
    "Successful metadata records:",
    (metadata_df["metadata_retrieval_status"] == "SUCCESS").sum()
)

print(
    "Missing metadata records:",
    (metadata_df["metadata_retrieval_status"] != "SUCCESS").sum()
)

print("API batch errors:", len(api_errors))

print("\nMetadata coverage by movie:")
print(
    metadata_df.groupby("movie_title")[
        "metadata_retrieval_status"
    ].value_counts().to_string()
)

print("\nSaved:", OUTPUT_FILE)

if api_errors:
    print("\nAPI errors:")
    print(pd.DataFrame(api_errors).to_string(index=False))

print("\nBATCH 05 METADATA RETRIEVAL COMPLETED")

In [ ]:
import pandas as pd
import re
from pathlib import Path

# ============================================================
# BATCH 05 — AUTOMATED YOUTUBE SOURCE SCREENING (CORRECTED)
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_METADATA_BATCH_05.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_05.csv"
)

if not INPUT_FILE.exists():
    raise FileNotFoundError(INPUT_FILE)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        "Screening file already exists. "
        "Stopping to prevent overwrite."
    )

df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")

# ------------------------------------------------------------
# 1. VERIFY REQUIRED COLUMNS
# ------------------------------------------------------------

required_columns = [
    "movie_title",
    "video_id",
    "metadata_title",
    "description_y",
    "channel_title_y",
    "duration",
    "view_count",
    "like_count",
    "comment_count"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# Use metadata-retrieved fields for screening.
df["screening_title"] = df["metadata_title"].fillna("")
df["screening_description"] = df["description_y"].fillna("")
df["screening_channel"] = df["channel_title_y"].fillna("")

df["search_text"] = (
    df["screening_title"] + " "
    + df["screening_description"]
).str.lower()

# ------------------------------------------------------------
# 2. PARSE VIDEO DURATION
# ------------------------------------------------------------

def parse_duration(value):
    if pd.isna(value):
        return 0

    match = re.fullmatch(
        r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?",
        str(value)
    )

    if not match:
        return 0

    h, m, s = [
        int(x) if x else 0
        for x in match.groups()
    ]

    return h * 3600 + m * 60 + s

df["duration_seconds"] = df["duration"].apply(
    parse_duration
)

# ------------------------------------------------------------
# 3. KEYWORD GROUPS
# ------------------------------------------------------------

review_terms = [
    "movie review", "film review", "review",
    "reaction", "reacts", "reacting",
    "first time watching", "watching for the first time",
    "revisit", "rewatch", "honest opinion",
    "movie discussion", "film discussion",
    "movie analysis", "film analysis"
]

full_movie_terms = [
    "full movie", "full film", "complete movie",
    "full hindi movie", "full tamil movie",
    "full telugu movie", "full hd movie"
]

exclude_terms = [
    "official trailer", "teaser", "full song",
    "lyrical video", "whatsapp status",
    "status video", "best scene", "movie clip",
    "shorts", "short video", "making",
    "behind the scenes", "cast salary",
    "unknown facts", "facts about",
    "box office collection"
]

secondary_terms = [
    "retrospective", "legacy", "then and now",
    "cinema history", "film history",
    "cult classic", "impact of",
    "success story", "career analysis"
]

# ------------------------------------------------------------
# 4. CLASSIFICATION
# ------------------------------------------------------------

def classify_video(row):

    text = row["search_text"]
    duration = row["duration_seconds"]

    if not str(row["screening_title"]).strip():
        return "Hold Manual Review", "Missing video title"

    # Very short videos are not full reviews.
    if duration < 120:
        return (
            "Exclude",
            "Very short video; unlikely to contain a substantive review"
        )

    # Full movie candidates
    if any(term in text for term in full_movie_terms):

        if duration >= 3600:
            return (
                "Separate Full Movie",
                "Full-movie keyword with long duration"
            )

        return (
            "Hold Manual Review",
            "Full-movie keyword but duration requires verification"
        )

    # Trailer, song, clip and unrelated content
    if any(term in text for term in exclude_terms):
        return (
            "Exclude",
            "Trailer, song, clip, short or unrelated content"
        )

    # Primary review/reaction
    if any(term in text for term in review_terms):
        return (
            "Primary Review/Reaction",
            "Review or reaction keyword with sufficient duration"
        )

    # Contextual discussion
    if any(term in text for term in secondary_terms):
        return (
            "Secondary Contextual Discussion",
            "Retrospective or contextual discussion"
        )

    return (
        "Hold Manual Review",
        "No decisive content classification"
    )


classified = df.apply(
    classify_video,
    axis=1,
    result_type="expand"
)

df["automated_category"] = classified[0]
df["classification_reason"] = classified[1]

# ------------------------------------------------------------
# 5. PROPOSED SOURCE TYPE
# ------------------------------------------------------------

def identify_source_type(row):

    if row["automated_category"] == "Separate Full Movie":
        return "Full Movie Upload"

    if row["automated_category"] == "Secondary Contextual Discussion":
        return "Contextual Discussion"

    if row["automated_category"] == "Primary Review/Reaction":

        text = row["search_text"]

        if "revisit" in text or "rewatch" in text:
            return "Revisit Reaction"

        if "review" in text:
            return "Movie Review"

        if "reaction" in text or "reacts" in text:
            return "Movie Reaction"

        return "Review/Discussion"

    return "Unclassified"


df["proposed_source_type"] = df.apply(
    identify_source_type,
    axis=1
)

# ------------------------------------------------------------
# 6. SORT AND SAVE
# ------------------------------------------------------------

category_order = {
    "Primary Review/Reaction": 1,
    "Separate Full Movie": 2,
    "Secondary Contextual Discussion": 3,
    "Hold Manual Review": 4,
    "Exclude": 5
}

df["_sort_order"] = df["automated_category"].map(
    category_order
)

df = df.sort_values(
    ["movie_title", "_sort_order", "candidate_rank"]
).drop(columns="_sort_order")

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("BATCH 05 — AUTOMATED SOURCE SCREENING SUMMARY")
print("=" * 70)

print("Total candidates:", len(df))
print("Unique video IDs:", df["video_id"].nunique())

print("\nCATEGORY DISTRIBUTION")
print(df["automated_category"].value_counts().to_string())

print("\nSOURCE TYPE DISTRIBUTION")
print(df["proposed_source_type"].value_counts().to_string())

print("\nRESULTS BY MOVIE")
print(
    pd.crosstab(
        df["movie_title"],
        df["automated_category"]
    ).to_string()
)

print("\nPRIMARY CANDIDATES")
print(
    df[df["automated_category"] == "Primary Review/Reaction"][
        [
            "movie_title",
            "video_id",
            "metadata_title",
            "channel_title_y",
            "duration",
            "view_count",
            "comment_count",
            "proposed_source_type"
        ]
    ].to_string(index=False)
)

print("\nFULL-MOVIE CANDIDATES")
print(
    df[df["automated_category"] == "Separate Full Movie"][
        [
            "movie_title",
            "video_id",
            "metadata_title",
            "channel_title_y",
            "duration",
            "view_count",
            "comment_count"
        ]
    ].to_string(index=False)
)

print("\nSaved:", OUTPUT_FILE)
print("\nBATCH 05 SCREENING COMPLETED")

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# BATCH 05 — MANUAL SOURCE REVIEW SHEET
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_05.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_05.csv"
)

if not INPUT_FILE.exists():
    raise FileNotFoundError(INPUT_FILE)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        "Manual-review file already exists. "
        "Stopping to prevent overwrite."
    )

df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")

# Preserve original automated screening.
df["original_automated_category"] = df["automated_category"]

# Manual decision fields.
df["manual_decision"] = "Pending"
df["manual_source_type"] = ""
df["manual_verification_status"] = "Not yet reviewed"
df["manual_review_notes"] = ""

# Keep metadata fields visible for review.
review_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "metadata_title",
    "channel_title_y",
    "duration",
    "view_count",
    "like_count",
    "comment_count",
    "original_automated_category",
    "classification_reason",
    "manual_decision",
    "manual_source_type",
    "manual_verification_status",
    "manual_review_notes",
    "description_y"
]

review_df = df[review_columns].copy()

review_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("BATCH 05 — MANUAL REVIEW SHEET")
print("=" * 70)

print("Total candidates:", len(review_df))
print("Movies covered:", review_df["movie_title"].nunique())
print("Unique video IDs:", review_df["video_id"].nunique())

print("\nManual decisions:")
print(review_df["manual_decision"].value_counts().to_string())

print("\nCandidates by movie:")
print(review_df.groupby("movie_title").size().to_string())

print("\nSaved:", OUTPUT_FILE)

print("\nMANUAL REVIEW SHEET CREATED")

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_05.csv"
)

df = pd.read_csv(FILE, encoding="utf-8-sig")

hero = df[
    df["movie_title"].str.contains(
        "The Hero: Love Story of a Spy",
        case=False,
        na=False
    )
].copy()

columns_to_show = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "channel_title_y",
    "duration",
    "view_count",
    "comment_count",
    "original_automated_category",
    "classification_reason",
    "description_y"
]

print("THE HERO — YOUTUBE CANDIDATE REVIEW")
print("=" * 80)
print("Total candidates:", len(hero))

display(hero[columns_to_show].reset_index(drop=True))

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_05.csv"
)

df = pd.read_csv(FILE, encoding="utf-8-sig")

video_id = "RD9WGhsGi_U"
mask = df["video_id"].astype(str).eq(video_id)

if mask.sum() != 1:
    raise ValueError(
        f"Expected exactly one matching row for {video_id}; "
        f"found {mask.sum()}. No changes saved."
    )

df.loc[mask, "manual_decision"] = "Include"
df.loc[mask, "manual_source_type"] = "Movie Review"
df.loc[mask, "manual_verification_status"] = "Content Reviewed"
df.loc[mask, "manual_review_notes"] = (
    "Watched and confirmed substantial discussion of the film. "
    "Suitable as a movie review/rewind discussion source."
)

df.to_csv(FILE, index=False, encoding="utf-8-sig")

print("Updated The Hero source successfully.")
display(df.loc[mask, [
    "movie_title",
    "video_id",
    "metadata_title",
    "manual_decision",
    "manual_source_type",
    "manual_verification_status",
    "manual_review_notes"
]])

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_05.csv"
)

df = pd.read_csv(FILE, encoding="utf-8-sig")

text_columns = [
    "manual_decision",
    "manual_source_type",
    "manual_verification_status",
    "manual_review_notes"
]

for col in text_columns:
    df[col] = df[col].astype("string")

df.to_csv(FILE, index=False, encoding="utf-8-sig")

print("Manual-review text columns converted to string.")
print("The Hero decision:", df.loc[
    df["video_id"].eq("RD9WGhsGi_U"),
    ["manual_decision", "manual_source_type"]
].to_dict("records"))

In [ ]:
dhoom = df[df["movie_title"].eq("Dhoom")].copy()

print("DHOOM — YOUTUBE CANDIDATE REVIEW")
print("=" * 80)
print("Total candidates:", len(dhoom))

display(dhoom[[
    "video_id",
    "metadata_title",
    "channel_title_y",
    "duration",
    "view_count",
    "comment_count",
    "original_automated_category",
    "classification_reason",
    "description_y"
]].reset_index(drop=True))

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_05.csv"
)

df = pd.read_csv(FILE, encoding="utf-8-sig")

text_columns = [
    "manual_decision",
    "manual_source_type",
    "manual_verification_status",
    "manual_review_notes"
]

for col in text_columns:
    df[col] = df[col].astype("string")

decisions = {
    "6Yqx7HiM3WM": (
        "Include",
        "Movie Review",
        "Metadata Reviewed",
        "Selected as the suitable long-form Dhoom discussion source."
    ),
    "T3ob1ganXTo": (
        "Exclude",
        "Not Suitable",
        "Metadata Reviewed",
        "Excluded: source considered irrelevant to the project."
    ),
    "OPtAJ_a5lSE": (
        "Exclude",
        "Short Review",
        "Metadata Reviewed",
        "Excluded: too short for the intended audience-comment analysis."
    )
}

for video_id, values in decisions.items():

    mask = df["video_id"].astype(str).eq(video_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected one row for {video_id}; found {mask.sum()}."
        )

    df.loc[mask, text_columns] = list(values)

df.to_csv(FILE, index=False, encoding="utf-8-sig")

print("Dhoom manual decisions updated successfully.")

result = df[df["video_id"].isin(decisions.keys())]

display(result[[
    "movie_title",
    "video_id",
    "metadata_title",
    "manual_decision",
    "manual_source_type",
    "manual_verification_status",
    "manual_review_notes"
]])

In [ ]:
mujhse = df[
    df["movie_title"].eq("Mujhse Shaadi Karogi")
].copy()

print("MUJHSE SHAADI KAROGI — YOUTUBE CANDIDATE REVIEW")
print("=" * 80)
print("Total candidates:", len(mujhse))

display(mujhse[[
    "video_id",
    "metadata_title",
    "channel_title_y",
    "duration",
    "view_count",
    "comment_count",
    "original_automated_category",
    "classification_reason",
    "description_y"
]].reset_index(drop=True))

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_05.csv"
)

df = pd.read_csv(FILE, encoding="utf-8-sig")

remaining_movies = [
    "Khakee",
    "Krrish",
    "Rang De Basanti",
    "Fanaa",
    "Om Shanti Om",
    "Sivaji: The Boss",
    "Welcome"
]

review = df[df["movie_title"].isin(remaining_movies)].copy()

columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "channel_title_y",
    "duration",
    "view_count",
    "comment_count",
    "original_automated_category",
    "classification_reason",
    "description_y"
]

print("=" * 90)
print("BATCH 05 — CONSOLIDATED MANUAL REVIEW")
print("=" * 90)
print("Movies:", review["movie_title"].nunique())
print("Candidates:", len(review))
print("Unique video IDs:", review["video_id"].nunique())

for movie in remaining_movies:
    movie_df = review[review["movie_title"].eq(movie)]

    print("\n" + "=" * 90)
    print(f"{movie.upper()} — {len(movie_df)} CANDIDATES")
    print("=" * 90)

    display(movie_df[columns].reset_index(drop=True))

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_05.csv"
)

df = pd.read_csv(FILE, encoding="utf-8-sig")

# Ensure manual-review fields support text
text_columns = [
    "manual_decision",
    "manual_source_type",
    "manual_verification_status",
    "manual_review_notes"
]

for col in text_columns:
    df[col] = df[col].astype("string")

# Seven provisional selections based on metadata
approved_candidates = {
    "x1Pm7ml-k20": (
        "Khakee",
        "Movie Review",
        "Metadata Reviewed",
        "Selected as a promising long-form film discussion."
    ),
    "ATe8vq4Gxh4": (
        "Krrish",
        "Movie Review",
        "Metadata Reviewed",
        "Selected as a standalone retrospective film review."
    ),
    "2paYFBFS7nk": (
        "Rang De Basanti",
        "Movie Review",
        "Metadata Reviewed",
        "Selected as a promising long-form film discussion."
    ),
    "22kbJFofWuk": (
        "Fanaa",
        "Movie Review",
        "Metadata Reviewed",
        "Selected as a promising long-form film discussion."
    ),
    "H9MB16OALS0": (
        "Om Shanti Om",
        "Movie Review",
        "Metadata Reviewed",
        "Selected as a long-form film discussion."
    ),
    "2ZgQKzhErzk": (
        "Sivaji: The Boss",
        "Movie Review",
        "Metadata Reviewed",
        "Selected as a Tamil film review."
    ),
    "0gZBvsbqiG8": (
        "Welcome",
        "Movie Review",
        "Metadata Reviewed",
        "Selected as a promising long-form film discussion."
    )
}

# Validate all seven selections before changing anything
for video_id, details in approved_candidates.items():
    mask = df["video_id"].astype(str).eq(video_id)

    if mask.sum() != 1:
        raise ValueError(
            f"Expected exactly one row for {video_id}; found {mask.sum()}."
        )

    actual_movie = df.loc[mask, "movie_title"].iloc[0]

    if actual_movie != details[0]:
        raise ValueError(
            f"Movie mismatch for {video_id}: {actual_movie}"
        )

# Apply the seven provisional selections
for video_id, details in approved_candidates.items():
    mask = df["video_id"].astype(str).eq(video_id)

    df.loc[mask, text_columns] = [
        "Include",
        details[1],
        details[2],
        details[3]
    ]

# Record no suitable source for Mujhse Shaadi Karogi
mujhse_mask = df["movie_title"].eq("Mujhse Shaadi Karogi")

df.loc[mujhse_mask, "manual_decision"] = "Exclude"
df.loc[mujhse_mask, "manual_source_type"] = "Not Suitable"
df.loc[mujhse_mask, "manual_verification_status"] = "Metadata Reviewed"
df.loc[mujhse_mask, "manual_review_notes"] = (
    "No suitable primary audience review/reaction source "
    "identified among the screened candidates."
)

# Confirm Hero and Dhoom decisions remain intact
required_existing = {
    "RD9WGhsGi_U": "Include",
    "6Yqx7HiM3WM": "Include"
}

for video_id, expected in required_existing.items():
    actual = df.loc[
        df["video_id"].astype(str).eq(video_id),
        "manual_decision"
    ]

    if len(actual) != 1 or actual.iloc[0] != expected:
        raise ValueError(
            f"Existing decision check failed for {video_id}."
        )

# Save the updated manual-review file
df.to_csv(FILE, index=False, encoding="utf-8-sig")

print("BATCH 05 MANUAL REVIEW UPDATED")
print("=" * 65)
print("Total candidate rows:", len(df))
print("Selected seven new provisional sources:", len(approved_candidates))
print("Mujhse Shaadi Karogi candidates excluded:", int(mujhse_mask.sum()))

# Display all included sources
included = df[df["manual_decision"].eq("Include")]

display(included[[
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "channel_title_y",
    "manual_decision",
    "manual_source_type",
    "manual_verification_status"
]].reset_index(drop=True))

# Movie-level decision summary
summary = (
    df.groupby("movie_title")
    .agg(
        total_candidates=("video_id", "count"),
        included=("manual_decision", lambda x: (x == "Include").sum()),
        excluded=("manual_decision", lambda x: (x == "Exclude").sum()),
        pending=("manual_decision", lambda x: x.isna().sum() + (x == "Pending").sum())
    )
    .reset_index()
)

display(summary)

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

REVIEW_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_05.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_05.csv"
)

df = pd.read_csv(REVIEW_FILE, encoding="utf-8-sig")

approved = df[
    df["manual_decision"].eq("Include")
].copy()

# Validate selected records
assert len(approved) == 9, (
    f"Expected 9 selected sources, found {len(approved)}"
)

assert approved["video_id"].nunique() == 9, (
    "Duplicate video IDs detected."
)

assert approved["movie_record_id"].nunique() == 9, (
    "Expected 9 distinct movie records."
)

assert approved["movie_record_id"].notna().all()
assert approved["movie_title"].notna().all()

# Prepare manifest
manifest = approved[[
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "metadata_title",
    "channel_title_y",
    "duration",
    "view_count",
    "comment_count",
    "manual_source_type",
    "manual_verification_status",
    "manual_review_notes"
]].copy()

manifest = manifest.rename(columns={
    "metadata_title": "video_title",
    "channel_title_y": "channel_title"
})

manifest["acquisition_status"] = "Pending"
manifest["target_comments"] = 100

# Save only after all validations pass
manifest.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("BATCH 05 SOURCE MANIFEST VALIDATION")
print("=" * 65)
print("Selected sources:", len(manifest))
print("Unique video IDs:", manifest["video_id"].nunique())
print("Distinct movie records:", manifest["movie_record_id"].nunique())
print("Maximum comment target:", manifest["target_comments"].sum())
print("Manifest saved:", OUTPUT_FILE.exists())

display(manifest[[
    "movie_record_id",
    "movie_title",
    "video_id",
    "channel_title",
    "manual_verification_status",
    "target_comments"
]])

In [ ]:
import pandas as pd
import requests
import time
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata
from IPython.display import display

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

AUDIENCE_ROOT = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

RAW_COMMENT_ROOT = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

MANIFEST_FILE = (
    AUDIENCE_ROOT
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_05.csv"
)

RAW_OUTPUT = (
    RAW_COMMENT_ROOT
    / "ICMB_130_YOUTUBE_COMMENT_BATCH_05.csv"
)

LOG_OUTPUT = (
    AUDIENCE_ROOT
    / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_05.csv"
)

CHECKPOINT_FILE = (
    RAW_COMMENT_ROOT
    / "ICMB_130_YOUTUBE_COMMENT_BATCH_05_CHECKPOINT.csv"
)

RAW_COMMENT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. SAFETY CHECKS
# ============================================================

if not MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"Approved manifest not found: {MANIFEST_FILE}"
    )

if RAW_OUTPUT.exists() or LOG_OUTPUT.exists():
    raise FileExistsError(
        "A final Batch 05 output already exists. "
        "No files or API requests were made. "
        "Inspect the existing outputs before proceeding."
    )

manifest = pd.read_csv(
    MANIFEST_FILE,
    encoding="utf-8-sig"
)

required_columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "channel_title",
    "target_comments"
]

missing_columns = [
    col for col in required_columns
    if col not in manifest.columns
]

if missing_columns:
    raise ValueError(
        f"Manifest is missing columns: {missing_columns}"
    )

if len(manifest) != 9:
    raise ValueError(
        f"Expected 9 approved sources, found {len(manifest)}"
    )

if manifest["video_id"].nunique() != 9:
    raise ValueError("Duplicate video IDs in approved manifest.")

if CHECKPOINT_FILE.exists():
    checkpoint = pd.read_csv(
        CHECKPOINT_FILE,
        encoding="utf-8-sig"
    )
    completed_ids = set(
        checkpoint["video_id"].astype(str).unique()
    )
    print(
        "Existing checkpoint found. Completed videos will be skipped:",
        len(completed_ids)
    )
else:
    checkpoint = pd.DataFrame()
    completed_ids = set()

# ============================================================
# 3. LOAD API KEY FROM COLAB SECRET
# ============================================================

API_KEY = userdata.get("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError(
        "YOUTUBE_API_KEY was not found in Colab Secrets."
    )

API_URL = (
    "https://www.googleapis.com/youtube/v3/commentThreads"
)

# ============================================================
# 4. COMMENT ACQUISITION
# ============================================================

acquisition_log = []

for index, row in manifest.iterrows():

    video_id = str(row["video_id"])

    if video_id in completed_ids:
        print(
            f"Skipping already checkpointed video: "
            f"{row['movie_title']} ({video_id})"
        )
        continue

    target = int(row["target_comments"])

    print("\n" + "=" * 65)
    print(f"Movie: {row['movie_title']}")
    print(f"Video ID: {video_id}")
    print(f"Target: {target}")
    print("=" * 65)

    request_time = datetime.now(
        timezone.utc
    ).isoformat()

    video_comments = []
    status = "Success"
    error_message = ""

    try:
        params = {
            "part": "snippet",
            "videoId": video_id,
            "maxResults": min(target, 100),
            "order": "time",
            "textFormat": "plainText",
            "key": API_KEY
        }

        response = requests.get(
            API_URL,
            params=params,
            timeout=60
        )

        data = response.json()

        if response.status_code != 200:

            status = "API Error"

            error_message = (
                data.get("error", {})
                .get("message", "Unknown API error")
            )

        else:

            for item in data.get("items", []):

                snippet = (
                    item.get("snippet", {})
                    .get("topLevelComment", {})
                    .get("snippet", {})
                )

                comment = {
                    "movie_record_id": row["movie_record_id"],
                    "movie_title": row["movie_title"],
                    "video_id": video_id,
                    "video_title": row.get("video_title", ""),
                    "channel_title": row["channel_title"],
                    "comment_id": (
                        item.get("snippet", {})
                        .get("topLevelComment", {})
                        .get("id", "")
                    ),
                    "author_display_name": (
                        snippet.get("authorDisplayName", "")
                    ),
                    "comment_text": (
                        snippet.get("textDisplay", "")
                    ),
                    "like_count": snippet.get("likeCount", 0),
                    "published_at": snippet.get("publishedAt", ""),
                    "updated_at": snippet.get("updatedAt", ""),
                    "total_reply_count": (
                        item.get("snippet", {})
                        .get("totalReplyCount", 0)
                    ),
                    "comment_source": "YouTube Top-Level Comment",
                    "acquisition_order": "time",
                    "acquisition_timestamp_utc": request_time
                }

                video_comments.append(comment)

    except Exception as e:

        status = "Request Error"
        error_message = str(e)

    # ========================================================
    # 5. SAVE CHECKPOINT AFTER EACH VIDEO
    # ========================================================

    if video_comments:

        new_comments = pd.DataFrame(video_comments)

        if checkpoint.empty:
            checkpoint = new_comments.copy()
        else:
            checkpoint = pd.concat(
                [checkpoint, new_comments],
                ignore_index=True
            )

    # A checkpoint marker is recorded even for zero-comment
    # or API-error videos to prevent accidental repeated calls.
    checkpoint_marker = pd.DataFrame([{
        "movie_record_id": row["movie_record_id"],
        "movie_title": row["movie_title"],
        "video_id": video_id,
        "video_title": row.get("video_title", ""),
        "channel_title": row["channel_title"],
        "comment_id": f"__CHECKPOINT__{video_id}",
        "author_display_name": "",
        "comment_text": "",
        "like_count": 0,
        "published_at": "",
        "updated_at": "",
        "total_reply_count": 0,
        "comment_source": "__CHECKPOINT__",
        "acquisition_order": "time",
        "acquisition_timestamp_utc": request_time
    }])

    if not checkpoint.empty:
        checkpoint = pd.concat(
            [checkpoint, checkpoint_marker],
            ignore_index=True
        )
    else:
        checkpoint = checkpoint_marker.copy()

    checkpoint.to_csv(
        CHECKPOINT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    actual_count = len(video_comments)

    acquisition_log.append({
        "movie_record_id": row["movie_record_id"],
        "movie_title": row["movie_title"],
        "video_id": video_id,
        "channel_title": row["channel_title"],
        "target_comments": target,
        "actual_comments": actual_count,
        "acquisition_status": status,
        "error_message": error_message,
        "request_timestamp_utc": request_time
    })

    print("Status:", status)
    print("Comments retrieved:", actual_count)

    if error_message:
        print("Error:", error_message)

    time.sleep(0.5)

# ============================================================
# 6. FINALIZE RAW COMMENTS AND ACQUISITION LOG
# ============================================================

if checkpoint.empty:
    raise RuntimeError("No acquisition checkpoint was created.")

# Remove checkpoint marker rows from actual comment dataset
raw_comments = checkpoint[
    ~checkpoint["comment_source"].eq("__CHECKPOINT__")
].copy()

raw_comments = raw_comments.drop_duplicates(
    subset=["comment_id"],
    keep="first"
)

# Save final raw comment file
raw_comments.to_csv(
    RAW_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

# Include existing completed videos in the final log
log_records = []

for _, row in manifest.iterrows():

    video_id = str(row["video_id"])

    if video_id in completed_ids:

        previous = checkpoint[
            checkpoint["video_id"].astype(str).eq(video_id)
        ]

        actual = previous[
            ~previous["comment_source"].eq("__CHECKPOINT__")
        ]

        log_records.append({
            "movie_record_id": row["movie_record_id"],
            "movie_title": row["movie_title"],
            "video_id": video_id,
            "channel_title": row["channel_title"],
            "target_comments": int(row["target_comments"]),
            "actual_comments": len(actual),
            "acquisition_status": "Previously Checkpointed",
            "error_message": "",
            "request_timestamp_utc": ""
        })

for record in acquisition_log:
    log_records.append(record)

final_log = pd.DataFrame(log_records)

final_log.to_csv(
    LOG_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 7. FINAL REPORT
# ============================================================

print("\n" + "=" * 75)
print("BATCH 05 — YOUTUBE COMMENT ACQUISITION SUMMARY")
print("=" * 75)

print("Approved videos:", len(manifest))
print("Retrieved comment rows:", len(raw_comments))
print(
    "Unique comment IDs:",
    raw_comments["comment_id"].nunique()
)
print("Maximum target:", int(manifest["target_comments"].sum()))
print("Actual total:", len(raw_comments))

print("\nAcquisition status:")
display(
    final_log[[
        "movie_title",
        "video_id",
        "target_comments",
        "actual_comments",
        "acquisition_status"
    ]]
)

print("\nSaved files:")
print("Raw comments:", RAW_OUTPUT)
print("Acquisition log:", LOG_OUTPUT)
print("Checkpoint:", CHECKPOINT_FILE)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# ============================================================
# BATCH 05 — COMMENT QUALITY AUDIT
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_FILE = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments/"
    / "ICMB_130_YOUTUBE_COMMENT_BATCH_05.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_FILE = (
    OUTPUT_DIR / "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_05.csv"
)

MOVIE_FILE = (
    OUTPUT_DIR / "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_05.csv"
)

if not RAW_FILE.exists():
    raise FileNotFoundError(RAW_FILE)

df = pd.read_csv(RAW_FILE, encoding="utf-8-sig")

required = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "comment_id",
    "comment_text"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Work on a copy. Raw file is not modified.
audit = df.copy()

audit["comment_text"] = (
    audit["comment_text"].fillna("").astype(str)
)

audit["char_count"] = audit["comment_text"].str.len()

audit["word_count"] = (
    audit["comment_text"]
    .str.split()
    .str.len()
)

audit["is_empty"] = audit["comment_text"].str.strip().eq("")

audit["is_very_short"] = (
    (audit["char_count"] <= 10) |
    (audit["word_count"] <= 2)
)

audit["contains_url"] = audit["comment_text"].str.contains(
    r"https?://|www\.",
    case=False,
    regex=True
)

audit["length_category"] = pd.cut(
    audit["word_count"],
    bins=[-1, 2, 5, 20, np.inf],
    labels=[
        "Very Short",
        "Short",
        "Medium",
        "Long"
    ]
)

# ============================================================
# SCRIPT DISTRIBUTION — NOT LANGUAGE IDENTIFICATION
# ============================================================

def classify_script(text):

    text = str(text)

    devanagari = any(
        "\u0900" <= ch <= "\u097F" for ch in text
    )

    tamil = any(
        "\u0B80" <= ch <= "\u0BFF" for ch in text
    )

    bengali = any(
        "\u0980" <= ch <= "\u09FF" for ch in text
    )

    latin = any(
        ("A" <= ch <= "Z") or ("a" <= ch <= "z")
        for ch in text
    )

    if sum([devanagari, tamil, bengali, latin]) > 1:
        return "Mixed"

    if devanagari:
        return "Devanagari"

    if tamil:
        return "Tamil"

    if bengali:
        return "Bengali"

    if latin:
        return "Latin"

    return "Other/Emoji/Symbols"

audit["script_category"] = audit["comment_text"].apply(
    classify_script
)

# ============================================================
# DUPLICATE ANALYSIS
# ============================================================

audit["normalized_text"] = (
    audit["comment_text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

nonempty = audit[audit["normalized_text"].ne("")].copy()

duplicate_text = nonempty[
    nonempty.duplicated(
        subset=["normalized_text"],
        keep=False
    )
]

within_video_duplicates = duplicate_text[
    duplicate_text.duplicated(
        subset=["video_id", "normalized_text"],
        keep=False
    )
]

cross_video_duplicates = duplicate_text[
    duplicate_text.groupby("normalized_text")["video_id"]
    .transform("nunique") > 1
]

# ============================================================
# OVERALL QUALITY PROFILE
# ============================================================

profile = pd.DataFrame({
    "metric": [
        "Total comment records",
        "Unique comment IDs",
        "Duplicate comment IDs",
        "Empty comments",
        "Very short comments",
        "Comments containing URLs",
        "Duplicate-text records",
        "Within-video duplicate-text records",
        "Cross-video duplicate-text records",
        "Average word count",
        "Median word count",
        "Average character count",
        "Videos represented",
        "Movies represented"
    ],
    "value": [
        len(audit),
        audit["comment_id"].nunique(),
        audit["comment_id"].duplicated().sum(),
        audit["is_empty"].sum(),
        audit["is_very_short"].sum(),
        audit["contains_url"].sum(),
        len(duplicate_text),
        len(within_video_duplicates),
        len(cross_video_duplicates),
        round(audit["word_count"].mean(), 2),
        round(audit["word_count"].median(), 2),
        round(audit["char_count"].mean(), 2),
        audit["video_id"].nunique(),
        audit["movie_record_id"].nunique()
    ]
})

# ============================================================
# MOVIE-WISE QUALITY PROFILE
# ============================================================

movie_quality = (
    audit.groupby(
        ["movie_record_id", "movie_title"],
        dropna=False
    )
    .agg(
        comment_count=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        empty_comments=("is_empty", "sum"),
        very_short_comments=("is_very_short", "sum"),
        url_comments=("contains_url", "sum"),
        average_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        average_characters=("char_count", "mean")
    )
    .reset_index()
)

movie_quality["average_words"] = (
    movie_quality["average_words"].round(2)
)

movie_quality["average_characters"] = (
    movie_quality["average_characters"].round(2)
)

# ============================================================
# SAVE REPORTS
# ============================================================

profile.to_csv(
    PROFILE_FILE,
    index=False,
    encoding="utf-8-sig"
)

movie_quality.to_csv(
    MOVIE_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 70)
print("BATCH 05 — COMMENT QUALITY AUDIT")
print("=" * 70)

display(profile)

print("\nMOVIE-WISE QUALITY")
display(movie_quality)

print("\nSCRIPT DISTRIBUTION")
display(
    audit["script_category"]
    .value_counts()
    .rename_axis("script_category")
    .reset_index(name="count")
)

print("\nCOMMENT LENGTH DISTRIBUTION")
display(
    audit["length_category"]
    .value_counts()
    .rename_axis("length_category")
    .reset_index(name="count")
)

print("\nREPORTS SAVED")
print(PROFILE_FILE)
print(MOVIE_FILE)

print("\nRaw comments preserved:", RAW_FILE.exists())

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_YOUTUBE_ROOT = (
    PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"
)

COMMENT_ROOT = RAW_YOUTUBE_ROOT / "comments"

# ============================================================
# 2. EXPECTED BATCH FILES
# ============================================================

expected_files = {
    "Batch 01": "ICMB_130_YOUTUBE_COMMENT_BATCH_01.csv",
    "Batch 02": "ICMB_130_YOUTUBE_COMMENT_BATCH_02.csv",
    "Batch 03": "ICMB_130_YOUTUBE_COMMENT_BATCH_03.csv",
    "Batch 04": "ICMB_130_YOUTUBE_COMMENT_BATCH_04.csv",
    "Batch 05": "ICMB_130_YOUTUBE_COMMENT_BATCH_05.csv"
}

print("=" * 75)
print("ICMB-130 — RAW YOUTUBE COMMENT CONSOLIDATION PREFLIGHT")
print("=" * 75)

# ============================================================
# 3. LOCATE PILOT COMMENT FILE
# ============================================================

pilot_candidates = []

for file in RAW_YOUTUBE_ROOT.rglob("*.csv"):

    name = file.name.upper()

    if (
        "COMMENT" in name
        and "PILOT" in name
        and "CHECKPOINT" not in name
        and "QUALITY" not in name
        and "PROFILE" not in name
    ):
        pilot_candidates.append(file)

print("\nPILOT COMMENT FILE SEARCH")

if pilot_candidates:
    for file in pilot_candidates:
        print("Found:", file)
else:
    print("No filename explicitly containing COMMENT and PILOT.")

# ============================================================
# 4. INSPECT EXPECTED BATCH FILES
# ============================================================

files_to_inspect = {}

for batch_name, filename in expected_files.items():

    file_path = COMMENT_ROOT / filename

    if file_path.exists():
        files_to_inspect[batch_name] = file_path
    else:
        print(f"\nMISSING: {batch_name} — {filename}")

# Add pilot candidates for inspection
for i, file_path in enumerate(pilot_candidates, start=1):
    files_to_inspect[f"Pilot candidate {i}"] = file_path

# ============================================================
# 5. DISPLAY SHAPES, COLUMNS AND BASIC VALIDATION
# ============================================================

inspection_results = []

for label, file_path in files_to_inspect.items():

    data = pd.read_csv(
        file_path,
        encoding="utf-8-sig",
        low_memory=False
    )

    inspection_results.append({
        "source": label,
        "filename": file_path.name,
        "rows": len(data),
        "columns": len(data.columns),
        "unique_comment_ids": (
            data["comment_id"].nunique()
            if "comment_id" in data.columns else None
        ),
        "has_comment_text": "comment_text" in data.columns,
        "has_video_id": "video_id" in data.columns,
        "path": str(file_path)
    })

    print("\n" + "-" * 75)
    print(label)
    print("File:", file_path.name)
    print("Shape:", data.shape)
    print("Columns:")
    print(data.columns.tolist())

display(pd.DataFrame(inspection_results))

print("\nPREFLIGHT COMPLETE")
print("No source files were modified.")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

OUTPUT_FILES = [
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_05.csv",
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_05.csv",
    "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_PILOT_BATCH_01_05.csv"
]

print("=" * 70)
print("CONSOLIDATION OUTPUT RECOVERY CHECK")
print("=" * 70)

for filename in OUTPUT_FILES:
    path = PROCESSED_ROOT / filename
    print(f"\n{filename}")
    print("Exists:", path.exists())

    if path.exists():
        print("Size (KB):", round(path.stat().st_size / 1024, 2))

# Check consolidated comments
CONSOLIDATED_FILE = PROCESSED_ROOT / OUTPUT_FILES[0]

if CONSOLIDATED_FILE.exists():

    combined = pd.read_csv(
        CONSOLIDATED_FILE,
        encoding="utf-8-sig",
        low_memory=False
    )

    print("\nCONSOLIDATED DATASET")
    print("Shape:", combined.shape)
    print("Unique comment IDs:", combined["comment_id"].nunique())
    print("Movies:", combined["movie_record_id"].nunique())

else:
    print("\nConsolidated dataset was not saved.")
    print("We will rebuild the reports from the six original raw files.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENT_ROOT = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

PROCESSED_ROOT = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment"
)

REFERENCE_FILE = (
    PROCESSED_ROOT
    / "ICMB_130_AUDIENCE_MOVIE_REFERENCE_VALIDATED.csv"
)

OUTPUT_FILE = (
    PROCESSED_ROOT
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_05.csv"
)

COVERAGE_FILE = (
    PROCESSED_ROOT
    / "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_05.csv"
)

DUPLICATE_FILE = (
    PROCESSED_ROOT
    / "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_PILOT_BATCH_01_05.csv"
)

# ============================================================
# 2. SOURCE FILES
# ============================================================

source_files = {
    "Pilot": "ICMB_130_YOUTUBE_COMMENT_PILOT.csv",
    "Batch 01": "ICMB_130_YOUTUBE_COMMENT_BATCH_01.csv",
    "Batch 02": "ICMB_130_YOUTUBE_COMMENT_BATCH_02.csv",
    "Batch 03": "ICMB_130_YOUTUBE_COMMENT_BATCH_03.csv",
    "Batch 04": "ICMB_130_YOUTUBE_COMMENT_BATCH_04.csv",
    "Batch 05": "ICMB_130_YOUTUBE_COMMENT_BATCH_05.csv"
}

if not REFERENCE_FILE.exists():
    raise FileNotFoundError(REFERENCE_FILE)

# Do not overwrite existing outputs.
for path in [OUTPUT_FILE, COVERAGE_FILE, DUPLICATE_FILE]:
    if path.exists():
        raise FileExistsError(
            f"Output already exists. Inspect before replacing: {path}"
        )

# ============================================================
# 3. LOAD AND STANDARDIZE
# ============================================================

frames = []

for batch_name, filename in source_files.items():

    path = COMMENT_ROOT / filename

    if not path.exists():
        raise FileNotFoundError(path)

    data = pd.read_csv(
        path,
        encoding="utf-8-sig",
        low_memory=False
    )

    required = [
        "movie_record_id",
        "movie_title",
        "video_id",
        "comment_id",
        "comment_text"
    ]

    missing = [c for c in required if c not in data.columns]

    if missing:
        raise ValueError(
            f"{batch_name} missing required columns: {missing}"
        )

    data = data.copy()
    data["source_batch"] = batch_name

    # Normalize common fields
    for col in ["movie_record_id", "video_id", "comment_id"]:
        data[col] = data[col].astype("string").str.strip()

    data["comment_text"] = (
        data["comment_text"].fillna("").astype(str)
    )

    data["normalized_comment_text"] = (
        data["comment_text"]
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    frames.append(data)

    print(
        f"{batch_name}: {len(data)} records | "
        f"{data['comment_id'].nunique()} unique comment IDs"
    )

# ============================================================
# 4. CONSOLIDATE
# ============================================================

combined = pd.concat(
    frames,
    ignore_index=True,
    sort=False
)

# ============================================================
# 5. DUPLICATE COMMENT ID AUDIT
# ============================================================

duplicate_ids = combined[
    combined.duplicated(
        subset=["comment_id"],
        keep=False
    )
].copy()

# ============================================================
# 6. DUPLICATE TEXT AUDIT
# ============================================================

nonempty = combined[
    combined["normalized_comment_text"].ne("")
].copy()

duplicate_text = nonempty[
    nonempty.duplicated(
        subset=["normalized_comment_text"],
        keep=False
    )
].copy()

if not duplicate_text.empty:

    duplicate_text["video_count"] = (
        duplicate_text.groupby("normalized_comment_text")["video_id"]
        .transform("nunique")
    )

    duplicate_text["batch_count"] = (
        duplicate_text.groupby("normalized_comment_text")["source_batch"]
        .transform("nunique")
    )

    duplicate_text["duplicate_scope"] = np.where(
        duplicate_text["video_count"] > 1,
        "Across Videos",
        "Within Video"
    )

else:
    duplicate_text["video_count"] = pd.Series(dtype="int64")
    duplicate_text["batch_count"] = pd.Series(dtype="int64")
    duplicate_text["duplicate_scope"] = pd.Series(dtype="object")

# ============================================================
# 7. LOAD VALIDATED MOVIE REFERENCE
# ============================================================

reference = pd.read_csv(
    REFERENCE_FILE,
    encoding="utf-8-sig"
)

reference["movie_record_id"] = (
    reference["movie_record_id"].astype("string").str.strip()
)

# Use reference movie titles as the authoritative titles.
movie_reference = reference[
    ["movie_record_id", "movie_title", "year", "rank", "language"]
].copy()

# Avoid overlapping movie_title column during merge.
comment_counts = (
    combined.groupby("movie_record_id")
    .agg(
        comment_records=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        acquisition_batches=("source_batch", "nunique")
    )
    .reset_index()
)

coverage = movie_reference.merge(
    comment_counts,
    on="movie_record_id",
    how="left"
)

count_columns = [
    "comment_records",
    "unique_comment_ids",
    "unique_videos",
    "acquisition_batches"
]

coverage[count_columns] = (
    coverage[count_columns].fillna(0).astype(int)
)

coverage["coverage_status"] = np.where(
    coverage["comment_records"] > 0,
    "Comments Acquired",
    "No Comments Acquired"
)

# ============================================================
# 8. BATCH SUMMARY
# ============================================================

batch_summary = (
    combined.groupby("source_batch")
    .agg(
        comment_records=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        unique_movies=("movie_record_id", "nunique")
    )
    .reset_index()
)

# ============================================================
# 9. SAVE ALL THREE OUTPUTS BEFORE DISPLAY
# ============================================================

combined.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

coverage.to_csv(
    COVERAGE_FILE,
    index=False,
    encoding="utf-8-sig"
)

duplicate_text.to_csv(
    DUPLICATE_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 10. VERIFY SAVED OUTPUTS
# ============================================================

for path in [OUTPUT_FILE, COVERAGE_FILE, DUPLICATE_FILE]:

    if not path.exists():
        raise RuntimeError(f"Output save verification failed: {path}")

    print(
        f"Verified: {path.name} | "
        f"{round(path.stat().st_size / 1024, 2)} KB"
    )

# ============================================================
# 11. FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("ICMB-130 — CONSOLIDATION RECOVERY COMPLETE")
print("=" * 70)

print("Total source records:", len(combined))
print("Unique comment IDs:", combined["comment_id"].nunique())
print("Duplicate comment IDs:", len(duplicate_ids))
print("Unique videos:", combined["video_id"].nunique())
print("Movies with comments:", (coverage["comment_records"] > 0).sum())
print("Movies without comments:", (coverage["comment_records"] == 0).sum())
print("Duplicate-text records:", len(duplicate_text))

print("\nBATCH SUMMARY")
display(batch_summary)

print("\nDUPLICATE TEXT SCOPE")
if not duplicate_text.empty:
    display(
        duplicate_text["duplicate_scope"]
        .value_counts()
        .rename_axis("duplicate_scope")
        .reset_index(name="records")
    )

print("\nMOVIES WITHOUT COMMENTS")
display(
    coverage.loc[
        coverage["comment_records"] == 0,
        ["movie_record_id", "movie_title", "year", "language"]
    ]
)

print("\nMOVIES WITH FEWER THAN 50 COMMENTS")
display(
    coverage.loc[
        (coverage["comment_records"] > 0) &
        (coverage["comment_records"] < 50),
        ["movie_record_id", "movie_title", "comment_records"]
    ]
)

print("\nSAVED OUTPUTS")
print(OUTPUT_FILE)
print(COVERAGE_FILE)
print(DUPLICATE_FILE)

print("\nOriginal six raw comment files remain unchanged.")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

COVERAGE_FILE = (
    PROCESSED_ROOT
    / "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_05.csv"
)

OUTPUT_FILE = (
    PROCESSED_ROOT
    / "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT.csv"
)

SUMMARY_FILE = (
    PROCESSED_ROOT
    / "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY.csv"
)

if not COVERAGE_FILE.exists():
    raise FileNotFoundError(COVERAGE_FILE)

for path in [OUTPUT_FILE, SUMMARY_FILE]:
    if path.exists():
        raise FileExistsError(
            f"Output already exists; refusing to overwrite: {path}"
        )

# ============================================================
# 2. LOAD COVERAGE DATA
# ============================================================

coverage = pd.read_csv(
    COVERAGE_FILE,
    encoding="utf-8-sig"
)

required_columns = [
    "movie_record_id",
    "movie_title",
    "year",
    "rank",
    "language",
    "comment_records",
    "unique_videos",
    "acquisition_batches"
]

missing = [
    col for col in required_columns
    if col not in coverage.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

report = coverage.copy()

# ============================================================
# 3. CLASSIFY COMMENT COVERAGE
# ============================================================

def classify_coverage(count):

    if count == 0:
        return "No Comments"

    elif count < 50:
        return "Low (1–49)"

    elif count < 100:
        return "Moderate (50–99)"

    else:
        return "Higher (100+)"

report["coverage_category"] = (
    report["comment_records"].apply(classify_coverage)
)

# ============================================================
# 4. ASSIGN ACQUISITION PRIORITY
# ============================================================

def assign_priority(category):

    if category == "No Comments":
        return "Priority 1"

    elif category == "Low (1–49)":
        return "Priority 2"

    elif category == "Moderate (50–99)":
        return "Priority 3"

    return "Covered — Monitor"

report["acquisition_priority"] = (
    report["coverage_category"].apply(assign_priority)
)

# ============================================================
# 5. SET PROVISIONAL TARGETS
# ============================================================

# These are planning targets, not mandatory quotas.
# They do not imply that every movie will have suitable sources.

def assign_target(count):

    if count == 0:
        return 100

    elif count < 50:
        return 100 - count

    elif count < 100:
        return 100 - count

    return 0

report["additional_comment_target"] = (
    report["comment_records"].apply(assign_target)
)

# ============================================================
# 6. PRIORITIZE BY YEAR AND LANGUAGE
# ============================================================

report = report.sort_values(
    ["acquisition_priority", "year", "rank"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ============================================================
# 7. CATEGORY SUMMARY
# ============================================================

category_order = [
    "No Comments",
    "Low (1–49)",
    "Moderate (50–99)",
    "Higher (100+)"
]

category_summary = (
    report.groupby("coverage_category")
    .agg(
        movie_count=("movie_record_id", "count"),
        existing_comments=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reindex(category_order)
    .fillna(0)
    .reset_index()
)

# ============================================================
# 8. LANGUAGE-WISE SUMMARY
# ============================================================

language_summary = (
    report.groupby("language")
    .agg(
        total_movies=("movie_record_id", "count"),
        movies_with_comments=(
            "comment_records",
            lambda x: (x > 0).sum()
        ),
        movies_without_comments=(
            "comment_records",
            lambda x: (x == 0).sum()
        ),
        existing_comments=("comment_records", "sum")
    )
    .reset_index()
)

# ============================================================
# 9. SAVE REPORTS
# ============================================================

report.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

category_summary.to_csv(
    SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

print("=" * 75)
print("ICMB-130 — YOUTUBE AUDIENCE ACQUISITION PLANNING")
print("=" * 75)

print("\nOVERALL COVERAGE")
print("Total movies:", len(report))
print("Movies with comments:", (report["comment_records"] > 0).sum())
print("Movies without comments:", (report["comment_records"] == 0).sum())
print("Existing comment records:", report["comment_records"].sum())

print(
    "\nProvisional additional comment target:",
    report["additional_comment_target"].sum()
)

print("\nCOVERAGE CATEGORY SUMMARY")
display(category_summary)

print("\nLANGUAGE-WISE COVERAGE")
display(language_summary)

print("\nPRIORITY 1 — MOVIES WITHOUT COMMENTS")
display(
    report.loc[
        report["coverage_category"] == "No Comments",
        [
            "movie_record_id",
            "movie_title",
            "year",
            "language",
            "additional_comment_target"
        ]
    ]
)

print("\nPRIORITY 2 — LOW-COVERAGE MOVIES")
display(
    report.loc[
        report["coverage_category"] == "Low (1–49)",
        [
            "movie_record_id",
            "movie_title",
            "comment_records",
            "additional_comment_target"
        ]
    ]
)

print("\nPRIORITY 3 — MODERATE-COVERAGE MOVIES")
display(
    report.loc[
        report["coverage_category"] == "Moderate (50–99)",
        [
            "movie_record_id",
            "movie_title",
            "comment_records",
            "additional_comment_target"
        ]
    ]
)

print("\nSAVED REPORTS")
print(OUTPUT_FILE)
print(SUMMARY_FILE)

print("\nOriginal comment datasets remain unchanged.")

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

DATA_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

REPORT_FILE = (
    DATA_DIR / "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT.csv"
)

OUTPUT_FILE = (
    DATA_DIR / "ICMB_130_YOUTUBE_BATCH_06_MOVIE_ALLOCATION.csv"
)

if not REPORT_FILE.exists():
    raise FileNotFoundError(REPORT_FILE)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        "Batch 06 allocation already exists. "
        "Inspect it before creating another."
    )

report = pd.read_csv(
    REPORT_FILE,
    encoding="utf-8-sig"
)

# Three movies with low existing coverage
low_coverage = report[
    report["coverage_category"] == "Low (1–49)"
].copy()

if len(low_coverage) != 3:
    raise ValueError(
        f"Expected 3 low-coverage movies, found {len(low_coverage)}"
    )

# Uncovered non-Hindi movies
uncovered = report[
    (report["coverage_category"] == "No Comments") &
    (report["language"].isin(["Kannada", "Tamil", "Telugu"]))
].copy()

# Allocate 7 slots with language-diversity priorities
language_targets = {
    "Kannada": 2,
    "Tamil": 2,
    "Telugu": 3
}

selected_non_hindi = []

for language, target in language_targets.items():

    candidates = uncovered[
        uncovered["language"] == language
    ].sort_values(["year", "rank"])

    if len(candidates) < target:
        raise ValueError(
            f"Not enough uncovered {language} movies."
        )

    selected_non_hindi.append(
        candidates.head(target)
    )

selected_non_hindi = pd.concat(
    selected_non_hindi,
    ignore_index=True
)

batch06 = pd.concat(
    [low_coverage, selected_non_hindi],
    ignore_index=True
)

batch06 = batch06.sort_values(
    ["year", "rank"]
).reset_index(drop=True)

batch06["batch_id"] = "Batch 06"
batch06["allocation_reason"] = batch06.apply(
    lambda row:
        "Improve low existing coverage"
        if row["coverage_category"] == "Low (1–49)"
        else "Improve non-Hindi language coverage",
    axis=1
)

if len(batch06) != 10:
    raise ValueError("Batch 06 must contain exactly 10 movies.")

if batch06["movie_record_id"].nunique() != 10:
    raise ValueError("Duplicate movie allocations detected.")

batch06.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("BATCH 06 — MOVIE ALLOCATION")
print("=" * 70)

display(
    batch06[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "rank",
            "language",
            "comment_records",
            "allocation_reason"
        ]
    ]
)

print("\nLanguage allocation:")
display(
    batch06["language"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="movies")
)

print("\nTotal movies:", len(batch06))
print("Unique movie IDs:", batch06["movie_record_id"].nunique())
print("Saved:", OUTPUT_FILE)

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 06 YOUTUBE DISCOVERY
# 10 allocated movies | 1 search request per movie
# ============================================================

import os
import time
import pandas as pd
from pathlib import Path
from googleapiclient.discovery import build
from google.colab import userdata
from IPython.display import display

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

AUDIENCE_ROOT = (
    PROJECT_ROOT / "01_data" / "raw" / "audience_sentiment" / "youtube"
)

ALLOCATION_FILE = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment" /
    "ICMB_130_YOUTUBE_BATCH_06_MOVIE_ALLOCATION.csv"
)

OUTPUT_FILE = (
    AUDIENCE_ROOT /
    "ICMB_130_YOUTUBE_DISCOVERY_BATCH_06.csv"
)

AUDIENCE_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. CHECK INPUT AND PREVENT OVERWRITING
# ------------------------------------------------------------

assert ALLOCATION_FILE.exists(), (
    f"Allocation file not found: {ALLOCATION_FILE}"
)

assert not OUTPUT_FILE.exists(), (
    "Batch 06 discovery file already exists. "
    "Inspect it before creating another discovery run."
)

allocation = pd.read_csv(ALLOCATION_FILE)

required_columns = [
    "movie_record_id",
    "movie_title",
    "year",
    "rank",
    "language"
]

missing = [
    col for col in required_columns
    if col not in allocation.columns
]

assert not missing, f"Missing allocation columns: {missing}"
assert allocation["movie_record_id"].nunique() == 10

print("Allocation loaded:", allocation.shape)
display(allocation[required_columns])

# ------------------------------------------------------------
# 3. CONNECT TO YOUTUBE DATA API
# ------------------------------------------------------------

API_KEY = userdata.get("YOUTUBE_API_KEY")

assert API_KEY, "YOUTUBE_API_KEY not found in Colab Secrets."

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY
)

print("YouTube API client initialized.")

# ------------------------------------------------------------
# 4. SEARCH QUERY BUILDER
# ------------------------------------------------------------

def build_search_query(row):

    title = str(row["movie_title"]).strip()
    year = int(row["year"])
    language = str(row["language"]).strip()

    if language == "Hindi":
        return f'"{title}" {year} movie review reaction'

    return f'"{title}" {year} Indian movie review reaction'

# ------------------------------------------------------------
# 5. DISCOVERY — ONE SEARCH PER MOVIE
# ------------------------------------------------------------

records = []
request_log = []

for index, row in allocation.iterrows():

    movie_id = row["movie_record_id"]
    movie_title = row["movie_title"]
    query = build_search_query(row)

    print(
        f"\n[{index + 1}/10] Searching: "
        f"{movie_title} ({row['year']})"
    )

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=10,
            order="relevance"
        ).execute()

        items = response.get("items", [])

        for item in items:

            snippet = item.get("snippet", {})
            video_id = item.get("id", {}).get("videoId")

            if not video_id:
                continue

            records.append({
                "movie_record_id": movie_id,
                "movie_title": movie_title,
                "movie_year": int(row["year"]),
                "language": row["language"],
                "search_query": query,
                "video_id": video_id,
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "discovery_source": "YouTube Data API v3",
                "discovery_batch": "Batch 06"
            })

        request_log.append({
            "movie_record_id": movie_id,
            "movie_title": movie_title,
            "search_status": "Success",
            "candidates_returned": len(items),
            "error": None
        })

        print("Candidates returned:", len(items))

    except Exception as e:

        request_log.append({
            "movie_record_id": movie_id,
            "movie_title": movie_title,
            "search_status": "Failed",
            "candidates_returned": 0,
            "error": str(e)
        })

        print("Search failed:", str(e))

    time.sleep(0.5)

# ------------------------------------------------------------
# 6. CREATE AND SAVE DISCOVERY DATA
# ------------------------------------------------------------

discovery = pd.DataFrame(records)

if not discovery.empty:

    discovery = discovery.drop_duplicates(
        subset=["video_id"],
        keep="first"
    )

    discovery.to_csv(OUTPUT_FILE, index=False)

    print("\nDiscovery file saved:", OUTPUT_FILE)

else:
    print("\nNo candidates returned. No discovery file created.")

# ------------------------------------------------------------
# 7. SAVE REQUEST LOG
# ------------------------------------------------------------

request_log_df = pd.DataFrame(request_log)

LOG_FILE = (
    AUDIENCE_ROOT /
    "ICMB_130_YOUTUBE_DISCOVERY_LOG_BATCH_06.csv"
)

request_log_df.to_csv(LOG_FILE, index=False)

# ------------------------------------------------------------
# 8. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BATCH 06 — YOUTUBE DISCOVERY SUMMARY")
print("=" * 65)

print("Allocated movies:", len(allocation))
print("Search requests:", len(request_log_df))
print(
    "Successful requests:",
    (request_log_df["search_status"] == "Success").sum()
)
print(
    "Failed requests:",
    (request_log_df["search_status"] == "Failed").sum()
)
print("Unique candidate videos:", discovery["video_id"].nunique()
      if not discovery.empty else 0)
print("Movies with candidates:",
      discovery["movie_record_id"].nunique()
      if not discovery.empty else 0)

print("\nCandidates by movie:")

if not discovery.empty:
    display(
        discovery.groupby(
            ["movie_record_id", "movie_title", "language"],
            as_index=False
        ).agg(
            candidate_count=("video_id", "nunique")
        )
    )

print("\nRequest log:")
display(request_log_df)

print("\nDiscovery file:", OUTPUT_FILE)
print("Request log:", LOG_FILE)

In [ ]:
# ============================================================
# BATCH 06 — METADATA OUTPUT INSPECTION AND PREVIEW
# No API calls | No file modification
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

METADATA_FILE = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" /
    "ICMB_130_YOUTUBE_METADATA_BATCH_06.csv"
)

assert METADATA_FILE.exists(), "Saved metadata file not found."

merged = pd.read_csv(METADATA_FILE)

print("=" * 65)
print("BATCH 06 — SAVED METADATA INSPECTION")
print("=" * 65)

print("Shape:", merged.shape)
print("\nActual columns:")
for col in merged.columns:
    print("-", col)

print("\nMetadata status:")
display(merged["metadata_status"].value_counts().to_frame("records"))

# Identify actual channel and description columns
channel_columns = [
    col for col in merged.columns
    if "channel_title" in col.lower()
]

description_columns = [
    col for col in merged.columns
    if "description" in col.lower()
]

print("\nChannel columns:", channel_columns)
print("Description columns:", description_columns)

# Build preview using columns that actually exist
preferred_columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "channel_title",
    "channel_title_x",
    "channel_title_y",
    "duration",
    "view_count",
    "comment_count",
    "metadata_status"
]

preview_columns = [
    col for col in preferred_columns
    if col in merged.columns
]

display(merged[preview_columns].head(20))

print("\nCandidates by movie:")
display(
    merged.groupby(
        ["movie_record_id", "movie_title", "language"],
        as_index=False
    ).agg(
        candidate_count=("video_id", "nunique"),
        metadata_retrieved=(
            "metadata_status",
            lambda x: (x == "Retrieved").sum()
        )
    )
)

In [ ]:
# ============================================================
# BATCH 06 — METADATA-BASED CANDIDATE SCREENING
# No API calls | Original files remain unchanged
# ============================================================

import pandas as pd
import re
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

YOUTUBE_ROOT = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube"
)

INPUT_FILE = (
    YOUTUBE_ROOT / "ICMB_130_YOUTUBE_METADATA_BATCH_06.csv"
)

OUTPUT_FILE = (
    YOUTUBE_ROOT / "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_06.csv"
)

assert INPUT_FILE.exists(), "Metadata file not found."
assert not OUTPUT_FILE.exists(), (
    "Screening file already exists. Inspect it before rerunning."
)

df = pd.read_csv(INPUT_FILE)

# ------------------------------------------------------------
# 1. NORMALIZE METADATA FIELDS
# ------------------------------------------------------------

df["screening_title"] = (
    df["metadata_title"].fillna(df["video_title"]).fillna("").astype(str)
)

df["screening_description"] = (
    df["description_y"].fillna("").astype(str)
)

df["screening_channel"] = (
    df["channel_title_y"].fillna("").astype(str)
)

def duration_seconds(value):
    if pd.isna(value):
        return None

    match = re.fullmatch(
        r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?",
        str(value)
    )

    if not match:
        return None

    hours, minutes, seconds = match.groups()

    return (
        int(hours or 0) * 3600
        + int(minutes or 0) * 60
        + int(seconds or 0)
    )

df["duration_seconds"] = df["duration"].apply(duration_seconds)

# ------------------------------------------------------------
# 2. METADATA-BASED SOURCE TYPE SCREENING
# ------------------------------------------------------------

def classify_source(row):

    title = row["screening_title"].lower()
    description = row["screening_description"].lower()
    text = title + " " + description

    # Strong exclusion signals
    if re.search(r"\b(song|lyrical|jukebox|audio jukebox|video song)\b", title):
        return "Song", "Exclude", "Title indicates a song/music upload"

    if re.search(r"\b(shorts|status video|whatsapp status)\b", title):
        return "Short Clip", "Exclude", "Title indicates short-form content"

    if re.search(r"\b(scene|clip|best scenes|fight scene|comedy scene)\b", title):
        return "Clip or Scene", "Exclude", "Title indicates an isolated film scene"

    # Full movie
    if re.search(
        r"\b(full movie|full film|complete movie|watch full movie)\b",
        title
    ):
        return "Full Movie", "Hold", "Full-film upload; distinct source type"

    # Review
    if re.search(
        r"\b(review|rewatch|rewatched|retrospective|"
        r"aged well|film analysis|movie analysis|explained)\b",
        title
    ):
        return "Movie Review / Analysis", "Include", (
            "Title indicates review, retrospective or film analysis"
        )

    # Reaction
    if re.search(
        r"\b(reaction|reacts|reacting|watches|watching|"
        r"first time watching|part\s*\d+)\b",
        title
    ):
        return "Movie Reaction", "Include", (
            "Title indicates a movie reaction/watch-along"
        )

    # Possible relevance from description only
    if re.search(r"\b(review|reaction|analysis|rewatch)\b", description):
        return "Possible Review / Reaction", "Hold", (
            "Relevant terms appear in description, not clearly in title"
        )

    return "Unclassified", "Hold", (
        "Metadata does not clearly establish source type"
    )

screening = df.apply(
    classify_source,
    axis=1,
    result_type="expand"
)

screening.columns = [
    "provisional_source_type",
    "automated_recommendation",
    "screening_reason"
]

df = pd.concat([df, screening], axis=1)

# ------------------------------------------------------------
# 3. FLAG POSSIBLE MOVIE TITLE MISMATCHES
# ------------------------------------------------------------

def title_match_flag(row):

    movie = str(row["movie_title"]).lower()
    title = str(row["screening_title"]).lower()

    # Normalize common punctuation differences
    normalize = lambda x: re.sub(r"[^a-z0-9]+", " ", x).strip()

    movie_norm = normalize(movie)
    title_norm = normalize(title)

    # Use meaningful title tokens; ignore generic words
    stopwords = {
        "the", "a", "an", "of", "and", "movie", "film",
        "full", "review", "reaction", "part"
    }

    tokens = [
        token for token in movie_norm.split()
        if token not in stopwords and len(token) > 1
    ]

    if not tokens:
        return "Manual title check required"

    matched = sum(token in title_norm for token in tokens)

    if matched == len(tokens):
        return "Title terms matched"

    if matched > 0:
        return "Partial title match — inspect"

    return "Possible mismatch — inspect"

df["title_match_flag"] = df.apply(
    title_match_flag,
    axis=1
)

# ------------------------------------------------------------
# 4. DURATION FLAGS
# ------------------------------------------------------------

def duration_flag(seconds):

    if pd.isna(seconds):
        return "Duration unavailable"

    if seconds < 60:
        return "Under 1 minute"

    if seconds < 180:
        return "Under 3 minutes"

    if seconds < 300:
        return "Under 5 minutes"

    return "5 minutes or longer"

df["duration_flag"] = df["duration_seconds"].apply(duration_flag)

# ------------------------------------------------------------
# 5. ADD VIDEO LINKS AND REVIEW FIELDS
# ------------------------------------------------------------

df["video_url"] = (
    "https://www.youtube.com/watch?v=" + df["video_id"].astype(str)
)

df["metadata_review_status"] = "Pending"

df["final_decision"] = ""
df["final_source_type"] = ""
df["content_review_status"] = "Not reviewed"
df["reviewer_notes"] = ""

# ------------------------------------------------------------
# 6. SELECT REVIEW SHEET COLUMNS
# ------------------------------------------------------------

review_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "video_url",
    "screening_title",
    "screening_channel",
    "duration",
    "duration_flag",
    "view_count",
    "comment_count",
    "provisional_source_type",
    "automated_recommendation",
    "screening_reason",
    "title_match_flag",
    "metadata_review_status",
    "final_decision",
    "final_source_type",
    "content_review_status",
    "reviewer_notes",
    "published_at_y",
    "screening_description"
]

review_columns = [
    col for col in review_columns
    if col in df.columns
]

review = df[review_columns].copy()

# ------------------------------------------------------------
# 7. SAVE SCREENING SHEET
# ------------------------------------------------------------

review.to_csv(OUTPUT_FILE, index=False)

# ------------------------------------------------------------
# 8. SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("BATCH 06 — METADATA SCREENING SUMMARY")
print("=" * 65)

print("Total candidates:", len(review))
print("Unique video IDs:", review["video_id"].nunique())

print("\nAutomated recommendations:")
display(
    review["automated_recommendation"]
    .value_counts()
    .to_frame("candidate_count")
)

print("\nProvisional source types:")
display(
    review["provisional_source_type"]
    .value_counts()
    .to_frame("candidate_count")
)

print("\nScreening by movie:")
display(
    review.groupby(
        ["movie_record_id", "movie_title"],
        as_index=False
    ).agg(
        candidates=("video_id", "nunique"),
        include=("automated_recommendation",
                 lambda x: (x == "Include").sum()),
        hold=("automated_recommendation",
              lambda x: (x == "Hold").sum()),
        exclude=("automated_recommendation",
                 lambda x: (x == "Exclude").sum())
    )
)

print("\nReview sheet preview:")
display(
    review[[
        "movie_record_id",
        "movie_title",
        "screening_title",
        "screening_channel",
        "duration",
        "provisional_source_type",
        "automated_recommendation",
        "title_match_flag"
    ]].head(25)
)

print("\nSaved screening sheet:")
print(OUTPUT_FILE)

In [ ]:
# ============================================================
# BATCH 06 — FOCUSED CANDIDATE SHORTLIST
# No API calls | No changes to existing files
# ============================================================

import pandas as pd
import re
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

YOUTUBE_ROOT = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube"
)

INPUT_FILE = (
    YOUTUBE_ROOT /
    "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_06.csv"
)

OUTPUT_FILE = (
    YOUTUBE_ROOT /
    "ICMB_130_YOUTUBE_CANDIDATE_SHORTLIST_BATCH_06.csv"
)

assert INPUT_FILE.exists()
assert not OUTPUT_FILE.exists(), (
    "Shortlist already exists. Inspect it before recreating."
)

df = pd.read_csv(INPUT_FILE)

# ------------------------------------------------------------
# 1. CORRECT OBVIOUS AUTOMATED CLASSIFICATION ISSUES
# ------------------------------------------------------------

def corrected_type(row):

    title = str(row["screening_title"]).lower()

    # A review title must not be mistaken for a full movie.
    if re.search(r"\b(review|rewatch|retrospective|analysis)\b", title):
        return "Movie Review / Analysis"

    if re.search(r"\b(full movie|full film|complete movie)\b", title):
        return "Full Movie"

    if re.search(r"\b(song|jukebox|lyrical|video song)\b", title):
        return "Song"

    if re.search(r"\b(scene|clip|shorts|status video)\b", title):
        return "Clip or Scene"

    if re.search(
        r"\b(reaction|reacts|reacting|watches|watching)\b",
        title
    ):
        return "Movie Reaction"

    return row["provisional_source_type"]

df["corrected_source_type"] = df.apply(
    corrected_type,
    axis=1
)

# ------------------------------------------------------------
# 2. IDENTIFY POSSIBLE TITLE MISMATCHES
# ------------------------------------------------------------

def mismatch_flag(row):

    title = str(row["screening_title"]).lower()
    movie = str(row["movie_title"]).lower()

    if (
        row["movie_record_id"] == "2006_2"
        and "munna bhai m.b.b.s" in title
    ):
        return "Possible wrong movie — exclude pending verification"

    if "possible mismatch" in str(row["title_match_flag"]).lower():
        return "Possible mismatch — hold"

    return "No obvious mismatch from title"

df["mismatch_review"] = df.apply(
    mismatch_flag,
    axis=1
)

# ------------------------------------------------------------
# 3. SELECT SHORTLIST CANDIDATES
# ------------------------------------------------------------

def shortlist_decision(row):

    source_type = row["corrected_source_type"]
    mismatch = row["mismatch_review"]

    if mismatch.startswith("Possible wrong movie"):
        return "Exclude"

    if mismatch.startswith("Possible mismatch"):
        return "Hold"

    if source_type in [
        "Movie Reaction",
        "Movie Review / Analysis"
    ]:
        return "Shortlist"

    if source_type == "Possible Review / Reaction":
        return "Hold"

    return "Exclude"

df["shortlist_decision"] = df.apply(
    shortlist_decision,
    axis=1
)

shortlist = df[
    df["shortlist_decision"].isin(["Shortlist", "Hold"])
].copy()

# ------------------------------------------------------------
# 4. KEEP USEFUL REVIEW FIELDS
# ------------------------------------------------------------

columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "video_url",
    "screening_title",
    "screening_channel",
    "duration",
    "view_count",
    "comment_count",
    "corrected_source_type",
    "mismatch_review",
    "shortlist_decision",
    "screening_description"
]

columns = [col for col in columns if col in shortlist.columns]

shortlist = shortlist[columns].copy()

shortlist.to_csv(OUTPUT_FILE, index=False)

# ------------------------------------------------------------
# 5. SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("BATCH 06 — FOCUSED SHORTLIST SUMMARY")
print("=" * 65)

print("Original candidates:", len(df))
print("Shortlist candidates:", len(shortlist))

print("\nShortlist decisions:")
display(
    shortlist["shortlist_decision"]
    .value_counts()
    .to_frame("candidate_count")
)

print("\nSource types:")
display(
    shortlist["corrected_source_type"]
    .value_counts()
    .to_frame("candidate_count")
)

print("\nCandidates by movie:")
display(
    shortlist.groupby(
        ["movie_record_id", "movie_title"],
        as_index=False
    ).agg(
        shortlist_candidates=("video_id", "nunique"),
        hold_candidates=(
            "shortlist_decision",
            lambda x: (x == "Hold").sum()
        )
    )
)

print("\nFocused shortlist:")
display(
    shortlist[[
        "movie_record_id",
        "movie_title",
        "screening_title",
        "screening_channel",
        "duration",
        "corrected_source_type",
        "mismatch_review",
        "shortlist_decision"
    ]].to_string(index=False)
)

print("\nSaved:", OUTPUT_FILE)

In [ ]:
# ============================================================
# BATCH 06 — COMPLETE MOVIE-WISE SHORTLIST INSPECTION
# No API calls | Read-only
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

FILE = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" /
    "ICMB_130_YOUTUBE_CANDIDATE_SHORTLIST_BATCH_06.csv"
)

assert FILE.exists(), "Batch 06 shortlist not found."

df = pd.read_csv(FILE)

print("Total shortlist records:", len(df))
print("Unique videos:", df["video_id"].nunique())
print("Movies:", df["movie_record_id"].nunique())

for movie_id, group in df.groupby(
    ["movie_record_id", "movie_title"],
    sort=False
):

    print("\n" + "=" * 100)
    print(f"{movie_id[0]} — {movie_id[1]}")
    print("=" * 100)

    display(
        group[[
            "video_id",
            "screening_title",
            "screening_channel",
            "duration",
            "corrected_source_type",
            "mismatch_review",
            "shortlist_decision",
            "video_url"
        ]].reset_index(drop=True)
    )

print("\n" + "=" * 100)
print("HOLD CANDIDATES — SEPARATE REVIEW")
print("=" * 100)

display(
    df[df["shortlist_decision"] == "Hold"][[
        "movie_record_id",
        "movie_title",
        "video_id",
        "screening_title",
        "screening_channel",
        "duration",
        "corrected_source_type",
        "mismatch_review",
        "video_url"
    ]].reset_index(drop=True)
)

In [ ]:
# ============================================================
# NOTEBOOK 10 — LOCATE BATCH 05 AND BATCH 06 FILES
# Read-only diagnostic | No files modified
# ============================================================

from pathlib import Path

possible_roots = [
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"),
    Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics"),
]

print("=" * 70)
print("GOOGLE DRIVE AND PROJECT PATH CHECK")
print("=" * 70)

for root in possible_roots:
    print(f"\nRoot: {root}")
    print("Exists:", root.exists())

    if root.exists():
        print("Contents:", [p.name for p in root.iterdir()])

print("\n" + "=" * 70)
print("SEARCHING FOR REQUIRED FILES")
print("=" * 70)

search_names = [
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_05.csv",
    "ICMB_130_YOUTUBE_CANDIDATE_SHORTLIST_BATCH_06.csv"
]

for filename in search_names:
    print(f"\nSearching: {filename}")
    matches = []

    for root in possible_roots:
        if root.exists():
            matches.extend(root.rglob(filename))

    if matches:
        for path in matches:
            print("FOUND:", path)
    else:
        print("NOT FOUND in either project root.")

In [ ]:
# ============================================================
# BATCH 06 — APPROVED MANIFEST SCHEMA CHECK
# Read-only | No API calls | No files modified
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

BATCH05_FILE = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment" /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_05.csv"
)

BATCH06_SHORTLIST = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" /
    "ICMB_130_YOUTUBE_CANDIDATE_SHORTLIST_BATCH_06.csv"
)

batch05 = pd.read_csv(BATCH05_FILE)
batch06 = pd.read_csv(BATCH06_SHORTLIST)

print("=" * 70)
print("BATCH 05 — APPROVED MANIFEST STRUCTURE")
print("=" * 70)

print("Shape:", batch05.shape)
print("\nColumns:")
for col in batch05.columns:
    print("-", col)

display(batch05.head(3))

print("\n" + "=" * 70)
print("BATCH 06 — PROPOSED SOURCES")
print("=" * 70)

approved_ids = [
    "7UzmGhuK68I",
    "XKe3iqirm5s",
    "bAcy86e7fGE",
    "FfVuduRk1WI",
    "H6ADfnLI4jY",
    "wL-4paUE1LE",
    "K1lAL6Qvf88",
    "Y5qt_6pnFkU",
    "_8GRLpjduVU"
]

assert "video_id" in batch06.columns, (
    "Expected video_id column not found. "
    f"Available columns: {batch06.columns.tolist()}"
)

proposed = batch06[
    batch06["video_id"].astype(str).isin(approved_ids)
].copy()

print("Proposed sources found:", len(proposed))
print("Unique video IDs:", proposed["video_id"].nunique())

display(proposed)

print("\nDasavathaaram remains on hold.")

In [ ]:
# ============================================================
# BATCH 06 — CREATE APPROVED ACQUISITION MANIFEST
# 9 provisional sources | Dasavathaaram excluded
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

SHORTLIST_FILE = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" /
    "ICMB_130_YOUTUBE_CANDIDATE_SHORTLIST_BATCH_06.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment" /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_06.csv"
)

assert SHORTLIST_FILE.exists(), "Batch 06 shortlist not found."

# Approved provisional source IDs
approved_sources = {
    "7UzmGhuK68I": (
        "Movie Review / Analysis",
        "Selected for retrospective discussion of Kaante. "
        "Provisionally approved from metadata; content not reviewed."
    ),
    "XKe3iqirm5s": (
        "Movie Review",
        "Selected as a dedicated Lage Raho Munnabhai movie review. "
        "Provisionally approved from metadata; content not reviewed."
    ),
    "bAcy86e7fGE": (
        "Movie Reaction",
        "Selected as a Sivaji: The Boss revisit/reaction. "
        "Provisionally approved from metadata; content not reviewed."
    ),
    "FfVuduRk1WI": (
        "Movie Reaction",
        "Selected as Magadheera reaction Part 1. "
        "Provisionally approved from metadata; content not reviewed."
    ),
    "H6ADfnLI4jY": (
        "Movie Reaction",
        "Selected as Enthiran reaction Part 1. "
        "Provisionally approved from metadata; content not reviewed."
    ),
    "wL-4paUE1LE": (
        "Movie Reaction",
        "Selected as a long-form Baahubali: The Beginning reaction. "
        "Provisionally approved from metadata; content not reviewed."
    ),
    "K1lAL6Qvf88": (
        "Movie Review",
        "Selected as a Baahubali 2: The Conclusion movie review. "
        "Provisionally approved from metadata; content not reviewed."
    ),
    "Y5qt_6pnFkU": (
        "Movie Review",
        "Selected as a KGF Chapter 2 movie review. "
        "Provisionally approved from metadata; content not reviewed."
    ),
    "_8GRLpjduVU": (
        "Movie Review",
        "Selected as a Kantara movie review. "
        "Provisionally approved from metadata; content not reviewed."
    )
}

shortlist = pd.read_csv(SHORTLIST_FILE, dtype={"video_id": str})

assert "video_id" in shortlist.columns
assert shortlist["video_id"].is_unique, (
    "Duplicate video IDs found in shortlist. Resolve before proceeding."
)

selected = shortlist[
    shortlist["video_id"].isin(approved_sources.keys())
].copy()

# Confirm all nine selections are present
found_ids = set(selected["video_id"])
missing_ids = set(approved_sources) - found_ids

assert not missing_ids, f"Approved IDs missing from shortlist: {missing_ids}"
assert len(selected) == 9, f"Expected 9 sources, found {len(selected)}"

# Match Batch 05 manifest schema
manifest = pd.DataFrame({
    "movie_record_id": selected["movie_record_id"],
    "movie_title": selected["movie_title"],
    "movie_year": selected["movie_year"],
    "language": selected["language"],
    "video_id": selected["video_id"],
    "video_title": selected["screening_title"],
    "channel_title": selected["screening_channel"],
    "duration": selected["duration"],
    "view_count": selected["view_count"],
    "comment_count": selected["comment_count"],
    "manual_source_type": selected["video_id"].map(
        lambda x: approved_sources[x][0]
    ),
    "manual_verification_status": "Metadata Reviewed",
    "manual_review_notes": selected["video_id"].map(
        lambda x: approved_sources[x][1]
    ),
    "acquisition_status": "Pending",
    "target_comments": 100
})

# Preserve the exact Batch 05 column order
manifest = manifest[[
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "video_title",
    "channel_title",
    "duration",
    "view_count",
    "comment_count",
    "manual_source_type",
    "manual_verification_status",
    "manual_review_notes",
    "acquisition_status",
    "target_comments"
]].reset_index(drop=True)

# Prevent accidental overwrite
assert not OUTPUT_FILE.exists(), (
    "Batch 06 manifest already exists. "
    "Inspect it before deciding whether to replace it."
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
manifest.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# Read-back verification
verified = pd.read_csv(OUTPUT_FILE, dtype={"video_id": str})

print("=" * 70)
print("BATCH 06 APPROVED MANIFEST — VERIFICATION")
print("=" * 70)

print("Saved:", OUTPUT_FILE.exists())
print("Shape:", verified.shape)
print("Unique video IDs:", verified["video_id"].nunique())
print("Unique movies:", verified["movie_record_id"].nunique())
print("Total target comments:", verified["target_comments"].sum())
print("Acquisition status:", verified["acquisition_status"].value_counts().to_dict())
print("Verification status:", verified["manual_verification_status"].value_counts().to_dict())

print("\nSource allocation:")
display(verified[[
    "movie_record_id",
    "movie_title",
    "video_id",
    "manual_source_type",
    "manual_verification_status",
    "target_comments"
]])

print("\nDasavathaaram: ON HOLD — not included in the manifest.")
print("Content Reviewed: No — all nine selections are metadata-based.")

In [ ]:
# ============================================================
# BATCH 06 — COMMENT ACQUISITION PREFLIGHT
# Read-only | No API calls | No files modified
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_COMMENTS = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

PROCESSED = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

files_to_check = [
    RAW_COMMENTS / "ICMB_130_YOUTUBE_COMMENT_BATCH_06.csv",
    RAW_COMMENTS / "ICMB_130_YOUTUBE_COMMENT_BATCH_06_CHECKPOINT.csv",
    RAW_COMMENTS / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_06.csv",
    PROCESSED / "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_06.csv",
    PROCESSED / "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_06.csv"
]

print("=" * 70)
print("BATCH 06 — EXISTING FILE CHECK")
print("=" * 70)

for path in files_to_check:
    print(f"\n{path.name}")
    print("Exists:", path.exists())

    if path.exists():
        print("Size (KB):", round(path.stat().st_size / 1024, 2))

print("\n" + "=" * 70)
print("APPROVED MANIFEST CHECK")
print("=" * 70)

manifest_path = (
    PROCESSED /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_06.csv"
)

manifest = pd.read_csv(manifest_path, dtype={"video_id": str})

print("Sources:", len(manifest))
print("Target comments:", manifest["target_comments"].sum())
print("Pending sources:", (manifest["acquisition_status"] == "Pending").sum())

In [ ]:
# ============================================================
# BATCH 06 — YOUTUBE COMMENT ACQUISITION
# 9 approved sources | Target: up to 900 comments
# Checkpoint after each video
# ============================================================

import os
import time
import requests
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata
from IPython.display import display

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

RAW_ROOT = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube"
)

COMMENTS_ROOT = RAW_ROOT / "comments"
COMMENTS_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = (
    PROCESSED_ROOT /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_06.csv"
)

OUTPUT_FILE = (
    COMMENTS_ROOT /
    "ICMB_130_YOUTUBE_COMMENT_BATCH_06.csv"
)

CHECKPOINT_FILE = (
    COMMENTS_ROOT /
    "ICMB_130_YOUTUBE_COMMENT_BATCH_06_CHECKPOINT.csv"
)

LOG_FILE = (
    COMMENTS_ROOT /
    "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_06.csv"
)

# Prevent accidental overwrite or duplicate acquisition.
for path in [OUTPUT_FILE, CHECKPOINT_FILE, LOG_FILE]:
    assert not path.exists(), (
        f"Output already exists: {path.name}. "
        "Inspect it before proceeding."
    )

# ------------------------------------------------------------
# 2. LOAD MANIFEST AND API KEY
# ------------------------------------------------------------

manifest = pd.read_csv(
    MANIFEST_FILE,
    dtype={"video_id": str}
)

assert len(manifest) == 9
assert manifest["video_id"].nunique() == 9
assert (manifest["acquisition_status"] == "Pending").all()

API_KEY = userdata.get("YOUTUBE_API_KEY")

assert API_KEY, "YOUTUBE_API_KEY not found in Colab Secrets."

API_URL = "https://www.googleapis.com/youtube/v3/commentThreads"

# ------------------------------------------------------------
# 3. ACQUISITION FUNCTION
# ------------------------------------------------------------

all_comments = []
acquisition_logs = []

for index, source in manifest.iterrows():

    video_id = str(source["video_id"])
    target = int(source["target_comments"])

    print("\n" + "=" * 65)
    print(f"VIDEO {index + 1}/9: {source['movie_title']}")
    print("Video ID:", video_id)
    print("Target:", target)
    print("=" * 65)

    video_comments = []
    next_page_token = None
    request_count = 0
    error_message = None
    status = "Success"

    while len(video_comments) < target:

        params = {
            "part": "snippet",
            "videoId": video_id,
            "maxResults": min(100, target - len(video_comments)),
            "order": "time",
            "textFormat": "plainText",
            "key": API_KEY
        }

        if next_page_token:
            params["pageToken"] = next_page_token

        try:
            response = requests.get(
                API_URL,
                params=params,
                timeout=30
            )

            request_count += 1

            if response.status_code != 200:
                status = "API Error"
                error_message = (
                    f"HTTP {response.status_code}: "
                    f"{response.text[:500]}"
                )
                break

            data = response.json()
            items = data.get("items", [])

            if not items:
                status = "No More Comments"
                break

            for item in items:
                snippet = item["snippet"]["topLevelComment"]
                comment = snippet["snippet"]

                video_comments.append({
                    "movie_record_id": source["movie_record_id"],
                    "movie_title": source["movie_title"],
                    "movie_year": source["movie_year"],
                    "language": source["language"],
                    "video_id": video_id,
                    "video_title": source["video_title"],
                    "channel_title": source["channel_title"],
                    "comment_id": snippet["id"],
                    "comment_text": comment.get("textDisplay", ""),
                    "author_name": comment.get("authorDisplayName", ""),
                    "like_count": comment.get("likeCount", 0),
                    "published_at": comment.get("publishedAt"),
                    "updated_at": comment.get("updatedAt"),
                    "comment_position": len(video_comments) + 1,
                    "acquisition_batch": "Batch 06",
                    "retrieval_order": "time"
                })

            next_page_token = data.get("nextPageToken")

            if not next_page_token:
                status = "Completed"
                break

            time.sleep(0.2)

        except requests.exceptions.RequestException as exc:
            status = "Request Error"
            error_message = str(exc)
            break

    # Append this video's results.
    all_comments.extend(video_comments)

    actual_count = len(video_comments)

    acquisition_logs.append({
        "movie_record_id": source["movie_record_id"],
        "movie_title": source["movie_title"],
        "video_id": video_id,
        "target_comments": target,
        "comments_collected": actual_count,
        "request_count": request_count,
        "acquisition_status": status,
        "error_message": error_message,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat()
    })

    # --------------------------------------------------------
    # 4. SAVE CHECKPOINT AFTER EACH VIDEO
    # --------------------------------------------------------

    checkpoint_df = pd.DataFrame(all_comments)
    log_df = pd.DataFrame(acquisition_logs)

    checkpoint_df.to_csv(
        CHECKPOINT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    log_df.to_csv(
        LOG_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("Collected:", actual_count)
    print("Status:", status)
    print("Checkpoint saved.")

    if error_message:
        print("Error:", error_message)

# ------------------------------------------------------------
# 5. SAVE FINAL RAW COMMENT FILE
# ------------------------------------------------------------

comments_df = pd.DataFrame(all_comments)
logs_df = pd.DataFrame(acquisition_logs)

comments_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

logs_df.to_csv(
    LOG_FILE,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 6. FINAL ACQUISITION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BATCH 06 — COMMENT ACQUISITION SUMMARY")
print("=" * 70)

print("Videos processed:", len(logs_df))
print("Total target:", int(logs_df["target_comments"].sum()))
print("Total collected:", len(comments_df))
print("Unique comment IDs:", comments_df["comment_id"].nunique())
print("API/request errors:", logs_df["acquisition_status"].isin(
    ["API Error", "Request Error"]
).sum())

print("\nMovie-wise acquisition:")
display(logs_df)

print("\nSaved files:")
print("Raw comments:", OUTPUT_FILE)
print("Checkpoint:", CHECKPOINT_FILE)
print("Acquisition log:", LOG_FILE)

In [ ]:
# ============================================================
# BATCH 06 — COMMENT QUALITY AUDIT
# Read-only raw inputs | Creates two processed reports
# ============================================================

import pandas as pd
import re
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_ROOT = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

RAW_FILE = RAW_ROOT / "ICMB_130_YOUTUBE_COMMENT_BATCH_06.csv"
LOG_FILE = RAW_ROOT / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_06.csv"

PROFILE_FILE = (
    PROCESSED_ROOT /
    "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_06.csv"
)

MOVIE_FILE = (
    PROCESSED_ROOT /
    "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_06.csv"
)

assert RAW_FILE.exists(), "Raw Batch 06 comments not found."
assert LOG_FILE.exists(), "Batch 06 acquisition log not found."
assert not PROFILE_FILE.exists(), "Quality profile already exists."
assert not MOVIE_FILE.exists(), "Movie quality report already exists."

comments = pd.read_csv(RAW_FILE, keep_default_na=False)
logs = pd.read_csv(LOG_FILE, keep_default_na=False)

required = {
    "movie_record_id", "movie_title", "video_id",
    "comment_id", "comment_text"
}
assert required.issubset(comments.columns), (
    f"Missing columns: {required - set(comments.columns)}"
)

# ------------------------------------------------------------
# 1. TEXT NORMALIZATION FOR DUPLICATE AUDIT
# ------------------------------------------------------------

def normalize_text(value):
    value = str(value).lower().strip()
    value = re.sub(r"\s+", " ", value)
    return value

comments["normalized_text"] = comments["comment_text"].map(normalize_text)

comments["word_count"] = comments["comment_text"].map(
    lambda x: len(str(x).split())
)

comments["character_count"] = comments["comment_text"].map(
    lambda x: len(str(x).strip())
)

comments["is_empty"] = comments["comment_text"].map(
    lambda x: len(str(x).strip()) == 0
)

comments["is_very_short"] = comments["word_count"] <= 3

comments["contains_url"] = comments["comment_text"].map(
    lambda x: bool(re.search(r"https?://|www\.", str(x), flags=re.I))
)

# Duplicate text records: count all records belonging to repeated text.
valid_text = comments["normalized_text"].ne("")

text_counts = comments.loc[valid_text, "normalized_text"].value_counts()
repeated_texts = set(text_counts[text_counts > 1].index)

comments["duplicate_text"] = (
    comments["normalized_text"].isin(repeated_texts)
    & valid_text
)

# Within-video vs across-video duplicate text.
text_video_counts = (
    comments.loc[valid_text]
    .groupby("normalized_text")["video_id"]
    .nunique()
)

cross_video_texts = set(
    text_video_counts[text_video_counts > 1].index
)

comments["duplicate_across_videos"] = (
    comments["normalized_text"].isin(cross_video_texts)
    & valid_text
)

comments["duplicate_within_video"] = (
    comments["duplicate_text"]
    & ~comments["duplicate_across_videos"]
)

# ------------------------------------------------------------
# 2. SCRIPT PROFILE — SCRIPT IS NOT LANGUAGE IDENTIFICATION
# ------------------------------------------------------------

def detect_script(text):
    text = str(text)

    counts = {
        "Devanagari": len(re.findall(r"[\u0900-\u097F]", text)),
        "Tamil": len(re.findall(r"[\u0B80-\u0BFF]", text)),
        "Telugu": len(re.findall(r"[\u0C00-\u0C7F]", text)),
        "Kannada": len(re.findall(r"[\u0C80-\u0CFF]", text)),
        "Bengali": len(re.findall(r"[\u0980-\u09FF]", text)),
        "Latin": len(re.findall(r"[A-Za-z]", text))
    }

    present = [k for k, v in counts.items() if v > 0]

    if len(present) == 0:
        return "Other/Emoji/Symbols"
    if len(present) > 1:
        return "Mixed"
    return present[0]

comments["script_category"] = comments["comment_text"].map(detect_script)

# ------------------------------------------------------------
# 3. OVERALL QUALITY PROFILE
# ------------------------------------------------------------

total = len(comments)
unique_ids = comments["comment_id"].nunique()
duplicate_ids = total - unique_ids

profile = pd.DataFrame([
    ["Total comment records", total],
    ["Unique comment IDs", unique_ids],
    ["Duplicate comment IDs", duplicate_ids],
    ["Empty comments", int(comments["is_empty"].sum())],
    ["Very short comments (<=3 words)", int(comments["is_very_short"].sum())],
    ["Comments containing URLs", int(comments["contains_url"].sum())],
    ["Duplicate-text records", int(comments["duplicate_text"].sum())],
    ["Duplicate-text records within video", int(comments["duplicate_within_video"].sum())],
    ["Duplicate-text records across videos", int(comments["duplicate_across_videos"].sum())],
    ["Average word count", round(comments["word_count"].mean(), 2)],
    ["Median word count", round(comments["word_count"].median(), 2)],
    ["Average character count", round(comments["character_count"].mean(), 2)]
], columns=["quality_metric", "value"])

# ------------------------------------------------------------
# 4. MOVIE-WISE QUALITY PROFILE
# ------------------------------------------------------------

movie_quality = (
    comments.groupby(
        ["movie_record_id", "movie_title", "video_id"],
        dropna=False
    )
    .agg(
        comment_records=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        empty_comments=("is_empty", "sum"),
        very_short_comments=("is_very_short", "sum"),
        url_comments=("contains_url", "sum"),
        duplicate_text_records=("duplicate_text", "sum"),
        duplicate_within_video=("duplicate_within_video", "sum"),
        duplicate_across_videos=("duplicate_across_videos", "sum"),
        average_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        average_characters=("character_count", "mean")
    )
    .reset_index()
)

movie_quality["average_words"] = movie_quality["average_words"].round(2)
movie_quality["average_characters"] = movie_quality["average_characters"].round(2)

# ------------------------------------------------------------
# 5. SCRIPT DISTRIBUTION
# ------------------------------------------------------------

script_profile = (
    comments["script_category"]
    .value_counts()
    .rename_axis("quality_metric")
    .reset_index(name="value")
)

script_profile["quality_metric"] = (
    "Script category: " + script_profile["quality_metric"]
)

profile = pd.concat(
    [profile, script_profile],
    ignore_index=True
)

# ------------------------------------------------------------
# 6. ACQUISITION RECONCILIATION
# ------------------------------------------------------------

print("=" * 70)
print("BATCH 06 — ACQUISITION RECONCILIATION")
print("=" * 70)

print("Raw comment records:", total)
print("Unique comment IDs:", unique_ids)
print("Log total collected:", int(logs["comments_collected"].sum()))
print("Log target:", int(logs["target_comments"].sum()))

assert total == int(logs["comments_collected"].sum()), (
    "Raw comments and acquisition log totals do not match."
)

# ------------------------------------------------------------
# 7. SAVE REPORTS
# ------------------------------------------------------------

PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

profile.to_csv(PROFILE_FILE, index=False, encoding="utf-8-sig")
movie_quality.to_csv(MOVIE_FILE, index=False, encoding="utf-8-sig")

print("\n" + "=" * 70)
print("OVERALL COMMENT QUALITY PROFILE")
print("=" * 70)
display(profile)

print("\n" + "=" * 70)
print("MOVIE-WISE COMMENT QUALITY")
print("=" * 70)
display(movie_quality)

print("\nSaved reports:")
print(PROFILE_FILE)
print(MOVIE_FILE)

print("\nNote: Script categories are not language labels.")
print("Duplicate text is flagged for analysis; no comments were deleted.")

In [ ]:
# ============================================================
# BATCH 06 — UPDATE ACQUISITION MANIFEST
# Uses actual acquisition log | No API calls
# Original manifest remains unchanged
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_ROOT = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

RAW_ROOT = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

MANIFEST_FILE = (
    PROCESSED_ROOT /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_06.csv"
)

LOG_FILE = (
    RAW_ROOT /
    "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_06.csv"
)

OUTPUT_FILE = (
    PROCESSED_ROOT /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_06_UPDATED.csv"
)

assert MANIFEST_FILE.exists()
assert LOG_FILE.exists()
assert not OUTPUT_FILE.exists(), (
    "Updated manifest already exists. Inspect it before proceeding."
)

manifest = pd.read_csv(MANIFEST_FILE, dtype={"video_id": str})
logs = pd.read_csv(LOG_FILE, dtype={"video_id": str})

assert manifest["video_id"].is_unique
assert logs["video_id"].is_unique
assert set(manifest["video_id"]) == set(logs["video_id"])

# Map actual acquisition results to the manifest
log_lookup = logs.set_index("video_id")

manifest["comments_collected"] = manifest["video_id"].map(
    log_lookup["comments_collected"]
).astype(int)

manifest["acquisition_status"] = manifest["video_id"].map(
    log_lookup["acquisition_status"]
)

def classify_acquisition(row):
    collected = row["comments_collected"]
    target = row["target_comments"]

    if collected == 0:
        return "No Comments Retrieved"
    elif collected < target:
        return "Partial"
    else:
        return "Completed"

manifest["acquisition_result"] = manifest.apply(
    classify_acquisition,
    axis=1
)

manifest.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

# Read-back verification
verified = pd.read_csv(OUTPUT_FILE, dtype={"video_id": str})

print("=" * 70)
print("BATCH 06 — UPDATED MANIFEST VERIFICATION")
print("=" * 70)

print("Shape:", verified.shape)
print("Total target:", verified["target_comments"].sum())
print("Total collected:", verified["comments_collected"].sum())

print("\nAcquisition result distribution:")
print(verified["acquisition_result"].value_counts())

print("\nMovie-wise status:")
display(verified[[
    "movie_record_id",
    "movie_title",
    "video_id",
    "target_comments",
    "comments_collected",
    "acquisition_status",
    "acquisition_result"
]])

print("\nSaved:", OUTPUT_FILE)
print("Original approved manifest preserved:", MANIFEST_FILE.exists())

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 06 CONSOLIDATION PREFLIGHT
# Read-only | No files modified
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

RAW_COMMENTS = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

files = {
    "Previous consolidated comments": (
        PROCESSED /
        "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_05.csv"
    ),
    "Previous movie coverage": (
        PROCESSED /
        "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_05.csv"
    ),
    "Previous duplicate audit": (
        PROCESSED /
        "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_PILOT_BATCH_01_05.csv"
    ),
    "Batch 06 raw comments": (
        RAW_COMMENTS /
        "ICMB_130_YOUTUBE_COMMENT_BATCH_06.csv"
    ),
    "Batch 06 updated manifest": (
        PROCESSED /
        "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_06_UPDATED.csv"
    )
}

print("=" * 70)
print("CONSOLIDATION PREFLIGHT")
print("=" * 70)

for label, path in files.items():
    print(f"\n{label}")
    print("Exists:", path.exists())

    if path.exists():
        print("Size (KB):", round(path.stat().st_size / 1024, 2))

assert all(path.exists() for path in files.values()), (
    "One or more required files are missing."
)

# Inspect schemas and record counts
old_comments = pd.read_csv(files["Previous consolidated comments"])
batch06 = pd.read_csv(files["Batch 06 raw comments"])

print("\n" + "=" * 70)
print("EXISTING CONSOLIDATED DATA")
print("=" * 70)
print("Shape:", old_comments.shape)
print("Columns:", old_comments.columns.tolist())
print("Unique comment IDs:", old_comments["comment_id"].nunique())

print("\n" + "=" * 70)
print("BATCH 06 RAW DATA")
print("=" * 70)
print("Shape:", batch06.shape)
print("Columns:", batch06.columns.tolist())
print("Unique comment IDs:", batch06["comment_id"].nunique())

print("\n" + "=" * 70)
print("BATCH 06 OVERLAP CHECK")
print("=" * 70)

overlap = set(old_comments["comment_id"]) & set(batch06["comment_id"])

print("Comment IDs already present:", len(overlap))
print("New Batch 06 comment IDs:", batch06["comment_id"].nunique() - len(overlap))

print("\nPreflight complete. No files modified.")

In [ ]:
# ============================================================
# BATCH 06 — CONSOLIDATION SCHEMA MAPPING INSPECTION
# Read-only | No files modified
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

RAW = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

old = pd.read_csv(
    PROCESSED /
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_05.csv"
)

new = pd.read_csv(
    RAW / "ICMB_130_YOUTUBE_COMMENT_BATCH_06.csv"
)

# ------------------------------------------------------------
# 1. Existing consolidated schema and non-null coverage
# ------------------------------------------------------------

print("=" * 75)
print("EXISTING CONSOLIDATED DATA — COLUMN PROFILE")
print("=" * 75)

column_profile = pd.DataFrame({
    "column": old.columns,
    "dtype": [str(old[c].dtype) for c in old.columns],
    "non_null": [old[c].notna().sum() for c in old.columns],
    "null": [old[c].isna().sum() for c in old.columns],
    "sample_value": [
        next(
            (str(v)[:100] for v in old[c] if pd.notna(v)),
            "<all null>"
        )
        for c in old.columns
    ]
})

display(column_profile)

# ------------------------------------------------------------
# 2. Existing source and batch conventions
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("EXISTING BATCH DISTRIBUTION")
print("=" * 75)

for col in ["source_batch", "batch_id", "source_type",
            "retrieval_order", "sampling_order",
            "sampling_unit", "comment_source"]:
    if col in old.columns:
        print(f"\n{col}:")
        print(old[col].value_counts(dropna=False).head(15))

# ------------------------------------------------------------
# 3. Batch 06 source field inspection
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BATCH 06 — SAMPLE RECORDS")
print("=" * 75)

display(new.head(3).T)

print("\n" + "=" * 75)
print("BATCH 06 FIELD COVERAGE")
print("=" * 75)

new_profile = pd.DataFrame({
    "column": new.columns,
    "dtype": [str(new[c].dtype) for c in new.columns],
    "non_null": [new[c].notna().sum() for c in new.columns],
    "null": [new[c].isna().sum() for c in new.columns]
})

display(new_profile)

# ------------------------------------------------------------
# 4. Verify movie and video coverage
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BATCH 06 COVERAGE")
print("=" * 75)

print("Movies:", new["movie_record_id"].nunique())
print("Videos:", new["video_id"].nunique())
print("Comment records:", len(new))
print("Unique comment IDs:", new["comment_id"].nunique())

print("\nMovies and videos:")
display(
    new.groupby(
        ["movie_record_id", "movie_title", "video_id"],
        dropna=False
    ).size().reset_index(name="comment_records")
)

print("\nInspection complete. No files modified.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — CONSOLIDATE PILOT + BATCHES 01–06
# Creates new versioned outputs; preserves all previous files
# ============================================================

import pandas as pd
import re
from pathlib import Path
from datetime import timezone
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

RAW = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

# ------------------------------------------------------------
# 1. INPUT FILES
# ------------------------------------------------------------

OLD_FILE = (
    PROCESSED /
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_05.csv"
)

BATCH06_FILE = (
    RAW / "ICMB_130_YOUTUBE_COMMENT_BATCH_06.csv"
)

LOG_FILE = (
    RAW / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_06.csv"
)

MANIFEST_FILE = (
    PROCESSED /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_06_UPDATED.csv"
)

REFERENCE_FILE = (
    PROCESSED /
    "ICMB_130_AUDIENCE_MOVIE_REFERENCE_VALIDATED.csv"
)

# ------------------------------------------------------------
# 2. OUTPUT FILES — NEW NAMES
# ------------------------------------------------------------

CONSOLIDATED_OUT = (
    PROCESSED /
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_06.csv"
)

COVERAGE_OUT = (
    PROCESSED /
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_06.csv"
)

DUPLICATE_OUT = (
    PROCESSED /
    "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_PILOT_BATCH_01_06.csv"
)

for path in [
    OLD_FILE, BATCH06_FILE, LOG_FILE,
    MANIFEST_FILE, REFERENCE_FILE
]:
    assert path.exists(), f"Missing input file: {path.name}"

for path in [
    CONSOLIDATED_OUT, COVERAGE_OUT, DUPLICATE_OUT
]:
    assert not path.exists(), (
        f"Output already exists: {path.name}. "
        "Inspect before replacing."
    )

# ------------------------------------------------------------
# 3. LOAD INPUTS
# ------------------------------------------------------------

old = pd.read_csv(OLD_FILE, keep_default_na=False)
new = pd.read_csv(BATCH06_FILE, keep_default_na=False)
logs = pd.read_csv(LOG_FILE, keep_default_na=False)
manifest = pd.read_csv(MANIFEST_FILE, keep_default_na=False)
reference = pd.read_csv(REFERENCE_FILE, keep_default_na=False)

assert len(old) == 3684
assert len(new) == 741
assert old["comment_id"].is_unique
assert new["comment_id"].is_unique

overlap = set(old["comment_id"]) & set(new["comment_id"])
assert len(overlap) == 0, (
    f"Found {len(overlap)} overlapping comment IDs."
)

# ------------------------------------------------------------
# 4. BUILD BATCH 06 IN THE EXISTING 32-COLUMN SCHEMA
# ------------------------------------------------------------

columns = old.columns.tolist()

batch06 = pd.DataFrame(index=new.index, columns=columns)

# Directly available fields
direct_fields = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "video_id",
    "video_title",
    "channel_title",
    "comment_id",
    "comment_text",
    "published_at",
    "updated_at",
    "like_count",
    "author_name",
    "comment_position"
]

for col in direct_fields:
    if col in new.columns and col in batch06.columns:
        batch06[col] = new[col].values

# Batch and source information
batch06["source_batch"] = "Batch 06"
batch06["batch_id"] = "BATCH_06"

# Source type from the approved, updated manifest
source_type_map = manifest.set_index(
    "video_id"
)["manual_source_type"].to_dict()

batch06["source_type"] = batch06["video_id"].map(
    source_type_map
)

assert batch06["source_type"].notna().all(), (
    "Some Batch 06 videos have no source type mapping."
)

# Preserve actual retrieval convention
batch06["retrieval_order"] = new["retrieval_order"].values
batch06["sampling_order"] = new["retrieval_order"].values
batch06["acquisition_order"] = new["retrieval_order"].values

batch06["sampling_unit"] = "top_level_comment"
batch06["comment_source"] = "YouTube Top-Level Comment"

# Author display name is the same field supplied by the API.
batch06["author_display_name"] = new["author_name"].values

# Normalize text using the same lowercase/whitespace convention.
def normalize_comment(value):
    value = str(value).lower().strip()
    return re.sub(r"\s+", " ", value)

batch06["normalized_comment_text"] = (
    new["comment_text"].map(normalize_comment).values
)

# Acquisition timestamps from the actual log.
timestamp_map = logs.set_index(
    "video_id"
)["retrieved_at_utc"].to_dict()

batch06["acquisition_timestamp_utc"] = (
    batch06["video_id"].map(timestamp_map)
)

batch06["acquired_at_utc"] = (
    batch06["video_id"].map(timestamp_map)
)

batch06["retrieved_at"] = (
    batch06["video_id"].map(timestamp_map)
)

# These fields were not retrieved in Batch 06.
# Leave them missing rather than invent values:
# author_channel_id, comment_reply_count, reply_count,
# total_reply_count, comment_author, series_part.

# Match existing consolidated column order.
batch06 = batch06[columns]

# ------------------------------------------------------------
# 5. CONCATENATE WITHOUT MODIFYING OLD RECORDS
# ------------------------------------------------------------

combined = pd.concat(
    [old, batch06],
    ignore_index=True
)

assert len(combined) == len(old) + len(new)
assert combined["comment_id"].is_unique, (
    "Duplicate comment IDs detected after consolidation."
)

# ------------------------------------------------------------
# 6. RECOMPUTE MOVIE COVERAGE AGAINST 130-MOVIE REFERENCE
# ------------------------------------------------------------

assert "movie_record_id" in reference.columns
assert reference["movie_record_id"].is_unique

comment_counts = (
    combined.groupby("movie_record_id")
    .agg(
        comment_records=("comment_id", "size"),
        unique_videos=("video_id", "nunique"),
        unique_comment_ids=("comment_id", "nunique")
    )
    .reset_index()
)

coverage = reference.merge(
    comment_counts,
    on="movie_record_id",
    how="left"
)

for col in [
    "comment_records",
    "unique_videos",
    "unique_comment_ids"
]:
    coverage[col] = coverage[col].fillna(0).astype(int)

coverage["coverage_status"] = coverage["comment_records"].map(
    lambda n: (
        "No Comments" if n == 0
        else "Low (1-49)" if n < 50
        else "Moderate (50-99)" if n < 100
        else "Higher (100+)"
    )
)

# ------------------------------------------------------------
# 7. RECOMPUTE DUPLICATE-TEXT AUDIT
# ------------------------------------------------------------

duplicate_source = combined[
    ["movie_record_id", "movie_title", "video_id",
     "comment_id", "comment_text"]
].copy()

duplicate_source["normalized_text"] = (
    duplicate_source["comment_text"].map(normalize_comment)
)

duplicate_source = duplicate_source[
    duplicate_source["normalized_text"].ne("")
]

duplicate_audit = (
    duplicate_source.groupby("normalized_text")
    .agg(
        record_count=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        unique_movies=("movie_record_id", "nunique"),
        movie_titles=(
            "movie_title",
            lambda x: " | ".join(sorted(set(x)))
        ),
        video_ids=(
            "video_id",
            lambda x: " | ".join(sorted(set(x)))
        ),
        comment_ids=(
            "comment_id",
            lambda x: " | ".join(x)
        )
    )
    .reset_index()
)

duplicate_audit = duplicate_audit[
    duplicate_audit["record_count"] > 1
].copy()

duplicate_audit["duplicate_scope"] = (
    duplicate_audit["unique_videos"].map(
        lambda n: (
            "Across videos" if n > 1
            else "Within video"
        )
    )
)

# ------------------------------------------------------------
# 8. SAVE NEW OUTPUTS
# ------------------------------------------------------------

combined.to_csv(
    CONSOLIDATED_OUT,
    index=False,
    encoding="utf-8-sig"
)

coverage.to_csv(
    COVERAGE_OUT,
    index=False,
    encoding="utf-8-sig"
)

duplicate_audit.to_csv(
    DUPLICATE_OUT,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 9. READ-BACK VERIFICATION
# ------------------------------------------------------------

check_combined = pd.read_csv(
    CONSOLIDATED_OUT,
    keep_default_na=False
)

check_coverage = pd.read_csv(COVERAGE_OUT)
check_duplicates = pd.read_csv(
    DUPLICATE_OUT,
    keep_default_na=False
)

assert len(check_combined) == 4425
assert check_combined["comment_id"].nunique() == 4425
assert len(check_coverage) == 130
assert check_combined.columns.tolist() == columns

print("=" * 75)
print("PILOT + BATCHES 01–06 — CONSOLIDATION SUCCESS")
print("=" * 75)

print("Previous records:", len(old))
print("Batch 06 records:", len(new))
print("Consolidated records:", len(check_combined))
print("Unique comment IDs:", check_combined["comment_id"].nunique())
print("Consolidated columns:", len(check_combined.columns))
print("Unique videos:", check_combined["video_id"].nunique())
print("Unique movies with comments:", check_combined["movie_record_id"].nunique())

print("\nBatch distribution:")
display(
    check_combined["source_batch"]
    .value_counts()
    .rename_axis("source_batch")
    .reset_index(name="records")
)

print("\nCoverage summary:")
display(
    check_coverage["coverage_status"]
    .value_counts()
    .rename_axis("coverage_status")
    .reset_index(name="movies")
)

print("\nDuplicate-text audit:")
print("Repeated-text groups:", len(check_duplicates))
print(
    "Records in repeated-text groups:",
    check_duplicates["record_count"].sum()
)
print(
    "Groups across videos:",
    (check_duplicates["duplicate_scope"] == "Across videos").sum()
)
print(
    "Groups within video:",
    (check_duplicates["duplicate_scope"] == "Within video").sum()
)

print("\nSaved files:")
print(CONSOLIDATED_OUT)
print(COVERAGE_OUT)
print(DUPLICATE_OUT)

print("\nPrevious Pilot–Batch 05 consolidated file preserved:", OLD_FILE.exists())
print("Raw Batch 06 file preserved:", BATCH06_FILE.exists())

In [ ]:
# ============================================================
# NOTEBOOK 10 — UPDATED ACQUISITION PLANNING PREFLIGHT
# Read-only | No files modified
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED = (
    PROJECT_ROOT / "01_data" / "processed" /
    "audience_sentiment"
)

planning_file = (
    PROCESSED /
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT.csv"
)

planning_summary_file = (
    PROCESSED /
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY.csv"
)

coverage_file = (
    PROCESSED /
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_06.csv"
)

assert planning_file.exists()
assert planning_summary_file.exists()
assert coverage_file.exists()

old_planning = pd.read_csv(planning_file)
old_summary = pd.read_csv(planning_summary_file)
coverage = pd.read_csv(coverage_file)

print("=" * 70)
print("PREVIOUS ACQUISITION PLANNING REPORT")
print("=" * 70)

print("Shape:", old_planning.shape)
print("Columns:", old_planning.columns.tolist())
display(old_planning.head(5))

print("\n" + "=" * 70)
print("PREVIOUS PLANNING SUMMARY")
print("=" * 70)

print("Shape:", old_summary.shape)
display(old_summary)

print("\n" + "=" * 70)
print("LATEST MOVIE COVERAGE")
print("=" * 70)

print("Reference movies:", len(coverage))
print("Covered movies:", (coverage["comment_records"] > 0).sum())
print("Uncovered movies:", (coverage["comment_records"] == 0).sum())
print("Total comments:", coverage["comment_records"].sum())

print("\nCoverage categories:")
display(
    coverage["coverage_status"]
    .value_counts()
    .rename_axis("coverage_status")
    .reset_index(name="movies")
)

print("\nLanguage-wise coverage:")
display(
    coverage.groupby("language")
    .agg(
        total_movies=("movie_record_id", "count"),
        covered_movies=("comment_records", lambda x: (x > 0).sum()),
        uncovered_movies=("comment_records", lambda x: (x == 0).sum()),
        comment_records=("comment_records", "sum")
    )
    .reset_index()
)

print("\nPreflight complete. No files modified.")

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
PROCESSED = PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"

coverage_file = PROCESSED / "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_06.csv"

coverage = pd.read_csv(coverage_file)

print("=" * 70)
print("LATEST MOVIE COVERAGE — SCHEMA AND SAMPLE")
print("=" * 70)
print("Shape:", coverage.shape)
print("Columns:", coverage.columns.tolist())
display(coverage.head(10))

print("\n" + "=" * 70)
print("COVERAGE CATEGORY COUNTS")
print("=" * 70)
for col in ["coverage_status", "coverage_category"]:
    if col in coverage.columns:
        print(f"\n{col}:")
        print(coverage[col].value_counts(dropna=False))

print("\n" + "=" * 70)
print("LOW-COVERAGE AND UNCOVERED MOVIES")
print("=" * 70)

category_col = (
    "coverage_category" if "coverage_category" in coverage.columns
    else "coverage_status" if "coverage_status" in coverage.columns
    else None
)

if category_col:
    display(
        coverage[
            coverage[category_col].astype(str).str.contains(
                "No Comments|Low|Moderate", case=False, na=False
            )
        ].head(30)
    )

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
PROCESSED = PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"

coverage_file = PROCESSED / "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_06.csv"

report_file = PROCESSED / "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_06.csv"
summary_file = PROCESSED / "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_06.csv"

# ------------------------------------------------------------
# 1. LOAD LATEST COVERAGE
# ------------------------------------------------------------
coverage = pd.read_csv(coverage_file)

required_columns = [
    "movie_record_id", "movie_title", "year", "rank", "language",
    "comment_records", "unique_videos", "unique_comment_ids",
    "coverage_status"
]

missing = [c for c in required_columns if c not in coverage.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if len(coverage) != 130 or coverage["movie_record_id"].nunique() != 130:
    raise ValueError("Reference validation failed: expected 130 unique movies.")

# ------------------------------------------------------------
# 2. BUILD UPDATED PLANNING REPORT
# ------------------------------------------------------------
planning = coverage[required_columns].copy()

planning["comment_records"] = planning["comment_records"].fillna(0).astype(int)
planning["unique_comment_ids"] = planning["unique_comment_ids"].fillna(0).astype(int)
planning["unique_videos"] = planning["unique_videos"].fillna(0).astype(int)

planning["acquisition_batches"] = 0
planning["coverage_category"] = planning["coverage_status"]

planning["additional_comment_target"] = (
    100 - planning["comment_records"]
).clip(lower=0)

planning.loc[
    planning["comment_records"] >= 100,
    "additional_comment_target"
] = 0

def assign_priority(row):
    comments = row["comment_records"]
    language = row["language"]

    if comments == 0:
        if language != "Hindi":
            return "High — Uncovered Non-Hindi"
        return "High — Uncovered Hindi"

    if comments < 50:
        return "High — Low Coverage"

    if comments < 100:
        return "Medium — Moderate Coverage"

    return "Covered — Monitor"

planning["acquisition_priority"] = planning.apply(assign_priority, axis=1)

planning = planning[
    [
        "movie_record_id", "movie_title", "year", "rank", "language",
        "comment_records", "unique_comment_ids", "unique_videos",
        "acquisition_batches", "coverage_status", "coverage_category",
        "acquisition_priority", "additional_comment_target"
    ]
]

# ------------------------------------------------------------
# 3. BUILD UPDATED SUMMARY
# ------------------------------------------------------------
category_order = [
    "No Comments",
    "Low (1-49)",
    "Moderate (50-99)",
    "Higher (100+)"
]

summary = (
    planning.groupby("coverage_category", dropna=False)
    .agg(
        movie_count=("movie_record_id", "count"),
        existing_comments=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reindex(category_order, fill_value=0)
    .reset_index()
)

# Language-wise coverage summary
language_summary = (
    planning.groupby("language")
    .agg(
        total_movies=("movie_record_id", "count"),
        covered_movies=("comment_records", lambda x: (x > 0).sum()),
        uncovered_movies=("comment_records", lambda x: (x == 0).sum()),
        comment_records=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 4. VALIDATION
# ------------------------------------------------------------
total_movies = len(planning)
covered_movies = int((planning["comment_records"] > 0).sum())
uncovered_movies = int((planning["comment_records"] == 0).sum())
total_comments = int(planning["comment_records"].sum())
total_target = int(planning["additional_comment_target"].sum())

assert total_movies == 130
assert covered_movies == 44
assert uncovered_movies == 86
assert total_comments == 4425
assert total_target == 8678

assert planning["movie_record_id"].is_unique
assert planning["additional_comment_target"].ge(0).all()

# ------------------------------------------------------------
# 5. SAVE AS NEW VERSIONED FILES
# ------------------------------------------------------------
if report_file.exists() or summary_file.exists():
    raise FileExistsError(
        "A versioned output already exists. Inspect it before overwriting."
    )

planning.to_csv(report_file, index=False)
summary.to_csv(summary_file, index=False)

language_summary_file = (
    PROCESSED /
    "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_06.csv"
)

if language_summary_file.exists():
    raise FileExistsError("Language summary already exists. Inspect before overwriting.")

language_summary.to_csv(language_summary_file, index=False)

# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------
print("=" * 70)
print("UPDATED YOUTUBE ACQUISITION PLANNING — PILOT + BATCHES 01–06")
print("=" * 70)

print(f"Reference movies       : {total_movies}")
print(f"Movies with comments   : {covered_movies}")
print(f"Movies without comments: {uncovered_movies}")
print(f"Total comment records  : {total_comments}")
print(f"Additional target      : {total_target:,}")

print("\nCOVERAGE SUMMARY")
display(summary)

print("\nLANGUAGE-WISE COVERAGE")
display(language_summary)

print("\nLOW AND MODERATE COVERAGE MOVIES")
display(
    planning[
        planning["coverage_category"].isin(["Low (1-49)", "Moderate (50-99)"])
    ][
        ["movie_record_id", "movie_title", "language",
         "comment_records", "additional_comment_target", "acquisition_priority"]
    ]
)

print("\nSAVED FILES")
print(report_file)
print(summary_file)
print(language_summary_file)

print("\nVALIDATION: PASSED")

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
PROCESSED = PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"

coverage_file = (
    PROCESSED /
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_06.csv"
)

coverage = pd.read_csv(coverage_file)

# Select movies that currently have no comments
uncovered = coverage[
    coverage["comment_records"].fillna(0).eq(0)
].copy()

# Language-wise summary
print("=" * 75)
print("BATCH 07 — UNCOVERED MOVIE ANALYSIS")
print("=" * 75)

print("\nLANGUAGE-WISE UNCOVERED MOVIES")
language_summary = (
    uncovered.groupby("language")
    .agg(
        uncovered_movies=("movie_record_id", "count"),
        year_start=("year", "min"),
        year_end=("year", "max")
    )
    .sort_values("uncovered_movies", ascending=False)
    .reset_index()
)
display(language_summary)

print("\nUNCOVERED MOVIES — SORTED BY LANGUAGE AND YEAR")
display(
    uncovered[
        [
            "movie_record_id", "movie_title", "year", "rank",
            "language", "india_gross_cr", "industry",
            "tmdb_original_language"
        ]
    ].sort_values(["language", "year", "rank"]).reset_index(drop=True)
)

print("\nUNCOVERED MOVIES BY YEAR")
display(
    uncovered.groupby("year")
    .size()
    .reset_index(name="uncovered_movies")
    .sort_values("year")
)

print("\nVALIDATION")
print("Uncovered movie records:", len(uncovered))
print("Unique movie IDs:", uncovered["movie_record_id"].nunique())
print("All have zero comments:", uncovered["comment_records"].fillna(0).eq(0).all())

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
PROCESSED = PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"

coverage = pd.read_csv(
    PROCESSED / "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_06.csv"
)

selected_ids = [
    "2022_2", "2023_5", "2024_1", "2024_2",
    "2008_3", "2008_1", "2009_1", "2011_1", "2013_1"
]

print("PROPOSED BATCH 07 — SELECTED RECORDS")
display(
    coverage[coverage["movie_record_id"].isin(selected_ids)][
        ["movie_record_id", "movie_title", "year", "rank",
         "language", "india_gross_cr", "comment_records"]
    ].sort_values(["language", "year"])
)

print("\nUNCOVERED KANNADA MOVIE")
display(
    coverage[
        (coverage["language"] == "Kannada") &
        (coverage["comment_records"] == 0)
    ][
        ["movie_record_id", "movie_title", "year", "rank",
         "language", "india_gross_cr", "comment_records"]
    ]
)

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
PROCESSED = PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"

coverage_file = (
    PROCESSED /
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_06.csv"
)

output_file = (
    PROCESSED /
    "ICMB_130_YOUTUBE_BATCH_07_MOVIE_ALLOCATION.csv"
)

coverage = pd.read_csv(coverage_file)

# Confirmed Batch 07 selection
selected_ids = [
    "2022_2",  # RRR
    "2023_5",  # Salaar: Part 1 – Ceasefire
    "2024_1",  # Pushpa 2: The Rule
    "2024_2",  # Kalki 2898 AD
    "2008_3",  # Dasavathaaram
    "2025_2",  # Kantara: A Legend – Chapter 1
    "2008_1",  # Ghajini
    "2009_1",  # 3 Idiots
    "2011_1",  # Bodyguard
    "2013_1"   # Dhoom 3
]

if len(selected_ids) != 10 or len(set(selected_ids)) != 10:
    raise ValueError("Selection must contain 10 unique movie IDs.")

selected = coverage[
    coverage["movie_record_id"].astype(str).isin(selected_ids)
].copy()

if len(selected) != 10:
    missing_ids = set(selected_ids) - set(selected["movie_record_id"].astype(str))
    raise ValueError(f"Selected IDs missing from coverage: {missing_ids}")

if not selected["comment_records"].fillna(0).eq(0).all():
    raise ValueError("At least one selected movie already has comments.")

# Preserve the intended selection order
selected["selection_order"] = selected["movie_record_id"].astype(str).map(
    {movie_id: i + 1 for i, movie_id in enumerate(selected_ids)}
)

selected["batch_id"] = "Batch 07"
selected["existing_comment_records"] = selected["comment_records"].fillna(0).astype(int)
selected["target_comments"] = 100
selected["allocation_status"] = "Selected for YouTube Discovery"

allocation = selected[
    [
        "batch_id", "selection_order", "movie_record_id",
        "movie_title", "year", "rank", "language",
        "india_gross_cr", "tmdb_id", "imdb_id",
        "existing_comment_records", "target_comments",
        "allocation_status"
    ]
].sort_values("selection_order")

if output_file.exists():
    raise FileExistsError(
        f"Allocation file already exists; inspect before overwriting: {output_file}"
    )

allocation.to_csv(output_file, index=False)

print("=" * 70)
print("BATCH 07 MOVIE ALLOCATION — SAVED")
print("=" * 70)

display(allocation)

print("\nLanguage distribution:")
display(allocation["language"].value_counts().rename_axis("language").reset_index(name="movies"))

print("\nTotal movies:", len(allocation))
print("Unique movie IDs:", allocation["movie_record_id"].nunique())
print("Existing comments:", allocation["existing_comment_records"].sum())
print("Potential comment target:", allocation["target_comments"].sum())
print("\nSaved:", output_file)

In [ ]:
import os
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from IPython.display import display

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
PROCESSED = PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"
RAW_YOUTUBE = PROJECT_ROOT / "01_data" / "raw" / "audience_sentiment" / "youtube"

allocation_file = PROCESSED / "ICMB_130_YOUTUBE_BATCH_07_MOVIE_ALLOCATION.csv"

discovery_file = RAW_YOUTUBE / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_07.csv"
log_file = RAW_YOUTUBE / "ICMB_130_YOUTUBE_DISCOVERY_LOG_BATCH_07.csv"

# Safety checks
for path in [allocation_file]:
    if not path.exists():
        raise FileNotFoundError(path)

if discovery_file.exists() or log_file.exists():
    raise FileExistsError(
        "Batch 07 discovery output already exists. Inspect existing files before rerunning."
    )

allocation = pd.read_csv(allocation_file)

if len(allocation) != 10 or allocation["movie_record_id"].nunique() != 10:
    raise ValueError("Batch 07 allocation must contain 10 unique movies.")

# Read API key from Colab Secrets; never print it.
api_key = userdata.get("YOUTUBE_API_KEY")
if not api_key:
    raise ValueError("YOUTUBE_API_KEY not found in Colab Secrets.")

youtube = build("youtube", "v3", developerKey=api_key)

discovery_rows = []
log_rows = []

print("=" * 70)
print("BATCH 07 — YOUTUBE VIDEO DISCOVERY")
print("=" * 70)

for _, movie in allocation.sort_values("selection_order").iterrows():

    movie_id = str(movie["movie_record_id"])
    title = str(movie["movie_title"])
    year = int(movie["year"])
    language = str(movie["language"])

    # Search using movie title, year and review/reaction context.
    query = f'"{title}" {year} movie review reaction'

    request_time = datetime.now(timezone.utc).isoformat()

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=10,
            order="relevance",
            relevanceLanguage={
                "Hindi": "hi",
                "Telugu": "te",
                "Tamil": "ta",
                "Kannada": "kn"
            }.get(language, "en")
        ).execute()

        items = response.get("items", [])

        for item in items:
            snippet = item.get("snippet", {})
            discovery_rows.append({
                "batch_id": "Batch 07",
                "movie_record_id": movie_id,
                "movie_title": title,
                "movie_year": year,
                "language": language,
                "search_query": query,
                "video_id": item["id"].get("videoId"),
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "discovery_source": "YouTube Data API v3",
                "discovered_at_utc": request_time
            })

        log_rows.append({
            "batch_id": "Batch 07",
            "movie_record_id": movie_id,
            "movie_title": title,
            "search_query": query,
            "request_status": "Success",
            "candidates_returned": len(items),
            "error_message": None,
            "requested_at_utc": request_time
        })

        print(f"SUCCESS | {movie_id} | {title} | Candidates: {len(items)}")

    except Exception as e:
        log_rows.append({
            "batch_id": "Batch 07",
            "movie_record_id": movie_id,
            "movie_title": title,
            "search_query": query,
            "request_status": "Error",
            "candidates_returned": 0,
            "error_message": str(e),
            "requested_at_utc": request_time
        })

        print(f"ERROR | {movie_id} | {title} | {str(e)[:200]}")

# Save discovery and request log
discovery = pd.DataFrame(discovery_rows)
request_log = pd.DataFrame(log_rows)

discovery.to_csv(discovery_file, index=False)
request_log.to_csv(log_file, index=False)

print("\n" + "=" * 70)
print("BATCH 07 DISCOVERY SUMMARY")
print("=" * 70)

print("Movies searched:", len(request_log))
print("Successful requests:", (request_log["request_status"] == "Success").sum())
print("Failed requests:", (request_log["request_status"] == "Error").sum())
print("Total candidates:", len(discovery))
print("Unique video IDs:", discovery["video_id"].nunique() if len(discovery) else 0)

print("\nCandidates by movie:")
if not discovery.empty:
    display(
        discovery.groupby(
            ["movie_record_id", "movie_title", "language"]
        ).agg(
            candidate_count=("video_id", "count"),
            unique_video_ids=("video_id", "nunique")
        ).reset_index()
    )

print("\nSaved files:")
print(discovery_file)
print(log_file)

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
RAW_YOUTUBE = PROJECT_ROOT / "01_data" / "raw" / "audience_sentiment" / "youtube"

discovery_file = RAW_YOUTUBE / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_07.csv"
metadata_file = RAW_YOUTUBE / "ICMB_130_YOUTUBE_METADATA_BATCH_07.csv"

if not discovery_file.exists():
    raise FileNotFoundError(discovery_file)

if metadata_file.exists():
    raise FileExistsError(
        f"Metadata file already exists. Inspect before rerunning: {metadata_file}"
    )

discovery = pd.read_csv(discovery_file)

if discovery.empty or discovery["video_id"].isna().any():
    raise ValueError("Discovery file is empty or contains missing video IDs.")

video_ids = discovery["video_id"].astype(str).drop_duplicates().tolist()

if len(video_ids) != 100:
    raise ValueError(f"Expected 100 unique video IDs; found {len(video_ids)}.")

api_key = userdata.get("YOUTUBE_API_KEY")
if not api_key:
    raise ValueError("YOUTUBE_API_KEY not found in Colab Secrets.")

youtube = build("youtube", "v3", developerKey=api_key)

metadata_rows = []
error_rows = []

print("=" * 70)
print("BATCH 07 — VIDEO METADATA RETRIEVAL")
print("=" * 70)

# videos.list accepts up to 50 IDs per request
for start in range(0, len(video_ids), 50):
    chunk = video_ids[start:start + 50]

    try:
        response = youtube.videos().list(
            part="snippet,contentDetails,statistics,status",
            id=",".join(chunk),
            maxResults=50
        ).execute()

        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            statistics = item.get("statistics", {})
            content = item.get("contentDetails", {})
            status = item.get("status", {})

            metadata_rows.append({
                "video_id": item.get("id"),
                "metadata_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "metadata_channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "tags": " | ".join(snippet.get("tags", [])),
                "category_id": snippet.get("categoryId"),
                "duration": content.get("duration"),
                "definition": content.get("definition"),
                "caption_available": content.get("caption"),
                "view_count": statistics.get("viewCount"),
                "like_count": statistics.get("likeCount"),
                "comment_count": statistics.get("commentCount"),
                "privacy_status": status.get("privacyStatus"),
                "embeddable": status.get("embeddable"),
                "metadata_retrieved_at_utc": datetime.now(timezone.utc).isoformat()
            })

        returned_ids = {item.get("id") for item in response.get("items", [])}
        missing_ids = set(chunk) - returned_ids

        for missing_id in missing_ids:
            error_rows.append({
                "video_id": missing_id,
                "error": "No metadata returned for requested video ID"
            })

        print(
            f"Retrieved chunk {start // 50 + 1}: "
            f"{len(response.get('items', []))}/{len(chunk)} videos"
        )

    except Exception as e:
        for video_id in chunk:
            error_rows.append({
                "video_id": video_id,
                "error": str(e)
            })
        print(f"ERROR in chunk {start // 50 + 1}: {str(e)[:200]}")

metadata = pd.DataFrame(metadata_rows)

if metadata.empty:
    raise RuntimeError("No video metadata was retrieved. Output not saved.")

# Preserve discovery fields and add metadata fields
merged = discovery.merge(
    metadata,
    on="video_id",
    how="left",
    suffixes=("_discovery", "_metadata"),
    validate="many_to_one"
)

merged.to_csv(metadata_file, index=False)

print("\n" + "=" * 70)
print("BATCH 07 METADATA SUMMARY")
print("=" * 70)
print("Discovery candidates:", len(discovery))
print("Unique requested IDs:", len(video_ids))
print("Metadata records:", len(metadata))
print("Merged shape:", merged.shape)
print("Candidates with metadata:", merged["metadata_title"].notna().sum())
print("Missing metadata:", merged["metadata_title"].isna().sum())
print("Metadata retrieval errors:", len(error_rows))

print("\nMetadata preview:")
display(
    merged[
        [
            "movie_record_id", "movie_title", "video_id",
            "metadata_title", "metadata_channel_title",
            "duration", "view_count", "comment_count"
        ]
    ].head(15)
)

if error_rows:
    print("\nMetadata exceptions:")
    display(pd.DataFrame(error_rows))

print("\nSaved:", metadata_file)

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
RAW_YOUTUBE = PROJECT_ROOT / "01_data" / "raw" / "audience_sentiment" / "youtube"

metadata_file = RAW_YOUTUBE / "ICMB_130_YOUTUBE_METADATA_BATCH_07.csv"

df = pd.read_csv(metadata_file)

print("=" * 70)
print("BATCH 07 METADATA — COLUMN INSPECTION")
print("=" * 70)

print("Shape:", df.shape)
print("\nAll columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

print("\nDescription-related columns:")
description_cols = [c for c in df.columns if "description" in c.lower()]
print(description_cols)

print("\nSample values:")
for col in description_cols:
    print(f"\n{col}:")
    display(df[col].head(3))

print("\nMetadata title and channel fields:")
display(
    df[
        [c for c in [
            "movie_record_id", "movie_title", "video_id",
            "metadata_title", "metadata_channel_title",
            "duration", "view_count", "comment_count"
        ] if c in df.columns]
    ].head(5)
)

In [ ]:
import pandas as pd
import re
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
RAW_YOUTUBE = PROJECT_ROOT / "01_data" / "raw" / "audience_sentiment" / "youtube"

metadata_file = RAW_YOUTUBE / "ICMB_130_YOUTUBE_METADATA_BATCH_07.csv"
output_file = RAW_YOUTUBE / "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_07.csv"

if not metadata_file.exists():
    raise FileNotFoundError(metadata_file)

if output_file.exists():
    raise FileExistsError(f"Output already exists: {output_file}")

df = pd.read_csv(metadata_file)

required = [
    "movie_record_id", "movie_title", "video_id",
    "metadata_title", "metadata_channel_title",
    "description_metadata", "duration", "view_count", "comment_count"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

def duration_seconds(value):
    if pd.isna(value):
        return None

    match = re.fullmatch(
        r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?",
        str(value)
    )
    if not match:
        return None

    h, m, s = [int(x or 0) for x in match.groups()]
    return h * 3600 + m * 60 + s

def classify_video(row):
    title = str(row["metadata_title"]).lower()
    description = str(row["description_metadata"]).lower()
    text = title + " " + description

    seconds = duration_seconds(row["duration"])

    trailer_terms = [
        "official trailer", "trailer reaction",
        "teaser reaction", "teaser trailer", "official teaser"
    ]

    full_movie_terms = [
        "full movie", "full film", "complete movie",
        "watch full movie"
    ]

    review_terms = [
        "movie review", "film review", "review and analysis",
        "ending explained", "movie analysis",
        "film analysis", "retrospective", "rewatched",
        "rewatch", "revisited"
    ]

    reaction_terms = [
        "movie reaction", "film reaction",
        "first time watching", "first time reaction",
        "watching for the first time",
        "foreigners react", "react to"
    ]

    if any(term in text for term in trailer_terms):
        return "Trailer Reaction", "Exclude from primary sample"

    if any(term in text for term in full_movie_terms):
        return "Full Movie", "Exclude from primary sample"

    if any(term in text for term in review_terms):
        return "Movie Review / Analysis", "Shortlist"

    if any(term in text for term in reaction_terms):
        if seconds is not None and seconds >= 1200:
            return "Movie Reaction", "Shortlist"
        return "Possible Reaction", "Hold for review"

    if seconds is not None and seconds < 180:
        return "Short / Unclear", "Hold for review"

    return "Unclassified", "Hold for review"

classified = df.apply(classify_video, axis=1, result_type="expand")
classified.columns = [
    "automated_source_type",
    "screening_decision"
]

df = pd.concat([df, classified], axis=1)

df["duration_seconds"] = df["duration"].apply(duration_seconds)
df["metadata_review_status"] = "Metadata Reviewed"
df["content_review_status"] = "Not Reviewed"
df["screening_method"] = "Rule-based title and metadata description screening"

decision_order = {
    "Shortlist": 0,
    "Hold for review": 1,
    "Exclude from primary sample": 2
}

df["_sort"] = df["screening_decision"].map(decision_order)

df = df.sort_values(
    ["movie_record_id", "_sort", "comment_count", "view_count"],
    ascending=[True, True, False, False]
).drop(columns="_sort")

df.to_csv(output_file, index=False)

print("=" * 70)
print("BATCH 07 — AUTOMATED METADATA SCREENING")
print("=" * 70)

print("Total candidates:", len(df))
print("Unique video IDs:", df["video_id"].nunique())

print("\nSCREENING DECISIONS")
display(df["screening_decision"].value_counts().reset_index())

print("\nPROVISIONAL SOURCE TYPES")
display(df["automated_source_type"].value_counts().reset_index())

print("\nMOVIE-WISE SCREENING")
movie_summary = (
    df.groupby(["movie_record_id", "movie_title", "language"])
    .agg(
        candidates=("video_id", "count"),
        shortlist=("screening_decision",
                   lambda x: (x == "Shortlist").sum()),
        hold=("screening_decision",
              lambda x: (x == "Hold for review").sum()),
        excluded=("screening_decision",
                  lambda x: x.str.startswith("Exclude").sum())
    )
    .reset_index()
)

display(movie_summary)

print("\nSAVED:", output_file)

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
RAW_YOUTUBE = PROJECT_ROOT / "01_data" / "raw" / "audience_sentiment" / "youtube"

input_file = RAW_YOUTUBE / "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_07.csv"
output_file = RAW_YOUTUBE / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_07.csv"

if not input_file.exists():
    raise FileNotFoundError(input_file)

if output_file.exists():
    raise FileExistsError(f"Review file already exists: {output_file}")

df = pd.read_csv(input_file)

# Retain shortlist and hold candidates.
review = df[
    df["screening_decision"].isin(["Shortlist", "Hold for review"])
].copy()

review["youtube_url"] = (
    "https://www.youtube.com/watch?v=" + review["video_id"].astype(str)
)

# Manual decision fields remain blank until reviewed.
review["manual_decision"] = ""
review["manual_source_type"] = ""
review["manual_verification_status"] = "Pending"
review["manual_review_notes"] = ""

columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "metadata_title",
    "metadata_channel_title",
    "duration",
    "view_count",
    "comment_count",
    "automated_source_type",
    "screening_decision",
    "youtube_url",
    "manual_decision",
    "manual_source_type",
    "manual_verification_status",
    "manual_review_notes"
]

review = review[columns].sort_values(
    ["movie_record_id", "screening_decision", "comment_count"],
    ascending=[True, True, False]
)

review.to_csv(output_file, index=False)

print("=" * 70)
print("BATCH 07 — MANUAL REVIEW SHEET")
print("=" * 70)

print("Total candidates:", len(df))
print("Shortlisted:", (df["screening_decision"] == "Shortlist").sum())
print("On hold:", (df["screening_decision"] == "Hold for review").sum())
print("Candidates in review sheet:", len(review))

print("\nCandidates by movie:")
display(
    review.groupby(["movie_record_id", "movie_title", "language"])
    .agg(
        candidates=("video_id", "count"),
        shortlist=("screening_decision",
                   lambda x: (x == "Shortlist").sum()),
        hold=("screening_decision",
              lambda x: (x == "Hold for review").sum())
    )
    .reset_index()
)

print("\nReview sheet preview:")
display(review.head(20))

print("\nSaved:", output_file)

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
file_path = (
    PROJECT_ROOT / "01_data" / "raw" / "audience_sentiment" /
    "youtube" / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_07.csv"
)

df = pd.read_csv(file_path)

for movie_id, group in df.groupby("movie_record_id", sort=True):

    movie_title = group["movie_title"].iloc[0]
    language = group["language"].iloc[0]

    print("\n" + "=" * 100)
    print(f"{movie_id} | {movie_title} | {language}")
    print("=" * 100)

    display(
        group[
            [
                "video_id",
                "metadata_title",
                "metadata_channel_title",
                "duration",
                "view_count",
                "comment_count",
                "automated_source_type",
                "screening_decision",
                "youtube_url"
            ]
        ].reset_index(drop=True)
    )

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

RAW_YOUTUBE = PROJECT_ROOT / "01_data" / "raw" / "audience_sentiment" / "youtube"
PROCESSED_AUDIENCE = PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"

metadata_file = RAW_YOUTUBE / "ICMB_130_YOUTUBE_METADATA_BATCH_07.csv"
output_file = PROCESSED_AUDIENCE / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_07.csv"

if not metadata_file.exists():
    raise FileNotFoundError(metadata_file)

if output_file.exists():
    raise FileExistsError(
        f"Output already exists. It will not be overwritten:\n{output_file}"
    )

PROCESSED_AUDIENCE.mkdir(parents=True, exist_ok=True)

# Approved source decisions
approved_sources = [
    ("2008_1", "2g6a0Q5Qkck", "Movie Reaction",
     "CineDesi Part 2/2; metadata reviewed; content not independently reviewed."),

    ("2009_1", "s_9d85zg6sA", "Movie Reaction",
     "ChaatNChat; metadata reviewed; content not independently reviewed."),

    ("2011_1", "fXDozL7rcZc", "Movie Review",
     "CineDesi; metadata reviewed; content not independently reviewed."),

    ("2013_1", "fXxMhaJXfyk", "Movie Reaction",
     "OUR STUPID REACTIONS Part 3/3; metadata reviewed; content not independently reviewed."),

    ("2022_2", "hW64tis1DJI", "Movie Review",
     "Jeremy Jahns; metadata reviewed; content not independently reviewed."),

    ("2022_2", "sTcZ5mYy5Os", "Movie Reaction",
     "Popcorn In Bed; metadata reviewed; content not independently reviewed."),

    ("2023_5", "Vg0deJOTsHI", "Movie Review",
     "OUR STUPID REACTIONS; metadata reviewed; content not independently reviewed."),

    ("2024_1", "fykmUadWgNQ", "Movie Review",
     "BnfTV; metadata reviewed; content not independently reviewed."),

    ("2024_2", "htDKfJz401k", "Movie Review",
     "Tried&Refused Productions; metadata reviewed; content not independently reviewed."),

    ("2024_2", "szrU_tpy3Ek", "Movie Reaction",
     "The Normies; metadata reviewed; content not independently reviewed."),

    ("2025_2", "u3ekQhaYeS4", "Movie Review",
     "FlickVerse; metadata reviewed; content not independently reviewed."),

    ("2025_2", "boDI1t8quZE", "Movie Reaction",
     "CineDesi Part 1; metadata reviewed; content not independently reviewed.")
]

decisions = pd.DataFrame(
    approved_sources,
    columns=[
        "movie_record_id",
        "video_id",
        "manual_source_type",
        "manual_review_notes"
    ]
)

metadata = pd.read_csv(metadata_file)

required_metadata = [
    "movie_record_id", "movie_title", "movie_year", "language",
    "video_id", "metadata_title", "metadata_channel_title",
    "duration", "view_count", "comment_count"
]

missing = [c for c in required_metadata if c not in metadata.columns]
if missing:
    raise ValueError(f"Missing metadata columns: {missing}")

# Validate approved video IDs against retrieved metadata.
if decisions["video_id"].duplicated().any():
    raise ValueError("Duplicate video IDs in approved source decisions.")

merged = decisions.merge(
    metadata[required_metadata],
    on=["movie_record_id", "video_id"],
    how="left",
    validate="one_to_one",
    indicator=True
)

missing_matches = merged[merged["_merge"] != "both"]

if not missing_matches.empty:
    raise ValueError(
        "Some approved videos were not found in Batch 07 metadata:\n"
        + missing_matches[["movie_record_id", "video_id"]].to_string(index=False)
    )

merged = merged.drop(columns="_merge")

# Match the established Batch 06 manifest schema.
manifest = pd.DataFrame({
    "movie_record_id": merged["movie_record_id"],
    "movie_title": merged["movie_title"],
    "movie_year": merged["movie_year"],
    "language": merged["language"],
    "video_id": merged["video_id"],
    "video_title": merged["metadata_title"],
    "channel_title": merged["metadata_channel_title"],
    "duration": merged["duration"],
    "view_count": merged["view_count"],
    "comment_count": merged["comment_count"],
    "manual_source_type": merged["manual_source_type"],
    "manual_verification_status": "Metadata Reviewed",
    "manual_review_notes": merged["manual_review_notes"],
    "acquisition_status": "Pending",
    "target_comments": 100
})

# Final validation before saving.
assert len(manifest) == 12
assert manifest["video_id"].nunique() == 12
assert manifest["movie_record_id"].nunique() == 9
assert manifest["target_comments"].sum() == 1200
assert manifest["acquisition_status"].eq("Pending").all()

manifest.to_csv(output_file, index=False)

print("=" * 70)
print("BATCH 07 — APPROVED ACQUISITION MANIFEST")
print("=" * 70)

print("Approved videos:", len(manifest))
print("Unique movies:", manifest["movie_record_id"].nunique())
print("Unique video IDs:", manifest["video_id"].nunique())
print("Maximum comment target:", manifest["target_comments"].sum())

print("\nAPPROVED SOURCES BY MOVIE")
display(
    manifest.groupby(
        ["movie_record_id", "movie_title", "language"]
    ).agg(
        approved_videos=("video_id", "count"),
        target_comments=("target_comments", "sum")
    ).reset_index()
)

print("\nSOURCE MANIFEST")
display(
    manifest[
        [
            "movie_title",
            "video_id",
            "channel_title",
            "manual_source_type",
            "manual_verification_status",
            "acquisition_status"
        ]
    ]
)

print("\nSCHEMA:", manifest.columns.tolist())
print("\nSAVED:", output_file)

In [ ]:
import pandas as pd
import requests
import time
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata

# ============================================================
# 1. PATHS AND CONFIGURATION
# ============================================================

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

PROCESSED_AUDIENCE = (
    PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"
)

COMMENTS_DIR = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

manifest_file = (
    PROCESSED_AUDIENCE /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_07.csv"
)

raw_file = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_07.csv"
checkpoint_file = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_07_CHECKPOINT.csv"
log_file = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_07.csv"

COMMENTS_DIR.mkdir(parents=True, exist_ok=True)

# Never overwrite existing acquisition outputs.
for path in [raw_file, checkpoint_file, log_file]:
    if path.exists():
        raise FileExistsError(
            f"Output already exists. Inspect it before proceeding:\n{path}"
        )

if not manifest_file.exists():
    raise FileNotFoundError(manifest_file)

# Retrieve API key securely from Colab Secrets.
API_KEY = userdata.get("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY is missing from Colab Secrets.")

# ============================================================
# 2. LOAD AND VALIDATE APPROVED MANIFEST
# ============================================================

manifest = pd.read_csv(manifest_file)

required_columns = [
    "movie_record_id", "movie_title", "movie_year", "language",
    "video_id", "video_title", "channel_title",
    "target_comments", "acquisition_status"
]

missing = [c for c in required_columns if c not in manifest.columns]

if missing:
    raise ValueError(f"Manifest missing columns: {missing}")

if manifest["video_id"].duplicated().any():
    raise ValueError("Duplicate video IDs found in approved manifest.")

if not manifest["acquisition_status"].eq("Pending").all():
    raise ValueError("Some manifest sources are not Pending. Inspect before acquisition.")

if len(manifest) != 12:
    raise ValueError(f"Expected 12 approved sources, found {len(manifest)}.")

# ============================================================
# 3. OUTPUT SCHEMAS — MATCH BATCH 06
# ============================================================

comment_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "video_title",
    "channel_title",
    "comment_id",
    "comment_text",
    "author_name",
    "like_count",
    "published_at",
    "updated_at",
    "comment_position",
    "acquisition_batch",
    "retrieval_order"
]

log_columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "target_comments",
    "comments_collected",
    "request_count",
    "acquisition_status",
    "error_message",
    "retrieved_at_utc"
]

all_comments = []
acquisition_logs = []

# ============================================================
# 4. YOUTUBE COMMENT ACQUISITION
# ============================================================

API_URL = "https://www.googleapis.com/youtube/v3/commentThreads"

print("=" * 70)
print("BATCH 07 — YOUTUBE COMMENT ACQUISITION")
print("=" * 70)
print("Approved videos:", len(manifest))
print("Maximum target:", manifest["target_comments"].sum())
print("Sampling order: time")
print("Sampling unit: top-level comments")
print("=" * 70)

for index, row in manifest.iterrows():

    movie_id = row["movie_record_id"]
    movie_title = row["movie_title"]
    video_id = row["video_id"]
    target = int(row["target_comments"])

    collected = 0
    request_count = 0
    next_page_token = None
    error_message = ""
    api_error = False

    print(f"\n[{index + 1}/{len(manifest)}] {movie_title}")
    print("Video ID:", video_id)

    while collected < target:

        remaining = target - collected

        params = {
            "part": "snippet",
            "videoId": video_id,
            "maxResults": min(100, remaining),
            "order": "time",
            "textFormat": "plainText",
            "key": API_KEY
        }

        if next_page_token:
            params["pageToken"] = next_page_token

        try:
            response = requests.get(
                API_URL,
                params=params,
                timeout=45
            )

            request_count += 1
            data = response.json()

            if response.status_code != 200:
                api_error = True
                error_info = data.get("error", {})
                error_message = error_info.get(
                    "message", f"HTTP {response.status_code}"
                )
                print("API error:", error_message)
                break

            items = data.get("items", [])

            if not items:
                break

            for item in items:

                top_comment = (
                    item.get("snippet", {})
                    .get("topLevelComment", {})
                )

                snippet = top_comment.get("snippet", {})

                comment_text = snippet.get("textDisplay", "")

                all_comments.append({
                    "movie_record_id": movie_id,
                    "movie_title": movie_title,
                    "movie_year": row["movie_year"],
                    "language": row["language"],
                    "video_id": video_id,
                    "video_title": row["video_title"],
                    "channel_title": row["channel_title"],
                    "comment_id": top_comment.get("id"),
                    "comment_text": comment_text,
                    "author_name": snippet.get("authorDisplayName"),
                    "like_count": snippet.get("likeCount", 0),
                    "published_at": snippet.get("publishedAt"),
                    "updated_at": snippet.get("updatedAt"),
                    "comment_position": collected + 1,
                    "acquisition_batch": "Batch 07",
                    "retrieval_order": "time"
                })

                collected += 1

                if collected >= target:
                    break

            next_page_token = data.get("nextPageToken")

            if not next_page_token:
                break

            time.sleep(0.2)

        except requests.RequestException as exc:
            api_error = True
            error_message = str(exc)
            print("Request exception:", error_message)
            break

        except Exception as exc:
            api_error = True
            error_message = str(exc)
            print("Unexpected error:", error_message)
            break

    # Determine acquisition status.
    if api_error:
        status = "API Error"
    elif collected >= target:
        status = "Completed"
    elif collected == 0:
        status = "No More Comments"
    else:
        status = "Partial"

    retrieved_at = datetime.now(timezone.utc).isoformat()

    acquisition_logs.append({
        "movie_record_id": movie_id,
        "movie_title": movie_title,
        "video_id": video_id,
        "target_comments": target,
        "comments_collected": collected,
        "request_count": request_count,
        "acquisition_status": status,
        "error_message": error_message,
        "retrieved_at_utc": retrieved_at
    })

    # Save accumulated progress after every video.
    comments_df = pd.DataFrame(
        all_comments,
        columns=comment_columns
    )

    logs_df = pd.DataFrame(
        acquisition_logs,
        columns=log_columns
    )

    comments_df.to_csv(raw_file, index=False)
    comments_df.to_csv(checkpoint_file, index=False)
    logs_df.to_csv(log_file, index=False)

    print("Comments collected:", collected, "/", target)
    print("Requests:", request_count)
    print("Status:", status)

# ============================================================
# 5. FINAL ACQUISITION SUMMARY
# ============================================================

final_comments = pd.DataFrame(
    all_comments,
    columns=comment_columns
)

final_logs = pd.DataFrame(
    acquisition_logs,
    columns=log_columns
)

print("\n" + "=" * 70)
print("BATCH 07 — ACQUISITION SUMMARY")
print("=" * 70)

print("Videos processed:", len(final_logs))
print("Total comments collected:", len(final_comments))
print("Unique comment IDs:", final_comments["comment_id"].nunique())
print("Unique videos with comments:", final_comments["video_id"].nunique())
print("Target comments:", manifest["target_comments"].sum())

print("\nACQUISITION STATUS")
display(
    final_logs["acquisition_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="videos")
)

print("\nVIDEO-WISE RESULTS")
display(final_logs)

print("\nSAVED FILES")
print("Raw comments:", raw_file)
print("Checkpoint:", checkpoint_file)
print("Acquisition log:", log_file)

# API key is not printed or written to any file.
del API_KEY

In [ ]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

COMMENTS_DIR = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"
)

raw_file = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_07.csv"
manifest_file = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_07.csv"
)
log_file = (
    COMMENTS_DIR /
    "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_07.csv"
)

quality_file = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_07.csv"
)
movie_quality_file = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_07.csv"
)
video_quality_file = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_COMMENT_VIDEO_QUALITY_BATCH_07.csv"
)
duplicate_audit_file = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_BATCH_07.csv"
)

output_files = [
    quality_file,
    movie_quality_file,
    video_quality_file,
    duplicate_audit_file
]

for path in output_files:
    if path.exists():
        raise FileExistsError(
            f"Output already exists; inspect before proceeding:\n{path}"
        )

# ============================================================
# 2. LOAD FILES
# ============================================================

comments = pd.read_csv(raw_file)
manifest = pd.read_csv(manifest_file)
logs = pd.read_csv(log_file)

print("=" * 70)
print("BATCH 07 — DATA VALIDATION")
print("=" * 70)

print("Raw comments shape:", comments.shape)
print("Manifest shape:", manifest.shape)
print("Acquisition log shape:", logs.shape)

print("\nRaw comment columns:")
print(comments.columns.tolist())

# ============================================================
# 3. STRUCTURAL AND ACQUISITION VALIDATION
# ============================================================

required_columns = [
    "movie_record_id", "movie_title", "movie_year", "language",
    "video_id", "video_title", "channel_title", "comment_id",
    "comment_text", "author_name", "like_count", "published_at",
    "updated_at", "comment_position", "acquisition_batch",
    "retrieval_order"
]

missing_columns = [
    c for c in required_columns if c not in comments.columns
]

if missing_columns:
    raise ValueError(f"Missing raw comment columns: {missing_columns}")

expected_video_ids = set(manifest["video_id"].astype(str))
actual_video_ids = set(comments["video_id"].dropna().astype(str))

missing_video_ids = expected_video_ids - actual_video_ids
unexpected_video_ids = actual_video_ids - expected_video_ids

duplicate_comment_ids = int(
    comments["comment_id"].duplicated().sum()
)

empty_comment_ids = int(
    comments["comment_id"].isna().sum()
)

empty_comment_text = int(
    comments["comment_text"].fillna("").str.strip().eq("").sum()
)

status_counts = logs["acquisition_status"].value_counts().to_dict()

print("\nACQUISITION CHECKS")
print("-" * 70)
print("Expected approved videos:", len(expected_video_ids))
print("Videos represented in raw comments:", len(actual_video_ids))
print("Missing video IDs:", missing_video_ids)
print("Unexpected video IDs:", unexpected_video_ids)
print("Raw comment records:", len(comments))
print("Unique comment IDs:", comments["comment_id"].nunique())
print("Duplicate comment IDs:", duplicate_comment_ids)
print("Empty comment IDs:", empty_comment_ids)
print("Empty comment text:", empty_comment_text)
print("Acquisition statuses:", status_counts)

# Verify per-video counts against manifest targets.
video_counts = (
    comments.groupby("video_id")
    .size()
    .rename("actual_comments")
    .reset_index()
)

target_check = manifest[
    ["movie_record_id", "movie_title", "video_id", "target_comments"]
].merge(
    video_counts,
    on="video_id",
    how="left"
)

target_check["actual_comments"] = (
    target_check["actual_comments"].fillna(0).astype(int)
)

target_check["target_met"] = (
    target_check["actual_comments"] >= target_check["target_comments"]
)

print("\nVIDEO-WISE TARGET VALIDATION")
display(target_check)

# ============================================================
# 4. NORMALIZE TEXT FOR DUPLICATE-TEXT AUDIT
#    Raw comment_text remains unchanged.
# ============================================================

def normalize_text(value):
    if pd.isna(value):
        return ""

    value = unicodedata.normalize("NFKC", str(value))
    value = value.lower()
    value = re.sub(r"https?://\S+|www\.\S+", " ", value)
    value = re.sub(r"@\w+", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()

comments["normalized_text_audit"] = (
    comments["comment_text"].apply(normalize_text)
)

# Exclude empty normalized text from duplicate analysis.
nonempty = comments[
    comments["normalized_text_audit"].ne("")
].copy()

duplicate_groups = (
    nonempty.groupby("normalized_text_audit")
    .agg(
        record_count=("comment_id", "size"),
        unique_videos=("video_id", "nunique"),
        unique_movies=("movie_record_id", "nunique"),
        example_comment=("comment_text", "first")
    )
    .reset_index()
)

duplicate_groups = duplicate_groups[
    duplicate_groups["record_count"] > 1
].copy()

duplicate_groups["duplicate_scope"] = duplicate_groups.apply(
    lambda r: (
        "Across Videos"
        if r["unique_videos"] > 1
        else "Within Video"
    ),
    axis=1
)

duplicate_audit = duplicate_groups[
    [
        "normalized_text_audit",
        "record_count",
        "unique_videos",
        "unique_movies",
        "duplicate_scope",
        "example_comment"
    ]
].copy()

# Record-level count within repeated-text groups.
duplicate_text_records = int(
    duplicate_groups["record_count"].sum()
)

within_video_groups = int(
    duplicate_groups["duplicate_scope"].eq("Within Video").sum()
)

across_video_groups = int(
    duplicate_groups["duplicate_scope"].eq("Across Videos").sum()
)

# ============================================================
# 5. COMMENT LENGTH AND BASIC TEXT PROFILE
# ============================================================

comments["word_count"] = (
    comments["comment_text"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

comments["character_count"] = (
    comments["comment_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

very_short_comments = int(
    comments["word_count"].le(3).sum()
)

url_comments = int(
    comments["comment_text"]
    .fillna("")
    .str.contains(r"https?://|www\.", case=False, regex=True)
    .sum()
)

# Rough Unicode-script profile only; this is NOT language identification.
def script_category(text):
    text = str(text)

    has_devanagari = bool(re.search(r"[\u0900-\u097F]", text))
    has_tamil = bool(re.search(r"[\u0B80-\u0BFF]", text))
    has_telugu = bool(re.search(r"[\u0C00-\u0C7F]", text))
    has_kannada = bool(re.search(r"[\u0C80-\u0CFF]", text))
    has_bengali = bool(re.search(r"[\u0980-\u09FF]", text))
    has_latin = bool(re.search(r"[A-Za-z]", text))

    scripts = [
        has_devanagari, has_tamil, has_telugu,
        has_kannada, has_bengali, has_latin
    ]

    if sum(scripts) > 1:
        return "Mixed"
    if has_devanagari:
        return "Devanagari"
    if has_tamil:
        return "Tamil"
    if has_telugu:
        return "Telugu"
    if has_kannada:
        return "Kannada"
    if has_bengali:
        return "Bengali"
    if has_latin:
        return "Latin"
    return "Other/Emoji/Symbols"

comments["script_category"] = (
    comments["comment_text"].fillna("").apply(script_category)
)

script_distribution = (
    comments["script_category"]
    .value_counts()
    .to_dict()
)

# ============================================================
# 6. OVERALL QUALITY PROFILE
# ============================================================

quality_profile = pd.DataFrame([{
    "batch_id": "Batch 07",
    "total_comments": len(comments),
    "unique_comment_ids": comments["comment_id"].nunique(),
    "duplicate_comment_ids": duplicate_comment_ids,
    "empty_comment_ids": empty_comment_ids,
    "empty_comment_text": empty_comment_text,
    "unique_videos": comments["video_id"].nunique(),
    "unique_movies": comments["movie_record_id"].nunique(),
    "very_short_comments_le_3_words": very_short_comments,
    "comments_containing_urls": url_comments,
    "average_word_count": round(comments["word_count"].mean(), 2),
    "median_word_count": round(comments["word_count"].median(), 2),
    "average_character_count": round(comments["character_count"].mean(), 2),
    "duplicate_text_groups": len(duplicate_groups),
    "records_in_duplicate_text_groups": duplicate_text_records,
    "within_video_duplicate_groups": within_video_groups,
    "across_video_duplicate_groups": across_video_groups,
    "script_distribution": str(script_distribution)
}])

# ============================================================
# 7. MOVIE-WISE AND VIDEO-WISE QUALITY
# ============================================================

movie_quality = (
    comments.groupby(
        ["movie_record_id", "movie_title", "language"],
        dropna=False
    )
    .agg(
        comment_count=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        average_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        average_characters=("character_count", "mean"),
        very_short_comments=("word_count", lambda s: int(s.le(3).sum()))
    )
    .reset_index()
)

movie_quality["average_words"] = movie_quality["average_words"].round(2)
movie_quality["median_words"] = movie_quality["median_words"].round(2)
movie_quality["average_characters"] = (
    movie_quality["average_characters"].round(2)
)

video_quality = (
    comments.groupby(
        ["movie_record_id", "movie_title", "video_id", "video_title"],
        dropna=False
    )
    .agg(
        comment_count=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        average_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        average_characters=("character_count", "mean"),
        very_short_comments=("word_count", lambda s: int(s.le(3).sum()))
    )
    .reset_index()
)

video_quality["average_words"] = video_quality["average_words"].round(2)
video_quality["median_words"] = video_quality["median_words"].round(2)
video_quality["average_characters"] = (
    video_quality["average_characters"].round(2)
)

# ============================================================
# 8. SAVE AUDIT OUTPUTS
# ============================================================

quality_profile.to_csv(quality_file, index=False)
movie_quality.to_csv(movie_quality_file, index=False)
video_quality.to_csv(video_quality_file, index=False)
duplicate_audit.to_csv(duplicate_audit_file, index=False)

# ============================================================
# 9. FINAL VALIDATION RESULT
# ============================================================

checks = {
    "Expected 12 approved videos": len(manifest) == 12,
    "All approved videos represented": len(missing_video_ids) == 0,
    "No unexpected videos": len(unexpected_video_ids) == 0,
    "Exactly 1200 comments": len(comments) == 1200,
    "All comment IDs unique": duplicate_comment_ids == 0,
    "No empty comment IDs": empty_comment_ids == 0,
    "No empty comment text": empty_comment_text == 0,
    "All video targets met": bool(target_check["target_met"].all()),
    "All API acquisitions completed": (
        logs["acquisition_status"].eq("Completed").all()
    )
}

print("\n" + "=" * 70)
print("BATCH 07 — FINAL VALIDATION")
print("=" * 70)

for check, passed in checks.items():
    print(f"{'PASS' if passed else 'CHECK'} | {check}")

print("\nOVERALL QUALITY PROFILE")
display(quality_profile.T.rename(columns={0: "value"}))

print("\nMOVIE-WISE QUALITY")
display(movie_quality)

print("\nVIDEO-WISE QUALITY")
display(video_quality)

print("\nDUPLICATE TEXT SUMMARY")
print("Repeated-text groups:", len(duplicate_groups))
print("Records in repeated-text groups:", duplicate_text_records)
print("Within-video groups:", within_video_groups)
print("Across-video groups:", across_video_groups)

print("\nSAVED REPORTS")
for path in output_files:
    print(path)

print("\nRaw acquisition file was not modified.")

In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

RAW_DIR = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"
)

old_file = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_06.csv"
)

new_file = RAW_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_07.csv"

# Intended new outputs; originals will remain untouched.
new_consolidated_file = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_07.csv"
)

new_coverage_file = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_07.csv"
)

new_duplicate_file = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_PILOT_BATCH_01_07.csv"
)

# ============================================================
# 2. FILE EXISTENCE CHECK
# ============================================================

for path in [old_file, new_file]:
    if not path.exists():
        raise FileNotFoundError(path)

for path in [
    new_consolidated_file,
    new_coverage_file,
    new_duplicate_file
]:
    if path.exists():
        raise FileExistsError(
            f"New output already exists. Inspect before proceeding:\n{path}"
        )

# ============================================================
# 3. LOAD DATA
# ============================================================

old = pd.read_csv(old_file)
batch07 = pd.read_csv(new_file)

print("=" * 70)
print("PILOT–BATCH 06 + BATCH 07 CONSOLIDATION PREFLIGHT")
print("=" * 70)

print("\nEXISTING CONSOLIDATED DATASET")
print("Shape:", old.shape)
print("Unique comment IDs:", old["comment_id"].nunique())
print("Unique videos:", old["video_id"].nunique())
print("Unique movies:", old["movie_record_id"].nunique())

print("\nBATCH 07 RAW DATA")
print("Shape:", batch07.shape)
print("Unique comment IDs:", batch07["comment_id"].nunique())
print("Unique videos:", batch07["video_id"].nunique())
print("Unique movies:", batch07["movie_record_id"].nunique())

# ============================================================
# 4. SCHEMA COMPARISON
# ============================================================

old_columns = old.columns.tolist()
new_columns = batch07.columns.tolist()

print("\nEXISTING CONSOLIDATED COLUMNS")
print(old_columns)

print("\nBATCH 07 RAW COLUMNS")
print(new_columns)

print("\nCOLUMNS COMMON TO BOTH")
print(sorted(set(old_columns) & set(new_columns)))

print("\nCOLUMNS ONLY IN EXISTING DATASET")
print(sorted(set(old_columns) - set(new_columns)))

print("\nCOLUMNS ONLY IN BATCH 07")
print(sorted(set(new_columns) - set(old_columns)))

# ============================================================
# 5. COMMENT ID OVERLAP
# ============================================================

old_ids = set(old["comment_id"].dropna().astype(str))
new_ids = set(batch07["comment_id"].dropna().astype(str))

overlap_ids = old_ids & new_ids

print("\nCOMMENT ID OVERLAP")
print("Existing IDs:", len(old_ids))
print("Batch 07 IDs:", len(new_ids))
print("Overlapping IDs:", len(overlap_ids))

if overlap_ids:
    print("Sample overlapping IDs:", list(overlap_ids)[:10])

# ============================================================
# 6. BATCH DISTRIBUTION IN EXISTING DATA
# ============================================================

if "source_batch" in old.columns:
    print("\nEXISTING SOURCE BATCH DISTRIBUTION")
    display(
        old["source_batch"]
        .value_counts(dropna=False)
        .rename_axis("source_batch")
        .reset_index(name="records")
    )

elif "batch_id" in old.columns:
    print("\nEXISTING BATCH DISTRIBUTION")
    display(
        old["batch_id"]
        .value_counts(dropna=False)
        .rename_axis("batch_id")
        .reset_index(name="records")
    )

# ============================================================
# 7. BATCH 07 MOVIE AND VIDEO COVERAGE
# ============================================================

print("\nBATCH 07 MOVIE COVERAGE")
display(
    batch07.groupby(
        ["movie_record_id", "movie_title", "language"],
        dropna=False
    )
    .agg(
        comments=("comment_id", "size"),
        videos=("video_id", "nunique")
    )
    .reset_index()
)

print("\nBATCH 07 VIDEO COVERAGE")
display(
    batch07.groupby(
        ["movie_title", "video_id"],
        dropna=False
    )
    .size()
    .reset_index(name="comments")
)

# ============================================================
# 8. PREFLIGHT DECISION
# ============================================================

checks = {
    "Existing dataset has comment_id": "comment_id" in old.columns,
    "Batch 07 has comment_id": "comment_id" in batch07.columns,
    "Existing IDs are unique": old["comment_id"].is_unique,
    "Batch 07 IDs are unique": batch07["comment_id"].is_unique,
    "No comment ID overlap": len(overlap_ids) == 0,
    "Batch 07 contains 1200 records": len(batch07) == 1200,
    "Existing consolidated file exists": old_file.exists(),
    "New output paths are available": not any(
        p.exists() for p in [
            new_consolidated_file,
            new_coverage_file,
            new_duplicate_file
        ]
    )
}

print("\n" + "=" * 70)
print("PREFLIGHT CHECKS")
print("=" * 70)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'CHECK'} | {name}")

print("\nNo files were modified.")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

print("=" * 70)
print("LOCATING ICMB-130 MASTER DATASET")
print("=" * 70)

# Search project data folders for likely master/reference files.
search_roots = [
    PROJECT_ROOT / "01_data" / "raw",
    PROJECT_ROOT / "01_data" / "processed"
]

matches = []

for root in search_roots:
    if root.exists():
        for path in root.rglob("*.csv"):
            name = path.name.lower()

            if (
                "icmb" in name
                and ("master" in name or "dataset" in name or "final" in name)
            ):
                matches.append(path)

if matches:
    for i, path in enumerate(matches, 1):
        print(f"\n{i}. {path}")
        print(f"   Size: {path.stat().st_size / 1024:.2f} KB")
else:
    print("No likely master dataset CSV found.")

print("\n" + "=" * 70)
print("CHECKING RECENTLY USED CONSOLIDATED FILES")
print("=" * 70)

for path in [
    PROJECT_ROOT / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_06.csv",

    PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube/comments/"
    / "ICMB_130_YOUTUBE_COMMENT_BATCH_07.csv"
]:
    print(path.name, "| Exists:", path.exists())

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

master_file = (
    PROJECT_ROOT / "01_data" / "processed" /
    "ICMB_130_MASTER_DATA.csv"
)

master = pd.read_csv(master_file)

print("=" * 70)
print("ICMB-130 MASTER DATASET — SCHEMA VERIFICATION")
print("=" * 70)

print("File:", master_file)
print("Shape:", master.shape)

print("\nColumns:")
for i, col in enumerate(master.columns, 1):
    print(f"{i:02d}. {col}")

print("\nFirst five records:")
display(master.head())

print("\nPotential movie identifier columns:")
print([
    c for c in master.columns
    if any(k in c.lower() for k in ["movie_record_id", "record_id", "movie_id"])
])

print("\nPotential title, year, rank and language columns:")
for keyword in ["title", "year", "rank", "language"]:
    print(
        keyword,
        "->",
        [c for c in master.columns if keyword in c.lower()]
    )

print("\nDuplicate movie identifiers:")
for col in master.columns:
    if "movie_record_id" in col.lower():
        print(col, ":", master[col].duplicated().sum())

print("\nRaw master dataset was not modified.")

In [ ]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = (
    PROJECT_ROOT / "01_data" / "raw" /
    "audience_sentiment" / "youtube" / "comments"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data" / "processed" / "audience_sentiment"
)

old_file = PROCESSED_DIR / (
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_06.csv"
)

batch07_file = RAW_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_07.csv"

master_file = (
    PROJECT_ROOT / "01_data" / "processed" /
    "ICMB_130_MASTER_DATA.csv"
)

log_file = RAW_DIR / (
    "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_07.csv"
)

manifest_file = PROCESSED_DIR / (
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_07.csv"
)

output_file = PROCESSED_DIR / (
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_07.csv"
)

coverage_file = PROCESSED_DIR / (
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_07.csv"
)

duplicate_file = PROCESSED_DIR / (
    "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_PILOT_BATCH_01_07.csv"
)

# Protect existing outputs.
for path in [
    output_file,
    coverage_file,
    duplicate_file
]:
    if path.exists():
        raise FileExistsError(
            f"Output already exists. Inspect before proceeding:\n{path}"
        )

for path in [
    old_file,
    batch07_file,
    master_file,
    log_file,
    manifest_file
]:
    if not path.exists():
        raise FileNotFoundError(path)

# ============================================================
# 2. LOAD SOURCE FILES
# ============================================================

old = pd.read_csv(old_file)
batch07 = pd.read_csv(batch07_file)
master = pd.read_csv(master_file)
batch07_log = pd.read_csv(log_file)
manifest = pd.read_csv(manifest_file)

print("=" * 70)
print("PILOT–BATCH 07 CONSOLIDATION")
print("=" * 70)

print("Existing consolidated:", old.shape)
print("Batch 07 raw:", batch07.shape)
print("Master reference:", master.shape)

# ============================================================
# 3. VALIDATE SOURCE DATA
# ============================================================

if not old["comment_id"].is_unique:
    raise ValueError("Existing consolidated comment IDs are not unique.")

if not batch07["comment_id"].is_unique:
    raise ValueError("Batch 07 comment IDs are not unique.")

if len(batch07) != 1200:
    raise ValueError(f"Expected 1200 Batch 07 records, got {len(batch07)}.")

if len(master) != 130:
    raise ValueError(f"Expected 130 reference movies, got {len(master)}.")

if master["movie_record_id"].duplicated().any():
    raise ValueError("Duplicate movie_record_id values in master dataset.")

overlap = (
    set(old["comment_id"].dropna().astype(str))
    & set(batch07["comment_id"].dropna().astype(str))
)

if overlap:
    raise ValueError(f"Comment ID overlap detected: {len(overlap)}")

# Verify Batch 07 source videos against approved manifest.
if set(batch07["video_id"].astype(str)) != set(manifest["video_id"].astype(str)):
    raise ValueError("Batch 07 video IDs do not match approved manifest.")

# ============================================================
# 4. STANDARDIZE BATCH 07 TO EXISTING 32-COLUMN SCHEMA
# ============================================================

schema = old.columns.tolist()

new = pd.DataFrame(index=batch07.index)

for col in schema:
    new[col] = pd.NA

# Direct mappings.
direct_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "video_id",
    "video_title",
    "channel_title",
    "comment_id",
    "comment_text",
    "author_name",
    "like_count",
    "published_at",
    "updated_at",
    "comment_position",
    "retrieval_order"
]

for col in direct_columns:
    if col in batch07.columns and col in new.columns:
        new[col] = batch07[col]

# Source and sampling information.
new["source_batch"] = "Batch 07"
new["batch_id"] = "Batch 07"
new["retrieval_order"] = "time"
new["sampling_order"] = "time"
new["acquisition_order"] = "time"
new["sampling_unit"] = "top_level_comment"
new["comment_source"] = "YouTube Top-Level Comment"

# Author fields.
new["author_display_name"] = batch07["author_name"]
new["comment_author"] = batch07["author_name"]

# Reply count and author channel ID are unavailable in Batch 07 raw data.
# These remain null.

# Acquisition timestamps from actual acquisition log.
timestamp_map = (
    batch07_log.set_index("video_id")["retrieved_at_utc"].to_dict()
)

new["retrieved_at"] = batch07["video_id"].map(timestamp_map)
new["acquired_at_utc"] = batch07["video_id"].map(timestamp_map)
new["acquisition_timestamp_utc"] = batch07["video_id"].map(timestamp_map)

# Source type from approved source manifest.
source_type_map = (
    manifest.set_index("video_id")["manual_source_type"].to_dict()
)

new["source_type"] = batch07["video_id"].map(source_type_map)

# Extract part information only when explicitly stated in the video title.
def extract_series_part(title):
    if pd.isna(title):
        return pd.NA

    match = re.search(
        r"\bpart\s*(\d+)\s*(?:/|of)\s*(\d+)\b",
        str(title),
        flags=re.IGNORECASE
    )

    if match:
        return f"{match.group(1)}/{match.group(2)}"

    return pd.NA

new["series_part"] = batch07["video_title"].apply(extract_series_part)

# Normalize text for duplicate analysis; preserve original comment_text.
def normalize_comment(value):
    if pd.isna(value):
        return ""

    value = unicodedata.normalize("NFKC", str(value))
    value = value.lower()
    value = re.sub(r"https?://\S+|www\.\S+", " ", value)
    value = re.sub(r"@\w+", " ", value)
    value = re.sub(r"\s+", " ", value)

    return value.strip()

new["normalized_comment_text"] = (
    batch07["comment_text"].apply(normalize_comment)
)

# Match the exact previous schema.
new = new[schema]

# ============================================================
# 5. CONSOLIDATE
# ============================================================

consolidated = pd.concat(
    [old, new],
    ignore_index=True
)

if len(consolidated) != len(old) + len(batch07):
    raise ValueError("Unexpected consolidated record count.")

if not consolidated["comment_id"].is_unique:
    raise ValueError("Duplicate comment IDs after consolidation.")

if consolidated.columns.tolist() != schema:
    raise ValueError("Consolidated schema mismatch.")

# ============================================================
# 6. BUILD MOVIE COVERAGE USING MASTER REFERENCE
# ============================================================

comment_coverage = (
    consolidated.groupby("movie_record_id")
    .agg(
        comment_records=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        acquisition_batches=("source_batch", "nunique")
    )
    .reset_index()
)

coverage = master[
    [
        "movie_record_id",
        "movie_title",
        "year",
        "rank",
        "language"
    ]
].merge(
    comment_coverage,
    on="movie_record_id",
    how="left"
)

count_columns = [
    "comment_records",
    "unique_comment_ids",
    "unique_videos",
    "acquisition_batches"
]

coverage[count_columns] = (
    coverage[count_columns].fillna(0).astype(int)
)

coverage["coverage_status"] = coverage["comment_records"].apply(
    lambda n: "Covered" if n > 0 else "No Comments"
)

coverage["coverage_category"] = coverage["comment_records"].apply(
    lambda n: (
        "No Comments" if n == 0
        else "Low (1-49)" if n < 50
        else "Moderate (50-99)" if n < 100
        else "Higher (100+)"
    )
)

# ============================================================
# 7. DUPLICATE TEXT AUDIT
# ============================================================

duplicate_groups = (
    consolidated[
        consolidated["normalized_comment_text"].fillna("").ne("")
    ]
    .groupby("normalized_comment_text")
    .agg(
        record_count=("comment_id", "size"),
        unique_videos=("video_id", "nunique"),
        unique_movies=("movie_record_id", "nunique"),
        example_comment=("comment_text", "first")
    )
    .reset_index()
)

duplicate_groups = duplicate_groups[
    duplicate_groups["record_count"] > 1
].copy()

duplicate_groups["duplicate_scope"] = duplicate_groups.apply(
    lambda r: (
        "Across Videos"
        if r["unique_videos"] > 1
        else "Within Video"
    ),
    axis=1
)

duplicate_audit = duplicate_groups[
    [
        "normalized_comment_text",
        "record_count",
        "unique_videos",
        "unique_movies",
        "duplicate_scope",
        "example_comment"
    ]
].copy()

# ============================================================
# 8. FINAL VALIDATION BEFORE SAVING
# ============================================================

checks = {
    "Consolidated rows equal source sum": (
        len(consolidated) == len(old) + len(batch07)
    ),
    "Consolidated IDs are unique": (
        consolidated["comment_id"].is_unique
    ),
    "Consolidated schema has 32 columns": (
        len(consolidated.columns) == 32
    ),
    "Reference coverage contains 130 movies": (
        len(coverage) == 130
    ),
    "All Batch 07 movies are in master reference": (
        set(batch07["movie_record_id"]).issubset(
            set(master["movie_record_id"])
        )
    ),
    "Batch 07 source types are populated": (
        new["source_type"].notna().all()
    )
}

print("\nPRE-SAVE VALIDATION")
for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

if not all(checks.values()):
    raise ValueError("Validation failed. No output files saved.")

# ============================================================
# 9. SAVE VERSIONED OUTPUTS
# ============================================================

consolidated.to_csv(output_file, index=False)
coverage.to_csv(coverage_file, index=False)
duplicate_audit.to_csv(duplicate_file, index=False)

# ============================================================
# 10. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PILOT–BATCH 07 CONSOLIDATION COMPLETE")
print("=" * 70)

print("Total comments:", len(consolidated))
print("Unique comment IDs:", consolidated["comment_id"].nunique())
print("Unique videos:", consolidated["video_id"].nunique())
print("Unique movies with comments:", consolidated["movie_record_id"].nunique())
print("Reference movies:", len(coverage))
print("Movies covered:", (coverage["comment_records"] > 0).sum())
print("Movies uncovered:", (coverage["comment_records"] == 0).sum())

print("\nSOURCE BATCH DISTRIBUTION")
display(
    consolidated["source_batch"]
    .value_counts()
    .rename_axis("source_batch")
    .reset_index(name="records")
)

print("\nCOVERAGE CATEGORY")
display(
    coverage["coverage_category"]
    .value_counts()
    .rename_axis("coverage_category")
    .reset_index(name="movies")
)

print("\nBATCH 07 MOVIE COVERAGE")
display(
    coverage[
        coverage["movie_record_id"].isin(
            batch07["movie_record_id"].unique()
        )
    ][
        [
            "movie_record_id",
            "movie_title",
            "language",
            "comment_records",
            "unique_videos",
            "coverage_category"
        ]
    ]
)

print("\nDUPLICATE TEXT AUDIT")
print("Repeated-text groups:", len(duplicate_groups))
print("Records in repeated-text groups:", int(duplicate_groups["record_count"].sum()))
print(
    "Within-video groups:",
    int(duplicate_groups["duplicate_scope"].eq("Within Video").sum())
)
print(
    "Across-video groups:",
    int(duplicate_groups["duplicate_scope"].eq("Across Videos").sum())
)

print("\nSAVED OUTPUTS")
print(output_file)
print(coverage_file)
print(duplicate_file)

print("\nOriginal Pilot–Batch 06 consolidated file and Batch 07 raw data remain unchanged.")

In [ ]:
import os
import pandas as pd
from IPython.display import display

# ============================================================
# NOTEBOOK 10 — BATCH 07 PLANNING REPORT PREFLIGHT
# ============================================================

PROJECT_ROOT = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "audience_sentiment"
)

MASTER_PATH = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "ICMB_130_MASTER_DATA.csv"
)

FILES = {
    "consolidated": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_07.csv"
    ),
    "coverage": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_07.csv"
    ),
    "old_planning_report": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_06.csv"
    ),
    "old_planning_summary": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_06.csv"
    ),
    "old_language_summary": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_06.csv"
    )
}

print("=" * 75)
print("BATCH 07 PLANNING REPORT PREFLIGHT")
print("=" * 75)

# 1. Verify files
for name, path in FILES.items():
    print(f"{name}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

print(
    f"master_dataset: "
    f"{'FOUND' if os.path.exists(MASTER_PATH) else 'MISSING'}"
)

required_paths = [
    FILES["consolidated"],
    FILES["coverage"],
    MASTER_PATH
]

missing = [p for p in required_paths if not os.path.exists(p)]

if missing:
    raise FileNotFoundError(
        "Required files are missing:\n" + "\n".join(missing)
    )

# 2. Load latest datasets
comments = pd.read_csv(FILES["consolidated"])
coverage = pd.read_csv(FILES["coverage"])
master = pd.read_csv(MASTER_PATH)

print("\n" + "=" * 75)
print("LATEST DATASET SHAPES")
print("=" * 75)

print("Consolidated comments:", comments.shape)
print("Movie coverage:", coverage.shape)
print("Master dataset:", master.shape)

# 3. Inspect schemas
for name, df in [
    ("CONSOLIDATED COMMENTS", comments),
    ("MOVIE COVERAGE", coverage),
    ("MASTER DATASET", master)
]:
    print(f"\n{name} COLUMNS:")
    print(df.columns.tolist())

# 4. Inspect previous planning report schemas
for key in [
    "old_planning_report",
    "old_planning_summary",
    "old_language_summary"
]:
    path = FILES[key]

    if os.path.exists(path):
        old_df = pd.read_csv(path)

        print("\n" + "=" * 75)
        print(key.upper())
        print("=" * 75)
        print("Shape:", old_df.shape)
        print("Columns:", old_df.columns.tolist())
        display(old_df.head(5))

# 5. Latest coverage distribution
print("\n" + "=" * 75)
print("LATEST COVERAGE DISTRIBUTION")
print("=" * 75)

coverage_column = next(
    (
        col for col in [
            "coverage_category",
            "coverage_status",
            "coverage_level"
        ]
        if col in coverage.columns
    ),
    None
)

if coverage_column:
    display(
        coverage[coverage_column]
        .value_counts(dropna=False)
        .rename_axis("coverage_category")
        .reset_index(name="movie_count")
    )
else:
    print("Coverage category column not identified.")
    display(coverage.head(10))

# 6. Comment provenance
print("\n" + "=" * 75)
print("COMMENT SOURCE BATCH DISTRIBUTION")
print("=" * 75)

if "source_batch" in comments.columns:
    display(
        comments["source_batch"]
        .value_counts(dropna=False)
        .rename_axis("source_batch")
        .reset_index(name="comment_count")
    )

# 7. Language distribution in master dataset
print("\n" + "=" * 75)
print("MASTER DATASET LANGUAGE DISTRIBUTION")
print("=" * 75)

if "language" in master.columns:
    display(
        master["language"]
        .value_counts(dropna=False)
        .rename_axis("language")
        .reset_index(name="movie_count")
    )

print("\nPreflight completed. No files were modified.")

In [ ]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# CORRECTED — PLANNING REPORTS PILOT + BATCH 01–07
# ============================================================

PROJECT_ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
PROCESSED_DIR = os.path.join(
    PROJECT_ROOT, "01_data", "processed", "audience_sentiment"
)

COMMENTS_PATH = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_07.csv"
)
COVERAGE_PATH = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_07.csv"
)
MASTER_PATH = os.path.join(
    PROJECT_ROOT, "01_data", "processed", "ICMB_130_MASTER_DATA.csv"
)

OUTPUTS = {
    "report": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_07.csv"
    ),
    "summary": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_07.csv"
    ),
    "language": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_07.csv"
    )
}

TARGET_PER_MOVIE = 100

# 1. Load source files
comments = pd.read_csv(COMMENTS_PATH)
coverage = pd.read_csv(COVERAGE_PATH)
master = pd.read_csv(MASTER_PATH)

# 2. Protect existing outputs
existing = [p for p in OUTPUTS.values() if os.path.exists(p)]
if existing:
    raise FileExistsError(
        "These output files already exist; no files were overwritten:\n"
        + "\n".join(existing)
    )

# 3. Calculate acquisition batches from comment provenance
batch_counts = (
    comments.groupby("movie_record_id")["source_batch"]
    .nunique()
    .rename("actual_acquisition_batches")
    .reset_index()
)

# 4. Remove the old acquisition_batches column before merging.
# This prevents Pandas from creating acquisition_batches_x/y.
coverage_clean = coverage.drop(
    columns=["acquisition_batches"],
    errors="ignore"
)

planning = master[
    ["movie_record_id", "movie_title", "year", "rank", "language"]
].merge(
    coverage_clean.drop(
        columns=["movie_title", "year", "rank", "language"],
        errors="ignore"
    ),
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

planning = planning.merge(
    batch_counts,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

planning["acquisition_batches"] = (
    planning["actual_acquisition_batches"]
    .fillna(0)
    .astype(int)
)

planning.drop(
    columns=["actual_acquisition_batches"],
    inplace=True
)

# 5. Normalize numeric coverage fields
for col in [
    "comment_records",
    "unique_comment_ids",
    "unique_videos"
]:
    planning[col] = planning[col].fillna(0).astype(int)

# 6. Derive coverage category and status
def classify_coverage(count):
    if count == 0:
        return "No Comments"
    elif count < 50:
        return "Low (1-49)"
    elif count < 100:
        return "Moderate (50-99)"
    return "Higher (100+)"

planning["coverage_category"] = (
    planning["comment_records"].apply(classify_coverage)
)

planning["coverage_status"] = np.where(
    planning["comment_records"] > 0,
    "Covered",
    "Uncovered"
)

# 7. Calculate provisional additional target
planning["additional_comment_target"] = (
    TARGET_PER_MOVIE - planning["comment_records"]
).clip(lower=0)

def assign_priority(row):
    count = row["comment_records"]
    if count == 0:
        return "High — No Comments"
    elif count < 50:
        return "High — Low Coverage"
    elif count < 100:
        return "Medium — Moderate Coverage"
    return "Covered — Monitor"

planning["acquisition_priority"] = planning.apply(
    assign_priority,
    axis=1
)

# 8. Arrange report columns
planning = planning[
    [
        "movie_record_id",
        "movie_title",
        "year",
        "rank",
        "language",
        "comment_records",
        "unique_comment_ids",
        "unique_videos",
        "acquisition_batches",
        "coverage_status",
        "coverage_category",
        "acquisition_priority",
        "additional_comment_target"
    ]
].sort_values(
    ["year", "rank"]
).reset_index(drop=True)

# 9. Coverage summary
category_order = [
    "No Comments",
    "Low (1-49)",
    "Moderate (50-99)",
    "Higher (100+)"
]

planning_summary = (
    planning.groupby("coverage_category")
    .agg(
        movie_count=("movie_record_id", "count"),
        existing_comments=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reindex(category_order, fill_value=0)
    .reset_index()
)

# 10. Language summary
language_summary = (
    planning.groupby("language")
    .agg(
        total_movies=("movie_record_id", "count"),
        covered_movies=(
            "comment_records", lambda x: int((x > 0).sum())
        ),
        uncovered_movies=(
            "comment_records", lambda x: int((x == 0).sum())
        ),
        comment_records=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reset_index()
    .sort_values("language")
    .reset_index(drop=True)
)

# 11. Validation
assert len(planning) == 130
assert planning["movie_record_id"].is_unique
assert planning["comment_records"].sum() == len(comments)
assert planning["unique_comment_ids"].sum() == comments["comment_id"].nunique()
assert planning["acquisition_batches"].sum() >= 0
assert planning_summary["movie_count"].sum() == 130
assert language_summary["total_movies"].sum() == 130

expected = {
    "No Comments": 77,
    "Low (1-49)": 1,
    "Moderate (50-99)": 1,
    "Higher (100+)": 51
}

actual = planning["coverage_category"].value_counts().to_dict()

for category, count in expected.items():
    assert actual.get(category, 0) == count

# 12. Save versioned outputs
planning.to_csv(OUTPUTS["report"], index=False)
planning_summary.to_csv(OUTPUTS["summary"], index=False)
language_summary.to_csv(OUTPUTS["language"], index=False)

# 13. Display results
print("=" * 70)
print("PLANNING REPORTS — PILOT + BATCH 01–07")
print("=" * 70)

print("\nOverall:")
print("Reference movies:", len(planning))
print("Consolidated comments:", len(comments))
print("Unique videos:", comments["video_id"].nunique())
print("Movies covered:", (planning["comment_records"] > 0).sum())
print("Movies uncovered:", (planning["comment_records"] == 0).sum())
print("Total provisional additional target:",
      planning["additional_comment_target"].sum())

print("\nCoverage summary:")
display(planning_summary)

print("\nLanguage summary:")
display(language_summary)

print("\nPartially covered movies:")
display(
    planning.loc[
        planning["comment_records"].between(1, 99),
        [
            "movie_record_id", "movie_title", "language",
            "comment_records", "acquisition_batches",
            "additional_comment_target", "acquisition_priority"
        ]
    ]
)

print("\nFiles saved:")
for name, path in OUTPUTS.items():
    print(name, "—", os.path.exists(path), "—", path)

print("\nAll validation checks passed.")

In [ ]:
import os
import pandas as pd
from IPython.display import display

# ============================================================
# NOTEBOOK 10 — UNCOVERED MOVIE PRIORITY REVIEW
# PILOT + BATCH 01–07
# ============================================================

PROJECT_ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT,
    "01_data",
    "processed",
    "audience_sentiment"
)

REPORT_PATH = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_07.csv"
)

if not os.path.exists(REPORT_PATH):
    raise FileNotFoundError(REPORT_PATH)

planning = pd.read_csv(REPORT_PATH)

# ------------------------------------------------------------
# 1. ISOLATE UNCOVERED MOVIES
# ------------------------------------------------------------

uncovered = planning[
    planning["comment_records"] == 0
].copy()

uncovered = uncovered.sort_values(
    ["language", "year", "rank"]
).reset_index(drop=True)

print("=" * 75)
print("UNCOVERED MOVIE REVIEW — PILOT + BATCH 01–07")
print("=" * 75)

print("Total reference movies:", len(planning))
print("Movies without comments:", len(uncovered))

# ------------------------------------------------------------
# 2. LANGUAGE-WISE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("UNCOVERED MOVIES BY LANGUAGE")
print("=" * 75)

language_summary = (
    uncovered.groupby("language")
    .agg(
        uncovered_movies=("movie_record_id", "count"),
        first_year=("year", "min"),
        last_year=("year", "max")
    )
    .reset_index()
    .sort_values("uncovered_movies", ascending=False)
)

display(language_summary)

# ------------------------------------------------------------
# 3. YEAR-WISE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("UNCOVERED MOVIES BY YEAR")
print("=" * 75)

year_summary = (
    uncovered.groupby("year")
    .agg(
        uncovered_movies=("movie_record_id", "count"),
        languages=("language", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
    .sort_values("year")
)

display(year_summary)

# ------------------------------------------------------------
# 4. YEAR × LANGUAGE DISTRIBUTION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("YEAR × LANGUAGE DISTRIBUTION")
print("=" * 75)

year_language = pd.crosstab(
    uncovered["year"],
    uncovered["language"]
)

display(year_language)

# ------------------------------------------------------------
# 5. COMPLETE LIST OF 77 UNCOVERED MOVIES
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("COMPLETE UNCOVERED MOVIE LIST")
print("=" * 75)

display(
    uncovered[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "rank",
            "language",
            "india_gross_cr"
        ]
    ] if "india_gross_cr" in uncovered.columns
    else uncovered[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "rank",
            "language",
            "comment_records",
            "additional_comment_target"
        ]
    ]
)

# ------------------------------------------------------------
# 6. HIGHEST PRIORITY YEARS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("YEARS WITH THE MOST UNCOVERED MOVIES")
print("=" * 75)

display(
    year_summary.sort_values(
        "uncovered_movies",
        ascending=False
    ).head(10)
)

print("\nInspection completed. No files were modified.")

In [ ]:
import os
import pandas as pd
from IPython.display import display

# ============================================================
# NOTEBOOK 10 — COMPLETE UNCOVERED MOVIE CANDIDATE REVIEW
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PLANNING_PATH = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment",
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_07.csv"
)

MASTER_PATH = os.path.join(
    ROOT, "01_data", "processed",
    "ICMB_130_MASTER_DATA.csv"
)

planning = pd.read_csv(PLANNING_PATH)
master = pd.read_csv(MASTER_PATH)

uncovered = planning.loc[
    planning["comment_records"] == 0
].copy()

# Add India Gross and verification information
gross_columns = [
    "movie_record_id",
    "india_gross_cr",
    "india_gross_source",
    "india_gross_verified"
]

uncovered = uncovered.merge(
    master[gross_columns],
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

uncovered = uncovered.sort_values(
    ["year", "rank"]
).reset_index(drop=True)

print("=" * 80)
print("ALL UNCOVERED MOVIES — 77 CANDIDATES")
print("=" * 80)

print("Total:", len(uncovered))
print("Unique movie IDs:", uncovered["movie_record_id"].nunique())

# Display all rows without truncation
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.width", 200
):
    display(
        uncovered[
            [
                "movie_record_id",
                "movie_title",
                "year",
                "rank",
                "language",
                "india_gross_cr",
                "india_gross_verified"
            ]
        ]
    )

print("\nLANGUAGE COUNTS")
display(
    uncovered["language"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="uncovered_movies")
)

print("\nYEAR COUNTS")
display(
    uncovered.groupby("year")
    .size()
    .rename("uncovered_movies")
    .reset_index()
    .sort_values(
        ["uncovered_movies", "year"],
        ascending=[False, True]
    )
)

print("\nNo files modified.")

In [ ]:
import os
import pandas as pd
from IPython.display import display

# ============================================================
# NOTEBOOK 10 — BATCH 08 MOVIE ALLOCATION
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

MASTER_PATH = os.path.join(
    ROOT, "01_data", "processed", "ICMB_130_MASTER_DATA.csv"
)

PLANNING_PATH = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_07.csv"
)

OUTPUT_PATH = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_08_MOVIE_ALLOCATION.csv"
)

# ------------------------------------------------------------
# 1. APPROVED MOVIE IDS
# ------------------------------------------------------------

BATCH_08_IDS = [
    "2012_1",
    "2014_1",
    "2015_2",
    "2016_1",
    "2019_1",
    "2023_1",
    "2025_1",
    "2022_3",
    "2025_5",
    "2020_2"
]

# ------------------------------------------------------------
# 2. SAFETY CHECKS
# ------------------------------------------------------------

if not os.path.exists(MASTER_PATH):
    raise FileNotFoundError(MASTER_PATH)

if not os.path.exists(PLANNING_PATH):
    raise FileNotFoundError(PLANNING_PATH)

if os.path.exists(OUTPUT_PATH):
    raise FileExistsError(
        "Batch 08 allocation already exists. "
        "No file has been overwritten:\n" + OUTPUT_PATH
    )

# ------------------------------------------------------------
# 3. LOAD DATA
# ------------------------------------------------------------

master = pd.read_csv(MASTER_PATH)
planning = pd.read_csv(PLANNING_PATH)

assert master["movie_record_id"].is_unique
assert planning["movie_record_id"].is_unique
assert len(BATCH_08_IDS) == 10
assert len(set(BATCH_08_IDS)) == 10

# ------------------------------------------------------------
# 4. SELECT APPROVED MOVIES
# ------------------------------------------------------------

allocation = master[
    master["movie_record_id"].isin(BATCH_08_IDS)
].copy()

assert len(allocation) == 10, (
    "One or more approved movie IDs were not found in master."
)

# Preserve approved order
allocation["batch_order"] = allocation[
    "movie_record_id"
].map({movie_id: i + 1 for i, movie_id in enumerate(BATCH_08_IDS)})

allocation = allocation.sort_values("batch_order")

# ------------------------------------------------------------
# 5. RECONCILE CURRENT COVERAGE
# ------------------------------------------------------------

coverage = planning[
    ["movie_record_id", "comment_records", "coverage_category"]
].copy()

allocation = allocation.merge(
    coverage,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

assert allocation["comment_records"].eq(0).all(), (
    "At least one selected movie already has comments. "
    "Review allocation before proceeding."
)

# ------------------------------------------------------------
# 6. FINAL ALLOCATION SCHEMA
# ------------------------------------------------------------

allocation = allocation[
    [
        "movie_record_id",
        "movie_title",
        "year",
        "rank",
        "language",
        "india_gross_cr",
        "comment_records",
        "coverage_category"
    ]
].rename(
    columns={
        "comment_records": "existing_comments"
    }
).reset_index(drop=True)

# ------------------------------------------------------------
# 7. VALIDATE ALLOCATION
# ------------------------------------------------------------

expected_language_counts = {
    "Hindi": 7,
    "Tamil": 2,
    "Telugu": 1
}

actual_language_counts = (
    allocation["language"].value_counts().to_dict()
)

assert actual_language_counts == expected_language_counts
assert allocation["movie_record_id"].is_unique
assert allocation["existing_comments"].sum() == 0

# ------------------------------------------------------------
# 8. SAVE
# ------------------------------------------------------------

allocation.to_csv(OUTPUT_PATH, index=False)

# ------------------------------------------------------------
# 9. DISPLAY RESULT
# ------------------------------------------------------------

print("=" * 75)
print("BATCH 08 MOVIE ALLOCATION — CREATED")
print("=" * 75)

print("Total movies:", len(allocation))
print("Existing comments:", allocation["existing_comments"].sum())

print("\nLanguage distribution:")
display(
    allocation["language"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="movie_count")
)

print("\nApproved allocation:")
display(allocation)

print("\nSaved:", os.path.exists(OUTPUT_PATH))
print("Path:", OUTPUT_PATH)
print("\nAll allocation validations passed.")
print("No YouTube API calls were made.")

In [ ]:
import os

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

SEARCH_DIRS = [
    os.path.join(ROOT, "02_notebooks"),
    os.path.join(ROOT, "03_scripts"),
    os.path.join(ROOT, "04_config")
]

print("=" * 75)
print("EXISTING YOUTUBE DISCOVERY WORKFLOW — FILE INSPECTION")
print("=" * 75)

for folder in SEARCH_DIRS:
    print(f"\nFOLDER: {folder}")

    if not os.path.exists(folder):
        print("Folder not found.")
        continue

    for current_root, dirs, files in os.walk(folder):
        dirs[:] = [
            d for d in dirs
            if d not in ["__pycache__", ".ipynb_checkpoints"]
        ]

        for filename in files:
            if filename.lower().endswith(
                (".ipynb", ".py", ".json", ".yaml", ".yml")
            ):
                print(os.path.join(current_root, filename))

print("\n" + "=" * 75)
print("BATCH 07 DISCOVERY OUTPUT SCHEMA")
print("=" * 75)

import pandas as pd

discovery_path = os.path.join(
    ROOT,
    "01_data",
    "raw",
    "audience_sentiment",
    "youtube",
    "ICMB_130_YOUTUBE_DISCOVERY_BATCH_07.csv"
)

if os.path.exists(discovery_path):
    df = pd.read_csv(discovery_path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(5))
else:
    print("Batch 07 discovery file not found.")

print("\nInspection complete. No files modified.")

In [ ]:
import json
import os
import re

notebook_path = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "02_notebooks/10_Audience_Sentiment_Analysis.ipynb"
)

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "search.list",
    "youtube.search",
    "search_query",
    "discovery",
    "maxresults",
    "publishedafter",
    "batch 07",
    "batch_07"
]

sensitive_pattern = re.compile(
    r"(API_KEY|api_key|YOUTUBE_API_KEY|secret|token|password)",
    re.IGNORECASE
)

matches = []

for i, cell in enumerate(notebook.get("cells", [])):
    source = "".join(cell.get("source", []))

    if cell.get("cell_type") != "code":
        continue

    if any(term in source.lower() for term in keywords):
        matches.append((i, source))

print("=" * 80)
print("NOTEBOOK DISCOVERY WORKFLOW — MATCHING CODE CELLS")
print("=" * 80)
print("Matching cells:", len(matches))

for cell_index, source in matches:
    print(f"\n{'=' * 80}")
    print(f"CELL INDEX: {cell_index}")
    print(f"{'=' * 80}")

    lines = source.splitlines()

    for line in lines:
        if sensitive_pattern.search(line):
            print("[REDACTED — possible credential/configuration line]")
        else:
            print(line)

print("\nInspection only: notebook was not executed or modified.")

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED = PROJECT_ROOT / "01_data/processed/audience_sentiment"
RAW_YOUTUBE = PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"

allocation_file = (
    PROCESSED / "ICMB_130_YOUTUBE_BATCH_08_MOVIE_ALLOCATION.csv"
)

discovery_file = (
    RAW_YOUTUBE / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_08.csv"
)

log_file = (
    RAW_YOUTUBE / "ICMB_130_YOUTUBE_DISCOVERY_LOG_BATCH_08.csv"
)

print("=" * 75)
print("BATCH 08 — DISCOVERY PREFLIGHT")
print("=" * 75)

if not allocation_file.exists():
    raise FileNotFoundError(allocation_file)

allocation = pd.read_csv(allocation_file)

print("Allocation shape:", allocation.shape)
print("Columns:", allocation.columns.tolist())

required = ["movie_record_id", "movie_title", "year", "language"]
missing = [c for c in required if c not in allocation.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

if len(allocation) != 10:
    raise ValueError(f"Expected 10 movies; found {len(allocation)}")

if allocation["movie_record_id"].nunique() != 10:
    raise ValueError("Allocation does not contain 10 unique movie IDs.")

display(
    allocation[
        [
            c for c in [
                "selection_order",
                "movie_record_id",
                "movie_title",
                "year",
                "language"
            ]
            if c in allocation.columns
        ]
    ]
)

print("\nDiscovery output already exists:", discovery_file.exists())
print("Discovery log already exists:", log_file.exists())

if discovery_file.exists() or log_file.exists():
    print("\nSTOP: Existing output detected. Do not overwrite it.")
else:
    print("\nPreflight passed. No API calls made.")
    print("Estimated search quota for discovery: 1,000 units.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — YOUTUBE DISCOVERY: BATCH 08
# 10 movies | Up to 10 candidates per movie
# Follows the established Batch 07 workflow
# ============================================================

import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED = PROJECT_ROOT / "01_data/processed/audience_sentiment"
RAW_YOUTUBE = PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"

allocation_file = (
    PROCESSED / "ICMB_130_YOUTUBE_BATCH_08_MOVIE_ALLOCATION.csv"
)

discovery_file = (
    RAW_YOUTUBE / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_08.csv"
)

log_file = (
    RAW_YOUTUBE / "ICMB_130_YOUTUBE_DISCOVERY_LOG_BATCH_08.csv"
)

# ------------------------------------------------------------
# 1. SAFETY CHECKS
# ------------------------------------------------------------

if not allocation_file.exists():
    raise FileNotFoundError(allocation_file)

if discovery_file.exists() or log_file.exists():
    raise FileExistsError(
        "Batch 08 discovery output already exists. "
        "Inspect existing files before rerunning."
    )

allocation = pd.read_csv(allocation_file)

required_columns = [
    "movie_record_id",
    "movie_title",
    "year",
    "language"
]

missing = [
    col for col in required_columns
    if col not in allocation.columns
]

if missing:
    raise KeyError(f"Missing allocation columns: {missing}")

if len(allocation) != 10:
    raise ValueError("Expected exactly 10 movies.")

if allocation["movie_record_id"].nunique() != 10:
    raise ValueError("Expected 10 unique movie IDs.")

# ------------------------------------------------------------
# 2. YOUTUBE API CONNECTION
# ------------------------------------------------------------

api_key = userdata.get("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError(
        "YOUTUBE_API_KEY not found in Colab Secrets."
    )

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

# ------------------------------------------------------------
# 3. DISCOVERY
# ------------------------------------------------------------

discovery_rows = []
log_rows = []

language_codes = {
    "Hindi": "hi",
    "Telugu": "te",
    "Tamil": "ta",
    "Kannada": "kn"
}

print("=" * 75)
print("BATCH 08 — YOUTUBE VIDEO DISCOVERY")
print("=" * 75)

print("Movies:", len(allocation))
print("Maximum candidates:", len(allocation) * 10)
print("Estimated search quota:", len(allocation) * 100)

for position, (_, movie) in enumerate(
    allocation.iterrows(), start=1
):

    movie_id = str(movie["movie_record_id"])
    title = str(movie["movie_title"])
    year = int(movie["year"])
    language = str(movie["language"])

    query = f'"{title}" {year} movie review reaction'

    request_time = datetime.now(timezone.utc).isoformat()

    print(
        f"\n[{position}/{len(allocation)}] "
        f"Searching: {title} ({year})"
    )

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=10,
            order="relevance",
            relevanceLanguage=language_codes.get(
                language, "en"
            )
        ).execute()

        items = response.get("items", [])

        for item in items:

            snippet = item.get("snippet", {})
            video_id = item.get("id", {}).get("videoId")

            if not video_id:
                continue

            discovery_rows.append({
                "batch_id": "Batch 08",
                "movie_record_id": movie_id,
                "movie_title": title,
                "movie_year": year,
                "language": language,
                "search_query": query,
                "video_id": video_id,
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "discovery_source": "YouTube Data API v3",
                "discovered_at_utc": request_time
            })

        log_rows.append({
            "batch_id": "Batch 08",
            "movie_record_id": movie_id,
            "movie_title": title,
            "search_query": query,
            "request_status": "Success",
            "candidates_returned": len(items),
            "error_message": None,
            "requested_at_utc": request_time
        })

        print(
            f"SUCCESS | {movie_id} | "
            f"Candidates: {len(items)}"
        )

    except Exception as e:

        log_rows.append({
            "batch_id": "Batch 08",
            "movie_record_id": movie_id,
            "movie_title": title,
            "search_query": query,
            "request_status": "Error",
            "candidates_returned": 0,
            "error_message": str(e),
            "requested_at_utc": request_time
        })

        print(
            f"ERROR | {movie_id} | {str(e)[:200]}"
        )

# ------------------------------------------------------------
# 4. SAVE DISCOVERY AND REQUEST LOG
# ------------------------------------------------------------

discovery = pd.DataFrame(discovery_rows)
request_log = pd.DataFrame(log_rows)

if discovery.empty:
    raise RuntimeError(
        "No discovery candidates retrieved. "
        "No output files saved."
    )

if discovery_file.exists() or log_file.exists():
    raise FileExistsError(
        "Output appeared during execution. "
        "Inspect before saving."
    )

discovery.to_csv(discovery_file, index=False)
request_log.to_csv(log_file, index=False)

# ------------------------------------------------------------
# 5. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BATCH 08 DISCOVERY SUMMARY")
print("=" * 75)

print("Movies searched:", len(request_log))
print(
    "Successful requests:",
    (request_log["request_status"] == "Success").sum()
)
print(
    "Failed requests:",
    (request_log["request_status"] == "Error").sum()
)
print("Total candidates:", len(discovery))
print("Unique video IDs:", discovery["video_id"].nunique())

print("\nRequest log:")
display(request_log)

print("\nCandidates by movie:")

display(
    discovery.groupby(
        ["movie_record_id", "movie_title", "language"]
    ).agg(
        candidate_count=("video_id", "count"),
        unique_video_ids=("video_id", "nunique")
    ).reset_index()
)

print("\nSaved discovery:", discovery_file)
print("Saved request log:", log_file)

print("\nDiscovery completed.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — YOUTUBE VIDEO METADATA RETRIEVAL: BATCH 08
# 100 candidates | 2 API requests (50 IDs per request)
# Follows the established Batch 07 metadata workflow
# ============================================================

import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from IPython.display import display

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_YOUTUBE = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube"
)

discovery_file = (
    RAW_YOUTUBE / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_08.csv"
)

metadata_file = (
    RAW_YOUTUBE / "ICMB_130_YOUTUBE_METADATA_BATCH_08.csv"
)

# ------------------------------------------------------------
# 2. SAFETY CHECKS
# ------------------------------------------------------------

if not discovery_file.exists():
    raise FileNotFoundError(discovery_file)

if metadata_file.exists():
    raise FileExistsError(
        f"Metadata file already exists. "
        f"Inspect before rerunning: {metadata_file}"
    )

discovery = pd.read_csv(discovery_file)

required_columns = [
    "movie_record_id",
    "movie_title",
    "video_id"
]

missing_columns = [
    col for col in required_columns
    if col not in discovery.columns
]

if missing_columns:
    raise KeyError(
        f"Missing discovery columns: {missing_columns}"
    )

if discovery.empty or discovery["video_id"].isna().any():
    raise ValueError(
        "Discovery file is empty or contains missing video IDs."
    )

video_ids = (
    discovery["video_id"]
    .astype(str)
    .drop_duplicates()
    .tolist()
)

if len(video_ids) != 100:
    raise ValueError(
        f"Expected 100 unique video IDs; found {len(video_ids)}."
    )

print("=" * 70)
print("BATCH 08 — METADATA PREFLIGHT")
print("=" * 70)

print("Discovery records:", len(discovery))
print("Unique video IDs:", len(video_ids))
print("Expected API requests:", (len(video_ids) + 49) // 50)
print("Metadata output already exists:", metadata_file.exists())

# ------------------------------------------------------------
# 3. YOUTUBE API CONNECTION
# ------------------------------------------------------------

api_key = userdata.get("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError(
        "YOUTUBE_API_KEY not found in Colab Secrets."
    )

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

# ------------------------------------------------------------
# 4. RETRIEVE METADATA
# ------------------------------------------------------------

metadata_rows = []
error_rows = []

print("\n" + "=" * 70)
print("BATCH 08 — VIDEO METADATA RETRIEVAL")
print("=" * 70)

# YouTube videos.list accepts up to 50 IDs per request.
for start in range(0, len(video_ids), 50):

    chunk = video_ids[start:start + 50]

    chunk_number = start // 50 + 1

    print(
        f"\nRequest {chunk_number}: "
        f"{len(chunk)} video IDs"
    )

    try:
        response = youtube.videos().list(
            part="snippet,contentDetails,statistics,status",
            id=",".join(chunk),
            maxResults=50
        ).execute()

        items = response.get("items", [])

        for item in items:

            snippet = item.get("snippet", {})
            statistics = item.get("statistics", {})
            content = item.get("contentDetails", {})
            status = item.get("status", {})

            metadata_rows.append({
                "video_id": item.get("id"),
                "metadata_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "metadata_channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "tags": " | ".join(snippet.get("tags", [])),
                "category_id": snippet.get("categoryId"),
                "duration": content.get("duration"),
                "definition": content.get("definition"),
                "caption_available": content.get("caption"),
                "view_count": statistics.get("viewCount"),
                "like_count": statistics.get("likeCount"),
                "comment_count": statistics.get("commentCount"),
                "privacy_status": status.get("privacyStatus"),
                "embeddable": status.get("embeddable"),
                "metadata_retrieved_at_utc": (
                    datetime.now(timezone.utc).isoformat()
                )
            })

        returned_ids = {
            item.get("id") for item in items
        }

        missing_ids = set(chunk) - returned_ids

        for missing_id in missing_ids:
            error_rows.append({
                "video_id": missing_id,
                "error": "No metadata returned for requested video ID"
            })

        print(
            f"Retrieved: {len(items)}/{len(chunk)}"
        )

        if missing_ids:
            print(
                "Unavailable IDs in this request:",
                len(missing_ids)
            )

    except Exception as e:

        for missing_id in chunk:
            error_rows.append({
                "video_id": missing_id,
                "error": str(e)[:500]
            })

        print(
            f"API ERROR in request {chunk_number}: "
            f"{str(e)[:300]}"
        )

# ------------------------------------------------------------
# 5. MERGE WITH DISCOVERY
# ------------------------------------------------------------

metadata = pd.DataFrame(metadata_rows)

if metadata.empty:
    raise RuntimeError(
        "No video metadata was retrieved. "
        "Output file was not saved."
    )

merged = discovery.merge(
    metadata,
    on="video_id",
    how="left",
    suffixes=("_discovery", "_metadata"),
    validate="many_to_one"
)

# ------------------------------------------------------------
# 6. SAVE METADATA OUTPUT
# ------------------------------------------------------------

if metadata_file.exists():
    raise FileExistsError(
        "Metadata output appeared during execution. "
        "Inspect before saving."
    )

merged.to_csv(
    metadata_file,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BATCH 08 METADATA SUMMARY")
print("=" * 70)

print("Discovery candidates:", len(discovery))
print("Unique requested IDs:", len(video_ids))
print("Metadata records retrieved:", len(metadata))
print("Merged shape:", merged.shape)

print(
    "Candidates with metadata:",
    merged["metadata_title"].notna().sum()
)

print(
    "Missing metadata:",
    merged["metadata_title"].isna().sum()
)

print("Metadata retrieval exceptions:", len(error_rows))

print("\nMetadata coverage by movie:")

display(
    merged.groupby(
        ["movie_record_id", "movie_title"]
    ).agg(
        candidates=("video_id", "count"),
        metadata_retrieved=("metadata_title", "count"),
        unique_video_ids=("video_id", "nunique")
    ).reset_index()
)

print("\nMetadata preview:")

display(
    merged[
        [
            "movie_record_id",
            "movie_title",
            "video_id",
            "metadata_title",
            "metadata_channel_title",
            "duration",
            "view_count",
            "comment_count"
        ]
    ].head(15)
)

if error_rows:
    print("\nMetadata exceptions:")
    display(pd.DataFrame(error_rows))

print("\nSaved:", metadata_file)
print("Batch 08 metadata retrieval completed.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 08 AUTOMATED METADATA SCREENING
# Same screening rules as Batch 07
# Metadata-based decisions only; no content verification
# ============================================================

import pandas as pd
import re
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_YOUTUBE = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube"
)

metadata_file = (
    RAW_YOUTUBE / "ICMB_130_YOUTUBE_METADATA_BATCH_08.csv"
)

output_file = (
    RAW_YOUTUBE / "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_08.csv"
)

# ------------------------------------------------------------
# 1. SAFETY CHECKS
# ------------------------------------------------------------

if not metadata_file.exists():
    raise FileNotFoundError(metadata_file)

if output_file.exists():
    raise FileExistsError(
        f"Output already exists: {output_file}"
    )

df = pd.read_csv(metadata_file)

required = [
    "movie_record_id",
    "movie_title",
    "language",
    "video_id",
    "metadata_title",
    "metadata_channel_title",
    "description_metadata",
    "duration",
    "view_count",
    "comment_count"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

if len(df) != 100 or df["video_id"].nunique() != 100:
    raise ValueError(
        "Expected 100 records and 100 unique video IDs."
    )

# ------------------------------------------------------------
# 2. DURATION CONVERSION
# ------------------------------------------------------------

def duration_seconds(value):

    if pd.isna(value):
        return None

    match = re.fullmatch(
        r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?",
        str(value)
    )

    if not match:
        return None

    h, m, s = [int(x or 0) for x in match.groups()]

    return h * 3600 + m * 60 + s


# ------------------------------------------------------------
# 3. RULE-BASED SOURCE CLASSIFICATION
# ------------------------------------------------------------

def classify_video(row):

    title = str(row["metadata_title"]).lower()
    description = str(row["description_metadata"]).lower()

    text = title + " " + description

    seconds = duration_seconds(row["duration"])

    trailer_terms = [
        "official trailer",
        "trailer reaction",
        "teaser reaction",
        "teaser trailer",
        "official teaser"
    ]

    full_movie_terms = [
        "full movie",
        "full film",
        "complete movie",
        "watch full movie"
    ]

    review_terms = [
        "movie review",
        "film review",
        "review and analysis",
        "ending explained",
        "movie analysis",
        "film analysis",
        "retrospective",
        "rewatched",
        "rewatch",
        "revisited"
    ]

    reaction_terms = [
        "movie reaction",
        "film reaction",
        "first time watching",
        "first time reaction",
        "watching for the first time",
        "foreigners react",
        "react to"
    ]

    if any(term in text for term in trailer_terms):
        return (
            "Trailer Reaction",
            "Exclude from primary sample"
        )

    if any(term in text for term in full_movie_terms):
        return (
            "Full Movie",
            "Exclude from primary sample"
        )

    if any(term in text for term in review_terms):
        return (
            "Movie Review / Analysis",
            "Shortlist"
        )

    if any(term in text for term in reaction_terms):

        if seconds is not None and seconds >= 1200:
            return (
                "Movie Reaction",
                "Shortlist"
            )

        return (
            "Possible Reaction",
            "Hold for review"
        )

    if seconds is not None and seconds < 180:
        return (
            "Short / Unclear",
            "Hold for review"
        )

    return (
        "Unclassified",
        "Hold for review"
    )


classified = df.apply(
    classify_video,
    axis=1,
    result_type="expand"
)

classified.columns = [
    "automated_source_type",
    "screening_decision"
]

df = pd.concat([df, classified], axis=1)

# ------------------------------------------------------------
# 4. REVIEW STATUS AND SCREENING PROVENANCE
# ------------------------------------------------------------

df["duration_seconds"] = df["duration"].apply(duration_seconds)

df["metadata_review_status"] = "Metadata Reviewed"
df["content_review_status"] = "Not Reviewed"

df["screening_method"] = (
    "Rule-based title and metadata description screening"
)

# ------------------------------------------------------------
# 5. SORT RESULTS
# ------------------------------------------------------------

decision_order = {
    "Shortlist": 0,
    "Hold for review": 1,
    "Exclude from primary sample": 2
}

df["_sort"] = df["screening_decision"].map(decision_order)

df = (
    df.sort_values(
        [
            "movie_record_id",
            "_sort",
            "comment_count",
            "view_count"
        ],
        ascending=[True, True, False, False]
    )
    .drop(columns="_sort")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. SAVE SCREENING OUTPUT
# ------------------------------------------------------------

if output_file.exists():
    raise FileExistsError(
        "Screening output appeared during execution. "
        "Inspect before saving."
    )

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. SCREENING SUMMARY
# ------------------------------------------------------------

print("=" * 70)
print("BATCH 08 — AUTOMATED METADATA SCREENING")
print("=" * 70)

print("Total candidates:", len(df))
print("Unique video IDs:", df["video_id"].nunique())

print("\nSCREENING DECISIONS")

display(
    df["screening_decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="candidate_count")
)

print("\nPROVISIONAL SOURCE TYPES")

display(
    df["automated_source_type"]
    .value_counts()
    .rename_axis("source_type")
    .reset_index(name="candidate_count")
)

print("\nMOVIE-WISE SCREENING")

movie_summary = (
    df.groupby(
        ["movie_record_id", "movie_title", "language"]
    )
    .agg(
        candidates=("video_id", "count"),
        shortlist=(
            "screening_decision",
            lambda x: (x == "Shortlist").sum()
        ),
        hold=(
            "screening_decision",
            lambda x: (x == "Hold for review").sum()
        ),
        excluded=(
            "screening_decision",
            lambda x: x.str.startswith("Exclude").sum()
        )
    )
    .reset_index()
)

display(movie_summary)

print("\nSCREENING STATUS CHECK")

print(
    "Metadata reviewed:",
    (df["metadata_review_status"] == "Metadata Reviewed").sum()
)

print(
    "Content not reviewed:",
    (df["content_review_status"] == "Not Reviewed").sum()
)

print("\nSaved:", output_file)
print("Batch 08 automated screening completed.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 08 MANUAL REVIEW SHEET
# Retain Shortlist + Hold candidates
# Read-only source; creates a separate review file
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_YOUTUBE = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube"
)

input_file = (
    RAW_YOUTUBE
    / "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_08.csv"
)

output_file = (
    RAW_YOUTUBE
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_08.csv"
)

if not input_file.exists():
    raise FileNotFoundError(input_file)

if output_file.exists():
    raise FileExistsError(
        f"Review file already exists: {output_file}"
    )

df = pd.read_csv(input_file)

required = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "metadata_title",
    "metadata_channel_title",
    "duration",
    "view_count",
    "comment_count",
    "automated_source_type",
    "screening_decision",
    "metadata_review_status",
    "content_review_status"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Retain shortlist and hold candidates only.
review = df[
    df["screening_decision"].isin(
        ["Shortlist", "Hold for review"]
    )
].copy()

# Add fields for final source-selection decisions.
review["manual_decision"] = "Pending"
review["final_source_type"] = ""
review["review_notes"] = ""

# Keep metadata and content review distinct.
review["metadata_review_status"] = "Metadata Reviewed"
review["content_review_status"] = "Not Reviewed"

# Sort by movie, then shortlist before hold.
decision_order = {
    "Shortlist": 0,
    "Hold for review": 1
}

review["_sort"] = review["screening_decision"].map(decision_order)

review = (
    review.sort_values(
        ["movie_record_id", "_sort", "comment_count", "view_count"],
        ascending=[True, True, False, False]
    )
    .drop(columns="_sort")
    .reset_index(drop=True)
)

# Save without modifying the metadata screening file.
review.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("BATCH 08 — MANUAL REVIEW SHEET")
print("=" * 70)

print("Automated screening records:", len(df))
print("Candidates retained for review:", len(review))
print("Unique video IDs:", review["video_id"].nunique())
print("Movies represented:", review["movie_record_id"].nunique())

print("\nReview candidates by movie:")

display(
    review.groupby(
        ["movie_record_id", "movie_title", "language"]
    ).agg(
        review_candidates=("video_id", "count"),
        shortlist=(
            "screening_decision",
            lambda x: (x == "Shortlist").sum()
        ),
        hold=(
            "screening_decision",
            lambda x: (x == "Hold for review").sum()
        )
    ).reset_index()
)

print("\nReview decision fields:")
display(
    review[
        [
            "movie_record_id",
            "movie_title",
            "metadata_title",
            "metadata_channel_title",
            "duration",
            "comment_count",
            "automated_source_type",
            "screening_decision",
            "manual_decision"
        ]
    ].head(20)
)

print("\nSaved:", output_file)
print("Original metadata screening file remains unchanged.")

In [ ]:
# ============================================================
# BATCH 08 — CONSOLIDATED CANDIDATE INSPECTION
# No API calls; no source files modified
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

review_file = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_08.csv"
)

review = pd.read_csv(review_file)

columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "metadata_channel_title",
    "duration",
    "view_count",
    "comment_count",
    "automated_source_type",
    "screening_decision"
]

missing = [c for c in columns if c not in review.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

for movie_id, group in review.groupby(
    "movie_record_id", sort=False
):
    print("\n" + "=" * 100)
    print(
        f"{group['movie_title'].iloc[0]} "
        f"({movie_id}) — {len(group)} candidates"
    )
    print("=" * 100)

    display(
        group[columns[2:]].reset_index(drop=True)
    )

print("\nTOTAL CANDIDATES:", len(review))
print("UNIQUE VIDEO IDs:", review["video_id"].nunique())

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

file = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_08.csv"
)

df = pd.read_csv(file)

columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "metadata_channel_title",
    "duration",
    "view_count",
    "comment_count",
    "automated_source_type",
    "screening_decision"
]

# Display records 21–88; first 20 are already available.
remaining = df[columns].iloc[20:].copy()

print("REMAINING CANDIDATES:", len(remaining))
print("UNIQUE VIDEO IDs:", remaining["video_id"].nunique())

for movie_id, group in remaining.groupby(
    "movie_record_id", sort=False
):
    print(
        f"\n{group['movie_title'].iloc[0]} "
        f"({movie_id}) — {len(group)} candidates"
    )
    display(group.drop(columns=["movie_record_id", "movie_title"]).reset_index(drop=True))

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

file = (
    ROOT
    / "01_data/raw/audience_sentiment/youtube/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_08.csv"
)

df = pd.read_csv(file)

cols = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "metadata_channel_title",
    "duration",
    "view_count",
    "comment_count",
    "automated_source_type",
    "screening_decision"
]

# Display all records without truncating the titles.
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 120)

display(
    df[cols].sort_values(
        ["movie_record_id", "screening_decision", "comment_count"],
        ascending=[True, True, False]
    ).reset_index(drop=True)
)

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 08 MANIFEST SCHEMA PREFLIGHT
# Read-only inspection; no files modified
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

manifest_file = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_07.csv"
)

review_file = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_08.csv"
)

if not manifest_file.exists():
    raise FileNotFoundError(manifest_file)

if not review_file.exists():
    raise FileNotFoundError(review_file)

previous = pd.read_csv(manifest_file)
review = pd.read_csv(review_file)

print("=" * 70)
print("BATCH 07 APPROVED MANIFEST — SCHEMA")
print("=" * 70)

print("Shape:", previous.shape)
print("\nColumns and data types:")
display(
    pd.DataFrame({
        "column": previous.columns,
        "dtype": previous.dtypes.astype(str).values
    })
)

print("\nSample records:")
display(previous.head(5))

print("\nBATCH 08 MANUAL REVIEW FILE")
print("Shape:", review.shape)
print("Columns:", review.columns.tolist())

print("\nExisting Batch 07 output path:")
print(manifest_file)

print("\nBatch 08 proposed manifest path:")
print(
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_08.csv"
)

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 08 APPROVED ACQUISITION MANIFEST
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

SOURCE_FILE = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/"
    / "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_08.csv"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_08.csv"
)

if not SOURCE_FILE.exists():
    raise FileNotFoundError(SOURCE_FILE)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        f"Output already exists. No overwrite performed:\n{OUTPUT_FILE}"
    )

review = pd.read_csv(SOURCE_FILE)

# Approved video IDs and source classifications
approved = [
    ("2012_1", "P9JgWmBHKk8", "Movie Review"),
    ("2012_1", "PLBFtBrZnpg", "Movie Reaction"),

    ("2014_1", "prp418Wp7ik", "Movie Review"),
    ("2014_1", "X3-SxU8GVaI", "Movie Reaction"),

    ("2015_2", "MGs5CE-cns8", "Movie Review"),
    ("2015_2", "y6wECZaDq0s", "Movie Reaction"),

    ("2016_1", "sZtWu280nkg", "Movie Reaction"),
    ("2016_1", "7RbEvrDXtX8", "Movie Reaction"),

    ("2020_2", "ER2rwYzDqOo", "Movie Review"),
    ("2020_2", "X8blRlsG7xU", "Movie Reaction"),

    ("2022_3", "npLKrm2qGS4", "Movie Review"),
    ("2022_3", "NQeykklUyBY", "Movie Reaction"),

    ("2023_1", "1oCqsOe_QRQ", "Movie Reaction"),
    ("2023_1", "7QSGk6XQaT8", "Movie Reaction"),

    ("2025_1", "lvglXTpsE-g", "Movie Review"),
    ("2025_1", "OlU6_qjaR7I", "Movie Reaction"),

    ("2025_5", "xhGdEPe_JHA", "Movie Reaction"),
    ("2025_5", "wgkq26wY5oE", "Movie Reaction"),
]

approved_df = pd.DataFrame(
    approved,
    columns=["movie_record_id", "video_id", "manual_source_type"]
)

# Validate uniqueness
assert len(approved_df) == 18
assert approved_df["video_id"].nunique() == 18
assert approved_df["movie_record_id"].nunique() == 9

# Match approved IDs to actual metadata records
matched = approved_df.merge(
    review,
    on=["movie_record_id", "video_id"],
    how="left",
    validate="one_to_one",
    indicator=True
)

missing = matched[matched["_merge"] != "both"]

if not missing.empty:
    display(missing[["movie_record_id", "video_id", "_merge"]])
    raise ValueError("Some approved video IDs were not found in the review file.")

# Construct manifest using the Batch 07 schema
manifest = pd.DataFrame({
    "movie_record_id": matched["movie_record_id"],
    "movie_title": matched["movie_title"],
    "movie_year": matched["movie_year"],
    "language": matched["language"],
    "video_id": matched["video_id"],
    "video_title": matched["metadata_title"],
    "channel_title": matched["metadata_channel_title"],
    "duration": matched["duration"],
    "view_count": matched["view_count"],
    "comment_count": matched["comment_count"],
    "manual_source_type": matched["manual_source_type"],
    "manual_verification_status": "Metadata Reviewed",
    "manual_review_notes": (
        "Selected for Batch 08 audience-comment acquisition. "
        "Metadata reviewed; video content not independently reviewed. "
        "Target is 50 top-level comments per selected video."
    ),
    "acquisition_status": "Pending",
    "target_comments": 50
})

# Final validation
assert manifest.shape == (18, 15)
assert manifest["video_id"].nunique() == 18
assert manifest["movie_record_id"].nunique() == 9
assert manifest["target_comments"].sum() == 900
assert manifest["acquisition_status"].eq("Pending").all()
assert manifest["manual_verification_status"].eq("Metadata Reviewed").all()

# Save without overwriting
manifest.to_csv(OUTPUT_FILE, index=False)

print("=" * 70)
print("BATCH 08 APPROVED MANIFEST CREATED")
print("=" * 70)

print("Shape:", manifest.shape)
print("Unique movies:", manifest["movie_record_id"].nunique())
print("Unique videos:", manifest["video_id"].nunique())
print("Target comments:", manifest["target_comments"].sum())

print("\nMovie-wise allocation:")
display(
    manifest.groupby(
        ["movie_record_id", "movie_title", "language"],
        as_index=False
    ).agg(
        selected_videos=("video_id", "nunique"),
        target_comments=("target_comments", "sum")
    )
)

print("\nApproved manifest preview:")
display(manifest)

print("\nSaved:", OUTPUT_FILE)

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 08 COMMENT ACQUISITION PREFLIGHT
# Read-only: no API calls and no files modified
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

MANIFEST_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_08.csv"
)

COMMENTS_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

OUTPUT_FILE = (
    COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08.csv"
)

LOG_FILE = (
    COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_08.csv"
)

CHECKPOINT_FILE = (
    COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08_CHECKPOINT.csv"
)

assert MANIFEST_FILE.exists(), "Approved manifest not found."
assert COMMENTS_DIR.exists(), "Comments directory not found."

manifest = pd.read_csv(MANIFEST_FILE)

print("=" * 70)
print("BATCH 08 COMMENT ACQUISITION PREFLIGHT")
print("=" * 70)

print("Manifest shape:", manifest.shape)
print("Unique videos:", manifest["video_id"].nunique())
print("Total target comments:", manifest["target_comments"].sum())

print("\nAcquisition status:")
print(manifest["acquisition_status"].value_counts(dropna=False))

print("\nOutput file checks:")
for file_path in [OUTPUT_FILE, LOG_FILE, CHECKPOINT_FILE]:
    print(f"{file_path.name}: {'EXISTS' if file_path.exists() else 'NOT FOUND'}")

print("\nExisting raw comment batches:")
for file_path in sorted(COMMENTS_DIR.glob("ICMB_130_YOUTUBE_COMMENT_BATCH_*.csv")):
    print(file_path.name)

print("\nPer-movie allocation:")
display(
    manifest.groupby(
        ["movie_record_id", "movie_title"],
        as_index=False
    ).agg(
        videos=("video_id", "nunique"),
        target_comments=("target_comments", "sum")
    )
)

assert manifest["acquisition_status"].eq("Pending").all()
assert manifest["video_id"].is_unique

print("\nPreflight completed. No files modified and no API calls made.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 08 YOUTUBE COMMENT ACQUISITION
# Target: 18 videos x 50 top-level comments = 900
# ============================================================

import os
import time
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)
MANIFEST_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_08.csv"
)

RAW_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08.csv"
LOG_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_08.csv"
CHECKPOINT_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08_CHECKPOINT.csv"

# Safety checks
for path in [RAW_FILE, LOG_FILE, CHECKPOINT_FILE]:
    if path.exists():
        raise FileExistsError(
            f"Existing file detected; refusing to overwrite: {path}"
        )

manifest = pd.read_csv(MANIFEST_FILE)

assert len(manifest) == 18
assert manifest["video_id"].nunique() == 18
assert manifest["target_comments"].sum() == 900
assert manifest["acquisition_status"].eq("Pending").all()

# Read API key securely from Colab Secrets
API_KEY = userdata.get("YOUTUBE_API_KEY")
if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY not found in Colab Secrets.")

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY,
    cache_discovery=False
)

all_comments = []
acquisition_logs = []

print("=" * 70)
print("BATCH 08 — COMMENT ACQUISITION STARTED")
print("=" * 70)

for index, row in manifest.iterrows():

    video_id = row["video_id"]
    target = int(row["target_comments"])

    retrieved = []
    next_page_token = None
    request_count = 0
    status = "Completed"
    error_message = ""

    print(
        f"\n[{index + 1}/18] {row['movie_title']} | "
        f"{row['channel_title']} | {video_id}"
    )

    try:
        while len(retrieved) < target:

            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, target - len(retrieved)),
                order="time",
                textFormat="plainText",
                pageToken=next_page_token
            )

            response = request.execute()
            request_count += 1

            items = response.get("items", [])

            if not items:
                break

            for item in items:
                snippet = item["snippet"]["topLevelComment"]["snippet"]
                comment_id = item["snippet"]["topLevelComment"]["id"]

                retrieved.append({
                    "movie_record_id": row["movie_record_id"],
                    "movie_title": row["movie_title"],
                    "movie_year": row["movie_year"],
                    "language": row["language"],
                    "video_id": video_id,
                    "video_title": row["video_title"],
                    "channel_title": row["channel_title"],
                    "comment_id": comment_id,
                    "comment_text": snippet.get("textDisplay", ""),
                    "author_name": snippet.get("authorDisplayName", ""),
                    "like_count": snippet.get("likeCount", 0),
                    "published_at": snippet.get("publishedAt", ""),
                    "updated_at": snippet.get("updatedAt", ""),
                    "comment_position": len(retrieved) + 1,
                    "acquisition_batch": "Batch 08",
                    "retrieval_order": "time"
                })

                if len(retrieved) >= target:
                    break

            next_page_token = response.get("nextPageToken")

            if not next_page_token:
                break

            time.sleep(0.15)

        if len(retrieved) < target:
            status = "Partial"

    except HttpError as e:
        status = "API Error"
        error_message = str(e)[:500]

    except Exception as e:
        status = "Error"
        error_message = str(e)[:500]

    all_comments.extend(retrieved)

    acquisition_logs.append({
        "movie_record_id": row["movie_record_id"],
        "movie_title": row["movie_title"],
        "video_id": video_id,
        "channel_title": row["channel_title"],
        "target_comments": target,
        "retrieved_comments": len(retrieved),
        "request_count": request_count,
        "acquisition_status": status,
        "error_message": error_message,
        "acquired_at_utc": datetime.now(timezone.utc).isoformat()
    })

    # Save checkpoint after every video
    pd.DataFrame(all_comments).to_csv(
        CHECKPOINT_FILE, index=False
    )
    pd.DataFrame(acquisition_logs).to_csv(
        LOG_FILE, index=False
    )

    print(
        f"Retrieved: {len(retrieved)}/{target} | "
        f"Status: {status}"
    )

    if status in ["API Error", "Error"]:
        print("Error:", error_message)

    time.sleep(0.2)

# Save final raw comments and acquisition log
comments_df = pd.DataFrame(all_comments)
logs_df = pd.DataFrame(acquisition_logs)

comments_df.to_csv(RAW_FILE, index=False)
logs_df.to_csv(LOG_FILE, index=False)

# Update manifest acquisition statuses
status_map = logs_df.set_index("video_id")["acquisition_status"]

updated_manifest = manifest.copy()
updated_manifest["acquisition_status"] = (
    updated_manifest["video_id"]
    .map(status_map)
    .fillna("Pending")
)

updated_manifest.to_csv(MANIFEST_FILE, index=False)

print("\n" + "=" * 70)
print("BATCH 08 ACQUISITION SUMMARY")
print("=" * 70)

print("Total comments:", len(comments_df))
print("Unique comment IDs:", comments_df["comment_id"].nunique())
print("Unique videos:", comments_df["video_id"].nunique())
print("Unique movies:", comments_df["movie_record_id"].nunique())
print("\nAcquisition statuses:")
print(logs_df["acquisition_status"].value_counts(dropna=False))

print("\nMovie-wise retrieval:")
display(
    comments_df.groupby(
        ["movie_record_id", "movie_title"],
        as_index=False
    ).agg(
        comments=("comment_id", "count"),
        unique_comment_ids=("comment_id", "nunique"),
        videos=("video_id", "nunique")
    )
)

print("\nSaved files:")
print("Raw comments:", RAW_FILE)
print("Acquisition log:", LOG_FILE)
print("Checkpoint:", CHECKPOINT_FILE)
print("Updated manifest:", MANIFEST_FILE)

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 08 FAILED VIDEO RETRY
# Only y6wECZaDq0s; existing comments are preserved
# ============================================================

import pandas as pd
import time
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

RAW_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08.csv"
LOG_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_08.csv"
CHECKPOINT_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08_CHECKPOINT.csv"
MANIFEST_FILE = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment/"
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_08.csv"
)

VIDEO_ID = "y6wECZaDq0s"
TARGET = 50

# Confirm files exist
for path in [RAW_FILE, LOG_FILE, CHECKPOINT_FILE, MANIFEST_FILE]:
    if not path.exists():
        raise FileNotFoundError(path)

# Read replacement secret without displaying it
API_KEY = userdata.get("YOUTUBE_API_KEY")
if not API_KEY:
    raise ValueError("Replacement YOUTUBE_API_KEY not found.")

API_KEY = "".join(API_KEY.split())

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY,
    cache_discovery=False
)

raw = pd.read_csv(RAW_FILE)
logs = pd.read_csv(LOG_FILE)
manifest = pd.read_csv(MANIFEST_FILE)

# Safety: do not retry if comments already exist for this video
existing = raw[raw["video_id"].astype(str) == VIDEO_ID]

if len(existing) > 0:
    raise ValueError(
        f"{len(existing)} comments already exist for this video. "
        "Manual reconciliation required; no retry performed."
    )

approved_row = manifest[
    manifest["video_id"].astype(str) == VIDEO_ID
]

if len(approved_row) != 1:
    raise ValueError("Expected exactly one manifest record for failed video.")

row = approved_row.iloc[0]

print("Retrying:", row["movie_title"], "|", row["channel_title"])
print("Video ID:", VIDEO_ID)

retrieved = []
next_page_token = None
request_count = 0
status = "Completed"
error_message = ""

try:
    while len(retrieved) < TARGET:

        response = youtube.commentThreads().list(
            part="snippet",
            videoId=VIDEO_ID,
            maxResults=min(100, TARGET - len(retrieved)),
            order="time",
            textFormat="plainText",
            pageToken=next_page_token
        ).execute()

        request_count += 1

        items = response.get("items", [])
        if not items:
            break

        for item in items:
            top_comment = item["snippet"]["topLevelComment"]
            snippet = top_comment["snippet"]

            retrieved.append({
                "movie_record_id": row["movie_record_id"],
                "movie_title": row["movie_title"],
                "movie_year": row["movie_year"],
                "language": row["language"],
                "video_id": VIDEO_ID,
                "video_title": row["video_title"],
                "channel_title": row["channel_title"],
                "comment_id": top_comment["id"],
                "comment_text": snippet.get("textDisplay", ""),
                "author_name": snippet.get("authorDisplayName", ""),
                "like_count": snippet.get("likeCount", 0),
                "published_at": snippet.get("publishedAt", ""),
                "updated_at": snippet.get("updatedAt", ""),
                "comment_position": len(retrieved) + 1,
                "acquisition_batch": "Batch 08",
                "retrieval_order": "time"
            })

            if len(retrieved) >= TARGET:
                break

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.2)

    if len(retrieved) < TARGET:
        status = "Partial"

except HttpError as e:
    status = "API Error"
    error_message = str(e)[:500]

except Exception as e:
    status = "Error"
    error_message = str(e)[:500]

# Append only newly retrieved records
if retrieved:
    new_df = pd.DataFrame(retrieved)

    combined = pd.concat([raw, new_df], ignore_index=True)

    # Ensure comment IDs are not duplicated
    combined = combined.drop_duplicates(
        subset=["comment_id"],
        keep="first"
    )

    combined.to_csv(RAW_FILE, index=False)
    combined.to_csv(CHECKPOINT_FILE, index=False)

# Update existing log entry for this video
logs = logs[logs["video_id"].astype(str) != VIDEO_ID]

retry_log = pd.DataFrame([{
    "movie_record_id": row["movie_record_id"],
    "movie_title": row["movie_title"],
    "video_id": VIDEO_ID,
    "channel_title": row["channel_title"],
    "target_comments": TARGET,
    "retrieved_comments": len(retrieved),
    "request_count": request_count,
    "acquisition_status": status,
    "error_message": error_message,
    "acquired_at_utc": datetime.now(timezone.utc).isoformat()
}])

logs = pd.concat([logs, retry_log], ignore_index=True)
logs.to_csv(LOG_FILE, index=False)

# Update manifest status
manifest.loc[
    manifest["video_id"].astype(str) == VIDEO_ID,
    "acquisition_status"
] = status

manifest.to_csv(MANIFEST_FILE, index=False)

# Final counts
final_raw = pd.read_csv(RAW_FILE)

print("\n" + "=" * 65)
print("FAILED VIDEO RETRY RESULT")
print("=" * 65)
print("Retrieved in retry:", len(retrieved))
print("Retry status:", status)
print("API requests:", request_count)
print("Total Batch 08 comments:", len(final_raw))
print("Unique comment IDs:", final_raw["comment_id"].nunique())
print("Unique videos:", final_raw["video_id"].nunique())

if error_message:
    print("Error:", error_message)

print("\nNo earlier batch files were modified.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 08 QUALITY PROFILE PREFLIGHT
# Inspect Batch 07 report structures; no files modified
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

FILES = [
    "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_07.csv",
    "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_07.csv",
    "ICMB_130_YOUTUBE_COMMENT_VIDEO_QUALITY_BATCH_07.csv",
    "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_BATCH_07.csv",
]

print("=" * 70)
print("BATCH 08 QUALITY PROFILE PREFLIGHT")
print("=" * 70)

for filename in FILES:
    path = COMMENTS_DIR / filename

    print(f"\n{'-' * 70}")
    print(filename)

    if not path.exists():
        print("NOT FOUND")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))

raw_file = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08.csv"

raw = pd.read_csv(raw_file)

print("\n" + "=" * 70)
print("BATCH 08 RAW DATA CHECK")
print("=" * 70)

print("Shape:", raw.shape)
print("Columns:", raw.columns.tolist())
print("Unique comment IDs:", raw["comment_id"].nunique())
print("Unique videos:", raw["video_id"].nunique())
print("Unique movies:", raw["movie_record_id"].nunique())
print("Empty comments:", raw["comment_text"].isna().sum())

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 08 OVERALL COMMENT QUALITY PROFILE
# ============================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

RAW_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08.csv"

OUTPUT_FILE = (
    COMMENTS_DIR
    / "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_08.csv"
)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        f"Report already exists; refusing to overwrite: {OUTPUT_FILE}"
    )

df = pd.read_csv(RAW_FILE)

# Preserve raw comments; create temporary analysis fields
text = df["comment_text"].fillna("").astype(str)

word_counts = text.apply(
    lambda x: len(re.findall(r"\b\w+\b", x, flags=re.UNICODE))
)

char_counts = text.str.len()

# Script classification is descriptive only, not language identification
def classify_script(value):
    value = str(value)

    if not value.strip():
        return "Empty"

    scripts = []

    checks = {
        "Devanagari": r"[\u0900-\u097F]",
        "Tamil": r"[\u0B80-\u0BFF]",
        "Telugu": r"[\u0C00-\u0C7F]",
        "Kannada": r"[\u0C80-\u0CFF]",
        "Bengali": r"[\u0980-\u09FF]",
        "Malayalam": r"[\u0D00-\u0D7F]"
    }

    for script, pattern in checks.items():
        if re.search(pattern, value):
            scripts.append(script)

    has_latin = bool(re.search(r"[A-Za-z]", value))

    if len(scripts) > 1 or (scripts and has_latin):
        return "Mixed Script"

    if scripts:
        return scripts[0]

    if has_latin:
        return "Latin Script"

    if re.search(r"[^\w\s]", value):
        return "Other/Emoji/Symbols"

    return "Other"

script_categories = text.apply(classify_script)

# URL detection
url_count = text.str.contains(
    r"https?://|www\.",
    case=False,
    regex=True
).sum()

# Duplicate text detection on normalized temporary text
normalized = (
    text.str.lower()
    .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

duplicate_mask = (
    normalized.ne("") &
    normalized.duplicated(keep=False)
)

# Overall profile
profile = pd.DataFrame([{
    "batch_id": "Batch 08",
    "total_records": len(df),
    "unique_comment_ids": df["comment_id"].nunique(),
    "unique_videos": df["video_id"].nunique(),
    "unique_movies": df["movie_record_id"].nunique(),
    "empty_comments": int(text.str.strip().eq("").sum()),
    "non_empty_comments": int(text.str.strip().ne("").sum()),
    "very_short_comments_le_3_words": int((word_counts <= 3).sum()),
    "short_comments_4_to_10_words": int(
        ((word_counts >= 4) & (word_counts <= 10)).sum()
    ),
    "medium_comments_11_to_30_words": int(
        ((word_counts >= 11) & (word_counts <= 30)).sum()
    ),
    "long_comments_over_30_words": int((word_counts > 30).sum()),
    "comments_containing_urls": int(url_count),
    "duplicate_text_records": int(duplicate_mask.sum()),
    "average_word_count": round(word_counts.mean(), 2),
    "median_word_count": round(word_counts.median(), 2),
    "average_character_count": round(char_counts.mean(), 2),
    "median_character_count": round(char_counts.median(), 2),
    "script_latin": int((script_categories == "Latin Script").sum()),
    "script_devanagari": int((script_categories == "Devanagari").sum()),
    "script_tamil": int((script_categories == "Tamil").sum()),
    "script_telugu": int((script_categories == "Telugu").sum()),
    "script_kannada": int((script_categories == "Kannada").sum()),
    "script_bengali": int((script_categories == "Bengali").sum()),
    "script_malayalam": int((script_categories == "Malayalam").sum()),
    "script_mixed": int((script_categories == "Mixed Script").sum()),
    "script_other_emoji_symbols": int(
        (script_categories == "Other/Emoji/Symbols").sum()
    ),
    "script_empty": int((script_categories == "Empty").sum()),
    "sampling_method": "YouTube top-level comments ordered by time",
    "replies_included": False,
    "profiled_at_utc": datetime.now(timezone.utc).isoformat()
}])

profile.to_csv(OUTPUT_FILE, index=False)

print("=" * 70)
print("BATCH 08 OVERALL QUALITY PROFILE")
print("=" * 70)

display(profile.T.rename(columns={0: "value"}))

print("\nScript distribution:")
display(
    script_categories.value_counts()
    .rename_axis("script_category")
    .reset_index(name="records")
)

print("\nSaved:", OUTPUT_FILE)

print(
    "\nNote: Script classification is not language identification. "
    "Latin script may represent English, transliterated Indian languages, "
    "or other languages."
)

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 08 MOVIE-WISE AND VIDEO-WISE QUALITY
# ============================================================

import pandas as pd
import re
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

RAW_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08.csv"

MOVIE_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_08.csv"
VIDEO_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_VIDEO_QUALITY_BATCH_08.csv"

for path in [MOVIE_FILE, VIDEO_FILE]:
    if path.exists():
        raise FileExistsError(
            f"Output already exists; refusing to overwrite: {path}"
        )

df = pd.read_csv(RAW_FILE)

df["comment_text"] = df["comment_text"].fillna("").astype(str)

df["word_count"] = df["comment_text"].apply(
    lambda x: len(re.findall(r"\b\w+\b", x, flags=re.UNICODE))
)

df["character_count"] = df["comment_text"].str.len()

df["is_empty"] = df["comment_text"].str.strip().eq("")
df["is_very_short"] = df["word_count"].le(3)
df["contains_url"] = df["comment_text"].str.contains(
    r"https?://|www\.",
    case=False,
    regex=True
)

# Temporary normalized text for duplicate analysis
df["normalized_text"] = (
    df["comment_text"].str.lower()
    .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["is_duplicate_text"] = (
    df["normalized_text"].ne("") &
    df["normalized_text"].duplicated(keep=False)
)

def build_quality_profile(group, group_columns):
    records = []

    for keys, part in group:
        if not isinstance(keys, tuple):
            keys = (keys,)

        record = dict(zip(group_columns, keys))

        record.update({
            "comment_records": len(part),
            "unique_comment_ids": part["comment_id"].nunique(),
            "unique_videos": part["video_id"].nunique(),
            "empty_comments": int(part["is_empty"].sum()),
            "very_short_comments_le_3_words": int(
                part["is_very_short"].sum()
            ),
            "comments_containing_urls": int(
                part["contains_url"].sum()
            ),
            "duplicate_text_records": int(
                part["is_duplicate_text"].sum()
            ),
            "average_word_count": round(
                part["word_count"].mean(), 2
            ),
            "median_word_count": round(
                part["word_count"].median(), 2
            ),
            "average_character_count": round(
                part["character_count"].mean(), 2
            ),
            "median_character_count": round(
                part["character_count"].median(), 2
            )
        })

        records.append(record)

    return pd.DataFrame(records)

movie_profile = build_quality_profile(
    df.groupby(
        ["movie_record_id", "movie_title", "movie_year", "language"],
        dropna=False
    ),
    ["movie_record_id", "movie_title", "movie_year", "language"]
)

video_profile = build_quality_profile(
    df.groupby(
        [
            "movie_record_id", "movie_title",
            "video_id", "video_title", "channel_title"
        ],
        dropna=False
    ),
    [
        "movie_record_id", "movie_title",
        "video_id", "video_title", "channel_title"
    ]
)

movie_profile.to_csv(MOVIE_FILE, index=False)
video_profile.to_csv(VIDEO_FILE, index=False)

print("=" * 70)
print("BATCH 08 MOVIE-WISE QUALITY PROFILE")
print("=" * 70)

display(movie_profile)

print("\n" + "=" * 70)
print("BATCH 08 VIDEO-WISE QUALITY PROFILE")
print("=" * 70)

display(video_profile)

print("\nSaved reports:")
print(MOVIE_FILE)
print(VIDEO_FILE)

print("\nValidation:")
print("Movie profile rows:", len(movie_profile))
print("Video profile rows:", len(video_profile))
print("Movie comments total:", movie_profile["comment_records"].sum())
print("Video comments total:", video_profile["comment_records"].sum())

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 08 DUPLICATE TEXT AUDIT
# ============================================================

import pandas as pd
import re
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

RAW_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08.csv"

OUTPUT_FILE = (
    COMMENTS_DIR
    / "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_BATCH_08.csv"
)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        f"Output already exists; refusing to overwrite: {OUTPUT_FILE}"
    )

df = pd.read_csv(RAW_FILE)

df["comment_text"] = df["comment_text"].fillna("").astype(str)

# Normalize only for comparison; original comments remain unchanged
df["normalized_text"] = (
    df["comment_text"].str.lower()
    .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Exclude empty normalized comments from duplicate matching
eligible = df[df["normalized_text"].ne("")].copy()

duplicate_groups = (
    eligible.groupby("normalized_text")
    .filter(lambda x: len(x) > 1)
    .groupby("normalized_text", sort=False)
)

audit_rows = []

for group_number, (normalized_text, group) in enumerate(
    duplicate_groups, start=1
):
    unique_videos = group["video_id"].nunique()
    unique_movies = group["movie_record_id"].nunique()

    if unique_videos > 1:
        duplicate_scope = "Across Videos"
    else:
        duplicate_scope = "Within Video"

    audit_rows.append({
        "duplicate_group_id": f"B08_DUP_{group_number:03d}",
        "normalized_text": normalized_text,
        "record_count": len(group),
        "unique_comment_ids": group["comment_id"].nunique(),
        "unique_videos": unique_videos,
        "unique_movies": unique_movies,
        "duplicate_scope": duplicate_scope,
        "movie_record_ids": " | ".join(
            sorted(group["movie_record_id"].dropna().astype(str).unique())
        ),
        "movie_titles": " | ".join(
            sorted(group["movie_title"].dropna().astype(str).unique())
        ),
        "video_ids": " | ".join(
            sorted(group["video_id"].dropna().astype(str).unique())
        ),
        "channels": " | ".join(
            sorted(group["channel_title"].dropna().astype(str).unique())
        ),
        "original_comment_examples": " || ".join(
            group["comment_text"].drop_duplicates().head(3).tolist()
        )
    })

audit = pd.DataFrame(audit_rows)

if not audit.empty:
    audit = audit.sort_values(
        ["record_count", "unique_videos"],
        ascending=[False, False]
    ).reset_index(drop=True)

audit.to_csv(OUTPUT_FILE, index=False)

print("=" * 70)
print("BATCH 08 DUPLICATE TEXT AUDIT")
print("=" * 70)

print("Repeated-text groups:", len(audit))
print(
    "Records in repeated-text groups:",
    audit["record_count"].sum() if not audit.empty else 0
)

print("\nDuplicate scope distribution:")
if not audit.empty:
    display(
        audit["duplicate_scope"]
        .value_counts()
        .rename_axis("scope")
        .reset_index(name="groups")
    )

    print("\nLargest repeated-text groups:")
    display(audit.head(15))
else:
    print("No repeated-text groups found.")

print("\nSaved:", OUTPUT_FILE)

print(
    "\nImportant: repeated text is an audit flag, not proof of spam, "
    "copying, or invalid comments. No records have been deleted."
)

In [ ]:
# ============================================================
# NOTEBOOK 10 — PILOT TO BATCH 08 CONSOLIDATION PREFLIGHT
# Read-only schema and integrity inspection
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

COMMENTS_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

OLD_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_07.csv"
)

BATCH08_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08.csv"

MANIFEST_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_08.csv"
)

LOG_FILE = (
    COMMENTS_DIR
    / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_08.csv"
)

for path in [OLD_FILE, BATCH08_FILE, MANIFEST_FILE, LOG_FILE]:
    if not path.exists():
        raise FileNotFoundError(path)

old = pd.read_csv(OLD_FILE)
batch08 = pd.read_csv(BATCH08_FILE)
manifest = pd.read_csv(MANIFEST_FILE)
logs = pd.read_csv(LOG_FILE)

print("=" * 70)
print("PILOT–BATCH 07 CONSOLIDATED DATASET")
print("=" * 70)

print("Shape:", old.shape)
print("Columns:", old.columns.tolist())
print("Unique comment IDs:", old["comment_id"].nunique())
print("Unique videos:", old["video_id"].nunique())
print("Unique movies:", old["movie_record_id"].nunique())

print("\n" + "=" * 70)
print("BATCH 08 RAW COMMENTS")
print("=" * 70)

print("Shape:", batch08.shape)
print("Columns:", batch08.columns.tolist())
print("Unique comment IDs:", batch08["comment_id"].nunique())
print("Unique videos:", batch08["video_id"].nunique())
print("Unique movies:", batch08["movie_record_id"].nunique())

print("\n" + "=" * 70)
print("BATCH 08 MANIFEST")
print("=" * 70)

print("Shape:", manifest.shape)
display(
    manifest[
        [
            "movie_record_id",
            "video_id",
            "manual_source_type",
            "manual_verification_status",
            "acquisition_status",
            "target_comments"
        ]
    ]
)

print("\n" + "=" * 70)
print("BATCH 08 ACQUISITION LOG")
print("=" * 70)

display(logs)

print("\n" + "=" * 70)
print("SCHEMA COMPATIBILITY")
print("=" * 70)

print("Old-only columns:", sorted(set(old.columns) - set(batch08.columns)))
print("Batch08-only columns:", sorted(set(batch08.columns) - set(old.columns)))

print("\nPotential overlapping comment IDs:",
      len(set(old["comment_id"]) & set(batch08["comment_id"])))

print("\nNo files modified.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — CONSOLIDATE PILOT + BATCH 01–08
# ============================================================

import pandas as pd
import re
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

COMMENTS_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

OLD_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_07.csv"
)

BATCH08_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_BATCH_08.csv"

MANIFEST_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_08.csv"
)

LOG_FILE = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_08.csv"

OUTPUT_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_08.csv"
)

# Safety checks
for path in [OLD_FILE, BATCH08_FILE, MANIFEST_FILE, LOG_FILE]:
    if not path.exists():
        raise FileNotFoundError(path)

if OUTPUT_FILE.exists():
    raise FileExistsError(
        f"Output already exists. No overwrite performed: {OUTPUT_FILE}"
    )

old = pd.read_csv(OLD_FILE)
batch08 = pd.read_csv(BATCH08_FILE)
manifest = pd.read_csv(MANIFEST_FILE)
logs = pd.read_csv(LOG_FILE)

# Validate source integrity
assert old["comment_id"].is_unique
assert batch08["comment_id"].is_unique
assert not set(old["comment_id"]) & set(batch08["comment_id"])

assert len(batch08) == 900
assert batch08["video_id"].nunique() == 18
assert manifest["acquisition_status"].eq("Completed").all()

# Prepare lookup dictionaries
source_type_map = manifest.set_index("video_id")[
    "manual_source_type"
].to_dict()

timestamp_map = logs.set_index("video_id")[
    "acquired_at_utc"
].to_dict()

# Build Batch 08 in the existing consolidated schema
new = pd.DataFrame(index=batch08.index)

new["movie_record_id"] = batch08["movie_record_id"]
new["movie_title"] = batch08["movie_title"]
new["movie_year"] = batch08["movie_year"]
new["video_id"] = batch08["video_id"]
new["video_title"] = batch08["video_title"]
new["channel_title"] = batch08["channel_title"]
new["comment_id"] = batch08["comment_id"]
new["comment_text"] = batch08["comment_text"]

# Not available in the Batch 08 API extraction
new["author_channel_id"] = pd.NA

new["published_at"] = batch08["published_at"]
new["updated_at"] = batch08["updated_at"]
new["like_count"] = batch08["like_count"]

# Replies were not acquired
new["comment_reply_count"] = pd.NA
new["total_reply_count"] = pd.NA
new["reply_count"] = pd.NA

new["retrieved_at"] = batch08["video_id"].map(timestamp_map)
new["source_batch"] = "Batch 08"

# Normalized text for duplicate analysis only
new["normalized_comment_text"] = (
    batch08["comment_text"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(
        r"https?://\S+|www\.\S+",
        " ",
        regex=True
    )
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

new["source_type"] = batch08["video_id"].map(source_type_map)
new["retrieval_order"] = batch08["retrieval_order"]
new["batch_id"] = "Batch 08"

# Preserve explicit series part information where present
new["series_part"] = batch08["video_title"].apply(
    lambda x: (
        re.search(r"\bPart\s*(\d+)\s*/\s*(\d+)\b", str(x), re.I)
    )
)

new["series_part"] = new["series_part"].apply(
    lambda m: f"Part {m.group(1)}/{m.group(2)}" if m else pd.NA
)

new["comment_author"] = batch08["author_name"]
new["reply_count"] = pd.NA
new["acquired_at_utc"] = batch08["video_id"].map(timestamp_map)
new["author_name"] = batch08["author_name"]
new["comment_position"] = batch08["comment_position"]
new["sampling_order"] = batch08["retrieval_order"]
new["sampling_unit"] = "top_level_comment"
new["author_display_name"] = batch08["author_name"]
new["comment_source"] = "YouTube Top-Level Comment"
new["acquisition_order"] = batch08["retrieval_order"]
new["acquisition_timestamp_utc"] = batch08["video_id"].map(timestamp_map)

# Match the exact existing schema and order
new = new[old.columns]

# Concatenate
combined = pd.concat(
    [old, new],
    ignore_index=True
)

# Final integrity checks
assert combined.shape[0] == 6525
assert combined.shape[1] == 32
assert combined["comment_id"].is_unique
assert combined["video_id"].nunique() == 80

# Save new consolidated file
combined.to_csv(OUTPUT_FILE, index=False)

print("=" * 70)
print("PILOT–BATCH 08 CONSOLIDATION COMPLETED")
print("=" * 70)

print("Previous records:", len(old))
print("Batch 08 records:", len(new))
print("Consolidated shape:", combined.shape)
print("Unique comment IDs:", combined["comment_id"].nunique())
print("Unique videos:", combined["video_id"].nunique())
print("Unique movies:", combined["movie_record_id"].nunique())

print("\nSource batch distribution:")
display(
    combined["source_batch"]
    .value_counts(dropna=False)
    .rename_axis("source_batch")
    .reset_index(name="records")
)

print("\nBatch 08 source type distribution:")
display(
    new["source_type"]
    .value_counts(dropna=False)
    .rename_axis("source_type")
    .reset_index(name="records")
)

print("\nNull counts in Batch 08 mapped records:")
display(new.isna().sum().rename("null_count").to_frame())

print("\nSaved:", OUTPUT_FILE)
print("\nPrevious Pilot–Batch 07 file was not modified.")

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT, "01_data", "processed", "audience_sentiment"
)

CONSOLIDATED_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_08.csv"
)

MASTER_FILE = os.path.join(
    PROJECT_ROOT, "01_data", "processed",
    "ICMB_130_MASTER_DATA.csv"
)

COVERAGE_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_08.csv"
)

DUPLICATE_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_PILOT_BATCH_01_08.csv"
)

# Never overwrite existing reports
for path in [COVERAGE_FILE, DUPLICATE_FILE]:
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists; stopping to prevent overwrite:\n{path}"
        )

# ============================================================
# 2. LOAD AND VALIDATE INPUTS
# ============================================================

comments = pd.read_csv(CONSOLIDATED_FILE)
master = pd.read_csv(MASTER_FILE)

required_comment_cols = {
    "movie_record_id", "movie_title", "video_id",
    "comment_id", "comment_text", "source_batch"
}

required_master_cols = {
    "movie_record_id", "movie_title", "year",
    "rank", "language"
}

assert required_comment_cols.issubset(comments.columns), (
    f"Missing comment columns: "
    f"{required_comment_cols - set(comments.columns)}"
)

assert required_master_cols.issubset(master.columns), (
    f"Missing master columns: "
    f"{required_master_cols - set(master.columns)}"
)

assert master["movie_record_id"].is_unique, (
    "Master dataset contains duplicate movie_record_id values."
)

assert comments["comment_id"].is_unique, (
    "Consolidated dataset contains duplicate comment IDs."
)

assert set(comments["movie_record_id"].dropna()).issubset(
    set(master["movie_record_id"])
), "Some comment records do not match the master movie dataset."

# ============================================================
# 3. MOVIE-LEVEL COVERAGE REPORT
# ============================================================

coverage_counts = (
    comments.groupby("movie_record_id")
    .agg(
        comment_records=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        acquisition_batches=("source_batch", "nunique")
    )
    .reset_index()
)

coverage = master[
    ["movie_record_id", "movie_title", "year", "rank", "language"]
].merge(
    coverage_counts,
    on="movie_record_id",
    how="left"
)

count_cols = [
    "comment_records",
    "unique_comment_ids",
    "unique_videos",
    "acquisition_batches"
]

coverage[count_cols] = coverage[count_cols].fillna(0).astype(int)

coverage["coverage_status"] = np.where(
    coverage["comment_records"] > 0,
    "Covered",
    "No Comments Retrieved"
)

coverage["coverage_category"] = np.select(
    [
        coverage["comment_records"] >= 100,
        coverage["comment_records"].between(50, 99),
        coverage["comment_records"].between(1, 49)
    ],
    ["Higher", "Moderate", "Low"],
    default="No Comments"
)

coverage = coverage.sort_values(
    ["year", "rank"]
).reset_index(drop=True)

coverage.to_csv(COVERAGE_FILE, index=False)

# ============================================================
# 4. DUPLICATE-TEXT AUDIT
# ============================================================

def normalize_comment(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

audit_data = comments.copy()

audit_data["audit_normalized_text"] = (
    audit_data["comment_text"].apply(normalize_comment)
)

# Exclude empty normalized text from duplicate grouping.
# Original comment records remain untouched.
valid_text = audit_data[
    audit_data["audit_normalized_text"].ne("")
].copy()

duplicate_groups = (
    valid_text.groupby("audit_normalized_text")
    .agg(
        record_count=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        unique_movies=("movie_record_id", "nunique"),
        unique_batches=("source_batch", "nunique"),
        movie_titles=(
            "movie_title",
            lambda x: " | ".join(sorted(x.dropna().astype(str).unique()))
        ),
        video_ids=(
            "video_id",
            lambda x: " | ".join(sorted(x.dropna().astype(str).unique()))
        ),
        source_batches=(
            "source_batch",
            lambda x: " | ".join(sorted(x.dropna().astype(str).unique()))
        )
    )
    .reset_index()
)

duplicate_groups = duplicate_groups[
    duplicate_groups["record_count"] > 1
].copy()

duplicate_groups["duplicate_scope"] = np.where(
    duplicate_groups["unique_videos"] > 1,
    "Across Videos",
    "Within Video"
)

duplicate_groups = duplicate_groups.sort_values(
    ["record_count", "unique_movies"],
    ascending=[False, False]
).reset_index(drop=True)

duplicate_groups.insert(
    0, "duplicate_group_id",
    [f"DUP_{i:04d}" for i in range(1, len(duplicate_groups) + 1)]
)

duplicate_groups.to_csv(DUPLICATE_FILE, index=False)

# ============================================================
# 5. VALIDATION AND SUMMARY
# ============================================================

print("=" * 70)
print("PILOT–BATCH 08 COVERAGE AND DUPLICATE AUDIT COMPLETED")
print("=" * 70)

print("\nCONSOLIDATED DATASET")
print("Comment records:", len(comments))
print("Unique comment IDs:", comments["comment_id"].nunique())
print("Unique videos:", comments["video_id"].nunique())
print("Movies with comments:", comments["movie_record_id"].nunique())
print("Master movies:", len(master))

print("\nMOVIE COVERAGE")
print(coverage["coverage_status"].value_counts().to_string())

print("\nCOVERAGE CATEGORIES")
print(coverage["coverage_category"].value_counts().to_string())

print("\nLANGUAGE-WISE COVERAGE")
language_summary = (
    coverage.groupby("language")
    .agg(
        total_movies=("movie_record_id", "count"),
        covered_movies=(
            "coverage_status",
            lambda x: (x == "Covered").sum()
        ),
        uncovered_movies=(
            "coverage_status",
            lambda x: (x == "No Comments Retrieved").sum()
        ),
        comment_records=("comment_records", "sum")
    )
    .reset_index()
)

display(language_summary)

print("\nDUPLICATE-TEXT AUDIT")
print("Repeated-text groups:", len(duplicate_groups))
print(
    "Records in repeated-text groups:",
    duplicate_groups["record_count"].sum()
    if not duplicate_groups.empty else 0
)

if not duplicate_groups.empty:
    print("\nDuplicate scope:")
    print(duplicate_groups["duplicate_scope"].value_counts().to_string())

    print("\nLargest repeated-text groups:")
    display(
        duplicate_groups[
            [
                "duplicate_group_id",
                "record_count",
                "unique_videos",
                "unique_movies",
                "duplicate_scope",
                "audit_normalized_text"
            ]
        ].head(10)
    )

print("\nSAVED REPORTS")
print(COVERAGE_FILE)
print(DUPLICATE_FILE)

# Final integrity checks
assert len(coverage) == len(master) == 130
assert coverage["comment_records"].sum() == len(comments)
assert coverage["unique_comment_ids"].sum() == comments["comment_id"].nunique()

print("\nAll validation checks passed.")
print("No comments were deleted or modified.")

In [ ]:
import os
import pandas as pd
from IPython.display import display

PROJECT_ROOT = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

REPORT_DIR = os.path.join(
    PROJECT_ROOT, "01_data", "processed", "audience_sentiment"
)

files = {
    "Previous planning report": (
        "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_07.csv"
    ),
    "Previous planning summary": (
        "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_07.csv"
    ),
    "Previous language summary": (
        "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_07.csv"
    )
}

for label, filename in files.items():
    path = os.path.join(REPORT_DIR, filename)

    print("\n" + "=" * 75)
    print(label)
    print("=" * 75)

    if not os.path.exists(path):
        print("FILE NOT FOUND:", filename)
        continue

    df = pd.read_csv(path)

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(10))

    if "acquisition_priority" in df.columns:
        print("\nExisting priority distribution:")
        print(df["acquisition_priority"].value_counts(dropna=False))

print("\nRead-only inspection completed. No files modified.")

In [ ]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT, "01_data", "processed", "audience_sentiment"
)

COVERAGE_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_08.csv"
)

CONSOLIDATED_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_08.csv"
)

REPORT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_08.csv"
)

SUMMARY_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_08.csv"
)

LANGUAGE_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_08.csv"
)

# Prevent overwriting existing reports
for path in [REPORT_FILE, SUMMARY_FILE, LANGUAGE_FILE]:
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists; stopping:\n{path}"
        )

# ============================================================
# 2. LOAD INPUTS
# ============================================================

coverage = pd.read_csv(COVERAGE_FILE)
comments = pd.read_csv(CONSOLIDATED_FILE)

assert len(coverage) == 130
assert comments["comment_id"].is_unique

# ============================================================
# 3. MOVIE-LEVEL PLANNING REPORT
# ============================================================

planning = coverage.copy()

planning["coverage_category"] = np.select(
    [
        planning["comment_records"] >= 100,
        planning["comment_records"].between(50, 99),
        planning["comment_records"].between(1, 49)
    ],
    [
        "Higher (100+)",
        "Moderate (50-99)",
        "Low (1-49)"
    ],
    default="No Comments"
)

planning["acquisition_priority"] = np.select(
    [
        planning["comment_records"] == 0,
        planning["comment_records"].between(1, 49),
        planning["comment_records"].between(50, 99),
        planning["comment_records"] >= 100
    ],
    [
        "High — No Comments",
        "High — Low Coverage",
        "Medium — Moderate Coverage",
        "Covered — Monitor"
    ],
    default="Review Required"
)

planning["additional_comment_target"] = (
    100 - planning["comment_records"]
).clip(lower=0).astype(int)

planning = planning[
    [
        "movie_record_id",
        "movie_title",
        "year",
        "rank",
        "language",
        "comment_records",
        "unique_comment_ids",
        "unique_videos",
        "acquisition_batches",
        "coverage_status",
        "coverage_category",
        "acquisition_priority",
        "additional_comment_target"
    ]
].sort_values(
    ["acquisition_priority", "year", "rank"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ============================================================
# 4. COVERAGE CATEGORY SUMMARY
# ============================================================

summary = (
    planning.groupby("coverage_category")
    .agg(
        movie_count=("movie_record_id", "count"),
        existing_comments=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reset_index()
)

category_order = [
    "No Comments",
    "Low (1-49)",
    "Moderate (50-99)",
    "Higher (100+)"
]

# Normalize labels for summary table
summary["coverage_category"] = summary[
    "coverage_category"
].replace({
    "Low (1-49)": "Low (1-49)",
    "Moderate (50-99)": "Moderate (50-99)"
})

summary = summary.set_index(
    "coverage_category"
).reindex(
    category_order,
    fill_value=0
).reset_index()

# ============================================================
# 5. LANGUAGE-WISE SUMMARY
# ============================================================

language_summary = (
    planning.groupby("language")
    .agg(
        total_movies=("movie_record_id", "count"),
        covered_movies=(
            "coverage_status",
            lambda x: (x == "Covered").sum()
        ),
        uncovered_movies=(
            "coverage_status",
            lambda x: (x == "No Comments Retrieved").sum()
        ),
        comment_records=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reset_index()
)

# ============================================================
# 6. VALIDATION
# ============================================================

assert len(planning) == 130
assert planning["comment_records"].sum() == len(comments)
assert planning["additional_comment_target"].sum() == (
    100 * 130 - planning["comment_records"].clip(upper=100).sum()
)

assert language_summary["total_movies"].sum() == 130
assert language_summary["comment_records"].sum() == len(comments)

# ============================================================
# 7. SAVE NEW REPORTS
# ============================================================

planning.to_csv(REPORT_FILE, index=False)
summary.to_csv(SUMMARY_FILE, index=False)
language_summary.to_csv(LANGUAGE_FILE, index=False)

# ============================================================
# 8. DISPLAY RESULTS
# ============================================================

print("=" * 75)
print("PILOT–BATCH 08 ACQUISITION PLANNING REPORTS CREATED")
print("=" * 75)

print("\nMOVIE-LEVEL PRIORITY DISTRIBUTION")
print(planning["acquisition_priority"].value_counts().to_string())

print("\nCOVERAGE SUMMARY")
display(summary)

print("\nLANGUAGE-WISE SUMMARY")
display(language_summary)

print("\nTOTALS")
print("Movies:", len(planning))
print("Existing comments:", planning["comment_records"].sum())
print("Provisional additional target:",
      planning["additional_comment_target"].sum())

print("\nSAVED FILES")
print(REPORT_FILE)
print(SUMMARY_FILE)
print(LANGUAGE_FILE)

print("\nAll validation checks passed.")
print("Previous Pilot–Batch 07 reports were not modified.")

In [ ]:
import os
import pandas as pd
from IPython.display import display

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT, "01_data", "processed", "audience_sentiment"
)

PLANNING_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_08.csv"
)

MASTER_FILE = os.path.join(
    PROJECT_ROOT, "01_data", "processed",
    "ICMB_130_MASTER_DATA.csv"
)

# ============================================================
# 2. LOAD DATA
# ============================================================

planning = pd.read_csv(PLANNING_FILE)
master = pd.read_csv(MASTER_FILE)

uncovered = planning[
    planning["comment_records"] == 0
].copy()

assert len(uncovered) == 68
assert uncovered["movie_record_id"].is_unique

# Add India Gross for prioritization context
gross_columns = [
    "movie_record_id",
    "india_gross_cr",
    "india_gross_source"
]

gross_columns = [
    col for col in gross_columns if col in master.columns
]

uncovered = uncovered.merge(
    master[gross_columns],
    on="movie_record_id",
    how="left"
)

# ============================================================
# 3. YEAR-WISE COVERAGE GAP
# ============================================================

print("=" * 75)
print("BATCH 09 — UNCOVERED MOVIE ANALYSIS")
print("=" * 75)

print("\n1. YEAR-WISE UNCOVERED MOVIES")

year_summary = (
    uncovered.groupby("year")
    .agg(
        uncovered_movies=("movie_record_id", "count"),
        languages=("language", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
    .sort_values(
        ["uncovered_movies", "year"],
        ascending=[False, True]
    )
)

display(year_summary)

# ============================================================
# 4. LANGUAGE-WISE GAP
# ============================================================

print("\n2. LANGUAGE-WISE UNCOVERED MOVIES")

language_summary = (
    uncovered.groupby("language")
    .agg(
        uncovered_movies=("movie_record_id", "count"),
        earliest_year=("year", "min"),
        latest_year=("year", "max")
    )
    .reset_index()
    .sort_values("uncovered_movies", ascending=False)
)

display(language_summary)

# ============================================================
# 5. INDIA GROSS DISTRIBUTION
# ============================================================

print("\n3. INDIA GROSS DISTRIBUTION")

if "india_gross_cr" in uncovered.columns:
    uncovered["india_gross_cr"] = pd.to_numeric(
        uncovered["india_gross_cr"],
        errors="coerce"
    )

    gross_summary = (
        uncovered.groupby("language")
        .agg(
            movies=("movie_record_id", "count"),
            average_india_gross=("india_gross_cr", "mean"),
            median_india_gross=("india_gross_cr", "median"),
            highest_india_gross=("india_gross_cr", "max")
        )
        .reset_index()
    )

    display(gross_summary)

# ============================================================
# 6. COMPLETE UNCOVERED MOVIE LIST
# ============================================================

print("\n4. ALL UNCOVERED MOVIES")

display(
    uncovered.sort_values(
        ["language", "year", "rank"]
    )[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "rank",
            "language",
            "india_gross_cr",
            "additional_comment_target"
        ]
    ].reset_index(drop=True)
)

# ============================================================
# 7. PRIORITY CANDIDATES
# ============================================================

print("\n5. TOP INDIA-GROSS MOVIES AMONG UNCOVERED RECORDS")

priority_candidates = uncovered.sort_values(
    "india_gross_cr",
    ascending=False,
    na_position="last"
)

display(
    priority_candidates[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "language",
            "india_gross_cr"
        ]
    ].head(20).reset_index(drop=True)
)

print("\nRead-only analysis completed.")
print("No files modified. No YouTube API calls made.")

In [ ]:
import os
import pandas as pd
from datetime import datetime, timezone
from IPython.display import display

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = os.path.join(
    PROJECT_ROOT, "01_data", "processed", "audience_sentiment"
)

MASTER_FILE = os.path.join(
    PROJECT_ROOT, "01_data", "processed",
    "ICMB_130_MASTER_DATA.csv"
)

COVERAGE_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_08.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_09_MOVIE_ALLOCATION.csv"
)

# ============================================================
# 2. APPROVED BATCH 09 MOVIES
# ============================================================

approved_ids = [
    "2025_3",
    "2024_3",
    "2023_2",
    "2023_3",
    "2018_1",
    "2017_2",
    "2016_2",
    "2024_4",
    "2019_2",
    "2022_5"
]

# Prevent accidental overwrite
if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        f"Batch 09 allocation already exists:\n{OUTPUT_FILE}"
    )

# ============================================================
# 3. LOAD REFERENCE DATA
# ============================================================

master = pd.read_csv(MASTER_FILE)
coverage = pd.read_csv(COVERAGE_FILE)

assert master["movie_record_id"].is_unique
assert coverage["movie_record_id"].is_unique

# ============================================================
# 4. VALIDATE APPROVED MOVIES
# ============================================================

allocation = master[
    master["movie_record_id"].isin(approved_ids)
].copy()

assert len(allocation) == 10, (
    f"Expected 10 movies, found {len(allocation)}"
)

assert set(allocation["movie_record_id"]) == set(approved_ids)

allocation = allocation.merge(
    coverage[
        [
            "movie_record_id",
            "comment_records",
            "coverage_status",
            "coverage_category"
        ]
    ],
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

assert allocation["coverage_status"].eq(
    "No Comments Retrieved"
).all(), "At least one selected movie already has comments."

# ============================================================
# 5. ADD ACQUISITION PLANNING FIELDS
# ============================================================

allocation["acquisition_batch"] = "Batch 09"
allocation["existing_comments"] = allocation["comment_records"]
allocation["target_comments"] = 100
allocation["additional_comment_target"] = (
    allocation["target_comments"] -
    allocation["existing_comments"]
).clip(lower=0)

allocation["allocation_status"] = "Approved"
allocation["allocation_timestamp_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

allocation = allocation[
    [
        "movie_record_id",
        "movie_title",
        "year",
        "rank",
        "language",
        "india_gross_cr",
        "tmdb_id",
        "existing_comments",
        "target_comments",
        "additional_comment_target",
        "acquisition_batch",
        "allocation_status",
        "allocation_timestamp_utc"
    ]
].sort_values(
    ["language", "year", "rank"]
).reset_index(drop=True)

# ============================================================
# 6. SAVE
# ============================================================

allocation.to_csv(OUTPUT_FILE, index=False)

# ============================================================
# 7. VALIDATION
# ============================================================

saved = pd.read_csv(OUTPUT_FILE)

assert saved.shape[0] == 10
assert saved["movie_record_id"].is_unique
assert saved["existing_comments"].sum() == 0
assert saved["additional_comment_target"].sum() == 1000
assert saved["allocation_status"].eq("Approved").all()

print("=" * 70)
print("BATCH 09 MOVIE ALLOCATION SAVED")
print("=" * 70)

print("Movies:", len(saved))
print("Existing comments:", saved["existing_comments"].sum())
print("Planned comment target:", saved["additional_comment_target"].sum())

print("\nLanguage distribution:")
print(saved["language"].value_counts().to_string())

print("\nApproved movie allocation:")
display(saved)

print("\nSaved file:")
print(OUTPUT_FILE)

print("\nAll validation checks passed.")
print("No YouTube API calls made.")

In [ ]:
import os
import pandas as pd
from IPython.display import display

PROJECT_ROOT = (
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

TARGET_FILES = [
    "ICMB_130_TMDB_MAPPING.csv",
    "ICMB_130_TMDB_METADATA.csv",
    "ICMB_130_MASTER_DATA.csv",
    "ICMB_130_YOUTUBE_BATCH_09_MOVIE_ALLOCATION.csv"
]

# Locate exact files
file_paths = {}

for root, dirs, files in os.walk(PROJECT_ROOT):
    for filename in TARGET_FILES:
        if filename in files:
            file_paths[filename] = os.path.join(root, filename)

print("=" * 75)
print("TMDB MAPPING — SCHEMA AND COVERAGE INSPECTION")
print("=" * 75)

dataframes = {}

for filename in TARGET_FILES:
    path = file_paths.get(filename)

    print(f"\n{'-' * 65}")
    print(filename)

    if not path:
        print("NOT FOUND")
        continue

    df = pd.read_csv(path)
    dataframes[filename] = df

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    display(df.head(3))

    id_cols = [
        col for col in df.columns
        if "tmdb" in col.lower() and "id" in col.lower()
    ]

    for col in id_cols:
        print(
            f"{col}: {df[col].notna().sum()} non-null, "
            f"{df[col].nunique()} unique"
        )

# Check Batch 09 IDs against mapping and metadata
allocation = dataframes[
    "ICMB_130_YOUTUBE_BATCH_09_MOVIE_ALLOCATION.csv"
]

for filename in [
    "ICMB_130_TMDB_MAPPING.csv",
    "ICMB_130_TMDB_METADATA.csv"
]:
    if filename not in dataframes:
        continue

    df = dataframes[filename]

    print(f"\n{'=' * 75}")
    print("BATCH 09 MATCH CHECK:", filename)

    possible_keys = [
        c for c in df.columns
        if c in ["movie_record_id", "movie_title", "original_title"]
    ]

    print("Potential matching columns:", possible_keys)

    if "movie_record_id" in df.columns:
        check = allocation[
            ["movie_record_id", "movie_title"]
        ].merge(
            df,
            on="movie_record_id",
            how="left",
            suffixes=("_allocation", "_source")
        )

        print(
            "Matched Batch 09 records:",
            check["movie_record_id"].notna().sum()
        )
        display(check.head(10))

print("\nInspection complete. No files modified.")

In [ ]:
import os
import pandas as pd
from IPython.display import display

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

# Locate files anywhere inside project directory
required_files = [
    "ICMB_130_MASTER_DATA.csv",
    "ICMB_130_TMDB_MAPPING.csv",
    "ICMB_130_TMDB_METADATA.csv",
    "ICMB_130_YOUTUBE_BATCH_09_MOVIE_ALLOCATION.csv"
]

found = {}

for folder, dirs, files in os.walk(ROOT):
    for filename in required_files:
        if filename in files:
            found[filename] = os.path.join(folder, filename)

missing = set(required_files) - set(found)

if missing:
    raise FileNotFoundError(
        f"Files not found in project: {missing}"
    )

print("Located files:")
for name, path in found.items():
    print(name, "->", path)

# Load
master = pd.read_csv(found["ICMB_130_MASTER_DATA.csv"])
mapping = pd.read_csv(found["ICMB_130_TMDB_MAPPING.csv"])
metadata = pd.read_csv(found["ICMB_130_TMDB_METADATA.csv"])
allocation = pd.read_csv(
    found["ICMB_130_YOUTUBE_BATCH_09_MOVIE_ALLOCATION.csv"]
)

# Validate original mapping
assert len(master) == 130
assert len(mapping) == 130
assert len(metadata) == 130

assert mapping["movie_record_id"].is_unique
assert metadata["movie_record_id"].is_unique
assert mapping["tmdb_id"].notna().all()
assert metadata["tmdb_id"].notna().all()

id_check = mapping[
    ["movie_record_id", "tmdb_id"]
].merge(
    metadata[["movie_record_id", "tmdb_id"]],
    on="movie_record_id",
    suffixes=("_mapping", "_metadata"),
    validate="one_to_one"
)

assert (
    id_check["tmdb_id_mapping"].astype("Int64")
    == id_check["tmdb_id_metadata"].astype("Int64")
).all()

# Restore master using metadata
metadata_fields = {
    "tmdb_id": "tmdb_id",
    "original_title": "tmdb_original_title",
    "release_date": "tmdb_release_date",
    "runtime": "runtime",
    "genres": "genres",
    "original_language": "tmdb_original_language",
    "production_countries": "production_countries",
    "budget": "budget",
    "worldwide_revenue": "revenue"
}

meta = metadata[
    ["movie_record_id"] + list(metadata_fields.values())
].rename(columns={
    source: target
    for target, source in metadata_fields.items()
})

repaired_master = master.drop(
    columns=list(metadata_fields.keys()),
    errors="ignore"
).merge(
    meta,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

repaired_master = repaired_master[master.columns]

assert len(repaired_master) == 130
assert repaired_master["tmdb_id"].notna().all()
assert repaired_master["tmdb_id"].nunique() == 130

# Restore Batch 09 allocation
repaired_allocation = allocation.drop(
    columns=["tmdb_id"],
    errors="ignore"
).merge(
    mapping[["movie_record_id", "tmdb_id"]],
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

assert len(repaired_allocation) == 10
assert repaired_allocation["tmdb_id"].notna().all()
assert repaired_allocation["tmdb_id"].nunique() == 10

# Save separately; never overwrite original files
master_output = os.path.join(
    os.path.dirname(found["ICMB_130_MASTER_DATA.csv"]),
    "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

allocation_output = os.path.join(
    os.path.dirname(
        found["ICMB_130_YOUTUBE_BATCH_09_MOVIE_ALLOCATION.csv"]
    ),
    "ICMB_130_YOUTUBE_BATCH_09_MOVIE_ALLOCATION_TMDB_REPAIRED.csv"
)

for path in [master_output, allocation_output]:
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists: {path}"
        )

repaired_master.to_csv(master_output, index=False)
repaired_allocation.to_csv(allocation_output, index=False)

print("\n" + "=" * 70)
print("TMDB RESTORATION SUCCESSFUL")
print("=" * 70)

print("Master TMDB IDs:", repaired_master["tmdb_id"].notna().sum())
print("Batch 09 TMDB IDs:", repaired_allocation["tmdb_id"].notna().sum())

display(
    repaired_allocation[
        ["movie_record_id", "movie_title", "tmdb_id"]
    ]
)

print("\nSaved:")
print(master_output)
print(allocation_output)

print("\nOriginal files remain unchanged.")

In [ ]:
import os
import pandas as pd
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from IPython.display import display

# ============================================================
# 1. PATHS AND CONFIGURATION
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

AUDIENCE_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

ALLOCATION_FILE = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment",
    "ICMB_130_YOUTUBE_BATCH_09_MOVIE_ALLOCATION_TMDB_REPAIRED.csv"
)

DISCOVERY_FILE = os.path.join(
    AUDIENCE_DIR, "ICMB_130_YOUTUBE_DISCOVERY_BATCH_09.csv"
)

DISCOVERY_LOG_FILE = os.path.join(
    AUDIENCE_DIR, "ICMB_130_YOUTUBE_DISCOVERY_LOG_BATCH_09.csv"
)

METADATA_FILE = os.path.join(
    AUDIENCE_DIR, "ICMB_130_YOUTUBE_METADATA_BATCH_09.csv"
)

os.makedirs(AUDIENCE_DIR, exist_ok=True)

for path in [DISCOVERY_FILE, DISCOVERY_LOG_FILE, METADATA_FILE]:
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists. Nothing overwritten:\n{path}"
        )

# ============================================================
# 2. LOAD APPROVED ALLOCATION AND API KEY
# ============================================================

allocation = pd.read_csv(ALLOCATION_FILE)

assert len(allocation) == 10
assert allocation["movie_record_id"].is_unique
assert allocation["tmdb_id"].notna().all()

API_KEY = userdata.get("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError(
        "YOUTUBE_API_KEY not found in Colab Secrets."
    )

youtube = build(
    "youtube", "v3",
    developerKey=API_KEY,
    cache_discovery=False
)

# ============================================================
# 3. DISCOVERY — ONE SEARCH PER MOVIE
# ============================================================

discovery_rows = []
discovery_logs = []

print("=" * 70)
print("BATCH 09 — YOUTUBE DISCOVERY")
print("=" * 70)

for _, movie in allocation.iterrows():

    movie_id = movie["movie_record_id"]
    title = movie["movie_title"]
    year = int(movie["year"])
    language = movie["language"]

    query = f'"{title}" {year} movie review reaction'

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=10,
            order="relevance",
            regionCode="IN"
        ).execute()

        items = response.get("items", [])

        for item in items:
            snippet = item.get("snippet", {})

            discovery_rows.append({
                "movie_record_id": movie_id,
                "movie_title": title,
                "movie_year": year,
                "language": language,
                "tmdb_id": movie["tmdb_id"],
                "search_query": query,
                "video_id": item["id"]["videoId"],
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "discovery_source": "YouTube Data API v3",
                "discovery_batch": "Batch 09"
            })

        discovery_logs.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "year": year,
            "query": query,
            "request_status": "Success",
            "candidates_returned": len(items),
            "error_type": None,
            "retrieved_at_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print(f"{title}: {len(items)} candidates")

    except Exception as e:
        # Sanitize errors; never print API request URLs or keys.
        discovery_logs.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "year": year,
            "query": query,
            "request_status": "Failed",
            "candidates_returned": 0,
            "error_type": type(e).__name__,
            "retrieved_at_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print(f"{title}: request failed ({type(e).__name__})")

# ============================================================
# 4. SAVE DISCOVERY FILES
# ============================================================

discovery = pd.DataFrame(discovery_rows)
discovery_log = pd.DataFrame(discovery_logs)

discovery.to_csv(DISCOVERY_FILE, index=False)
discovery_log.to_csv(DISCOVERY_LOG_FILE, index=False)

print("\nDiscovery candidates:", len(discovery))
print("Unique video IDs:", discovery["video_id"].nunique())
print(
    "Successful movie searches:",
    (discovery_log["request_status"] == "Success").sum()
)
print(
    "Failed movie searches:",
    (discovery_log["request_status"] == "Failed").sum()
)

# ============================================================
# 5. METADATA RETRIEVAL — CHUNKS OF 50
# ============================================================

metadata_rows = []

video_ids = (
    discovery["video_id"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

print("\n" + "=" * 70)
print("BATCH 09 — VIDEO METADATA RETRIEVAL")
print("=" * 70)

for start in range(0, len(video_ids), 50):

    chunk = video_ids[start:start + 50]

    try:
        response = youtube.videos().list(
            part="snippet,contentDetails,statistics,status",
            id=",".join(chunk),
            maxResults=50
        ).execute()

        for item in response.get("items", []):

            snippet = item.get("snippet", {})
            statistics = item.get("statistics", {})
            status = item.get("status", {})
            content = item.get("contentDetails", {})

            metadata_rows.append({
                "video_id": item.get("id"),
                "metadata_title": snippet.get("title"),
                "metadata_channel_id": snippet.get("channelId"),
                "metadata_channel_title": snippet.get("channelTitle"),
                "metadata_description": snippet.get("description"),
                "metadata_published_at": snippet.get("publishedAt"),
                "duration": content.get("duration"),
                "view_count": statistics.get("viewCount"),
                "like_count": statistics.get("likeCount"),
                "comment_count": statistics.get("commentCount"),
                "privacy_status": status.get("privacyStatus"),
                "upload_status": status.get("uploadStatus"),
                "metadata_status": "Retrieved"
            })

        print(
            f"Metadata chunk {start // 50 + 1}: "
            f"{len(response.get('items', []))} records"
        )

    except Exception as e:
        print(
            f"Metadata chunk {start // 50 + 1} failed: "
            f"{type(e).__name__}"
        )

metadata = pd.DataFrame(metadata_rows)

if not metadata.empty:
    metadata = metadata.drop_duplicates(
        subset=["video_id"]
    )

# Save metadata separately
metadata.to_csv(METADATA_FILE, index=False)

# ============================================================
# 6. MERGE AND VALIDATE
# ============================================================

combined = discovery.merge(
    metadata,
    on="video_id",
    how="left",
    validate="many_to_one"
)

print("\n" + "=" * 70)
print("BATCH 09 DISCOVERY + METADATA SUMMARY")
print("=" * 70)

print("Discovery records:", len(discovery))
print("Unique discovered videos:", discovery["video_id"].nunique())
print("Metadata records:", len(metadata))
print("Candidates with metadata:", combined["metadata_title"].notna().sum())
print("Candidates missing metadata:", combined["metadata_title"].isna().sum())

print("\nCandidates by movie:")
display(
    discovery.groupby(
        ["movie_record_id", "movie_title"]
    ).agg(
        candidate_records=("video_id", "size"),
        unique_video_ids=("video_id", "nunique")
    ).reset_index()
)

print("\nDiscovery request log:")
display(discovery_log)

print("\nSaved files:")
print(DISCOVERY_FILE)
print(DISCOVERY_LOG_FILE)
print(METADATA_FILE)

print("\nDiscovery and metadata retrieval completed.")
print("No existing files overwritten.")

In [ ]:
import os
import re
import pandas as pd
from IPython.display import display

# ============================================================
# 1. PATHS AND LOAD DATA
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

YOUTUBE_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

DISCOVERY_FILE = os.path.join(
    YOUTUBE_DIR, "ICMB_130_YOUTUBE_DISCOVERY_BATCH_09.csv"
)

METADATA_FILE = os.path.join(
    YOUTUBE_DIR, "ICMB_130_YOUTUBE_METADATA_BATCH_09.csv"
)

SCREENING_FILE = os.path.join(
    YOUTUBE_DIR, "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_09.csv"
)

REVIEW_FILE = os.path.join(
    YOUTUBE_DIR, "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_09.csv"
)

for path in [SCREENING_FILE, REVIEW_FILE]:
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists; no file overwritten:\n{path}"
        )

discovery = pd.read_csv(DISCOVERY_FILE)
metadata = pd.read_csv(METADATA_FILE)

assert discovery["video_id"].nunique() == len(discovery)
assert metadata["video_id"].nunique() == len(metadata)

df = discovery.merge(
    metadata,
    on="video_id",
    how="left",
    validate="one_to_one"
)

assert len(df) == 100

# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def parse_duration(value):
    """Convert ISO 8601 YouTube duration to seconds."""
    if pd.isna(value):
        return None

    match = re.fullmatch(
        r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?",
        str(value)
    )

    if not match:
        return None

    hours = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)

    return hours * 3600 + minutes * 60 + seconds


def classify_candidate(row):

    title = str(row.get("metadata_title", "")).lower()
    description = str(
        row.get("metadata_description", "")
    ).lower()

    combined = title + " " + description
    duration = parse_duration(row.get("duration"))

    # Strong title indicators
    full_movie = bool(re.search(
        r"\b(full movie|full film|complete movie|entire movie|"
        r"watch full|full hindi movie|full tamil movie|"
        r"full telugu movie)\b",
        title
    ))

    trailer = bool(re.search(
        r"\b(trailer|teaser|official teaser|official trailer|"
        r"announcement video)\b",
        title
    ))

    reaction = bool(re.search(
        r"\b(reaction|reacts|reacting|first time watching|"
        r"watching for the first time|movie watchalong)\b",
        title
    ))

    review = bool(re.search(
        r"\b(review|reviewed|analysis|explained|"
        r"ending explained|movie discussion|"
        r"film discussion|rewatch|retrospective|"
        r"revisited|rewind|breakdown)\b",
        title
    ))

    trailer_reaction = trailer and reaction

    song_or_music = bool(re.search(
        r"\b(song|jukebox|lyrical|lyric video|"
        r"video song|music video|ost|soundtrack)\b",
        title
    ))

    short_or_clip = bool(re.search(
        r"\b(shorts|short film|clip|scene|"
        r"best scenes|fight scene|comedy scene)\b",
        title
    ))

    # Classify source type
    if full_movie:
        source_type = "Full Movie"
    elif trailer_reaction:
        source_type = "Trailer Reaction"
    elif trailer:
        source_type = "Trailer"
    elif song_or_music:
        source_type = "Song / Music"
    elif short_or_clip:
        source_type = "Clip / Short"
    elif reaction:
        source_type = "Movie Reaction"
    elif review:
        source_type = "Movie Review / Analysis"
    elif duration is not None and duration < 180:
        source_type = "Short / Unclear"
    else:
        source_type = "Unclassified"

    # Preliminary screening decision
    if full_movie or trailer or song_or_music or short_or_clip:
        decision = "Exclude"

    elif reaction or review:
        decision = "Shortlist"

    else:
        decision = "Hold"

    # Add duration warning
    if duration is None:
        duration_note = "Duration unavailable"
    elif duration < 180:
        duration_note = "Very short video"
    elif duration < 600:
        duration_note = "Short-form discussion; inspect context"
    else:
        duration_note = "Longer-form video"

    # Notes
    notes = []

    if decision == "Shortlist":
        notes.append(
            "Title contains review/reaction indicator; "
            "metadata-based selection only"
        )

    if decision == "Hold":
        notes.append(
            "No clear review/reaction indicator in title"
        )

    if decision == "Exclude":
        notes.append(
            "Title indicates a non-primary discussion source"
        )

    if trailer_reaction:
        notes.append(
            "Trailer reaction is not equivalent to a full-film reaction"
        )

    if duration is not None and duration < 600:
        notes.append("Check whether substantive film discussion is present")

    return pd.Series({
        "source_type": source_type,
        "screening_decision": decision,
        "duration_seconds": duration,
        "duration_note": duration_note,
        "screening_notes": "; ".join(notes),
        "metadata_review_status": "Metadata Reviewed",
        "content_review_status": "Not Reviewed"
    })


# ============================================================
# 3. AUTOMATED SCREENING
# ============================================================

screening = pd.concat(
    [df, df.apply(classify_candidate, axis=1)],
    axis=1
)

# Keep all original records; add screening results
screening.to_csv(SCREENING_FILE, index=False)

# ============================================================
# 4. PREPARE MANUAL REVIEW SHEET
# ============================================================

review_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "metadata_title",
    "metadata_channel_title",
    "duration",
    "duration_seconds",
    "view_count",
    "comment_count",
    "source_type",
    "screening_decision",
    "metadata_description",
    "metadata_review_status",
    "content_review_status",
    "screening_notes"
]

review = screening[review_columns].copy()

review["final_decision"] = ""
review["final_source_type"] = ""
review["manual_review_notes"] = ""

review.to_csv(REVIEW_FILE, index=False)

# ============================================================
# 5. SUMMARY
# ============================================================

print("=" * 70)
print("BATCH 09 — AUTOMATED SCREENING RESULTS")
print("=" * 70)

print("Total candidates:", len(screening))
print("Unique video IDs:", screening["video_id"].nunique())

print("\nScreening decision distribution:")
display(
    screening["screening_decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="candidate_count")
)

print("\nSource type distribution:")
display(
    screening["source_type"]
    .value_counts()
    .rename_axis("source_type")
    .reset_index(name="candidate_count")
)

print("\nMovie-wise screening:")
movie_summary = screening.pivot_table(
    index=["movie_record_id", "movie_title"],
    columns="screening_decision",
    values="video_id",
    aggfunc="count",
    fill_value=0
).reset_index()

display(movie_summary)

# Show shortlisted candidates first, grouped by movie
priority = {
    "Shortlist": 1,
    "Hold": 2,
    "Exclude": 3
}

screening["_priority"] = screening[
    "screening_decision"
].map(priority)

shortlist_view = screening.sort_values(
    ["movie_title", "_priority", "view_count"],
    ascending=[True, True, False]
)

print("\nCandidate recommendations — first 5 per movie:")
display(
    shortlist_view.groupby(
        ["movie_record_id", "movie_title"],
        group_keys=False
    ).head(5)[[
        "movie_title",
        "video_id",
        "metadata_title",
        "metadata_channel_title",
        "duration",
        "source_type",
        "screening_decision"
    ]]
)

print("\nSaved screening:", SCREENING_FILE)
print("Saved review sheet:", REVIEW_FILE)

print("\nContent verification remains pending.")

In [ ]:
import os
import pandas as pd
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from IPython.display import display

# ============================================================
# 1. PATHS
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

COMMENTS_DIR = os.path.join(RAW_DIR, "comments")

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

os.makedirs(COMMENTS_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

SCREENING_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_09.csv"
)

MANIFEST_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_09.csv"
)

RAW_FILE = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_BATCH_09.csv"
)

LOG_FILE = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_09.csv"
)

CHECKPOINT_FILE = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_BATCH_09_CHECKPOINT.csv"
)

# Protect existing outputs
for path in [MANIFEST_FILE, RAW_FILE, LOG_FILE, CHECKPOINT_FILE]:
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists. No files overwritten:\n{path}"
        )

# ============================================================
# 2. APPROVED SOURCE LIST
# ============================================================

approved_sources = [
    ("2016_2", "zEAnNY6pRXA", "Movie Reaction"),
    ("2016_2", "tOEYsdOpM1c", "Movie Reaction"),
    ("2017_2", "DLSUS1ot76I", "Movie Reaction"),
    ("2017_2", "Mh9YhdzMVNQ", "Movie Review"),
    ("2023_2", "1_oH06Z5FRw", "Movie Review / Analysis"),
    ("2023_2", "PY3HIYpPQ4Q", "Movie Reaction"),
    ("2023_3", "gczxO19Td8E", "Movie Review"),
    ("2023_3", "h3dImehRxwA", "Movie Review"),
    ("2024_3", "7f3DiaLQRL0", "Movie Review"),
    ("2024_3", "xAI9qcU5Jnk", "Movie Reaction"),
    ("2025_3", "US4Y8GG5MN8", "Movie Review"),
    ("2025_3", "b6Nm0VXPPAA", "Movie Reaction"),
    ("2018_1", "Zm3Ag425Aj0", "Movie Review"),
    ("2018_1", "CqpsQ4XSE5o", "Movie Reaction"),
    ("2022_5", "Fsr6CTWHyog", "Movie Review"),
    ("2022_5", "ECEeI4F9A0c", "Movie Reaction"),
    ("2019_2", "PJimSQqORTg", "Movie Review"),
    ("2019_2", "HQ0dQ0x66sI", "Movie Review"),
    ("2024_4", "CH2wR4uUXns", "Movie Review"),
    ("2024_4", "QDkELBBmbUs", "Movie Review")
]

# ============================================================
# 3. LOAD DISCOVERY + METADATA AND BUILD MANIFEST
# ============================================================

screening = pd.read_csv(SCREENING_FILE)

selected_ids = [x[1] for x in approved_sources]

assert len(selected_ids) == 20
assert len(set(selected_ids)) == 20

selected = screening[
    screening["video_id"].astype(str).isin(selected_ids)
].copy()

assert len(selected) == 20
assert selected["video_id"].nunique() == 20

source_type_map = {
    video_id: source_type
    for _, video_id, source_type in approved_sources
}

selected["manual_source_type"] = (
    selected["video_id"].map(source_type_map)
)

selected["manual_verification_status"] = "Metadata Reviewed"
selected["content_review_status"] = "Not Reviewed"
selected["target_comments"] = 50
selected["acquisition_status"] = "Pending"

selected["manual_review_notes"] = (
    "User-approved metadata-based selection; "
    "content not independently reviewed. "
    "YouTube top-level comments ordered by time."
)

manifest_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "metadata_title",
    "metadata_channel_title",
    "duration",
    "view_count",
    "comment_count",
    "manual_source_type",
    "manual_verification_status",
    "manual_review_notes",
    "content_review_status",
    "acquisition_status",
    "target_comments"
]

manifest = selected[manifest_columns].copy()

manifest = manifest.rename(columns={
    "metadata_title": "video_title",
    "metadata_channel_title": "channel_title"
})

manifest.to_csv(MANIFEST_FILE, index=False)

print("Approved manifest saved:", len(manifest), "sources")
print("Movies:", manifest["movie_record_id"].nunique())
print("Planned comments:", manifest["target_comments"].sum())

# ============================================================
# 4. CONNECT TO YOUTUBE API
# ============================================================

API_KEY = userdata.get("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY missing from Colab Secrets.")

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY,
    cache_discovery=False
)

# ============================================================
# 5. COMMENT ACQUISITION
# ============================================================

all_comments = []
logs = []

def save_checkpoint():
    pd.DataFrame(all_comments).to_csv(
        CHECKPOINT_FILE, index=False
    )

print("\n" + "=" * 70)
print("BATCH 09 — COMMENT ACQUISITION")
print("=" * 70)

for index, source in manifest.iterrows():

    video_id = str(source["video_id"])
    target = int(source["target_comments"])
    collected = 0
    pages = 0
    next_page_token = None
    status = "Pending"
    error_type = None

    try:
        while collected < target:

            request_size = min(100, target - collected)

            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=request_size,
                order="time",
                textFormat="plainText",
                pageToken=next_page_token
            ).execute()

            pages += 1
            items = response.get("items", [])

            if not items:
                break

            for item in items:
                top_comment = item["snippet"]["topLevelComment"]
                snippet = top_comment["snippet"]

                all_comments.append({
                    "movie_record_id": source["movie_record_id"],
                    "movie_title": source["movie_title"],
                    "movie_year": source["movie_year"],
                    "language": source["language"],
                    "video_id": video_id,
                    "video_title": source["video_title"],
                    "channel_title": source["channel_title"],
                    "comment_id": top_comment["id"],
                    "comment_text": snippet.get("textDisplay"),
                    "author_name": snippet.get("authorDisplayName"),
                    "like_count": snippet.get("likeCount", 0),
                    "published_at": snippet.get("publishedAt"),
                    "updated_at": snippet.get("updatedAt"),
                    "comment_position": collected + 1,
                    "acquisition_batch": "Batch 09",
                    "retrieval_order": "time"
                })

                collected += 1

                if collected >= target:
                    break

            next_page_token = response.get("nextPageToken")

            if not next_page_token:
                break

        status = (
            "Completed" if collected == target
            else "Partial" if collected > 0
            else "No Comments Retrieved"
        )

    except Exception as e:
        status = (
            "Partial" if collected > 0
            else "Failed"
        )
        error_type = type(e).__name__

    # Save checkpoint after each video
    save_checkpoint()

    acquired_at = datetime.now(timezone.utc).isoformat()

    logs.append({
        "movie_record_id": source["movie_record_id"],
        "movie_title": source["movie_title"],
        "video_id": video_id,
        "target_comments": target,
        "comments_collected": collected,
        "api_pages": pages,
        "acquisition_status": status,
        "error_type": error_type,
        "acquired_at_utc": acquired_at
    })

    # Update manifest with actual status
    manifest.loc[
        manifest["video_id"] == video_id,
        "acquisition_status"
    ] = status

    # Persist progress after each source
    manifest.to_csv(MANIFEST_FILE, index=False)

    print(
        f"{source['movie_title']}: {collected}/{target} "
        f"| {status}"
    )

# ============================================================
# 6. SAVE FINAL RAW DATA AND LOG
# ============================================================

comments_df = pd.DataFrame(all_comments)
logs_df = pd.DataFrame(logs)

comments_df.to_csv(RAW_FILE, index=False)
logs_df.to_csv(LOG_FILE, index=False)

# ============================================================
# 7. FINAL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("BATCH 09 — ACQUISITION SUMMARY")
print("=" * 70)

print("Total comments:", len(comments_df))
print("Unique comment IDs:", comments_df["comment_id"].nunique())
print("Unique videos:", comments_df["video_id"].nunique())
print("Unique movies:", comments_df["movie_record_id"].nunique())
print("Successful sources:", (logs_df["acquisition_status"] == "Completed").sum())
print("Partial sources:", (logs_df["acquisition_status"] == "Partial").sum())
print("Failed sources:", (logs_df["acquisition_status"] == "Failed").sum())
print("Total API pages:", logs_df["api_pages"].sum())

print("\nAcquisition by movie:")
display(
    logs_df.groupby(
        ["movie_record_id", "movie_title"]
    ).agg(
        videos=("video_id", "nunique"),
        comments=("comments_collected", "sum"),
        status_count=("acquisition_status", "count")
    ).reset_index()
)

print("\nSaved files:")
print("Manifest:", MANIFEST_FILE)
print("Raw comments:", RAW_FILE)
print("Acquisition log:", LOG_FILE)
print("Checkpoint:", CHECKPOINT_FILE)

print("\nRaw comments remain unmodified. No replies were collected.")

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from collections import Counter
from IPython.display import display

# ============================================================
# 1. PATHS AND LOAD
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

COMMENTS_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube", "comments"
)

RAW_FILE = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_BATCH_09.csv"
)

OUTPUTS = {
    "overall": os.path.join(
        COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_09.csv"
    ),
    "movie": os.path.join(
        COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_09.csv"
    ),
    "video": os.path.join(
        COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_VIDEO_QUALITY_BATCH_09.csv"
    ),
    "duplicates": os.path.join(
        COMMENTS_DIR, "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_BATCH_09.csv"
    )
}

for path in OUTPUTS.values():
    if os.path.exists(path):
        raise FileExistsError(f"Output already exists: {path}")

df = pd.read_csv(RAW_FILE)

assert len(df) == 995
assert df["comment_id"].nunique() == 995

# ============================================================
# 2. NORMALIZATION AND SCRIPT PROFILE
# ============================================================

df["comment_text"] = df["comment_text"].fillna("").astype(str)

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["normalized_comment_text"] = df["comment_text"].apply(
    normalize_text
)

df["word_count"] = df["comment_text"].apply(
    lambda x: len(str(x).split())
)

df["character_count"] = df["comment_text"].str.len()

df["is_empty"] = df["comment_text"].str.strip().eq("")
df["is_very_short"] = (
    (~df["is_empty"]) & (df["word_count"] <= 3)
)
df["contains_url"] = df["comment_text"].str.contains(
    r"http\S+|www\.", case=False, regex=True
)

def detect_script(text):
    text = str(text)

    if not text.strip():
        return "Empty"

    scripts = []

    ranges = {
        "Devanagari": r"[\u0900-\u097F]",
        "Tamil": r"[\u0B80-\u0BFF]",
        "Telugu": r"[\u0C00-\u0C7F]",
        "Kannada": r"[\u0C80-\u0CFF]",
        "Bengali": r"[\u0980-\u09FF]",
        "Malayalam": r"[\u0D00-\u0D7F]"
    }

    for name, pattern in ranges.items():
        if re.search(pattern, text):
            scripts.append(name)

    if len(scripts) > 1:
        return "Mixed Indian Scripts"

    if len(scripts) == 1:
        return scripts[0]

    if re.search(r"[A-Za-z]", text):
        return "Latin"

    return "Other/Emoji/Symbols"

df["script_category"] = df["comment_text"].apply(detect_script)

# ============================================================
# 3. QUALITY SUMMARY FUNCTION
# ============================================================

def quality_summary(data, group_columns=None):

    if group_columns is None:
        groups = [(None, data)]
    else:
        groups = data.groupby(
            group_columns, dropna=False
        )

    records = []

    for key, part in groups:

        if group_columns is None:
            key_values = {}
        else:
            if not isinstance(key, tuple):
                key = (key,)
            key_values = dict(zip(group_columns, key))

        nonempty = part.loc[~part["is_empty"]]

        records.append({
            **key_values,
            "comment_records": len(part),
            "unique_comment_ids": part["comment_id"].nunique(),
            "unique_videos": part["video_id"].nunique(),
            "empty_comments": int(part["is_empty"].sum()),
            "very_short_comments_le_3_words": int(
                part["is_very_short"].sum()
            ),
            "comments_with_urls": int(part["contains_url"].sum()),
            "average_word_count": round(
                nonempty["word_count"].mean(), 2
            ) if len(nonempty) else 0,
            "median_word_count": round(
                nonempty["word_count"].median(), 2
            ) if len(nonempty) else 0,
            "average_character_count": round(
                nonempty["character_count"].mean(), 2
            ) if len(nonempty) else 0,
            "median_character_count": round(
                nonempty["character_count"].median(), 2
            ) if len(nonempty) else 0
        })

    return pd.DataFrame(records)

# ============================================================
# 4. OVERALL QUALITY PROFILE
# ============================================================

overall = quality_summary(df)

script_counts = (
    df["script_category"]
    .value_counts()
    .to_dict()
)

length_distribution = {
    "Very Short (1-3 words)": int(
        ((df["word_count"] <= 3) & (~df["is_empty"])).sum()
    ),
    "Short (4-10 words)": int(
        df["word_count"].between(4, 10).sum()
    ),
    "Medium (11-30 words)": int(
        df["word_count"].between(11, 30).sum()
    ),
    "Long (>30 words)": int(
        (df["word_count"] > 30).sum()
    )
}

overall["sampling_method"] = (
    "YouTube top-level comments ordered by time"
)
overall["replies_included"] = False
overall["script_distribution"] = str(script_counts)
overall["length_distribution"] = str(length_distribution)

# ============================================================
# 5. MOVIE-WISE AND VIDEO-WISE QUALITY
# ============================================================

movie_quality = quality_summary(
    df, ["movie_record_id", "movie_title", "movie_year"]
)

video_quality = quality_summary(
    df, [
        "movie_record_id",
        "movie_title",
        "video_id",
        "video_title",
        "channel_title"
    ]
)

# ============================================================
# 6. REPEATED TEXT AUDIT
# ============================================================

duplicate_base = df[
    (df["normalized_comment_text"] != "")
].copy()

duplicate_groups = (
    duplicate_base.groupby("normalized_comment_text")
    .agg(
        repeated_records=("comment_id", "size"),
        unique_videos=("video_id", "nunique"),
        unique_movies=("movie_record_id", "nunique"),
        comment_ids=("comment_id", lambda x: " | ".join(x.astype(str))),
        video_ids=("video_id", lambda x: " | ".join(sorted(set(x.astype(str))))),
        movie_titles=("movie_title", lambda x: " | ".join(sorted(set(x.astype(str))))),
        example_comment=("comment_text", "first")
    )
    .reset_index()
)

duplicate_audit = duplicate_groups[
    duplicate_groups["repeated_records"] > 1
].copy()

if not duplicate_audit.empty:
    duplicate_audit["duplicate_scope"] = np.where(
        duplicate_audit["unique_videos"] > 1,
        "Across Videos",
        "Within Video"
    )

    duplicate_audit = duplicate_audit.sort_values(
        ["repeated_records", "unique_videos"],
        ascending=False
    )

# ============================================================
# 7. SAVE REPORTS
# ============================================================

overall.to_csv(OUTPUTS["overall"], index=False)
movie_quality.to_csv(OUTPUTS["movie"], index=False)
video_quality.to_csv(OUTPUTS["video"], index=False)
duplicate_audit.to_csv(OUTPUTS["duplicates"], index=False)

# ============================================================
# 8. VALIDATION AND SUMMARY
# ============================================================

print("=" * 70)
print("BATCH 09 — COMMENT QUALITY PROFILE")
print("=" * 70)

print("Total comments:", len(df))
print("Unique comment IDs:", df["comment_id"].nunique())
print("Unique videos:", df["video_id"].nunique())
print("Unique movies:", df["movie_record_id"].nunique())
print("Empty comments:", int(df["is_empty"].sum()))
print("Very short comments:", int(df["is_very_short"].sum()))
print("Comments containing URLs:", int(df["contains_url"].sum()))

print("\nOverall quality:")
display(overall)

print("\nMovie-wise quality:")
display(movie_quality)

print("\nVideo-wise quality:")
display(video_quality)

print("\nScript distribution:")
display(
    df["script_category"]
    .value_counts()
    .rename_axis("script_category")
    .reset_index(name="comment_count")
)

print("\nRepeated-text audit:")
print("Repeated-text groups:", len(duplicate_audit))
print(
    "Records in repeated groups:",
    int(duplicate_audit["repeated_records"].sum())
    if len(duplicate_audit) else 0
)

if len(duplicate_audit):
    display(duplicate_audit.head(15))

print("\nSaved reports:")
for name, path in OUTPUTS.items():
    print(name, "->", path)

print("\nRaw comments were not modified or deleted.")
print("Script classification is not language identification.")

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# 1. PATHS
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed"
)

AUDIENCE_DIR = os.path.join(
    PROCESSED_DIR, "audience_sentiment"
)

COMMENTS_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment",
    "youtube", "comments"
)

OLD_CONSOLIDATED = os.path.join(
    AUDIENCE_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_08.csv"
)

MASTER_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

BATCH09_RAW = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_BATCH_09.csv"
)

BATCH09_MANIFEST = os.path.join(
    AUDIENCE_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_09.csv"
)

BATCH09_LOG = os.path.join(
    COMMENTS_DIR,
    "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_09.csv"
)

OUTPUTS = {
    "consolidated": os.path.join(
        AUDIENCE_DIR,
        "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_09.csv"
    ),
    "coverage": os.path.join(
        AUDIENCE_DIR,
        "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_09.csv"
    ),
    "duplicates": os.path.join(
        AUDIENCE_DIR,
        "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_PILOT_BATCH_01_09.csv"
    )
}

for path in OUTPUTS.values():
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists. No file overwritten:\n{path}"
        )

# ============================================================
# 2. LOAD AND PREFLIGHT
# ============================================================

old = pd.read_csv(OLD_CONSOLIDATED)
master = pd.read_csv(MASTER_FILE)
new = pd.read_csv(BATCH09_RAW)
manifest = pd.read_csv(BATCH09_MANIFEST)
logs = pd.read_csv(BATCH09_LOG)

assert len(master) == 130
assert master["movie_record_id"].is_unique
assert len(new) == 995
assert new["comment_id"].nunique() == 995
assert len(manifest) == 20
assert manifest["video_id"].nunique() == 20

required_old = {
    "movie_record_id", "comment_id", "comment_text",
    "video_id", "source_batch", "source_type"
}
assert required_old.issubset(old.columns)

required_new = {
    "movie_record_id", "movie_title", "movie_year",
    "video_id", "video_title", "channel_title",
    "comment_id", "comment_text", "author_name",
    "like_count", "published_at", "updated_at",
    "comment_position", "retrieval_order"
}
assert required_new.issubset(new.columns)

overlap = set(old["comment_id"].dropna().astype(str)) & set(
    new["comment_id"].dropna().astype(str)
)

if overlap:
    raise ValueError(
        f"Found {len(overlap)} overlapping comment IDs. "
        "Review before consolidation."
    )

print("Preflight passed.")
print("Old consolidated:", old.shape)
print("Batch 09 raw:", new.shape)
print("Master:", master.shape)
print("Overlapping comment IDs:", len(overlap))

# ============================================================
# 3. PREPARE BATCH 09 SCHEMA
# ============================================================

source_map = manifest.set_index(
    "video_id"
)["manual_source_type"].to_dict()

timestamp_map = logs.set_index(
    "video_id"
)["acquired_at_utc"].to_dict()

new["source_type"] = new["video_id"].map(source_map)
new["retrieved_at"] = new["video_id"].map(timestamp_map)

new["normalized_comment_text"] = (
    new["comment_text"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"http\S+|www\.\S+", " ", regex=True)
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

new["source_batch"] = "Batch 09"
new["batch_id"] = "Batch 09"

new["sampling_order"] = new["comment_position"]
new["acquisition_order"] = new["comment_position"]

new["sampling_unit"] = "top_level_comment"
new["comment_source"] = "YouTube Top-Level Comment"

new["author_display_name"] = new["author_name"]
new["comment_author"] = new["author_name"]

new["acquired_at_utc"] = new["retrieved_at"]
new["acquisition_timestamp_utc"] = new["retrieved_at"]

new["author_channel_id"] = np.nan
new["comment_reply_count"] = np.nan
new["reply_count"] = np.nan
new["total_reply_count"] = np.nan

def extract_series_part(title):
    match = re.search(
        r"\bPart\s*(\d+)\s*/\s*(\d+)\b",
        str(title),
        flags=re.IGNORECASE
    )
    return (
        f"Part {match.group(1)}/{match.group(2)}"
        if match else None
    )

new["series_part"] = new["video_title"].apply(
    extract_series_part
)

# Standardize acquisition fields
new["retrieval_order"] = "time"

consolidated_columns = list(old.columns)

for col in consolidated_columns:
    if col not in new.columns:
        new[col] = np.nan

new = new[consolidated_columns]

# ============================================================
# 4. CONSOLIDATE WITHOUT MODIFYING OLD DATA
# ============================================================

consolidated = pd.concat(
    [old, new],
    ignore_index=True
)

assert consolidated["comment_id"].nunique() == len(consolidated)
assert consolidated["movie_record_id"].notna().all()

consolidated.to_csv(
    OUTPUTS["consolidated"],
    index=False
)

# ============================================================
# 5. GENERATE 130-MOVIE COVERAGE REPORT
# ============================================================

coverage_counts = (
    consolidated.groupby("movie_record_id")
    .agg(
        comment_records=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        acquisition_batches=("source_batch", "nunique")
    )
    .reset_index()
)

coverage = master[
    ["movie_record_id", "movie_title", "year", "rank", "language"]
].merge(
    coverage_counts,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

count_columns = [
    "comment_records",
    "unique_comment_ids",
    "unique_videos",
    "acquisition_batches"
]

coverage[count_columns] = (
    coverage[count_columns].fillna(0).astype(int)
)

coverage["coverage_status"] = np.where(
    coverage["comment_records"] > 0,
    "Covered",
    "Not Covered"
)

coverage["coverage_category"] = np.select(
    [
        coverage["comment_records"] >= 100,
        coverage["comment_records"].between(50, 99),
        coverage["comment_records"].between(1, 49)
    ],
    [
        "Higher (100+)",
        "Moderate (50-99)",
        "Low (1-49)"
    ],
    default="No Comments"
)

coverage = coverage[
    [
        "movie_record_id", "movie_title", "year", "rank",
        "language", "comment_records", "unique_comment_ids",
        "unique_videos", "acquisition_batches",
        "coverage_status", "coverage_category"
    ]
]

assert len(coverage) == 130
assert coverage["movie_record_id"].is_unique

coverage.to_csv(
    OUTPUTS["coverage"],
    index=False
)

# ============================================================
# 6. REPEATED-TEXT AUDIT ON FULL CONSOLIDATED DATA
# ============================================================

duplicate_base = consolidated.copy()

duplicate_base["normalized_comment_text"] = (
    duplicate_base["comment_text"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"http\S+|www\.\S+", " ", regex=True)
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

duplicate_base = duplicate_base[
    duplicate_base["normalized_comment_text"] != ""
]

duplicate_audit = (
    duplicate_base.groupby("normalized_comment_text")
    .agg(
        repeated_records=("comment_id", "size"),
        unique_videos=("video_id", "nunique"),
        unique_movies=("movie_record_id", "nunique"),
        comment_ids=("comment_id", lambda x: " | ".join(x.astype(str))),
        video_ids=("video_id", lambda x: " | ".join(sorted(set(x.astype(str))))),
        movie_titles=("movie_title", lambda x: " | ".join(sorted(set(x.astype(str))))),
        source_batches=("source_batch", lambda x: " | ".join(sorted(set(x.astype(str))))),
        example_comment=("comment_text", "first")
    )
    .reset_index()
)

duplicate_audit = duplicate_audit[
    duplicate_audit["repeated_records"] > 1
].copy()

if not duplicate_audit.empty:
    duplicate_audit["duplicate_scope"] = np.where(
        duplicate_audit["unique_videos"] > 1,
        "Across Videos",
        "Within Video"
    )

    duplicate_audit = duplicate_audit.sort_values(
        ["repeated_records", "unique_videos"],
        ascending=False
    )

duplicate_audit.to_csv(
    OUTPUTS["duplicates"],
    index=False
)

# ============================================================
# 7. FINAL VALIDATION AND SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PILOT + BATCH 01–09 CONSOLIDATION COMPLETE")
print("=" * 70)

print("Consolidated rows:", len(consolidated))
print("Unique comment IDs:", consolidated["comment_id"].nunique())
print("Unique videos:", consolidated["video_id"].nunique())
print("Unique movies with comments:", consolidated["movie_record_id"].nunique())

print("\nCoverage summary:")
display(
    coverage["coverage_category"]
    .value_counts()
    .rename_axis("coverage_category")
    .reset_index(name="movie_count")
)

print("Covered movies:", (coverage["comment_records"] > 0).sum())
print("Uncovered movies:", (coverage["comment_records"] == 0).sum())
print("Total comments:", coverage["comment_records"].sum())

print("\nBatch 09 coverage contribution:")
display(
    coverage[
        coverage["movie_record_id"].isin(
            manifest["movie_record_id"].unique()
        )
    ][[
        "movie_record_id", "movie_title",
        "comment_records", "unique_videos",
        "coverage_category"
    ]]
)

print("\nRepeated-text audit:")
print("Repeated-text groups:", len(duplicate_audit))
print(
    "Records in repeated groups:",
    int(duplicate_audit["repeated_records"].sum())
    if len(duplicate_audit) else 0
)

print("\nSaved outputs:")
for name, path in OUTPUTS.items():
    print(name, "->", path)

print("\nPrevious consolidated data and raw files remain unchanged.")

In [ ]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# 1. PATHS
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(ROOT, "01_data", "processed")
AUDIENCE_DIR = os.path.join(PROCESSED_DIR, "audience_sentiment")

MASTER_FILE = os.path.join(
    PROCESSED_DIR, "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

COVERAGE_FILE = os.path.join(
    AUDIENCE_DIR,
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_09.csv"
)

CONSOLIDATED_FILE = os.path.join(
    AUDIENCE_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_09.csv"
)

OUTPUTS = {
    "report": os.path.join(
        AUDIENCE_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_09.csv"
    ),
    "summary": os.path.join(
        AUDIENCE_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_09.csv"
    ),
    "language": os.path.join(
        AUDIENCE_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_09.csv"
    )
}

for path in OUTPUTS.values():
    if os.path.exists(path):
        raise FileExistsError(f"Output already exists: {path}")

# ============================================================
# 2. LOAD AND VALIDATE
# ============================================================

master = pd.read_csv(MASTER_FILE)
coverage = pd.read_csv(COVERAGE_FILE)
consolidated = pd.read_csv(CONSOLIDATED_FILE)

assert len(master) == 130
assert len(coverage) == 130
assert master["movie_record_id"].is_unique
assert coverage["movie_record_id"].is_unique
assert consolidated["comment_id"].nunique() == len(consolidated)

print("Preflight passed.")
print("Master:", master.shape)
print("Coverage:", coverage.shape)
print("Consolidated comments:", consolidated.shape)

# ============================================================
# 3. MOVIE-LEVEL ACQUISITION PLANNING
# ============================================================

planning = coverage.copy()

TARGET_PER_MOVIE = 100

planning["acquisition_priority"] = np.select(
    [
        planning["coverage_category"] == "No Comments",
        planning["coverage_category"] == "Low (1-49)",
        planning["coverage_category"] == "Moderate (50-99)"
    ],
    [
        "High — No Comments",
        "High — Low Coverage",
        "Medium — Moderate Coverage"
    ],
    default="Covered — Monitor"
)

planning["additional_comment_target"] = (
    TARGET_PER_MOVIE - planning["comment_records"]
).clip(lower=0).astype(int)

planning = planning[
    [
        "movie_record_id",
        "movie_title",
        "year",
        "rank",
        "language",
        "comment_records",
        "unique_comment_ids",
        "unique_videos",
        "acquisition_batches",
        "coverage_status",
        "coverage_category",
        "acquisition_priority",
        "additional_comment_target"
    ]
]

# ============================================================
# 4. COVERAGE SUMMARY
# ============================================================

summary = (
    planning.groupby("coverage_category")
    .agg(
        movie_count=("movie_record_id", "count"),
        existing_comments=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reset_index()
)

category_order = [
    "No Comments",
    "Low (1-49)",
    "Moderate (50-99)",
    "Higher (100+)"
]

summary["coverage_category"] = pd.Categorical(
    summary["coverage_category"],
    categories=category_order,
    ordered=True
)

summary = summary.sort_values(
    "coverage_category"
).reset_index(drop=True)

summary["coverage_category"] = (
    summary["coverage_category"].astype(str)
)

# ============================================================
# 5. LANGUAGE-WISE SUMMARY
# ============================================================

language_summary = (
    planning.groupby("language")
    .agg(
        total_movies=("movie_record_id", "count"),
        covered_movies=(
            "coverage_status",
            lambda x: (x == "Covered").sum()
        ),
        uncovered_movies=(
            "coverage_status",
            lambda x: (x == "Not Covered").sum()
        ),
        comment_records=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reset_index()
    .sort_values("language")
)

# ============================================================
# 6. VALIDATION
# ============================================================

assert len(planning) == 130
assert planning["movie_record_id"].is_unique

assert planning["comment_records"].sum() == len(consolidated)
assert planning["additional_comment_target"].sum() == 5883

assert language_summary["total_movies"].sum() == 130
assert language_summary["additional_target"].sum() == 5883

# ============================================================
# 7. SAVE REPORTS
# ============================================================

planning.to_csv(OUTPUTS["report"], index=False)
summary.to_csv(OUTPUTS["summary"], index=False)
language_summary.to_csv(OUTPUTS["language"], index=False)

# ============================================================
# 8. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("PILOT + BATCH 01–09 — ACQUISITION PLANNING SUMMARY")
print("=" * 70)

print("Total movies:", len(planning))
print("Total comments:", planning["comment_records"].sum())
print("Covered movies:", (planning["coverage_status"] == "Covered").sum())
print("Uncovered movies:", (planning["coverage_status"] == "Not Covered").sum())
print("Additional provisional target:",
      planning["additional_comment_target"].sum())

print("\nCoverage summary:")
display(summary)

print("\nLanguage-wise summary:")
display(language_summary)

print("\nAcquisition priority:")
display(
    planning["acquisition_priority"]
    .value_counts()
    .rename_axis("priority")
    .reset_index(name="movie_count")
)

print("\nSaved reports:")
for name, path in OUTPUTS.items():
    print(name, "->", path)

print("\nPrevious reports and source datasets remain unchanged.")

In [ ]:
import os
import pandas as pd
from IPython.display import display

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

REPORT_FILE = os.path.join(
    ROOT,
    "01_data",
    "processed",
    "audience_sentiment",
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_09.csv"
)

planning = pd.read_csv(REPORT_FILE)

# Movies with no comments
uncovered = planning[
    planning["coverage_category"] == "No Comments"
].copy()

# Movies with partial coverage
partial = planning[
    planning["coverage_category"].isin(
        ["Low (1-49)", "Moderate (50-99)"]
    )
].copy()

# Sort by language, year and rank
uncovered = uncovered.sort_values(
    ["language", "year", "rank"]
).reset_index(drop=True)

partial = partial.sort_values(
    ["coverage_category", "language", "year", "rank"]
).reset_index(drop=True)

print("=" * 75)
print("BATCH 10 — REMAINING MOVIE ALLOCATION")
print("=" * 75)

print("\n1. MOVIES WITH NO COMMENTS:", len(uncovered))
display(
    uncovered[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "rank",
            "language",
            "additional_comment_target"
        ]
    ]
)

print("\n2. MOVIES WITH PARTIAL COVERAGE:", len(partial))
display(
    partial[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "language",
            "comment_records",
            "unique_videos",
            "coverage_category",
            "additional_comment_target"
        ]
    ]
)

print("\n3. UNCOVERED MOVIES BY LANGUAGE")
display(
    uncovered.groupby("language")
    .agg(
        uncovered_movies=("movie_record_id", "count"),
        provisional_target=("additional_comment_target", "sum")
    )
    .reset_index()
)

print("\nTotal movies requiring further acquisition:",
      len(uncovered) + len(partial))

print("Total provisional additional target:",
      planning["additional_comment_target"].sum())

In [ ]:
import os
import pandas as pd
from IPython.display import display

# ============================================================
# BATCH 10 — MOVIE ALLOCATION
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

AUDIENCE_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

MASTER_FILE = os.path.join(
    ROOT, "01_data", "processed",
    "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

PLANNING_FILE = os.path.join(
    AUDIENCE_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_09.csv"
)

OUTPUT_FILE = os.path.join(
    AUDIENCE_DIR,
    "ICMB_130_YOUTUBE_BATCH_10_MOVIE_ALLOCATION.csv"
)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        f"Batch 10 allocation already exists: {OUTPUT_FILE}"
    )

# Selected movie IDs
BATCH_10_IDS = [
    "2007_4",  # Chak De! India
    "2007_5",  # Partner
    "2008_2",  # Rab Ne Bana Di Jodi
    "2008_4",  # Singh Is Kinng
    "2008_5",  # Jodhaa Akbar
    "2009_3",  # Love Aaj Kal
    "2009_4",  # Ajab Prem Ki Ghazab Kahani
    "2009_5",  # Wanted
    "2008_3",  # Dasavathaaram
    "2020_3"   # Sarileru Neekevvaru
]

master = pd.read_csv(MASTER_FILE)
planning = pd.read_csv(PLANNING_FILE)

assert master["movie_record_id"].is_unique
assert planning["movie_record_id"].is_unique

# Select only currently uncovered movies
selected_plan = planning[
    planning["movie_record_id"].isin(BATCH_10_IDS)
].copy()

assert len(selected_plan) == 10
assert set(selected_plan["movie_record_id"]) == set(BATCH_10_IDS)

assert (selected_plan["coverage_category"] == "No Comments").all()

# Attach TMDB metadata
allocation = selected_plan.merge(
    master[
        [
            "movie_record_id",
            "tmdb_id",
            "original_title",
            "original_language"
        ]
    ],
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

# Validate TMDB IDs
assert allocation["tmdb_id"].notna().all()
assert allocation["tmdb_id"].is_unique

# Add batch identifier
allocation.insert(0, "batch_id", "Batch 10")

allocation = allocation[
    [
        "batch_id",
        "movie_record_id",
        "movie_title",
        "year",
        "rank",
        "language",
        "tmdb_id",
        "original_title",
        "original_language",
        "comment_records",
        "coverage_category",
        "additional_comment_target"
    ]
].sort_values(
    ["language", "year", "rank"]
).reset_index(drop=True)

# Save
allocation.to_csv(OUTPUT_FILE, index=False)

# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("BATCH 10 MOVIE ALLOCATION CREATED")
print("=" * 70)

print("Movies allocated:", len(allocation))
print("Unique TMDB IDs:", allocation["tmdb_id"].nunique())
print("Hindi:", (allocation["language"] == "Hindi").sum())
print("Tamil:", (allocation["language"] == "Tamil").sum())
print("Telugu:", (allocation["language"] == "Telugu").sum())
print(
    "Provisional comment target:",
    allocation["additional_comment_target"].sum()
)

display(allocation)

print("\nSaved:", OUTPUT_FILE)

In [ ]:
import os
import time
import pandas as pd
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from IPython.display import display

# ============================================================
# BATCH 10 — YOUTUBE VIDEO DISCOVERY
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

AUDIENCE_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

RAW_YOUTUBE_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

ALLOCATION_FILE = os.path.join(
    AUDIENCE_DIR,
    "ICMB_130_YOUTUBE_BATCH_10_MOVIE_ALLOCATION.csv"
)

DISCOVERY_FILE = os.path.join(
    RAW_YOUTUBE_DIR,
    "ICMB_130_YOUTUBE_DISCOVERY_BATCH_10.csv"
)

LOG_FILE = os.path.join(
    RAW_YOUTUBE_DIR,
    "ICMB_130_YOUTUBE_DISCOVERY_LOG_BATCH_10.csv"
)

os.makedirs(RAW_YOUTUBE_DIR, exist_ok=True)

# Protect existing outputs
for path in [DISCOVERY_FILE, LOG_FILE]:
    if os.path.exists(path):
        raise FileExistsError(f"Output already exists: {path}")

# ============================================================
# 1. LOAD ALLOCATION AND API SECRET
# ============================================================

allocation = pd.read_csv(ALLOCATION_FILE)

assert len(allocation) == 10
assert allocation["movie_record_id"].is_unique
assert allocation["tmdb_id"].notna().all()

API_KEY = userdata.get("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError(
        "YOUTUBE_API_KEY not found in Colab Secrets. "
        "Add it without printing the key."
    )

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY,
    cache_discovery=False
)

print("Allocation loaded:", len(allocation))
print("YouTube API client initialized.")

# ============================================================
# 2. DISCOVERY CONFIGURATION
# ============================================================

MAX_RESULTS_PER_MOVIE = 10

industry_map = {
    "Hindi": "Bollywood",
    "Tamil": "Kollywood",
    "Telugu": "Tollywood",
    "Kannada": "Sandalwood"
}

discovery_records = []
discovery_logs = []

# ============================================================
# 3. SEARCH EACH MOVIE
# ============================================================

for index, movie in allocation.iterrows():

    movie_id = str(movie["movie_record_id"])
    title = str(movie["movie_title"])
    year = int(movie["year"])
    language = str(movie["language"])

    query = f'"{title}" {year} movie review reaction'

    print(
        f"\n[{index + 1}/10] Searching: {title} ({year})"
    )

    request_time = datetime.now(
        timezone.utc
    ).isoformat()

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=MAX_RESULTS_PER_MOVIE,
            order="relevance",
            regionCode="IN"
        ).execute()

        items = response.get("items", [])

        for item in items:

            snippet = item.get("snippet", {})
            video_id = item.get("id", {}).get("videoId")

            if not video_id:
                continue

            discovery_records.append({
                "movie_record_id": movie_id,
                "movie_title": title,
                "movie_year": year,
                "industry": industry_map.get(
                    language, "Other Indian Cinema"
                ),
                "search_query": query,
                "video_id": video_id,
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "discovery_source": "YouTube Data API v3"
            })

        discovery_logs.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "search_query": query,
            "request_status": "Success",
            "results_returned": len(items),
            "request_timestamp_utc": request_time,
            "error_message": None
        })

        print("Candidates returned:", len(items))

    except Exception as e:

        # Do not print raw API exception or request URL.
        error_message = str(e).splitlines()[0][:250]

        discovery_logs.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "search_query": query,
            "request_status": "Failed",
            "results_returned": 0,
            "request_timestamp_utc": request_time,
            "error_message": error_message
        })

        print("Search failed. Sanitized error:", error_message)

    time.sleep(0.2)

# ============================================================
# 4. CREATE DATAFRAMES
# ============================================================

discovery_df = pd.DataFrame(discovery_records)
log_df = pd.DataFrame(discovery_logs)

if discovery_df.empty:
    raise RuntimeError(
        "No candidate videos were retrieved. "
        "Check the discovery log before proceeding."
    )

# Remove duplicate video IDs while retaining the first match
discovery_df = discovery_df.drop_duplicates(
    subset=["video_id"],
    keep="first"
).reset_index(drop=True)

# ============================================================
# 5. VALIDATE AND SAVE
# ============================================================

assert discovery_df["video_id"].is_unique
assert discovery_df["movie_record_id"].isin(
    allocation["movie_record_id"]
).all()

discovery_df.to_csv(DISCOVERY_FILE, index=False)
log_df.to_csv(LOG_FILE, index=False)

# ============================================================
# 6. DISCOVERY SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("BATCH 10 — YOUTUBE DISCOVERY COMPLETE")
print("=" * 70)

print("Movies searched:", len(log_df))
print("Successful searches:",
      (log_df["request_status"] == "Success").sum())
print("Failed searches:",
      (log_df["request_status"] == "Failed").sum())
print("Unique candidate videos:", len(discovery_df))

print("\nCandidates by movie:")

display(
    discovery_df.groupby(
        ["movie_record_id", "movie_title"]
    )
    .agg(
        candidate_count=("video_id", "nunique")
    )
    .reset_index()
)

print("\nDiscovery source type:")
print(discovery_df["discovery_source"].value_counts())

print("\nSaved files:")
print("Discovery:", DISCOVERY_FILE)
print("Log:", LOG_FILE)

print("\nOriginal allocation and existing batches remain unchanged.")

In [ ]:
import os
import pandas as pd
from google.colab import userdata
from googleapiclient.discovery import build
from IPython.display import display

# ============================================================
# BATCH 10 — YOUTUBE METADATA RETRIEVAL
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

DISCOVERY_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_DISCOVERY_BATCH_10.csv"
)

METADATA_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_BATCH_10.csv"
)

if os.path.exists(METADATA_FILE):
    raise FileExistsError(
        f"Metadata output already exists: {METADATA_FILE}"
    )

discovery = pd.read_csv(DISCOVERY_FILE)

assert discovery["video_id"].notna().all()

video_ids = discovery["video_id"].drop_duplicates().tolist()

print("Discovery records:", len(discovery))
print("Unique video IDs:", len(video_ids))

# ============================================================
# INITIALIZE API
# ============================================================

API_KEY = userdata.get("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY not found in Colab Secrets.")

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY,
    cache_discovery=False
)

# ============================================================
# RETRIEVE METADATA IN CHUNKS OF 50
# ============================================================

metadata_records = []
api_errors = []

for start in range(0, len(video_ids), 50):

    chunk = video_ids[start:start + 50]

    print(
        f"\nRetrieving metadata: "
        f"{start + 1}–{start + len(chunk)} of {len(video_ids)}"
    )

    try:
        response = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(chunk),
            maxResults=50
        ).execute()

        for item in response.get("items", []):

            snippet = item.get("snippet", {})
            statistics = item.get("statistics", {})
            content = item.get("contentDetails", {})

            metadata_records.append({
                "video_id": item.get("id"),
                "metadata_title": snippet.get("title"),
                "metadata_channel_id": snippet.get("channelId"),
                "metadata_channel_title": snippet.get("channelTitle"),
                "description": snippet.get("description"),
                "published_at_metadata": snippet.get("publishedAt"),
                "duration": content.get("duration"),
                "view_count": statistics.get("viewCount"),
                "like_count": statistics.get("likeCount"),
                "comment_count": statistics.get("commentCount"),
                "category_id": snippet.get("categoryId"),
                "default_language": snippet.get("defaultLanguage"),
                "default_audio_language": snippet.get("defaultAudioLanguage")
            })

        print("Metadata records returned:", len(response.get("items", [])))

    except Exception as e:
        # Do not print raw API exception or request URL.
        error_message = str(e).splitlines()[0][:250]
        api_errors.append({
            "chunk_start": start,
            "chunk_size": len(chunk),
            "error_message": error_message
        })
        print("Chunk failed. Sanitized error:", error_message)

# ============================================================
# BUILD METADATA DATAFRAME
# ============================================================

metadata = pd.DataFrame(metadata_records)

if metadata.empty:
    raise RuntimeError(
        "No metadata returned. Check API errors before proceeding."
    )

assert metadata["video_id"].is_unique

# Identify videos missing from metadata response
missing_ids = sorted(set(video_ids) - set(metadata["video_id"]))

metadata.to_csv(METADATA_FILE, index=False)

# ============================================================
# VALIDATION AND SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("BATCH 10 — METADATA RETRIEVAL COMPLETE")
print("=" * 70)

print("Discovery video IDs:", len(video_ids))
print("Metadata records:", len(metadata))
print("Missing metadata:", len(missing_ids))
print("API errors:", len(api_errors))

if missing_ids:
    print("\nMissing video IDs:")
    print(missing_ids)

if api_errors:
    print("\nAPI error summary:")
    display(pd.DataFrame(api_errors))

# Merge for movie-level retrieval coverage
metadata_check = discovery[
    ["movie_record_id", "movie_title", "video_id"]
].drop_duplicates().merge(
    metadata[["video_id", "metadata_title"]],
    on="video_id",
    how="left"
)

print("\nMetadata retrieval by movie:")
display(
    metadata_check.groupby(
        ["movie_record_id", "movie_title"]
    )
    .agg(
        candidate_videos=("video_id", "nunique"),
        metadata_retrieved=("metadata_title", "count")
    )
    .reset_index()
)

print("\nSaved metadata:", METADATA_FILE)
print("Original discovery and allocation files remain unchanged.")

In [ ]:
# ============================================================
# BATCH 10 — SCREENING COLUMN MISMATCH FIX
# Uses existing merged dataframe: df
# No API calls and no reprocessing of metadata
# ============================================================

import os
import pandas as pd
import numpy as np
from IPython.display import display

# ------------------------------------------------------------
# 1. INSPECT AND RESOLVE LANGUAGE COLUMN
# ------------------------------------------------------------

print("Current dataframe shape:", df.shape)
print("Available columns:")
print(df.columns.tolist())

if "language" not in df.columns:

    # Recover language from the original allocation
    allocation = pd.read_csv(
        "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
        "01_data/processed/audience_sentiment/"
        "ICMB_130_YOUTUBE_BATCH_10_MOVIE_ALLOCATION.csv"
    )

    language_map = allocation.set_index(
        "movie_record_id"
    )["language"]

    df["language"] = df["movie_record_id"].map(language_map)

assert df["language"].notna().all()
assert len(df) == 100

# ------------------------------------------------------------
# 2. CREATE SCREENING DATAFRAME
# ------------------------------------------------------------

screening_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "metadata_title",
    "metadata_channel_title",
    "published_at_metadata",
    "duration",
    "duration_seconds",
    "view_count",
    "comment_count",
    "provisional_source_type",
    "title_identity_overlap",
    "screening_score",
    "automated_decision",
    "metadata_reviewed",
    "content_reviewed",
    "review_status",
    "screening_notes",
    "description_metadata"
]

missing_columns = [
    col for col in screening_columns
    if col not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Other required columns are missing: {missing_columns}"
    )

screening = df[screening_columns].copy()
manual_review = screening.copy()

# ------------------------------------------------------------
# 3. OUTPUT PATHS
# ------------------------------------------------------------

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

SCREENING_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_10.csv"
)

REVIEW_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_MANUAL_REVIEW_BATCH_10.csv"
)

# Prevent accidental overwrite
for path in [SCREENING_FILE, REVIEW_FILE]:
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists. No file overwritten: {path}"
        )

# ------------------------------------------------------------
# 4. SAVE
# ------------------------------------------------------------

screening.to_csv(SCREENING_FILE, index=False)
manual_review.to_csv(REVIEW_FILE, index=False)

# ------------------------------------------------------------
# 5. VALIDATE
# ------------------------------------------------------------

assert len(screening) == 100
assert screening["video_id"].nunique() == 100
assert screening["language"].notna().all()
assert screening["content_reviewed"].eq("No").all()

print("\n" + "=" * 70)
print("BATCH 10 — SCREENING FILES SAVED SUCCESSFULLY")
print("=" * 70)

print("Screening shape:", screening.shape)
print("Manual review shape:", manual_review.shape)

print("\nDecision distribution:")
display(
    screening["automated_decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="candidate_count")
)

print("\nSource type distribution:")
display(
    screening["provisional_source_type"]
    .value_counts()
    .rename_axis("source_type")
    .reset_index(name="candidate_count")
)

print("\nMovie-wise screening:")
display(
    screening.pivot_table(
        index=["movie_record_id", "movie_title"],
        columns="automated_decision",
        values="video_id",
        aggfunc="count",
        fill_value=0
    ).reset_index()
)

print("\nSaved:")
print(SCREENING_FILE)
print(REVIEW_FILE)

print("\nNo API calls made. No existing files overwritten.")

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# BATCH 10 — CONSOLIDATED CANDIDATE REVIEW
# ============================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

SCREENING_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_10.csv"
)

OUTPUT_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_CANDIDATE_REVIEW_BATCH_10.csv"
)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        f"Output already exists: {OUTPUT_FILE}"
    )

# ============================================================
# 1. LOAD SCREENING DATA
# ============================================================

df = pd.read_csv(SCREENING_FILE)

assert len(df) == 100
assert df["video_id"].is_unique

# ============================================================
# 2. ADD REVIEW PRIORITY
# ============================================================

priority_map = {
    "Shortlist": 1,
    "Hold / Needs Review": 2,
    "Exclude / Not Primary Source": 3
}

df["review_priority"] = (
    df["automated_decision"].map(priority_map)
)

# ============================================================
# 3. CREATE CANDIDATE RANK WITHIN EACH MOVIE
# ============================================================

df = df.sort_values(
    [
        "movie_record_id",
        "review_priority",
        "screening_score",
        "view_count"
    ],
    ascending=[True, True, False, False],
    na_position="last"
).reset_index(drop=True)

df["candidate_rank"] = (
    df.groupby("movie_record_id").cumcount() + 1
)

# ============================================================
# 4. ADD REVIEW GUIDANCE
# ============================================================

def review_guidance(row):

    source_type = row["provisional_source_type"]

    if source_type in [
        "Movie Review / Analysis",
        "Movie Reaction",
        "Review + Reaction"
    ]:
        return (
            "Priority candidate: check film identity, "
            "substantial movie discussion and relevance."
        )

    if source_type == "Unclassified":
        return (
            "Check title, description and video content "
            "before considering inclusion."
        )

    if source_type == "Short / Unclear":
        return (
            "Check whether the video contains sufficient "
            "substantive discussion."
        )

    return (
        "Not a primary audience review/reaction source "
        "under the current criteria."
    )


df["review_guidance"] = df.apply(
    review_guidance,
    axis=1
)

df["manual_decision"] = "Pending"
df["manual_source_type"] = ""
df["manual_review_notes"] = ""
df["content_reviewed"] = "No"

# ============================================================
# 5. FINAL REVIEW COLUMN ORDER
# ============================================================

review_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "candidate_rank",
    "video_id",
    "metadata_title",
    "metadata_channel_title",
    "duration",
    "duration_seconds",
    "view_count",
    "comment_count",
    "provisional_source_type",
    "screening_score",
    "automated_decision",
    "review_priority",
    "review_guidance",
    "metadata_reviewed",
    "content_reviewed",
    "manual_decision",
    "manual_source_type",
    "manual_review_notes"
]

review = df[review_columns].copy()

# ============================================================
# 6. SAVE
# ============================================================

review.to_csv(OUTPUT_FILE, index=False)

# ============================================================
# 7. VALIDATION AND SUMMARY
# ============================================================

assert len(review) == 100
assert review["video_id"].nunique() == 100
assert review["candidate_rank"].between(1, 10).all()
assert review["manual_decision"].eq("Pending").all()

print("\n" + "=" * 75)
print("BATCH 10 — CONSOLIDATED CANDIDATE REVIEW CREATED")
print("=" * 75)

print("Total candidates:", len(review))
print("Unique movies:", review["movie_record_id"].nunique())
print("Unique videos:", review["video_id"].nunique())

print("\nTop candidates per movie:")

display(
    review[
        review["candidate_rank"] <= 5
    ][
        [
            "movie_title",
            "candidate_rank",
            "metadata_title",
            "metadata_channel_title",
            "duration",
            "provisional_source_type",
            "screening_score",
            "automated_decision"
        ]
    ]
)

print("\nDecision distribution:")
display(
    review.groupby(
        ["movie_title", "automated_decision"]
    ).size().unstack(fill_value=0)
)

print("\nSaved review file:")
print(OUTPUT_FILE)

print("\nNo videos approved automatically.")
print("No content marked as independently reviewed.")
print("Original screening files remain unchanged.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

REVIEW_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_CANDIDATE_REVIEW_BATCH_10.csv"
)

OUTPUT_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_PROVISIONAL_SELECTION_BATCH_10.csv"
)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(f"File already exists: {OUTPUT_FILE}")

review = pd.read_csv(REVIEW_FILE)

# Candidate ranks selected for preliminary review.
# Ajab Prem Ki Ghazab Kahani and Dasavathaaram are
# deliberately left for further discovery.

selected = {
    "2007_4": [1],   # Chak De! India
    "2007_5": [2],   # Partner
    "2008_2": [1],   # Rab Ne Bana Di Jodi
    "2008_4": [1],   # Singh Is Kinng
    "2008_5": [1],   # Jodhaa Akbar
    "2009_3": [1],   # Love Aaj Kal
    "2009_5": [1],   # Wanted
    "2020_3": [2]    # Sarileru Neekevvaru
}

selection = review[
    review.apply(
        lambda r: r["candidate_rank"] in selected.get(
            str(r["movie_record_id"]), []
        ),
        axis=1
    )
].copy()

selection["provisional_decision"] = "Pending Confirmation"
selection["manual_source_type"] = selection["provisional_source_type"]
selection["metadata_reviewed"] = "Yes"
selection["content_reviewed"] = "No"

selection["selection_notes"] = selection["movie_title"].map({
    "Chak De! India": "Reaction Part 1/3; metadata-based selection.",
    "Partner": "Reaction Part 1/3; metadata-based selection.",
    "Rab Ne Bana Di Jodi": "Long-form reaction; verify film identity.",
    "Singh Is Kinng": "Reaction Part 1/3; verify substantive discussion.",
    "Jodhaa Akbar": "Long-form reaction; metadata-based selection.",
    "Love Aaj Kal": "Verify 2009 film, not 2020 film.",
    "Wanted": "Reaction Part 3/3; partial-series limitation.",
    "Sarileru Neekevvaru": "Reaction Part 1/3; metadata-based selection."
})

selection.to_csv(OUTPUT_FILE, index=False)

print("Provisional selections:", len(selection))
print("Unique movies:", selection["movie_record_id"].nunique())
print("Unique video IDs:", selection["video_id"].nunique())
print("No source approved automatically.")

display(
    selection[
        [
            "movie_title",
            "video_id",
            "metadata_title",
            "metadata_channel_title",
            "duration",
            "manual_source_type",
            "provisional_decision",
            "selection_notes"
        ]
    ]
)

print("\nSaved:", OUTPUT_FILE)

In [ ]:
import os
import pandas as pd
import re
from IPython.display import display

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

SCREENING_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_10.csv"
)

METADATA_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_BATCH_10.csv"
)

OUTPUT_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_FOCUSED_VERIFICATION_BATCH_10.csv"
)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(f"File already exists: {OUTPUT_FILE}")

screening = pd.read_csv(SCREENING_FILE)
metadata = pd.read_csv(METADATA_FILE)

# Merge full descriptions without changing original files
evidence = screening.merge(
    metadata[["video_id", "description"]],
    on="video_id",
    how="left",
    validate="one_to_one"
)

# Focus on all candidates for problematic movies
# and top candidates for the remaining movies.
focus_ids = {
    "2009_3",  # Love Aaj Kal — film-year identity
    "2009_4",  # Ajab Prem — music vs film discussion
    "2008_3"   # Dasavathaaram — music/scene vs film discussion
}

focus = evidence[
    evidence["movie_record_id"].astype(str).isin(focus_ids)
].copy()

other = evidence[
    ~evidence["movie_record_id"].astype(str).isin(focus_ids)
].copy()

other = other[
    other["automated_decision"].isin(
        ["Shortlist", "Hold / Needs Review"]
    )
].sort_values(
    ["movie_record_id", "screening_score"],
    ascending=[True, False]
).groupby(
    "movie_record_id"
).head(3)

verification = pd.concat(
    [focus, other],
    ignore_index=True
)

# Flag obvious music-centric and year-specific signals
def evidence_flags(row):
    text = (
        str(row["metadata_title"]) + " " +
        str(row["description"])
    ).lower()

    flags = []

    if re.search(
        r"\b(song|lyrical|jukebox|music video|audio song|"
        r"video song|official song)\b", text
    ):
        flags.append("Music-related wording")

    if re.search(r"\b2020\b", text):
        flags.append("Mentions 2020")

    if re.search(r"\b2009\b", text):
        flags.append("Mentions 2009")

    if re.search(
        r"\b(review|analysis|explained|discussion|"
        r"reaction|reacts|movie commentary)\b", text
    ):
        flags.append("Film-discussion wording")

    return "; ".join(flags) if flags else "No strong keyword signal"

verification["metadata_evidence_flags"] = verification.apply(
    evidence_flags, axis=1
)

verification["description_excerpt"] = (
    verification["description"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 700)
)

verification["youtube_url"] = (
    "https://www.youtube.com/watch?v=" +
    verification["video_id"].astype(str)
)

columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "metadata_channel_title",
    "duration",
    "provisional_source_type",
    "automated_decision",
    "screening_score",
    "metadata_evidence_flags",
    "description_excerpt",
    "youtube_url"
]

verification = verification[columns].sort_values(
    ["movie_record_id", "screening_score"],
    ascending=[True, False]
)

verification.to_csv(OUTPUT_FILE, index=False)

print("=" * 75)
print("BATCH 10 — FOCUSED METADATA VERIFICATION")
print("=" * 75)

print("Candidates for verification:", len(verification))
print("Movies covered:", verification["movie_record_id"].nunique())

for movie_id, group in verification.groupby("movie_record_id"):
    print("\n" + "-" * 75)
    print(group.iloc[0]["movie_title"], "|", movie_id)
    display(group.drop(columns=["movie_record_id", "movie_title"]))

print("\nSaved:", OUTPUT_FILE)
print("No API calls made. No sources approved automatically.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

OLD_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_PROVISIONAL_SELECTION_BATCH_10.csv"
)
EVIDENCE_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_FOCUSED_VERIFICATION_BATCH_10.csv"
)
OUTPUT_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_PROVISIONAL_SELECTION_REVISED_BATCH_10.csv"
)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(f"Output already exists: {OUTPUT_FILE}")

old = pd.read_csv(OLD_FILE)
evidence = pd.read_csv(EVIDENCE_FILE)

# Retain the existing provisional selections except the Love Aaj Kal candidate.
revised = old[
    old["movie_record_id"].astype(str) != "2009_3"
].copy()

# Retrieve the replacement Love Aaj Kal (2009) and Ajab Prem candidates.
new_ids = {
    "bdZhJh2oNb4": {
        "decision": "Pending Confirmation",
        "notes": "Love Aaj Kal (2009) identified in description; reaction Part 1/2. Metadata Reviewed; content not independently reviewed."
    },
    "TIEq1KGrBPQ": {
        "decision": "Pending Confirmation",
        "notes": "Part 1 reaction; potential film-level discussion. Metadata Reviewed; content not independently reviewed."
    }
}

for video_id, details in new_ids.items():
    match = evidence[evidence["video_id"].astype(str) == video_id]

    if len(match) != 1:
        raise ValueError(
            f"Expected exactly one metadata record for {video_id}; found {len(match)}"
        )

    r = match.iloc[0]

    revised = pd.concat(
        [
            revised,
            pd.DataFrame([{
                "movie_record_id": r["movie_record_id"],
                "movie_title": r["movie_title"],
                "video_id": r["video_id"],
                "metadata_title": r["metadata_title"],
                "metadata_channel_title": r["metadata_channel_title"],
                "duration": r["duration"],
                "manual_source_type": r["provisional_source_type"],
                "provisional_decision": details["decision"],
                "selection_notes": details["notes"]
            }])
        ],
        ignore_index=True
    )

# Keep all decisions pending; do not approve automatically.
revised["provisional_decision"] = "Pending Confirmation"

revised = revised.sort_values(
    ["movie_record_id", "video_id"]
).reset_index(drop=True)

# Validation
assert revised["video_id"].is_unique
assert revised["movie_record_id"].nunique() == 9
assert len(revised) == 9

revised.to_csv(OUTPUT_FILE, index=False)

print("REVISED BATCH 10 PROVISIONAL SELECTIONS")
print("=" * 65)
print("Provisional sources:", len(revised))
print("Unique movies:", revised["movie_record_id"].nunique())
print("Unique video IDs:", revised["video_id"].nunique())
print("\nMovie-wise selection:")
display(revised[[
    "movie_title",
    "video_id",
    "metadata_channel_title",
    "manual_source_type",
    "provisional_decision",
    "selection_notes"
]])

print("\nSaved:", OUTPUT_FILE)
print("\nNo source approved automatically.")
print("Dasavathaaram remains on hold.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

SELECTION_FILE = os.path.join(
    RAW_DIR,
    "ICMB_130_YOUTUBE_PROVISIONAL_SELECTION_REVISED_BATCH_10.csv"
)

ALLOCATION_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_10_MOVIE_ALLOCATION.csv"
)

REFERENCE_MANIFEST = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_09.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_10.csv"
)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        f"Manifest already exists. Original preserved:\n{OUTPUT_FILE}"
    )

# Load source files
selection = pd.read_csv(SELECTION_FILE)
allocation = pd.read_csv(ALLOCATION_FILE)
reference = pd.read_csv(REFERENCE_MANIFEST)

# Preserve the exact reference manifest schema
manifest_columns = reference.columns.tolist()

required = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "metadata_channel_title"
]

missing = [c for c in required if c not in selection.columns]
if missing:
    raise ValueError(f"Missing selection columns: {missing}")

# Merge approved source selections with movie allocation details
manifest = selection.merge(
    allocation,
    on="movie_record_id",
    how="left",
    suffixes=("", "_allocation"),
    validate="many_to_one"
)

if manifest["tmdb_id"].isna().any():
    raise ValueError("Some selected movies could not be matched to allocation.")

# Map selection fields to the reference manifest structure
field_map = {
    "video_title": "metadata_title",
    "channel_title": "metadata_channel_title",
    "source_type": "manual_source_type"
}

for target, source in field_map.items():
    if target in manifest_columns and source in manifest.columns:
        manifest[target] = manifest[source]

if "selection_status" in manifest_columns:
    manifest["selection_status"] = "Approved"

if "selection_notes" in manifest_columns:
    manifest["selection_notes"] = manifest["selection_notes"].fillna("")

if "metadata_reviewed" in manifest_columns:
    manifest["metadata_reviewed"] = "Yes"

if "content_reviewed" in manifest_columns:
    manifest["content_reviewed"] = "No"

if "acquisition_status" in manifest_columns:
    manifest["acquisition_status"] = "Pending"

# One selected source per movie; target 100 comments per source
if "target_comments" in manifest_columns:
    manifest["target_comments"] = 100

# Retain only fields present in the previous approved manifest
for col in manifest_columns:
    if col not in manifest.columns:
        manifest[col] = pd.NA

manifest = manifest[manifest_columns].copy()

# Validation
assert len(manifest) == 9
assert manifest["video_id"].nunique() == 9
assert manifest["movie_record_id"].nunique() == 9

if "selection_status" in manifest.columns:
    assert (manifest["selection_status"] == "Approved").all()

if "content_reviewed" in manifest.columns:
    assert (manifest["content_reviewed"] == "No").all()

manifest.to_csv(OUTPUT_FILE, index=False)

print("=" * 70)
print("BATCH 10 — APPROVED ACQUISITION MANIFEST")
print("=" * 70)

print("Manifest shape:", manifest.shape)
print("Approved sources:", len(manifest))
print("Unique movies:", manifest["movie_record_id"].nunique())
print("Unique video IDs:", manifest["video_id"].nunique())

if "target_comments" in manifest.columns:
    print("Planned comment target:", manifest["target_comments"].sum())

print("\nManifest preview:")
display(manifest)

print("\nSaved:", OUTPUT_FILE)
print("\nDasavathaaram remains excluded from Batch 10.")
print("No YouTube API calls made.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

allocation_path = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment",
    "ICMB_130_YOUTUBE_BATCH_10_MOVIE_ALLOCATION.csv"
)

manifest_path = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment",
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_10.csv"
)

allocation = pd.read_csv(allocation_path, dtype={"movie_record_id": str})
manifest = pd.read_csv(manifest_path, dtype={"movie_record_id": str})

print("ALLOCATION COLUMNS")
print(allocation.columns.tolist())

print("\nALLOCATION — ALL NINE SELECTED MOVIES")
display(
    allocation[
        allocation["movie_record_id"].isin(
            manifest["movie_record_id"].unique()
        )
    ].sort_values("movie_record_id")
)

print("\nCURRENT MANIFEST — YEAR AND LANGUAGE CHECK")
display(
    manifest[[
        "movie_record_id",
        "movie_title",
        "movie_year",
        "language",
        "manual_verification_status",
        "manual_review_notes",
        "content_review_status",
        "acquisition_status"
    ]]
)

print("\nMISSING VALUE COUNTS")
print(
    manifest[[
        "movie_year",
        "language",
        "manual_verification_status",
        "content_review_status"
    ]].isna().sum()
)

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

MANIFEST_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_10.csv"
)

ALLOCATION_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_10_MOVIE_ALLOCATION.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_10_CORRECTED.csv"
)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(f"Corrected file already exists: {OUTPUT_FILE}")

manifest = pd.read_csv(
    MANIFEST_FILE,
    dtype={"movie_record_id": str}
)

allocation = pd.read_csv(
    ALLOCATION_FILE,
    dtype={"movie_record_id": str}
)

# Preserve all existing manifest fields and source selections.
corrected = manifest.merge(
    allocation[
        ["movie_record_id", "year", "rank", "language", "tmdb_id"]
    ],
    on="movie_record_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_allocation")
)

# Restore year and language from the authoritative allocation.
corrected["movie_year"] = corrected["year"]
corrected["language"] = corrected["language_allocation"]

# Populate review status accurately.
corrected["manual_verification_status"] = "Metadata Reviewed"
corrected["content_review_status"] = "Not Reviewed"

# Preserve the acquisition status and target.
corrected["acquisition_status"] = "Pending"
corrected["target_comments"] = 100

# Remove temporary merge columns.
corrected.drop(
    columns=["year", "rank", "language_allocation", "tmdb_id"],
    inplace=True
)

# Validation
assert len(corrected) == 9
assert corrected["video_id"].nunique() == 9
assert corrected["movie_record_id"].nunique() == 9
assert corrected["movie_year"].notna().all()
assert corrected["language"].notna().all()
assert (corrected["manual_verification_status"] == "Metadata Reviewed").all()
assert (corrected["content_review_status"] == "Not Reviewed").all()
assert (corrected["acquisition_status"] == "Pending").all()

corrected.to_csv(OUTPUT_FILE, index=False)

print("=" * 70)
print("BATCH 10 — CORRECTED APPROVED ACQUISITION MANIFEST")
print("=" * 70)

print("Shape:", corrected.shape)
print("Sources:", corrected["video_id"].nunique())
print("Movies:", corrected["movie_record_id"].nunique())
print("Planned comments:", corrected["target_comments"].sum())

display(corrected[[
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "manual_source_type",
    "manual_verification_status",
    "content_review_status",
    "acquisition_status",
    "target_comments"
]])

print("\nSaved:", OUTPUT_FILE)
print("\nOriginal manifest preserved.")
print("No API calls made.")

In [ ]:
import os
import time
import pandas as pd
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from IPython.display import display

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)
RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)
COMMENTS_DIR = os.path.join(RAW_DIR, "comments")

MANIFEST_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_10_CORRECTED.csv"
)
ALLOCATION_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_10_MOVIE_ALLOCATION.csv"
)
REFERENCE_RAW = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_BATCH_09.csv"
)

RAW_OUTPUT = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_BATCH_10.csv"
)
LOG_OUTPUT = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_10.csv"
)
CHECKPOINT_OUTPUT = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_BATCH_10_CHECKPOINT.csv"
)

for path in [RAW_OUTPUT, LOG_OUTPUT, CHECKPOINT_OUTPUT]:
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists; refusing to overwrite:\n{path}"
        )

# Load and validate inputs
manifest = pd.read_csv(
    MANIFEST_FILE, dtype={"movie_record_id": str}
)
allocation = pd.read_csv(
    ALLOCATION_FILE, dtype={"movie_record_id": str}
)
reference = pd.read_csv(REFERENCE_RAW, nrows=1)

assert len(manifest) == 9
assert manifest["video_id"].nunique() == 9
assert manifest["movie_record_id"].nunique() == 9
assert manifest["movie_year"].notna().all()
assert manifest["language"].notna().all()

manifest = manifest.merge(
    allocation[
        ["movie_record_id", "year", "rank", "language"]
    ],
    on="movie_record_id",
    how="left",
    suffixes=("", "_allocation"),
    validate="many_to_one"
)

# Use the established Batch 09 raw schema.
raw_columns = reference.columns.tolist()

# Candidate values for each comment record.
def build_raw_record(movie, comment, position):
    top = comment["snippet"]["topLevelComment"]["snippet"]

    values = {
        "batch_id": "Batch 10",
        "movie_record_id": movie["movie_record_id"],
        "movie_title": movie["movie_title"],
        "year": movie["year"],
        "rank": movie["rank"],
        "language": movie["language"],
        "video_id": movie["video_id"],
        "video_title": movie["video_title"],
        "channel_title": movie["channel_title"],
        "comment_id": comment["id"],
        "comment_text": top.get("textDisplay", ""),
        "author_display_name": top.get("authorDisplayName", ""),
        "author_channel_id": top.get("authorChannelId", {}).get("value", ""),
        "like_count": top.get("likeCount", 0),
        "published_at": top.get("publishedAt", ""),
        "updated_at": top.get("updatedAt", ""),
        "source_type": movie["manual_source_type"],
        "comment_position": position,
        "manual_source_type": movie["manual_source_type"]
    }

    return {col: values.get(col, pd.NA) for col in raw_columns}

# Retrieve API key from Colab Secret without displaying it.
api_key = userdata.get("YOUTUBE_API_KEY")
if not api_key:
    raise ValueError(
        "YOUTUBE_API_KEY was not found in Colab Secrets."
    )

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

all_comments = []
logs = []

print("=" * 75)
print("BATCH 10 — YOUTUBE COMMENT ACQUISITION")
print("=" * 75)
print("Approved sources:", len(manifest))
print("Target comments:", int(manifest["target_comments"].sum()))
print("Sampling: recent top-level comments (order=time)")
print("Replies: not collected")
print("Raw schema:", raw_columns)

for idx, movie in manifest.iterrows():
    video_id = movie["video_id"]
    target = int(movie["target_comments"])
    retrieved = []
    status = "Completed"
    error_code = ""
    error_message = ""
    pages = 0

    try:
        response = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=min(target, 100),
            order="time",
            textFormat="plainText"
        ).execute()

        pages += 1

        for position, item in enumerate(
            response.get("items", []), start=1
        ):
            retrieved.append(
                build_raw_record(movie, item, position)
            )

        if len(retrieved) == 0:
            status = "No Comments Retrieved"
        elif len(retrieved) < target:
            status = "Partial"

    except HttpError as e:
        status = "Failed"
        error_code = getattr(e.resp, "status", "HTTP Error")
        error_message = (
            "YouTube API request failed; details sanitized."
        )
    except Exception:
        status = "Failed"
        error_code = "Client/Runtime Error"
        error_message = (
            "Unexpected acquisition error; details sanitized."
        )

    all_comments.extend(retrieved)

    logs.append({
        "batch_id": "Batch 10",
        "movie_record_id": movie["movie_record_id"],
        "movie_title": movie["movie_title"],
        "video_id": video_id,
        "channel_title": movie["channel_title"],
        "target_comments": target,
        "retrieved_comments": len(retrieved),
        "unique_comment_ids": len({
            r.get("comment_id") for r in retrieved
        }),
        "api_pages": pages,
        "acquisition_status": status,
        "error_code": error_code,
        "error_message": error_message,
        "sampling_order": "time",
        "collected_at_utc": datetime.now(
            timezone.utc
        ).isoformat()
    })

    # Persist checkpoint and log after every source.
    pd.DataFrame(all_comments, columns=raw_columns).to_csv(
        CHECKPOINT_OUTPUT, index=False
    )
    pd.DataFrame(logs).to_csv(LOG_OUTPUT, index=False)

    print(
        f"{movie['movie_title']}: "
        f"{len(retrieved)}/{target} comments — {status}"
    )

    time.sleep(0.2)

# Final raw file
raw_df = pd.DataFrame(all_comments, columns=raw_columns)
log_df = pd.DataFrame(logs)

raw_df.to_csv(RAW_OUTPUT, index=False)

print("\n" + "=" * 75)
print("BATCH 10 ACQUISITION SUMMARY")
print("=" * 75)

print("Sources processed:", len(log_df))
print("Completed:", (log_df["acquisition_status"] == "Completed").sum())
print("Partial:", (log_df["acquisition_status"] == "Partial").sum())
print("Failed:", (log_df["acquisition_status"] == "Failed").sum())
print("No comments:", (log_df["acquisition_status"] == "No Comments Retrieved").sum())
print("Total comments retrieved:", len(raw_df))
print("Unique comment IDs:", raw_df["comment_id"].nunique())
print("Unique videos:", raw_df["video_id"].nunique())
print("Unique movies:", raw_df["movie_record_id"].nunique())

print("\nMovie-wise acquisition:")
display(log_df[[
    "movie_title",
    "video_id",
    "target_comments",
    "retrieved_comments",
    "acquisition_status",
    "api_pages"
]])

print("\nRaw comments saved:", RAW_OUTPUT)
print("Acquisition log saved:", LOG_OUTPUT)
print("Checkpoint saved:", CHECKPOINT_OUTPUT)

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from collections import Counter

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

COMMENTS_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube", "comments"
)

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

RAW_FILE = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_BATCH_10.csv"
)

OUTPUTS = {
    "overall": os.path.join(
        PROCESSED_DIR, "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_10.csv"
    ),
    "movie": os.path.join(
        PROCESSED_DIR, "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_10.csv"
    ),
    "video": os.path.join(
        PROCESSED_DIR, "ICMB_130_YOUTUBE_COMMENT_VIDEO_QUALITY_BATCH_10.csv"
    ),
    "duplicates": os.path.join(
        PROCESSED_DIR, "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_BATCH_10.csv"
    )
}

# Protect existing reports.
for name, path in OUTPUTS.items():
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists ({name}); refusing to overwrite:\n{path}"
        )

df = pd.read_csv(RAW_FILE)

required = [
    "movie_record_id", "movie_title", "movie_year", "language",
    "video_id", "video_title", "channel_title",
    "comment_id", "comment_text"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Normalize text for analysis only; raw comment_text remains unchanged.
df["comment_text"] = df["comment_text"].fillna("").astype(str)
df["normalized_text"] = (
    df["comment_text"]
    .str.lower()
    .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["word_count"] = df["comment_text"].str.findall(r"\b[\w']+\b").str.len()
df["character_count"] = df["comment_text"].str.len()
df["is_empty"] = df["comment_text"].str.strip().eq("")
df["is_very_short"] = df["word_count"].le(3) & ~df["is_empty"]
df["contains_url"] = df["comment_text"].str.contains(
    r"https?://|www\.", case=False, regex=True
)

# Script profiling: this is NOT language identification.
def detect_script(text):
    if not text.strip():
        return "Empty"

    scripts = set()

    if re.search(r"[A-Za-z]", text):
        scripts.add("Latin")
    if re.search(r"[\u0900-\u097F]", text):
        scripts.add("Devanagari")
    if re.search(r"[\u0B80-\u0BFF]", text):
        scripts.add("Tamil")
    if re.search(r"[\u0C00-\u0C7F]", text):
        scripts.add("Telugu")
    if re.search(r"[\u0C80-\u0CFF]", text):
        scripts.add("Kannada")
    if re.search(r"[\u0980-\u09FF]", text):
        scripts.add("Bengali")
    if re.search(r"[\u0A00-\u0A7F]", text):
        scripts.add("Gurmukhi")
    if re.search(r"[\u0D00-\u0D7F]", text):
        scripts.add("Malayalam")

    if len(scripts) > 1:
        return "Mixed"
    if len(scripts) == 1:
        return next(iter(scripts))
    return "Other/Emoji/Symbols"

df["script_category"] = df["comment_text"].apply(detect_script)

# Length categories
df["length_category"] = pd.cut(
    df["word_count"],
    bins=[-1, 3, 10, 30, np.inf],
    labels=["Very Short", "Short", "Medium", "Long"]
)

def quality_metrics(group):
    nonempty = group.loc[~group["is_empty"]]

    return pd.Series({
        "comment_records": len(group),
        "unique_comment_ids": group["comment_id"].nunique(),
        "unique_videos": group["video_id"].nunique(),
        "empty_comments": int(group["is_empty"].sum()),
        "nonempty_comments": len(nonempty),
        "very_short_comments_le_3_words": int(group["is_very_short"].sum()),
        "comments_with_urls": int(group["contains_url"].sum()),
        "average_word_count": round(nonempty["word_count"].mean(), 2)
            if len(nonempty) else 0,
        "median_word_count": round(nonempty["word_count"].median(), 2)
            if len(nonempty) else 0,
        "average_character_count": round(nonempty["character_count"].mean(), 2)
            if len(nonempty) else 0,
        "median_character_count": round(nonempty["character_count"].median(), 2)
            if len(nonempty) else 0
    })

# 1. Overall quality
overall = quality_metrics(df).to_frame().T
overall.insert(0, "batch_id", "Batch 10")

# 2. Movie-wise quality
movie_rows = []

for movie_id, group in df.groupby("movie_record_id", dropna=False):
    row = {
        "movie_record_id": movie_id,
        "movie_title": group["movie_title"].iloc[0],
        "movie_year": group["movie_year"].iloc[0],
        "language": group["language"].iloc[0]
    }
    row.update(quality_metrics(group).to_dict())
    movie_rows.append(row)

movie_quality = pd.DataFrame(movie_rows)

# 3. Video-wise quality
video_rows = []

for video_id, group in df.groupby("video_id", dropna=False):
    row = {
        "movie_record_id": group["movie_record_id"].iloc[0],
        "movie_title": group["movie_title"].iloc[0],
        "video_id": video_id,
        "video_title": group["video_title"].iloc[0],
        "channel_title": group["channel_title"].iloc[0]
    }
    row.update(quality_metrics(group).to_dict())
    video_rows.append(row)

video_quality = pd.DataFrame(video_rows)

# 4. Duplicate text audit. Repeated text is flagged, never deleted.
valid_duplicates = df[
    df["normalized_text"].ne("")
].copy()

duplicate_groups = (
    valid_duplicates.groupby("normalized_text")
    .agg(
        repeated_text_records=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        unique_movies=("movie_record_id", "nunique"),
        movie_titles=("movie_title", lambda x: " | ".join(sorted(set(x)))),
        video_ids=("video_id", lambda x: " | ".join(sorted(set(x)))),
        original_text_examples=("comment_text", lambda x: " || ".join(
            pd.Series(x).drop_duplicates().head(3).tolist()
        ))
    )
    .reset_index()
    .rename(columns={"normalized_text": "normalized_comment_text"})
)

duplicate_audit = duplicate_groups[
    duplicate_groups["repeated_text_records"] > 1
].copy()

duplicate_audit["duplicate_scope"] = np.where(
    duplicate_audit["unique_videos"] > 1,
    "Across Videos",
    "Within Video"
)

duplicate_audit = duplicate_audit.sort_values(
    ["repeated_text_records", "unique_videos"],
    ascending=False
).reset_index(drop=True)

# Save all four reports.
overall.to_csv(OUTPUTS["overall"], index=False)
movie_quality.to_csv(OUTPUTS["movie"], index=False)
video_quality.to_csv(OUTPUTS["video"], index=False)
duplicate_audit.to_csv(OUTPUTS["duplicates"], index=False)

# Summary
print("=" * 75)
print("BATCH 10 — COMMENT QUALITY PROFILING")
print("=" * 75)

print("\nOVERALL QUALITY")
display(overall)

print("\nMOVIE-WISE QUALITY")
display(movie_quality)

print("\nVIDEO-WISE QUALITY")
display(video_quality)

print("\nSCRIPT DISTRIBUTION")
display(
    df["script_category"]
    .value_counts(dropna=False)
    .rename_axis("script_category")
    .reset_index(name="comment_records")
)

print("\nCOMMENT LENGTH DISTRIBUTION")
display(
    df["length_category"]
    .value_counts(sort=False)
    .rename_axis("length_category")
    .reset_index(name="comment_records")
)

print("\nDUPLICATE TEXT AUDIT")
print("Repeated-text groups:", len(duplicate_audit))
print(
    "Records in repeated-text groups:",
    int(duplicate_audit["repeated_text_records"].sum())
)
print(
    "Within-video groups:",
    int((duplicate_audit["duplicate_scope"] == "Within Video").sum())
)
print(
    "Across-video groups:",
    int((duplicate_audit["duplicate_scope"] == "Across Videos").sum())
)
display(duplicate_audit.head(20))

print("\nVALIDATION")
print("Raw records:", len(df))
print("Unique comment IDs:", df["comment_id"].nunique())
print("Unique videos:", df["video_id"].nunique())
print("Unique movies:", df["movie_record_id"].nunique())
print("Empty comments:", int(df["is_empty"].sum()))

assert len(df) == 900
assert df["comment_id"].nunique() == 900
assert df["video_id"].nunique() == 9
assert df["movie_record_id"].nunique() == 9

print("\nAll four quality reports saved successfully.")
for name, path in OUTPUTS.items():
    print(f"{name}: {path}")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

MOVIE_QUALITY_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_10.csv"
)

ALLOCATION_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_10_MOVIE_ALLOCATION.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_10_CORRECTED.csv"
)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        f"Corrected report already exists:\n{OUTPUT_FILE}"
    )

quality = pd.read_csv(
    MOVIE_QUALITY_FILE,
    dtype={"movie_record_id": str}
)

allocation = pd.read_csv(
    ALLOCATION_FILE,
    dtype={"movie_record_id": str}
)

# Restore year and language from the authoritative allocation.
corrected = quality.drop(
    columns=["movie_year", "language"],
    errors="ignore"
).merge(
    allocation[
        ["movie_record_id", "year", "language"]
    ],
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

corrected.rename(
    columns={"year": "movie_year"},
    inplace=True
)

# Validate all nine movie records.
assert len(corrected) == 9
assert corrected["movie_record_id"].nunique() == 9
assert corrected["movie_year"].notna().all()
assert corrected["language"].notna().all()
assert corrected["comment_records"].sum() == 900

corrected.to_csv(OUTPUT_FILE, index=False)

print("=" * 70)
print("BATCH 10 — CORRECTED MOVIE QUALITY REPORT")
print("=" * 70)

display(corrected[[
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "comment_records",
    "average_word_count",
    "median_word_count",
    "very_short_comments_le_3_words"
]])

print("\nTotal comments:", corrected["comment_records"].sum())
print("Missing years:", corrected["movie_year"].isna().sum())
print("Missing languages:", corrected["language"].isna().sum())
print("\nSaved:", OUTPUT_FILE)
print("Original quality report preserved.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

COMMENTS_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube", "comments"
)

FILES = {
    "previous_consolidated": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_09.csv"
    ),
    "batch10_raw": os.path.join(
        COMMENTS_DIR,
        "ICMB_130_YOUTUBE_COMMENT_BATCH_10.csv"
    ),
    "previous_coverage": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_09.csv"
    ),
    "master_dataset": os.path.join(
        ROOT, "01_data", "processed",
        "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
    ),
    "batch10_allocation": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_BATCH_10_MOVIE_ALLOCATION.csv"
    )
}

for name, path in FILES.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"{name} not found:\n{path}")

previous = pd.read_csv(FILES["previous_consolidated"])
batch10 = pd.read_csv(FILES["batch10_raw"])
coverage = pd.read_csv(FILES["previous_coverage"])
master = pd.read_csv(FILES["master_dataset"])
allocation = pd.read_csv(FILES["batch10_allocation"])

print("=" * 75)
print("PILOT + BATCHES 01–10 — CONSOLIDATION PREFLIGHT")
print("=" * 75)

for name, df in [
    ("Previous consolidated", previous),
    ("Batch 10 raw", batch10),
    ("Previous coverage", coverage),
    ("Master dataset", master),
    ("Batch 10 allocation", allocation)
]:
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

print("\n" + "=" * 75)
print("KEY VALIDATION")
print("=" * 75)

print("Previous comment IDs:", previous["comment_id"].nunique())
print("Batch 10 comment IDs:", batch10["comment_id"].nunique())
print("Batch 10 duplicate IDs:", batch10["comment_id"].duplicated().sum())

print("Previous videos:", previous["video_id"].nunique())
print("Batch 10 videos:", batch10["video_id"].nunique())

print("Previous movies:", previous["movie_record_id"].nunique())
print("Batch 10 movies:", batch10["movie_record_id"].nunique())

print("\nColumns in previous consolidated but missing from Batch 10:")
print(sorted(set(previous.columns) - set(batch10.columns)))

print("\nColumns in Batch 10 but missing from previous consolidated:")
print(sorted(set(batch10.columns) - set(previous.columns)))

print("\nPreflight complete. No files modified.")

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import unicodedata

# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------
ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)
COMMENTS_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube", "comments"
)

OLD_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_09.csv"
)
BATCH10_FILE = os.path.join(
    COMMENTS_DIR,
    "ICMB_130_YOUTUBE_COMMENT_BATCH_10.csv"
)
OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_10.csv"
)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        "Output already exists. No file was overwritten:\n" + OUTPUT_FILE
    )

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
old = pd.read_csv(OLD_FILE)
new = pd.read_csv(BATCH10_FILE)

# ---------------------------------------------------------
# NORMALIZE BATCH 10 TO THE EXISTING 32-COLUMN SCHEMA
# ---------------------------------------------------------
new["normalized_comment_text"] = (
    new["comment_text"]
    .fillna("")
    .astype(str)
    .str.lower()
    .apply(lambda x: unicodedata.normalize("NFKC", x))
    .str.replace(r"http\S+|www\.\S+", " ", regex=True)
    .str.replace(r"[^\w\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

new["source_batch"] = "Batch 10"
new["batch_id"] = "10"
new["source_type"] = "YouTube Reaction/Review"
new["comment_source"] = "YouTube"
new["sampling_order"] = "time"
new["sampling_unit"] = "Top-level comments"
new["acquisition_order"] = new["comment_position"]

# Fields unavailable in Batch 10 raw data remain missing.
new["author_channel_id"] = pd.NA
new["comment_reply_count"] = pd.NA
new["retrieved_at"] = pd.NA
new["series_part"] = pd.NA
new["comment_author"] = new["author_name"]
new["reply_count"] = pd.NA
new["acquired_at_utc"] = pd.NA
new["author_display_name"] = new["author_name"]
new["total_reply_count"] = pd.NA
new["acquisition_timestamp_utc"] = pd.NA

# Select the exact existing column order.
new = new[old.columns]

# ---------------------------------------------------------
# ALIGN DATA TYPES AND CONCATENATE
# ---------------------------------------------------------
combined = pd.concat(
    [old, new],
    ignore_index=True,
    sort=False
)

# ---------------------------------------------------------
# VALIDATION
# ---------------------------------------------------------
print("=" * 75)
print("PILOT + BATCHES 01–10 — CONSOLIDATION VALIDATION")
print("=" * 75)

print("Previous records:", len(old))
print("Batch 10 records:", len(new))
print("Combined records:", len(combined))
print("Combined columns:", len(combined.columns))

print("\nUnique comment IDs:", combined["comment_id"].nunique())
print("Duplicate comment IDs:", combined["comment_id"].duplicated().sum())
print("Unique videos:", combined["video_id"].nunique())
print("Unique movies:", combined["movie_record_id"].nunique())

print("\nBatch-wise records:")
print(combined["source_batch"].value_counts(dropna=False).to_string())

print("\nBatch 10 metadata validation:")
batch10_check = combined[
    combined["source_batch"].astype(str).str.contains("10", na=False)
]

print("Batch 10 records:", len(batch10_check))
print("Missing movie years:", batch10_check["movie_year"].isna().sum())
print("Missing movie IDs:", batch10_check["movie_record_id"].isna().sum())
print("Missing video IDs:", batch10_check["video_id"].isna().sum())
print("Missing comment IDs:", batch10_check["comment_id"].isna().sum())

# Stop if comment IDs are duplicated or expected Batch 10 records are missing.
assert combined["comment_id"].nunique() == len(combined), \
    "Duplicate comment IDs detected. Output not saved."

assert len(batch10_check) == 900, \
    "Batch 10 record count mismatch. Output not saved."

assert combined["movie_record_id"].nunique() == 81, \
    "Unexpected movie coverage count. Inspect before saving."

assert len(combined.columns) == 32, \
    "Consolidated schema mismatch."

# ---------------------------------------------------------
# SAVE NEW VERSION
# ---------------------------------------------------------
combined.to_csv(OUTPUT_FILE, index=False)

print("\n" + "=" * 75)
print("CONSOLIDATION SAVED")
print("=" * 75)
print(OUTPUT_FILE)
print("Final shape:", combined.shape)
print("Original consolidated and raw files preserved.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

old_file = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_10.csv"
)

allocation_file = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_10_MOVIE_ALLOCATION.csv"
)

manifest_file = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_10_CORRECTED.csv"
)

output_file = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_10_REPAIRED.csv"
)

if os.path.exists(output_file):
    raise FileExistsError("Output already exists. No overwrite.")

df = pd.read_csv(old_file)
allocation = pd.read_csv(allocation_file)
manifest = pd.read_csv(manifest_file)

original_columns = df.columns.tolist()
batch10_mask = df["source_batch"].eq("Batch 10")

year_map = allocation.set_index("movie_record_id")["year"]
language_map = allocation.set_index("movie_record_id")["language"]
source_map = (
    manifest.drop_duplicates("video_id")
    .set_index("video_id")["manual_source_type"]
)

# Restore movie year.
df.loc[batch10_mask, "movie_year"] = (
    df.loc[batch10_mask, "movie_record_id"].map(year_map)
)

# Preserve language in the existing schema only if the column exists.
if "language" in original_columns:
    df.loc[batch10_mask, "language"] = (
        df.loc[batch10_mask, "movie_record_id"].map(language_map)
    )

# Restore source type.
df.loc[batch10_mask, "source_type"] = (
    df.loc[batch10_mask, "video_id"].map(source_map)
)

# Ensure the original 32-column schema is retained.
df = df[original_columns]

# Validation.
assert len(df) == 8420
assert len(df.columns) == 32
assert df["comment_id"].nunique() == 8420
assert batch10_mask.sum() == 900
assert df.loc[batch10_mask, "movie_year"].notna().all()
assert df.loc[batch10_mask, "source_type"].notna().all()

print("Schema preserved:", len(df.columns), "columns")
print("Total records:", len(df))
print("Batch 10 records:", batch10_mask.sum())
print("Missing Batch 10 years:",
      df.loc[batch10_mask, "movie_year"].isna().sum())
print("Missing Batch 10 source types:",
      df.loc[batch10_mask, "source_type"].isna().sum())
print("Unique comment IDs:", df["comment_id"].nunique())

df.to_csv(output_file, index=False)

print("\nREPAIRED FILE SAVED:")
print(output_file)
print("Original files preserved.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

files = {
    "repaired_consolidated": "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_10_REPAIRED.csv",
    "previous_coverage": "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_09.csv",
    "previous_duplicate_audit": "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_PILOT_BATCH_01_09.csv",
    "master": "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
}

data = {}

for name, filename in files.items():
    path = os.path.join(
        PROCESSED_DIR if name != "master" else os.path.join(ROOT, "01_data", "processed"),
        filename
    )
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    data[name] = pd.read_csv(path)

for name, df in data.items():
    print("\n" + "=" * 70)
    print(name.upper())
    print("=" * 70)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

print("\nInspection complete. No files modified.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

INPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_10_REPAIRED.csv"
)

MASTER_FILE = os.path.join(
    ROOT, "01_data", "processed",
    "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

OUTPUTS = {
    "coverage": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_10.csv"
    ),
    "duplicates": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_PILOT_BATCH_01_10.csv"
    ),
    "summary": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_COVERAGE_SUMMARY_PILOT_BATCH_01_10.csv"
    )
}

for path in [INPUT_FILE, MASTER_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

for path in OUTPUTS.values():
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists; no file overwritten:\n{path}"
        )

comments = pd.read_csv(INPUT_FILE)
master = pd.read_csv(MASTER_FILE)

# ---------------------------------------------------------
# 1. MOVIE-LEVEL COVERAGE
# ---------------------------------------------------------
coverage_counts = (
    comments.groupby("movie_record_id")
    .agg(
        comment_records=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        acquisition_batches=("source_batch", "nunique")
    )
    .reset_index()
)

coverage = master[
    ["movie_record_id", "movie_title", "year", "rank", "language"]
].merge(
    coverage_counts,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

count_columns = [
    "comment_records",
    "unique_comment_ids",
    "unique_videos",
    "acquisition_batches"
]

coverage[count_columns] = coverage[count_columns].fillna(0).astype(int)

coverage["coverage_status"] = coverage["comment_records"].apply(
    lambda x: "Covered" if x > 0 else "Not Covered"
)

def classify_coverage(count):
    if count == 0:
        return "No Comments"
    elif count < 50:
        return "Low (1–49)"
    elif count < 100:
        return "Moderate (50–99)"
    return "Higher (100+)"

coverage["coverage_category"] = coverage["comment_records"].apply(
    classify_coverage
)

coverage = coverage[
    [
        "movie_record_id", "movie_title", "year", "rank", "language",
        "comment_records", "unique_comment_ids", "unique_videos",
        "acquisition_batches", "coverage_status", "coverage_category"
    ]
]

# ---------------------------------------------------------
# 2. DUPLICATE-TEXT AUDIT
# ---------------------------------------------------------
valid_text = comments[
    comments["normalized_comment_text"].notna()
    & comments["normalized_comment_text"].astype(str).str.strip().ne("")
].copy()

duplicate_rows = valid_text[
    valid_text.duplicated(
        "normalized_comment_text", keep=False
    )
].copy()

duplicate_audit = (
    duplicate_rows.groupby("normalized_comment_text")
    .agg(
        repeated_records=("comment_id", "size"),
        unique_videos=("video_id", "nunique"),
        unique_movies=("movie_record_id", "nunique"),
        comment_ids=("comment_id", lambda x: " | ".join(x.astype(str))),
        video_ids=("video_id", lambda x: " | ".join(x.dropna().astype(str).unique())),
        movie_titles=("movie_title", lambda x: " | ".join(x.dropna().astype(str).unique())),
        source_batches=("source_batch", lambda x: " | ".join(x.dropna().astype(str).unique())),
        example_comment=("comment_text", "first")
    )
    .reset_index()
)

duplicate_audit["duplicate_scope"] = duplicate_audit.apply(
    lambda row: (
        "Across Videos"
        if row["unique_videos"] > 1
        else "Within Video"
    ),
    axis=1
)

duplicate_audit = duplicate_audit.sort_values(
    ["repeated_records", "unique_movies"],
    ascending=False
).reset_index(drop=True)

# ---------------------------------------------------------
# 3. COVERAGE SUMMARY
# ---------------------------------------------------------
coverage_summary = (
    coverage.groupby("coverage_category")
    .agg(
        movie_count=("movie_record_id", "count"),
        comment_records=("comment_records", "sum")
    )
    .reset_index()
)

# ---------------------------------------------------------
# 4. VALIDATION
# ---------------------------------------------------------
print("=" * 75)
print("PILOT + BATCHES 01–10 — FULL CORPUS AUDIT")
print("=" * 75)

print("\nCOMMENT CORPUS")
print("Total comments:", len(comments))
print("Unique comment IDs:", comments["comment_id"].nunique())
print("Duplicate comment IDs:", comments["comment_id"].duplicated().sum())
print("Unique videos:", comments["video_id"].nunique())
print("Unique movies:", comments["movie_record_id"].nunique())

print("\nMOVIE COVERAGE")
print("Master movies:", len(master))
print("Coverage records:", len(coverage))
print(coverage["coverage_category"].value_counts().to_string())

print("\nCOVERAGE BY CATEGORY")
print(coverage_summary.to_string(index=False))

print("\nDUPLICATE-TEXT AUDIT")
print("Repeated-text groups:", len(duplicate_audit))
print(
    "Records in repeated-text groups:",
    duplicate_audit["repeated_records"].sum()
)
print(
    "Within-video groups:",
    (duplicate_audit["duplicate_scope"] == "Within Video").sum()
)
print(
    "Across-video groups:",
    (duplicate_audit["duplicate_scope"] == "Across Videos").sum()
)

# Stop before saving if integrity checks fail.
assert len(master) == 130
assert len(coverage) == 130
assert len(comments) == 8420
assert comments["comment_id"].nunique() == 8420
assert coverage["movie_record_id"].nunique() == 130
assert coverage["comment_records"].sum() == 8420

# ---------------------------------------------------------
# 5. SAVE NEW VERSIONED REPORTS
# ---------------------------------------------------------
coverage.to_csv(OUTPUTS["coverage"], index=False)
duplicate_audit.to_csv(OUTPUTS["duplicates"], index=False)
coverage_summary.to_csv(OUTPUTS["summary"], index=False)

print("\n" + "=" * 75)
print("ALL THREE REPORTS SAVED")
print("=" * 75)

for name, path in OUTPUTS.items():
    print(f"{name}: {path}")

print("\nOriginal reports and raw comment files remain unchanged.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

FILES = {
    "previous_report": "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_09.csv",
    "previous_summary": "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_09.csv",
    "previous_language_summary": "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_09.csv",
    "current_coverage": "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_10.csv",
    "batch10_allocation": "ICMB_130_YOUTUBE_BATCH_10_MOVIE_ALLOCATION.csv"
}

data = {}

for name, filename in FILES.items():
    path = os.path.join(PROCESSED_DIR, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    data[name] = pd.read_csv(path)

print("=" * 75)
print("ACQUISITION PLANNING — PREFLIGHT")
print("=" * 75)

for name, df in data.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))

print("\n" + "=" * 75)
print("CURRENT COVERAGE CHECK")
print("=" * 75)

coverage = data["current_coverage"]

print("Total movies:", len(coverage))
print("Covered movies:", (coverage["comment_records"] > 0).sum())
print("Uncovered movies:", (coverage["comment_records"] == 0).sum())
print("Total comments:", coverage["comment_records"].sum())

print("\nPreflight complete. No files modified.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

report = pd.read_csv(os.path.join(
    DIR, "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_09.csv"
))

summary = pd.read_csv(os.path.join(
    DIR, "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_09.csv"
))

language_summary = pd.read_csv(os.path.join(
    DIR, "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_09.csv"
))

print("=" * 70)
print("EXISTING ACQUISITION PRIORITY LABELS")
print("=" * 70)
print(report["acquisition_priority"].value_counts(dropna=False).to_string())

print("\n" + "=" * 70)
print("PREVIOUS COVERAGE SUMMARY — ALL ROWS")
print("=" * 70)
print(summary.to_string(index=False))

print("\n" + "=" * 70)
print("PREVIOUS LANGUAGE SUMMARY — ALL ROWS")
print("=" * 70)
print(language_summary.to_string(index=False))

print("\n" + "=" * 70)
print("TARGET VALIDATION")
print("=" * 70)
print("Previous total additional target:",
      report["additional_comment_target"].sum())

print("Priority labels and planning rules inspected.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

COVERAGE_FILE = os.path.join(
    DIR, "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_10.csv"
)

OUTPUTS = {
    "report": os.path.join(
        DIR, "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_10.csv"
    ),
    "summary": os.path.join(
        DIR, "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_10.csv"
    ),
    "language": os.path.join(
        DIR, "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_10.csv"
    )
}

if not os.path.exists(COVERAGE_FILE):
    raise FileNotFoundError(COVERAGE_FILE)

for path in OUTPUTS.values():
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists. No overwrite:\n{path}"
        )

coverage = pd.read_csv(COVERAGE_FILE)

# ---------------------------------------------------------
# 1. MOVIE-LEVEL PLANNING REPORT
# ---------------------------------------------------------
report = coverage.copy()

report["acquisition_priority"] = report["coverage_category"].map({
    "No Comments": "High — No Comments",
    "Low (1–49)": "High — Low Coverage",
    "Moderate (50–99)": "Medium — Moderate Coverage",
    "Higher (100+)": "Covered — Monitor"
})

report["additional_comment_target"] = (
    100 - report["comment_records"]
).clip(lower=0).astype(int)

# Preserve established column order.
report = report[
    [
        "movie_record_id", "movie_title", "year", "rank", "language",
        "comment_records", "unique_comment_ids", "unique_videos",
        "acquisition_batches", "coverage_status", "coverage_category",
        "acquisition_priority", "additional_comment_target"
    ]
]

# ---------------------------------------------------------
# 2. COVERAGE CATEGORY SUMMARY
# ---------------------------------------------------------
summary = (
    report.groupby("coverage_category")
    .agg(
        movie_count=("movie_record_id", "count"),
        existing_comments=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reset_index()
)

category_order = [
    "No Comments", "Low (1–49)",
    "Moderate (50–99)", "Higher (100+)"
]

summary["coverage_category"] = pd.Categorical(
    summary["coverage_category"],
    categories=category_order,
    ordered=True
)

summary = summary.sort_values(
    "coverage_category"
).reset_index(drop=True)

summary["coverage_category"] = summary[
    "coverage_category"
].astype(str)

# ---------------------------------------------------------
# 3. LANGUAGE-LEVEL SUMMARY
# ---------------------------------------------------------
language_summary = (
    report.groupby("language", dropna=False)
    .agg(
        total_movies=("movie_record_id", "count"),
        covered_movies=(
            "comment_records",
            lambda x: int((x > 0).sum())
        ),
        uncovered_movies=(
            "comment_records",
            lambda x: int((x == 0).sum())
        ),
        comment_records=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reset_index()
)

# ---------------------------------------------------------
# 4. VALIDATION
# ---------------------------------------------------------
print("=" * 75)
print("PILOT + BATCHES 01–10 — UPDATED ACQUISITION PLAN")
print("=" * 75)

print("\nMOVIE-LEVEL COVERAGE")
print("Total movies:", len(report))
print("Covered movies:", (report["comment_records"] > 0).sum())
print("Uncovered movies:", (report["comment_records"] == 0).sum())
print("Total comments:", report["comment_records"].sum())
print("Remaining provisional target:",
      report["additional_comment_target"].sum())

print("\nCOVERAGE SUMMARY")
print(summary.to_string(index=False))

print("\nLANGUAGE SUMMARY")
print(language_summary.to_string(index=False))

# Integrity checks.
assert len(report) == 130
assert report["movie_record_id"].nunique() == 130
assert report["comment_records"].sum() == 8420
assert report["additional_comment_target"].sum() == 4983
assert language_summary["total_movies"].sum() == 130
assert language_summary["comment_records"].sum() == 8420

# ---------------------------------------------------------
# 5. SAVE NEW REPORTS
# ---------------------------------------------------------
report.to_csv(OUTPUTS["report"], index=False)
summary.to_csv(OUTPUTS["summary"], index=False)
language_summary.to_csv(OUTPUTS["language"], index=False)

print("\n" + "=" * 75)
print("THREE PLANNING REPORTS SAVED")
print("=" * 75)

for name, path in OUTPUTS.items():
    print(f"{name}: {path}")

print("\nPrevious planning reports preserved.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

INPUT = os.path.join(
    DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_10.csv"
)

OUTPUT = os.path.join(
    DIR,
    "ICMB_130_YOUTUBE_BATCH_11_MOVIE_CANDIDATE_ALLOCATION.csv"
)

if not os.path.exists(INPUT):
    raise FileNotFoundError(INPUT)

if os.path.exists(OUTPUT):
    raise FileExistsError("Batch 11 allocation already exists.")

df = pd.read_csv(INPUT)

# Select only movies that need additional comments.
candidates = df[df["additional_comment_target"] > 0].copy()

# Priority order: uncovered Tamil/Telugu, partial Tamil/Telugu,
# uncovered Hindi, then partial Hindi and other languages.
def priority(row):
    language = row["language"]
    count = row["comment_records"]

    if count == 0 and language in ["Tamil", "Telugu"]:
        return 1
    if 0 < count < 100 and language in ["Tamil", "Telugu"]:
        return 2
    if count == 0 and language == "Hindi":
        return 3
    if 0 < count < 100:
        return 4
    return 5

candidates["planning_priority"] = candidates.apply(priority, axis=1)

candidates = candidates.sort_values(
    ["planning_priority", "year", "rank"],
    ascending=[True, True, True]
).reset_index(drop=True)

candidates.insert(
    0, "proposed_batch_id", "Batch 11"
)

candidates.to_csv(OUTPUT, index=False)

print("=" * 70)
print("BATCH 11 — MOVIE CANDIDATE ALLOCATION")
print("=" * 70)

print("Total candidate movies:", len(candidates))
print("Remaining target:", candidates["additional_comment_target"].sum())

print("\nPriority distribution:")
print(candidates["planning_priority"].value_counts().sort_index())

print("\nLanguage distribution:")
print(candidates.groupby("language").agg(
    movies=("movie_record_id", "count"),
    remaining_target=("additional_comment_target", "sum")
).to_string())

print("\nTop 20 candidate movies:")
print(candidates[
    [
        "movie_record_id", "movie_title", "year",
        "language", "comment_records",
        "additional_comment_target", "planning_priority"
    ]
].head(20).to_string(index=False))

print("\nSaved:", OUTPUT)
print("No comments or previous reports modified.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

FILE = os.path.join(
    DIR,
    "ICMB_130_YOUTUBE_BATCH_11_MOVIE_CANDIDATE_ALLOCATION.csv"
)

df = pd.read_csv(FILE)

hindi = df[
    (df["language"] == "Hindi") &
    (df["comment_records"] == 0)
].copy()

hindi = hindi.sort_values(
    ["year", "rank"]
)

print("=" * 75)
print("BATCH 11 — UNCOVERED HINDI MOVIE CANDIDATES")
print("=" * 75)

print("Total uncovered Hindi movies:", len(hindi))

print(
    hindi[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "rank",
            "additional_comment_target"
        ]
    ].to_string(index=False)
)

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

INPUT = os.path.join(
    DIR,
    "ICMB_130_YOUTUBE_BATCH_11_MOVIE_CANDIDATE_ALLOCATION.csv"
)

OUTPUT = os.path.join(
    DIR,
    "ICMB_130_YOUTUBE_BATCH_11_MOVIE_ALLOCATION_PROPOSED.csv"
)

if not os.path.exists(INPUT):
    raise FileNotFoundError(INPUT)

if os.path.exists(OUTPUT):
    raise FileExistsError("Proposed allocation already exists.")

candidates = pd.read_csv(INPUT)

selected_ids = [
    "2008_3", "2016_3", "2021_1", "2021_3", "2021_5",
    "2010_2", "2010_3", "2010_4", "2010_5", "2011_2"
]

batch11 = candidates[
    candidates["movie_record_id"].isin(selected_ids)
].copy()

assert len(batch11) == 10
assert batch11["movie_record_id"].nunique() == 10
assert batch11["additional_comment_target"].sum() == 1000

batch11.insert(0, "batch_id", "Batch 11")
batch11["allocation_status"] = "Proposed — Pending Discovery"

batch11.to_csv(OUTPUT, index=False)

print("=" * 70)
print("BATCH 11 — PROPOSED MOVIE ALLOCATION")
print("=" * 70)

print(batch11[
    ["movie_record_id", "movie_title", "year",
     "language", "additional_comment_target"]
].to_string(index=False))

print("\nMovies:", len(batch11))
print("Provisional comment target:",
      batch11["additional_comment_target"].sum())
print("Saved:", OUTPUT)
print("Original candidate allocation preserved.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

ALLOCATION_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_11_MOVIE_ALLOCATION_PROPOSED.csv"
)

MASTER_FILE = os.path.join(
    ROOT, "01_data", "processed",
    "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_11_DISCOVERY_PREPARATION.csv"
)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError("Discovery preparation file already exists.")

allocation = pd.read_csv(ALLOCATION_FILE)
master = pd.read_csv(MASTER_FILE)

master_columns = [
    "movie_record_id", "tmdb_id",
    "original_title", "original_language"
]

discovery = allocation.merge(
    master[master_columns],
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

assert len(discovery) == 10
assert discovery["tmdb_id"].notna().all()
assert discovery["movie_record_id"].nunique() == 10

discovery["search_query"] = (
    discovery["movie_title"].astype(str)
    + " "
    + discovery["year"].astype(str)
    + " movie review reaction"
)

discovery["discovery_status"] = "Pending"

discovery.to_csv(OUTPUT_FILE, index=False)

print("=" * 70)
print("BATCH 11 — DISCOVERY PREPARATION")
print("=" * 70)

print(discovery[
    [
        "movie_record_id", "movie_title", "year",
        "language", "tmdb_id", "search_query",
        "discovery_status"
    ]
].to_string(index=False))

print("\nTMDB IDs available:", discovery["tmdb_id"].notna().sum())
print("Movies prepared:", len(discovery))
print("Saved:", OUTPUT_FILE)

In [ ]:
import os
import pandas as pd
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# ---------------------------------------------------------
# PATHS
# ---------------------------------------------------------
ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

INPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_11_DISCOVERY_PREPARATION.csv"
)

DISCOVERY_FILE = os.path.join(
    RAW_DIR,
    "ICMB_130_YOUTUBE_DISCOVERY_BATCH_11.csv"
)

LOG_FILE = os.path.join(
    RAW_DIR,
    "ICMB_130_YOUTUBE_DISCOVERY_LOG_BATCH_11.csv"
)

for path in [DISCOVERY_FILE, LOG_FILE]:
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists. No overwrite:\n{path}"
        )

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(INPUT_FILE)

# ---------------------------------------------------------
# LOAD API SECRET
# ---------------------------------------------------------
api_key = userdata.get("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError(
        "YOUTUBE_API_KEY was not found in Colab Secrets."
    )

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

movies = pd.read_csv(INPUT_FILE)

assert len(movies) == 10
assert movies["tmdb_id"].notna().all()

# ---------------------------------------------------------
# LANGUAGE-SPECIFIC SEARCH TERMS
# ---------------------------------------------------------
language_terms = {
    "Hindi": "Hindi",
    "Tamil": "Tamil",
    "Telugu": "Telugu",
    "Kannada": "Kannada"
}

discovery_records = []
log_records = []

print("=" * 75)
print("BATCH 11 — YOUTUBE VIDEO DISCOVERY")
print("=" * 75)

# ---------------------------------------------------------
# SEARCH — ONE REQUEST PER MOVIE
# ---------------------------------------------------------
for index, movie in movies.iterrows():

    movie_id = movie["movie_record_id"]
    title = movie["movie_title"]
    year = int(movie["year"])
    language = movie["language"]

    query = (
        f'"{title}" {year} '
        f'{language_terms.get(language, "")} '
        f'movie review reaction'
    ).strip()

    print(
        f"\n[{index + 1}/10] Searching: {title} ({year})"
    )

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=10,
            order="relevance",
            regionCode="IN"
        ).execute()

        items = response.get("items", [])

        for item in items:
            snippet = item.get("snippet", {})
            video_id = item.get("id", {}).get("videoId")

            if not video_id:
                continue

            discovery_records.append({
                "movie_record_id": movie_id,
                "movie_title": title,
                "movie_year": year,
                "industry": language,
                "search_query": query,
                "video_id": video_id,
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "discovery_source": "YouTube Data API v3"
            })

        log_records.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "year": year,
            "language": language,
            "search_query": query,
            "search_status": "Success",
            "candidates_returned": len(items),
            "error_type": None,
            "discovery_timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print("Candidates returned:", len(items))

    except HttpError as error:

        # Do not print the raw API exception or request URL.
        status = getattr(error.resp, "status", None)

        log_records.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "year": year,
            "language": language,
            "search_query": query,
            "search_status": "Failed",
            "candidates_returned": 0,
            "error_type": f"HTTP error {status}",
            "discovery_timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print("Search failed. HTTP status:", status)

    except Exception as error:

        log_records.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "year": year,
            "language": language,
            "search_query": query,
            "search_status": "Failed",
            "candidates_returned": 0,
            "error_type": type(error).__name__,
            "discovery_timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print("Search failed:", type(error).__name__)

# ---------------------------------------------------------
# SAVE DISCOVERY AND LOG
# ---------------------------------------------------------
discovery_df = pd.DataFrame(discovery_records)
log_df = pd.DataFrame(log_records)

discovery_df.to_csv(DISCOVERY_FILE, index=False)
log_df.to_csv(LOG_FILE, index=False)

# ---------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------
print("\n" + "=" * 75)
print("BATCH 11 — DISCOVERY RESULTS")
print("=" * 75)

print("Movies searched:", len(log_df))
print("Successful searches:",
      (log_df["search_status"] == "Success").sum())
print("Failed searches:",
      (log_df["search_status"] == "Failed").sum())
print("Total candidates:", len(discovery_df))
print("Unique video IDs:", discovery_df["video_id"].nunique())
print("Movies with candidates:",
      discovery_df["movie_record_id"].nunique())

print("\nCandidates by movie:")
if not discovery_df.empty:
    print(
        discovery_df.groupby(
            ["movie_record_id", "movie_title"]
        )["video_id"].nunique().to_string()
    )

print("\nDiscovery file:", DISCOVERY_FILE)
print("Discovery log:", LOG_FILE)

print("\nAPI key was not displayed.")
print("No comments were acquired in this step.")

In [ ]:
import os
import pandas as pd
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)
PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

DISCOVERY_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_DISCOVERY_BATCH_11.csv"
)

OUTPUT_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_BATCH_11.csv"
)

LOG_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_LOG_BATCH_11.csv"
)

if os.path.exists(OUTPUT_FILE) or os.path.exists(LOG_FILE):
    raise FileExistsError("Metadata output already exists. No overwrite.")

discovery = pd.read_csv(DISCOVERY_FILE)

video_ids = discovery["video_id"].dropna().astype(str).unique().tolist()

assert len(video_ids) == 100

api_key = userdata.get("YOUTUBE_API_KEY")
if not api_key:
    raise ValueError("YOUTUBE_API_KEY missing from Colab Secrets.")

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

metadata_records = []
metadata_log = []

print("=" * 70)
print("BATCH 11 — VIDEO METADATA RETRIEVAL")
print("=" * 70)

for start in range(0, len(video_ids), 50):

    chunk = video_ids[start:start + 50]

    try:
        response = youtube.videos().list(
            part="snippet,contentDetails,statistics",
            id=",".join(chunk),
            maxResults=50
        ).execute()

        items = response.get("items", [])

        for item in items:
            snippet = item.get("snippet", {})
            statistics = item.get("statistics", {})
            content = item.get("contentDetails", {})

            metadata_records.append({
                "video_id": item.get("id"),
                "metadata_title": snippet.get("title"),
                "description": snippet.get("description"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "duration": content.get("duration"),
                "view_count": statistics.get("viewCount"),
                "like_count": statistics.get("likeCount"),
                "comment_count": statistics.get("commentCount"),
                "metadata_retrieved_at_utc": datetime.now(
                    timezone.utc
                ).isoformat()
            })

        metadata_log.append({
            "chunk_start": start,
            "requested_ids": len(chunk),
            "returned_records": len(items),
            "status": "Success",
            "error_type": None
        })

        print(
            f"Chunk {start // 50 + 1}: "
            f"{len(items)}/{len(chunk)} metadata records"
        )

    except HttpError as error:

        status = getattr(error.resp, "status", None)

        metadata_log.append({
            "chunk_start": start,
            "requested_ids": len(chunk),
            "returned_records": 0,
            "status": "Failed",
            "error_type": f"HTTP error {status}"
        })

        print("Chunk failed. HTTP status:", status)

metadata_df = pd.DataFrame(metadata_records)
log_df = pd.DataFrame(metadata_log)

# Preserve all discovery candidates, including unavailable videos.
final_metadata = discovery.merge(
    metadata_df,
    on="video_id",
    how="left",
    suffixes=("_discovery", "_metadata"),
    validate="one_to_one"
)

final_metadata["metadata_status"] = final_metadata[
    "metadata_title"
].notna().map({
    True: "Retrieved",
    False: "Unavailable or Missing"
})

final_metadata.to_csv(OUTPUT_FILE, index=False)
log_df.to_csv(LOG_FILE, index=False)

print("\n" + "=" * 70)
print("METADATA RETRIEVAL SUMMARY")
print("=" * 70)

print("Discovery candidates:", len(discovery))
print("Unique discovered video IDs:", len(video_ids))
print("Metadata retrieved:", len(metadata_df))
print("Metadata missing:", final_metadata["metadata_title"].isna().sum())
print("Metadata API errors:", (log_df["status"] == "Failed").sum())

print("\nMetadata status:")
print(final_metadata["metadata_status"].value_counts().to_string())

print("\nSaved:", OUTPUT_FILE)
print("Log:", LOG_FILE)
print("API key was not displayed.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

FILE = os.path.join(
    ROOT,
    "01_data", "raw", "audience_sentiment", "youtube",
    "ICMB_130_YOUTUBE_METADATA_BATCH_11.csv"
)

df = pd.read_csv(FILE)

print("=" * 70)
print("BATCH 11 — METADATA SCHEMA INSPECTION")
print("=" * 70)

print("Shape:", df.shape)

print("\nActual columns:")
for col in df.columns:
    print("-", col)

print("\nDescription-related columns:")
description_cols = [
    col for col in df.columns
    if "description" in col.lower()
]
print(description_cols)

print("\nTitle-related columns:")
title_cols = [
    col for col in df.columns
    if "title" in col.lower()
]
print(title_cols)

print("\nSample metadata:")
display(df[
    [c for c in title_cols + description_cols if c in df.columns]
].head(3))

print("\nNo files modified.")

In [ ]:
import os
import re
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

INPUT_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_BATCH_11.csv"
)

OUTPUT_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_11.csv"
)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(INPUT_FILE)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError("Screening file already exists.")

df = pd.read_csv(INPUT_FILE)

# ---------------------------------------------------------
# TEXT PREPARATION
# ---------------------------------------------------------
def safe_text(value):
    return "" if pd.isna(value) else str(value).lower()

df["screening_text"] = (
    df["metadata_title"].apply(safe_text)
    + " "
    + df["description_metadata"].apply(safe_text)
)

# ---------------------------------------------------------
# DURATION CONVERSION
# ---------------------------------------------------------
def duration_seconds(value):
    if pd.isna(value):
        return None

    match = re.fullmatch(
        r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?",
        str(value)
    )

    if not match:
        return None

    h, m, s = [
        int(x) if x else 0
        for x in match.groups()
    ]

    return h * 3600 + m * 60 + s

df["duration_seconds"] = df["duration"].apply(duration_seconds)

# ---------------------------------------------------------
# KEYWORD PATTERNS
# ---------------------------------------------------------
patterns = {
    "review": r"\breview\b|\banalysis\b|\bexplained\b|\bretrospective\b|\bfilm discussion\b",
    "reaction": r"\breaction\b|\breacts\b|\breacting\b|\bwatch party\b|\bfirst time watching\b",
    "trailer": r"\btrailer\b|\bteaser\b|\bpromo\b|\bpreview\b",
    "song": r"\bsong\b|\blyrics\b|\bvideo song\b|\bmusic video\b",
    "clip": r"\bscene\b|\bclip\b|\bshorts\b|\bshort film\b",
    "full_movie": r"\bfull movie\b|\bcomplete movie\b|\bentire movie\b"
}

for key, pattern in patterns.items():
    df[f"has_{key}"] = df["screening_text"].str.contains(
        pattern, case=False, regex=True, na=False
    )

# ---------------------------------------------------------
# SOURCE CLASSIFICATION
# ---------------------------------------------------------
def classify_source(row):

    if row["has_song"]:
        return "Song / Music Video"
    if row["has_trailer"]:
        return "Trailer / Teaser"
    if row["has_clip"]:
        return "Clip / Scene"
    if row["has_full_movie"]:
        return "Full Movie"
    if row["has_review"] and row["has_reaction"]:
        return "Review + Reaction"
    if row["has_review"]:
        return "Movie Review / Analysis"
    if row["has_reaction"]:
        return "Movie Reaction"

    return "Unclassified"

df["automated_source_type"] = df.apply(
    classify_source, axis=1
)

# ---------------------------------------------------------
# HEURISTIC SCREENING SCORE
# ---------------------------------------------------------
def calculate_score(row):

    score = 0
    source_type = row["automated_source_type"]

    if row["has_review"]:
        score += 40

    if row["has_reaction"]:
        score += 40

    duration = row["duration_seconds"]

    if pd.notna(duration):
        if duration >= 600:
            score += 20
        elif duration >= 300:
            score += 10

    if source_type in [
        "Song / Music Video",
        "Trailer / Teaser",
        "Clip / Scene",
        "Full Movie"
    ]:
        score = min(score, 20)

    return score

df["screening_score"] = df.apply(
    calculate_score, axis=1
)

# ---------------------------------------------------------
# AUTOMATED DECISION
# ---------------------------------------------------------
def screening_decision(row):

    source_type = row["automated_source_type"]

    if source_type in [
        "Song / Music Video",
        "Trailer / Teaser",
        "Clip / Scene",
        "Full Movie"
    ]:
        return "Exclude"

    if source_type in [
        "Movie Review / Analysis",
        "Movie Reaction",
        "Review + Reaction"
    ]:
        return (
            "Shortlist"
            if row["screening_score"] >= 40
            else "Hold"
        )

    return "Hold"

df["automated_decision"] = df.apply(
    screening_decision, axis=1
)

df["screening_reason"] = df["automated_source_type"].map({
    "Movie Review / Analysis":
        "Review keywords detected; content not verified.",
    "Movie Reaction":
        "Reaction keywords detected; content not verified.",
    "Review + Reaction":
        "Review and reaction keywords detected; content not verified.",
    "Song / Music Video":
        "Song/music indicators detected.",
    "Trailer / Teaser":
        "Trailer/teaser indicators detected.",
    "Clip / Scene":
        "Scene/clip indicators detected.",
    "Full Movie":
        "Full-movie indicators detected.",
    "Unclassified":
        "Insufficient metadata evidence; requires review."
})

df["metadata_review_status"] = "Pending"
df["content_review_status"] = "Not Reviewed"

# ---------------------------------------------------------
# SAVE AND VALIDATE
# ---------------------------------------------------------
assert len(df) == 100
assert df["video_id"].nunique() == 100

df.to_csv(OUTPUT_FILE, index=False)

print("=" * 75)
print("BATCH 11 — AUTOMATED SCREENING RESULTS")
print("=" * 75)

print("Total candidates:", len(df))
print("Unique videos:", df["video_id"].nunique())

print("\nAutomated decisions:")
print(df["automated_decision"].value_counts().to_string())

print("\nSource-type distribution:")
print(df["automated_source_type"].value_counts().to_string())

print("\nResults by movie:")
print(
    df.groupby("movie_title").agg(
        candidates=("video_id", "nunique"),
        shortlist=("automated_decision",
                   lambda x: (x == "Shortlist").sum()),
        hold=("automated_decision",
              lambda x: (x == "Hold").sum()),
        exclude=("automated_decision",
                 lambda x: (x == "Exclude").sum())
    ).to_string()
)

print("\nSaved:", OUTPUT_FILE)
print("Screening is metadata-based, not content verification.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

INPUT_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_11.csv"
)

OUTPUT_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_CANDIDATE_REVIEW_BATCH_11.csv"
)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(INPUT_FILE)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        "Review sheet already exists. Existing file will not be overwritten."
    )

df = pd.read_csv(INPUT_FILE)

# Prepare direct video links
df["youtube_url"] = (
    "https://www.youtube.com/watch?v=" + df["video_id"].astype(str)
)

# Use metadata description, falling back to discovery description
df["description_excerpt"] = (
    df["description_metadata"]
    .fillna(df["description_discovery"])
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.slice(0, 500)
)

# Keep metadata and content verification separate
df["suggested_decision"] = df["automated_decision"]
df["metadata_review_status"] = "Pending"
df["content_review_status"] = "Not Reviewed"
df["manual_review_notes"] = ""

# Sort: shortlist first, then hold, then exclude
priority = {
    "Shortlist": 1,
    "Hold": 2,
    "Exclude": 3
}

df["review_priority"] = df["automated_decision"].map(priority)

df = df.sort_values(
    ["review_priority", "movie_title", "screening_score"],
    ascending=[True, True, False]
).drop(columns=["review_priority"])

# Select useful review fields while retaining all 100 candidates
review_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "video_id",
    "video_title",
    "metadata_title",
    "channel_title_metadata",
    "duration",
    "view_count",
    "like_count",
    "comment_count",
    "youtube_url",
    "description_excerpt",
    "automated_source_type",
    "screening_score",
    "automated_decision",
    "suggested_decision",
    "screening_reason",
    "metadata_review_status",
    "content_review_status",
    "manual_review_notes"
]

review_df = df[review_columns].copy()

assert len(review_df) == 100
assert review_df["video_id"].nunique() == 100

review_df.to_csv(OUTPUT_FILE, index=False)

print("=" * 75)
print("BATCH 11 — CONSOLIDATED CANDIDATE REVIEW SHEET")
print("=" * 75)

print("Total candidates:", len(review_df))
print("Unique videos:", review_df["video_id"].nunique())
print("\nDecision distribution:")
print(review_df["suggested_decision"].value_counts().to_string())

print("\nCandidates by movie:")
print(
    review_df.groupby(
        ["movie_title", "suggested_decision"]
    ).size().unstack(fill_value=0).to_string()
)

print("\nReview sheet columns:")
print(review_df.columns.tolist())

print("\nSaved successfully:")
print(OUTPUT_FILE)

display(
    review_df[
        review_df["suggested_decision"].isin(["Shortlist", "Hold"])
    ][[
        "movie_title",
        "video_title",
        "channel_title_metadata",
        "duration",
        "automated_source_type",
        "screening_score",
        "suggested_decision",
        "youtube_url"
    ]].head(20)
)

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

FILE = os.path.join(
    ROOT,
    "01_data", "raw", "audience_sentiment", "youtube",
    "ICMB_130_YOUTUBE_CANDIDATE_REVIEW_BATCH_11.csv"
)

df = pd.read_csv(FILE)

review = df[
    df["suggested_decision"].isin(["Shortlist", "Hold"])
].copy()

review["duration"] = review["duration"].fillna("Unknown")
review["description_excerpt"] = (
    review["description_excerpt"]
    .fillna("")
    .astype(str)
    .str.slice(0, 350)
)

review = review.sort_values(
    ["movie_title", "suggested_decision", "screening_score"],
    ascending=[True, True, False]
)

for movie, group in review.groupby("movie_title", sort=False):

    print("\n" + "=" * 100)
    print(f"{movie} | Candidates: {len(group)}")
    print("=" * 100)

    for _, row in group.iterrows():
        print(f"\nDecision: {row['suggested_decision']}")
        print(f"Title: {row['video_title']}")
        print(f"Channel: {row['channel_title_metadata']}")
        print(f"Duration: {row['duration']}")
        print(f"Type: {row['automated_source_type']}")
        print(f"Score: {row['screening_score']}")
        print(f"Description: {row['description_excerpt']}")
        print(f"Link: {row['youtube_url']}")

print("\nTotal shortlisted/held candidates:", len(review))
print("No files modified.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

REVIEW_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_CANDIDATE_REVIEW_BATCH_11.csv"
)

ALLOCATION_FILE = os.path.join(
    PROCESSED_DIR, "ICMB_130_YOUTUBE_BATCH_11_MOVIE_ALLOCATION_PROPOSED.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_11.csv"
)

if not os.path.exists(REVIEW_FILE):
    raise FileNotFoundError(REVIEW_FILE)

if not os.path.exists(ALLOCATION_FILE):
    raise FileNotFoundError(ALLOCATION_FILE)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        "Batch 11 approved manifest already exists. No overwrite performed."
    )

review = pd.read_csv(REVIEW_FILE)
allocation = pd.read_csv(ALLOCATION_FILE)

# ---------------------------------------------------------
# USER-APPROVED VIDEO SELECTION
# ---------------------------------------------------------
approved = {
    "OB2IMYPQ14Y": "Movie Review / Analysis",

    "gBkh2C7YDhM": "Review + Reaction",

    "TpSylBv5K-E": "Movie Reaction",
    "Iqj5B6f8uX8": "Movie Reaction",

    "Ip3Ol6vBDFg": "Movie Reaction",
    "YaTDKtmK0u4": "Movie Reaction",
    "crwikGe3fRI": "Movie Reaction",

    "amBGPnQwNqY": "Movie Review / Analysis",
    "rpNpYPcjQmE": "Movie Reaction",

    "cOTqLIjNuac": "Movie Reaction",
    "iAsLnN5LDzM": "Movie Review / Analysis",

    "lCXkJzghgqc": "Movie Reaction",
    "kmW1546r4Uc": "Movie Reaction"
}

# ---------------------------------------------------------
# VERIFY ALL VIDEO IDS
# ---------------------------------------------------------
if review["video_id"].astype(str).duplicated().any():
    raise ValueError("Duplicate video IDs found in review sheet.")

selected = review[
    review["video_id"].astype(str).isin(approved.keys())
].copy()

missing_ids = set(approved.keys()) - set(selected["video_id"].astype(str))

if missing_ids:
    raise ValueError(f"Approved video IDs missing from review sheet: {missing_ids}")

selected["manual_source_type"] = (
    selected["video_id"].astype(str).map(approved)
)

# ---------------------------------------------------------
# RESTORE MOVIE LANGUAGE AND ALLOCATION METADATA
# ---------------------------------------------------------
allocation_columns = allocation.columns.tolist()

year_col = next(
    (c for c in ["movie_year", "year"] if c in allocation_columns),
    None
)

if year_col is None or "language" not in allocation_columns:
    raise ValueError(
        f"Required year/language fields not found. Allocation columns: {allocation_columns}"
    )

allocation_lookup = allocation[
    ["movie_record_id", "movie_title", year_col, "language"]
].drop_duplicates("movie_record_id")

selected = selected.merge(
    allocation_lookup,
    on=["movie_record_id", "movie_title"],
    how="left",
    validate="many_to_one"
)

selected["movie_year"] = selected[year_col]

if selected["movie_year"].isna().any():
    raise ValueError("Missing movie years in selected sources.")

if selected["language"].isna().any():
    raise ValueError("Missing movie languages in selected sources.")

# ---------------------------------------------------------
# BUILD MANIFEST
# ---------------------------------------------------------
manifest = pd.DataFrame({
    "movie_record_id": selected["movie_record_id"],
    "movie_title": selected["movie_title"],
    "movie_year": selected["movie_year"],
    "language": selected["language"],
    "video_id": selected["video_id"].astype(str),
    "video_title": selected["video_title"],
    "channel_title": selected["channel_title_metadata"],
    "duration": selected["duration"],
    "view_count": selected["view_count"],
    "comment_count": selected["comment_count"],
    "manual_source_type": selected["manual_source_type"],
    "manual_verification_status": "Metadata Reviewed",
    "manual_review_notes": (
        "Source selected and approved by user based on candidate metadata. "
        "Video content not independently reviewed."
    ),
    "content_review_status": "Not Reviewed",
    "acquisition_status": "Pending",
    "target_comments": 100
})

# ---------------------------------------------------------
# FINAL VALIDATION AND SAVE
# ---------------------------------------------------------
assert len(manifest) == 13
assert manifest["video_id"].nunique() == 13
assert manifest["movie_record_id"].nunique() == 7
assert manifest["target_comments"].sum() == 1300
assert manifest["movie_year"].notna().all()
assert manifest["language"].notna().all()

manifest.to_csv(OUTPUT_FILE, index=False)

print("=" * 75)
print("BATCH 11 — APPROVED ACQUISITION MANIFEST")
print("=" * 75)

print("Approved sources:", len(manifest))
print("Unique movies:", manifest["movie_record_id"].nunique())
print("Unique videos:", manifest["video_id"].nunique())
print("Planned comment target:", manifest["target_comments"].sum())

print("\nSources per movie:")
print(
    manifest.groupby("movie_title").agg(
        sources=("video_id", "nunique"),
        planned_comments=("target_comments", "sum")
    ).to_string()
)

print("\nLanguage distribution:")
print(manifest.groupby("language")["video_id"].nunique().to_string())

print("\nVerification status:")
print(manifest["manual_verification_status"].value_counts().to_string())

print("\nContent review status:")
print(manifest["content_review_status"].value_counts().to_string())

print("\nSaved:", OUTPUT_FILE)

display(manifest)

In [ ]:
import os
import re
import time
import pandas as pd
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# =========================================================
# 1. PROJECT PATHS
# =========================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

COMMENTS_DIR = os.path.join(RAW_DIR, "comments")

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

os.makedirs(COMMENTS_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

MANIFEST_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_11.csv"
)

RAW_FILE = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_BATCH_11.csv"
)

LOG_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_COMMENT_ACQUISITION_LOG_BATCH_11.csv"
)

CHECKPOINT_FILE = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_BATCH_11_CHECKPOINT.csv"
)

for path in [RAW_FILE, LOG_FILE, CHECKPOINT_FILE]:
    if os.path.exists(path):
        raise FileExistsError(
            f"Existing file detected; refusing to overwrite: {path}"
        )

# =========================================================
# 2. LOAD APPROVED MANIFEST
# =========================================================

manifest = pd.read_csv(MANIFEST_FILE)

required_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "video_title",
    "channel_title",
    "manual_source_type",
    "acquisition_status",
    "target_comments"
]

missing = [
    col for col in required_columns
    if col not in manifest.columns
]

if missing:
    raise ValueError(f"Manifest missing columns: {missing}")

if len(manifest) != 13:
    raise ValueError(f"Expected 13 approved sources, found {len(manifest)}")

if manifest["video_id"].nunique() != 13:
    raise ValueError("Video ID uniqueness validation failed.")

if not (manifest["acquisition_status"] == "Pending").all():
    raise ValueError("Manifest contains sources not marked Pending.")

# =========================================================
# 3. LOAD API KEY FROM COLAB SECRET
# =========================================================

API_KEY = userdata.get("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY was not found in Colab Secrets.")

youtube = build(
    "youtube",
    "v3",
    developerKey=API_KEY,
    cache_discovery=False
)

print("=" * 75)
print("BATCH 11 — YOUTUBE COMMENT ACQUISITION")
print("=" * 75)
print("Approved videos:", len(manifest))
print("Planned maximum comments:", manifest["target_comments"].sum())

# =========================================================
# 4. HELPER FUNCTIONS
# =========================================================

def clean_api_error(error):
    """Return a sanitized error without exposing API credentials."""
    message = str(error)
    message = re.sub(
        r"key=[^&\s]+",
        "key=REDACTED",
        message,
        flags=re.IGNORECASE
    )
    message = re.sub(
        r"https?://\S+",
        "[API URL REDACTED]",
        message
    )
    return message[:300]


def retrieve_comments(video_id, target):
    records = []
    page_token = None
    request_count = 0
    status = "Completed"
    error_message = ""

    while len(records) < target:
        try:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, target - len(records)),
                order="time",
                textFormat="plainText",
                pageToken=page_token
            )

            response = request.execute()
            request_count += 1

            for item in response.get("items", []):
                snippet = item["snippet"]["topLevelComment"]
                comment = snippet["snippet"]

                records.append({
                    "comment_id": snippet.get("id"),
                    "comment_text": comment.get("textDisplay"),
                    "author_name": comment.get("authorDisplayName"),
                    "like_count": comment.get("likeCount", 0),
                    "published_at": comment.get("publishedAt"),
                    "updated_at": comment.get("updatedAt")
                })

                if len(records) >= target:
                    break

            page_token = response.get("nextPageToken")

            if not page_token:
                break

        except HttpError as error:
            status = "Partial" if records else "Failed"
            error_message = clean_api_error(error)
            break

        except Exception as error:
            status = "Partial" if records else "Failed"
            error_message = clean_api_error(error)
            break

    if status != "Failed" and len(records) < target:
        status = "Partial"

    return records, status, request_count, error_message


# =========================================================
# 5. ACQUIRE COMMENTS
# =========================================================

all_comments = []
acquisition_logs = []

for index, row in manifest.iterrows():

    video_id = str(row["video_id"])
    target = int(row["target_comments"])

    print("\n" + "-" * 70)
    print(f"Movie: {row['movie_title']}")
    print(f"Video: {video_id}")
    print(f"Target: {target}")

    comments, status, requests, error = retrieve_comments(
        video_id, target
    )

    retrieved_at = datetime.now(timezone.utc).isoformat()

    for position, comment in enumerate(comments, start=1):

        all_comments.append({
            "movie_record_id": row["movie_record_id"],
            "movie_title": row["movie_title"],
            "movie_year": row["movie_year"],
            "language": row["language"],
            "video_id": video_id,
            "video_title": row["video_title"],
            "channel_title": row["channel_title"],
            "comment_id": comment["comment_id"],
            "comment_text": comment["comment_text"],
            "author_name": comment["author_name"],
            "like_count": comment["like_count"],
            "published_at": comment["published_at"],
            "updated_at": comment["updated_at"],
            "comment_position": position,
            "acquisition_batch": "Batch 11",
            "retrieval_order": "time",
            "retrieved_at_utc": retrieved_at
        })

    acquisition_logs.append({
        "movie_record_id": row["movie_record_id"],
        "movie_title": row["movie_title"],
        "movie_year": row["movie_year"],
        "video_id": video_id,
        "channel_title": row["channel_title"],
        "target_comments": target,
        "retrieved_comments": len(comments),
        "request_count": requests,
        "acquisition_status": status,
        "error_message": error,
        "retrieved_at_utc": retrieved_at
    })

    print("Retrieved:", len(comments))
    print("Status:", status)
    print("API requests:", requests)

    if error:
        print("Sanitized error:", error)

    time.sleep(0.2)

# =========================================================
# 6. BUILD DATAFRAMES
# =========================================================

comments_df = pd.DataFrame(all_comments)
log_df = pd.DataFrame(acquisition_logs)

# =========================================================
# 7. VALIDATION
# =========================================================

if not comments_df.empty:

    duplicate_ids = comments_df["comment_id"].duplicated().sum()

    print("\n" + "=" * 75)
    print("BATCH 11 — ACQUISITION VALIDATION")
    print("=" * 75)

    print("Total comments:", len(comments_df))
    print("Unique comment IDs:", comments_df["comment_id"].nunique())
    print("Duplicate comment IDs:", duplicate_ids)
    print("Unique videos:", comments_df["video_id"].nunique())
    print("Unique movies:", comments_df["movie_record_id"].nunique())
    print("Empty comments:", comments_df["comment_text"].isna().sum())

    if duplicate_ids:
        raise ValueError(
            "Duplicate comment IDs detected. Review before saving."
        )

else:
    print("No comments were retrieved.")

# =========================================================
# 8. SAVE RAW COMMENTS, LOG AND CHECKPOINT
# =========================================================

comments_df.to_csv(RAW_FILE, index=False)
log_df.to_csv(LOG_FILE, index=False)
log_df.to_csv(CHECKPOINT_FILE, index=False)

# =========================================================
# 9. FINAL SUMMARY
# =========================================================

print("\n" + "=" * 75)
print("BATCH 11 — FINAL ACQUISITION SUMMARY")
print("=" * 75)

print("Sources processed:", len(log_df))
print("\nAcquisition status:")
print(log_df["acquisition_status"].value_counts().to_string())

print("\nComments by movie:")
if not comments_df.empty:
    print(
        comments_df.groupby("movie_title").agg(
            comments=("comment_id", "count"),
            unique_ids=("comment_id", "nunique"),
            videos=("video_id", "nunique")
        ).to_string()
    )

print("\nTotal comments retrieved:", len(comments_df))
print("Total target:", int(manifest["target_comments"].sum()))

print("\nSaved files:")
print("Raw comments:", RAW_FILE)
print("Acquisition log:", LOG_FILE)
print("Checkpoint:", CHECKPOINT_FILE)

print("\nNote: Comments are recent top-level comments (order=time).")
print("Replies are not included. This is not a random audience sample.")
print("API key was not printed or saved in the output files.")

In [ ]:
import os
import re
import pandas as pd
import unicodedata
from collections import Counter

# =========================================================
# 1. PATHS
# =========================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

COMMENTS_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment",
    "youtube", "comments"
)

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

INPUT_FILE = os.path.join(
    COMMENTS_DIR, "ICMB_130_YOUTUBE_COMMENT_BATCH_11.csv"
)

OUTPUTS = {
    "overall": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_COMMENT_QUALITY_PROFILE_BATCH_11.csv"
    ),
    "movie": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_COMMENT_MOVIE_QUALITY_BATCH_11.csv"
    ),
    "video": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_COMMENT_VIDEO_QUALITY_BATCH_11.csv"
    ),
    "duplicate": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_BATCH_11.csv"
    )
}

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(INPUT_FILE)

existing = [p for p in OUTPUTS.values() if os.path.exists(p)]

if existing:
    raise FileExistsError(
        "Existing report files detected; refusing to overwrite:\n"
        + "\n".join(existing)
    )

# =========================================================
# 2. LOAD AND VALIDATE RAW DATA
# =========================================================

df = pd.read_csv(INPUT_FILE)

required = [
    "movie_record_id", "movie_title", "movie_year",
    "language", "video_id", "video_title",
    "channel_title", "comment_id", "comment_text",
    "published_at", "like_count"
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

assert len(df) == 1200
assert df["comment_id"].nunique() == 1200
assert df["comment_id"].duplicated().sum() == 0

# =========================================================
# 3. TEXT NORMALIZATION AND BASIC FEATURES
# =========================================================

def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).lower().strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\u0900-\u097f\u0b80-\u0bff\u0c00-\u0c7f\u0c80-\u0cff\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


df["comment_text"] = df["comment_text"].fillna("").astype(str)
df["normalized_comment_text"] = df["comment_text"].apply(normalize_text)

df["word_count"] = df["comment_text"].apply(
    lambda x: len(str(x).split()) if str(x).strip() else 0
)

df["character_count"] = df["comment_text"].str.len()

df["is_empty"] = df["comment_text"].str.strip().eq("")

df["is_very_short"] = (
    (df["word_count"] > 0) & (df["word_count"] <= 3)
)

df["contains_url"] = df["comment_text"].str.contains(
    r"https?://|www\.",
    case=False,
    regex=True,
    na=False
)

df["length_category"] = pd.cut(
    df["word_count"],
    bins=[-1, 3, 10, 30, float("inf")],
    labels=["Very Short", "Short", "Medium", "Long"]
)

# =========================================================
# 4. SCRIPT CLASSIFICATION
# NOTE: Script is NOT language identification.
# =========================================================

def detect_script(text):
    if not str(text).strip():
        return "Empty"

    counts = Counter()

    for char in str(text):
        if not char.isalpha():
            continue

        name = unicodedata.name(char, "")

        if "DEVANAGARI" in name:
            counts["Devanagari"] += 1
        elif "TAMIL" in name:
            counts["Tamil"] += 1
        elif "TELUGU" in name:
            counts["Telugu"] += 1
        elif "KANNADA" in name:
            counts["Kannada"] += 1
        elif "BENGALI" in name:
            counts["Bengali"] += 1
        elif "LATIN" in name:
            counts["Latin"] += 1
        else:
            counts["Other"] += 1

    if not counts:
        return "Other/Emoji/Symbols"

    if len(counts) > 1:
        return "Mixed"

    return counts.most_common(1)[0][0]


df["script_category"] = df["comment_text"].apply(detect_script)

# =========================================================
# 5. OVERALL QUALITY PROFILE
# =========================================================

overall = pd.DataFrame([{
    "batch": "Batch 11",
    "total_comments": len(df),
    "unique_comment_ids": df["comment_id"].nunique(),
    "duplicate_comment_ids": df["comment_id"].duplicated().sum(),
    "unique_videos": df["video_id"].nunique(),
    "unique_movies": df["movie_record_id"].nunique(),
    "empty_comments": df["is_empty"].sum(),
    "nonempty_comments": (~df["is_empty"]).sum(),
    "very_short_comments_le_3_words": df["is_very_short"].sum(),
    "comments_with_urls": df["contains_url"].sum(),
    "average_words": round(df["word_count"].mean(), 2),
    "median_words": round(df["word_count"].median(), 2),
    "average_characters": round(df["character_count"].mean(), 2),
    "median_characters": round(df["character_count"].median(), 2)
}])

# =========================================================
# 6. MOVIE-WISE PROFILE
# =========================================================

movie_quality = (
    df.groupby(
        ["movie_record_id", "movie_title", "movie_year", "language"],
        dropna=False
    )
    .agg(
        comments=("comment_id", "count"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        average_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        average_characters=("character_count", "mean"),
        very_short_comments=("is_very_short", "sum"),
        empty_comments=("is_empty", "sum"),
        comments_with_urls=("contains_url", "sum")
    )
    .reset_index()
)

movie_quality["average_words"] = movie_quality["average_words"].round(2)
movie_quality["average_characters"] = movie_quality["average_characters"].round(2)

# =========================================================
# 7. VIDEO-WISE PROFILE
# =========================================================

video_quality = (
    df.groupby(
        [
            "movie_record_id", "movie_title",
            "video_id", "video_title", "channel_title"
        ],
        dropna=False
    )
    .agg(
        comments=("comment_id", "count"),
        unique_comment_ids=("comment_id", "nunique"),
        average_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        average_characters=("character_count", "mean"),
        very_short_comments=("is_very_short", "sum"),
        comments_with_urls=("contains_url", "sum")
    )
    .reset_index()
)

video_quality["average_words"] = video_quality["average_words"].round(2)
video_quality["average_characters"] = video_quality["average_characters"].round(2)

# =========================================================
# 8. DUPLICATE TEXT AUDIT
# Repeated normalized text is flagged, never deleted.
# =========================================================

nonempty = df[
    df["normalized_comment_text"].str.len() > 0
].copy()

duplicate_groups = (
    nonempty.groupby("normalized_comment_text")
    .agg(
        repeated_records=("comment_id", "count"),
        unique_videos=("video_id", "nunique"),
        unique_movies=("movie_record_id", "nunique"),
        comment_ids=("comment_id", lambda x: " | ".join(x.astype(str))),
        video_ids=("video_id", lambda x: " | ".join(sorted(set(x.astype(str))))),
        movie_titles=("movie_title", lambda x: " | ".join(sorted(set(x.astype(str))))),
        example_comment=("comment_text", "first")
    )
    .reset_index()
)

duplicate_groups = duplicate_groups[
    duplicate_groups["repeated_records"] > 1
].copy()

duplicate_groups["duplicate_scope"] = duplicate_groups.apply(
    lambda row: (
        "Across Videos"
        if row["unique_videos"] > 1
        else "Within Video"
    ),
    axis=1
)

duplicate_groups = duplicate_groups.sort_values(
    "repeated_records", ascending=False
)

# =========================================================
# 9. SAVE REPORTS
# =========================================================

os.makedirs(PROCESSED_DIR, exist_ok=True)

overall.to_csv(OUTPUTS["overall"], index=False)
movie_quality.to_csv(OUTPUTS["movie"], index=False)
video_quality.to_csv(OUTPUTS["video"], index=False)
duplicate_groups.to_csv(OUTPUTS["duplicate"], index=False)

# =========================================================
# 10. DISPLAY RESULTS
# =========================================================

print("=" * 75)
print("BATCH 11 — OVERALL COMMENT QUALITY PROFILE")
print("=" * 75)

display(overall)

print("\nLength distribution:")
print(df["length_category"].value_counts().to_string())

print("\nScript distribution:")
print(df["script_category"].value_counts().to_string())

print("\n" + "=" * 75)
print("MOVIE-WISE QUALITY PROFILE")
print("=" * 75)

display(movie_quality)

print("\n" + "=" * 75)
print("VIDEO-WISE QUALITY PROFILE")
print("=" * 75)

display(video_quality)

print("\n" + "=" * 75)
print("DUPLICATE TEXT AUDIT")
print("=" * 75)

print("Repeated-text groups:", len(duplicate_groups))
print(
    "Records in repeated groups:",
    duplicate_groups["repeated_records"].sum()
)

if not duplicate_groups.empty:
    print("\nDuplicate scope:")
    print(duplicate_groups["duplicate_scope"].value_counts().to_string())
    display(duplicate_groups.head(15))

print("\n" + "=" * 75)
print("FILES SAVED")
print("=" * 75)

for name, path in OUTPUTS.items():
    print(f"{name}: {path}")

print("\nRaw comments file was not modified.")
print("Repeated comments are flagged only; no records were deleted.")
print("Script classification does not identify comment language.")

In [ ]:
import os
import pandas as pd

# =========================================================
# 1. PATHS
# =========================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

COMMENTS_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment",
    "youtube", "comments"
)

OLD_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_10_REPAIRED.csv"
)

BATCH11_FILE = os.path.join(
    COMMENTS_DIR,
    "ICMB_130_YOUTUBE_COMMENT_BATCH_11.csv"
)

if not os.path.exists(OLD_FILE):
    raise FileNotFoundError(OLD_FILE)

if not os.path.exists(BATCH11_FILE):
    raise FileNotFoundError(BATCH11_FILE)

# =========================================================
# 2. LOAD DATA
# =========================================================

old = pd.read_csv(OLD_FILE, low_memory=False)
batch11 = pd.read_csv(BATCH11_FILE, low_memory=False)

print("=" * 75)
print("PILOT–BATCH 10 + BATCH 11 — CONSOLIDATION PREFLIGHT")
print("=" * 75)

print("\nExisting consolidated dataset:")
print("Shape:", old.shape)
print("Unique comment IDs:", old["comment_id"].nunique())
print("Unique videos:", old["video_id"].nunique())
print("Unique movies:", old["movie_record_id"].nunique())

print("\nBatch 11 raw dataset:")
print("Shape:", batch11.shape)
print("Unique comment IDs:", batch11["comment_id"].nunique())
print("Unique videos:", batch11["video_id"].nunique())
print("Unique movies:", batch11["movie_record_id"].nunique())

# =========================================================
# 3. SCHEMA COMPARISON
# =========================================================

old_columns = set(old.columns)
new_columns = set(batch11.columns)

print("\nSchema comparison:")
print("Existing columns:", len(old_columns))
print("Batch 11 columns:", len(new_columns))
print("Columns common to both:", len(old_columns & new_columns))

print("\nColumns missing from Batch 11:")
print(sorted(old_columns - new_columns))

print("\nAdditional Batch 11 columns:")
print(sorted(new_columns - old_columns))

# =========================================================
# 4. COMMENT ID VALIDATION
# =========================================================

old_ids = set(old["comment_id"].dropna().astype(str))
new_ids = set(batch11["comment_id"].dropna().astype(str))

overlap_ids = old_ids & new_ids

print("\nComment ID validation:")
print("Existing unique IDs:", len(old_ids))
print("Batch 11 unique IDs:", len(new_ids))
print("Cross-dataset overlapping IDs:", len(overlap_ids))
print("Expected combined records:", len(old) + len(batch11))

# =========================================================
# 5. BATCH 11 METADATA VALIDATION
# =========================================================

print("\nBatch 11 missing values:")

metadata_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "language",
    "video_id",
    "video_title",
    "channel_title",
    "comment_id",
    "comment_text"
]

for col in metadata_columns:
    if col in batch11.columns:
        print(f"{col}: {batch11[col].isna().sum()}")

print("\nBatch 11 movie-wise records:")
display(
    batch11.groupby(
        ["movie_record_id", "movie_title", "movie_year", "language"],
        dropna=False
    ).agg(
        comments=("comment_id", "count"),
        unique_ids=("comment_id", "nunique"),
        videos=("video_id", "nunique")
    ).reset_index()
)

# =========================================================
# 6. BATCH PROVENANCE CHECK
# =========================================================

if "source_batch" in old.columns:
    print("\nExisting source batch distribution:")
    print(old["source_batch"].value_counts(dropna=False).to_string())

elif "batch_id" in old.columns:
    print("\nExisting batch ID distribution:")
    print(old["batch_id"].value_counts(dropna=False).to_string())

print("\nBatch 11 acquisition batch values:")
if "acquisition_batch" in batch11.columns:
    print(batch11["acquisition_batch"].value_counts(dropna=False).to_string())

# =========================================================
# 7. FINAL PREFLIGHT DECISION
# =========================================================

print("\n" + "=" * 75)
print("PREFLIGHT SUMMARY")
print("=" * 75)

print("Expected total rows:", len(old) + len(batch11))
print("Expected unique IDs:", len(old_ids) + len(new_ids))
print("ID overlap:", len(overlap_ids))
print("Schema compatibility requires mapping:", old_columns != new_columns)

if len(overlap_ids) == 0 and batch11["comment_id"].notna().all():
    print("\nPRELIMINARY RESULT: Comment IDs are compatible for consolidation.")
else:
    print("\nREVIEW REQUIRED: Resolve ID or missing-value issues before consolidation.")

print("\nNo files modified.")

In [ ]:
import os
import re
import pandas as pd

# =========================================================
# 1. PATHS
# =========================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

COMMENTS_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment",
    "youtube", "comments"
)

OLD_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_10_REPAIRED.csv"
)

BATCH11_FILE = os.path.join(
    COMMENTS_DIR,
    "ICMB_130_YOUTUBE_COMMENT_BATCH_11.csv"
)

MANIFEST_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_11.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_11.csv"
)

# =========================================================
# 2. SAFE INPUT VALIDATION
# =========================================================

for path in [OLD_FILE, BATCH11_FILE, MANIFEST_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        "Consolidated Batch 11 output already exists. No overwrite performed."
    )

old = pd.read_csv(OLD_FILE, low_memory=False)
new = pd.read_csv(BATCH11_FILE, low_memory=False)
manifest = pd.read_csv(MANIFEST_FILE)

original_columns = old.columns.tolist()

assert len(original_columns) == 32
assert old["comment_id"].nunique() == len(old)
assert new["comment_id"].nunique() == len(new)
assert not set(old["comment_id"].astype(str)).intersection(
    set(new["comment_id"].astype(str))
)

# =========================================================
# 3. MAP BATCH 11 METADATA
# =========================================================

source_type_map = manifest.set_index(
    "video_id"
)["manual_source_type"].to_dict()

new["source_batch"] = "Batch 11"
new["batch_id"] = "Batch 11"

new["source_type"] = new["video_id"].map(source_type_map)

if new["source_type"].isna().any():
    raise ValueError("Some Batch 11 videos lack source-type metadata.")

new["retrieved_at"] = new["retrieved_at_utc"]
new["acquired_at_utc"] = new["retrieved_at_utc"]
new["acquisition_timestamp_utc"] = new["retrieved_at_utc"]

new["normalized_comment_text"] = (
    new["comment_text"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

new["comment_author"] = new["author_name"]
new["author_display_name"] = new["author_name"]
new["comment_position"] = pd.to_numeric(
    new["comment_position"], errors="coerce"
)
new["acquisition_order"] = new["comment_position"]
new["sampling_order"] = new["comment_position"]
new["sampling_unit"] = "Top-level comment"
new["comment_source"] = "YouTube"
new["series_part"] = pd.NA
new["author_channel_id"] = pd.NA
new["comment_reply_count"] = 0
new["reply_count"] = 0
new["total_reply_count"] = 0

# =========================================================
# 4. ALIGN TO EXISTING 32-COLUMN SCHEMA
# =========================================================

for column in original_columns:
    if column not in new.columns:
        new[column] = pd.NA

new = new[original_columns]

# =========================================================
# 5. CONSOLIDATE
# =========================================================

combined = pd.concat(
    [old, new],
    ignore_index=True
)

# =========================================================
# 6. VALIDATE CONSOLIDATED DATA
# =========================================================

assert len(combined) == 9620
assert combined.columns.tolist() == original_columns
assert combined["comment_id"].nunique() == 9620
assert combined["video_id"].nunique() == 122
assert combined["movie_record_id"].nunique() == 88

batch11_check = combined[
    combined["source_batch"] == "Batch 11"
]

assert len(batch11_check) == 1200
assert batch11_check["comment_id"].nunique() == 1200
assert batch11_check["source_type"].notna().all()

# =========================================================
# 7. SAVE NEW CONSOLIDATED DATASET
# =========================================================

combined.to_csv(OUTPUT_FILE, index=False)

print("=" * 75)
print("PILOT + BATCHES 01–11 — CONSOLIDATION COMPLETE")
print("=" * 75)

print("Existing records:", len(old))
print("Batch 11 records:", len(new))
print("Consolidated records:", len(combined))
print("Unique comment IDs:", combined["comment_id"].nunique())
print("Unique videos:", combined["video_id"].nunique())
print("Unique movies:", combined["movie_record_id"].nunique())
print("Columns:", len(combined.columns))

print("\nBatch 11 source-type distribution:")
print(batch11_check["source_type"].value_counts().to_string())

print("\nBatch 11 movie-wise comments:")
print(
    batch11_check.groupby("movie_title")["comment_id"]
    .count().to_string()
)

print("\nSaved:", OUTPUT_FILE)
print("\nPrevious consolidated dataset remains unchanged.")

In [ ]:
import os
import re
import pandas as pd

# =========================================================
# 1. PATHS
# =========================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

MASTER_FILE = os.path.join(
    ROOT, "01_data", "processed",
    "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

CONSOLIDATED_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_11.csv"
)

OUTPUTS = {
    "coverage": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_11.csv"
    ),
    "duplicate": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_DUPLICATE_TEXT_AUDIT_PILOT_BATCH_01_11.csv"
    ),
    "summary": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_COVERAGE_SUMMARY_PILOT_BATCH_01_11.csv"
    )
}

for path in [MASTER_FILE, CONSOLIDATED_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

existing = [p for p in OUTPUTS.values() if os.path.exists(p)]
if existing:
    raise FileExistsError(
        "Existing outputs detected; refusing to overwrite:\n"
        + "\n".join(existing)
    )

# =========================================================
# 2. LOAD AND VALIDATE
# =========================================================

master = pd.read_csv(MASTER_FILE)
comments = pd.read_csv(CONSOLIDATED_FILE, low_memory=False)

assert len(master) == 130
assert master["movie_record_id"].nunique() == 130
assert len(comments) == 9620
assert comments["comment_id"].nunique() == 9620

required_master = [
    "movie_record_id", "movie_title", "year", "rank", "language"
]
required_comments = [
    "movie_record_id", "movie_title", "comment_id",
    "video_id", "source_batch", "comment_text"
]

for col in required_master:
    if col not in master.columns:
        raise ValueError(f"Master dataset missing: {col}")

for col in required_comments:
    if col not in comments.columns:
        raise ValueError(f"Consolidated dataset missing: {col}")

# =========================================================
# 3. MOVIE-WISE COVERAGE
# =========================================================

movie_counts = (
    comments.groupby("movie_record_id")
    .agg(
        comment_records=("comment_id", "count"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        acquisition_batches=("source_batch", "nunique")
    )
    .reset_index()
)

coverage = master[required_master].merge(
    movie_counts,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

count_columns = [
    "comment_records", "unique_comment_ids",
    "unique_videos", "acquisition_batches"
]

coverage[count_columns] = (
    coverage[count_columns].fillna(0).astype(int)
)

coverage["coverage_status"] = coverage[
    "comment_records"
].apply(lambda x: "Covered" if x > 0 else "Uncovered")

def coverage_category(count):
    if count == 0:
        return "No Comments"
    elif count < 50:
        return "Low"
    elif count < 100:
        return "Moderate"
    return "Higher"

coverage["coverage_category"] = (
    coverage["comment_records"].apply(coverage_category)
)

coverage = coverage[
    required_master + [
        "comment_records",
        "unique_comment_ids",
        "unique_videos",
        "acquisition_batches",
        "coverage_status",
        "coverage_category"
    ]
]

# =========================================================
# 4. FULL-CORPUS DUPLICATE TEXT AUDIT
# =========================================================

def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value).lower().strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(
        r"[^a-z0-9\u0900-\u097f\u0b80-\u0bff\u0c00-\u0c7f\u0c80-\u0cff\s]",
        " ",
        text
    )
    return re.sub(r"\s+", " ", text).strip()

duplicate_source = comments.copy()

duplicate_source["normalized_comment_text"] = (
    duplicate_source["comment_text"].apply(normalize_text)
)

duplicate_source = duplicate_source[
    duplicate_source["normalized_comment_text"].str.len() > 0
].copy()

duplicate_audit = (
    duplicate_source.groupby("normalized_comment_text")
    .agg(
        repeated_records=("comment_id", "count"),
        unique_videos=("video_id", "nunique"),
        unique_movies=("movie_record_id", "nunique"),
        comment_ids=("comment_id", lambda x: " | ".join(x.astype(str))),
        video_ids=("video_id", lambda x: " | ".join(sorted(set(x.astype(str))))),
        movie_titles=("movie_title", lambda x: " | ".join(sorted(set(x.astype(str))))),
        source_batches=("source_batch", lambda x: " | ".join(sorted(set(x.dropna().astype(str))))),
        example_comment=("comment_text", "first")
    )
    .reset_index()
)

duplicate_audit = duplicate_audit[
    duplicate_audit["repeated_records"] > 1
].copy()

duplicate_audit["duplicate_scope"] = duplicate_audit.apply(
    lambda row: (
        "Across Videos"
        if row["unique_videos"] > 1
        else "Within Video"
    ),
    axis=1
)

duplicate_audit = duplicate_audit.sort_values(
    "repeated_records", ascending=False
).reset_index(drop=True)

# =========================================================
# 5. COVERAGE SUMMARY
# =========================================================

coverage_summary = (
    coverage.groupby("coverage_category")
    .agg(
        movies=("movie_record_id", "count"),
        comment_records=("comment_records", "sum"),
        unique_videos=("unique_videos", "sum")
    )
    .reset_index()
)

category_order = [
    "No Comments", "Low", "Moderate", "Higher"
]

coverage_summary["coverage_category"] = pd.Categorical(
    coverage_summary["coverage_category"],
    categories=category_order,
    ordered=True
)

coverage_summary = coverage_summary.sort_values(
    "coverage_category"
).reset_index(drop=True)

# =========================================================
# 6. VALIDATE AND SAVE
# =========================================================

assert len(coverage) == 130
assert coverage["movie_record_id"].nunique() == 130
assert coverage["comment_records"].sum() == 9620
assert coverage["unique_comment_ids"].sum() == 9620

coverage.to_csv(OUTPUTS["coverage"], index=False)
duplicate_audit.to_csv(OUTPUTS["duplicate"], index=False)
coverage_summary.to_csv(OUTPUTS["summary"], index=False)

print("=" * 75)
print("PILOT + BATCHES 01–11 — FULL CORPUS AUDIT")
print("=" * 75)

print("\nTotal comments:", len(comments))
print("Unique comment IDs:", comments["comment_id"].nunique())
print("Unique videos:", comments["video_id"].nunique())
print("Unique movies with comments:", comments["movie_record_id"].nunique())

print("\nCoverage categories:")
print(coverage["coverage_category"].value_counts().reindex(
    category_order, fill_value=0
).to_string())

print("\nCoverage summary:")
display(coverage_summary)

print("\nCoverage totals:")
print("Master movies:", len(coverage))
print("Covered movies:", (coverage["comment_records"] > 0).sum())
print("Uncovered movies:", (coverage["comment_records"] == 0).sum())
print("Total comments:", coverage["comment_records"].sum())

print("\nDuplicate-text audit:")
print("Repeated-text groups:", len(duplicate_audit))
print(
    "Records in repeated groups:",
    duplicate_audit["repeated_records"].sum()
)

if not duplicate_audit.empty:
    print("\nDuplicate scope:")
    print(duplicate_audit["duplicate_scope"].value_counts().to_string())
    display(duplicate_audit.head(15))

print("\nSaved reports:")
for name, path in OUTPUTS.items():
    print(f"{name}: {path}")

print("\nNo comments were deleted. Previous reports and raw files remain unchanged.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

OLD_REPORT = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_10.csv"
)

NEW_COVERAGE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_11.csv"
)

OLD_SUMMARY = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_10.csv"
)

OLD_LANGUAGE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_10.csv"
)

OUTPUT_FILES = [
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_11.csv",
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_11.csv",
    "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_11.csv"
]

for path in [OLD_REPORT, NEW_COVERAGE, OLD_SUMMARY, OLD_LANGUAGE]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

old_report = pd.read_csv(OLD_REPORT)
coverage = pd.read_csv(NEW_COVERAGE)
old_summary = pd.read_csv(OLD_SUMMARY)
old_language = pd.read_csv(OLD_LANGUAGE)

print("=" * 75)
print("ACQUISITION PLANNING — PREFLIGHT")
print("=" * 75)

print("\nPrevious planning report:", old_report.shape)
print("Columns:", old_report.columns.tolist())

print("\nPrevious planning summary:", old_summary.shape)
print("Columns:", old_summary.columns.tolist())
display(old_summary)

print("\nPrevious language summary:", old_language.shape)
print("Columns:", old_language.columns.tolist())
display(old_language)

print("\nNew coverage report:", coverage.shape)
print("Covered movies:", (coverage["comment_records"] > 0).sum())
print("Uncovered movies:", (coverage["comment_records"] == 0).sum())
print("Total comments:", coverage["comment_records"].sum())

print("\nPrevious priority distribution:")
print(old_report["acquisition_priority"].value_counts(dropna=False))

print("\nPrevious planning target:")
print(old_report["additional_comment_target"].sum())

print("\nPrevious planning categories:")
display(
    old_report.groupby(
        ["language", "coverage_category", "acquisition_priority"],
        dropna=False
    ).agg(
        movies=("movie_record_id", "count"),
        comments=("comment_records", "sum"),
        target=("additional_comment_target", "sum")
    ).reset_index()
)

print("\nOutput-file availability:")
for filename in OUTPUT_FILES:
    path = os.path.join(PROCESSED_DIR, filename)
    print(filename, "EXISTS" if os.path.exists(path) else "Available")

print("\nNo files modified.")

In [ ]:
import os
import pandas as pd

# =========================================================
# 1. PATHS
# =========================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

COVERAGE_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_11.csv"
)

OUTPUTS = {
    "report": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_11.csv"
    ),
    "summary": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_SUMMARY_PILOT_BATCH_01_11.csv"
    ),
    "language": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_ACQUISITION_LANGUAGE_SUMMARY_PILOT_BATCH_01_11.csv"
    )
}

if not os.path.exists(COVERAGE_FILE):
    raise FileNotFoundError(COVERAGE_FILE)

existing = [p for p in OUTPUTS.values() if os.path.exists(p)]
if existing:
    raise FileExistsError(
        "Refusing to overwrite existing reports:\n" + "\n".join(existing)
    )

# =========================================================
# 2. LOAD COVERAGE
# =========================================================

coverage = pd.read_csv(COVERAGE_FILE)

assert len(coverage) == 130
assert coverage["movie_record_id"].nunique() == 130
assert coverage["comment_records"].sum() == 9620

planning = coverage.copy()

# =========================================================
# 3. PRIORITY AND TARGET
# =========================================================

def assign_priority(category):
    if category == "No Comments":
        return "High — No Comments"
    elif category == "Low":
        return "High — Low Coverage"
    elif category == "Moderate":
        return "Medium — Moderate Coverage"
    return "Covered — Monitor"

planning["acquisition_priority"] = (
    planning["coverage_category"].apply(assign_priority)
)

planning["additional_comment_target"] = (
    100 - planning["comment_records"]
).clip(lower=0).astype(int)

# Retain the established 13-column planning schema.
planning = planning[
    [
        "movie_record_id",
        "movie_title",
        "year",
        "rank",
        "language",
        "comment_records",
        "unique_comment_ids",
        "unique_videos",
        "acquisition_batches",
        "coverage_status",
        "coverage_category",
        "acquisition_priority",
        "additional_comment_target"
    ]
]

# =========================================================
# 4. COVERAGE SUMMARY
# =========================================================

summary = (
    planning.groupby("coverage_category")
    .agg(
        movie_count=("movie_record_id", "count"),
        existing_comments=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reset_index()
)

category_order = [
    "No Comments", "Low", "Moderate", "Higher"
]

summary["coverage_category"] = pd.Categorical(
    summary["coverage_category"],
    categories=category_order,
    ordered=True
)

summary = summary.sort_values(
    "coverage_category"
).reset_index(drop=True)

summary["coverage_category"] = summary[
    "coverage_category"
].astype(str)

# =========================================================
# 5. LANGUAGE SUMMARY
# =========================================================

language_summary = (
    planning.groupby("language")
    .agg(
        total_movies=("movie_record_id", "count"),
        covered_movies=("comment_records", lambda x: (x > 0).sum()),
        uncovered_movies=("comment_records", lambda x: (x == 0).sum()),
        comment_records=("comment_records", "sum"),
        additional_target=("additional_comment_target", "sum")
    )
    .reset_index()
    .sort_values("language")
    .reset_index(drop=True)
)

# =========================================================
# 6. VALIDATION
# =========================================================

assert len(planning) == 130
assert planning["additional_comment_target"].sum() == (
    130 * 100 - planning["comment_records"].clip(upper=100).sum()
)

assert summary["movie_count"].sum() == 130
assert summary["existing_comments"].sum() == 9620
assert summary["additional_target"].sum() == (
    planning["additional_comment_target"].sum()
)

assert language_summary["total_movies"].sum() == 130
assert language_summary["comment_records"].sum() == 9620
assert language_summary["additional_target"].sum() == (
    planning["additional_comment_target"].sum()
)

# =========================================================
# 7. SAVE REPORTS
# =========================================================

planning.to_csv(OUTPUTS["report"], index=False)
summary.to_csv(OUTPUTS["summary"], index=False)
language_summary.to_csv(OUTPUTS["language"], index=False)

# =========================================================
# 8. DISPLAY RESULTS
# =========================================================

print("=" * 75)
print("PILOT + BATCHES 01–11 — ACQUISITION PLANNING UPDATED")
print("=" * 75)

print("\nOverall:")
print("Total movies:", len(planning))
print("Covered movies:", (planning["comment_records"] > 0).sum())
print("Uncovered movies:", (planning["comment_records"] == 0).sum())
print("Total comments:", planning["comment_records"].sum())
print("Provisional additional target:",
      planning["additional_comment_target"].sum())

print("\nPriority distribution:")
print(planning["acquisition_priority"].value_counts().to_string())

print("\nCoverage summary:")
display(summary)

print("\nLanguage summary:")
display(language_summary)

print("\nHighest-priority uncovered Tamil/Telugu movies:")
display(
    planning[
        planning["language"].isin(["Tamil", "Telugu"])
        & (planning["comment_records"] == 0)
    ][
        ["movie_record_id", "movie_title", "year",
         "language", "additional_comment_target"]
    ].sort_values(["language", "year"])
)

print("\nSaved reports:")
for name, path in OUTPUTS.items():
    print(f"{name}: {path}")

print("\nPrevious planning reports remain unchanged.")
print("The 100-comments-per-movie target remains provisional.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

PLANNING_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_11.csv"
)

BATCH11_ALLOCATION = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_11_MOVIE_CANDIDATE_ALLOCATION.csv"
)

BATCH11_PROPOSED = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_11_MOVIE_ALLOCATION_PROPOSED.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_12_MOVIE_CANDIDATE_ALLOCATION.csv"
)

for path in [PLANNING_FILE, BATCH11_ALLOCATION, BATCH11_PROPOSED]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        "Batch 12 candidate allocation already exists. No overwrite performed."
    )

planning = pd.read_csv(PLANNING_FILE)
batch11 = pd.read_csv(BATCH11_ALLOCATION)
batch11_proposed = pd.read_csv(BATCH11_PROPOSED)

print("=" * 75)
print("BATCH 12 — ALLOCATION PREFLIGHT")
print("=" * 75)

print("\nUpdated planning report:", planning.shape)
print("Columns:", planning.columns.tolist())

print("\nBatch 11 candidate allocation:", batch11.shape)
print("Columns:", batch11.columns.tolist())
display(batch11.head(15))

print("\nBatch 11 proposed allocation:", batch11_proposed.shape)
print("Columns:", batch11_proposed.columns.tolist())
display(batch11_proposed)

print("\nCurrent Hindi coverage:")
hindi = planning[planning["language"] == "Hindi"]

print("Total Hindi movies:", len(hindi))
print("Uncovered Hindi movies:",
      (hindi["comment_records"] == 0).sum())
print("Partially covered Hindi movies:",
      ((hindi["comment_records"] > 0) &
       (hindi["comment_records"] < 100)).sum())
print("Hindi comments:", hindi["comment_records"].sum())
print("Hindi remaining target:",
      hindi["additional_comment_target"].sum())

print("\nUncovered Hindi movies:")
display(
    hindi[hindi["comment_records"] == 0][
        [
            "movie_record_id", "movie_title", "year", "rank",
            "comment_records", "unique_videos",
            "acquisition_priority", "additional_comment_target"
        ]
    ].sort_values(["year", "rank"])
)

print("\nPartially covered movies:")
display(
    planning[
        (planning["comment_records"] > 0) &
        (planning["comment_records"] < 100)
    ][
        [
            "movie_record_id", "movie_title", "year",
            "language", "comment_records",
            "additional_comment_target"
        ]
    ].sort_values(["language", "year"])
)

print("\nBatch 12 output available:",
      not os.path.exists(OUTPUT_FILE))

print("\nNo files modified.")

In [ ]:
import os
import pandas as pd

# =========================================================
# 1. PATHS
# =========================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

PLANNING_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_ACQUISITION_PLANNING_REPORT_PILOT_BATCH_01_11.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_12_MOVIE_CANDIDATE_ALLOCATION.csv"
)

if not os.path.exists(PLANNING_FILE):
    raise FileNotFoundError(PLANNING_FILE)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        "Batch 12 candidate allocation already exists. "
        "No overwrite performed."
    )

# =========================================================
# 2. LOAD CURRENT PLANNING DATA
# =========================================================

planning = pd.read_csv(PLANNING_FILE)

assert len(planning) == 130
assert planning["movie_record_id"].nunique() == 130

# Only movies with a remaining provisional target.
candidates = planning[
    planning["additional_comment_target"] > 0
].copy()

# =========================================================
# 3. ASSIGN BATCH 12 PRIORITIES
# =========================================================

def assign_batch12_priority(row):
    if row["comment_records"] == 0:
        return 1
    elif row["language"] == "Hindi":
        return 2
    else:
        return 3

candidates["planning_priority"] = candidates.apply(
    assign_batch12_priority, axis=1
)

candidates["proposed_batch_id"] = "Batch 12"

# =========================================================
# 4. RETAIN ESTABLISHED COLUMN STRUCTURE
# =========================================================

columns = [
    "proposed_batch_id",
    "movie_record_id",
    "movie_title",
    "year",
    "rank",
    "language",
    "comment_records",
    "unique_comment_ids",
    "unique_videos",
    "acquisition_batches",
    "coverage_status",
    "coverage_category",
    "acquisition_priority",
    "additional_comment_target",
    "planning_priority"
]

candidates = candidates[columns].copy()

candidates = candidates.sort_values(
    ["planning_priority", "year", "rank", "movie_record_id"]
).reset_index(drop=True)

# =========================================================
# 5. VALIDATION
# =========================================================

assert len(candidates) == 45
assert candidates["movie_record_id"].nunique() == 45

assert (
    candidates["comment_records"].eq(0).sum() == 42
)

assert candidates["additional_comment_target"].sum() == 4283

assert not candidates["movie_record_id"].duplicated().any()

# =========================================================
# 6. SAVE
# =========================================================

candidates.to_csv(OUTPUT_FILE, index=False)

print("=" * 75)
print("BATCH 12 — CANDIDATE ALLOCATION CREATED")
print("=" * 75)

print("\nTotal candidate movies:", len(candidates))
print("Total provisional target:",
      candidates["additional_comment_target"].sum())

print("\nPlanning priority distribution:")
print(candidates["planning_priority"].value_counts().sort_index())

print("\nLanguage distribution:")
print(candidates["language"].value_counts())

print("\nPriority 1 — Uncovered movies:")
display(
    candidates[candidates["planning_priority"] == 1][
        ["movie_record_id", "movie_title", "year",
         "rank", "language", "additional_comment_target"]
    ]
)

print("\nPriority 2 and 3 — Partially covered movies:")
display(
    candidates[candidates["planning_priority"] > 1][
        ["movie_record_id", "movie_title", "year",
         "language", "comment_records",
         "additional_comment_target", "planning_priority"]
    ]
)

print("\nSaved:", OUTPUT_FILE)
print("\nNo discovery searches or comment acquisition performed.")

In [ ]:
import os
import pandas as pd

# =========================================================
# 1. PATHS
# =========================================================

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

MASTER_FILE = os.path.join(
    ROOT, "01_data", "processed",
    "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

ALLOCATION_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_12_MOVIE_CANDIDATE_ALLOCATION.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_12_DISCOVERY_PREPARATION.csv"
)

for path in [MASTER_FILE, ALLOCATION_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        "Batch 12 discovery preparation already exists. No overwrite."
    )

# =========================================================
# 2. APPROVED MOVIE SELECTION
# =========================================================

selected_ids = [
    "2011_3",  # Ra.One
    "2011_4",  # Don 2
    "2011_5",  # Singham
    "2012_2",  # Dabangg 2
    "2012_3",  # Rowdy Rathore
    "2012_4",  # Agneepath
    "2012_5",  # Housefull 2
    "2013_2",  # Chennai Express
    "2013_3",  # Krrish 3
    "2013_4"   # Yeh Jawaani Hai Deewani
]

allocation = pd.read_csv(ALLOCATION_FILE)
master = pd.read_csv(MASTER_FILE)

assert len(selected_ids) == 10
assert allocation["movie_record_id"].is_unique
assert master["movie_record_id"].is_unique

selected = allocation[
    allocation["movie_record_id"].isin(selected_ids)
].copy()

if len(selected) != 10:
    raise ValueError(
        f"Expected 10 approved movies, found {len(selected)}."
    )

# =========================================================
# 3. RETRIEVE TMDB METADATA
# =========================================================

tmdb_columns = [
    "movie_record_id",
    "tmdb_id",
    "original_title",
    "release_date",
    "runtime",
    "genres",
    "original_language"
]

for col in tmdb_columns:
    if col not in master.columns:
        raise ValueError(f"Master dataset missing column: {col}")

metadata = master[tmdb_columns].copy()

prepared = selected.merge(
    metadata,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

# Validate selected movie identity against master.
identity_check = selected.merge(
    master[["movie_record_id", "movie_title", "year", "rank", "language"]],
    on="movie_record_id",
    how="left",
    suffixes=("_allocation", "_master"),
    validate="one_to_one"
)

for field in ["movie_title", "year", "rank", "language"]:
    if not (
        identity_check[f"{field}_allocation"].astype(str).values
        == identity_check[f"{field}_master"].astype(str).values
    ).all():
        raise ValueError(f"Movie identity mismatch in field: {field}")

if prepared["tmdb_id"].isna().any():
    raise ValueError("One or more selected movies have missing TMDB IDs.")

if prepared["tmdb_id"].nunique() != 10:
    raise ValueError("TMDB IDs are not unique across selected movies.")

# =========================================================
# 4. PREPARE DISCOVERY QUERIES
# =========================================================

prepared["search_query"] = (
    prepared["movie_title"].astype(str)
    + " "
    + prepared["year"].astype(str)
    + " Hindi movie review reaction"
)

prepared["discovery_status"] = "Pending"
prepared["metadata_status"] = "Pending"
prepared["allocation_status"] = "Approved for Discovery"

prepared = prepared.sort_values(
    ["year", "rank"]
).reset_index(drop=True)

# =========================================================
# 5. VALIDATE AND SAVE
# =========================================================

assert len(prepared) == 10
assert prepared["movie_record_id"].nunique() == 10
assert prepared["tmdb_id"].nunique() == 10
assert prepared["search_query"].notna().all()

prepared.to_csv(OUTPUT_FILE, index=False)

print("=" * 75)
print("BATCH 12 — DISCOVERY PREPARATION COMPLETE")
print("=" * 75)

print("\nApproved movies:", len(prepared))
print("Unique TMDB IDs:", prepared["tmdb_id"].nunique())
print("Missing TMDB IDs:", prepared["tmdb_id"].isna().sum())

display(
    prepared[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "rank",
            "language",
            "tmdb_id",
            "original_title",
            "search_query",
            "discovery_status"
        ]
    ]
)

print("\nSaved:", OUTPUT_FILE)
print("\nNo YouTube searches or comment acquisition performed.")

In [ ]:
# ============================================================
# BATCH 12 — YOUTUBE VIDEO DISCOVERY
# ============================================================

import os
import re
import time
import pandas as pd
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# ------------------------------------------------------------
# 1. PATHS AND CONFIGURATION
# ------------------------------------------------------------

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

PREP_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_12_DISCOVERY_PREPARATION.csv"
)

DISCOVERY_FILE = os.path.join(
    RAW_DIR,
    "ICMB_130_YOUTUBE_DISCOVERY_BATCH_12.csv"
)

LOG_FILE = os.path.join(
    RAW_DIR,
    "ICMB_130_YOUTUBE_DISCOVERY_LOG_BATCH_12.csv"
)

os.makedirs(RAW_DIR, exist_ok=True)

for path in [PREP_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

for path in [DISCOVERY_FILE, LOG_FILE]:
    if os.path.exists(path):
        raise FileExistsError(
            f"Output already exists. No overwrite: {path}"
        )

# ------------------------------------------------------------
# 2. LOAD PREPARATION FILE
# ------------------------------------------------------------

movies = pd.read_csv(PREP_FILE)

required_columns = [
    "movie_record_id",
    "movie_title",
    "year",
    "language",
    "tmdb_id",
    "search_query"
]

missing = [
    c for c in required_columns if c not in movies.columns
]

if missing:
    raise ValueError(f"Missing preparation columns: {missing}")

if len(movies) != 10:
    raise ValueError(f"Expected 10 movies, found {len(movies)}")

if movies["movie_record_id"].nunique() != 10:
    raise ValueError("Duplicate movie_record_id detected.")

# ------------------------------------------------------------
# 3. CONNECT TO YOUTUBE API
# ------------------------------------------------------------

api_key = userdata.get("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError(
        "YOUTUBE_API_KEY not found in Colab Secrets."
    )

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

del api_key

# ------------------------------------------------------------
# 4. DISCOVERY SEARCH
# ------------------------------------------------------------

discovery_rows = []
log_rows = []

print("=" * 75)
print("BATCH 12 — YOUTUBE DISCOVERY STARTED")
print("=" * 75)

for idx, movie in movies.iterrows():

    movie_id = str(movie["movie_record_id"])
    movie_title = str(movie["movie_title"])
    query = str(movie["search_query"])

    print(
        f"\n[{idx + 1}/10] Searching: {movie_title} ({movie['year']})"
    )

    started_at = datetime.now(timezone.utc).isoformat()

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=10,
            order="relevance",
            relevanceLanguage="hi",
            regionCode="IN",
            safeSearch="none"
        ).execute()

        items = response.get("items", [])

        for item in items:

            video_id = item.get("id", {}).get("videoId")

            if not video_id:
                continue

            snippet = item.get("snippet", {})

            discovery_rows.append({
                "movie_record_id": movie_id,
                "movie_title": movie_title,
                "movie_year": int(movie["year"]),
                "industry": movie["language"],
                "search_query": query,
                "video_id": video_id,
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "discovery_source": "YouTube Data API v3",
                "discovered_at_utc": datetime.now(
                    timezone.utc
                ).isoformat()
            })

        log_rows.append({
            "movie_record_id": movie_id,
            "movie_title": movie_title,
            "search_query": query,
            "search_status": "Success",
            "candidates_found": len(items),
            "error_type": None,
            "error_message": None,
            "started_at_utc": started_at,
            "completed_at_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print(f"Candidates found: {len(items)}")

    except HttpError as e:

        # Do not print raw API URLs or credentials.
        status_code = getattr(
            getattr(e, "resp", None), "status", None
        )

        log_rows.append({
            "movie_record_id": movie_id,
            "movie_title": movie_title,
            "search_query": query,
            "search_status": "Failed",
            "candidates_found": 0,
            "error_type": "YouTube API HTTP error",
            "error_message": f"HTTP status: {status_code}",
            "started_at_utc": started_at,
            "completed_at_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print(f"API request failed. HTTP status: {status_code}")

    except Exception as e:

        log_rows.append({
            "movie_record_id": movie_id,
            "movie_title": movie_title,
            "search_query": query,
            "search_status": "Failed",
            "candidates_found": 0,
            "error_type": type(e).__name__,
            "error_message": "Non-HTTP discovery error; details withheld.",
            "started_at_utc": started_at,
            "completed_at_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print(f"Discovery failed: {type(e).__name__}")

    # Small delay between requests.
    time.sleep(1)

# ------------------------------------------------------------
# 5. CREATE DATAFRAMES
# ------------------------------------------------------------

discovery_df = pd.DataFrame(discovery_rows)
log_df = pd.DataFrame(log_rows)

if not discovery_df.empty:
    discovery_df = discovery_df.drop_duplicates(
        subset=["video_id"],
        keep="first"
    ).reset_index(drop=True)

# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BATCH 12 — DISCOVERY VALIDATION")
print("=" * 75)

print("Movies searched:", len(log_df))
print(
    "Successful searches:",
    (log_df["search_status"] == "Success").sum()
)
print(
    "Failed searches:",
    (log_df["search_status"] == "Failed").sum()
)

print("\nCandidate records:", len(discovery_df))

if not discovery_df.empty:
    print(
        "Unique video IDs:",
        discovery_df["video_id"].nunique()
    )
    print(
        "Unique matched movies:",
        discovery_df["movie_record_id"].nunique()
    )

    print("\nCandidates by movie:")
    display(
        discovery_df.groupby(
            ["movie_record_id", "movie_title"]
        ).agg(
            candidate_count=("video_id", "count")
        ).reset_index()
    )

# ------------------------------------------------------------
# 7. SAVE OUTPUTS
# ------------------------------------------------------------

discovery_df.to_csv(DISCOVERY_FILE, index=False)
log_df.to_csv(LOG_FILE, index=False)

print("\n" + "=" * 75)
print("BATCH 12 — DISCOVERY FILES SAVED")
print("=" * 75)

print("\nDiscovery file:", DISCOVERY_FILE)
print("Discovery log:", LOG_FILE)

print("\nNo video metadata retrieval or comment acquisition performed.")

In [ ]:
# ============================================================
# BATCH 12 — YOUTUBE VIDEO METADATA RETRIEVAL
# ============================================================

import os
import time
import pandas as pd
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

DISCOVERY_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_DISCOVERY_BATCH_12.csv"
)

METADATA_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_BATCH_12.csv"
)

LOG_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_LOG_BATCH_12.csv"
)

if not os.path.exists(DISCOVERY_FILE):
    raise FileNotFoundError(DISCOVERY_FILE)

for path in [METADATA_FILE, LOG_FILE]:
    if os.path.exists(path):
        raise FileExistsError(f"No overwrite allowed: {path}")

# ------------------------------------------------------------
# 2. LOAD DISCOVERY DATA
# ------------------------------------------------------------

discovery = pd.read_csv(DISCOVERY_FILE)

required = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "video_id"
]

missing = [c for c in required if c not in discovery.columns]

if missing:
    raise ValueError(f"Missing discovery columns: {missing}")

discovery["video_id"] = discovery["video_id"].astype(str)

video_ids = discovery["video_id"].drop_duplicates().tolist()

print("=" * 75)
print("BATCH 12 — METADATA RETRIEVAL")
print("=" * 75)
print("Discovery records:", len(discovery))
print("Unique video IDs:", len(video_ids))
print("Expected API chunks:", (len(video_ids) + 49) // 50)

# ------------------------------------------------------------
# 3. CONNECT TO YOUTUBE API
# ------------------------------------------------------------

api_key = userdata.get("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError("YOUTUBE_API_KEY not found in Colab Secrets.")

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

del api_key

# ------------------------------------------------------------
# 4. FETCH METADATA IN CHUNKS OF 50
# ------------------------------------------------------------

metadata_rows = []
log_rows = []

chunks = [
    video_ids[i:i + 50]
    for i in range(0, len(video_ids), 50)
]

for chunk_number, chunk in enumerate(chunks, start=1):

    started_at = datetime.now(timezone.utc).isoformat()

    print(
        f"\nProcessing chunk {chunk_number}/{len(chunks)} "
        f"({len(chunk)} video IDs)"
    )

    try:
        response = youtube.videos().list(
            part="snippet,contentDetails,statistics,status",
            id=",".join(chunk),
            maxResults=50
        ).execute()

        items = response.get("items", [])

        for item in items:

            snippet = item.get("snippet", {})
            details = item.get("contentDetails", {})
            stats = item.get("statistics", {})
            status = item.get("status", {})

            metadata_rows.append({
                "video_id": item.get("id"),
                "metadata_title": snippet.get("title"),
                "description_metadata": snippet.get("description"),
                "channel_id_metadata": snippet.get("channelId"),
                "channel_title_metadata": snippet.get("channelTitle"),
                "published_at_metadata": snippet.get("publishedAt"),
                "duration": details.get("duration"),
                "definition": details.get("definition"),
                "caption_available": details.get("caption"),
                "view_count": stats.get("viewCount"),
                "like_count": stats.get("likeCount"),
                "comment_count": stats.get("commentCount"),
                "privacy_status": status.get("privacyStatus"),
                "upload_status": status.get("uploadStatus"),
                "metadata_retrieved_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
                "metadata_status": "Retrieved"
            })

        returned_ids = {
            item.get("id") for item in items
        }

        missing_ids = set(chunk) - returned_ids

        log_rows.append({
            "chunk_number": chunk_number,
            "requested_video_ids": len(chunk),
            "metadata_records_returned": len(items),
            "missing_video_ids": len(missing_ids),
            "missing_ids": ";".join(sorted(missing_ids)),
            "request_status": "Success",
            "error_type": None,
            "error_message": None,
            "started_at_utc": started_at,
            "completed_at_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print("Metadata records returned:", len(items))
        print("IDs not returned:", len(missing_ids))

    except HttpError as e:

        status_code = getattr(
            getattr(e, "resp", None), "status", None
        )

        log_rows.append({
            "chunk_number": chunk_number,
            "requested_video_ids": len(chunk),
            "metadata_records_returned": 0,
            "missing_video_ids": len(chunk),
            "missing_ids": ";".join(chunk),
            "request_status": "Failed",
            "error_type": "YouTube API HTTP error",
            "error_message": f"HTTP status: {status_code}",
            "started_at_utc": started_at,
            "completed_at_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print(f"Chunk failed. HTTP status: {status_code}")

    except Exception as e:

        log_rows.append({
            "chunk_number": chunk_number,
            "requested_video_ids": len(chunk),
            "metadata_records_returned": 0,
            "missing_video_ids": len(chunk),
            "missing_ids": ";".join(chunk),
            "request_status": "Failed",
            "error_type": type(e).__name__,
            "error_message": "Error details withheld.",
            "started_at_utc": started_at,
            "completed_at_utc": datetime.now(
                timezone.utc
            ).isoformat()
        })

        print("Chunk failed:", type(e).__name__)

    time.sleep(1)

# ------------------------------------------------------------
# 5. MERGE DISCOVERY AND METADATA
# ------------------------------------------------------------

metadata = pd.DataFrame(metadata_rows)
metadata_log = pd.DataFrame(log_rows)

if not metadata.empty:

    if metadata["video_id"].duplicated().any():
        raise ValueError("Duplicate video IDs in metadata results.")

    merged = discovery.merge(
        metadata,
        on="video_id",
        how="left",
        validate="many_to_one"
    )

    merged["metadata_status"] = merged[
        "metadata_status"
    ].fillna("Not Retrieved")

else:
    merged = discovery.copy()
    merged["metadata_status"] = "Not Retrieved"

# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

retrieved_count = (
    merged["metadata_status"] == "Retrieved"
).sum()

missing_count = (
    merged["metadata_status"] != "Retrieved"
).sum()

print("\n" + "=" * 75)
print("BATCH 12 — METADATA VALIDATION")
print("=" * 75)

print("Discovery candidates:", len(discovery))
print("Unique discovered IDs:", len(video_ids))
print("Metadata records retrieved:", retrieved_count)
print("Metadata missing:", missing_count)
print(
    "Unique metadata IDs:",
    metadata["video_id"].nunique() if not metadata.empty else 0
)

print("\nMetadata status:")
print(merged["metadata_status"].value_counts())

print("\nMetadata retrieval log:")
display(metadata_log)

print("\nMovie-wise metadata coverage:")
display(
    merged.groupby(
        ["movie_record_id", "movie_title"]
    ).agg(
        candidates=("video_id", "nunique"),
        metadata_retrieved=(
            "metadata_status",
            lambda x: (x == "Retrieved").sum()
        )
    ).reset_index()
)

# ------------------------------------------------------------
# 7. SAVE OUTPUTS
# ------------------------------------------------------------

merged.to_csv(METADATA_FILE, index=False)
metadata_log.to_csv(LOG_FILE, index=False)

print("\n" + "=" * 75)
print("BATCH 12 — METADATA FILES SAVED")
print("=" * 75)

print("Metadata file:", METADATA_FILE)
print("Metadata log:", LOG_FILE)

print("\nNo source screening or comment acquisition performed.")

In [ ]:
# ============================================================
# BATCH 12 — AUTOMATED YOUTUBE SOURCE SCREENING
# ============================================================

import os
import re
import pandas as pd
from datetime import datetime, timezone

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

METADATA_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_BATCH_12.csv"
)

SCREENING_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_12.csv"
)

REVIEW_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_CANDIDATE_REVIEW_BATCH_12.csv"
)

if not os.path.exists(METADATA_FILE):
    raise FileNotFoundError(METADATA_FILE)

for path in [SCREENING_FILE, REVIEW_FILE]:
    if os.path.exists(path):
        raise FileExistsError(f"No overwrite allowed: {path}")

# ------------------------------------------------------------
# 2. LOAD AND VALIDATE METADATA
# ------------------------------------------------------------

df = pd.read_csv(METADATA_FILE)

required_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "video_id",
    "metadata_title",
    "description_metadata",
    "channel_title_metadata",
    "duration",
    "view_count",
    "comment_count",
    "metadata_status"
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

if df["video_id"].nunique() != 100:
    raise ValueError("Expected 100 unique video IDs.")

# ------------------------------------------------------------
# 3. TEXT NORMALIZATION AND KEYWORD GROUPS
# ------------------------------------------------------------

def normalize_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).lower()).strip()


REVIEW_TERMS = [
    "movie review", "film review", "review", "explained",
    "analysis", "ending explained", "cinema review",
    "honest review", "detailed review", "rewatch review",
    "retrospective", "rewind", "film analysis"
]

REACTION_TERMS = [
    "movie reaction", "film reaction", "reaction",
    "first time watching", "watching for the first time",
    "reacts to", "reacting to", "part 1", "part 2",
    "part 3", "full reaction"
]

TRAILER_TERMS = [
    "official trailer", "trailer", "teaser",
    "official teaser", "announcement teaser"
]

SONG_TERMS = [
    "full song", "video song", "lyrical video",
    "music video", "jukebox", "audio jukebox",
    "song", "lyrics", "ost"
]

CLIP_TERMS = [
    "scene", "clip", "fight scene", "comedy scene",
    "best scenes", "dialogue", "entry scene",
    "action scene", "status video"
]

FULL_MOVIE_TERMS = [
    "full movie", "full hindi movie", "complete movie",
    "watch full movie", "entire movie"
]

UNRELATED_TERMS = [
    "gameplay", "game walkthrough", "cover song",
    "dance cover", "tutorial", "reaction to trailer"
]

# ------------------------------------------------------------
# 4. CLASSIFICATION FUNCTION
# ------------------------------------------------------------

def classify_candidate(row):

    title = normalize_text(row["metadata_title"])
    description = normalize_text(row["description_metadata"])

    combined = title + " " + description

    # Explicit exclusion indicators
    if any(term in title for term in FULL_MOVIE_TERMS):
        return "Exclude", "Full Movie", "Full movie upload"

    if any(term in title for term in SONG_TERMS):
        return "Exclude", "Song/Music Video", "Song or music-focused title"

    if any(term in title for term in TRAILER_TERMS):
        return "Exclude", "Trailer/Teaser", "Trailer or teaser title"

    if any(term in title for term in CLIP_TERMS):
        return "Exclude", "Clip/Scene", "Scene or clip-focused title"

    if any(term in title for term in UNRELATED_TERMS):
        return "Exclude", "Unrelated", "Unrelated content indicator"

    review_hits = sum(term in combined for term in REVIEW_TERMS)
    reaction_hits = sum(term in combined for term in REACTION_TERMS)

    # Review + reaction
    if review_hits > 0 and reaction_hits > 0:
        return (
            "Shortlist",
            "Review+Reaction",
            "Review and reaction indicators found"
        )

    # Review
    if review_hits > 0:
        return (
            "Shortlist",
            "Movie Review/Analysis",
            "Review or analysis indicators found"
        )

    # Reaction
    if reaction_hits > 0:
        return (
            "Shortlist",
            "Movie Reaction",
            "Reaction indicators found"
        )

    # Additional cautious checks
    if any(
        term in description
        for term in ["movie review", "film review", "movie reaction"]
    ):
        return (
            "Hold",
            "Unclassified",
            "Potential review/reaction evidence in description"
        )

    return (
        "Hold",
        "Unclassified",
        "Insufficient metadata evidence"
    )


# ------------------------------------------------------------
# 5. APPLY SCREENING
# ------------------------------------------------------------

results = df.apply(classify_candidate, axis=1)

df["automated_decision"] = [x[0] for x in results]
df["automated_source_type"] = [x[1] for x in results]
df["screening_reason"] = [x[2] for x in results]

df["metadata_review_status"] = "Metadata Reviewed"
df["content_review_status"] = "Not Reviewed"
df["screened_at_utc"] = datetime.now(
    timezone.utc
).isoformat()

# ------------------------------------------------------------
# 6. CREATE CONSOLIDATED REVIEW SHEET
# ------------------------------------------------------------

review_columns = [
    "movie_record_id",
    "movie_title",
    "movie_year",
    "video_id",
    "metadata_title",
    "channel_title_metadata",
    "published_at_metadata",
    "duration",
    "view_count",
    "comment_count",
    "automated_decision",
    "automated_source_type",
    "screening_reason",
    "metadata_review_status",
    "content_review_status"
]

review_df = df[review_columns].copy()

review_df["manual_decision"] = "Pending"
review_df["manual_source_type"] = "Pending"
review_df["manual_verification_status"] = "Pending"
review_df["manual_review_notes"] = ""

# ------------------------------------------------------------
# 7. VALIDATION AND SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("BATCH 12 — AUTOMATED SCREENING RESULTS")
print("=" * 75)

print("Total candidates:", len(df))
print("Unique video IDs:", df["video_id"].nunique())
print("Movies represented:", df["movie_record_id"].nunique())

print("\nAutomated decision distribution:")
display(df["automated_decision"].value_counts().to_frame("count"))

print("\nSource type distribution:")
display(df["automated_source_type"].value_counts().to_frame("count"))

print("\nResults by movie:")
display(
    df.groupby(
        ["movie_record_id", "movie_title", "automated_decision"]
    ).size().unstack(fill_value=0).reset_index()
)

print("\nReview sheet preview:")
display(
    review_df[
        [
            "movie_title",
            "metadata_title",
            "channel_title_metadata",
            "duration",
            "automated_decision",
            "automated_source_type"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 8. SAVE OUTPUTS
# ------------------------------------------------------------

df.to_csv(SCREENING_FILE, index=False)
review_df.to_csv(REVIEW_FILE, index=False)

print("\n" + "=" * 75)
print("BATCH 12 — SCREENING FILES SAVED")
print("=" * 75)

print("Screening:", SCREENING_FILE)
print("Review sheet:", REVIEW_FILE)

print("\nNo videos watched or content independently verified.")
print("No comment acquisition performed.")

In [ ]:
# ============================================================
# BATCH 12 — CONSOLIDATED SHORTLIST AND HOLD REVIEW
# ============================================================

import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

SCREENING_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_METADATA_SCREENING_BATCH_12.csv"
)

OUTPUT_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_SHORTLIST_HOLD_REVIEW_BATCH_12.csv"
)

if not os.path.exists(SCREENING_FILE):
    raise FileNotFoundError(SCREENING_FILE)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError("Review output already exists. No overwrite.")

df = pd.read_csv(SCREENING_FILE)

# Keep potential sources and ambiguous candidates.
review = df[
    df["automated_decision"].isin(["Shortlist", "Hold"])
].copy()

# Add direct video links.
review["youtube_url"] = (
    "https://www.youtube.com/watch?v="
    + review["video_id"].astype(str)
)

# Keep descriptions readable.
review["description_preview"] = (
    review["description_metadata"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 350)
)

review["manual_decision"] = "Pending"
review["manual_source_type"] = "Pending"
review["manual_verification_status"] = "Pending"
review["manual_review_notes"] = ""

columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "channel_title_metadata",
    "duration",
    "view_count",
    "comment_count",
    "automated_decision",
    "automated_source_type",
    "screening_reason",
    "description_preview",
    "youtube_url",
    "manual_decision",
    "manual_source_type",
    "manual_verification_status",
    "manual_review_notes"
]

review = review[columns].sort_values(
    ["movie_title", "automated_decision", "channel_title_metadata"]
).reset_index(drop=True)

review.to_csv(OUTPUT_FILE, index=False)

print("=" * 75)
print("BATCH 12 — SHORTLIST AND HOLD REVIEW")
print("=" * 75)

print("Total candidates for review:", len(review))
print("Shortlist:", (review["automated_decision"] == "Shortlist").sum())
print("Hold:", (review["automated_decision"] == "Hold").sum())

print("\nCandidates by movie:")
display(
    review.groupby(
        ["movie_title", "automated_decision"]
    ).size().unstack(fill_value=0).reset_index()
)

print("\nComplete consolidated review table:")
display(review)

print("\nSaved:", OUTPUT_FILE)
print("\nNo videos watched and no sources approved by this cell.")

In [ ]:
import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

INPUT_FILE = os.path.join(
    RAW_DIR,
    "ICMB_130_YOUTUBE_SHORTLIST_HOLD_REVIEW_BATCH_12.csv"
)

OUTPUT_FILE = os.path.join(
    RAW_DIR,
    "ICMB_130_YOUTUBE_CANDIDATE_DECISION_TABLE_BATCH_12.csv"
)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(INPUT_FILE)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError("Output already exists. No overwrite.")

df = pd.read_csv(INPUT_FILE)

# Keep a compact, complete decision table.
columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "channel_title_metadata",
    "duration",
    "view_count",
    "comment_count",
    "automated_decision",
    "automated_source_type",
    "description_preview",
    "youtube_url"
]

decision_table = df[columns].copy()

decision_table = decision_table.sort_values(
    ["movie_title", "automated_decision", "channel_title_metadata"]
).reset_index(drop=True)

decision_table.to_csv(OUTPUT_FILE, index=False)

print("=" * 75)
print("BATCH 12 — COMPLETE CANDIDATE DECISION TABLE")
print("=" * 75)

for movie, group in decision_table.groupby("movie_title", sort=False):

    print(f"\n{'=' * 75}")
    print(f"MOVIE: {movie}")
    print(f"{'=' * 75}")

    for _, row in group.iterrows():

        print(f"\nVideo ID: {row['video_id']}")
        print(f"Title: {row['metadata_title']}")
        print(f"Channel: {row['channel_title_metadata']}")
        print(f"Duration: {row['duration']}")
        print(f"Views: {row['view_count']}")
        print(f"Comments: {row['comment_count']}")
        print(f"Automated decision: {row['automated_decision']}")
        print(f"Source type: {row['automated_source_type']}")
        print(f"Description: {row['description_preview']}")
        print(f"URL: {row['youtube_url']}")

print("\nSaved:", OUTPUT_FILE)

In [ ]:
# ============================================================
# BATCH 12 — CREATE APPROVED ACQUISITION SOURCE MANIFEST
# ============================================================

import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

INPUT_FILE = os.path.join(
    RAW_DIR,
    "ICMB_130_YOUTUBE_CANDIDATE_DECISION_TABLE_BATCH_12.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_12.csv"
)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(INPUT_FILE)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        f"Manifest already exists. No overwrite: {OUTPUT_FILE}"
    )

# ------------------------------------------------------------
# 1. APPROVED SOURCE SELECTION
# ------------------------------------------------------------

approved_sources = [
    # Agneepath
    {
        "movie_record_id": "2012_4",
        "video_id": "wSoyDwzZ1YQ",
        "manual_source_type": "Movie Review/Retrospective",
        "manual_review_notes": "Agneepath retrospective review."
    },
    {
        "movie_record_id": "2012_4",
        "video_id": "Jhl4EsGGwrA",
        "manual_source_type": "Movie Reaction",
        "manual_review_notes": "OUR STUPID REACTIONS; Part 1/3."
    },

    # Chennai Express
    {
        "movie_record_id": "2013_2",
        "video_id": "vw6rg4tO99g",
        "manual_source_type": "Movie Reaction",
        "manual_review_notes": "CineDesi; Part 1/3."
    },
    {
        "movie_record_id": "2013_2",
        "video_id": "JyRn6ZjA-BA",
        "manual_source_type": "Movie Reaction",
        "manual_review_notes": "VI3ION Movies; long-form movie reaction."
    },

    # Don 2
    {
        "movie_record_id": "2011_4",
        "video_id": "ceseovhL7Ug",
        "manual_source_type": "Movie Review",
        "manual_review_notes": "Jimmy Cage; dedicated movie review."
    },
    {
        "movie_record_id": "2011_4",
        "video_id": "yCmowCnwYEY",
        "manual_source_type": "Movie Reaction",
        "manual_review_notes": "CineDesi; Part 1 reaction."
    },

    # Krrish 3
    {
        "movie_record_id": "2013_3",
        "video_id": "b8rdMhQN6jI",
        "manual_source_type": "Movie Review/Analysis",
        "manual_review_notes": "Feature Friday Movies (INDIA); long-form review."
    },
    {
        "movie_record_id": "2013_3",
        "video_id": "T_Jrfs39fnk",
        "manual_source_type": "Movie Reaction",
        "manual_review_notes": "CineDesi; Part 1 reaction."
    },

    # Ra.One
    {
        "movie_record_id": "2011_3",
        "video_id": "AP-fgtQR8MU",
        "manual_source_type": "Movie Review/Retrospective",
        "manual_review_notes": "BnfTV; retrospective review."
    },
    {
        "movie_record_id": "2011_3",
        "video_id": "2zqfOs9cVLg",
        "manual_source_type": "Movie Reaction",
        "manual_review_notes": "ChaatNChat; Part 1/3."
    },

    # Singham
    {
        "movie_record_id": "2011_5",
        "video_id": "I87-GVLoIbo",
        "manual_source_type": "Movie Review",
        "manual_review_notes": "BnfTV; dedicated movie review."
    },
    {
        "movie_record_id": "2011_5",
        "video_id": "XRnLckG_p0Y",
        "manual_source_type": "Movie Reaction",
        "manual_review_notes": "ChaatNChat; Part 1/3."
    },

    # Yeh Jawaani Hai Deewani
    {
        "movie_record_id": "2013_4",
        "video_id": "_R80RUHoff0",
        "manual_source_type": "Movie Reaction",
        "manual_review_notes": "Badd Medicine; long-form first-time movie reaction."
    },
    {
        "movie_record_id": "2013_4",
        "video_id": "pslwigCtXuI",
        "manual_source_type": "Review+Reaction",
        "manual_review_notes": "Feature Friday Movies (INDIA); long-form review/reaction."
    }
]

approved = pd.DataFrame(approved_sources)

# ------------------------------------------------------------
# 2. LOAD ACTUAL CANDIDATE METADATA
# ------------------------------------------------------------

candidates = pd.read_csv(INPUT_FILE)

required_columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "metadata_title",
    "channel_title_metadata",
    "duration",
    "view_count",
    "comment_count"
]

missing = [
    col for col in required_columns
    if col not in candidates.columns
]

if missing:
    raise ValueError(f"Missing candidate columns: {missing}")

candidates["video_id"] = candidates["video_id"].astype(str)
approved["video_id"] = approved["video_id"].astype(str)

# ------------------------------------------------------------
# 3. MATCH APPROVED VIDEOS TO CANDIDATE RECORDS
# ------------------------------------------------------------

manifest = approved.merge(
    candidates[required_columns],
    on=["movie_record_id", "video_id"],
    how="left",
    validate="one_to_one",
    indicator=True
)

if not (manifest["_merge"] == "both").all():
    raise ValueError(
        "One or more approved video IDs were not found in candidate metadata."
    )

manifest.drop(columns=["_merge"], inplace=True)

# ------------------------------------------------------------
# 4. ADD MOVIE YEAR AND LANGUAGE FROM ALLOCATION
# ------------------------------------------------------------

allocation_file = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_12_MOVIE_ALLOCATION.csv"
)

# Use the discovery preparation file if the allocation
# file is not present in the processed directory.
if not os.path.exists(allocation_file):
    allocation_file = os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_BATCH_12_DISCOVERY_PREPARATION.csv"
    )

if not os.path.exists(allocation_file):
    raise FileNotFoundError(
        "Batch 12 allocation/discovery preparation file not found."
    )

allocation = pd.read_csv(allocation_file)

allocation_columns = [
    "movie_record_id", "year", "language"
]

for col in allocation_columns:
    if col not in allocation.columns:
        raise ValueError(f"Allocation file missing {col}")

allocation = allocation[allocation_columns].drop_duplicates(
    subset=["movie_record_id"]
)

manifest = manifest.merge(
    allocation,
    on="movie_record_id",
    how="left",
    validate="many_to_one"
)

manifest.rename(
    columns={"year": "movie_year"},
    inplace=True
)

# ------------------------------------------------------------
# 5. ADD ACQUISITION STATUS FIELDS
# ------------------------------------------------------------

manifest["manual_verification_status"] = "Metadata Reviewed"
manifest["content_review_status"] = "Not Reviewed"
manifest["acquisition_status"] = "Pending"
manifest["target_comments"] = 100

manifest = manifest.rename(columns={
    "metadata_title": "video_title",
    "channel_title_metadata": "channel_title"
})

# Keep a consistent manifest structure.
manifest = manifest[
    [
        "movie_record_id",
        "movie_title",
        "movie_year",
        "language",
        "video_id",
        "video_title",
        "channel_title",
        "duration",
        "view_count",
        "comment_count",
        "manual_source_type",
        "manual_verification_status",
        "manual_review_notes",
        "content_review_status",
        "acquisition_status",
        "target_comments"
    ]
].copy()

# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

assert len(manifest) == 14
assert manifest["video_id"].nunique() == 14
assert manifest["movie_record_id"].nunique() == 7
assert manifest["movie_year"].notna().all()
assert manifest["language"].notna().all()
assert manifest["movie_title"].notna().all()

print("=" * 75)
print("BATCH 12 — APPROVED SOURCE MANIFEST")
print("=" * 75)

print("Approved sources:", len(manifest))
print("Unique videos:", manifest["video_id"].nunique())
print("Movies covered:", manifest["movie_record_id"].nunique())
print("Provisional comment target:", manifest["target_comments"].sum())

print("\nSources per movie:")
display(
    manifest.groupby(
        ["movie_record_id", "movie_title"]
    ).agg(
        approved_sources=("video_id", "nunique"),
        planned_comments=("target_comments", "sum")
    ).reset_index()
)

print("\nAcquisition status:")
print(manifest["acquisition_status"].value_counts())

print("\nContent review status:")
print(manifest["content_review_status"].value_counts())

# ------------------------------------------------------------
# 7. SAVE
# ------------------------------------------------------------

manifest.to_csv(OUTPUT_FILE, index=False)

print("\nSaved:", OUTPUT_FILE)
print("\nNo comment acquisition performed.")

In [ ]:
# ============================================================
# BATCH 12 — YOUTUBE COMMENT ACQUISITION
# Approved 14-video manifest only
# ============================================================

import os
import re
import time
import pandas as pd
from datetime import datetime, timezone
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import userdata

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube", "comments"
)

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

MANIFEST_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_12.csv"
)

os.makedirs(RAW_DIR, exist_ok=True)

BATCH_TAG = "BATCH_12"
MAX_COMMENTS_PER_VIDEO = 100
MAX_PAGES_PER_VIDEO = 2

RAW_OUTPUT = os.path.join(
    RAW_DIR,
    f"ICMB_130_YOUTUBE_COMMENTS_{BATCH_TAG}.csv"
)

LOG_OUTPUT = os.path.join(
    RAW_DIR,
    f"ICMB_130_YOUTUBE_COMMENTS_ACQUISITION_LOG_{BATCH_TAG}.csv"
)

# ------------------------------------------------------------
# 1. SAFETY CHECKS
# ------------------------------------------------------------

if not os.path.exists(MANIFEST_FILE):
    raise FileNotFoundError(MANIFEST_FILE)

if os.path.exists(RAW_OUTPUT) or os.path.exists(LOG_OUTPUT):
    raise FileExistsError(
        "Batch 12 output already exists. No files will be overwritten."
    )

manifest = pd.read_csv(MANIFEST_FILE)

required = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "acquisition_status"
]

missing = [c for c in required if c not in manifest.columns]
if missing:
    raise ValueError(f"Manifest missing columns: {missing}")

if len(manifest) != 14 or manifest["video_id"].nunique() != 14:
    raise ValueError("Manifest validation failed: expected 14 unique videos.")

if not manifest["acquisition_status"].eq("Pending").all():
    raise ValueError("Some sources are not Pending. Review before acquisition.")

# ------------------------------------------------------------
# 2. CONNECT USING COLAB SECRET
# ------------------------------------------------------------

api_key = userdata.get("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError("YOUTUBE_API_KEY is unavailable in Colab Secrets.")

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

del api_key

print("YouTube API connection initialized.")
print("Approved videos:", len(manifest))
print("Maximum comments per video:", MAX_COMMENTS_PER_VIDEO)

# ------------------------------------------------------------
# 3. FETCH TOP-LEVEL COMMENTS
# ------------------------------------------------------------

all_comments = []
acquisition_logs = []

for index, row in manifest.iterrows():

    video_id = str(row["video_id"])
    movie_title = row["movie_title"]
    movie_record_id = row["movie_record_id"]

    video_comments = []
    page_token = None
    pages_fetched = 0
    status = "Success"
    error_message = ""

    print(
        f"\n[{index + 1}/{len(manifest)}] "
        f"{movie_title} | {video_id}"
    )

    try:
        while (
            len(video_comments) < MAX_COMMENTS_PER_VIDEO
            and pages_fetched < MAX_PAGES_PER_VIDEO
        ):

            remaining = MAX_COMMENTS_PER_VIDEO - len(video_comments)

            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, remaining),
                order="time",
                textFormat="plainText",
                pageToken=page_token
            ).execute()

            pages_fetched += 1

            for item in response.get("items", []):

                snippet = item["snippet"]["topLevelComment"]["snippet"]

                comment_id = item["snippet"]["topLevelComment"]["id"]

                video_comments.append({
                    "batch_id": BATCH_TAG,
                    "movie_record_id": movie_record_id,
                    "movie_title": movie_title,
                    "video_id": video_id,
                    "comment_id": comment_id,
                    "comment_text": snippet.get("textDisplay", ""),
                    "author_name": snippet.get("authorDisplayName", ""),
                    "published_at": snippet.get("publishedAt", ""),
                    "updated_at": snippet.get("updatedAt", ""),
                    "like_count": snippet.get("likeCount", 0),
                    "comment_source_type": row.get(
                        "manual_source_type", ""
                    ),
                    "collection_order": "time",
                    "collection_timestamp_utc": datetime.now(
                        timezone.utc
                    ).isoformat()
                })

            page_token = response.get("nextPageToken")

            if not page_token:
                break

        if not video_comments:
            status = "No comments returned"

    except HttpError as e:
        status = "API Error"

        # Do not print raw API error URLs or credentials.
        error_message = (
            f"HTTP {getattr(e.resp, 'status', 'Unknown')}: "
            "YouTube API request failed."
        )

    except Exception as e:
        status = "Error"
        error_message = re.sub(
            r"https?://\S+",
            "[URL REDACTED]",
            str(e)
        )[:300]

    all_comments.extend(video_comments)

    acquisition_logs.append({
        "batch_id": BATCH_TAG,
        "movie_record_id": movie_record_id,
        "movie_title": movie_title,
        "video_id": video_id,
        "pages_fetched": pages_fetched,
        "comments_collected": len(video_comments),
        "acquisition_status": status,
        "error_message": error_message,
        "collection_timestamp_utc": datetime.now(
            timezone.utc
        ).isoformat()
    })

    print(
        f"Status: {status} | "
        f"Comments: {len(video_comments)} | "
        f"Pages: {pages_fetched}"
    )

    time.sleep(0.2)

# ------------------------------------------------------------
# 4. SAVE RAW COMMENTS AND ACQUISITION LOG
# ------------------------------------------------------------

comments_df = pd.DataFrame(all_comments)
logs_df = pd.DataFrame(acquisition_logs)

comments_df.to_csv(RAW_OUTPUT, index=False)
logs_df.to_csv(LOG_OUTPUT, index=False)

# ------------------------------------------------------------
# 5. VALIDATION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BATCH 12 — COMMENT ACQUISITION SUMMARY")
print("=" * 75)

print("Approved videos:", len(manifest))
print("Videos processed:", len(logs_df))
print("Total comments:", len(comments_df))
print("Unique comment IDs:", comments_df["comment_id"].nunique()
      if not comments_df.empty else 0)
print("Unique videos with comments:", comments_df["video_id"].nunique()
      if not comments_df.empty else 0)

print("\nAcquisition status:")
print(logs_df["acquisition_status"].value_counts(dropna=False))

print("\nMovie-wise comment counts:")
if not comments_df.empty:
    display(
        comments_df.groupby(
            ["movie_record_id", "movie_title"]
        ).agg(
            comments=("comment_id", "count"),
            unique_comments=("comment_id", "nunique"),
            videos=("video_id", "nunique")
        ).reset_index()
    )

print("\nAcquisition log:")
display(logs_df)

print("\nRaw comments saved:", RAW_OUTPUT)
print("Acquisition log saved:", LOG_OUTPUT)

print("\nPrevious batches were not modified.")

In [ ]:
# ============================================================
# BATCH 12 — DATA QUALITY & CONSOLIDATION READINESS AUDIT
# Read-only audit; no source files modified
# ============================================================

import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube", "comments"
)

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

BATCH_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_COMMENTS_BATCH_12.csv"
)

LOG_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_COMMENTS_ACQUISITION_LOG_BATCH_12.csv"
)

MANIFEST_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_APPROVED_ACQUISITION_SOURCES_BATCH_12.csv"
)

EXISTING_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_11.csv"
)

REPORT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_BATCH_12_DATA_QUALITY_AUDIT.csv"
)

# ------------------------------------------------------------
# 1. LOAD FILES
# ------------------------------------------------------------

for path in [BATCH_FILE, LOG_FILE, MANIFEST_FILE, EXISTING_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

batch = pd.read_csv(BATCH_FILE)
logs = pd.read_csv(LOG_FILE)
manifest = pd.read_csv(MANIFEST_FILE)
existing = pd.read_csv(EXISTING_FILE)

print("=" * 75)
print("BATCH 12 — DATA QUALITY AUDIT")
print("=" * 75)

print("Batch 12 rows:", len(batch))
print("Existing consolidated rows:", len(existing))

# ------------------------------------------------------------
# 2. CHECK REQUIRED FIELDS
# ------------------------------------------------------------

required_columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "comment_id",
    "comment_text"
]

missing_columns = [
    col for col in required_columns
    if col not in batch.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# ------------------------------------------------------------
# 3. QUALITY CHECKS
# ------------------------------------------------------------

checks = []

def add_check(name, value, details=""):
    checks.append({
        "audit_check": name,
        "result": value,
        "details": details
    })

add_check(
    "Expected approved videos",
    len(manifest) == 14 and manifest["video_id"].nunique() == 14,
    f"Manifest videos: {manifest['video_id'].nunique()}"
)

add_check(
    "All videos processed",
    logs["video_id"].nunique() == 14,
    f"Logged videos: {logs['video_id'].nunique()}"
)

add_check(
    "All acquisition statuses successful",
    logs["acquisition_status"].eq("Success").all(),
    str(logs["acquisition_status"].value_counts().to_dict())
)

add_check(
    "All approved videos represented in comments",
    set(manifest["video_id"]).issubset(set(batch["video_id"])),
    f"Videos in comments: {batch['video_id'].nunique()}"
)

add_check(
    "No missing comment IDs",
    batch["comment_id"].notna().all(),
    f"Missing IDs: {batch['comment_id'].isna().sum()}"
)

add_check(
    "No duplicate comment IDs within Batch 12",
    batch["comment_id"].nunique() == len(batch),
    f"Duplicate rows: {batch['comment_id'].duplicated().sum()}"
)

add_check(
    "No empty comment text",
    batch["comment_text"].fillna("").str.strip().ne("").all(),
    f"Empty text: {batch['comment_text'].fillna('').str.strip().eq('').sum()}"
)

existing_ids = set(existing["comment_id"].dropna().astype(str))
batch_ids = batch["comment_id"].dropna().astype(str)

cross_batch_duplicates = batch_ids.isin(existing_ids).sum()

add_check(
    "No overlap with Batches 01–11",
    cross_batch_duplicates == 0,
    f"Overlapping comment IDs: {cross_batch_duplicates}"
)

# ------------------------------------------------------------
# 4. MOVIE-WISE AND VIDEO-WISE DISTRIBUTION
# ------------------------------------------------------------

movie_summary = (
    batch.groupby(["movie_record_id", "movie_title"])
    .agg(
        total_comments=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        videos=("video_id", "nunique"),
        empty_text=("comment_text",
                    lambda x: x.fillna("").str.strip().eq("").sum())
    )
    .reset_index()
)

video_summary = (
    batch.groupby(["movie_title", "video_id"])
    .agg(
        comments=("comment_id", "count"),
        unique_comments=("comment_id", "nunique")
    )
    .reset_index()
)

print("\nMOVIE-WISE SUMMARY")
display(movie_summary)

print("\nVIDEO-WISE SUMMARY")
display(video_summary)

# ------------------------------------------------------------
# 5. FINAL AUDIT REPORT
# ------------------------------------------------------------

audit_df = pd.DataFrame(checks)

print("\nAUDIT CHECKS")
display(audit_df)

print("\nOVERALL READINESS:",
      "PASS" if audit_df["result"].all() else "REVIEW REQUIRED")

print("\nNew unique records eligible for consolidation:",
      len(batch) - cross_batch_duplicates)

# ------------------------------------------------------------
# 6. SAVE AUDIT REPORT
# ------------------------------------------------------------

if os.path.exists(REPORT_FILE):
    raise FileExistsError(
        f"Audit report already exists; refusing to overwrite: {REPORT_FILE}"
    )

audit_df.to_csv(REPORT_FILE, index=False)

print("\nAudit report saved:", REPORT_FILE)
print("No raw or consolidated comment files were modified.")

In [ ]:
# ============================================================
# BATCH 12 — CREATE CLEAN ANALYSIS COPY
# Raw data remains unchanged
# ============================================================

import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube", "comments"
)

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

RAW_FILE = os.path.join(
    RAW_DIR, "ICMB_130_YOUTUBE_COMMENTS_BATCH_12.csv"
)

CLEAN_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CLEAN_BATCH_12.csv"
)

EXCLUDED_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_EMPTY_TEXT_EXCLUSIONS_BATCH_12.csv"
)

for path in [CLEAN_FILE, EXCLUDED_FILE]:
    if os.path.exists(path):
        raise FileExistsError(f"Refusing to overwrite: {path}")

# ------------------------------------------------------------
# 1. LOAD RAW DATA
# ------------------------------------------------------------

raw = pd.read_csv(RAW_FILE)

if "comment_text" not in raw.columns:
    raise ValueError("comment_text column missing.")

# ------------------------------------------------------------
# 2. IDENTIFY EMPTY TEXT RECORDS
# ------------------------------------------------------------

text = raw["comment_text"].fillna("").astype(str).str.strip()

empty_mask = text.eq("")

clean = raw.loc[~empty_mask].copy()
excluded = raw.loc[empty_mask].copy()

clean["cleaning_status"] = "Retained - Non-empty text"
excluded["cleaning_status"] = "Excluded from analysis - Empty text"

# ------------------------------------------------------------
# 3. VALIDATE
# ------------------------------------------------------------

assert len(clean) + len(excluded) == len(raw)
assert clean["comment_id"].is_unique
assert clean["comment_text"].fillna("").str.strip().ne("").all()

print("=" * 75)
print("BATCH 12 — CLEANING SUMMARY")
print("=" * 75)

print("Raw records:", len(raw))
print("Retained records:", len(clean))
print("Excluded empty-text records:", len(excluded))
print("Unique retained comment IDs:", clean["comment_id"].nunique())
print("Unique videos retained:", clean["video_id"].nunique())

print("\nExcluded records:")
if len(excluded):
    display(
        excluded[
            ["movie_title", "video_id", "comment_id", "comment_text"]
        ]
    )

# ------------------------------------------------------------
# 4. SAVE SEPARATE OUTPUTS
# ------------------------------------------------------------

clean.to_csv(CLEAN_FILE, index=False)
excluded.to_csv(EXCLUDED_FILE, index=False)

print("\nClean analysis copy saved:", CLEAN_FILE)
print("Empty-text exclusion audit saved:", EXCLUDED_FILE)
print("\nOriginal raw file remains unchanged.")

In [ ]:
# ============================================================
# DIAGNOSE EMPTY COMMENT TEXT — BATCHES 01–11 vs BATCH 12
# Read-only diagnostic
# ============================================================

import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

OLD_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_11.csv"
)

BATCH12_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CLEAN_BATCH_12.csv"
)

old_df = pd.read_csv(OLD_FILE)
batch12_df = pd.read_csv(BATCH12_FILE)

def inspect_empty_text(df, dataset_name):
    text = df["comment_text"].fillna("").astype(str).str.strip()
    mask = text.eq("")

    print("\n" + "=" * 70)
    print(dataset_name)
    print("=" * 70)

    print("Total records:", len(df))
    print("Empty comment text:", int(mask.sum()))
    print("Non-empty comment text:", int((~mask).sum()))

    if mask.any():
        cols = [
            c for c in [
                "batch_id",
                "movie_record_id",
                "movie_title",
                "video_id",
                "comment_id",
                "comment_text"
            ] if c in df.columns
        ]

        display(df.loc[mask, cols].head(20))

inspect_empty_text(old_df, "EXISTING BATCHES 01–11")
inspect_empty_text(batch12_df, "NEW BATCH 12")

print("\nDiagnostic completed. No files modified.")

In [ ]:
# ============================================================
# ICMB-130 — FINAL CONSOLIDATION: BATCHES 01–12
# Preserve historical records; validate new batch separately
# ============================================================

import os
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

EXISTING_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_11.csv"
)

BATCH12_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CLEAN_BATCH_12.csv"
)

OUTPUT_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_12.csv"
)

# ------------------------------------------------------------
# 1. SAFETY CHECKS
# ------------------------------------------------------------

for path in [EXISTING_FILE, BATCH12_FILE]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        f"Output already exists. No overwrite: {OUTPUT_FILE}"
    )

old_df = pd.read_csv(EXISTING_FILE)
batch12_df = pd.read_csv(BATCH12_FILE)

# ------------------------------------------------------------
# 2. VALIDATE INPUT DATA
# ------------------------------------------------------------

for name, df in [
    ("Batches 01–11", old_df),
    ("Batch 12", batch12_df)
]:
    if "comment_id" not in df.columns:
        raise ValueError(f"{name}: comment_id missing.")

    if df["comment_id"].isna().any():
        raise ValueError(f"{name}: missing comment IDs.")

    df["comment_id"] = df["comment_id"].astype(str).str.strip()

    if df["comment_id"].duplicated().any():
        raise ValueError(f"{name}: duplicate comment IDs found.")

# Batch 12 must be completely clean.
batch12_empty = (
    batch12_df["comment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

if batch12_empty.any():
    raise ValueError(
        f"Batch 12 contains {batch12_empty.sum()} empty comments."
    )

# Confirm the known historical exception.
old_empty = (
    old_df["comment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

old_empty_count = int(old_empty.sum())

if old_empty_count != 1:
    raise ValueError(
        f"Expected 1 documented historical empty record; found {old_empty_count}. "
        "Investigate before consolidation."
    )

if "comment_id" not in old_df.columns:
    raise ValueError("Existing dataset comment_id column missing.")

# ------------------------------------------------------------
# 3. CROSS-BATCH DUPLICATE CHECK
# ------------------------------------------------------------

overlap = set(old_df["comment_id"]) & set(batch12_df["comment_id"])

print("Cross-batch duplicate IDs:", len(overlap))

if overlap:
    raise ValueError(
        "Duplicate comment IDs across batches. Consolidation stopped."
    )

# ------------------------------------------------------------
# 4. ALIGN AND CONCATENATE
# ------------------------------------------------------------

all_columns = list(old_df.columns)

for col in batch12_df.columns:
    if col not in all_columns:
        all_columns.append(col)

old_aligned = old_df.reindex(columns=all_columns)
batch12_aligned = batch12_df.reindex(columns=all_columns)

consolidated = pd.concat(
    [old_aligned, batch12_aligned],
    ignore_index=True
)

# ------------------------------------------------------------
# 5. FINAL VALIDATION
# ------------------------------------------------------------

expected_total = len(old_df) + len(batch12_df)

assert len(consolidated) == expected_total
assert consolidated["comment_id"].is_unique
assert consolidated["comment_id"].notna().all()

final_empty = (
    consolidated["comment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
)

assert int(final_empty.sum()) == old_empty_count
assert not (
    consolidated.iloc[len(old_df):]["comment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
).any()

print("\n" + "=" * 75)
print("CONSOLIDATION VALIDATION")
print("=" * 75)

print("Batches 01–11 records:", len(old_df))
print("Batch 12 records added:", len(batch12_df))
print("Final records:", len(consolidated))
print("Unique comment IDs:", consolidated["comment_id"].nunique())
print("Empty historical records preserved:", int(final_empty.sum()))
print("Non-empty comments:", int((~final_empty).sum()))
print("Total columns:", len(consolidated.columns))

if "movie_record_id" in consolidated.columns:
    print("Unique movies:", consolidated["movie_record_id"].nunique())

if "video_id" in consolidated.columns:
    print("Unique videos:", consolidated["video_id"].nunique())

# ------------------------------------------------------------
# 6. SAVE NEW FILE
# ------------------------------------------------------------

consolidated.to_csv(OUTPUT_FILE, index=False)

print("\nSaved:", OUTPUT_FILE)
print("Previous Batches 01–11 file remains unchanged.")
print("All historical records preserved.")

In [ ]:
# ============================================================
# ICMB-130 — UPDATED YOUTUBE COVERAGE REPORT (BATCH 01–12)
# Uses the TMDB-repaired master dataset
# ============================================================

import os
import pandas as pd
import numpy as np

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

MASTER_FILE = os.path.join(
    ROOT, "01_data", "processed",
    "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

COMMENTS_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_12.csv"
)

if not os.path.exists(MASTER_FILE):
    raise FileNotFoundError(MASTER_FILE)

if not os.path.exists(COMMENTS_FILE):
    raise FileNotFoundError(COMMENTS_FILE)

master = pd.read_csv(MASTER_FILE)
comments = pd.read_csv(COMMENTS_FILE)

print("=" * 75)
print("ICMB-130 — UPDATED YOUTUBE COVERAGE")
print("=" * 75)

print("Master dataset:", os.path.basename(MASTER_FILE))
print("Master shape:", master.shape)
print("Comments shape:", comments.shape)

# ------------------------------------------------------------
# 1. VALIDATE SCHEMAS
# ------------------------------------------------------------

master_required = [
    "movie_record_id", "movie_title", "year", "language"
]

comment_required = [
    "movie_record_id", "comment_id", "video_id", "comment_text"
]

for col in master_required:
    if col not in master.columns:
        raise ValueError(f"Master missing column: {col}")

for col in comment_required:
    if col not in comments.columns:
        raise ValueError(f"Comments missing column: {col}")

if len(master) != 130:
    raise ValueError(f"Expected 130 movies, found {len(master)}")

if master["movie_record_id"].duplicated().any():
    raise ValueError("Duplicate movie_record_id in master dataset.")

# ------------------------------------------------------------
# 2. CALCULATE MOVIE-WISE COVERAGE
# ------------------------------------------------------------

comment_counts = (
    comments.groupby("movie_record_id")
    .agg(
        total_comments=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        video_count=("video_id", "nunique")
    )
    .reset_index()
)

coverage = master.merge(
    comment_counts,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

for col in ["total_comments", "unique_comments", "video_count"]:
    coverage[col] = coverage[col].fillna(0).astype(int)

coverage["coverage_category"] = np.select(
    [
        coverage["total_comments"].eq(0),
        coverage["total_comments"].between(1, 49),
        coverage["total_comments"].between(50, 199),
        coverage["total_comments"].ge(200)
    ],
    ["No Comments", "Low", "Moderate", "Higher"],
    default="Unclassified"
)

# ------------------------------------------------------------
# 3. LANGUAGE-WISE COVERAGE
# ------------------------------------------------------------

language_summary = (
    coverage.groupby("language")
    .agg(
        total_movies=("movie_record_id", "nunique"),
        covered_movies=(
            "total_comments", lambda x: int((x > 0).sum())
        ),
        uncovered_movies=(
            "total_comments", lambda x: int((x == 0).sum())
        ),
        total_comments=("total_comments", "sum"),
        total_videos=("video_count", "sum")
    )
    .reset_index()
)

language_summary["coverage_percentage"] = (
    language_summary["covered_movies"]
    / language_summary["total_movies"] * 100
).round(2)

# ------------------------------------------------------------
# 4. OVERALL COVERAGE
# ------------------------------------------------------------

total_movies = len(coverage)
covered_movies = int((coverage["total_comments"] > 0).sum())
uncovered_movies = total_movies - covered_movies

category_summary = (
    coverage["coverage_category"]
    .value_counts()
    .reindex(
        ["No Comments", "Low", "Moderate", "Higher"],
        fill_value=0
    )
    .rename_axis("coverage_category")
    .reset_index(name="movie_count")
)

print("\nOVERALL COVERAGE")
print("-" * 75)
print("Total movies:", total_movies)
print("Covered movies:", covered_movies)
print("Uncovered movies:", uncovered_movies)
print("Coverage percentage:",
      round(covered_movies / total_movies * 100, 2), "%")
print("Comment records:", len(comments))
print("Unique comment IDs:", comments["comment_id"].nunique())
print("Unique videos:", comments["video_id"].nunique())

print("\nCOVERAGE CATEGORIES")
display(category_summary)

print("\nLANGUAGE-WISE COVERAGE")
display(language_summary)

print("\nLOWEST COVERAGE MOVIES")
display(
    coverage.sort_values(
        ["total_comments", "year"],
        ascending=[True, True]
    )[
        [
            "movie_record_id", "movie_title", "year",
            "language", "total_comments", "video_count",
            "coverage_category"
        ]
    ].head(20)
)

# ------------------------------------------------------------
# 5. SAVE REPORTS — NO OVERWRITING
# ------------------------------------------------------------

outputs = {
    "movie_coverage": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_12.csv"
    ),
    "language_coverage": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_LANGUAGE_COVERAGE_PILOT_BATCH_01_12.csv"
    ),
    "category_summary": os.path.join(
        PROCESSED_DIR,
        "ICMB_130_YOUTUBE_COVERAGE_CATEGORY_SUMMARY_BATCH_01_12.csv"
    )
}

for path in outputs.values():
    if os.path.exists(path):
        raise FileExistsError(f"Refusing to overwrite: {path}")

coverage.to_csv(outputs["movie_coverage"], index=False)
language_summary.to_csv(outputs["language_coverage"], index=False)
category_summary.to_csv(outputs["category_summary"], index=False)

print("\nUPDATED REPORTS SAVED")
for name, path in outputs.items():
    print(f"{name}: {path}")

print("\nBoth master datasets remain unchanged.")

In [ ]:
# ============================================================
# BATCH 13 — EXISTING YOUTUBE DISCOVERY HISTORY INVENTORY
# Read-only inspection; no files modified
# ============================================================

import os
import re
import pandas as pd
from datetime import datetime

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

SEARCH_DIRS = [
    os.path.join(
        ROOT, "01_data", "raw",
        "audience_sentiment", "youtube"
    ),
    os.path.join(
        ROOT, "01_data", "processed",
        "audience_sentiment"
    )
]

patterns = [
    "DISCOVERY",
    "METADATA",
    "SCREENING",
    "CANDIDATE",
    "REVIEW",
    "ACQUISITION"
]

records = []

for folder in SEARCH_DIRS:
    if not os.path.exists(folder):
        print("Directory not found:", folder)
        continue

    for base, dirs, files in os.walk(folder):
        for filename in files:

            upper_name = filename.upper()

            if (
                filename.lower().endswith(".csv")
                and any(p in upper_name for p in patterns)
            ):
                path = os.path.join(base, filename)
                stat = os.stat(path)

                batch_match = re.search(
                    r"BATCH[_\s-]*(\d+)",
                    upper_name
                )

                records.append({
                    "file_name": filename,
                    "batch_number": (
                        int(batch_match.group(1))
                        if batch_match else None
                    ),
                    "folder": os.path.relpath(base, ROOT),
                    "size_kb": round(stat.st_size / 1024, 2),
                    "modified": datetime.fromtimestamp(
                        stat.st_mtime
                    ).strftime("%Y-%m-%d %H:%M:%S"),
                    "full_path": path
                })

inventory = pd.DataFrame(records)

if inventory.empty:
    print("No matching CSV files found.")
else:
    inventory = inventory.sort_values(
        ["batch_number", "file_name"],
        na_position="first"
    ).reset_index(drop=True)

    print("=" * 75)
    print("YOUTUBE DISCOVERY HISTORY INVENTORY")
    print("=" * 75)

    print("Total matching files:", len(inventory))

    print("\nFiles by batch:")
    display(
        inventory.groupby("batch_number", dropna=False)
        .size()
        .reset_index(name="file_count")
    )

    print("\nComplete file inventory:")
    display(inventory)

print("\nInspection complete. No files modified.")

In [ ]:
# ============================================================
# BATCH 13 PLANNING — DISCOVERY HISTORY SCHEMA AUDIT
# Read-only inspection
# ============================================================

import os
import glob
import pandas as pd
import re

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

SEARCH_DIRS = [
    os.path.join(
        ROOT, "01_data", "raw",
        "audience_sentiment", "youtube"
    ),
    os.path.join(
        ROOT, "01_data", "processed",
        "audience_sentiment"
    )
]

keywords = [
    "DISCOVERY",
    "SCREENING",
    "CANDIDATE",
    "REVIEW"
]

files = []

for folder in SEARCH_DIRS:
    for base, dirs, filenames in os.walk(folder):
        for filename in filenames:
            upper = filename.upper()

            if (
                filename.lower().endswith(".csv")
                and any(k in upper for k in keywords)
                and "LOG" not in upper
                and "SUMMARY" not in upper
                and "AUDIT" not in upper
                and "PREPARATION" not in upper
                and "ALLOCATION" not in upper
            ):
                files.append(os.path.join(base, filename))

schema_records = []
frames = {}

for path in sorted(set(files)):
    try:
        df = pd.read_csv(path, low_memory=False)
        name = os.path.basename(path)

        frames[name] = df

        batch_match = re.search(
            r"BATCH[_\s-]*(\d+)",
            name.upper()
        )

        schema_records.append({
            "file_name": name,
            "batch": (
                int(batch_match.group(1))
                if batch_match else "Pilot/General"
            ),
            "rows": len(df),
            "columns": len(df.columns),
            "column_names": " | ".join(df.columns.tolist()),
            "movie_records": (
                df["movie_record_id"].nunique()
                if "movie_record_id" in df.columns else None
            ),
            "video_ids": (
                df["video_id"].nunique()
                if "video_id" in df.columns else None
            ),
            "path": path
        })

    except Exception as e:
        schema_records.append({
            "file_name": os.path.basename(path),
            "batch": "Unknown",
            "rows": None,
            "columns": None,
            "column_names": None,
            "movie_records": None,
            "video_ids": None,
            "path": path,
            "error": str(e)[:200]
        })

schema_df = pd.DataFrame(schema_records)

print("=" * 75)
print("DISCOVERY HISTORY — SCHEMA AND RECORD AUDIT")
print("=" * 75)

print("Files inspected:", len(schema_df))
print("Total rows across inspected files:",
      schema_df["rows"].fillna(0).sum())

print("\nFILE SUMMARY")
display(
    schema_df[
        [
            "file_name",
            "batch",
            "rows",
            "columns",
            "movie_records",
            "video_ids"
        ]
    ]
)

print("\nDISTINCT COLUMN SCHEMAS")
schema_groups = (
    schema_df.groupby("column_names", dropna=False)
    .agg(
        files=("file_name", "count"),
        examples=("file_name", lambda x: "; ".join(x.head(4)))
    )
    .reset_index()
)

display(schema_groups)

print("\nInspection complete. No files modified.")

In [ ]:
# ============================================================
# BATCH 13 — UNCOVERED MOVIE DISCOVERY HISTORY
# Read-only candidate history analysis
# ============================================================

import os
import re
import pandas as pd

ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

RAW_DIR = os.path.join(
    ROOT, "01_data", "raw", "audience_sentiment", "youtube"
)

PROCESSED_DIR = os.path.join(
    ROOT, "01_data", "processed", "audience_sentiment"
)

COVERAGE_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_12.csv"
)

COMMENTS_FILE = os.path.join(
    PROCESSED_DIR,
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_12.csv"
)

if not os.path.exists(COVERAGE_FILE):
    raise FileNotFoundError(COVERAGE_FILE)

if not os.path.exists(COMMENTS_FILE):
    raise FileNotFoundError(COMMENTS_FILE)

coverage = pd.read_csv(COVERAGE_FILE)
comments = pd.read_csv(COMMENTS_FILE)

# ------------------------------------------------------------
# 1. IDENTIFY UNCOVERED MOVIES
# ------------------------------------------------------------

uncovered = coverage[
    coverage["total_comments"] == 0
].copy()

uncovered_ids = set(
    uncovered["movie_record_id"].astype(str)
)

print("=" * 75)
print("UNCOVERED MOVIES — DISCOVERY HISTORY")
print("=" * 75)

print("Uncovered movies:", len(uncovered))

# ------------------------------------------------------------
# 2. DISCOVER RELEVANT HISTORY FILES
# ------------------------------------------------------------

history_files = []

for base, dirs, filenames in os.walk(RAW_DIR):

    for filename in filenames:

        upper = filename.upper()

        if not filename.lower().endswith(".csv"):
            continue

        if not any(
            word in upper
            for word in [
                "DISCOVERY",
                "SCREENING",
                "CANDIDATE",
                "REVIEW",
                "SHORTLIST"
            ]
        ):
            continue

        if any(
            word in upper
            for word in [
                "LOG",
                "AUDIT",
                "SUMMARY",
                "ALLOCATION",
                "PREPARATION"
            ]
        ):
            continue

        history_files.append(os.path.join(base, filename))

# ------------------------------------------------------------
# 3. STANDARDIZE COLUMN IDENTIFICATION
# ------------------------------------------------------------

def find_column(df, possible_names):

    normalized = {
        re.sub(r"[^a-z0-9]", "", str(c).lower()): c
        for c in df.columns
    }

    for name in possible_names:
        key = re.sub(r"[^a-z0-9]", "", name.lower())

        if key in normalized:
            return normalized[key]

    return None


movie_id_names = [
    "movie_record_id",
    "record_id"
]

video_id_names = [
    "video_id"
]

title_names = [
    "movie_title",
    "title"
]

decision_names = [
    "manual_decision",
    "final_decision",
    "review_decision",
    "automated_decision",
    "decision",
    "screening_decision"
]

status_names = [
    "manual_verification_status",
    "verification_status",
    "review_status",
    "candidate_status",
    "acquisition_status"
]

history_rows = []
file_errors = []

# ------------------------------------------------------------
# 4. EXTRACT ONLY UNCOVERED MOVIE RECORDS
# ------------------------------------------------------------

for path in sorted(set(history_files)):

    try:
        df = pd.read_csv(path, low_memory=False)

        movie_col = find_column(df, movie_id_names)
        video_col = find_column(df, video_id_names)
        title_col = find_column(df, title_names)
        decision_col = find_column(df, decision_names)
        status_col = find_column(df, status_names)

        if movie_col is None:
            continue

        temp = df.copy()

        temp["_movie_id"] = temp[movie_col].astype(str).str.strip()

        temp = temp[
            temp["_movie_id"].isin(uncovered_ids)
        ].copy()

        if temp.empty:
            continue

        for _, row in temp.iterrows():

            history_rows.append({
                "movie_record_id": row["_movie_id"],
                "movie_title": (
                    row[title_col] if title_col else None
                ),
                "video_id": (
                    str(row[video_col]).strip()
                    if video_col and pd.notna(row[video_col])
                    else None
                ),
                "decision": (
                    row[decision_col]
                    if decision_col else None
                ),
                "review_status": (
                    row[status_col]
                    if status_col else None
                ),
                "source_file": os.path.basename(path),
                "source_path": path
            })

    except Exception as e:
        file_errors.append({
            "file": os.path.basename(path),
            "error": str(e)[:200]
        })

history = pd.DataFrame(history_rows)

# ------------------------------------------------------------
# 5. EXCLUDE VIDEOS ALREADY USED FOR COMMENT ACQUISITION
# ------------------------------------------------------------

acquired_video_ids = set(
    comments["video_id"].dropna().astype(str)
)

if history.empty:
    history["already_acquired"] = pd.Series(dtype=bool)
    history["candidate_usage"] = pd.Series(dtype=str)
else:
    history["already_acquired"] = (
        history["video_id"].isin(acquired_video_ids)
    )

    history["candidate_usage"] = history[
        "already_acquired"
    ].map({
        True: "Already acquired",
        False: "Not found in consolidated comments"
    })

# ------------------------------------------------------------
# 6. MOVIE-WISE SEARCH HISTORY SUMMARY
# ------------------------------------------------------------

if not history.empty:

    movie_history = (
        history.groupby(
            ["movie_record_id", "movie_title"],
            dropna=False
        )
        .agg(
            history_records=("source_file", "size"),
            distinct_video_ids=("video_id", "nunique"),
            files=("source_file", "nunique"),
            acquired_video_ids=(
                "already_acquired",
                "sum"
            )
        )
        .reset_index()
    )

else:
    movie_history = pd.DataFrame()

summary = uncovered[
    [
        "movie_record_id",
        "movie_title",
        "year",
        "language",
        "total_comments"
    ]
].copy()

if not movie_history.empty:
    summary = summary.merge(
        movie_history,
        on=["movie_record_id", "movie_title"],
        how="left"
    )

    for col in [
        "history_records",
        "distinct_video_ids",
        "files",
        "acquired_video_ids"
    ]:
        summary[col] = summary[col].fillna(0).astype(int)

    summary["discovery_history_status"] = (
        summary["distinct_video_ids"]
        .gt(0)
        .map({
            True: "Previously searched",
            False: "No candidate history found"
        })
    )

else:
    summary["history_records"] = 0
    summary["distinct_video_ids"] = 0
    summary["files"] = 0
    summary["acquired_video_ids"] = 0
    summary["discovery_history_status"] = "No candidate history found"

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

print("\nMOVIE-WISE HISTORY SUMMARY")
display(
    summary.sort_values(
        ["distinct_video_ids", "year"],
        ascending=[True, True]
    )
)

print("\nEXISTING CANDIDATE RECORDS FOR UNCOVERED MOVIES")

if not history.empty:
    display(
        history.sort_values(
            ["movie_record_id", "video_id"]
        ).head(100)
    )
else:
    print("No matching candidate records found.")

print("\nAcquired video IDs found in history:",
      int(history["already_acquired"].sum())
      if not history.empty else 0)

print("History records:", len(history))
print("Distinct historical video IDs:",
      history["video_id"].nunique()
      if not history.empty else 0)

if file_errors:
    print("\nFiles that could not be read:")
    display(pd.DataFrame(file_errors))

print("\nRead-only inspection complete. No files modified.")

In [ ]:
# ============================================================
# BATCH 13 — HISTORICAL CANDIDATE REVIEW
# Independent reconstruction | Read-only | No API calls
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
RAW_YT_DIR = PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"

assert RAW_YT_DIR.exists(), f"YouTube folder not found: {RAW_YT_DIR}"

searched_movies = {
    "2004_4": "Mujhse Shaadi Karogi",
    "2010_3": "Golmaal 3",
    "2010_4": "Raajneeti",
    "2011_2": "Ready",
    "2012_2": "Dabangg 2",
    "2012_3": "Rowdy Rathore",
    "2012_5": "Housefull 2",
    "2019_1": "War"
}

# Read candidate, discovery, screening and review files
candidate_frames = []
files_scanned = 0

for path in RAW_YT_DIR.rglob("*.csv"):
    name = path.name.lower()

    if not any(term in name for term in [
        "discovery", "screening", "candidate",
        "manual_review", "shortlist", "decision_table"
    ]):
        continue

    if any(term in name for term in [
        "log", "audit", "summary", "allocation", "preparation"
    ]):
        continue

    files_scanned += 1

    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception:
        continue

    # Identify movie ID and video ID columns
    normalized = {str(c).strip().lower(): c for c in df.columns}
    movie_col = normalized.get("movie_record_id")
    video_col = normalized.get("video_id")

    if movie_col is None or video_col is None:
        continue

    subset = df[
        df[movie_col].astype(str).isin(searched_movies)
    ].copy()

    if subset.empty:
        continue

    subset["source_file"] = path.name
    subset["movie_record_id"] = subset[movie_col].astype(str)
    subset["video_id"] = subset[video_col].astype(str).str.strip()

    # Standardise useful fields where present
    for target, possibilities in {
        "candidate_title": [
            "video_title", "metadata_title", "title"
        ],
        "channel": ["channel_title", "metadata_channel_title"],
        "decision": ["decision", "screening_decision", "automated_decision"],
        "review_status": ["review_status", "manual_review_status"],
        "duration": ["duration"],
        "views": ["view_count"],
        "comment_count": ["comment_count"],
        "video_type": ["video_type", "source_type", "content_type"],
        "youtube_url": ["youtube_url"]
    }.items():
        source = next(
            (normalized[c] for c in possibilities if c in normalized),
            None
        )
        subset[target] = subset[source] if source else None

    candidate_frames.append(subset[[
        "movie_record_id", "video_id", "candidate_title",
        "channel", "decision", "review_status", "duration",
        "views", "comment_count", "video_type", "youtube_url",
        "source_file"
    ]])

print("Candidate/review CSV files scanned:", files_scanned)

if not candidate_frames:
    raise ValueError(
        "No matching historical candidate records found. "
        "Check the folder and CSV schemas."
    )

all_history = pd.concat(candidate_frames, ignore_index=True)

all_history = all_history[
    all_history["video_id"].notna()
    & all_history["video_id"].ne("")
].copy()

# One row per movie/video pair; preserve all historical source stages
candidate_summary = (
    all_history.groupby(
        ["movie_record_id", "video_id"], as_index=False
    )
    .agg(
        candidate_title=("candidate_title", "first"),
        channel=("channel", "first"),
        decisions=("decision", lambda x: " | ".join(
            sorted(set(str(v) for v in x.dropna()
                       if str(v).strip().lower() not in ["", "none", "nan"]))
        )),
        review_statuses=("review_status", lambda x: " | ".join(
            sorted(set(str(v) for v in x.dropna()
                       if str(v).strip().lower() not in ["", "none", "nan"]))
        )),
        duration=("duration", "first"),
        views=("views", "first"),
        comment_count=("comment_count", "first"),
        video_type=("video_type", "first"),
        youtube_url=("youtube_url", "first"),
        source_files=("source_file", lambda x: " | ".join(sorted(set(x))))
    )
)

candidate_summary["movie_title"] = (
    candidate_summary["movie_record_id"].map(searched_movies)
)

# Check against the consolidated acquired-comment dataset
CONSOLIDATED_FILE = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_12.csv"
)

assert CONSOLIDATED_FILE.exists(), (
    f"Consolidated comments file not found: {CONSOLIDATED_FILE}"
)

acquired = pd.read_csv(
    CONSOLIDATED_FILE, usecols=["video_id"], low_memory=False
)
acquired_ids = set(acquired["video_id"].dropna().astype(str))

candidate_summary["already_acquired"] = (
    candidate_summary["video_id"].isin(acquired_ids)
)

print("\n" + "=" * 75)
print("HISTORICAL CANDIDATE SUMMARY")
print("=" * 75)

display(
    candidate_summary.groupby(
        ["movie_record_id", "movie_title"], as_index=False
    ).agg(
        distinct_video_ids=("video_id", "nunique"),
        acquired_video_ids=("already_acquired", "sum")
    )
)

print("\n" + "=" * 75)
print("CONSOLIDATED DISTINCT CANDIDATES")
print("=" * 75)

display(
    candidate_summary.sort_values(
        ["movie_record_id", "video_id"]
    ).reset_index(drop=True)
)

print("\nTotal historical source-stage rows:", len(all_history))
print("Distinct movie/video pairs:", len(candidate_summary))
print("Distinct video IDs:", candidate_summary["video_id"].nunique())
print("Already acquired video IDs:", candidate_summary.loc[
    candidate_summary["already_acquired"], "video_id"
].nunique())

print("\nRead-only inspection complete. No files modified and no API quota used.")

In [ ]:
# ============================================================
# BATCH 13 — HISTORICAL SCREENING DECISION AUDIT
# Read-only | No API calls | No files modified
# ============================================================

import pandas as pd

assert "all_history" in globals(), "all_history not found."
assert "candidate_summary" in globals(), "candidate_summary not found."

# Keep only records with an actual screening/review decision
decision_data = all_history.copy()

decision_data["decision_clean"] = (
    decision_data["decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)

decision_data = decision_data[
    decision_data["decision_clean"].ne("")
    & ~decision_data["decision_clean"].str.lower().isin(["none", "nan"])
].copy()

print("=" * 75)
print("HISTORICAL DECISION DISTRIBUTION")
print("=" * 75)

if decision_data.empty:
    print("No screening decisions found in the historical records.")
else:
    display(pd.crosstab(
        decision_data["movie_record_id"],
        decision_data["decision_clean"],
        margins=True
    ))

# Summarise decisions at the unique movie/video level
video_decisions = (
    decision_data.groupby(
        ["movie_record_id", "video_id"], as_index=False
    )
    .agg(
        decisions=("decision_clean", lambda x: " | ".join(
            sorted(set(x))
        )),
        decision_records=("decision_clean", "size"),
        decision_source_files=("source_file", lambda x: " | ".join(
            sorted(set(x))
        ))
    )
)

# Identify videos with multiple different decisions
video_decisions["conflicting_decisions"] = (
    video_decisions["decisions"].str.contains(r"\|", regex=True)
)

print("\n" + "=" * 75)
print("CANDIDATES WITH HISTORICAL SCREENING DECISIONS")
print("=" * 75)

display(
    video_decisions.merge(
        candidate_summary[
            ["movie_record_id", "video_id", "movie_title", "candidate_title",
             "channel", "duration", "views", "comment_count"]
        ],
        on=["movie_record_id", "video_id"],
        how="left"
    ).sort_values(["movie_record_id", "video_id"])
)

print("\n" + "=" * 75)
print("CONFLICTING DECISIONS")
print("=" * 75)

display(video_decisions[
    video_decisions["conflicting_decisions"]
])

print("\n" + "=" * 75)
print("MOVIES WITH NO POSITIVE HISTORICAL DECISION")
print("=" * 75)

positive_terms = [
    "shortlist", "approved", "include", "accept",
    "high confidence", "selected"
]

for movie_id, movie_name in searched_movies.items():
    movie_rows = video_decisions[
        video_decisions["movie_record_id"] == movie_id
    ]

    decisions = " ".join(movie_rows["decisions"].str.lower())

    has_positive = any(term in decisions for term in positive_terms)

    print(
        f"{movie_id} | {movie_name} | "
        f"{'Positive decision exists' if has_positive else 'No positive decision found'}"
    )

print("\nRead-only audit complete.")

In [ ]:
# ============================================================
# BATCH 13 — HISTORICAL SHORTLIST EXTRACTION
# Read-only | No API calls | No files modified
# ============================================================

assert "all_history" in globals()
assert "candidate_summary" in globals()

# Extract exact Shortlist decisions only.
# Exclude "Hold", "Hold for review", and all exclusion decisions.
shortlisted_history = all_history.copy()

shortlisted_history["decision_clean"] = (
    shortlisted_history["decision"]
    .fillna("")
    .astype(str)
    .str.strip()
)

shortlisted_history = shortlisted_history[
    shortlisted_history["decision_clean"].str.lower().eq("shortlist")
].copy()

# Retain unique movie/video combinations
shortlist_ids = (
    shortlisted_history[
        ["movie_record_id", "video_id"]
    ]
    .drop_duplicates()
)

shortlist_review = shortlist_ids.merge(
    candidate_summary,
    on=["movie_record_id", "video_id"],
    how="left"
)

# Add the actual source files where Shortlist was assigned
shortlist_sources = (
    shortlisted_history.groupby(
        ["movie_record_id", "video_id"]
    )["source_file"]
    .apply(lambda x: " | ".join(sorted(set(x.dropna().astype(str)))))
    .reset_index(name="shortlist_decision_sources")
)

shortlist_review = shortlist_review.merge(
    shortlist_sources,
    on=["movie_record_id", "video_id"],
    how="left"
)

# Add acquisition status
if "already_acquired" not in shortlist_review.columns:
    shortlist_review["already_acquired"] = False

print("UNIQUE HISTORICAL SHORTLISTED CANDIDATES:",
      len(shortlist_review))

print("\nSHORTLIST COUNT BY MOVIE:")
display(
    shortlist_review.groupby(
        ["movie_record_id", "movie_title"]
    ).size().reset_index(name="unique_shortlisted_videos")
)

print("\nCANDIDATE DETAILS:")
display(
    shortlist_review[[
        "movie_record_id",
        "movie_title",
        "video_id",
        "candidate_title",
        "channel",
        "duration",
        "views",
        "comment_count",
        "decisions",
        "source_files",
        "shortlist_decision_sources",
        "already_acquired"
    ]].sort_values(["movie_record_id", "video_id"])
)

print("\nNo files modified. No YouTube API quota used.")

In [ ]:
# ============================================================
# BATCH 13 — FINAL METADATA CHECK OF HISTORICAL CANDIDATES
# One videos.list request | No new search.list calls
# ============================================================

import os
import pandas as pd
from datetime import datetime, timezone
from googleapiclient.discovery import build
from google.colab import userdata

PROJECT_ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

candidate_map = {
    "2011_2": {
        "movie_title": "Ready",
        "video_ids": ["149jpjWmMzg"]
    },
    "2010_4": {
        "movie_title": "Raajneeti",
        "video_ids": ["F4_Lfln6k-o"]
    },
    "2012_5": {
        "movie_title": "Housefull 2",
        "video_ids": [
            "E-Rwdu1g550",
            "ZXwlWfuI4Ug",
            "jCP3DCwhSHk",
            "o3s0PD40YcM",
            "t79Q6-OBwqE"
        ]
    }
}

video_to_movie = {
    vid: (record_id, details["movie_title"])
    for record_id, details in candidate_map.items()
    for vid in details["video_ids"]
}

assert len(video_to_movie) == 7

# Retrieve API key securely from Colab Secrets
api_key = userdata.get("YOUTUBE_API_KEY")
assert api_key, "YOUTUBE_API_KEY not found in Colab Secrets."

youtube = build(
    "youtube", "v3",
    developerKey=api_key,
    cache_discovery=False
)

response = youtube.videos().list(
    part="snippet,contentDetails,statistics,status",
    id=",".join(video_to_movie.keys())
).execute()

items = response.get("items", [])
metadata_rows = []

for item in items:
    vid = item["id"]
    record_id, movie_title = video_to_movie[vid]
    snippet = item.get("snippet", {})
    stats = item.get("statistics", {})
    status = item.get("status", {})

    metadata_rows.append({
        "movie_record_id": record_id,
        "movie_title": movie_title,
        "video_id": vid,
        "video_title": snippet.get("title"),
        "channel_title": snippet.get("channelTitle"),
        "published_at": snippet.get("publishedAt"),
        "duration": item.get("contentDetails", {}).get("duration"),
        "view_count": stats.get("viewCount"),
        "like_count": stats.get("likeCount"),
        "comment_count": stats.get("commentCount"),
        "privacy_status": status.get("privacyStatus"),
        "upload_status": status.get("uploadStatus"),
        "metadata_check_status": "Available"
    })

found_ids = {row["video_id"] for row in metadata_rows}

for vid, (record_id, movie_title) in video_to_movie.items():
    if vid not in found_ids:
        metadata_rows.append({
            "movie_record_id": record_id,
            "movie_title": movie_title,
            "video_id": vid,
            "metadata_check_status": "Not returned by videos.list"
        })

metadata_check = pd.DataFrame(metadata_rows)

print("=" * 75)
print("BATCH 13 — HISTORICAL CANDIDATE METADATA CHECK")
print("=" * 75)

print("Requested video IDs:", len(video_to_movie))
print("Returned metadata records:", len(items))
print("Not returned:", len(video_to_movie) - len(items))
print("Checked at UTC:", datetime.now(timezone.utc).isoformat())

display(metadata_check.sort_values(
    ["movie_record_id", "video_id"]
).reset_index(drop=True))

print("\nNo search.list requests were made.")
print("No files were modified.")

In [ ]:
# ============================================================
# BATCH 13 — HISTORICAL CANDIDATE COMMENT ACQUISITION
# 7 approved-for-testing videos | Maximum 100 comments/video
# ============================================================

import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import userdata
import re

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = (
    RAW_DIR / "ICMB_130_YOUTUBE_COMMENTS_BATCH_13_HISTORICAL.csv"
)

LOG_FILE = (
    RAW_DIR / "ICMB_130_YOUTUBE_COMMENTS_ACQUISITION_LOG_BATCH_13_HISTORICAL.csv"
)

# Protect existing data
if OUTPUT_FILE.exists() or LOG_FILE.exists():
    raise FileExistsError(
        "Batch 13 output already exists. No files were overwritten."
    )

candidate_map = {
    "F4_Lfln6k-o": ("2010_4", "Raajneeti", "Aura Of Stories"),
    "149jpjWmMzg": ("2011_2", "Ready", "Bollywood Hungama"),
    "E-Rwdu1g550": ("2012_5", "Housefull 2", "React Bro"),
    "ZXwlWfuI4Ug": ("2012_5", "Housefull 2", "React Bro"),
    "jCP3DCwhSHk": ("2012_5", "Housefull 2", "React Bro"),
    "o3s0PD40YcM": ("2012_5", "Housefull 2", "React Bro"),
    "t79Q6-OBwqE": ("2012_5", "Housefull 2", "React Bro")
}

api_key = userdata.get("YOUTUBE_API_KEY")
assert api_key, "YOUTUBE_API_KEY not found in Colab Secrets."

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

comment_rows = []
log_rows = []

for video_id, (record_id, movie_title, channel) in candidate_map.items():

    retrieved = 0
    status = "Success"
    error_message = ""

    try:
        response = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            order="time",
            textFormat="plainText"
        ).execute()

        while True:
            for item in response.get("items", []):
                top = item["snippet"]["topLevelComment"]
                snippet = top["snippet"]

                comment_rows.append({
                    "movie_record_id": record_id,
                    "movie_title": movie_title,
                    "video_id": video_id,
                    "channel_title": channel,
                    "comment_id": top["id"],
                    "comment_text": snippet.get("textDisplay"),
                    "author_display_name": snippet.get("authorDisplayName"),
                    "published_at": snippet.get("publishedAt"),
                    "updated_at": snippet.get("updatedAt"),
                    "like_count": snippet.get("likeCount", 0),
                    "comment_acquisition_order": "time",
                    "comment_level": "top_level",
                    "acquired_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat()
                })

                retrieved += 1

            next_token = response.get("nextPageToken")

            if not next_token or retrieved >= 100:
                break

            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, 100 - retrieved),
                pageToken=next_token,
                order="time",
                textFormat="plainText"
            ).execute()

    except HttpError as e:
        status = "API Error"
        # Do not print raw API error details or URLs.
        error_message = f"HTTP {getattr(e.resp, 'status', 'unknown')}"

    except Exception as e:
        status = "Error"
        error_message = type(e).__name__

    log_rows.append({
        "movie_record_id": record_id,
        "movie_title": movie_title,
        "video_id": video_id,
        "channel_title": channel,
        "comments_retrieved": retrieved,
        "acquisition_status": status,
        "error_type": error_message,
        "acquired_at_utc": datetime.now(
            timezone.utc
        ).isoformat()
    })

    print(
        f"{movie_title:15} | {video_id} | "
        f"{retrieved:3} comments | {status}"
    )

# Construct outputs
comments_df = pd.DataFrame(comment_rows)
log_df = pd.DataFrame(log_rows)

# Save raw acquisition and log separately
comments_df.to_csv(OUTPUT_FILE, index=False)
log_df.to_csv(LOG_FILE, index=False)

print("\n" + "=" * 75)
print("BATCH 13 HISTORICAL ACQUISITION SUMMARY")
print("=" * 75)

print("Videos attempted:", len(candidate_map))
print("Videos successful:", (log_df["acquisition_status"] == "Success").sum())
print("Videos with errors:", (log_df["acquisition_status"] != "Success").sum())
print("Total comments retrieved:", len(comments_df))
print("Unique comment IDs:", comments_df["comment_id"].nunique())
print("Unique video IDs:", comments_df["video_id"].nunique())

print("\nMovie-wise acquisition:")
display(
    log_df.groupby("movie_title", as_index=False).agg(
        videos_attempted=("video_id", "nunique"),
        comments_retrieved=("comments_retrieved", "sum")
    )
)

print("\nSaved files:")
print(OUTPUT_FILE)
print(LOG_FILE)

print(
    "\nSampling note: comments are top-level comments ordered by "
    "recency, not a random audience sample. Replies were not retrieved."
)

In [ ]:
# ============================================================
# BATCH 13 — COMMENT QUALITY AND DUPLICATE AUDIT
# Read-only | No files modified
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment"
)

BATCH13_FILE = (
    RAW_DIR / "ICMB_130_YOUTUBE_COMMENTS_BATCH_13_HISTORICAL.csv"
)

PREVIOUS_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_12.csv"
)

assert BATCH13_FILE.exists()
assert PREVIOUS_FILE.exists()

batch13 = pd.read_csv(BATCH13_FILE, low_memory=False)
previous = pd.read_csv(PREVIOUS_FILE, low_memory=False)

# Basic integrity
print("=" * 75)
print("BATCH 13 BASIC INTEGRITY")
print("=" * 75)

print("Rows:", len(batch13))
print("Unique comment IDs:", batch13["comment_id"].nunique())
print("Unique video IDs:", batch13["video_id"].nunique())
print("Missing comment IDs:", batch13["comment_id"].isna().sum())
print("Missing video IDs:", batch13["video_id"].isna().sum())

# Duplicate checks
batch13_duplicate_ids = batch13[
    batch13["comment_id"].duplicated(keep=False)
].copy()

previous_ids = set(previous["comment_id"].dropna().astype(str))
batch13_ids = batch13["comment_id"].dropna().astype(str)

cross_batch_duplicates = batch13[
    batch13["comment_id"].astype(str).isin(previous_ids)
].copy()

print("\n" + "=" * 75)
print("DUPLICATE AUDIT")
print("=" * 75)

print("Duplicate rows within Batch 13:", len(batch13_duplicate_ids))
print("Cross-batch duplicate rows:", len(cross_batch_duplicates))

# Empty text check
batch13["comment_text"] = batch13["comment_text"].fillna("").astype(str)

empty_text = batch13[
    batch13["comment_text"].str.strip().eq("")
].copy()

print("\n" + "=" * 75)
print("COMMENT TEXT QUALITY")
print("=" * 75)

print("Empty comment text rows:", len(empty_text))
print("Non-empty comment rows:", len(batch13) - len(empty_text))

# Movie-wise summary
print("\n" + "=" * 75)
print("MOVIE-WISE SUMMARY")
print("=" * 75)

display(
    batch13.groupby("movie_title", as_index=False).agg(
        comments=("comment_id", "size"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique"),
        empty_text_comments=(
            "comment_text",
            lambda x: x.str.strip().eq("").sum()
        )
    )
)

# Video-wise summary
print("\n" + "=" * 75)
print("VIDEO-WISE SUMMARY")
print("=" * 75)

display(
    batch13.groupby(
        ["movie_title", "video_id", "channel_title"],
        as_index=False
    ).agg(
        comments=("comment_id", "size"),
        empty_comments=(
            "comment_text",
            lambda x: x.str.strip().eq("").sum()
        )
    )
)

# Duplicate details if any
if len(batch13_duplicate_ids):
    print("\nDUPLICATES WITHIN BATCH 13:")
    display(batch13_duplicate_ids)

if len(cross_batch_duplicates):
    print("\nDUPLICATES AGAINST BATCHES 01–12:")
    display(cross_batch_duplicates)

if len(empty_text):
    print("\nEMPTY COMMENT RECORDS:")
    display(empty_text)

print("\nRead-only audit complete. No files modified.")

In [ ]:
# ============================================================
# BATCH 13 — CLEANING AND CONSOLIDATION (BATCHES 01–13)
# Raw files preserved | Existing consolidated file preserved
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = (
    PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube/comments"
)
PROCESSED_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

RAW_FILE = (
    RAW_DIR / "ICMB_130_YOUTUBE_COMMENTS_BATCH_13_HISTORICAL.csv"
)
OLD_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_12.csv"
)
CLEAN_FILE = (
    PROCESSED_DIR / "ICMB_130_YOUTUBE_COMMENTS_CLEAN_BATCH_13.csv"
)
NEW_FILE = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_13.csv"
)

# Safeguard against overwriting previous work
for path in [RAW_FILE, OLD_FILE]:
    assert path.exists(), f"Required input missing: {path}"

for path in [CLEAN_FILE, NEW_FILE]:
    assert not path.exists(), (
        f"Output already exists; refusing to overwrite: {path}"
    )

batch13 = pd.read_csv(RAW_FILE, low_memory=False)
previous = pd.read_csv(OLD_FILE, low_memory=False)

# Validate required identifiers
required = ["comment_id", "video_id", "comment_text"]

for col in required:
    assert col in batch13.columns, f"Missing Batch 13 column: {col}"
    assert col in previous.columns, f"Missing consolidated column: {col}"

# Validate Batch 13 before cleaning
assert batch13["comment_id"].notna().all()
assert batch13["video_id"].notna().all()
assert batch13["comment_id"].astype(str).is_unique
assert batch13["comment_text"].fillna("").astype(str).str.strip().ne("").all()

old_ids = set(previous["comment_id"].dropna().astype(str))
new_ids = set(batch13["comment_id"].dropna().astype(str))

cross_duplicates = old_ids.intersection(new_ids)

assert len(cross_duplicates) == 0, (
    f"Cross-batch duplicate IDs detected: {len(cross_duplicates)}"
)

# Align schemas without dropping existing columns
all_columns = list(dict.fromkeys(
    list(previous.columns) + list(batch13.columns)
))

previous_aligned = previous.reindex(columns=all_columns)
batch13_aligned = batch13.reindex(columns=all_columns)

# Preserve all valid records
clean_batch13 = batch13_aligned.copy()

consolidated = pd.concat(
    [previous_aligned, clean_batch13],
    ignore_index=True
)

# Final integrity checks
assert consolidated["comment_id"].notna().all()
assert consolidated["comment_id"].astype(str).is_unique

# Save separate processed outputs
clean_batch13.to_csv(CLEAN_FILE, index=False)
consolidated.to_csv(NEW_FILE, index=False)

print("=" * 75)
print("BATCH 13 CLEANING AND CONSOLIDATION COMPLETE")
print("=" * 75)

print("Previous consolidated rows:", len(previous))
print("Batch 13 raw rows:", len(batch13))
print("Batch 13 clean rows:", len(clean_batch13))
print("Cross-batch duplicate IDs:", len(cross_duplicates))

print("\nNEW CONSOLIDATED DATASET")
print("Shape:", consolidated.shape)
print("Total comments:", len(consolidated))
print("Unique comment IDs:", consolidated["comment_id"].nunique())
print("Unique video IDs:", consolidated["video_id"].nunique())

print("\nMovie coverage:")
print("Covered movies:", consolidated["movie_record_id"].nunique())

print("\nSaved clean Batch 13:", CLEAN_FILE)
print("Saved consolidated Batches 01–13:", NEW_FILE)

print("\nRaw Batch 13 and previous consolidated file remain unchanged.")

In [ ]:
# ============================================================
# BATCH 13 — FINAL MOVIE AND LANGUAGE COVERAGE UPDATE
# Read-only inputs | New report outputs
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED = PROJECT_ROOT / "01_data/processed"
AUDIENCE = PROCESSED / "audience_sentiment"

MASTER_FILE = (
    PROCESSED / "ICMB_130_MASTER_DATA_TMDB_REPAIRED.csv"
)

COMMENTS_FILE = (
    AUDIENCE
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_13.csv"
)

OUTPUTS = {
    "movie": AUDIENCE / "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_13.csv",
    "language": AUDIENCE / "ICMB_130_YOUTUBE_LANGUAGE_COVERAGE_PILOT_BATCH_01_13.csv",
    "category": AUDIENCE / "ICMB_130_YOUTUBE_COVERAGE_CATEGORY_SUMMARY_BATCH_01_13.csv"
}

for path in [MASTER_FILE, COMMENTS_FILE]:
    assert path.exists(), f"Required input missing: {path}"

for path in OUTPUTS.values():
    assert not path.exists(), f"Refusing to overwrite: {path}"

master = pd.read_csv(MASTER_FILE, low_memory=False)
comments = pd.read_csv(COMMENTS_FILE, low_memory=False)

# Identify required master columns
required_master = ["movie_record_id", "movie_title", "year", "language"]
for col in required_master:
    assert col in master.columns, f"Master column missing: {col}"

assert "movie_record_id" in comments.columns

# Movie-wise comment counts
movie_counts = (
    comments.groupby("movie_record_id")
    .agg(
        total_comments=("comment_id", "count"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_video_ids=("video_id", "nunique")
    )
    .reset_index()
)

coverage = master.merge(
    movie_counts,
    on="movie_record_id",
    how="left"
)

for col in ["total_comments", "unique_comment_ids", "unique_video_ids"]:
    coverage[col] = coverage[col].fillna(0).astype(int)

coverage["coverage_status"] = coverage["total_comments"].apply(
    lambda x: "Uncovered" if x == 0 else "Covered"
)

coverage["coverage_category"] = pd.cut(
    coverage["total_comments"],
    bins=[-1, 0, 49, 199, float("inf")],
    labels=["No Comments", "Low", "Moderate", "Higher"]
).astype(str)

# Language-wise summary
language_coverage = (
    coverage.groupby("language", as_index=False, observed=True)
    .agg(
        total_movies=("movie_record_id", "nunique"),
        covered_movies=("coverage_status", lambda x: (x == "Covered").sum()),
        uncovered_movies=("coverage_status", lambda x: (x == "Uncovered").sum()),
        total_comments=("total_comments", "sum"),
        total_videos=("unique_video_ids", "sum")
    )
)

language_coverage["coverage_percentage"] = (
    language_coverage["covered_movies"]
    / language_coverage["total_movies"] * 100
).round(2)

# Category summary
category_summary = (
    coverage.groupby("coverage_category", observed=False)
    .agg(
        movies=("movie_record_id", "nunique"),
        total_comments=("total_comments", "sum")
    )
    .reset_index()
)

# Save reports
coverage.to_csv(OUTPUTS["movie"], index=False)
language_coverage.to_csv(OUTPUTS["language"], index=False)
category_summary.to_csv(OUTPUTS["category"], index=False)

# Final validation and summary
total_movies = coverage["movie_record_id"].nunique()
covered = (coverage["total_comments"] > 0).sum()
uncovered = total_movies - covered

print("=" * 75)
print("OFFICIAL BATCH 13 COVERAGE BASELINE")
print("=" * 75)

print("Master dataset movies:", total_movies)
print("Consolidated comments:", len(comments))
print("Unique comment IDs:", comments["comment_id"].nunique())
print("Unique video IDs:", comments["video_id"].nunique())
print("Covered movies:", covered)
print("Uncovered movies:", uncovered)
print("Coverage percentage:", round(covered / total_movies * 100, 2), "%")

print("\nLANGUAGE-WISE COVERAGE")
display(language_coverage)

print("\nCOVERAGE CATEGORY DISTRIBUTION")
display(category_summary)

print("\nREMAINING UNCOVERED MOVIES")
display(
    coverage.loc[
        coverage["total_comments"] == 0,
        ["movie_record_id", "movie_title", "year", "language"]
    ].sort_values(["year", "movie_title"])
)

print("\nSAVED REPORTS")
for name, path in OUTPUTS.items():
    print(name, ":", path)

print("\nCoverage report generation complete.")

In [ ]:
# ============================================================
# FAST ACQUISITION WORKFLOW — BATCH 14 PLANNING
# Current uncovered movies + historical discovery inventory
# Read-only | No API calls | No files modified
# ============================================================

import pandas as pd
from pathlib import Path

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
AUD = ROOT / "01_data/processed/audience_sentiment"
RAW = ROOT / "01_data/raw/audience_sentiment/youtube"

coverage_file = AUD / "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_13.csv"
comments_file = AUD / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_13.csv"

coverage = pd.read_csv(coverage_file, low_memory=False)
comments = pd.read_csv(comments_file, low_memory=False)

uncovered = coverage[
    coverage["total_comments"].fillna(0).eq(0)
].copy()

# Collect historical candidate records for currently uncovered movies
frames = []
files_scanned = 0

skip_terms = ["log", "audit", "summary", "allocation", "preparation"]

for path in RAW.rglob("*.csv"):
    name = path.name.lower()

    if not any(term in name for term in [
        "discovery", "screening", "candidate",
        "manual_review", "shortlist", "decision_table"
    ]):
        continue

    if any(term in name for term in skip_terms):
        continue

    try:
        df = pd.read_csv(path, low_memory=False)
    except Exception:
        continue

    cols = {str(c).strip().lower(): c for c in df.columns}
    movie_col = cols.get("movie_record_id")
    video_col = cols.get("video_id")

    if movie_col is None or video_col is None:
        continue

    part = df[
        df[movie_col].astype(str).isin(
            uncovered["movie_record_id"].astype(str)
        )
    ].copy()

    if part.empty:
        continue

    files_scanned += 1
    part["movie_record_id"] = part[movie_col].astype(str)
    part["video_id"] = part[video_col].astype(str).str.strip()
    part["source_file"] = path.name

    decision_col = next(
        (cols[c] for c in [
            "decision", "screening_decision", "automated_decision"
        ] if c in cols),
        None
    )
    part["decision"] = (
        part[decision_col].fillna("").astype(str).str.strip()
        if decision_col else ""
    )

    frames.append(part[[
        "movie_record_id", "video_id", "decision", "source_file"
    ]])

history = (
    pd.concat(frames, ignore_index=True)
    if frames else
    pd.DataFrame(columns=[
        "movie_record_id", "video_id", "decision", "source_file"
    ])
)

history = history[
    history["video_id"].ne("")
    & history["video_id"].ne("nan")
]

# Summarise unique historical candidates per movie
if not history.empty:
    history_summary = history.groupby(
        "movie_record_id"
    ).agg(
        historical_video_ids=("video_id", "nunique"),
        source_stage_records=("video_id", "size"),
        shortlisted_records=(
            "decision",
            lambda x: x.str.lower().eq("shortlist").sum()
        ),
        held_records=(
            "decision",
            lambda x: x.str.lower().str.contains("hold").sum()
        ),
        excluded_records=(
            "decision",
            lambda x: x.str.lower().str.contains("exclude").sum()
        )
    ).reset_index()
else:
    history_summary = pd.DataFrame(
        columns=[
            "movie_record_id", "historical_video_ids",
            "source_stage_records", "shortlisted_records",
            "held_records", "excluded_records"
        ]
    )

plan = uncovered[[
    "movie_record_id", "movie_title", "year", "language"
]].merge(history_summary, on="movie_record_id", how="left")

count_cols = [
    "historical_video_ids", "source_stage_records",
    "shortlisted_records", "held_records", "excluded_records"
]
plan[count_cols] = plan[count_cols].fillna(0).astype(int)

plan["next_action"] = plan["historical_video_ids"].apply(
    lambda n: "Reassess existing candidates" if n > 0
    else "Fresh discovery required"
)

plan = plan.sort_values(
    ["historical_video_ids", "year"],
    ascending=[False, True]
).reset_index(drop=True)

print("=" * 75)
print("BATCH 14 — CURRENT ACQUISITION PLAN")
print("=" * 75)
print("Uncovered movies:", len(uncovered))
print("Movies with historical candidates:",
      (plan["historical_video_ids"] > 0).sum())
print("Movies needing fresh discovery:",
      (plan["historical_video_ids"] == 0).sum())
print("Historical candidate/review files scanned:", files_scanned)
print("Existing consolidated comments:", len(comments))
print("Existing unique comment IDs:", comments["comment_id"].nunique())

display(plan)

print("\nPRIORITY SUMMARY")
print(plan.groupby("next_action").agg(
    movies=("movie_record_id", "count"),
    historical_candidates=("historical_video_ids", "sum")
).reset_index())

print("\nRead-only planning complete.")

In [ ]:
# ============================================================
# BATCH 14 — HISTORICAL CANDIDATE REASSESSMENT
# Read-only | No API calls | No files modified
# ============================================================

from pathlib import Path
import pandas as pd
import re

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
RAW_YT = ROOT / "01_data/raw/audience_sentiment/youtube"

TARGET_MOVIES = {
    "2004_4": "Mujhse Shaadi Karogi",
    "2010_3": "Golmaal 3",
    "2012_2": "Dabangg 2",
    "2012_3": "Rowdy Rathore",
    "2019_1": "War"
}

# Load existing consolidated comments
CONSOLIDATED = (
    ROOT / "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_13.csv"
)
comments = pd.read_csv(CONSOLIDATED, low_memory=False)

acquired_ids = set(
    comments["video_id"].dropna().astype(str).str.strip()
)

# Locate relevant historical CSV files
files = [
    p for p in RAW_YT.rglob("*.csv")
    if any(k in p.name.lower() for k in
           ["discovery", "screening", "candidate",
            "shortlist", "decision", "manual_review"])
    and not any(k in p.name.lower() for k in
                ["audit", "summary", "allocation", "log",
                 "report", "preparation"])
]

frames = []

for file in files:
    try:
        df = pd.read_csv(file, low_memory=False)
        df.columns = [str(c).strip().lower() for c in df.columns]

        movie_col = next(
            (c for c in ["movie_record_id", "record_id"] if c in df.columns),
            None
        )
        video_col = next(
            (c for c in ["video_id", "youtube_video_id"] if c in df.columns),
            None
        )

        if not movie_col or not video_col:
            continue

        temp = df.copy()
        temp["_file"] = file.name
        temp["_movie_id"] = temp[movie_col].astype(str).str.strip()
        temp["_video_id"] = temp[video_col].astype(str).str.strip()

        temp = temp[
            temp["_movie_id"].isin(TARGET_MOVIES)
            & temp["_video_id"].ne("")
            & temp["_video_id"].ne("nan")
        ]

        if not temp.empty:
            frames.append(temp)

    except Exception as e:
        print(f"Skipped unreadable file: {file.name} ({type(e).__name__})")

if not frames:
    raise ValueError("No historical candidate records were found.")

history = pd.concat(frames, ignore_index=True)

# Identify available metadata and decision columns
decision_cols = [
    c for c in ["decision", "screening_decision", "automated_decision"]
    if c in history.columns
]
title_col = next(
    (c for c in ["video_title", "title"] if c in history.columns), None
)
channel_col = next(
    (c for c in ["channel_title", "channel_name"] if c in history.columns),
    None
)
views_col = next(
    (c for c in ["view_count", "views"] if c in history.columns), None
)
duration_col = next(
    (c for c in ["duration", "duration_seconds"] if c in history.columns),
    None
)
comments_col = next(
    (c for c in ["comment_count", "comments"] if c in history.columns),
    None
)

# One record per movie/video; retain all historical decision evidence
rows = []

for (movie_id, video_id), group in history.groupby(
    ["_movie_id", "_video_id"], dropna=False
):
    decisions = []

    for col in decision_cols:
        decisions.extend(
            group[col].dropna().astype(str).str.strip().tolist()
        )

    decisions = sorted(set(d for d in decisions if d))

    def first_value(col):
        if col and col in group.columns:
            vals = group[col].dropna()
            if len(vals):
                return vals.iloc[0]
        return None

    rows.append({
        "movie_record_id": movie_id,
        "movie_title": TARGET_MOVIES[movie_id],
        "video_id": video_id,
        "video_title": first_value(title_col),
        "channel": first_value(channel_col),
        "views": first_value(views_col),
        "duration": first_value(duration_col),
        "comment_count": first_value(comments_col),
        "historical_decisions": " | ".join(decisions),
        "already_acquired": video_id in acquired_ids,
        "historical_files": " | ".join(sorted(set(group["_file"])))
    })

reassessment = pd.DataFrame(rows)

# Priority: previously shortlisted, then held, then undecided;
# exclude previously rejected candidates from immediate acquisition.
def priority(decisions):
    d = str(decisions).lower()
    if "shortlist" in d:
        return 1
    if "hold" in d:
        return 2
    if not d:
        return 3
    return 4

reassessment["priority"] = reassessment[
    "historical_decisions"
].apply(priority)

reassessment["next_action"] = reassessment.apply(
    lambda r:
        "Already acquired — skip" if r["already_acquired"]
        else "Reassess shortlisted candidate" if r["priority"] == 1
        else "Reassess held candidate" if r["priority"] == 2
        else "Review undecided candidate" if r["priority"] == 3
        else "Previously excluded — skip unless new evidence",
    axis=1
)

reassessment = reassessment.sort_values(
    ["priority", "movie_title", "views"],
    ascending=[True, True, False],
    na_position="last"
)

print("=" * 90)
print("BATCH 14 — HISTORICAL CANDIDATE REASSESSMENT")
print("=" * 90)
print("Unique historical candidates:", len(reassessment))
print("Already acquired:", int(reassessment["already_acquired"].sum()))
print("Remaining candidates:", int((~reassessment["already_acquired"]).sum()))

print("\nPRIORITY SUMMARY")
print(reassessment["next_action"].value_counts().to_string())

print("\nCANDIDATE TABLE")
display(reassessment.drop(columns=["historical_files"]).reset_index(drop=True))

print("\nMOVIE-WISE SUMMARY")
display(
    reassessment.groupby(
        ["movie_record_id", "movie_title", "next_action"],
        dropna=False
    ).size().reset_index(name="candidate_count")
)

print("\nRead-only reassessment complete. No files were modified.")

In [ ]:
# ============================================================
# BATCH 14 — FINAL METADATA DISPLAY (NO MERGING)
# Reuses existing API results; no new API calls
# ============================================================

# Build a clean lookup from the historical candidate table
movie_lookup = (
    priority_candidates[
        ["video_id", "movie_title", "historical_decisions"]
    ]
    .drop_duplicates("video_id")
)

# Use only the columns actually returned by videos.list
display_df = meta_df.merge(
    movie_lookup,
    on="video_id",
    how="left",
    validate="one_to_one"
)

# Add acquisition screening priority
display_df["acquisition_priority"] = display_df[
    "current_comment_count"
].apply(
    lambda x: "High" if x >= 20 else "Low / Review"
)

print("=" * 85)
print("BATCH 14 — VERIFIED HISTORICAL CANDIDATES")
print("=" * 85)

print("Metadata records:", len(display_df))
print("Movies represented:", display_df["movie_title"].nunique())
print("Videos with comments:", (display_df["current_comment_count"] > 0).sum())

print("\nAVAILABLE METADATA COLUMNS:")
print(display_df.columns.tolist())

print("\nCANDIDATE DETAILS:")
display(
    display_df.sort_values(
        ["movie_title", "current_comment_count"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

print("\nMOVIE-WISE SUMMARY:")
display(
    display_df.groupby("movie_title").agg(
        videos=("video_id", "nunique"),
        available_comments=("current_comment_count", "sum"),
        high_priority_videos=(
            "acquisition_priority",
            lambda x: (x == "High").sum()
        )
    ).reset_index()
)

print("\nNo comments acquired. No files modified.")

In [ ]:
# ============================================================
# BATCH 14 — YOUTUBE COMMENT ACQUISITION
# 10 approved historical candidates
# Raw output protected by existence checks
# ============================================================

from pathlib import Path
from google.colab import userdata
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd
from datetime import datetime, timezone
import time

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
RAW_DIR = ROOT / "01_data/raw/audience_sentiment/youtube/comments"
LOG_DIR = ROOT / "05_logs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = RAW_DIR / "ICMB_130_YOUTUBE_COMMENTS_BATCH_14.csv"
LOG_FILE = LOG_DIR / "ICMB_130_YOUTUBE_COMMENTS_ACQUISITION_LOG_BATCH_14.csv"

if RAW_FILE.exists() or LOG_FILE.exists():
    raise FileExistsError(
        "Batch 14 output already exists. No files were overwritten."
    )

# Approved candidates and their source classifications
approved = [
    ("2004_4", "Mujhse Shaadi Karogi", "LGAEvDsKCn4", "Full Movie Reaction"),
    ("2004_4", "Mujhse Shaadi Karogi", "tMZI9sZvAH0", "Full Movie Reaction"),
    ("2004_4", "Mujhse Shaadi Karogi", "7GEBA9QVA0I", "Movie Reaction"),
    ("2004_4", "Mujhse Shaadi Karogi", "mo0TQAM-QLM", "Scene Reaction"),
    ("2012_2", "Dabangg 2", "kbPcGzsDy0s", "Scene Reaction"),
    ("2012_2", "Dabangg 2", "eZ5KaYVUjFE", "Movie Review / Summary"),
    ("2019_1", "War", "0JblhRtvdvQ", "Public Review"),
    ("2019_1", "War", "Xlr2SSDF_1c", "Public Reaction"),
    ("2019_1", "War", "3ZTJr-nnUkw", "Scene Reaction"),
    ("2019_1", "War", "MERys7i7OIk", "Song / Dance Reaction"),
]

api_key = userdata.get("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=api_key)

all_comments = []
logs = []

for movie_id, movie_title, video_id, source_type in approved:
    fetched = 0
    status = "SUCCESS"
    error_type = ""

    try:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            order="time",
            textFormat="plainText"
        )

        while request:
            response = request.execute()

            for item in response.get("items", []):
                top = item["snippet"]["topLevelComment"]
                snippet = top["snippet"]

                all_comments.append({
                    "movie_record_id": movie_id,
                    "movie_title": movie_title,
                    "video_id": video_id,
                    "source_type": source_type,
                    "comment_id": top["id"],
                    "comment_text": snippet.get("textDisplay", ""),
                    "author": snippet.get("authorDisplayName", ""),
                    "published_at": snippet.get("publishedAt"),
                    "updated_at": snippet.get("updatedAt"),
                    "like_count": snippet.get("likeCount", 0),
                    "reply_count": item["snippet"].get("totalReplyCount", 0),
                    "acquisition_order": fetched + 1,
                    "sampling_method": "Recent top-level comments; order=time"
                })
                fetched += 1

            request = (
                youtube.commentThreads().list_next(request, response)
                if fetched < 100 else None
            )

        if fetched == 0:
            status = "NO_COMMENTS_RETURNED"

    except HttpError as e:
        status = "API_ERROR"
        error_type = f"HTTP_{e.resp.status}"

    except Exception as e:
        status = "ERROR"
        error_type = type(e).__name__

    logs.append({
        "movie_record_id": movie_id,
        "movie_title": movie_title,
        "video_id": video_id,
        "source_type": source_type,
        "comments_retrieved": fetched,
        "acquisition_status": status,
        "error_type": error_type,
        "acquired_at_utc": datetime.now(timezone.utc).isoformat()
    })

    print(f"{movie_title}: {fetched} comments | {status}")
    time.sleep(0.2)

comments_df = pd.DataFrame(all_comments)
logs_df = pd.DataFrame(logs)

# Save separate raw acquisition and log files
comments_df.to_csv(RAW_FILE, index=False, encoding="utf-8-sig")
logs_df.to_csv(LOG_FILE, index=False, encoding="utf-8-sig")

print("\n" + "=" * 75)
print("BATCH 14 ACQUISITION SUMMARY")
print("=" * 75)
print("Videos attempted:", len(approved))
print("Successful videos:", (logs_df["acquisition_status"] == "SUCCESS").sum())
print("Total comments:", len(comments_df))
print("Unique comment IDs:", comments_df["comment_id"].nunique())
print("Raw file:", RAW_FILE)
print("Log file:", LOG_FILE)

display(logs_df)

In [ ]:
# ============================================================
# BATCH 14 — CLEANING, VALIDATION AND CONSOLIDATION
# Preserves raw data and previous consolidated dataset
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

RAW_FILE = (
    ROOT / "01_data/raw/audience_sentiment/youtube/comments/"
    "ICMB_130_YOUTUBE_COMMENTS_BATCH_14.csv"
)

OLD_FILE = (
    ROOT / "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_13.csv"
)

PROCESSED_DIR = ROOT / "01_data/processed/audience_sentiment"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_FILE = PROCESSED_DIR / "ICMB_130_YOUTUBE_COMMENTS_CLEAN_BATCH_14.csv"
NEW_FILE = PROCESSED_DIR / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_14.csv"
AUDIT_FILE = PROCESSED_DIR / "ICMB_130_YOUTUBE_COMMENTS_AUDIT_BATCH_14.csv"

# Protect existing outputs
for path in [CLEAN_FILE, NEW_FILE, AUDIT_FILE]:
    if path.exists():
        raise FileExistsError(f"Output already exists: {path}")

raw = pd.read_csv(RAW_FILE, low_memory=False)
old = pd.read_csv(OLD_FILE, low_memory=False)

print("Raw Batch 14:", raw.shape)
print("Previous consolidated:", old.shape)

# Normalize identifiers and text without changing the raw file
raw["comment_id"] = raw["comment_id"].astype("string").str.strip()
raw["video_id"] = raw["video_id"].astype("string").str.strip()
raw["comment_text"] = raw["comment_text"].fillna("").astype(str).str.strip()

# Validation
missing_ids = raw["comment_id"].isna() | raw["comment_id"].eq("")
missing_video_ids = raw["video_id"].isna() | raw["video_id"].eq("")
empty_text = raw["comment_text"].eq("")

duplicate_within_batch = raw["comment_id"].duplicated(keep="first")
old_ids = set(old["comment_id"].dropna().astype(str).str.strip())
cross_batch_duplicate = raw["comment_id"].astype(str).isin(old_ids)

audit = pd.DataFrame([{
    "raw_rows": len(raw),
    "missing_comment_ids": int(missing_ids.sum()),
    "missing_video_ids": int(missing_video_ids.sum()),
    "empty_comment_text": int(empty_text.sum()),
    "duplicate_ids_within_batch": int(duplicate_within_batch.sum()),
    "duplicates_against_previous_consolidated": int(cross_batch_duplicate.sum())
}])

# Keep only valid new records; do not delete duplicate text
valid = raw.loc[
    ~(missing_ids | missing_video_ids | empty_text |
      duplicate_within_batch | cross_batch_duplicate)
].copy()

# Save clean batch
valid.to_csv(CLEAN_FILE, index=False, encoding="utf-8-sig")

# Align columns safely for consolidation
all_columns = list(dict.fromkeys(list(old.columns) + list(valid.columns)))

old_aligned = old.reindex(columns=all_columns)
valid_aligned = valid.reindex(columns=all_columns)

combined = pd.concat(
    [old_aligned, valid_aligned],
    ignore_index=True
)

if combined["comment_id"].duplicated().any():
    raise AssertionError("Duplicate comment IDs detected in final consolidation.")

combined.to_csv(NEW_FILE, index=False, encoding="utf-8-sig")

audit.to_csv(AUDIT_FILE, index=False, encoding="utf-8-sig")

print("\n" + "=" * 75)
print("BATCH 14 CLEANING AND CONSOLIDATION")
print("=" * 75)

display(audit)

print("Clean Batch 14 rows:", len(valid))
print("Excluded rows:", len(raw) - len(valid))
print("New consolidated rows:", len(combined))
print("Unique comment IDs:", combined["comment_id"].nunique())
print("Unique videos:", combined["video_id"].nunique())
print("Unique movies:", combined["movie_record_id"].nunique())

print("\nSaved files:")
print(CLEAN_FILE)
print(NEW_FILE)
print(AUDIT_FILE)

print("\nPrevious consolidated file remains unchanged.")
print("Raw Batch 14 file remains unchanged.")

In [ ]:
# ============================================================
# ICMB-130 — MASTER DATASET LOCATION DIAGNOSTIC
# Read-only | No files modified
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

print("=" * 80)
print("PROJECT DIRECTORY CHECK")
print("=" * 80)
print("Project exists:", ROOT.exists())

SEARCH_DIRS = [
    ROOT / "01_data",
    ROOT / "01_data/raw",
    ROOT / "01_data/processed",
]

found = {}

for folder in SEARCH_DIRS:
    if folder.exists():
        for file in folder.rglob("*.csv"):
            name = file.name.lower()

            if any(term in name for term in [
                "master", "icmb_130", "feature", "final_dataset"
            ]):
                found[str(file)] = file

print("\nPOTENTIAL DATASET FILES")
for i, (path, file) in enumerate(found.items(), 1):
    print(f"\n{i}. {file.name}")
    print("Path:", path)
    print("Size (KB):", round(file.stat().st_size / 1024, 2))

    try:
        sample = pd.read_csv(file, nrows=3, low_memory=False)
        print("Shape preview:", sample.shape)
        print("Columns:", sample.columns.tolist())
    except Exception as e:
        print("Read error:", type(e).__name__)

print("\n" + "=" * 80)
print("FILES IN 01_DATA/PROCESSED")
print("=" * 80)

processed = ROOT / "01_data/processed"

if processed.exists():
    for file in sorted(processed.glob("*.csv")):
        print(file.name)
else:
    print("Processed directory not found.")

print("\nDiagnostic complete. No files modified.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 14 STATUS & COVERAGE PREFLIGHT
# Read-only | No API calls | No files modified
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

MASTER = ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
RAW = ROOT / "01_data/raw/audience_sentiment/youtube/comments"
PROC = ROOT / "01_data/processed/audience_sentiment"

BATCH14 = RAW / "ICMB_130_YOUTUBE_COMMENTS_BATCH_14.csv"
CONSOLIDATED = PROC / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_13.csv"

# Load verified files
master = pd.read_csv(MASTER)
batch14 = pd.read_csv(BATCH14) if BATCH14.exists() else pd.DataFrame()
old = pd.read_csv(CONSOLIDATED)

print("=" * 75)
print("BATCH 14 STATUS")
print("=" * 75)

print("Master dataset:", master.shape)
print("Existing consolidated:", old.shape)
print("Batch 14 raw:", batch14.shape)

if not batch14.empty:
    print("\nBATCH 14 MOVIE-WISE COMMENTS")
    display(
        batch14.groupby(
            ["movie_record_id", "movie_title", "video_id"],
            dropna=False
        )
        .agg(comments=("comment_id", "count"))
        .reset_index()
    )

    print("\nBATCH 14 UNIQUE COMMENT IDs:", batch14["comment_id"].nunique())
    print("BATCH 14 UNIQUE VIDEOS:", batch14["video_id"].nunique())
    print(
        "OVERLAP WITH BATCHES 01–13:",
        len(
            set(batch14["comment_id"].dropna().astype(str))
            & set(old["comment_id"].dropna().astype(str))
        )
    )

print("\nBATCH 14 RAW FILE COLUMNS")
print(batch14.columns.tolist() if not batch14.empty else "No Batch 14 records")

print("\nExisting master columns:")
print(master.columns.tolist())

print("\nNo files modified.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 14 CONSOLIDATION PREFLIGHT
# Read-only | No files modified
# ============================================================

import pandas as pd
from pathlib import Path

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

PROC = ROOT / "01_data/processed/audience_sentiment"
RAW = ROOT / "01_data/raw/audience_sentiment/youtube/comments"

old_file = PROC / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_13.csv"
batch_file = RAW / "ICMB_130_YOUTUBE_COMMENTS_BATCH_14.csv"
master_file = ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"

old = pd.read_csv(old_file)
batch = pd.read_csv(batch_file)
master = pd.read_csv(master_file)

print("=" * 75)
print("CONSOLIDATION PREFLIGHT — BATCH 14")
print("=" * 75)

print("\nEXISTING CONSOLIDATED SCHEMA")
print(old.columns.tolist())

print("\nBATCH 14 SCHEMA")
print(batch.columns.tolist())

print("\nCOLUMNS PRESENT ONLY IN CONSOLIDATED")
print(sorted(set(old.columns) - set(batch.columns)))

print("\nCOLUMNS PRESENT ONLY IN BATCH 14")
print(sorted(set(batch.columns) - set(old.columns)))

print("\nBATCH 14 MOVIE ID VALIDATION")
movie_check = (
    batch[["movie_record_id", "movie_title"]]
    .drop_duplicates()
    .merge(
        master[["movie_record_id", "movie_title", "year", "language"]],
        on="movie_record_id",
        how="left",
        suffixes=("_batch", "_master"),
        indicator=True
    )
)
display(movie_check)

print("\nCOMMENT ID VALIDATION")
print("Batch rows:", len(batch))
print("Unique IDs:", batch["comment_id"].nunique())
print("Missing IDs:", batch["comment_id"].isna().sum())
print("Duplicate IDs within Batch 14:", batch["comment_id"].duplicated().sum())
print(
    "Overlap with Batches 01–13:",
    len(
        set(batch["comment_id"].dropna().astype(str))
        & set(old["comment_id"].dropna().astype(str))
    )
)

print("\nEXISTING OUTPUT NAME CHECK")
for name in [
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_14.csv",
    "ICMB_130_YOUTUBE_MOVIE_COVERAGE_PILOT_BATCH_01_14.csv",
    "ICMB_130_YOUTUBE_LANGUAGE_COVERAGE_PILOT_BATCH_01_14.csv"
]:
    print(name, "|", "ALREADY EXISTS" if (PROC / name).exists() else "AVAILABLE")

print("\nNo files modified.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — EXISTING BATCH 14 CONSOLIDATION CHECK
# Read-only | No files modified
# ============================================================

import pandas as pd
from pathlib import Path

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
PROC = ROOT / "01_data/processed/audience_sentiment"
RAW = ROOT / "01_data/raw/audience_sentiment/youtube/comments"

old = pd.read_csv(
    PROC / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_13.csv"
)

batch14 = pd.read_csv(
    RAW / "ICMB_130_YOUTUBE_COMMENTS_BATCH_14.csv"
)

existing_path = (
    PROC / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_14.csv"
)

print("=" * 75)
print("EXISTING BATCH 01–14 FILE INSPECTION")
print("=" * 75)

if existing_path.exists():
    existing = pd.read_csv(existing_path)

    old_ids = set(old["comment_id"].dropna().astype(str))
    batch14_ids = set(batch14["comment_id"].dropna().astype(str))
    existing_ids = set(existing["comment_id"].dropna().astype(str))

    expected_ids = old_ids | batch14_ids

    print("Existing file shape:", existing.shape)
    print("Existing unique comment IDs:", len(existing_ids))
    print("Expected combined unique IDs:", len(expected_ids))
    print("Existing duplicate IDs:", existing["comment_id"].duplicated().sum())

    print("\nCOMPARISON")
    print("Missing expected IDs:", len(expected_ids - existing_ids))
    print("Unexpected IDs:", len(existing_ids - expected_ids))
    print("Batch 14 IDs present:", len(batch14_ids & existing_ids))
    print("Batch 14 IDs missing:", len(batch14_ids - existing_ids))

    print("\nMOVIE-WISE EXISTING COUNTS")
    display(
        existing.groupby(
            ["movie_record_id", "movie_title"],
            dropna=False
        )
        .agg(
            comments=("comment_id", "count"),
            unique_comments=("comment_id", "nunique"),
            videos=("video_id", "nunique")
        )
        .reset_index()
        .tail(15)
    )

    print("\nSOURCE BATCH DISTRIBUTION")
    if "source_batch" in existing.columns:
        print(existing["source_batch"].value_counts(dropna=False).to_string())

    print("\nSCHEMA MATCHES BATCH 01–13:",
          set(existing.columns) == set(old.columns))

else:
    print("Existing Batches 01–14 file not found.")

print("\nNo files modified.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 15: FINAL 29-MOVIE SOURCE INVENTORY
# Read-only | Reuse historical discovery records
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
YOUTUBE_ROOT = ROOT / "01_data/raw/audience_sentiment/youtube"

remaining_movies = [
    ("2004_4", "Mujhse Shaadi Karogi"),
    ("2010_3", "Golmaal 3"),
    ("2012_2", "Dabangg 2"),
    ("2012_3", "Rowdy Rathore"),
    ("2013_5", "Goliyon Ki Raasleela Ram-Leela"),
    ("2014_2", "Kick"),
    ("2014_3", "Happy New Year"),
    ("2014_4", "Bang Bang!"),
    ("2014_5", "Singham Returns"),
    ("2015_3", "Prem Ratan Dhan Payo"),
    ("2015_4", "Bajirao Mastani"),
    ("2015_5", "Tanu Weds Manu Returns"),
    ("2016_4", "M.S. Dhoni: The Untold Story"),
    ("2016_5", "Rustom"),
    ("2017_3", "Golmaal Again"),
    ("2017_4", "Raees"),
    ("2017_5", "Judwaa 2"),
    ("2018_2", "Sanju"),
    ("2018_3", "Padmaavat"),
    ("2018_4", "Simmba"),
    ("2019_1", "War"),
    ("2019_3", "Kabir Singh"),
    ("2019_4", "Uri: The Surgical Strike"),
    ("2019_5", "Bharat"),
    ("2020_1", "Tanhaji: The Unsung Warrior"),
    ("2020_4", "Baaghi 3"),
    ("2020_5", "Street Dancer 3D"),
    ("2021_2", "Sooryavanshi"),
    ("2021_4", "83"),
    ("2023_4", "Gadar 2"),
    ("2024_5", "Bhool Bhulaiyaa 3"),
    ("2025_4", "Saiyaara"),
]

# Batch 14 has already supplied comments for these three movies.
batch14_covered = {"2004_4", "2012_2", "2019_1"}
targets = [
    (mid, title) for mid, title in remaining_movies
    if mid not in batch14_covered
]

print("=" * 75)
print("BATCH 15 — FINAL ACQUISITION TARGETS")
print("=" * 75)
print("Remaining movie targets:", len(targets))
for mid, title in targets:
    print(f"{mid}: {title}")

print("\n" + "=" * 75)
print("HISTORICAL YOUTUBE FILE INVENTORY")
print("=" * 75)

files = sorted(YOUTUBE_ROOT.rglob("*.csv"))
keywords = ("discover", "candidate", "screen", "verification", "manifest")

matches = [
    p for p in files
    if any(k in p.name.lower() for k in keywords)
]

print("Relevant CSV files found:", len(matches))

for path in matches:
    try:
        df = pd.read_csv(path, nrows=5)
        print(f"\nFILE: {path.relative_to(ROOT)}")
        print("Columns:", list(df.columns))
        print("Sample rows:", len(df))
    except Exception as e:
        print(f"\nFILE: {path.name} | Read error: {type(e).__name__}")

print("\nNo files modified. No API quota used.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 15: HISTORICAL CANDIDATE CONSOLIDATION
# Read-only | No API calls | No files modified
# ============================================================

import pandas as pd
from pathlib import Path

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
YT = ROOT / "01_data/raw/audience_sentiment/youtube"
PROC = ROOT / "01_data/processed/audience_sentiment"

targets = {
    "2010_3": "Golmaal 3",
    "2012_3": "Rowdy Rathore",
    "2013_5": "Goliyon Ki Raasleela Ram-Leela",
    "2014_2": "Kick",
    "2014_3": "Happy New Year",
    "2014_4": "Bang Bang!",
    "2014_5": "Singham Returns",
    "2015_3": "Prem Ratan Dhan Payo",
    "2015_4": "Bajirao Mastani",
    "2015_5": "Tanu Weds Manu Returns",
    "2016_4": "M.S. Dhoni: The Untold Story",
    "2016_5": "Rustom",
    "2017_3": "Golmaal Again",
    "2017_4": "Raees",
    "2017_5": "Judwaa 2",
    "2018_2": "Sanju",
    "2018_3": "Padmaavat",
    "2018_4": "Simmba",
    "2019_3": "Kabir Singh",
    "2019_4": "Uri: The Surgical Strike",
    "2019_5": "Bharat",
    "2020_1": "Tanhaji: The Unsung Warrior",
    "2020_4": "Baaghi 3",
    "2020_5": "Street Dancer 3D",
    "2021_2": "Sooryavanshi",
    "2021_4": "83",
    "2023_4": "Gadar 2",
    "2024_5": "Bhool Bhulaiyaa 3",
    "2025_4": "Saiyaara"
}

# Load historical discovery, candidate, shortlist and screening files.
candidate_frames = []
read_errors = []

keywords = (
    "DISCOVERY", "CANDIDATE", "SHORTLIST",
    "SCREENING", "VERIFICATION"
)

for path in sorted(YT.glob("*.csv")):
    if not any(word in path.name.upper() for word in keywords):
        continue

    try:
        df = pd.read_csv(path, dtype={"video_id": "string"})
        if {"movie_record_id", "video_id"}.issubset(df.columns):
            df["historical_file"] = path.name
            candidate_frames.append(df)
    except Exception as e:
        read_errors.append((path.name, type(e).__name__))

if not candidate_frames:
    raise ValueError("No usable historical candidate files were found.")

history = pd.concat(candidate_frames, ignore_index=True, sort=False)

history["movie_record_id"] = history["movie_record_id"].astype("string").str.strip()
history["video_id"] = history["video_id"].astype("string").str.strip()

history = history[
    history["movie_record_id"].isin(targets)
    & history["video_id"].notna()
    & history["video_id"].ne("")
].copy()

# One row per movie/video pair, retaining all file provenance.
history["historical_file"] = history["historical_file"].fillna("")
candidate_pool = (
    history.groupby(
        ["movie_record_id", "video_id"],
        as_index=False,
        sort=False
    )
    .agg(
        movie_title=("movie_title", "first"),
        video_title=("video_title", "first"),
        metadata_title=("metadata_title", "first"),
        channel_title=("channel_title", "first"),
        metadata_channel_title=("metadata_channel_title", "first"),
        comment_count=("comment_count", "max"),
        automated_decision=("automated_decision", "first"),
        automated_source_type=("automated_source_type", "first"),
        source_type=("source_type", "first"),
        historical_files=(
            "historical_file",
            lambda x: " | ".join(sorted(set(v for v in x if v)))
        )
    )
)

# Existing acquired video IDs from the current consolidated dataset.
consolidated_path = (
    PROC / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_14.csv"
)
if not consolidated_path.exists():
    raise FileNotFoundError(f"Expected consolidated file not found: {consolidated_path}")

comments = pd.read_csv(
    consolidated_path,
    usecols=["movie_record_id", "video_id"],
    dtype={"movie_record_id": "string", "video_id": "string"}
)

acquired_pairs = set(
    zip(
        comments["movie_record_id"].str.strip(),
        comments["video_id"].str.strip()
    )
)

candidate_pool["already_acquired"] = [
    (mid, vid) in acquired_pairs
    for mid, vid in zip(
        candidate_pool["movie_record_id"],
        candidate_pool["video_id"]
    )
]

candidate_pool["movie_title"] = candidate_pool.apply(
    lambda r: targets.get(r["movie_record_id"], r["movie_title"]),
    axis=1
)

summary = (
    candidate_pool.groupby("movie_record_id")
    .agg(
        movie_title=("movie_title", "first"),
        historical_candidates=("video_id", "nunique"),
        already_acquired_videos=("already_acquired", "sum")
    )
    .reset_index()
)

summary["not_yet_acquired"] = (
    summary["historical_candidates"] - summary["already_acquired_videos"]
)

# Include targets with no historical candidate records.
summary = (
    pd.DataFrame([
        {"movie_record_id": mid, "movie_title": title}
        for mid, title in targets.items()
    ])
    .merge(summary, on=["movie_record_id", "movie_title"], how="left")
    .fillna({
        "historical_candidates": 0,
        "already_acquired_videos": 0,
        "not_yet_acquired": 0
    })
)

for col in ["historical_candidates", "already_acquired_videos", "not_yet_acquired"]:
    summary[col] = summary[col].astype(int)

print("=" * 75)
print("BATCH 15 — HISTORICAL CANDIDATE COVERAGE")
print("=" * 75)
print("Target movies:", len(targets))
print("Historical files read:", len(candidate_frames))
print("Unique movie/video candidates:", len(candidate_pool))
print("Candidates already represented in comments:", int(candidate_pool["already_acquired"].sum()))
print("Candidates not yet represented in comments:", int((~candidate_pool["already_acquired"]).sum()))

display(summary.sort_values(
    ["historical_candidates", "movie_record_id"],
    ascending=[True, True]
).reset_index(drop=True))

print("\n" + "=" * 75)
print("CANDIDATE DETAILS — NOT YET ACQUIRED")
print("=" * 75)

display(
    candidate_pool.loc[
        ~candidate_pool["already_acquired"],
        [
            "movie_record_id", "movie_title", "video_id",
            "video_title", "metadata_title", "channel_title",
            "metadata_channel_title", "comment_count",
            "automated_decision", "automated_source_type",
            "source_type", "historical_files"
        ]
    ].sort_values(["movie_record_id", "video_id"]).reset_index(drop=True)
)

if read_errors:
    print("\nFiles that could not be read:")
    print(read_errors)

print("\nRead-only inventory complete. No API quota used; no files modified.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 15: FINAL 27-MOVIE YOUTUBE DISCOVERY
# 27 search requests | Up to 10 candidates per movie
# ============================================================

import pandas as pd
import time
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
YT = ROOT / "01_data/raw/audience_sentiment/youtube"

discovery_path = YT / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_15.csv"
log_path = YT / "ICMB_130_YOUTUBE_DISCOVERY_LOG_BATCH_15.csv"

if discovery_path.exists() or log_path.exists():
    raise FileExistsError(
        "Batch 15 output already exists. No files were changed. "
        "Inspect the existing files before proceeding."
    )

# The two movies with historical candidates are deliberately excluded.
movies = [
    ("2013_5", "Goliyon Ki Raasleela Ram-Leela", 2013),
    ("2014_2", "Kick", 2014),
    ("2014_3", "Happy New Year", 2014),
    ("2014_4", "Bang Bang", 2014),
    ("2014_5", "Singham Returns", 2014),
    ("2015_3", "Prem Ratan Dhan Payo", 2015),
    ("2015_4", "Bajirao Mastani", 2015),
    ("2015_5", "Tanu Weds Manu Returns", 2015),
    ("2016_4", "M.S. Dhoni The Untold Story", 2016),
    ("2016_5", "Rustom", 2016),
    ("2017_3", "Golmaal Again", 2017),
    ("2017_4", "Raees", 2017),
    ("2017_5", "Judwaa 2", 2017),
    ("2018_2", "Sanju", 2018),
    ("2018_3", "Padmaavat", 2018),
    ("2018_4", "Simmba", 2018),
    ("2019_3", "Kabir Singh", 2019),
    ("2019_4", "Uri The Surgical Strike", 2019),
    ("2019_5", "Bharat", 2019),
    ("2020_1", "Tanhaji The Unsung Warrior", 2020),
    ("2020_4", "Baaghi 3", 2020),
    ("2020_5", "Street Dancer 3D", 2020),
    ("2021_2", "Sooryavanshi", 2021),
    ("2021_4", "83 movie", 2021),
    ("2023_4", "Gadar 2", 2023),
    ("2024_5", "Bhool Bhulaiyaa 3", 2024),
    ("2025_4", "Saiyaara", 2025),
]

# Retrieve API key from Colab Secret without displaying it.
api_key = userdata.get("YOUTUBE_API_KEY")
if not api_key:
    raise ValueError(
        "YOUTUBE_API_KEY was not found in Colab Secrets. "
        "Add the existing key there and rerun this cell."
    )

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

discovery_rows = []
log_rows = []

print("=" * 75)
print("BATCH 15 — YOUTUBE DISCOVERY")
print("=" * 75)
print("Movies to search:", len(movies))
print("Planned search quota:", len(movies) * 100)
print("Maximum candidates:", len(movies) * 10)
print()

for i, (movie_id, title, year) in enumerate(movies, start=1):
    query = f'"{title}" {year} movie review reaction Hindi'

    try:
        response = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            maxResults=10,
            order="relevance",
            regionCode="IN",
            relevanceLanguage="hi"
        ).execute()

        items = response.get("items", [])

        for item in items:
            snippet = item.get("snippet", {})
            video_id = item.get("id", {}).get("videoId")

            if not video_id:
                continue

            discovery_rows.append({
                "movie_record_id": movie_id,
                "movie_title": title,
                "movie_year": year,
                "search_query": query,
                "video_id": video_id,
                "video_title": snippet.get("title"),
                "channel_id": snippet.get("channelId"),
                "channel_title": snippet.get("channelTitle"),
                "published_at": snippet.get("publishedAt"),
                "description": snippet.get("description"),
                "discovery_source": "YouTube Data API v3",
                "discovery_batch": "Batch 15"
            })

        log_rows.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "search_query": query,
            "request_status": "SUCCESS",
            "candidates_returned": len(items),
            "error_type": None,
            "requested_at_utc": datetime.now(timezone.utc).isoformat()
        })

        print(f"[{i:02d}/{len(movies)}] {title}: {len(items)} candidates")

    except Exception as e:
        # Do not print raw API exceptions; they may contain sensitive request data.
        log_rows.append({
            "movie_record_id": movie_id,
            "movie_title": title,
            "search_query": query,
            "request_status": "FAILED",
            "candidates_returned": 0,
            "error_type": type(e).__name__,
            "requested_at_utc": datetime.now(timezone.utc).isoformat()
        })

        print(f"[{i:02d}/{len(movies)}] {title}: REQUEST FAILED ({type(e).__name__})")

    time.sleep(0.2)

discovery = pd.DataFrame(discovery_rows)
log = pd.DataFrame(log_rows)

if not discovery.empty:
    discovery = discovery.drop_duplicates(
        subset=["movie_record_id", "video_id"]
    )

# Save only after all requests finish.
discovery.to_csv(discovery_path, index=False)
log.to_csv(log_path, index=False)

print("\n" + "=" * 75)
print("BATCH 15 DISCOVERY SUMMARY")
print("=" * 75)
print("Successful requests:", (log["request_status"] == "SUCCESS").sum())
print("Failed requests:", (log["request_status"] == "FAILED").sum())
print("Unique candidates discovered:", len(discovery))
print("Movies with candidates:", discovery["movie_record_id"].nunique())
print("Movies without candidates:", len(movies) - discovery["movie_record_id"].nunique())

print("\nREQUEST LOG:")
display(log[[
    "movie_record_id", "movie_title",
    "request_status", "candidates_returned"
]])

print("\nDISCOVERY FILE:", discovery_path)
print("LOG FILE:", log_path)
print("\nAPI key was not printed.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 15: YOUTUBE METADATA RETRIEVAL
# 270 candidates | 6 videos.list requests
# ============================================================

import pandas as pd
import time
from pathlib import Path
from datetime import datetime, timezone
from google.colab import userdata
from googleapiclient.discovery import build

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
YT = ROOT / "01_data/raw/audience_sentiment/youtube"

discovery_path = YT / "ICMB_130_YOUTUBE_DISCOVERY_BATCH_15.csv"
metadata_path = YT / "ICMB_130_YOUTUBE_METADATA_BATCH_15.csv"
log_path = YT / "ICMB_130_YOUTUBE_METADATA_LOG_BATCH_15.csv"

if not discovery_path.exists():
    raise FileNotFoundError("Batch 15 discovery file not found.")

if metadata_path.exists() or log_path.exists():
    raise FileExistsError(
        "Batch 15 metadata output already exists. Inspect before rerunning."
    )

discovery = pd.read_csv(
    discovery_path,
    dtype={"movie_record_id": "string", "video_id": "string"}
)

discovery = discovery.drop_duplicates(
    subset=["movie_record_id", "video_id"]
).copy()

video_ids = discovery["video_id"].dropna().unique().tolist()

api_key = userdata.get("YOUTUBE_API_KEY")
if not api_key:
    raise ValueError("YOUTUBE_API_KEY not found in Colab Secrets.")

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

metadata_rows = []
error_rows = []

print("=" * 75)
print("BATCH 15 — METADATA RETRIEVAL")
print("=" * 75)
print("Candidate records:", len(discovery))
print("Unique video IDs:", len(video_ids))
print("Planned videos.list requests:", (len(video_ids) + 49) // 50)

for start in range(0, len(video_ids), 50):
    chunk = video_ids[start:start + 50]
    request_number = start // 50 + 1

    try:
        response = youtube.videos().list(
            part="snippet,contentDetails,statistics,status",
            id=",".join(chunk),
            maxResults=50
        ).execute()

        returned = response.get("items", [])

        for item in returned:
            snippet = item.get("snippet", {})
            stats = item.get("statistics", {})
            details = item.get("contentDetails", {})
            status = item.get("status", {})

            metadata_rows.append({
                "video_id": item.get("id"),
                "metadata_title": snippet.get("title"),
                "metadata_description": snippet.get("description"),
                "channel_id_metadata": snippet.get("channelId"),
                "channel_title_metadata": snippet.get("channelTitle"),
                "published_at_metadata": snippet.get("publishedAt"),
                "tags": " | ".join(snippet.get("tags", [])),
                "category_id": snippet.get("categoryId"),
                "duration": details.get("duration"),
                "definition": details.get("definition"),
                "caption_available": details.get("caption"),
                "view_count": stats.get("viewCount"),
                "like_count": stats.get("likeCount"),
                "comment_count": stats.get("commentCount"),
                "privacy_status": status.get("privacyStatus"),
                "embeddable": status.get("embeddable"),
                "metadata_retrieved_at_utc": datetime.now(
                    timezone.utc
                ).isoformat()
            })

        returned_ids = {item.get("id") for item in returned}
        missing_ids = set(chunk) - returned_ids

        for vid in missing_ids:
            error_rows.append({
                "video_id": vid,
                "request_number": request_number,
                "status": "NOT_RETURNED",
                "error_type": None
            })

        print(
            f"Request {request_number}: "
            f"{len(returned)} returned; "
            f"{len(missing_ids)} not returned"
        )

    except Exception as e:
        for vid in chunk:
            error_rows.append({
                "video_id": vid,
                "request_number": request_number,
                "status": "FAILED",
                "error_type": type(e).__name__
            })

        print(f"Request {request_number}: FAILED ({type(e).__name__})")

    time.sleep(0.2)

metadata = pd.DataFrame(metadata_rows)
errors = pd.DataFrame(error_rows)

if metadata.empty:
    raise RuntimeError(
        "No metadata returned. Files were not saved; inspect the API connection."
    )

# Join metadata to each movie/video candidate record.
screening = discovery.merge(
    metadata,
    on="video_id",
    how="left",
    validate="many_to_one"
)

screening.to_csv(metadata_path, index=False)
errors.to_csv(log_path, index=False)

print("\n" + "=" * 75)
print("METADATA RETRIEVAL SUMMARY")
print("=" * 75)
print("Candidate records:", len(discovery))
print("Unique video IDs requested:", len(video_ids))
print("Unique video IDs returned:", metadata["video_id"].nunique())
print("Candidate records with metadata:", screening["metadata_title"].notna().sum())
print("Candidate records without metadata:", screening["metadata_title"].isna().sum())
print("Metadata error/not-returned records:", len(errors))
print("Saved metadata:", metadata_path)
print("Saved log:", log_path)

print("\nMOVIE-WISE METADATA COVERAGE")
display(
    screening.groupby(["movie_record_id", "movie_title"], dropna=False)
    .agg(
        candidates=("video_id", "nunique"),
        metadata_returned=("metadata_title", "count"),
        total_reported_comments=("comment_count", "sum")
    )
    .reset_index()
)

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 15: AUTOMATED CANDIDATE SCREENING
# Preliminary metadata screening | No API calls
# ============================================================

import pandas as pd
import re
from pathlib import Path

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
YT = ROOT / "01_data/raw/audience_sentiment/youtube"

INPUT = YT / "ICMB_130_YOUTUBE_METADATA_BATCH_15.csv"
OUTPUT = YT / "ICMB_130_YOUTUBE_AUTOMATED_SCREENING_BATCH_15.csv"

if not INPUT.exists():
    raise FileNotFoundError(INPUT)

if OUTPUT.exists():
    raise FileExistsError(
        "Screening output already exists. No existing file will be overwritten."
    )

df = pd.read_csv(INPUT, dtype={"video_id": "string"})

# Convert API statistics to numeric values.
df["comment_count"] = pd.to_numeric(
    df["comment_count"], errors="coerce"
).fillna(0).astype(int)

df["view_count"] = pd.to_numeric(
    df["view_count"], errors="coerce"
).fillna(0).astype("int64")

df["like_count"] = pd.to_numeric(
    df["like_count"], errors="coerce"
).fillna(0).astype("int64")

# Convert ISO 8601 YouTube duration to seconds.
def duration_to_seconds(value):
    if pd.isna(value):
        return None

    match = re.fullmatch(
        r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?",
        str(value)
    )

    if not match:
        return None

    hours, minutes, seconds = [
        int(x) if x else 0 for x in match.groups()
    ]

    return hours * 3600 + minutes * 60 + seconds

df["duration_seconds"] = df["duration"].apply(duration_to_seconds)

# Build text evidence from metadata.
def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).lower()

df["screening_text"] = (
    df["metadata_title"].apply(clean_text) + " " +
    df["metadata_description"].apply(clean_text)
)

# Conservative classification rules.
patterns = {
    "Full Movie": r"\b(full movie|full film|complete movie|entire movie|"
                  r"full hd movie|watch full movie|full hindi movie)\b",

    "Trailer / Teaser": r"\b(trailer|teaser|official promo|movie promo)\b",

    "Song / Music Video": r"\b(official song|video song|full song|"
                          r"lyrical video|lyric video|jukebox|"
                          r"music video|audio jukebox)\b",

    "Clip / Scene": r"\b(scene|clip|fight scene|comedy scene|"
                    r"best scenes|dialogue scene|movie scene|"
                    r"part \d+|shorts)\b",

    "Review / Analysis": r"\b(review|movie review|film review|"
                         r"analysis|explained|ending explained|"
                         r"honest review|public review|"
                         r"critique|rating)\b",

    "Reaction": r"\b(reaction|reacts|reacting|first time watching|"
                r"watching for the first time)\b"
}

def classify_source(text):
    # Priority order prevents a full movie or trailer from
    # being incorrectly treated as an audience review.
    for category in [
        "Full Movie",
        "Trailer / Teaser",
        "Song / Music Video",
        "Clip / Scene"
    ]:
        if re.search(patterns[category], text):
            return category

    has_review = bool(re.search(patterns["Review / Analysis"], text))
    has_reaction = bool(re.search(patterns["Reaction"], text))

    if has_review and has_reaction:
        return "Review + Reaction"
    if has_review:
        return "Movie Review / Analysis"
    if has_reaction:
        return "Movie Reaction"

    return "Unclassified"

df["automated_source_type"] = df["screening_text"].apply(classify_source)

# Title identity evidence: compare meaningful movie-title tokens
# against the YouTube metadata title.
def title_identity_match(row):
    movie_title = clean_text(row.get("movie_title"))
    video_title = clean_text(row.get("metadata_title"))

    tokens = [
        token for token in re.findall(r"[a-z0-9]+", movie_title)
        if len(token) >= 3
    ]

    if not tokens:
        return False

    matched = sum(token in video_title for token in tokens)
    return matched >= max(1, min(2, len(tokens)))

df["title_identity_match"] = df.apply(title_identity_match, axis=1)

# Preliminary decisions. These are not manual content verification.
def make_decision(row):
    source = row["automated_source_type"]

    if source in [
        "Full Movie", "Trailer / Teaser",
        "Song / Music Video", "Clip / Scene"
    ]:
        return "Exclude"

    if source in [
        "Movie Review / Analysis",
        "Movie Reaction",
        "Review + Reaction"
    ]:
        if row["comment_count"] > 0 and row["title_identity_match"]:
            return "Shortlist"
        return "Needs Review"

    return "Needs Review"

df["automated_decision"] = df.apply(make_decision, axis=1)

# Keep metadata and content review status distinct.
df["metadata_review_status"] = "Retrieved — automated screening"
df["content_review_status"] = "Not reviewed"
df["manual_verification_status"] = "Pending"

# Save the screening output.
df.to_csv(OUTPUT, index=False)

print("=" * 75)
print("BATCH 15 — AUTOMATED SCREENING RESULTS")
print("=" * 75)

print("Candidate records:", len(df))
print("Unique videos:", df["video_id"].nunique())
print("Missing metadata titles:", df["metadata_title"].isna().sum())
print("Comment count numeric:", pd.api.types.is_numeric_dtype(df["comment_count"]))
print("Total reported comments across candidates:", f"{df['comment_count'].sum():,}")

print("\nAUTOMATED DECISION DISTRIBUTION")
display(
    df["automated_decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="candidate_count")
)

print("\nSOURCE TYPE DISTRIBUTION")
display(
    df["automated_source_type"]
    .value_counts()
    .rename_axis("source_type")
    .reset_index(name="candidate_count")
)

print("\nMOVIE-WISE SCREENING SUMMARY")
display(
    df.groupby(["movie_record_id", "movie_title"], dropna=False)
    .agg(
        candidates=("video_id", "nunique"),
        shortlist=("automated_decision",
                   lambda x: (x == "Shortlist").sum()),
        needs_review=("automated_decision",
                      lambda x: (x == "Needs Review").sum()),
        excluded=("automated_decision",
                  lambda x: (x == "Exclude").sum()),
        reported_comments=("comment_count", "sum")
    )
    .reset_index()
)

print("\nSaved:", OUTPUT)
print("Content review has NOT been marked complete.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — BATCH 15: CONSOLIDATED SOURCE SHORTLIST
# 29 movies | Maximum 2 candidate sources per movie
# No API calls | No existing files overwritten
# ============================================================

import pandas as pd
from pathlib import Path

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
YT = ROOT / "01_data/raw/audience_sentiment/youtube"

screening_path = YT / "ICMB_130_YOUTUBE_AUTOMATED_SCREENING_BATCH_15.csv"
history11_path = YT / "ICMB_130_YOUTUBE_CANDIDATE_REVIEW_BATCH_11.csv"
history12_path = YT / "ICMB_130_YOUTUBE_CANDIDATE_DECISION_TABLE_BATCH_12.csv"
history12_review_path = YT / "ICMB_130_YOUTUBE_CANDIDATE_REVIEW_BATCH_12.csv"

output_path = YT / "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_15.csv"

if output_path.exists():
    raise FileExistsError(
        "Batch 15 shortlist already exists. Inspect it before rerunning."
    )

screen = pd.read_csv(screening_path, dtype={"video_id": "string"})

# Normalize the new screening records.
new = pd.DataFrame({
    "movie_record_id": screen["movie_record_id"].astype(str),
    "movie_title": screen["movie_title"],
    "video_id": screen["video_id"].astype(str),
    "video_title": screen["metadata_title"],
    "channel_title": screen["channel_title_metadata"],
    "comment_count": pd.to_numeric(screen["comment_count"], errors="coerce").fillna(0),
    "source_type": screen["automated_source_type"],
    "decision": screen["automated_decision"],
    "candidate_origin": "Batch 15 discovery"
})

historical_frames = []

# Historical candidates from Batch 11.
if history11_path.exists():
    h = pd.read_csv(history11_path, dtype={"video_id": "string"})
    h = h[h["movie_record_id"].astype(str).isin(["2010_3"])].copy()

    if not h.empty:
        historical_frames.append(pd.DataFrame({
            "movie_record_id": h["movie_record_id"].astype(str),
            "movie_title": h["movie_title"],
            "video_id": h["video_id"].astype(str),
            "video_title": h.get("metadata_title", h.get("video_title")),
            "channel_title": h.get("channel_title_metadata"),
            "comment_count": pd.to_numeric(
                h.get("comment_count"), errors="coerce"
            ).fillna(0),
            "source_type": h.get("automated_source_type"),
            "decision": h.get("automated_decision"),
            "candidate_origin": "Historical Batch 11"
        }))

# Historical Rowdy Rathore candidates from Batch 12 files.
for path in [history12_path, history12_review_path]:
    if path.exists():
        h = pd.read_csv(path, dtype={"video_id": "string"})
        h = h[h["movie_record_id"].astype(str).isin(["2012_3"])].copy()

        if not h.empty:
            title_col = (
                "metadata_title" if "metadata_title" in h.columns
                else "video_title"
            )
            channel_col = (
                "channel_title_metadata"
                if "channel_title_metadata" in h.columns
                else "channel_title"
            )
            source_col = (
                "automated_source_type"
                if "automated_source_type" in h.columns
                else "source_type"
            )
            decision_col = (
                "automated_decision"
                if "automated_decision" in h.columns
                else "manual_decision"
            )

            historical_frames.append(pd.DataFrame({
                "movie_record_id": h["movie_record_id"].astype(str),
                "movie_title": h["movie_title"],
                "video_id": h["video_id"].astype(str),
                "video_title": h[title_col] if title_col in h.columns else None,
                "channel_title": h[channel_col] if channel_col in h.columns else None,
                "comment_count": pd.to_numeric(
                    h["comment_count"], errors="coerce"
                ).fillna(0) if "comment_count" in h.columns else 0,
                "source_type": h[source_col] if source_col in h.columns else None,
                "decision": h[decision_col] if decision_col in h.columns else None,
                "candidate_origin": path.name
            }))

# Combine candidates and remove duplicate movie/video pairs.
parts = [new] + historical_frames
pool = pd.concat(parts, ignore_index=True, sort=False)

pool["video_id"] = pool["video_id"].astype(str).str.strip()
pool["movie_record_id"] = pool["movie_record_id"].astype(str).str.strip()
pool = pool.drop_duplicates(
    subset=["movie_record_id", "video_id"],
    keep="first"
).copy()

# Identify video IDs already represented in the current comments dataset.
comments_path = (
    ROOT / "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_14.csv"
)
comments = pd.read_csv(
    comments_path,
    usecols=["movie_record_id", "video_id"],
    dtype={"movie_record_id": "string", "video_id": "string"}
)

acquired_pairs = set(zip(
    comments["movie_record_id"].astype(str).str.strip(),
    comments["video_id"].astype(str).str.strip()
))

pool["already_acquired"] = [
    (mid, vid) in acquired_pairs
    for mid, vid in zip(pool["movie_record_id"], pool["video_id"])
]

# Only review/reaction sources are eligible for this audience-comment shortlist.
eligible_types = {
    "Movie Review / Analysis",
    "Movie Reaction",
    "Review + Reaction",
    "Movie Review/Analysis",
    "Review/Reaction"
}

pool["eligible_source"] = (
    pool["source_type"].fillna("").str.strip().isin(eligible_types)
)

# Exclude previously acquired video sources.
eligible = pool[
    pool["eligible_source"] &
    (~pool["already_acquired"]) &
    (pool["comment_count"] > 0)
].copy()

# Prioritize confirmed automated shortlist, then comment availability.
eligible["decision_priority"] = eligible["decision"].map({
    "Shortlist": 0,
    "Needs Review": 1
}).fillna(2)

eligible = eligible.sort_values(
    ["movie_record_id", "decision_priority", "comment_count"],
    ascending=[True, True, False]
)

# Up to two candidate sources per movie.
shortlist = (
    eligible.groupby("movie_record_id", group_keys=False)
    .head(2)
    .copy()
)

shortlist["selection_rank"] = (
    shortlist.groupby("movie_record_id").cumcount() + 1
)

# Coverage for all 29 targets.
targets = {
    "2010_3": "Golmaal 3",
    "2012_3": "Rowdy Rathore",
    "2013_5": "Goliyon Ki Raasleela Ram-Leela",
    "2014_2": "Kick",
    "2014_3": "Happy New Year",
    "2014_4": "Bang Bang!",
    "2014_5": "Singham Returns",
    "2015_3": "Prem Ratan Dhan Payo",
    "2015_4": "Bajirao Mastani",
    "2015_5": "Tanu Weds Manu Returns",
    "2016_4": "M.S. Dhoni: The Untold Story",
    "2016_5": "Rustom",
    "2017_3": "Golmaal Again",
    "2017_4": "Raees",
    "2017_5": "Judwaa 2",
    "2018_2": "Sanju",
    "2018_3": "Padmaavat",
    "2018_4": "Simmba",
    "2019_3": "Kabir Singh",
    "2019_4": "Uri: The Surgical Strike",
    "2019_5": "Bharat",
    "2020_1": "Tanhaji: The Unsung Warrior",
    "2020_4": "Baaghi 3",
    "2020_5": "Street Dancer 3D",
    "2021_2": "Sooryavanshi",
    "2021_4": "83",
    "2023_4": "Gadar 2",
    "2024_5": "Bhool Bhulaiyaa 3",
    "2025_4": "Saiyaara"
}

coverage = pd.DataFrame([
    {"movie_record_id": mid, "movie_title": title}
    for mid, title in targets.items()
])

counts = shortlist.groupby("movie_record_id").agg(
    shortlisted_sources=("video_id", "nunique"),
    available_reported_comments=("comment_count", "sum")
).reset_index()

coverage = coverage.merge(counts, on="movie_record_id", how="left")
coverage[["shortlisted_sources", "available_reported_comments"]] = (
    coverage[["shortlisted_sources", "available_reported_comments"]]
    .fillna(0).astype(int)
)

coverage["source_status"] = coverage["shortlisted_sources"].map(
    lambda x: "Selected" if x > 0 else "No eligible source"
)

shortlist.to_csv(output_path, index=False)

print("=" * 75)
print("BATCH 15 — CONSOLIDATED SOURCE SHORTLIST")
print("=" * 75)
print("Unique candidates in combined pool:", len(pool))
print("Eligible review/reaction sources:", len(eligible))
print("Selected sources:", len(shortlist))
print("Movies with selected sources:", (coverage["shortlisted_sources"] > 0).sum())
print("Movies without selected sources:", (coverage["shortlisted_sources"] == 0).sum())

print("\nMOVIE-WISE SOURCE COVERAGE")
display(coverage)

print("\nSELECTED CANDIDATES")
display(shortlist[[
    "movie_record_id", "movie_title", "selection_rank",
    "video_id", "video_title", "channel_title",
    "source_type", "comment_count", "decision", "candidate_origin"
]].sort_values(["movie_record_id", "selection_rank"]))

print("\nSaved:", output_path)
print("Selection is metadata-based; content review remains pending.")

In [ ]:
# NOTEBOOK 10 — BATCH 15: FINAL SHORTLIST QUALITY REVIEW

import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

shortlist_path = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube"
    / "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_15.csv"
)

shortlist = pd.read_csv(shortlist_path)

quality_flags = {
    "BJqkSpvuVsI": "Full-movie upload — classify separately",
    "9MjEHViluh4": "Song reaction — classify separately",
    "QK0xstPCryw": "Wrong movie — Baaghi 4; exclude",
    "53HdBer8HDg": "Box-office discussion — exclude from audience reviews"
}

corrected_types = {
    "BJqkSpvuVsI": "Full Movie",
    "9MjEHViluh4": "Song / Music Reaction",
    "QK0xstPCryw": "Exclude — Wrong Movie",
    "53HdBer8HDg": "Box Office / Commercial Discussion"
}

shortlist["quality_flag"] = (
    shortlist["video_id"].map(quality_flags).fillna("")
)

shortlist["reviewed_source_type"] = shortlist.apply(
    lambda row: corrected_types.get(
        row["video_id"], row["source_type"]
    ),
    axis=1
)

display_columns = [
    col for col in [
        "movie_record_id",
        "movie_title",
        "video_id",
        "video_title",
        "channel_title",
        "source_type",
        "reviewed_source_type",
        "comment_count",
        "quality_flag"
    ]
    if col in shortlist.columns
]

print("=" * 75)
print("BATCH 15 — ALL SELECTED SOURCES")
print("=" * 75)
print("Records:", len(shortlist))
print("Movies:", shortlist["movie_record_id"].nunique())

# Display in original shortlist order; no sorting assumptions
display(shortlist[display_columns].reset_index(drop=True))

print("\nREVIEWED SOURCE TYPE DISTRIBUTION")
display(
    shortlist["reviewed_source_type"]
    .value_counts()
    .rename_axis("reviewed_source_type")
    .reset_index(name="records")
)

print("\nKNOWN QUALITY FLAGS")
display(
    shortlist.loc[
        shortlist["quality_flag"] != "",
        display_columns
    ].reset_index(drop=True)
)

In [ ]:
# NOTEBOOK 10 — BATCH 15: YOUTUBE COMMENT ACQUISITION

import os
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import userdata
from IPython.display import display

# --------------------------------------------------
# 1. PATHS AND API CLIENT
# --------------------------------------------------

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

YOUTUBE_DIR = (
    PROJECT_ROOT / "01_data/raw/audience_sentiment/youtube"
)

COMMENTS_DIR = YOUTUBE_DIR / "comments"
COMMENTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_OUTPUT = COMMENTS_DIR / "ICMB_130_YOUTUBE_COMMENTS_BATCH_15.csv"
LOG_OUTPUT = YOUTUBE_DIR / "ICMB_130_YOUTUBE_COMMENTS_ACQUISITION_LOG_BATCH_15.csv"

if RAW_OUTPUT.exists() or LOG_OUTPUT.exists():
    raise FileExistsError(
        "Batch 15 output already exists. No files were overwritten."
    )

api_key = userdata.get("YOUTUBE_API_KEY")

if not api_key:
    raise ValueError("YOUTUBE_API_KEY is unavailable in Colab Secrets.")

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

# --------------------------------------------------
# 2. PREPARE ELIGIBLE SOURCE LIST
# --------------------------------------------------

shortlist_path = (
    YOUTUBE_DIR / "ICMB_130_YOUTUBE_SOURCE_SHORTLIST_BATCH_15.csv"
)

shortlist = pd.read_csv(shortlist_path)

excluded_ids = {
    "QK0xstPCryw",  # Baaghi 4, not Baaghi 3
    "53HdBer8HDg"   # Box-office discussion
}

eligible = shortlist[
    ~shortlist["video_id"].isin(excluded_ids)
].copy()

# Preserve corrected source classifications
corrected_types = {
    "BJqkSpvuVsI": "Full Movie",
    "9MjEHViluh4": "Song / Music Reaction"
}

eligible["acquisition_source_type"] = eligible.apply(
    lambda row: corrected_types.get(
        row["video_id"], row["source_type"]
    ),
    axis=1
)

eligible = eligible.drop_duplicates(
    subset=["movie_record_id", "video_id"]
).reset_index(drop=True)

print("Eligible videos:", len(eligible))
print("Movies represented:", eligible["movie_record_id"].nunique())

# --------------------------------------------------
# 3. FETCH TOP-LEVEL COMMENTS
# --------------------------------------------------

all_comments = []
acquisition_log = []

for index, source in eligible.iterrows():

    video_id = source["video_id"]
    retrieved = 0
    status = "Success"
    error_type = ""

    try:
        response = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            order="time",
            textFormat="plainText"
        ).execute()

        for item in response.get("items", []):

            top_comment = item["snippet"]["topLevelComment"]
            comment = top_comment["snippet"]

            text = comment.get("textDisplay", "").strip()

            if not text:
                continue

            all_comments.append({
                "movie_record_id": source["movie_record_id"],
                "movie_title": source["movie_title"],
                "video_id": video_id,
                "video_title": source["video_title"],
                "channel_title": source["channel_title"],
                "source_type": source["acquisition_source_type"],
                "comment_id": top_comment["id"],
                "comment_text": text,
                "author": comment.get("authorDisplayName"),
                "published_at": comment.get("publishedAt"),
                "updated_at": comment.get("updatedAt"),
                "like_count": comment.get("likeCount", 0),
                "reply_count": item["snippet"].get("totalReplyCount", 0),
                "acquisition_order": retrieved + 1,
                "sampling_method": "top_level_comments_order_time_latest_100"
            })

            retrieved += 1

    except HttpError as exc:
        status = "API Error"
        error_type = type(exc).__name__

    except Exception as exc:
        status = "Failed"
        error_type = type(exc).__name__

    acquisition_log.append({
        "movie_record_id": source["movie_record_id"],
        "movie_title": source["movie_title"],
        "video_id": video_id,
        "video_title": source["video_title"],
        "source_type": source["acquisition_source_type"],
        "retrieved_comments": retrieved,
        "status": status,
        "error_type": error_type,
        "retrieved_at_utc": datetime.now(timezone.utc).isoformat()
    })

    print(
        f"{index + 1:02d}/{len(eligible)} | "
        f"{source['movie_title']} | "
        f"{video_id} | "
        f"{retrieved} comments | {status}"
    )

# --------------------------------------------------
# 4. SAVE RAW COMMENTS AND LOG
# --------------------------------------------------

comments_df = pd.DataFrame(all_comments)
log_df = pd.DataFrame(acquisition_log)

comments_df.to_csv(RAW_OUTPUT, index=False)
log_df.to_csv(LOG_OUTPUT, index=False)

# --------------------------------------------------
# 5. ACQUISITION SUMMARY
# --------------------------------------------------

print("\n" + "=" * 75)
print("BATCH 15 — COMMENT ACQUISITION SUMMARY")
print("=" * 75)

print("Eligible videos:", len(eligible))
print("Videos with comments:", (log_df["retrieved_comments"] > 0).sum())
print("Videos with zero comments:", (log_df["retrieved_comments"] == 0).sum())
print("Successful API requests:", (log_df["status"] == "Success").sum())
print("Failed/API error requests:", (log_df["status"] != "Success").sum())
print("Total comments acquired:", len(comments_df))
print("Unique comment IDs:", comments_df["comment_id"].nunique())
print("Raw file saved:", RAW_OUTPUT.exists())
print("Log file saved:", LOG_OUTPUT.exists())

print("\nSOURCE TYPE DISTRIBUTION")
if not comments_df.empty:
    display(
        comments_df.groupby("source_type")
        .agg(
            videos=("video_id", "nunique"),
            comments=("comment_id", "count")
        )
        .reset_index()
    )

print("\nACQUISITION LOG")
display(log_df)

In [ ]:
# NOTEBOOK 10 — BATCH 15: COMMENT CLEANING AND VALIDATION

import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_PATH = (
    PROJECT_ROOT
    / "01_data/raw/audience_sentiment/youtube/comments"
    / "ICMB_130_YOUTUBE_COMMENTS_BATCH_15.csv"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENTS_BATCH_15_CLEANED.csv"
)

if not RAW_PATH.exists():
    raise FileNotFoundError(f"Raw file missing: {RAW_PATH}")

if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"Output already exists; refusing to overwrite: {OUTPUT_PATH}"
    )

# --------------------------------------------------
# 1. LOAD RAW COMMENTS
# --------------------------------------------------

raw = pd.read_csv(RAW_PATH)

print("=" * 75)
print("BATCH 15 — RAW DATA VALIDATION")
print("=" * 75)

print("Raw shape:", raw.shape)
print("Columns:", raw.columns.tolist())

required_columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "video_title",
    "source_type",
    "comment_id",
    "comment_text",
    "published_at"
]

missing_columns = [
    col for col in required_columns
    if col not in raw.columns
]

if missing_columns:
    raise ValueError(f"Required columns missing: {missing_columns}")

# --------------------------------------------------
# 2. CHECK EMPTY TEXT AND DUPLICATES
# --------------------------------------------------

raw["comment_text"] = (
    raw["comment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

empty_text_count = (raw["comment_text"] == "").sum()
duplicate_id_count = raw["comment_id"].duplicated().sum()

# --------------------------------------------------
# 3. CLEAN WITHOUT MODIFYING RAW FILE
# --------------------------------------------------

clean = raw[raw["comment_text"] != ""].copy()

clean = clean.drop_duplicates(
    subset=["comment_id"],
    keep="first"
).reset_index(drop=True)

# --------------------------------------------------
# 4. VALIDATE CLEAN DATA
# --------------------------------------------------

print("\n" + "=" * 75)
print("CLEANING SUMMARY")
print("=" * 75)

print("Raw records:", len(raw))
print("Empty comment texts:", empty_text_count)
print("Duplicate comment IDs:", duplicate_id_count)
print("Clean records:", len(clean))
print("Unique comment IDs:", clean["comment_id"].nunique())
print("Movies represented:", clean["movie_record_id"].nunique())
print("Videos represented:", clean["video_id"].nunique())

print("\nMissing values in important fields:")
display(
    clean[
        ["movie_record_id", "video_id", "comment_id",
         "comment_text", "published_at", "source_type"]
    ].isna().sum().rename("missing_count").to_frame()
)

# --------------------------------------------------
# 5. SAVE CLEANED DATA
# --------------------------------------------------

clean.to_csv(OUTPUT_PATH, index=False)

print("\nCleaned file saved:", OUTPUT_PATH.exists())
print("Saved shape:", clean.shape)

print("\nMOVIE-WISE COMMENT COUNTS")
display(
    clean.groupby(
        ["movie_record_id", "movie_title"],
        dropna=False
    )
    .agg(
        comments=("comment_id", "count"),
        videos=("video_id", "nunique")
    )
    .reset_index()
)

print("\nSOURCE TYPE COUNTS")
display(
    clean["source_type"]
    .value_counts()
    .rename_axis("source_type")
    .reset_index(name="comments")
)

In [ ]:
# NOTEBOOK 10 — CONSOLIDATE YOUTUBE COMMENTS: BATCHES 01–15

import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

OLD_PATH = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_PILOT_BATCH_01_14.csv"
)

BATCH15_PATH = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENTS_BATCH_15_CLEANED.csv"
)

OUTPUT_PATH = (
    PROCESSED_DIR
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_15.csv"
)

# --------------------------------------------------
# 1. SAFETY CHECKS
# --------------------------------------------------

for path in [OLD_PATH, BATCH15_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

if OUTPUT_PATH.exists():
    raise FileExistsError(
        "01–15 consolidated output already exists. "
        "No existing file will be overwritten."
    )

# --------------------------------------------------
# 2. LOAD LATEST CONSOLIDATED FILE AND BATCH 15
# --------------------------------------------------

old = pd.read_csv(OLD_PATH, low_memory=False)
batch15 = pd.read_csv(BATCH15_PATH, low_memory=False)

print("=" * 75)
print("CONSOLIDATION INPUTS")
print("=" * 75)

print("Batches 01–14:", old.shape)
print("Batch 15:", batch15.shape)

# --------------------------------------------------
# 3. SCHEMA ALIGNMENT
# --------------------------------------------------

all_columns = list(dict.fromkeys(
    list(old.columns) + list(batch15.columns)
))

old_aligned = old.reindex(columns=all_columns)
batch15_aligned = batch15.reindex(columns=all_columns)

# --------------------------------------------------
# 4. COMBINE
# --------------------------------------------------

combined = pd.concat(
    [old_aligned, batch15_aligned],
    ignore_index=True
)

# --------------------------------------------------
# 5. DUPLICATE COMMENT ID AUDIT
# --------------------------------------------------

if "comment_id" not in combined.columns:
    raise ValueError("comment_id column missing.")

duplicate_mask = combined["comment_id"].duplicated(keep="first")
duplicate_count = int(duplicate_mask.sum())

# Preserve first occurrence
combined_clean = combined.loc[~duplicate_mask].copy()

# --------------------------------------------------
# 6. SAVE NEW CONSOLIDATED DATASET
# --------------------------------------------------

combined_clean.to_csv(OUTPUT_PATH, index=False)

# --------------------------------------------------
# 7. FINAL SUMMARY
# --------------------------------------------------

print("\n" + "=" * 75)
print("BATCHES 01–15 CONSOLIDATION SUMMARY")
print("=" * 75)

print("Batches 01–14 rows:", len(old))
print("Batch 15 rows:", len(batch15))
print("Rows before duplicate handling:", len(combined))
print("Duplicate comment IDs:", duplicate_count)
print("Final consolidated rows:", len(combined_clean))
print("Unique comment IDs:", combined_clean["comment_id"].nunique())
print("Unique videos:", combined_clean["video_id"].nunique())
print("Unique movies:", combined_clean["movie_record_id"].nunique())
print("Output saved:", OUTPUT_PATH.exists())
print("Output shape:", combined_clean.shape)

# --------------------------------------------------
# 8. MOVIE-WISE COVERAGE
# --------------------------------------------------

coverage = (
    combined_clean.groupby(
        ["movie_record_id", "movie_title"],
        dropna=False
    )
    .agg(
        comments=("comment_id", "count"),
        videos=("video_id", "nunique")
    )
    .reset_index()
)

print("\nMOVIE-WISE COMMENT COVERAGE")
display(coverage.sort_values("movie_record_id"))

print("\nConsolidation completed. Original 01–14 file preserved.")

In [ ]:
# NOTEBOOK 10 — FINAL AUDIENCE COMMENT COVERAGE AUDIT
# STEP 1: LOCATE ICMB-130 MASTER DATASET

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

print("=" * 75)
print("SEARCHING FOR ICMB-130 MASTER DATASET")
print("=" * 75)

master_candidates = []

for path in PROJECT_ROOT.rglob("*.csv"):
    name = path.name.upper()

    if "MASTER" in name and "DATASET" in name:
        master_candidates.append(path)

if not master_candidates:
    print("No master dataset CSV found.")
else:
    for i, path in enumerate(master_candidates, 1):
        print(f"\n{i}. {path}")
        print(f"   Size: {path.stat().st_size / 1024:.2f} KB")

print("\n" + "=" * 75)
print("CONSOLIDATED AUDIENCE DATASET")
print("=" * 75)

consolidated_path = (
    PROJECT_ROOT
    / "01_data/processed/audience_sentiment"
    / "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_15.csv"
)

print("Exists:", consolidated_path.exists())
if consolidated_path.exists():
    print("Path:", consolidated_path)

In [ ]:
# NOTEBOOK 10 — LOCATE ICMB-130 MASTER DATASET BY SCHEMA

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

required_columns = {
    "movie_record_id",
    "movie_title",
    "year",
    "rank",
    "language",
    "india_gross_cr",
    "tmdb_id"
}

matches = []

print("=" * 75)
print("SEARCHING PROJECT FILES FOR ICMB-130 SCHEMA")
print("=" * 75)

for path in PROJECT_ROOT.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in [".csv", ".xlsx", ".xls"]:
        continue

    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path, nrows=5)
        else:
            df = pd.read_excel(path, nrows=5)

        columns = set(
            str(c).strip().lower() for c in df.columns
        )

        overlap = required_columns.intersection(columns)

        if len(overlap) >= 4:
            matches.append((path, overlap, list(df.columns)))

    except Exception:
        continue

if matches:
    for i, (path, overlap, columns) in enumerate(matches, 1):
        print(f"\nCandidate {i}: {path}")
        print("Matching schema columns:", sorted(overlap))
        print("Available columns:", columns)
else:
    print("No matching master dataset found in CSV/Excel files.")

print("\nSearch completed. No files were modified.")

In [ ]:
# NOTEBOOK 10 — FINAL AUDIENCE COVERAGE AUDIT (BATCHES 01–15)

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

MASTER_PATH = (
    PROJECT_ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
)

COMMENTS_PATH = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_15.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT / "06_outputs/audience_sentiment"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COVERAGE_PATH = (
    OUTPUT_DIR / "ICMB_130_FINAL_YOUTUBE_COMMENT_COVERAGE_BATCH_01_15.csv"
)

# Input validation
for path in [MASTER_PATH, COMMENTS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing input: {path}")

if COVERAGE_PATH.exists():
    raise FileExistsError(
        f"Output already exists; preserving it: {COVERAGE_PATH}"
    )

# Load datasets
master = pd.read_csv(MASTER_PATH)
comments = pd.read_csv(COMMENTS_PATH)

print("=" * 75)
print("ICMB-130 FINAL AUDIENCE COVERAGE AUDIT")
print("=" * 75)

print("\nMASTER DATASET:", master.shape)
print("CONSOLIDATED COMMENTS:", comments.shape)

# Validate required columns
required_master = {
    "movie_record_id", "movie_title", "year", "rank", "language"
}
required_comments = {
    "movie_record_id", "comment_id", "video_id"
}

assert required_master.issubset(master.columns), (
    f"Master columns missing: {required_master - set(master.columns)}"
)
assert required_comments.issubset(comments.columns), (
    f"Comment columns missing: {required_comments - set(comments.columns)}"
)

# Verify master uniqueness
assert master["movie_record_id"].is_unique, (
    "Master dataset contains duplicate movie_record_id values."
)

# Movie-wise coverage
movie_counts = (
    comments.groupby("movie_record_id")
    .agg(
        total_comments=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique")
    )
    .reset_index()
)

coverage = master.merge(
    movie_counts,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

coverage["total_comments"] = (
    coverage["total_comments"].fillna(0).astype(int)
)
coverage["unique_videos"] = (
    coverage["unique_videos"].fillna(0).astype(int)
)

coverage["coverage_status"] = coverage["total_comments"].apply(
    lambda x: "Covered" if x > 0 else "Not Covered"
)

coverage["coverage_category"] = coverage["total_comments"].apply(
    lambda x:
        "No Comments" if x == 0
        else "Below 100" if x < 100
        else "100 or More"
)

# Summary
total_movies = len(coverage)
covered_movies = (coverage["total_comments"] > 0).sum()
uncovered_movies = total_movies - covered_movies

print("\n" + "=" * 75)
print("FINAL COVERAGE SUMMARY")
print("=" * 75)

print(f"Master movies:             {total_movies}")
print(f"Movies with comments:      {covered_movies}")
print(f"Movies without comments:   {uncovered_movies}")
print(f"Coverage percentage:       {covered_movies / total_movies * 100:.2f}%")
print(f"Total comment records:     {len(comments):,}")
print(f"Unique comment IDs:        {comments['comment_id'].nunique():,}")
print(f"Unique videos:             {comments['video_id'].nunique():,}")

print("\n" + "=" * 75)
print("UNCOVERED MOVIES")
print("=" * 75)

uncovered = coverage[
    coverage["coverage_status"] == "Not Covered"
][["movie_record_id", "year", "rank", "movie_title", "language"]]

if uncovered.empty:
    print("All 130 movies have audience comments.")
else:
    display(uncovered.reset_index(drop=True))

print("\n" + "=" * 75)
print("COVERAGE CATEGORY DISTRIBUTION")
print("=" * 75)

display(
    coverage["coverage_category"]
    .value_counts()
    .rename_axis("coverage_category")
    .reset_index(name="movies")
)

# Save final coverage report
coverage.to_csv(COVERAGE_PATH, index=False)

print("\nCoverage report saved:", COVERAGE_PATH)
print("Saved shape:", coverage.shape)

In [ ]:
from google.colab import userdata
from googleapiclient.discovery import build
import pandas as pd

# --------------------------------------------------
# BATCH 16 — VALIDATE THREE REMAINING MOVIE SOURCES
# --------------------------------------------------

video_sources = [
    {
        "movie_record_id": "2014_2",
        "movie_title": "Kick",
        "year": 2014,
        "video_id": "pIDVoQWzCyY"
    },
    {
        "movie_record_id": "2018_2",
        "movie_title": "Sanju",
        "year": 2018,
        "video_id": "pRWsH7UsrIg"
    },
    {
        "movie_record_id": "2010_3",
        "movie_title": "Golmaal 3",
        "year": 2010,
        "video_id": "XaXKh-s9zws"
    }
]

api_key = userdata.get("YOUTUBE_API_KEY")

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

video_ids = [x["video_id"] for x in video_sources]

response = youtube.videos().list(
    part="snippet,statistics",
    id=",".join(video_ids),
    maxResults=50
).execute()

metadata = {
    item["id"]: item
    for item in response.get("items", [])
}

results = []

for source in video_sources:
    item = metadata.get(source["video_id"])

    if item:
        snippet = item["snippet"]
        stats = item.get("statistics", {})

        results.append({
            **source,
            "video_found": True,
            "video_title": snippet.get("title"),
            "channel_title": snippet.get("channelTitle"),
            "published_at": snippet.get("publishedAt"),
            "comment_count": stats.get("commentCount"),
            "view_count": stats.get("viewCount"),
            "like_count": stats.get("likeCount")
        })
    else:
        results.append({
            **source,
            "video_found": False,
            "video_title": None,
            "channel_title": None,
            "published_at": None,
            "comment_count": None,
            "view_count": None,
            "like_count": None
        })

validation_df = pd.DataFrame(results)

print("=" * 75)
print("BATCH 16 — YOUTUBE VIDEO VALIDATION")
print("=" * 75)
display(validation_df)

print("\nVideos found:", validation_df["video_found"].sum(), "/ 3")
print(
    "Videos with comments:",
    validation_df["comment_count"].notna().sum()
)

In [ ]:
from pathlib import Path
from google.colab import userdata
from googleapiclient.discovery import build
import pandas as pd
from datetime import datetime, timezone
import time

# ============================================================
# BATCH 16 — YOUTUBE COMMENT ACQUISITION
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_DIR = (
    PROJECT_ROOT /
    "01_data/raw/audience_sentiment/youtube/comments"
)
RAW_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = RAW_DIR / "ICMB_130_YOUTUBE_COMMENTS_BATCH_16.csv"
LOG_PATH = RAW_DIR / "ICMB_130_YOUTUBE_COMMENTS_ACQUISITION_LOG_BATCH_16.csv"

if OUTPUT_PATH.exists() or LOG_PATH.exists():
    raise FileExistsError(
        "Batch 16 output already exists. Nothing was overwritten. "
        "Inspect the existing files before proceeding."
    )

sources = [
    {
        "movie_record_id": "2014_2",
        "movie_title": "Kick",
        "video_id": "pIDVoQWzCyY",
        "video_title": "KICK (Part 1) Movie Reaction",
        "channel_title": "CineDesi",
        "source_type": "Movie Reaction"
    },
    {
        "movie_record_id": "2018_2",
        "movie_title": "Sanju",
        "video_id": "pRWsH7UsrIg",
        "video_title": "SANJU Movie Reaction Part 1/3",
        "channel_title": "CineDesi",
        "source_type": "Movie Reaction"
    },
    {
        "movie_record_id": "2010_3",
        "movie_title": "Golmaal 3",
        "video_id": "XaXKh-s9zws",
        "video_title": "Golmaal 3 Full Movie",
        "channel_title": "Shemaroo Movies",
        "source_type": "Full Movie"
    }
]

api_key = userdata.get("YOUTUBE_API_KEY")

youtube = build(
    "youtube",
    "v3",
    developerKey=api_key,
    cache_discovery=False
)

all_comments = []
acquisition_log = []

for source in sources:

    print(f"\nAcquiring: {source['movie_title']}")

    movie_comments = []
    next_page_token = None
    request_success = True
    error_type = None

    try:
        while len(movie_comments) < 100:

            response = youtube.commentThreads().list(
                part="snippet",
                videoId=source["video_id"],
                maxResults=min(100, 100 - len(movie_comments)),
                order="time",
                textFormat="plainText",
                pageToken=next_page_token
            ).execute()

            for item in response.get("items", []):

                comment = item["snippet"]["topLevelComment"]
                snippet = comment["snippet"]

                movie_comments.append({
                    "movie_record_id": source["movie_record_id"],
                    "movie_title": source["movie_title"],
                    "video_id": source["video_id"],
                    "video_title": source["video_title"],
                    "channel_title": source["channel_title"],
                    "source_type": source["source_type"],
                    "comment_id": comment["id"],
                    "comment_text": snippet.get("textDisplay", ""),
                    "author": snippet.get("authorDisplayName"),
                    "published_at": snippet.get("publishedAt"),
                    "updated_at": snippet.get("updatedAt"),
                    "like_count": snippet.get("likeCount", 0),
                    "reply_count": item["snippet"].get(
                        "totalReplyCount", 0
                    ),
                    "acquisition_order": len(movie_comments) + 1,
                    "sampling_method": (
                        "YouTube order=time; latest top-level comments; "
                        "maximum 100 per video; replies not acquired"
                    )
                })

                if len(movie_comments) >= 100:
                    break

            next_page_token = response.get("nextPageToken")

            if not next_page_token or not response.get("items"):
                break

            time.sleep(0.2)

    except Exception as e:
        request_success = False
        error_type = type(e).__name__

    all_comments.extend(movie_comments)

    acquisition_log.append({
        "movie_record_id": source["movie_record_id"],
        "movie_title": source["movie_title"],
        "video_id": source["video_id"],
        "source_type": source["source_type"],
        "comments_acquired": len(movie_comments),
        "request_status": (
            "Success" if request_success else "Failed"
        ),
        "error_type": error_type,
        "sampling_method": "Latest top-level comments; order=time"
    })

    print("Comments acquired:", len(movie_comments))
    print("Status:", "Success" if request_success else "Failed")

# ------------------------------------------------------------
# SAVE RAW COMMENTS AND ACQUISITION LOG
# ------------------------------------------------------------

columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "video_title",
    "channel_title",
    "source_type",
    "comment_id",
    "comment_text",
    "author",
    "published_at",
    "updated_at",
    "like_count",
    "reply_count",
    "acquisition_order",
    "sampling_method"
]

comments_df = pd.DataFrame(all_comments, columns=columns)
log_df = pd.DataFrame(acquisition_log)

comments_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
log_df.to_csv(LOG_PATH, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# ACQUISITION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BATCH 16 — ACQUISITION SUMMARY")
print("=" * 75)

print("Total comments:", len(comments_df))
print("Unique comment IDs:", comments_df["comment_id"].nunique())
print("Unique videos:", comments_df["video_id"].nunique())
print("Movies covered:", comments_df["movie_record_id"].nunique())

print("\nSOURCE-WISE COMMENT COUNTS")
display(
    comments_df.groupby(
        ["movie_title", "source_type"]
    ).size().reset_index(name="comments_acquired")
)

print("\nACQUISITION LOG")
display(log_df)

print("\nRaw comments saved:", OUTPUT_PATH)
print("Acquisition log saved:", LOG_PATH)
print("Saved shape:", comments_df.shape)

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# BATCH 16 — VALIDATION AND CLEANING
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

RAW_PATH = (
    PROJECT_ROOT /
    "01_data/raw/audience_sentiment/youtube/comments/"
    "ICMB_130_YOUTUBE_COMMENTS_BATCH_16.csv"
)

PROCESSED_DIR = (
    PROJECT_ROOT /
    "01_data/processed/audience_sentiment"
)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_COMMENTS_BATCH_16_CLEANED.csv"
)

if not RAW_PATH.exists():
    raise FileNotFoundError(RAW_PATH)

if CLEAN_PATH.exists():
    raise FileExistsError(
        f"Cleaned file already exists: {CLEAN_PATH}"
    )

df = pd.read_csv(RAW_PATH)

print("=" * 75)
print("BATCH 16 — DATA VALIDATION")
print("=" * 75)

print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

required_columns = [
    "movie_record_id",
    "movie_title",
    "video_id",
    "source_type",
    "comment_id",
    "comment_text",
    "published_at"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("\nMissing required columns:", missing_columns)
print("Duplicate comment IDs:", df["comment_id"].duplicated().sum())
print("Missing comment IDs:", df["comment_id"].isna().sum())

empty_text = (
    df["comment_text"].fillna("").str.strip().eq("")
).sum()

print("Empty comment text:", empty_text)

# Preserve original records; exclude only invalid IDs or empty text.
clean_df = df.copy()

clean_df = clean_df[
    clean_df["comment_id"].notna()
    & clean_df["comment_text"].fillna("").str.strip().ne("")
].copy()

clean_df["comment_text"] = (
    clean_df["comment_text"].astype(str).str.strip()
)

print("\n" + "=" * 75)
print("CLEANING SUMMARY")
print("=" * 75)

print("Raw records:", len(df))
print("Clean records:", len(clean_df))
print("Excluded records:", len(df) - len(clean_df))
print("Unique comment IDs:", clean_df["comment_id"].nunique())
print("Unique videos:", clean_df["video_id"].nunique())
print("Unique movies:", clean_df["movie_record_id"].nunique())

print("\nMovie-wise counts:")
display(
    clean_df.groupby(
        ["movie_record_id", "movie_title", "source_type"]
    ).size().reset_index(name="comments")
)

clean_df.to_csv(
    CLEAN_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nCleaned file saved:", CLEAN_PATH)
print("Cleaned shape:", clean_df.shape)

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# FINAL CONSOLIDATION — BATCHES 01–16
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

PROCESSED_DIR = (
    PROJECT_ROOT / "01_data/processed/audience_sentiment"
)

OLD_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_15.csv"
)

BATCH16_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_COMMENTS_BATCH_16_CLEANED.csv"
)

NEW_PATH = (
    PROCESSED_DIR /
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_16.csv"
)

if not OLD_PATH.exists():
    raise FileNotFoundError(OLD_PATH)

if not BATCH16_PATH.exists():
    raise FileNotFoundError(BATCH16_PATH)

if NEW_PATH.exists():
    raise FileExistsError(
        f"Output already exists; no file overwritten: {NEW_PATH}"
    )

# Load datasets
old_df = pd.read_csv(OLD_PATH, low_memory=False)
batch16_df = pd.read_csv(BATCH16_PATH, low_memory=False)

print("=" * 75)
print("FINAL CONSOLIDATION — BATCHES 01–16")
print("=" * 75)

print("Batches 01–15:", old_df.shape)
print("Batch 16:", batch16_df.shape)

# Align columns without discarding source-specific information
all_columns = list(old_df.columns)

for col in batch16_df.columns:
    if col not in all_columns:
        all_columns.append(col)

old_df = old_df.reindex(columns=all_columns)
batch16_df = batch16_df.reindex(columns=all_columns)

combined_df = pd.concat(
    [old_df, batch16_df],
    ignore_index=True
)

# Duplicate audit
duplicate_ids = combined_df["comment_id"].duplicated().sum()
missing_ids = combined_df["comment_id"].isna().sum()

print("\nRecords before duplicate handling:", len(combined_df))
print("Duplicate comment IDs:", duplicate_ids)
print("Missing comment IDs:", missing_ids)

if duplicate_ids > 0 or missing_ids > 0:
    raise ValueError(
        "Comment ID integrity check failed. "
        "Do not save until duplicates/missing IDs are reviewed."
    )

# Final save
combined_df.to_csv(
    NEW_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 75)
print("CONSOLIDATION COMPLETE")
print("=" * 75)

print("Final records:", len(combined_df))
print("Unique comment IDs:", combined_df["comment_id"].nunique())
print("Unique videos:", combined_df["video_id"].nunique())
print("Unique movies:", combined_df["movie_record_id"].nunique())
print("Final shape:", combined_df.shape)
print("\nSaved:", NEW_PATH)

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# ICMB-130 — FINAL AUDIENCE COVERAGE AUDIT (BATCHES 01–16)
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

MASTER_PATH = (
    PROJECT_ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
)

COMMENTS_PATH = (
    PROJECT_ROOT /
    "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_16.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "06_outputs/audience_sentiment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = (
    OUTPUT_DIR /
    "ICMB_130_FINAL_YOUTUBE_COMMENT_COVERAGE_BATCH_01_16.csv"
)

if not MASTER_PATH.exists():
    raise FileNotFoundError(MASTER_PATH)

if not COMMENTS_PATH.exists():
    raise FileNotFoundError(COMMENTS_PATH)

if REPORT_PATH.exists():
    raise FileExistsError(
        f"Report already exists. No file overwritten: {REPORT_PATH}"
    )

master_df = pd.read_csv(MASTER_PATH)
comments_df = pd.read_csv(COMMENTS_PATH, low_memory=False)

# ------------------------------------------------------------
# SCHEMA VALIDATION
# ------------------------------------------------------------

required_master = [
    "movie_record_id", "year", "rank",
    "movie_title", "language"
]

required_comments = [
    "movie_record_id", "comment_id",
    "video_id", "comment_text"
]

assert all(c in master_df.columns for c in required_master)
assert all(c in comments_df.columns for c in required_comments)

assert master_df["movie_record_id"].is_unique, (
    "Master dataset contains duplicate movie_record_id values."
)

# ------------------------------------------------------------
# MOVIE-WISE COMMENT COVERAGE
# ------------------------------------------------------------

comment_summary = (
    comments_df.groupby("movie_record_id")
    .agg(
        total_comments=("comment_id", "count"),
        unique_comment_ids=("comment_id", "nunique"),
        unique_videos=("video_id", "nunique")
    )
    .reset_index()
)

coverage_df = master_df[
    required_master
].merge(
    comment_summary,
    on="movie_record_id",
    how="left"
)

for col in [
    "total_comments",
    "unique_comment_ids",
    "unique_videos"
]:
    coverage_df[col] = coverage_df[col].fillna(0).astype(int)

coverage_df["coverage_status"] = coverage_df[
    "total_comments"
].apply(
    lambda x: "Covered" if x > 0 else "Not Covered"
)

coverage_df["coverage_category"] = coverage_df[
    "total_comments"
].apply(
    lambda x:
        "No Comments" if x == 0
        else "Below 100" if x < 100
        else "100 or More"
)

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

total_movies = len(coverage_df)
covered_movies = (
    coverage_df["coverage_status"] == "Covered"
).sum()

uncovered_movies = total_movies - covered_movies

print("=" * 75)
print("ICMB-130 FINAL AUDIENCE COVERAGE AUDIT — BATCHES 01–16")
print("=" * 75)

print("MASTER DATASET:", master_df.shape)
print("CONSOLIDATED COMMENTS:", comments_df.shape)

print("\nFINAL COVERAGE SUMMARY")
print("-" * 75)
print("Master movies:            ", total_movies)
print("Movies with comments:     ", covered_movies)
print("Movies without comments:  ", uncovered_movies)
print(
    "Coverage percentage:      ",
    round(covered_movies / total_movies * 100, 2), "%"
)
print("Total comment records:    ", len(comments_df))
print("Unique comment IDs:       ", comments_df["comment_id"].nunique())
print("Unique videos:            ", comments_df["video_id"].nunique())

print("\nUNCOVERED MOVIES")
print("-" * 75)

uncovered_df = coverage_df[
    coverage_df["coverage_status"] == "Not Covered"
]

if uncovered_df.empty:
    print("None — all 130 movies have audience comments.")
else:
    display(uncovered_df[required_master])

print("\nCOVERAGE CATEGORY DISTRIBUTION")
print("-" * 75)

display(
    coverage_df.groupby(
        "coverage_category"
    ).size().reset_index(name="movies")
)

# ------------------------------------------------------------
# SAVE FINAL REPORT
# ------------------------------------------------------------

coverage_df.to_csv(
    REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nFinal coverage report saved:", REPORT_PATH)
print("Saved shape:", coverage_df.shape)

In [ ]:
from pathlib import Path
import pandas as pd

# ============================================================
# ICMB-130 — LANGUAGE IDENTIFICATION: INITIAL INSPECTION
# ============================================================

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_PATH = (
    PROJECT_ROOT /
    "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_16.csv"
)

df = pd.read_csv(COMMENTS_PATH, low_memory=False)

print("=" * 75)
print("ICMB-130 — LANGUAGE IDENTIFICATION: INITIAL INSPECTION")
print("=" * 75)

print("\nDataset shape:", df.shape)

# Check existing language-related columns
language_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in [
        "language", "lang", "text", "comment"
    ])
]

print("\nLanguage / text-related columns:")
print(language_columns)

# Check source language metadata
if "language" in df.columns:
    print("\nMovie language distribution:")
    display(
        df["language"]
        .value_counts(dropna=False)
        .rename_axis("movie_language")
        .reset_index(name="movie_count")
    )

# Comment text inspection
print("\nComment text availability:")
print("Missing comments:", df["comment_text"].isna().sum())
print(
    "Empty comments:",
    df["comment_text"].fillna("").str.strip().eq("").sum()
)

print("\nSample audience comments:")
for i, comment in enumerate(
    df["comment_text"].dropna().head(15),
    start=1
):
    print(f"{i}. {str(comment)[:300]}")

print("\nUnique movie records:", df["movie_record_id"].nunique())
print("Unique videos:", df["video_id"].nunique())
print("Unique comments:", df["comment_id"].nunique())

In [ ]:
# ============================================================
# ICMB-130 — LANGUAGE DETECTION ENVIRONMENT CHECK
# ============================================================

import importlib.util
import sys

print("=" * 75)
print("LANGUAGE IDENTIFICATION — ENVIRONMENT CHECK")
print("=" * 75)

print("Python version:", sys.version)

libraries = [
    "langdetect",
    "lingua",
    "fasttext",
    "transformers",
    "torch",
    "indicnlp",
    "regex",
    "pandas",
    "numpy"
]

for library in libraries:
    available = importlib.util.find_spec(library) is not None
    print(f"{library:<15} : {'Available' if available else 'Not installed'}")

print("\nEnvironment inspection completed.")

In [ ]:
# ============================================================
# ICMB-130 — MULTILINGUAL MODEL ENVIRONMENT CHECK
# ============================================================

import torch
import transformers

print("=" * 75)
print("MULTILINGUAL LANGUAGE IDENTIFICATION ENVIRONMENT")
print("=" * 75)

print("Transformers version:", transformers.__version__)
print("PyTorch version:", torch.__version__)

print("\nGPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
    print(
        "GPU memory (GB):",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3), 2
        )
    )
else:
    print("Running on CPU")

print("\nAvailable model pipeline:",
      hasattr(transformers, "pipeline"))

print("\nEnvironment check completed.")

In [ ]:
# ============================================================
# ICMB-130 — ALTERNATIVE LANGUAGE MODEL ACCESS CHECK
# ============================================================

from transformers import AutoConfig

MODEL_NAME = "papluca/xlm-roberta-base-language-detection"

print("=" * 75)
print("HUGGING FACE LANGUAGE MODEL ACCESS CHECK")
print("=" * 75)

try:
    config = AutoConfig.from_pretrained(MODEL_NAME)

    print("Model:", MODEL_NAME)
    print("Model type:", config.model_type)
    print("Number of labels:", config.num_labels)

    print("\nAvailable language labels:")

    if hasattr(config, "id2label"):
        for label_id, label in config.id2label.items():
            print(label_id, ":", label)

    print("\nModel configuration loaded successfully.")

except Exception as e:
    print("Model access failed.")
    print("Error type:", type(e).__name__)
    print("Error:", str(e)[:500])

In [ ]:
# ============================================================
# ICMB-130 — LANGUAGE IDENTIFICATION PILOT
# ============================================================

import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_PATH = (
    PROJECT_ROOT /
    "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_16.csv"
)

MODEL_NAME = "papluca/xlm-roberta-base-language-detection"

df = pd.read_csv(COMMENTS_PATH, low_memory=False)

# Select 30 non-empty comments for pilot testing
pilot_df = (
    df[df["comment_text"].notna()]
    .loc[
        df["comment_text"].fillna("").str.strip().ne(""),
        ["movie_title", "comment_text"]
    ]
    .sample(n=min(30, len(df)), random_state=42)
    .reset_index(drop=True)
)

print("=" * 75)
print("ICMB-130 — LANGUAGE IDENTIFICATION PILOT")
print("=" * 75)

print("Pilot comments:", len(pilot_df))
print("Model:", MODEL_NAME)
print("Device: CPU")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=-1
)

# Run pilot predictions
predictions = classifier(
    pilot_df["comment_text"].astype(str).tolist(),
    batch_size=8,
    truncation=True,
    max_length=128
)

pilot_df["predicted_language"] = [
    p["label"] for p in predictions
]

pilot_df["language_confidence"] = [
    round(float(p["score"]), 4) for p in predictions
]

print("\nPILOT PREDICTIONS")
display(pilot_df)

print("\nPREDICTED LANGUAGE DISTRIBUTION")
display(
    pilot_df["predicted_language"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="comments")
)

print("\nPilot completed. Original dataset remains unchanged.")

In [ ]:
# ============================================================
# ICMB-130 — SCRIPT DISTRIBUTION ANALYSIS
# ============================================================

import pandas as pd
import regex
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_PATH = (
    PROJECT_ROOT /
    "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_16.csv"
)

df = pd.read_csv(COMMENTS_PATH, low_memory=False)

# Unicode script patterns
script_patterns = {
    "Devanagari": r"\p{Devanagari}",
    "Bengali": r"\p{Bengali}",
    "Tamil": r"\p{Tamil}",
    "Telugu": r"\p{Telugu}",
    "Kannada": r"\p{Kannada}",
    "Malayalam": r"\p{Malayalam}",
    "Gujarati": r"\p{Gujarati}",
    "Gurmukhi": r"\p{Gurmukhi}",
    "Arabic": r"\p{Arabic}",
    "Latin": r"\p{Latin}"
}

def detect_scripts(text):
    if pd.isna(text) or not str(text).strip():
        return "Empty"

    found = [
        script for script, pattern in script_patterns.items()
        if regex.search(pattern, str(text))
    ]

    if not found:
        return "No recognized script"

    if len(found) > 1:
        return "Mixed: " + " + ".join(found)

    return found[0]

df["script_type"] = df["comment_text"].apply(detect_scripts)

print("=" * 75)
print("ICMB-130 — SCRIPT DISTRIBUTION")
print("=" * 75)

print("\nScript distribution:")
display(
    df["script_type"]
    .value_counts(dropna=False)
    .rename_axis("script_type")
    .reset_index(name="comment_count")
)

print("\nScript distribution by movie language:")
if "language" in df.columns:
    display(
        pd.crosstab(
            df["language"],
            df["script_type"]
        )
    )

print("\nMixed-script examples:")
display(
    df.loc[
        df["script_type"].str.startswith("Mixed", na=False),
        ["movie_title", "comment_text", "script_type"]
    ].head(15)
)

print("\nScript analysis completed.")

In [ ]:
# ============================================================
# ICMB-130 — LATIN SCRIPT LANGUAGE ERROR ANALYSIS
# ============================================================

import pandas as pd
from pathlib import Path
from transformers import pipeline

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_PATH = (
    PROJECT_ROOT /
    "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_16.csv"
)

MODEL_NAME = "papluca/xlm-roberta-base-language-detection"

df = pd.read_csv(COMMENTS_PATH, low_memory=False)

# Select comments containing Latin script
latin_df = df[
    df["comment_text"].fillna("").str.contains(
        r"[A-Za-z]", regex=True
    )
].copy()

# Deterministic sample for reproducibility
sample_df = latin_df.sample(
    n=min(100, len(latin_df)),
    random_state=42
).reset_index(drop=True)

# Reuse model if already loaded in notebook
if "classifier" not in globals():
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME
    )

    classifier = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        device=-1
    )

predictions = classifier(
    sample_df["comment_text"].astype(str).tolist(),
    batch_size=8,
    truncation=True,
    max_length=128
)

sample_df["predicted_language"] = [
    p["label"] for p in predictions
]

sample_df["confidence"] = [
    round(float(p["score"]), 4) for p in predictions
]

print("=" * 75)
print("LATIN SCRIPT LANGUAGE ERROR ANALYSIS")
print("=" * 75)

print("Latin-script comments:", len(latin_df))
print("Sample analyzed:", len(sample_df))

print("\nPREDICTED LANGUAGE DISTRIBUTION")
display(
    sample_df["predicted_language"]
    .value_counts()
    .rename_axis("predicted_language")
    .reset_index(name="comments")
)

print("\nSAMPLE PREDICTIONS")
display(
    sample_df[
        [
            "movie_title",
            "comment_text",
            "predicted_language",
            "confidence"
        ]
    ]
)

print("\nAnalysis completed. Original data unchanged.")

In [ ]:
# ============================================================
# ICMB-130 — MANUAL LANGUAGE VALIDATION SAMPLE
# ============================================================

import pandas as pd
from pathlib import Path
import regex

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

COMMENTS_PATH = (
    PROJECT_ROOT /
    "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_16.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT / "06_outputs/audience_sentiment"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PATH = (
    OUTPUT_DIR /
    "ICMB_130_LANGUAGE_MANUAL_VALIDATION_SAMPLE.csv"
)

df = pd.read_csv(COMMENTS_PATH, low_memory=False)

if SAMPLE_PATH.exists():
    raise FileExistsError(
        "Validation sample already exists. "
        "No existing file will be overwritten."
    )

# Select a reproducible sample of non-empty comments
sample_df = (
    df[df["comment_text"].notna()]
    .loc[
        df["comment_text"].fillna("").str.strip().ne("")
    ]
    .sample(n=100, random_state=130)
    .copy()
)

# Unicode script identification
script_patterns = {
    "Devanagari": r"\p{Devanagari}",
    "Bengali": r"\p{Bengali}",
    "Tamil": r"\p{Tamil}",
    "Telugu": r"\p{Telugu}",
    "Kannada": r"\p{Kannada}",
    "Malayalam": r"\p{Malayalam}",
    "Arabic": r"\p{Arabic}",
    "Latin": r"\p{Latin}"
}

def identify_script(text):
    found = [
        name for name, pattern in script_patterns.items()
        if regex.search(pattern, str(text))
    ]

    if not found:
        return "No recognized script"

    return " + ".join(found)

sample_df["script_type"] = (
    sample_df["comment_text"].apply(identify_script)
)

# Use model predictions if the pilot classifier is loaded
if "classifier" in globals():
    predictions = classifier(
        sample_df["comment_text"].astype(str).tolist(),
        batch_size=8,
        truncation=True,
        max_length=128
    )

    sample_df["predicted_language"] = [
        p["label"] for p in predictions
    ]

    sample_df["language_confidence"] = [
        round(float(p["score"]), 4)
        for p in predictions
    ]
else:
    sample_df["predicted_language"] = ""
    sample_df["language_confidence"] = None

# Blank column for manual annotation
sample_df["manual_language"] = ""

sample_df["annotation_guidance"] = (
    "Enter Hindi, English, Hinglish, Bengali, Tamil, "
    "Telugu, Other, or Uncertain"
)

validation_columns = [
    "movie_title",
    "comment_text",
    "script_type",
    "predicted_language",
    "language_confidence",
    "manual_language",
    "annotation_guidance"
]

sample_df[validation_columns].to_csv(
    SAMPLE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 75)
print("MANUAL LANGUAGE VALIDATION SAMPLE CREATED")
print("=" * 75)

print("Sample size:", len(sample_df))
print("\nScript distribution:")
display(sample_df["script_type"].value_counts())

print("\nModel prediction distribution:")
display(sample_df["predicted_language"].value_counts())

print("\nSaved file:", SAMPLE_PATH)

In [ ]:
# ============================================================
# ICMB-130 — AUTOMATED LANGUAGE ANNOTATION
# ============================================================

from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_PATH = (
    PROJECT_ROOT /
    "06_outputs/audience_sentiment/"
    "ICMB_130_LANGUAGE_MANUAL_VALIDATION_SAMPLE.csv"
)

OUTPUT_PATH = (
    PROJECT_ROOT /
    "06_outputs/audience_sentiment/"
    "ICMB_130_LANGUAGE_AUTO_ANNOTATED_SAMPLE.csv"
)

if OUTPUT_PATH.exists():
    raise FileExistsError(
        "Output already exists. Original files will not be overwritten."
    )

df = pd.read_csv(INPUT_PATH)

# Roman Hindi vocabulary and expressions
hindi_patterns = [
    r"\bbhai\b", r"\byaar\b", r"\bmera\b", r"\bmeri\b",
    r"\bmein\b", r"\bmain\b", r"\bhai\b", r"\bhain\b",
    r"\btha\b", r"\bthi\b", r"\bthe\b", r"\bki\b",
    r"\bka\b", r"\bke\b", r"\bko\b", r"\bse\b",
    r"\bbohot\b", r"\bbahut\b", r"\bacha\b",
    r"\bachhi\b", r"\bachha\b", r"\bnahi\b",
    r"\bnahin\b", r"\bkya\b", r"\bkyu\b",
    r"\bkyun\b", r"\bkaise\b", r"\baisa\b",
    r"\baise\b", r"\bmovie mein\b", r"\bdekh\b",
    r"\bdekhne\b", r"\bwala\b", r"\bwali\b",
    r"\bwaah\b", r"\bzindagi\b", r"\bpyaar\b",
    r"\byaad\b", r"\baj\b", r"\baaj\b",
    r"\babhi\b", r"\bhum\b", r"\btum\b",
    r"\bwoh\b", r"\bwo\b", r"\bmera\b",
    r"\bkar\b", r"\bkarna\b", r"\bkiya\b",
    r"\bliye\b", r"\bliye\b", r"\bmatlab\b"
]

def annotate_language(row):

    text = str(row["comment_text"]).strip()
    predicted = str(row.get("predicted_language", "")).lower()
    script = str(row.get("script_type", ""))

    if not text or text.lower() == "nan":
        return pd.Series(["Uncertain", "Empty comment"])

    # Unicode script evidence
    devanagari = bool(re.search(r"[\u0900-\u097F]", text))
    tamil = bool(re.search(r"[\u0B80-\u0BFF]", text))
    telugu = bool(re.search(r"[\u0C00-\u0C7F]", text))
    bengali = bool(re.search(r"[\u0980-\u09FF]", text))
    arabic = bool(re.search(r"[\u0600-\u06FF]", text))

    latin_words = re.findall(r"[A-Za-z]+", text.lower())

    # Very short / non-linguistic text
    if len(latin_words) == 0 and not any(
        [devanagari, tamil, telugu, bengali, arabic]
    ):
        return pd.Series(["Uncertain", "No identifiable language text"])

    # Script-based identification
    if devanagari:
        latin_words_lower = set(latin_words)

        english_markers = {
            "movie", "film", "acting", "best", "love",
            "amazing", "awesome", "super", "review"
        }

        if latin_words_lower.intersection(english_markers):
            return pd.Series(["Hinglish", "Mixed Devanagari and English"])
        return pd.Series(["Hindi", "Devanagari script"])

    if tamil:
        return pd.Series(["Tamil", "Tamil script"])

    if telugu:
        return pd.Series(["Telugu", "Telugu script"])

    if bengali:
        return pd.Series(["Bengali", "Bengali script"])

    if arabic:
        return pd.Series(["Other", "Arabic script; language needs confirmation"])

    # Roman-script language heuristics
    normalized = " " + re.sub(r"[^a-z\s]", " ", text.lower()) + " "

    hindi_matches = sum(
        bool(re.search(pattern, normalized))
        for pattern in hindi_patterns
    )

    english_words = {
        "the", "is", "are", "was", "were", "this",
        "that", "movie", "film", "acting", "beautiful",
        "amazing", "great", "best", "because", "with",
        "from", "after", "before", "watching", "really",
        "very", "love", "awesome", "what", "who",
        "anyone", "everyone", "and", "but", "not"
    }

    english_matches = sum(
        word in english_words for word in latin_words
    )

    if hindi_matches >= 2 and english_matches >= 2:
        return pd.Series(["Hinglish", "Roman Hindi and English indicators"])

    if hindi_matches >= 2:
        return pd.Series(["Hindi", "Roman Hindi lexical indicators"])

    if predicted == "hi" and hindi_matches >= 1:
        return pd.Series(["Hindi", "Model prediction supported by text"])

    if predicted == "en" and english_matches >= 2:
        return pd.Series(["English", "Model prediction supported by text"])

    if predicted in ["ta", "te", "bn"]:
        return pd.Series(["Other", "Model-indicated regional language"])

    return pd.Series(["Uncertain", "Insufficient linguistic evidence"])


df[["auto_language", "annotation_reason"]] = df.apply(
    annotate_language,
    axis=1
)

df["annotation_method"] = "Rule-assisted preliminary annotation"

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 75)
print("AUTOMATED LANGUAGE ANNOTATION SUMMARY")
print("=" * 75)

display(
    df["auto_language"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="comments")
)

print("\nUncertain examples:")
display(
    df.loc[
        df["auto_language"] == "Uncertain",
        ["comment_text", "predicted_language", "annotation_reason"]
    ].head(15)
)

print("\nSaved:", OUTPUT_PATH)
print("Annotated records:", len(df))

In [ ]:
# ============================================================
# ICMB-130 — REVISED LANGUAGE ANNOTATION PILOT
# ============================================================

from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_PATH = (
    PROJECT_ROOT /
    "06_outputs/audience_sentiment/"
    "ICMB_130_LANGUAGE_AUTO_ANNOTATED_SAMPLE.csv"
)

OUTPUT_PATH = (
    PROJECT_ROOT /
    "06_outputs/audience_sentiment/"
    "ICMB_130_LANGUAGE_REVISED_ANNOTATION_SAMPLE.csv"
)

if OUTPUT_PATH.exists():
    raise FileExistsError("Output exists. No file overwritten.")

df = pd.read_csv(INPUT_PATH)

# Expanded Roman Hindi vocabulary
HINDI_WORDS = set("""
aaj ab abhi achha achhi achhe agar alag apna apne
aap aur bahut badi bada bachpan baki bas bhi bhai
bhi bilkul chahta chahiye dekha dekh dekhne diya
dil duniya ek film gaya gayi gaye hai hain ham hum
hamein humein ho hua hui huye iska iski iske iss
itna itni jaise jaisa jahan jaan jab jindagi ji
kaise kaam kaafi kaha kahani kahin kaisa kaisi
kar kare karta karti karna karo ki kya kyun kyu
ko koi kuch lage laga lagi lagta lagti lekar lekin
liye log love maza mein mera meri mere mila mujhe
movie nahi nahin nehin nahiye nhi naya naye naam
pata pe pehle phir pyar pyaar raha rahi rahe sab
sach saath samajh se sirf soch suna suna tha thi
the thoda tum tumhe us uski uske wale wali wala
wajah waqt wo woh yaar yaad ye yeh zindagi zaroor
""".split())

ENGLISH_WORDS = set("""
a an the and or but is are was were be been being
this that these those i you he she it we they my
your his her our their me him them who what when
where why how have has had do does did can could
will would should just really very movie film
watch watched watching review reaction amazing
awesome great good bad best worst love liked
hated beautiful story acting actor actress
because after before with from about into only
please thanks thank anyone everyone someone
never always again more much too so if not
""".split())

URDU_WORDS = set("""
hai hain hay ho ka kay ki ko lia liye aur mera
meri tum aap acha achi nahi nahin kya
""".split())

def revised_annotation(row):

    text = str(row["comment_text"]).strip()

    if not text or text.lower() == "nan":
        return pd.Series(["Uncertain", "Empty"])

    words = re.findall(r"[a-z]+", text.lower())

    if not words:
        return pd.Series(["Uncertain", "No Latin words"])

    # Native-script detection
    if re.search(r"[\u0900-\u097F]", text):
        if re.search(r"[A-Za-z]", text):
            return pd.Series(["Hinglish", "Devanagari + Latin"])
        return pd.Series(["Hindi", "Devanagari"])

    if re.search(r"[\u0B80-\u0BFF]", text):
        return pd.Series(["Tamil", "Tamil script"])

    if re.search(r"[\u0C00-\u0C7F]", text):
        return pd.Series(["Telugu", "Telugu script"])

    if re.search(r"[\u0980-\u09FF]", text):
        return pd.Series(["Bengali", "Bengali script"])

    hindi_score = sum(w in HINDI_WORDS for w in words)
    english_score = sum(w in ENGLISH_WORDS for w in words)
    urdu_score = sum(w in URDU_WORDS for w in words)

    predicted = str(
        row.get("predicted_language", "")
    ).lower()

    # Roman Hindi and mixed Hindi-English
    if hindi_score >= 2 and english_score >= 2:
        return pd.Series(["Hinglish", "Hindi and English vocabulary"])

    if hindi_score >= 2:
        return pd.Series(["Hindi", "Roman Hindi vocabulary"])

    if english_score >= 2 and predicted == "en":
        return pd.Series(["English", "English vocabulary + model support"])

    if predicted == "en" and len(words) >= 4:
        return pd.Series(["English", "Model prediction; Latin text"])

    if predicted == "hi" and hindi_score >= 1:
        return pd.Series(["Hindi", "Model + Hindi vocabulary"])

    if predicted == "ur" and urdu_score >= 2:
        return pd.Series(["Urdu", "Model + vocabulary"])

    return pd.Series(["Uncertain", "Insufficient evidence"])


df[["revised_language", "revised_reason"]] = df.apply(
    revised_annotation,
    axis=1
)

print("=" * 75)
print("REVISED LANGUAGE ANNOTATION RESULTS")
print("=" * 75)

display(
    df["revised_language"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="comments")
)

print("\nREVISED SAMPLE")
display(
    df[
        [
            "comment_text",
            "predicted_language",
            "revised_language",
            "revised_reason"
        ]
    ].head(30)
)

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", OUTPUT_PATH)

In [ ]:
# ============================================================
# ICMB-130 — FINAL REFINED LANGUAGE IDENTIFICATION PILOT
# ============================================================

from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_PATH = (
    PROJECT_ROOT /
    "06_outputs/audience_sentiment/"
    "ICMB_130_LANGUAGE_REVISED_ANNOTATION_SAMPLE.csv"
)

OUTPUT_PATH = (
    PROJECT_ROOT /
    "06_outputs/audience_sentiment/"
    "ICMB_130_LANGUAGE_REFINED_PILOT.csv"
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(INPUT_PATH)

if OUTPUT_PATH.exists():
    raise FileExistsError(
        "Output already exists. Existing files preserved."
    )

df = pd.read_csv(INPUT_PATH)

# ------------------------------------------------------------
# LANGUAGE VOCABULARIES
# ------------------------------------------------------------

HINDI_WORDS = set("""
aaj ab abhi achha achhi achhe agar apna apne
aur bahut badi bada baki bas bhai bhi bilkul
chahta chahiye dekha dekh dekhne diya dil
ek gaya gayi hai hain ham hum humein hume
ho hua hui humara humari iska iski iske
iss itna itni jaise jaisa jahan jaan jab
kaise kaam kaha kahani kaisa kaisi kar kare
karta karti karna karo ki kya kyun kyu
ko koi kuch lage laga lagi lagta lagti
lekin liye log maza mein mera meri mere
mila mujhe nahi nahin nehin nhi naya naye
naam pata pe pehle phir pyar pyaar raha
rahi rahe sab sach saath samajh se sirf
soch suna tha thi the thoda tum tumhe
us uski uske wale wali wala wajah wo woh
yaar yaad ye yeh zindagi zaroor maine
bilkul bhi nahi aaya aayi dekha movie mein
kar di kar diya karne
""".split())

# English function words and distinctive expressions
ENGLISH_MARKERS = set("""
the this that these those is are was were have has had
i you he she it we they my your his her our their
anyone everyone someone who what when where why how
because after before with from about into only
never always again much really very watching watched
found story budget based real many times behavior
completely please join channel amazing beautiful
boring reaction review listening usual hear
""".split())

# Words common to both languages are deliberately excluded
AMBIGUOUS_WORDS = set("""
hai ho ki ka ke ko mein me movie film story love
best good bad please like
""".split())

# ------------------------------------------------------------
# SCRIPT DETECTION
# ------------------------------------------------------------

def has_script(text, pattern):
    return bool(re.search(pattern, text))

def classify_language(row):

    text = str(row["comment_text"]).strip()
    words = re.findall(r"[a-z]+", text.lower())

    predicted = str(
        row.get("predicted_language", "")
    ).lower()

    confidence = pd.to_numeric(
        row.get("language_confidence", 0),
        errors="coerce"
    )

    if pd.isna(confidence):
        confidence = 0

    if not text or text.lower() == "nan":
        return pd.Series([
            "Uncertain", "Non-linguistic or empty",
            "Needs review"
        ])

    # Native-script evidence
    devanagari = has_script(text, r"[\u0900-\u097F]")
    tamil = has_script(text, r"[\u0B80-\u0BFF]")
    telugu = has_script(text, r"[\u0C00-\u0C7F]")
    bengali = has_script(text, r"[\u0980-\u09FF]")

    if devanagari:
        latin_words = set(words)

        if latin_words.intersection(ENGLISH_MARKERS):
            return pd.Series([
                "Hinglish",
                "Devanagari and English mixed",
                "Rule-assisted"
            ])

        return pd.Series([
            "Hindi", "Devanagari script", "Rule-assisted"
        ])

    if tamil:
        return pd.Series(["Tamil", "Tamil script", "Rule-assisted"])

    if telugu:
        return pd.Series(["Telugu", "Telugu script", "Rule-assisted"])

    if bengali:
        return pd.Series(["Bengali", "Bengali script", "Rule-assisted"])

    if not words:
        return pd.Series([
            "Uncertain", "No identifiable words", "Needs review"
        ])

    hindi_count = len(set(words) & HINDI_WORDS)
    english_count = len(set(words) & ENGLISH_MARKERS)

    # Require meaningful Hindi evidence
    if hindi_count >= 2 and english_count >= 2:
        return pd.Series([
            "Hinglish", "Roman Hindi and English indicators",
            "Rule-assisted"
        ])

    if hindi_count >= 2:
        return pd.Series([
            "Hindi", "Roman Hindi lexical evidence",
            "Rule-assisted"
        ])

    if predicted == "hi" and confidence >= 0.85 and hindi_count >= 1:
        return pd.Series([
            "Hindi", "Model and Hindi lexical evidence",
            "Rule-assisted"
        ])

    if predicted == "en" and confidence >= 0.75:
        return pd.Series([
            "English", "Multilingual model prediction",
            "Automatic"
        ])

    if predicted == "ur" and confidence >= 0.85:
        return pd.Series([
            "Urdu", "Multilingual model prediction",
            "Automatic"
        ])

    return pd.Series([
        "Uncertain", "Insufficient language evidence",
        "Needs review"
    ])

# ------------------------------------------------------------
# APPLY CLASSIFICATION
# ------------------------------------------------------------

df[
    [
        "refined_language",
        "refined_reason",
        "language_status"
    ]
] = df.apply(classify_language, axis=1)

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("ICMB-130 — REFINED LANGUAGE IDENTIFICATION PILOT")
print("=" * 75)

print("\nLanguage distribution:")
display(
    df["refined_language"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="comments")
)

print("\nClassification status:")
display(
    df["language_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="comments")
)

print("\nUncertain examples:")
display(
    df.loc[
        df["refined_language"] == "Uncertain",
        [
            "comment_text",
            "predicted_language",
            "language_confidence",
            "refined_reason"
        ]
    ].head(15)
)

print("\nSaved:", OUTPUT_PATH)
print("Annotated records:", len(df))
print("Original data unchanged.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — FULL DATASET LANGUAGE IDENTIFICATION
# ICMB-130 | YouTube Audience Comments | Batches 01–16
# ============================================================

import os
import re
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from transformers import pipeline

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

INPUT_FILE = os.path.join(
    PROJECT_ROOT,
    "01_data/processed/audience_sentiment/"
    "ICMB_130_YOUTUBE_COMMENTS_CONSOLIDATED_BATCH_01_16.csv"
)

OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "06_outputs/audience_sentiment"
)

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_LANGUAGE_IDENTIFICATION_MODEL_OUTPUT_BATCH_01_16.csv"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Prevent accidental overwriting
if os.path.exists(OUTPUT_FILE):
    raise FileExistsError(
        f"Output already exists. No file was overwritten:\n{OUTPUT_FILE}"
    )

# ------------------------------------------------------------
# 2. LOAD ORIGINAL DATASET
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("=" * 65)
print("FULL DATASET LANGUAGE IDENTIFICATION")
print("=" * 65)

print("Dataset shape:", df.shape)
print("Unique comments:", df["comment_id"].nunique())

assert "comment_text" in df.columns, "comment_text column missing"

# Preserve original comment text
df["comment_text"] = df["comment_text"].fillna("").astype(str)

# ------------------------------------------------------------
# 3. IDENTIFY COMMENT SCRIPT
# ------------------------------------------------------------

def detect_script(text):

    if not text or not text.strip():
        return "Empty"

    scripts = {
        "Devanagari": r"\u0900-\u097F",
        "Tamil": r"\u0B80-\u0BFF",
        "Bengali": r"\u0980-\u09FF",
        "Telugu": r"\u0C00-\u0C7F",
        "Kannada": r"\u0C80-\u0CFF",
        "Malayalam": r"\u0D00-\u0D7F",
        "Gujarati": r"\u0A80-\u0AFF",
        "Arabic": r"\u0600-\u06FF",
        "Latin": r"A-Za-z"
    }

    detected = []

    for name, pattern in scripts.items():
        if re.search(f"[{pattern}]", text):
            detected.append(name)

    if not detected:
        return "No recognized script"

    if len(detected) == 1:
        return detected[0]

    return "Mixed: " + " + ".join(detected)


df["script_type"] = df["comment_text"].apply(detect_script)

# ------------------------------------------------------------
# 4. LOAD OR REUSE LANGUAGE DETECTION MODEL
# ------------------------------------------------------------

MODEL_NAME = "papluca/xlm-roberta-base-language-detection"

if "classifier" not in globals():

    print("\nLoading language detection model...")

    classifier = pipeline(
        "text-classification",
        model=MODEL_NAME,
        tokenizer=MODEL_NAME,
        device=-1
    )

else:
    print("\nReusing previously loaded language detection model.")

# ------------------------------------------------------------
# 5. PREPARE NON-EMPTY COMMENTS
# ------------------------------------------------------------

valid_mask = df["comment_text"].str.strip().ne("")

valid_indices = df.index[valid_mask].tolist()
valid_texts = df.loc[valid_mask, "comment_text"].tolist()

df["predicted_language"] = None
df["language_confidence"] = np.nan
df["language_inference_status"] = "Missing/Empty"

print("\nComments eligible for prediction:", len(valid_texts))
print("Empty comments:", len(df) - len(valid_texts))

# ------------------------------------------------------------
# 6. RUN MODEL IN BATCHES
# ------------------------------------------------------------

BATCH_SIZE = 16
MAX_LENGTH = 128

predicted_labels = []
predicted_scores = []

print("\nRunning language predictions...")

for start in tqdm(range(0, len(valid_texts), BATCH_SIZE)):

    batch_texts = valid_texts[start:start + BATCH_SIZE]

    results = classifier(
        batch_texts,
        truncation=True,
        max_length=MAX_LENGTH,
        batch_size=BATCH_SIZE
    )

    for result in results:
        predicted_labels.append(result["label"])
        predicted_scores.append(float(result["score"]))

# ------------------------------------------------------------
# 7. STORE RAW MODEL PREDICTIONS
# ------------------------------------------------------------

df.loc[valid_indices, "predicted_language"] = predicted_labels
df.loc[valid_indices, "language_confidence"] = predicted_scores
df.loc[valid_indices, "language_inference_status"] = "Predicted"

df["language_model"] = MODEL_NAME

# ------------------------------------------------------------
# 8. SAVE SEPARATE OUTPUT
# ------------------------------------------------------------

df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 9. VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("LANGUAGE IDENTIFICATION COMPLETED")
print("=" * 65)

print("Final shape:", df.shape)
print("Total comments:", len(df))
print("Predicted comments:", (df["language_inference_status"] == "Predicted").sum())
print("Missing/empty comments:", (df["language_inference_status"] == "Missing/Empty").sum())
print("Unique comment IDs:", df["comment_id"].nunique())

print("\nLANGUAGE PREDICTION DISTRIBUTION")
print("-" * 65)

print(
    df["predicted_language"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nSCRIPT DISTRIBUTION")
print("-" * 65)

print(df["script_type"].value_counts().to_string())

print("\nSaved output:")
print(OUTPUT_FILE)

assert len(df) == 15960, "Unexpected row count"
assert df["comment_id"].nunique() == 15960, "Duplicate comment IDs detected"
assert os.path.exists(OUTPUT_FILE), "Output file was not saved"

print("\nAll verification checks passed.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — LANGUAGE PREDICTION QUALITY AUDIT
# ============================================================

import os
import pandas as pd

PROJECT_ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

INPUT_FILE = os.path.join(
    PROJECT_ROOT,
    "06_outputs/audience_sentiment/",
    "ICMB_130_LANGUAGE_IDENTIFICATION_MODEL_OUTPUT_BATCH_01_16.csv"
)

df_lang = pd.read_csv(INPUT_FILE)

print("=" * 75)
print("LANGUAGE PREDICTION QUALITY AUDIT")
print("=" * 75)

# ------------------------------------------------------------
# 1. PREDICTION BY SCRIPT TYPE
# ------------------------------------------------------------

print("\n1. LANGUAGE PREDICTIONS BY SCRIPT TYPE")
print("-" * 75)

script_language = pd.crosstab(
    df_lang["script_type"],
    df_lang["predicted_language"],
    normalize="index"
).round(3) * 100

print(script_language.to_string())

# ------------------------------------------------------------
# 2. CONFIDENCE DISTRIBUTION
# ------------------------------------------------------------

print("\n2. MODEL CONFIDENCE DISTRIBUTION")
print("-" * 75)

print(
    df_lang["language_confidence"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90])
    .round(3)
    .to_string()
)

df_lang["confidence_band"] = pd.cut(
    df_lang["language_confidence"],
    bins=[-0.001, 0.50, 0.70, 0.85, 1.001],
    labels=["Below 0.50", "0.50–0.70", "0.70–0.85", "Above 0.85"]
)

print("\nConfidence bands:")
print(df_lang["confidence_band"].value_counts(dropna=False).to_string())

# ------------------------------------------------------------
# 3. SWAHILI PREDICTION AUDIT
# ------------------------------------------------------------

print("\n3. SWAHILI PREDICTIONS — SAMPLE")
print("-" * 75)

sw_sample = (
    df_lang[df_lang["predicted_language"] == "sw"]
    .sample(
        n=min(30, (df_lang["predicted_language"] == "sw").sum()),
        random_state=130
    )
)

for _, row in sw_sample.iterrows():
    print(
        f"\nPrediction: {row['predicted_language']} | "
        f"Confidence: {row['language_confidence']:.3f} | "
        f"Script: {row['script_type']}"
    )
    print("Comment:", str(row["comment_text"])[:300])

# ------------------------------------------------------------
# 4. OTHER UNEXPECTED LANGUAGE PREDICTIONS
# ------------------------------------------------------------

print("\n4. OTHER NON-ENGLISH/HINDI/URDU PREDICTIONS")
print("-" * 75)

unexpected = df_lang[
    ~df_lang["predicted_language"].isin(["en", "hi", "ur"])
    & df_lang["predicted_language"].notna()
]

unexpected_sample = unexpected.sample(
    n=min(30, len(unexpected)),
    random_state=130
)

for _, row in unexpected_sample.iterrows():
    print(
        f"\nPrediction: {row['predicted_language']} | "
        f"Confidence: {row['language_confidence']:.3f} | "
        f"Script: {row['script_type']}"
    )
    print("Comment:", str(row["comment_text"])[:300])

# ------------------------------------------------------------
# 5. LOW-CONFIDENCE PREDICTIONS
# ------------------------------------------------------------

print("\n5. LOW-CONFIDENCE PREDICTIONS")
print("-" * 75)

low_conf = df_lang[
    df_lang["language_confidence"].notna()
    & (df_lang["language_confidence"] < 0.50)
].sort_values("language_confidence")

print("Total low-confidence comments:", len(low_conf))

for _, row in low_conf.head(20).iterrows():
    print(
        f"\nPrediction: {row['predicted_language']} | "
        f"Confidence: {row['language_confidence']:.3f} | "
        f"Script: {row['script_type']}"
    )
    print("Comment:", str(row["comment_text"])[:300])

# ------------------------------------------------------------
# 6. HINDI AND ENGLISH EXAMPLES
# ------------------------------------------------------------

for language in ["hi", "en", "ur"]:

    sample = df_lang[
        df_lang["predicted_language"] == language
    ].sample(
        n=min(
            10,
            (df_lang["predicted_language"] == language).sum()
        ),
        random_state=130
    )

    print(f"\n6. SAMPLE COMMENTS — {language.upper()}")
    print("-" * 75)

    for _, row in sample.iterrows():
        print(
            f"\nConfidence: {row['language_confidence']:.3f} | "
            f"Script: {row['script_type']}"
        )
        print("Comment:", str(row["comment_text"])[:300])

print("\n" + "=" * 75)
print("LANGUAGE QUALITY AUDIT COMPLETED")
print("=" * 75)

In [ ]:
# ============================================================
# NOTEBOOK 10 — LATIN-SCRIPT LANGUAGE ERROR ANALYSIS
# ============================================================

import pandas as pd
import os

PROJECT_ROOT = "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"

INPUT_FILE = os.path.join(
    PROJECT_ROOT,
    "06_outputs/audience_sentiment/",
    "ICMB_130_LANGUAGE_IDENTIFICATION_MODEL_OUTPUT_BATCH_01_16.csv"
)

df_lang = pd.read_csv(INPUT_FILE)

# ------------------------------------------------------------
# 1. LATIN-SCRIPT PREDICTION DISTRIBUTION
# ------------------------------------------------------------

latin_df = df_lang[df_lang["script_type"] == "Latin"].copy()

print("=" * 70)
print("LATIN-SCRIPT LANGUAGE ANALYSIS")
print("=" * 70)

print("Total Latin-script comments:", len(latin_df))

print("\nPrediction distribution:")
print(
    latin_df["predicted_language"]
    .value_counts(dropna=False)
    .to_string()
)

# ------------------------------------------------------------
# 2. SWAHILI PREDICTIONS BY MOVIE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SWAHILI PREDICTIONS BY MOVIE")
print("=" * 70)

sw_movie = (
    latin_df[latin_df["predicted_language"] == "sw"]
    .groupby("movie_title")
    .agg(
        comment_count=("comment_id", "count"),
        average_confidence=("language_confidence", "mean")
    )
    .sort_values("comment_count", ascending=False)
)

print(sw_movie.head(25).round(3).to_string())

# ------------------------------------------------------------
# 3. SWAHILI CONFIDENCE DISTRIBUTION
# ------------------------------------------------------------

sw_df = latin_df[latin_df["predicted_language"] == "sw"]

print("\nSwahili predictions:", len(sw_df))

print("\nConfidence distribution:")
print(
    sw_df["language_confidence"]
    .describe()
    .round(3)
    .to_string()
)

# ------------------------------------------------------------
# 4. LATIN-SCRIPT COMMENTS BY MODEL CONFIDENCE
# ------------------------------------------------------------

latin_df["confidence_band"] = pd.cut(
    latin_df["language_confidence"],
    bins=[-0.001, 0.50, 0.70, 0.85, 1.001],
    labels=["Below 0.50", "0.50–0.70", "0.70–0.85", "Above 0.85"]
)

print("\nConfidence by predicted language:")
print(
    pd.crosstab(
        latin_df["predicted_language"],
        latin_df["confidence_band"]
    ).to_string()
)

# ------------------------------------------------------------
# 5. SAMPLE HIGH-CONFIDENCE SWAHILI PREDICTIONS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HIGH-CONFIDENCE SWAHILI PREDICTIONS")
print("=" * 70)

high_sw = sw_df[
    sw_df["language_confidence"] >= 0.85
].sample(
    n=min(25, len(sw_df[sw_df["language_confidence"] >= 0.85])),
    random_state=130
)

for _, row in high_sw.iterrows():
    print("\nMovie:", row["movie_title"])
    print("Confidence:", round(row["language_confidence"], 3))
    print("Comment:", str(row["comment_text"])[:250])

print("\n" + "=" * 70)
print("ANALYSIS COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# NOTEBOOK 10 — INDIAN LANGUAGE MODEL AVAILABILITY CHECK
# ============================================================

import importlib.util
import os

print("=" * 70)
print("INDIAN LANGUAGE MODEL ENVIRONMENT CHECK")
print("=" * 70)

# 1. Installed libraries
libraries = [
    "transformers",
    "torch",
    "sentencepiece",
    "indicnlp",
    "datasets",
    "huggingface_hub"
]

print("\nINSTALLED LIBRARIES")
print("-" * 70)

for library in libraries:
    print(
        f"{library}:",
        "Available" if importlib.util.find_spec(library) else "Not installed"
    )

# 2. Check cached Hugging Face models
print("\nCACHED HUGGING FACE MODEL DIRECTORIES")
print("-" * 70)

cache_dir = os.path.expanduser("~/.cache/huggingface/hub")

if os.path.exists(cache_dir):

    model_dirs = [
        name for name in os.listdir(cache_dir)
        if name.startswith("models--")
    ]

    for name in model_dirs:
        print(name)

    if not model_dirs:
        print("No cached model directories found.")

else:
    print("Hugging Face cache directory not found.")

# 3. Existing classifier information
print("\nCURRENT CLASSIFIER")
print("-" * 70)

if "classifier" in globals():
    print("Existing classifier:", type(classifier).__name__)

    try:
        print("Model:", classifier.model.name_or_path)
        print("Supported labels:", classifier.model.config.id2label)
    except Exception as e:
        print("Model details unavailable:", str(e))

else:
    print("No classifier currently loaded.")

print("\n" + "=" * 70)
print("ENVIRONMENT CHECK COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# NOTEBOOK 10 — SEARCH FOR INDICLID MODELS
# ============================================================

from huggingface_hub import HfApi

api = HfApi()

print("=" * 70)
print("SEARCHING FOR INDIAN LANGUAGE IDENTIFICATION MODELS")
print("=" * 70)

try:
    models = list(
        api.list_models(
            search="IndicLID",
            sort="downloads",
            direction=-1,
            limit=20
        )
    )

    if models:
        for model in models:
            print("\nModel ID:", model.id)
            print("Pipeline:", getattr(model, "pipeline_tag", None))
            print("Downloads:", getattr(model, "downloads", None))
    else:
        print("No IndicLID repositories found.")

except Exception as e:
    print("Model search failed:", str(e))

print("\n" + "=" * 70)
print("MODEL SEARCH COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# NOTEBOOK 10 — INDICLID MODEL SEARCH (COMPATIBILITY FIX)
# ============================================================

from huggingface_hub import HfApi

api = HfApi()

print("=" * 70)
print("SEARCHING FOR INDICLID MODELS")
print("=" * 70)

try:
    models = list(
        api.list_models(
            search="IndicLID",
            sort="downloads",
            limit=20
        )
    )

    if models:
        for model in models:
            print("\nModel ID:", model.id)
            print("Pipeline:", getattr(model, "pipeline_tag", None))
            print("Downloads:", getattr(model, "downloads", None))
    else:
        print("No matching repositories found.")

except Exception as e:
    print("Search error:", repr(e))

print("\n" + "=" * 70)
print("SEARCH COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# NOTEBOOK 10 — INSPECT OFFICIAL INDICLID MODEL REPOSITORIES
# ============================================================

from huggingface_hub import HfApi

api = HfApi()

model_ids = [
    "ai4bharat/IndicLID-BERT",
    "ai4bharat/IndicLID-FTR",
    "ai4bharat/IndicLID-FTN"
]

for model_id in model_ids:

    print("\n" + "=" * 75)
    print("MODEL:", model_id)
    print("=" * 75)

    try:
        info = api.model_info(model_id, files_metadata=True)

        print("Repository:", info.id)
        print("Pipeline:", info.pipeline_tag)
        print("Library:", info.library_name)

        print("\nRepository files:")

        for file in info.siblings:
            print(
                file.rfilename,
                "| Size:", file.size
            )

    except Exception as e:
        print("Could not inspect repository:", repr(e))

print("\nRepository inspection completed.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — RETRIEVE OFFICIAL INDICLID INFERENCE CODE
# ============================================================

import os
import subprocess

REPO_DIR = "/content/IndicLID_official_source"

print("=" * 70)
print("OFFICIAL INDICLID INFERENCE CODE INSPECTION")
print("=" * 70)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):

    result = subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/AI4Bharat/IndicLID.git",
            REPO_DIR
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

    print("Official repository cloned successfully.")

else:
    print("Existing repository clone found. Reusing it.")

# ------------------------------------------------------------
# 1. FIND INFERENCE-RELATED PYTHON FILES
# ------------------------------------------------------------

print("\nINFERENCE FILES")
print("-" * 70)

inference_files = []

for root, dirs, files in os.walk(REPO_DIR):

    dirs[:] = [
        d for d in dirs
        if d not in [".git", "__pycache__"]
    ]

    for filename in files:

        if filename.endswith(".py") and (
            "inference" in filename.lower()
            or filename == "IndicLID.py"
            or filename == "test_script.py"
        ):
            full_path = os.path.join(root, filename)
            inference_files.append(full_path)
            print(os.path.relpath(full_path, REPO_DIR))

# ------------------------------------------------------------
# 2. DISPLAY MAIN DEPLOYMENT IMPLEMENTATION
# ------------------------------------------------------------

main_file = os.path.join(
    REPO_DIR,
    "deployement",
    "working",
    "IndicLID.py"
)

if os.path.exists(main_file):

    print("\nOFFICIAL INDICLID.PY")
    print("=" * 70)

    with open(main_file, "r", encoding="utf-8") as f:
        print(f.read())

else:
    print("\nMain deployment file not found.")

# ------------------------------------------------------------
# 3. DISPLAY TEST SCRIPT
# ------------------------------------------------------------

test_file = os.path.join(
    REPO_DIR,
    "deployement",
    "test_script.py"
)

if os.path.exists(test_file):

    print("\nOFFICIAL TEST SCRIPT")
    print("=" * 70)

    with open(test_file, "r", encoding="utf-8") as f:
        print(f.read())

print("\n" + "=" * 70)
print("SOURCE CODE INSPECTION COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# NOTEBOOK 10 — INDICLID DEPENDENCY COMPATIBILITY CHECK
# ============================================================

import sys
import importlib.util
import torch
import transformers

print("=" * 70)
print("INDICLID DEPENDENCY COMPATIBILITY")
print("=" * 70)

print("\nPYTHON AND LIBRARY VERSIONS")
print("-" * 70)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

print("\nFASTTEXT AVAILABILITY")
print("-" * 70)

try:
    import fasttext
    print("FastText: Available")
    print("FastText module:", fasttext.__file__)

except Exception as e:
    print("FastText: Not usable")
    print("Reason:", repr(e))

print("\nPYTORCH CHECKPOINT LOADING SUPPORT")
print("-" * 70)

import inspect

print("torch.load signature:")
print(inspect.signature(torch.load))

print("\nGPU availability:", torch.cuda.is_available())

print("\n" + "=" * 70)
print("COMPATIBILITY CHECK COMPLETED")
print("=" * 70)

In [ ]:
# ============================================================
# NOTEBOOK 10 — INSPECT INDICLID-BERT MODEL ARCHITECTURE
# ============================================================

import os
import re

REPO_DIR = "/content/IndicLID_official_source"

print("=" * 75)
print("INDICLID-BERT ARCHITECTURE INSPECTION")
print("=" * 75)

search_terms = [
    "class IndicLID",
    "class IndicBERT",
    "AutoModel.from_pretrained",
    "IndicBERTv2-MLM-only",
    "basline_nn_simple.pt",
    "torch.save"
]

matches = []

for root, dirs, files in os.walk(REPO_DIR):

    dirs[:] = [
        d for d in dirs
        if d not in [".git", "__pycache__"]
    ]

    for filename in files:

        if not filename.endswith(".py"):
            continue

        filepath = os.path.join(root, filename)

        try:
            with open(filepath, "r", encoding="utf-8") as f:
                lines = f.readlines()
        except Exception:
            continue

        for line_no, line in enumerate(lines):

            if any(term in line for term in search_terms):

                matches.append(
                    (filepath, line_no, lines)
                )

# Display architecture-related code with surrounding context
shown = set()

for filepath, line_no, lines in matches:

    key = (filepath, line_no)

    if key in shown:
        continue

    shown.add(key)

    start = max(0, line_no - 8)
    end = min(len(lines), line_no + 15)

    print("\nFILE:", os.path.relpath(filepath, REPO_DIR))
    print("-" * 75)

    for i in range(start, end):
        print(f"{i + 1:4}: {lines[i].rstrip()}")

print("\n" + "=" * 75)
print("TOTAL MATCHES:", len(shown))
print("ARCHITECTURE INSPECTION COMPLETED")
print("=" * 75)

In [ ]:
# ============================================================
# NOTEBOOK 10 — EXTRACT OFFICIAL INDICLID MAPPING & INFERENCE
# ============================================================

import os

REPO_DIR = "/content/IndicLID_official_source"

files_to_inspect = {
    "Official IndicLID wrapper":
        "Inference/ai4bharat/IndicLID.py",

    "IndicBERT 22-class inference":
        "final_runs_train/roman_model/finetuning/IndicBERT/freezed_bert_all_layer/inference.py"
}

for title, relative_path in files_to_inspect.items():

    filepath = os.path.join(REPO_DIR, relative_path)

    print("\n" + "=" * 80)
    print(title)
    print("FILE:", relative_path)
    print("=" * 80)

    if not os.path.exists(filepath):
        print("FILE NOT FOUND")
        continue

    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    # Print mapping, preprocessing and prediction sections
    keywords = [
        "IndicLID_lang_code_dict",
        "confusion_matrix_mapping",
        "tokenizer",
        "input_ids",
        "attention_mask",
        "softmax",
        "argmax",
        "predicted",
        "model("
    ]

    selected = set()

    for i, line in enumerate(lines):
        if any(term in line for term in keywords):
            for j in range(max(0, i - 3), min(len(lines), i + 9)):
                selected.add(j)

    for i in sorted(selected):
        print(f"{i + 1:4}: {lines[i].rstrip()}")

print("\n" + "=" * 80)
print("EXTRACTION COMPLETED")
print("=" * 80)

In [ ]:
from pathlib import Path

repo = Path("/content/IndicLID_official_source")
wrapper = repo / "Inference/ai4bharat/IndicLID.py"

if not wrapper.exists():
    print("Wrapper not found at:", wrapper)
else:
    lines = wrapper.read_text(encoding="utf-8").splitlines()

    sections = [
        ("Constructor and language mappings", 54, 155),
        ("Language detection and model routing", 190, 290),
        ("Post-processing", 286, min(len(lines), 370))
    ]

    for title, start, end in sections:
        print("\n" + "=" * 85)
        print(title)
        print("=" * 85)

        for i in range(start - 1, min(end, len(lines))):
            print(f"{i + 1:4}: {lines[i]}")

    print("\n" + "=" * 85)
    print("CHECKPOINT FILES")
    print("=" * 85)

    for path in [
        repo / "Inference/models/indiclid-bert/basline_nn_simple.pt",
        repo / "Inference/models/indiclid-ftn/model_baseline_roman.bin",
        repo / "Inference/models/indiclid-ftr/model_baseline_roman.bin"
    ]:
        print(f"{path}")
        print("Exists:", path.exists())
        if path.exists():
            print("Size (MB):", round(path.stat().st_size / (1024 * 1024), 2))

In [ ]:
from pathlib import Path

repo = Path("/content/IndicLID_official_source")

print("=" * 80)
print("INDICLID REPOSITORY MODEL AND DOWNLOAD AUDIT")
print("=" * 80)

# 1. Find model-related files and directories
print("\nMODEL-RELATED PATHS")
for path in repo.rglob("*"):
    if any(term in str(path).lower() for term in
           ["model", "checkpoint", "weight", "download"]):
        if path.is_file():
            print(
                path.relative_to(repo),
                "| Size:",
                round(path.stat().st_size / (1024 * 1024), 2),
                "MB"
            )
        elif path.is_dir():
            print(path.relative_to(repo), "/")

# 2. Inspect README and download-related documentation
print("\n" + "=" * 80)
print("README / DOWNLOAD DOCUMENTATION")
print("=" * 80)

docs = [
    p for p in repo.rglob("*")
    if p.is_file()
    and (
        p.name.lower().startswith("readme")
        or "download" in p.name.lower()
        or "install" in p.name.lower()
    )
]

for path in docs[:10]:
    print(f"\n--- {path.relative_to(repo)} ---")
    try:
        lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
        matches = [
            (i, line) for i, line in enumerate(lines)
            if any(word in line.lower() for word in
                   ["model", "checkpoint", "drive", "download",
                    "huggingface", "fasttext", "weight", "inference"])
        ]
        for i, line in matches[:35]:
            print(f"{i+1}: {line[:250]}")
    except Exception as e:
        print("Could not read:", e)

# 3. Check Git LFS pointers or submodules
print("\n" + "=" * 80)
print("GIT LFS / SUBMODULE CHECK")
print("=" * 80)

for name in [".gitattributes", ".gitmodules"]:
    path = repo / name
    print(f"\n{name}: Exists =", path.exists())
    if path.exists():
        print(path.read_text(encoding="utf-8", errors="ignore")[:4000])

In [ ]:
import requests
from pathlib import Path
import zipfile

download_dir = Path("/content/IndicLID_model_downloads")
download_dir.mkdir(parents=True, exist_ok=True)

models = {
    "FTN": "https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-ftn.zip",
    "FTR": "https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-ftr.zip",
    "BERT": "https://github.com/AI4Bharat/IndicLID/releases/download/v1.0/indiclid-bert.zip"
}

print("=" * 75)
print("OFFICIAL INDICLID MODEL DOWNLOAD")
print("=" * 75)

for name, url in models.items():
    archive = download_dir / f"indiclid-{name.lower()}.zip"

    if not archive.exists():
        print(f"\nDownloading {name}...")
        response = requests.get(url, stream=True, timeout=120)
        response.raise_for_status()

        with open(archive, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    print(f"\n{name}")
    print("Archive:", archive)
    print("Size (MB):", round(archive.stat().st_size / (1024 * 1024), 2))

    with zipfile.ZipFile(archive, "r") as z:
        print("Archive contents:")
        for item in z.infolist():
            print(
                " ",
                item.filename,
                "| Size (MB):",
                round(item.file_size / (1024 * 1024), 2)
            )

print("\nDownload and archive inspection completed.")

In [ ]:
from pathlib import Path
import zipfile
import shutil

archive = Path(
    "/content/IndicLID_model_downloads/indiclid-bert.zip"
)

model_dir = Path(
    "/content/IndicLID_official_source/Inference/models/indiclid-bert"
)

checkpoint = model_dir / "basline_nn_simple.pt"

print("=" * 75)
print("INDICLID OFFICIAL BERT EXTRACTION")
print("=" * 75)

free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
print("Available Colab storage (GB):", round(free_gb, 2))

if checkpoint.exists():
    print("\nCheckpoint already exists.")
    print("Size (MB):", round(checkpoint.stat().st_size / (1024 * 1024), 2))

else:
    if not archive.exists():
        raise FileNotFoundError(f"Archive not found: {archive}")

    with zipfile.ZipFile(archive, "r") as z:
        target_member = next(
            (
                name for name in z.namelist()
                if name.endswith("basline_nn_simple.pt")
            ),
            None
        )

        if target_member is None:
            raise FileNotFoundError(
                "Expected BERT checkpoint not found in archive."
            )

        required_gb = (
            z.getinfo(target_member).file_size / (1024 ** 3)
        )

        if free_gb < required_gb + 0.5:
            raise RuntimeError(
                "Insufficient free storage for safe extraction."
            )

        model_dir.mkdir(parents=True, exist_ok=True)

        with z.open(target_member) as source:
            with open(checkpoint, "wb") as destination:
                shutil.copyfileobj(source, destination)

    print("\nBERT checkpoint extracted successfully.")

print("\nCheckpoint:", checkpoint)
print("Exists:", checkpoint.exists())

if checkpoint.exists():
    print("Size (MB):", round(checkpoint.stat().st_size / (1024 * 1024), 2))

print("\nExtraction check completed.")

In [ ]:
import torch
from pathlib import Path

checkpoint = Path(
    "/content/IndicLID_official_source/Inference/models/indiclid-bert/basline_nn_simple.pt"
)

print("=" * 75)
print("OFFICIAL INDICLID BERT CHECKPOINT INSPECTION")
print("=" * 75)

print("PyTorch:", torch.__version__)
print("Device: CPU")
print("Checkpoint size (MB):", round(checkpoint.stat().st_size / (1024 * 1024), 2))

print("\nLoading official checkpoint...")

model = torch.load(
    checkpoint,
    map_location=torch.device("cpu"),
    weights_only=False
)

model.eval()

print("\nCHECKPOINT LOADED SUCCESSFULLY")
print("Model class:", type(model).__name__)

print("\nMODEL ARCHITECTURE")
print(model)

print("\nCLASSIFIER DIMENSIONS")

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(
            name,
            "| Input:", module.in_features,
            "| Output:", module.out_features
        )

print("\nPARAMETER SUMMARY")
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", f"{total_params:,}")
print("Model device:", next(model.parameters()).device)

print("\nCheckpoint inspection completed.")

In [ ]:
from pathlib import Path

repo = Path("/content/IndicLID_official_source")

inference_file = (
    repo /
    "final_runs_train/roman_model/finetuning/IndicBERT/"
    "freezed_bert_all_layer/inference.py"
)

print("=" * 80)
print("INDICLID BERT CHECKPOINT AND LABEL MAPPING VERIFICATION")
print("=" * 80)

print("\nCheckpoint classifier output classes:", model.classifier.out_features)
print("Model configuration classes:", model.config.num_labels)

print("\nMODEL CONFIG LABEL MAPPING")
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

print("\n" + "=" * 80)
print("OFFICIAL 22-CLASS INFERENCE MAPPING")
print("=" * 80)

if not inference_file.exists():
    print("Inference file not found:", inference_file)
else:
    lines = inference_file.read_text(
        encoding="utf-8"
    ).splitlines()

    for i, line in enumerate(lines):
        if any(key in line for key in [
            "confusion_matrix_mapping",
            "confusion_matrix_reverse_mapping",
            "classes ="
        ]):
            start = max(0, i - 2)
            end = min(len(lines), i + 65)

            for j in range(start, end):
                print(f"{j+1:4}: {lines[j]}")

print("\n" + "=" * 80)
print("VERIFICATION")
print("=" * 80)

print("Classifier output dimensions:", model.classifier.out_features)
print("Expected mapping size: 22")
print("Checkpoint mapping verification pending.")

In [ ]:
import transformers
import torch
import inspect
from pathlib import Path

repo = Path("/content/IndicLID_official_source")

print("=" * 75)
print("INDICLID CHECKPOINT COMPATIBILITY AUDIT")
print("=" * 75)

print("\nCURRENT ENVIRONMENT")
print("Python:", __import__("sys").version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

print("\nCHECKPOINT CONFIGURATION")
print("Model type:", model.config.model_type)
print("Hidden size:", model.config.hidden_size)
print("Layers:", model.config.num_hidden_layers)
print("Attention heads:", model.config.num_attention_heads)
print("Classifier classes:", model.classifier.out_features)

print("\nATTENTION MODULE ATTRIBUTES")
attention = model.bert.encoder.layer[0].attention
print("Class:", type(attention))
print("Attributes:", [
    x for x in dir(attention)
    if not x.startswith("__")
])

print("\nREPOSITORY DEPENDENCY FILES")
for filename in [
    "requirements.txt",
    "environment.yml",
    "setup.py",
    "pyproject.toml"
]:
    path = repo / filename
    if path.exists():
        print(f"\n--- {filename} ---")
        print(path.read_text(encoding="utf-8", errors="ignore")[:5000])

print("\nTRAINING SCRIPT IMPORTS")
training_file = (
    repo /
    "final_runs_train/roman_model/finetuning/IndicBERT/"
    "freezed_bert_all_layer/train.py"
)

if training_file.exists():
    for line in training_file.read_text(
        encoding="utf-8", errors="ignore"
    ).splitlines()[:45]:
        print(line)

print("\nAudit completed. No packages changed.")

In [ ]:
import torch
import json
from pathlib import Path
import transformers

checkpoint_path = Path(
    "/content/IndicLID_official_source/Inference/models/"
    "indiclid-bert/basline_nn_simple.pt"
)

print("=" * 70)
print("CHECKPOINT METADATA")
print("=" * 70)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))

    for key in ["config", "model_config", "args", "epoch"]:
        if key in checkpoint:
            value = checkpoint[key]
            print(f"\n{key}:")
            print(str(value)[:2000])

print("\nMODEL CONFIG")
print("Transformers version:", transformers.__version__)
print("Model class:", type(model).__name__)
print("BERT class:", type(model.bert).__name__)

print("\nTOKENIZER DETAILS")
print("Tokenizer class:", type(tokenizer).__name__)
print("Vocabulary size:", len(tokenizer))
print("Model vocabulary size:", model.config.vocab_size)

print("\nCHECKPOINT FILE SIZE (GB):",
      round(checkpoint_path.stat().st_size / (1024**3), 3))

print("\nInspection complete. No files or packages changed.")

In [ ]:
%pip install --no-cache-dir "transformers==4.48.3" "tokenizers==0.21.1"

In [ ]:
import torch
import transformers
import pandas as pd
from transformers import AutoTokenizer

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)

assert transformers.__version__ == "4.48.3", (
    "Please verify the runtime restart and Transformers version."
)

checkpoint_path = (
    "/content/IndicLID_official_source/Inference/models/"
    "indiclid-bert/basline_nn_simple.pt"
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "ai4bharat/IndicBERTv2-MLM-only"
)

# Load the complete serialized model
model = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

model.eval()

print("\nMODEL RELOADED")
print("Model class:", type(model).__name__)
print("Number of classes:", model.classifier.out_features)
print("Tokenizer vocabulary:", len(tokenizer))
print("Checkpoint loaded:", True)

# Verify the existing pilot data is available
print("\nPilot available:", "pilot" in globals())

if "pilot" in globals():
    print("Pilot shape:", pilot.shape)

print("\nReload verification completed.")

In [ ]:
import torch

# Restore the attention implementation setting on the loaded BERT model.
model.bert.attn_implementation = "eager"

# Also restore it on the nested BERT configuration if required.
if not hasattr(model.bert.config, "_attn_implementation"):
    model.bert.config._attn_implementation = "eager"

model.eval()

print("BERT attention implementation:",
      model.bert.attn_implementation)

# Test just one comment first.
test_text = str(pilot["comment_text"].iloc[0])

encoded = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

try:
    with torch.inference_mode():
        output = model(**encoded)

    probabilities = torch.softmax(output.logits, dim=1)
    confidence, predicted = probabilities.max(dim=1)

    labels_22 = [
        "Assamese", "Bangla", "Bodo", "Konkani", "Gujarati",
        "Hindi", "Kannada", "Kashmiri", "Maithili", "Malayalam",
        "Manipuri_Mei", "Marathi", "Nepali", "Oriya", "Punjabi",
        "Sanskrit", "Sindhi", "Tamil", "Telugu", "Urdu",
        "English", "Other"
    ]

    print("\nComment:", test_text[:250])
    print("Predicted language:", labels_22[predicted.item()])
    print("Confidence:", round(confidence.item(), 4))
    print("\nSingle-comment inference: SUCCESS")

except Exception as e:
    print("\nInference failed:", type(e).__name__, str(e))
    raise

In [ ]:
import torch
import pandas as pd

labels_22 = [
    "Assamese", "Bangla", "Bodo", "Konkani", "Gujarati",
    "Hindi", "Kannada", "Kashmiri", "Maithili", "Malayalam",
    "Manipuri_Mei", "Marathi", "Nepali", "Oriya", "Punjabi",
    "Sanskrit", "Sindhi", "Tamil", "Telugu", "Urdu",
    "English", "Other"
]

results = []
model.eval()

for start in range(0, len(pilot), 2):
    batch = (
        pilot["comment_text"]
        .iloc[start:start + 2]
        .fillna("")
        .astype(str)
        .tolist()
    )

    encoded = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    with torch.inference_mode():
        outputs = model(**encoded)

    probabilities = torch.softmax(outputs.logits, dim=1)
    confidence, predicted = probabilities.max(dim=1)

    for text, pred, conf in zip(
        batch, predicted.tolist(), confidence.tolist()
    ):
        results.append({
            "comment_text": text,
            "predicted_language": labels_22[pred],
            "language_confidence": round(conf, 4)
        })

pilot_predictions = pd.DataFrame(results)

print("=" * 70)
print("INDICBERT 22-CLASS PILOT RESULTS")
print("=" * 70)

print("Total comments:", len(pilot_predictions))

print("\nLanguage prediction distribution:")
display(
    pilot_predictions["predicted_language"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="comments")
)

print("\nConfidence distribution:")
display(
    pilot_predictions["language_confidence"].describe().round(4)
)

print("\nIndividual predictions:")
display(pilot_predictions)

In [ ]:
import re
import pandas as pd
import torch

# More specific Roman Hindi expressions; excludes generic words like "the".
hindi_terms = [
    "hai", "hain", "tha", "thi", "the", "nahi", "nahin", "nhi",
    "kya", "kyu", "kyun", "kaise", "mera", "meri", "mere",
    "tera", "teri", "tujhe", "mujhe", "tumhe", "bahut", "bohot",
    "acha", "achha", "accha", "wala", "wali", "wale", "yaar",
    "bhai", "matlab", "bilkul", "dekho", "dekha", "lagta",
    "lagti", "bol", "bola", "raha", "rahi", "rahe", "karna",
    "karo", "kiya", "hoon", "hun", "hain", "aap", "hum",
    "isko", "usko", "iska", "uska", "kaun", "kuch", "sab"
]

pattern = re.compile(
    r"\b(" + "|".join(map(re.escape, set(hindi_terms))) + r")\b",
    flags=re.IGNORECASE
)

diagnostic_pool = df[
    (df["latin_ratio"] > 0.5) &
    (df["comment_text"].fillna("").str.strip() != "")
].copy()

diagnostic_pool["hindi_keyword_count"] = (
    diagnostic_pool["comment_text"].astype(str)
    .apply(lambda x: len(pattern.findall(x)))
)

# Prioritize comments with at least two Hindi-specific expressions.
eligible = diagnostic_pool[
    diagnostic_pool["hindi_keyword_count"] >= 2
].copy()

print("Comments with 2+ Hindi-specific expressions:", len(eligible))

diagnostic = eligible.sample(
    n=min(30, len(eligible)),
    random_state=431
).reset_index(drop=True)

results = []

for start in range(0, len(diagnostic), 2):
    batch = diagnostic["comment_text"].iloc[start:start+2].astype(str).tolist()

    encoded = tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    with torch.inference_mode():
        outputs = model(**encoded)

    probs = torch.softmax(outputs.logits, dim=1)
    confidence, predicted = probs.max(dim=1)

    for text, pred, conf in zip(
        batch, predicted.tolist(), confidence.tolist()
    ):
        results.append({
            "comment_text": text,
            "predicted_language": labels_22[pred],
            "language_confidence": round(conf, 4)
        })

roman_hindi_diagnostic = pd.DataFrame(results)

print("\nPREDICTION DISTRIBUTION")
display(
    roman_hindi_diagnostic["predicted_language"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="comments")
)

print("\nINDIVIDUAL RESULTS")
display(roman_hindi_diagnostic)

In [ ]:
import pandas as pd

print("=" * 70)
print("ICMB-130 LANGUAGE IDENTIFICATION — FINAL PIPELINE PREPARATION")
print("=" * 70)

print("Dataset shape:", df.shape)

print("\nAvailable columns:")
print(df.columns.tolist())

print("\nScript distribution:")
if "script_type" in df.columns:
    display(df["script_type"].value_counts(dropna=False).to_frame("comments"))
else:
    print("script_type not present in the source dataset.")

print("\nComment text availability:")
print("Missing:", df["comment_text"].isna().sum())
print("Empty:", df["comment_text"].fillna("").str.strip().eq("").sum())

print("\nExisting language-related columns:")
print([
    c for c in df.columns
    if any(x in c.lower() for x in ["language", "script", "confidence"])
])

In [ ]:
import re
import pandas as pd

# Remove ambiguous words from the Hindi vocabulary.
HINDI_WORDS_FINAL = {
    "hai", "hain", "tha", "thi", "nahi", "nahin", "nhi",
    "kya", "kyu", "kyun", "kaise", "kaun", "kahan", "kab",
    "mera", "meri", "mere", "tera", "teri", "tujhe", "mujhe",
    "tumhe", "aap", "hum", "bahut", "bohot", "acha", "achha",
    "accha", "yaar", "bhai", "matlab", "bilkul", "wala",
    "wali", "wale", "isko", "usko", "iska", "uska",
    "dekho", "dekha", "lagta", "lagti", "raha", "rahi",
    "rahe", "karna", "karo", "kiya", "hoon", "hun",
    "doston", "pehle", "phir", "mein", "mai", "liye",
    "kyunki", "lekin", "aur", "bhi", "ye", "woh", "vo",
    "aisa", "aisi", "ka", "ki", "ke", "se", "ko",
    "par", "apne", "apni", "apna", "usne", "isne",
    "maine", "bolta", "bolti", "bola", "boli", "acchi",
    "achhi", "chahiye", "gaya", "gayi", "gaye"
}

# Use stronger English indicators; exclude generic words such as "the", "is".
ENGLISH_WORDS_FINAL = {
    "this", "that", "with", "you", "your", "have", "has",
    "had", "because", "but", "and", "for", "from", "they",
    "them", "their", "what", "when", "where", "who",
    "which", "would", "could", "should", "really", "very",
    "just", "there", "here", "been", "being", "about",
    "think", "know", "never", "always", "watching",
    "watched", "enjoyed", "amazing", "excellent",
    "beautiful", "story", "acting", "character",
    "although", "however", "everyone", "someone",
    "something", "anything", "please", "thank",
    "thanks", "love", "loved", "agree", "disagree"
}

def classify_roman_final(text):
    words = re.findall(r"[a-zA-Z]+", str(text).lower())

    if len(words) < 2:
        return "Uncertain", "Insufficient_Text"

    hindi_count = sum(w in HINDI_WORDS_FINAL for w in words)
    english_count = sum(w in ENGLISH_WORDS_FINAL for w in words)

    if hindi_count >= 2 and english_count >= 2:
        return "Hinglish", "Rule_Assisted"

    if hindi_count >= 2:
        return "Hindi_Roman", "Rule_Assisted"

    if english_count >= 3 and hindi_count == 0:
        return "English", "Rule_Assisted"

    return "Uncertain", "Insufficient_Language_Evidence"


def final_classification(row):
    script = row["script_type"]
    text = row["text_for_language"]

    if not text.strip():
        return pd.Series(["Uncertain", "Empty_Text"])

    native_labels = {
        "Devanagari": "Devanagari_Language_Unverified",
        "Bengali": "Bengali",
        "Gujarati": "Gujarati",
        "Gurmukhi": "Punjabi",
        "Odia": "Odia",
        "Tamil": "Tamil",
        "Telugu": "Telugu",
        "Kannada": "Kannada",
        "Malayalam": "Malayalam",
        "Arabic": "Arabic_Script_Language_Unverified"
    }

    if script in native_labels:
        return pd.Series([native_labels[script], "Unicode_Script"])

    if script.startswith("Mixed_"):
        return pd.Series(["Mixed_Script_Uncertain", "Unicode_Script"])

    if script == "Latin":
        return pd.Series(classify_roman_final(text))

    return pd.Series(["Uncertain", "No_Recognized_Script"])


lang_df[["final_language", "final_language_method"]] = (
    lang_df.apply(final_classification, axis=1)
)

print("FINAL BASELINE CLASSIFICATION")
print("Total comments:", len(lang_df))

print("\nLanguage distribution:")
display(
    lang_df["final_language"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="comments")
)

print("\nMethod distribution:")
display(lang_df["final_language_method"].value_counts().to_frame("comments"))

print("\nTargeted classification checks:")
display(
    lang_df[
        lang_df["text_for_language"].str.contains(
            r"bhai aap|kya baat hai|budget for this movie|"
            r"me pehle|rating cheppu|the budget",
            case=False, na=False, regex=True
        )
    ][["comment_text", "final_language", "final_language_method"]]
)

In [ ]:
# ============================================================
# NOTEBOOK 10 — SAVE FINAL BASELINE LANGUAGE CLASSIFICATION
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

OUTPUT_DIR = PROJECT_ROOT / "01_data/processed/audience_sentiment"
REPORT_DIR = PROJECT_ROOT / "06_outputs/audience_sentiment"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Separate output files; original consolidated data remains untouched
classified_path = (
    OUTPUT_DIR /
    "ICMB_130_YOUTUBE_COMMENTS_LANGUAGE_BASELINE.csv"
)

distribution_path = (
    REPORT_DIR /
    "ICMB_130_LANGUAGE_BASELINE_DISTRIBUTION.csv"
)

method_path = (
    REPORT_DIR /
    "ICMB_130_LANGUAGE_BASELINE_METHOD_DISTRIBUTION.csv"
)

# Validate before saving
assert len(lang_df) == 15960, "Unexpected comment count."
assert "final_language" in lang_df.columns
assert "final_language_method" in lang_df.columns
assert lang_df["final_language"].notna().all()

# Save classified comments
lang_df.to_csv(classified_path, index=False, encoding="utf-8-sig")

# Language distribution
language_report = (
    lang_df["final_language"]
    .value_counts()
    .rename_axis("language")
    .reset_index(name="comments")
)

language_report["percentage"] = (
    language_report["comments"] / len(lang_df) * 100
).round(2)

language_report.to_csv(
    distribution_path, index=False, encoding="utf-8-sig"
)

# Classification method distribution
method_report = (
    lang_df["final_language_method"]
    .value_counts()
    .rename_axis("classification_method")
    .reset_index(name="comments")
)

method_report["percentage"] = (
    method_report["comments"] / len(lang_df) * 100
).round(2)

method_report.to_csv(
    method_path, index=False, encoding="utf-8-sig"
)

print("=" * 65)
print("LANGUAGE BASELINE SAVED SUCCESSFULLY")
print("=" * 65)

print("Classified dataset:", classified_path)
print("Language report:", distribution_path)
print("Method report:", method_path)

print("\nDataset shape:", lang_df.shape)
print("Unique comment IDs:", lang_df["comment_id"].nunique())
print("Total comments:", len(lang_df))

print("\nLanguage distribution:")
display(language_report)

print("\nClassification method distribution:")
display(method_report)

print("\nFile existence verification:")
print("Classified dataset:", classified_path.exists())
print("Language report:", distribution_path.exists())
print("Method report:", method_path.exists())

In [ ]:
# ============================================================
# NOTEBOOK 10 — RECOVER MISSING SENTIMENT INPUT TEXT
# ============================================================

import pandas as pd

# Preserve normalized text wherever available.
normalized = (
    sentiment_df["normalized_comment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Use original comment text as fallback.
original = (
    sentiment_df["comment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

sentiment_df["sentiment_input_text"] = normalized.where(
    normalized.ne(""),
    original
)

# Record which source supplied the text.
sentiment_df["sentiment_text_source"] = "Normalized_Text"

sentiment_df.loc[
    normalized.eq("") & original.ne(""),
    "sentiment_text_source"
] = "Original_Comment_Text"

sentiment_df.loc[
    sentiment_df["sentiment_input_text"].eq(""),
    "sentiment_text_source"
] = "No_Text_Available"

sentiment_df["sentiment_input_status"] = (
    sentiment_df["sentiment_input_text"]
    .ne("")
    .map({True: "Available", False: "Empty"})
)

print("=" * 65)
print("SENTIMENT TEXT RECOVERY REPORT")
print("=" * 65)

print("\nText source distribution:")
display(
    sentiment_df["sentiment_text_source"]
    .value_counts()
    .rename_axis("text_source")
    .reset_index(name="comments")
)

print("\nFinal input availability:")
display(
    sentiment_df["sentiment_input_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="comments")
)

print("\nOriginal empty normalized text — recovery check:")

recovered = sentiment_df[
    normalized.eq("") & original.ne("")
]

print("Recovered comments:", len(recovered))

print("\nExamples of recovered comments:")
display(
    recovered[
        ["comment_id", "normalized_comment_text", "comment_text",
         "sentiment_input_text"]
    ].head(10)
)

print("\nFinal checks:")
print("Total comments:", len(sentiment_df))
print("Unique comment IDs:", sentiment_df["comment_id"].nunique())
print(
    "Available sentiment inputs:",
    sentiment_df["sentiment_input_text"].ne("").sum()
)
print(
    "Still empty:",
    sentiment_df["sentiment_input_text"].eq("").sum()
)

In [ ]:
# ============================================================
# NOTEBOOK 10 — SENTIMENT INPUT TYPE AUDIT
# ============================================================

import re
import pandas as pd

def classify_input_type(text):
    text = str(text).strip()

    if not text:
        return "Empty"

    # Remove URLs, whitespace, and punctuation.
    remaining = re.sub(r"https?://\S+|www\.\S+", "", text)
    remaining = re.sub(r"[\W_]+", "", remaining, flags=re.UNICODE)

    if not remaining:
        return "Emoji_or_Symbol_Only"

    return "Text_Available"

sentiment_df["sentiment_input_type"] = (
    sentiment_df["sentiment_input_text"]
    .apply(classify_input_type)
)

input_audit = (
    sentiment_df["sentiment_input_type"]
    .value_counts()
    .rename_axis("input_type")
    .reset_index(name="comments")
)

input_audit["percentage"] = (
    input_audit["comments"] / len(sentiment_df) * 100
).round(2)

print("=" * 65)
print("SENTIMENT INPUT TYPE AUDIT")
print("=" * 65)

display(input_audit)

print("\nExamples of emoji/symbol-only comments:")
display(
    sentiment_df.loc[
        sentiment_df["sentiment_input_type"] == "Emoji_or_Symbol_Only",
        ["comment_id", "comment_text", "final_language"]
    ].head(15)
)

print("\nExamples of text-bearing comments:")
display(
    sentiment_df.loc[
        sentiment_df["sentiment_input_type"] == "Text_Available",
        ["comment_id", "comment_text", "final_language"]
    ].head(10)
)

print("\nIntegrity:")
print("Total rows:", len(sentiment_df))
print("Unique comment IDs:", sentiment_df["comment_id"].nunique())

In [ ]:
# ============================================================
# NOTEBOOK 10 — FINAL SENTIMENT MODEL INPUT
# ============================================================

import re
import pandas as pd

def prepare_model_text(text):
    text = str(text).strip()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove standalone URL remnants and normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

sentiment_df["sentiment_model_text"] = (
    sentiment_df["sentiment_input_text"]
    .apply(prepare_model_text)
)

def assign_sentiment_input_group(row):
    text = row["sentiment_model_text"]

    if not text:
        if row["sentiment_input_type"] == "Empty":
            return "Empty"
        if row["sentiment_input_type"] == "Emoji_or_Symbol_Only":
            return "Emoji_Only"
        return "URL_or_NonText_Only"

    # Check whether any Unicode letter or number remains
    if not re.search(r"[\w]", text, flags=re.UNICODE):
        return "Emoji_or_Symbol_Only"

    return "Text_For_Sentiment_Model"

sentiment_df["sentiment_input_group"] = sentiment_df.apply(
    assign_sentiment_input_group,
    axis=1
)

input_group_report = (
    sentiment_df["sentiment_input_group"]
    .value_counts()
    .rename_axis("input_group")
    .reset_index(name="comments")
)

input_group_report["percentage"] = (
    input_group_report["comments"] / len(sentiment_df) * 100
).round(2)

print("=" * 65)
print("FINAL SENTIMENT MODEL INPUT GROUPS")
print("=" * 65)

display(input_group_report)

print("\nText samples after URL removal:")
display(
    sentiment_df.loc[
        sentiment_df["sentiment_input_group"] == "Text_For_Sentiment_Model",
        ["comment_id", "final_language", "sentiment_model_text"]
    ].head(10)
)

print("\nNon-text samples:")
display(
    sentiment_df.loc[
        sentiment_df["sentiment_input_group"] != "Text_For_Sentiment_Model",
        ["comment_id", "sentiment_input_group", "sentiment_input_text"]
    ].head(10)
)

print("\nIntegrity checks:")
print("Total rows:", len(sentiment_df))
print("Unique comment IDs:", sentiment_df["comment_id"].nunique())
print(
    "Text comments for model:",
    (sentiment_df["sentiment_input_group"] == "Text_For_Sentiment_Model").sum()
)

In [ ]:
# ============================================================
# NOTEBOOK 10 — LOAD MULTILINGUAL SENTIMENT BASELINE
# Model: cardiffnlp/twitter-xlm-roberta-base-sentiment
# ============================================================

import torch
from transformers import pipeline

MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

print("=" * 65)
print("SENTIMENT MODEL ENVIRONMENT")
print("=" * 65)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = 0 if torch.cuda.is_available() else -1

print("Selected device:", "GPU" if device == 0 else "CPU")
print("Loading model:", MODEL_NAME)

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=device
)

print("\nModel loaded successfully.")

# Verify label mapping with a few illustrative examples
test_texts = [
    "This movie is absolutely fantastic!",
    "I hated this film. Very disappointing.",
    "The movie was okay, nothing special."
]

test_results = sentiment_pipeline(
    test_texts,
    truncation=True,
    max_length=256
)

print("\nModel test:")
for text, result in zip(test_texts, test_results):
    print({
        "text": text,
        "prediction": result["label"],
        "score": round(result["score"], 4)
    })

In [ ]:
# ============================================================
# NOTEBOOK 10 — SENTIMENT MODEL LABEL AND CONFIDENCE CHECK
# ============================================================

print("=" * 65)
print("MODEL LABEL MAPPING")
print("=" * 65)

print(sentiment_pipeline.model.config.id2label)

test_texts = [
    "This movie is absolutely fantastic!",
    "I hated this film. Very disappointing.",
    "The movie was okay, nothing special.",
    "Average movie, could have been better.",
    "Bhai kya zabardast movie hai!",
    "Worst film ever. Total waste of time."
]

# Return scores for all sentiment classes
all_scores = sentiment_pipeline(
    test_texts,
    truncation=True,
    max_length=256,
    top_k=None
)

rows = []

for text, scores in zip(test_texts, all_scores):
    score_map = {
        item["label"]: round(item["score"], 4)
        for item in scores
    }

    predicted = max(score_map, key=score_map.get)

    rows.append({
        "text": text,
        "predicted_sentiment": predicted,
        "confidence": score_map[predicted],
        **score_map
    })

label_check_df = pd.DataFrame(rows)

display(label_check_df)

print("\nModel maximum input length:",
      sentiment_pipeline.tokenizer.model_max_length)

In [ ]:
# ============================================================
# NOTEBOOK 10 — FULL DATASET SENTIMENT INFERENCE
# Model: CardiffNLP XLM-RoBERTa Multilingual Sentiment
# ============================================================

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

CONFIDENCE_THRESHOLD = 0.60
BATCH_SIZE = 32
MAX_LENGTH = 256

# Initialize output columns for every comment
sentiment_df["model_sentiment"] = "Not_Processed"
sentiment_df["sentiment_confidence"] = np.nan
sentiment_df["positive_score"] = np.nan
sentiment_df["neutral_score"] = np.nan
sentiment_df["negative_score"] = np.nan
sentiment_df["final_sentiment"] = "Not_Processed"

# Select only eligible text comments
eligible_mask = (
    sentiment_df["sentiment_input_group"]
    == "Text_For_Sentiment_Model"
)

eligible_indices = sentiment_df.index[eligible_mask].tolist()
eligible_texts = (
    sentiment_df.loc[eligible_indices, "sentiment_model_text"]
    .astype(str)
    .tolist()
)

print("=" * 65)
print("STARTING SENTIMENT INFERENCE")
print("=" * 65)
print("Eligible comments:", len(eligible_texts))
print("Batch size:", BATCH_SIZE)
print("Maximum token length:", MAX_LENGTH)

# Run inference in batches
all_predictions = []

for start in tqdm(
    range(0, len(eligible_texts), BATCH_SIZE),
    desc="Sentiment inference"
):
    batch_texts = eligible_texts[start:start + BATCH_SIZE]

    batch_results = sentiment_pipeline(
        batch_texts,
        batch_size=BATCH_SIZE,
        truncation=True,
        max_length=MAX_LENGTH,
        top_k=None
    )

    all_predictions.extend(batch_results)

# Map predictions back to the original dataframe
for idx, scores in zip(eligible_indices, all_predictions):

    score_map = {
        item["label"].lower(): float(item["score"])
        for item in scores
    }

    predicted_label = max(score_map, key=score_map.get)
    confidence = score_map[predicted_label]

    sentiment_df.at[idx, "model_sentiment"] = predicted_label
    sentiment_df.at[idx, "sentiment_confidence"] = confidence

    sentiment_df.at[idx, "positive_score"] = score_map["positive"]
    sentiment_df.at[idx, "neutral_score"] = score_map["neutral"]
    sentiment_df.at[idx, "negative_score"] = score_map["negative"]

    sentiment_df.at[idx, "final_sentiment"] = (
        predicted_label
        if confidence >= CONFIDENCE_THRESHOLD
        else "Uncertain"
    )

# Preserve separate statuses for comments not sent to the model
sentiment_df.loc[
    sentiment_df["sentiment_input_group"] == "Emoji_or_Symbol_Only",
    "final_sentiment"
] = "Emoji_Only_Not_Modelled"

sentiment_df.loc[
    sentiment_df["sentiment_input_group"] == "Emoji_Only",
    "final_sentiment"
] = "Emoji_Only_Not_Modelled"

sentiment_df.loc[
    sentiment_df["sentiment_input_group"] == "Empty",
    "final_sentiment"
] = "Empty"

sentiment_df.loc[
    sentiment_df["sentiment_input_group"] == "URL_or_NonText_Only",
    "final_sentiment"
] = "NonText_Not_Modelled"

print("\n" + "=" * 65)
print("SENTIMENT INFERENCE COMPLETED")
print("=" * 65)

print("Total comments:", len(sentiment_df))
print("Processed comments:", len(all_predictions))
print("Unique comment IDs:", sentiment_df["comment_id"].nunique())

print("\nFinal sentiment distribution:")
display(
    sentiment_df["final_sentiment"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="comments")
)

print("\nRaw model prediction distribution:")
display(
    sentiment_df.loc[eligible_mask, "model_sentiment"]
    .value_counts()
    .rename_axis("model_sentiment")
    .reset_index(name="comments")
)

print("\nConfidence summary:")
display(
    sentiment_df.loc[eligible_mask, "sentiment_confidence"]
    .describe()
    .to_frame("confidence")
)

print("\nSample predictions:")
display(
    sentiment_df.loc[
        eligible_mask,
        [
            "comment_text",
            "final_language",
            "model_sentiment",
            "sentiment_confidence",
            "final_sentiment"
        ]
    ].head(15)
)

assert len(sentiment_df) == 15960
assert sentiment_df["comment_id"].nunique() == 15960
assert sentiment_df.loc[eligible_mask, "model_sentiment"].ne(
    "Not_Processed"
).all()

print("\nIntegrity checks passed.")

In [ ]:
# ============================================================
# NOTEBOOK 10 — SENTIMENT QUALITY AUDIT BY LANGUAGE
# ============================================================

# Analyse only comments actually processed by the model
model_df = sentiment_df[
    sentiment_df["sentiment_input_group"]
    == "Text_For_Sentiment_Model"
].copy()

# Language-wise sentiment distribution
language_sentiment = pd.crosstab(
    model_df["final_language"],
    model_df["final_sentiment"]
)

language_sentiment["Total"] = language_sentiment.sum(axis=1)

language_sentiment["Modelled_Confident"] = (
    language_sentiment.get("positive", 0)
    + language_sentiment.get("negative", 0)
    + language_sentiment.get("neutral", 0)
)

language_sentiment["Confident_Percentage"] = (
    language_sentiment["Modelled_Confident"]
    / language_sentiment["Total"] * 100
).round(2)

# Average confidence by language
confidence_by_language = (
    model_df.groupby("final_language")["sentiment_confidence"]
    .agg(["count", "mean", "median"])
    .round(3)
)

print("=" * 70)
print("SENTIMENT DISTRIBUTION BY COMMENT LANGUAGE")
print("=" * 70)

display(language_sentiment.sort_values("Total", ascending=False))

print("\nCONFIDENCE BY LANGUAGE")
display(confidence_by_language.sort_values("count", ascending=False))

# Targeted sample for Roman Hindi and Hinglish
print("\nROMAN HINDI AND HINGLISH SAMPLE REVIEW")

display(
    model_df[
        model_df["final_language"].isin(
            ["Hindi_Roman", "Hinglish"]
        )
    ][
        [
            "comment_text",
            "final_language",
            "model_sentiment",
            "sentiment_confidence",
            "final_sentiment"
        ]
    ].sample(
        n=min(
            20,
            len(model_df[
                model_df["final_language"].isin(
                    ["Hindi_Roman", "Hinglish"]
                )
            ])
        ),
        random_state=42
    )
)

print("\nTOTAL MODELLED COMMENTS:", len(model_df))
print("UNIQUE COMMENT IDs:", model_df["comment_id"].nunique())

In [ ]:
# ============================================================
# NOTEBOOK 10 — FINAL SENTIMENT DATASET AND REPORTS
# ============================================================

from pathlib import Path
import pandas as pd
import json

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "01_data/processed/audience_sentiment"
)

REPORT_DIR = (
    PROJECT_ROOT /
    "06_outputs/audience_sentiment"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Assign reporting reliability groups
# ------------------------------------------------------------

def assign_reliability(language):
    if language == "English":
        return "English_Baseline_Not_Validated"

    if language in ["Hindi_Roman", "Hinglish"]:
        return "Roman_Hindi_Low_Confidence"

    if language == "Uncertain":
        return "Language_Uncertain_Exploratory"

    return "Other_Languages_Exploratory"

sentiment_df["sentiment_reliability_group"] = (
    sentiment_df["final_language"]
    .apply(assign_reliability)
)

# ------------------------------------------------------------
# 2. Validate dataset
# ------------------------------------------------------------

assert len(sentiment_df) == 15960
assert sentiment_df["comment_id"].nunique() == 15960

assert sentiment_df.loc[
    sentiment_df["sentiment_input_group"]
    == "Text_For_Sentiment_Model",
    "model_sentiment"
].ne("Not_Processed").all()

# ------------------------------------------------------------
# 3. Save final sentiment dataset
# ------------------------------------------------------------

dataset_path = (
    OUTPUT_DIR /
    "ICMB_130_YOUTUBE_COMMENTS_SENTIMENT_BASELINE.csv"
)

sentiment_df.to_csv(
    dataset_path,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 4. Overall sentiment distribution
# ------------------------------------------------------------

overall_report = (
    sentiment_df["final_sentiment"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="comments")
)

overall_report["percentage"] = (
    overall_report["comments"] / len(sentiment_df) * 100
).round(2)

overall_path = (
    REPORT_DIR /
    "ICMB_130_OVERALL_SENTIMENT_DISTRIBUTION.csv"
)

overall_report.to_csv(
    overall_path, index=False, encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 5. Language-wise sentiment report
# ------------------------------------------------------------

language_report = pd.crosstab(
    sentiment_df["final_language"],
    sentiment_df["final_sentiment"]
).reset_index()

language_report_path = (
    REPORT_DIR /
    "ICMB_130_LANGUAGE_WISE_SENTIMENT.csv"
)

language_report.to_csv(
    language_report_path, index=False, encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 6. Reliability group report
# ------------------------------------------------------------

reliability_report = (
    sentiment_df["sentiment_reliability_group"]
    .value_counts()
    .rename_axis("reliability_group")
    .reset_index(name="comments")
)

reliability_report["percentage"] = (
    reliability_report["comments"] / len(sentiment_df) * 100
).round(2)

reliability_path = (
    REPORT_DIR /
    "ICMB_130_SENTIMENT_RELIABILITY_GROUPS.csv"
)

reliability_report.to_csv(
    reliability_path, index=False, encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. Save model and methodology information
# ------------------------------------------------------------

metadata = {
    "project": "ICMB-130",
    "task": "YouTube audience comment sentiment analysis",
    "model": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
    "framework": "Hugging Face Transformers",
    "execution_device": "CPU",
    "total_comments": 15960,
    "modelled_text_comments": 15569,
    "confidence_threshold": 0.60,
    "sentiment_labels": ["positive", "negative", "neutral"],
    "low_confidence_handling": "Assigned Uncertain",
    "emoji_only_handling": "Not modelled",
    "empty_comment_handling": "Not modelled",
    "language_limitation": (
        "Roman Hindi and Hinglish show low model confidence. "
        "Predictions are exploratory and are not manually validated."
    ),
    "confidence_interpretation": (
        "Model score is not equivalent to prediction accuracy."
    )
}

metadata_path = (
    REPORT_DIR /
    "ICMB_130_SENTIMENT_MODEL_METHODOLOGY.json"
)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

# ------------------------------------------------------------
# 8. Final verification
# ------------------------------------------------------------

print("=" * 70)
print("FINAL SENTIMENT BASELINE — SAVED")
print("=" * 70)

print("Dataset shape:", sentiment_df.shape)
print("Total comments:", len(sentiment_df))
print("Unique comment IDs:", sentiment_df["comment_id"].nunique())

print("\nOverall sentiment:")
display(overall_report)

print("\nReliability groups:")
display(reliability_report)

print("\nSaved files:")
for path in [
    dataset_path,
    overall_path,
    language_report_path,
    reliability_path,
    metadata_path
]:
    print(path.name, "| Exists:", path.exists())

In [ ]:
# ============================================================
# NOTEBOOK 10 — MOVIE-WISE AND YEAR-WISE SENTIMENT ANALYSIS
# ============================================================

import pandas as pd
from pathlib import Path

# Identify actual columns without assuming schema
title_candidates = ["movie_title", "tmdb_title"]
year_candidates = ["year", "movie_year"]

title_col = next(
    (c for c in title_candidates if c in sentiment_df.columns),
    None
)

year_col = next(
    (c for c in year_candidates if c in sentiment_df.columns),
    None
)

required = ["movie_record_id", "final_sentiment"]

missing = [c for c in required if c not in sentiment_df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

if title_col is None or year_col is None:
    raise ValueError(
        f"Movie/year fields not found. Available columns: "
        f"{sentiment_df.columns.tolist()}"
    )

# Work on a copy
analysis_df = sentiment_df.copy()

analysis_df[year_col] = pd.to_numeric(
    analysis_df[year_col], errors="coerce"
)

# Keep only confidently classified sentiment labels for polarity analysis
confident_labels = ["positive", "negative", "neutral"]

analysis_df["is_confident"] = (
    analysis_df["final_sentiment"].isin(confident_labels)
)

# ------------------------------------------------------------
# 1. MOVIE-WISE SENTIMENT
# ------------------------------------------------------------

movie_summary = (
    analysis_df.groupby(
        ["movie_record_id", title_col, year_col],
        dropna=False
    )
    .agg(
        total_comments=("comment_id", "nunique"),
        confident_comments=("is_confident", "sum"),
        positive_comments=(
            "final_sentiment",
            lambda x: (x == "positive").sum()
        ),
        negative_comments=(
            "final_sentiment",
            lambda x: (x == "negative").sum()
        ),
        neutral_comments=(
            "final_sentiment",
            lambda x: (x == "neutral").sum()
        ),
        uncertain_comments=(
            "final_sentiment",
            lambda x: (x == "Uncertain").sum()
        )
    )
    .reset_index()
)

movie_summary["confident_coverage_pct"] = (
    movie_summary["confident_comments"]
    / movie_summary["total_comments"] * 100
).round(2)

for label in ["positive", "negative", "neutral"]:
    movie_summary[f"{label}_pct"] = (
        movie_summary[f"{label}_comments"]
        / movie_summary["confident_comments"].replace(0, pd.NA)
        * 100
    ).round(2)

# ------------------------------------------------------------
# 2. YEAR-WISE SENTIMENT
# ------------------------------------------------------------

year_summary = (
    analysis_df.groupby(year_col)
    .agg(
        total_comments=("comment_id", "nunique"),
        confident_comments=("is_confident", "sum"),
        positive_comments=(
            "final_sentiment",
            lambda x: (x == "positive").sum()
        ),
        negative_comments=(
            "final_sentiment",
            lambda x: (x == "negative").sum()
        ),
        neutral_comments=(
            "final_sentiment",
            lambda x: (x == "neutral").sum()
        ),
        uncertain_comments=(
            "final_sentiment",
            lambda x: (x == "Uncertain").sum()
        )
    )
    .reset_index()
    .rename(columns={year_col: "year"})
)

year_summary["confident_coverage_pct"] = (
    year_summary["confident_comments"]
    / year_summary["total_comments"] * 100
).round(2)

for label in ["positive", "negative", "neutral"]:
    year_summary[f"{label}_pct"] = (
        year_summary[f"{label}_comments"]
        / year_summary["confident_comments"].replace(0, pd.NA)
        * 100
    ).round(2)

# ------------------------------------------------------------
# 3. REPORTS
# ------------------------------------------------------------

REPORT_DIR = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "06_outputs/audience_sentiment"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

movie_path = REPORT_DIR / "ICMB_130_MOVIE_WISE_SENTIMENT_ANALYSIS.csv"
year_path = REPORT_DIR / "ICMB_130_YEAR_WISE_SENTIMENT_TRENDS.csv"

movie_summary.to_csv(movie_path, index=False, encoding="utf-8-sig")
year_summary.to_csv(year_path, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 4. VERIFICATION
# ------------------------------------------------------------

print("=" * 70)
print("MOVIE-WISE AND YEAR-WISE SENTIMENT ANALYSIS")
print("=" * 70)

print("Movie title column:", title_col)
print("Year column:", year_col)
print("Unique movies:", movie_summary["movie_record_id"].nunique())
print("Years covered:", year_summary["year"].nunique())

print("\nMovie-wise summary — first 10 records:")
display(movie_summary.head(10))

print("\nYear-wise sentiment trends:")
display(year_summary)

print("\nSaved reports:")
print(movie_path, "| Exists:", movie_path.exists())
print(year_path, "| Exists:", year_path.exists())

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
DATA_DIR = ROOT / "01_data/processed"
OUT_DIR = ROOT / "06_outputs/audience_sentiment"
OUT_DIR.mkdir(parents=True, exist_ok=True)

master = pd.read_csv(DATA_DIR / "ICMB_130_MASTER_DATA.csv")
df = pd.read_csv(
    DATA_DIR / "audience_sentiment/ICMB_130_YOUTUBE_COMMENTS_SENTIMENT_BASELINE.csv"
)

# Use master dataset as the authoritative movie reference
master["movie_record_id"] = master["movie_record_id"].astype(str)
df["movie_record_id"] = df["movie_record_id"].astype(str)

# Aggregate each movie exactly once
movie_counts = (
    df.groupby("movie_record_id")
    .agg(
        total_comments=("comment_id", "nunique"),
        positive_comments=("final_sentiment", lambda x: (x == "positive").sum()),
        negative_comments=("final_sentiment", lambda x: (x == "negative").sum()),
        neutral_comments=("final_sentiment", lambda x: (x == "neutral").sum())
    )
    .reset_index()
)

movie_counts["confident_comments"] = (
    movie_counts["positive_comments"]
    + movie_counts["negative_comments"]
    + movie_counts["neutral_comments"]
)

movie_counts["uncertain_comments"] = (
    movie_counts["total_comments"] - movie_counts["confident_comments"]
)

# Join to the 130 authoritative movie records
movie_summary = master[
    ["movie_record_id", "movie_title", "year", "rank", "language", "india_gross_cr"]
].merge(
    movie_counts,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

count_cols = [
    "total_comments", "positive_comments", "negative_comments",
    "neutral_comments", "confident_comments", "uncertain_comments"
]
movie_summary[count_cols] = movie_summary[count_cols].fillna(0).astype(int)

movie_summary["confident_coverage_pct"] = (
    movie_summary["confident_comments"]
    / movie_summary["total_comments"].replace(0, pd.NA) * 100
).fillna(0).round(2)

for sentiment in ["positive", "negative", "neutral"]:
    movie_summary[f"{sentiment}_pct"] = (
        movie_summary[f"{sentiment}_comments"]
        / movie_summary["confident_comments"].replace(0, pd.NA) * 100
    ).fillna(0).round(2)

# Year-wise aggregation from corrected movie-level records
year_summary = (
    movie_summary.groupby("year")
    .agg(
        total_comments=("total_comments", "sum"),
        confident_comments=("confident_comments", "sum"),
        positive_comments=("positive_comments", "sum"),
        negative_comments=("negative_comments", "sum"),
        neutral_comments=("neutral_comments", "sum"),
        uncertain_comments=("uncertain_comments", "sum"),
        movies_in_dataset=("movie_record_id", "nunique"),
        movies_with_comments=("total_comments", lambda x: (x > 0).sum())
    )
    .reset_index()
    .rename(columns={"year": "year"})
)

year_summary["confident_coverage_pct"] = (
    year_summary["confident_comments"]
    / year_summary["total_comments"].replace(0, pd.NA) * 100
).fillna(0).round(2)

for sentiment in ["positive", "negative", "neutral"]:
    year_summary[f"{sentiment}_pct"] = (
        year_summary[f"{sentiment}_comments"]
        / year_summary["confident_comments"].replace(0, pd.NA) * 100
    ).fillna(0).round(2)

# Save corrected reports under new filenames
movie_path = OUT_DIR / "ICMB_130_MOVIE_WISE_SENTIMENT_ANALYSIS_CORRECTED.csv"
year_path = OUT_DIR / "ICMB_130_YEAR_WISE_SENTIMENT_TRENDS_CORRECTED.csv"

movie_summary.to_csv(movie_path, index=False)
year_summary.to_csv(year_path, index=False)

print("=" * 70)
print("CORRECTED SENTIMENT AGGREGATION — VERIFICATION")
print("=" * 70)
print("Unique movies:", movie_summary["movie_record_id"].nunique())
print("Years covered:", year_summary["year"].nunique())
print("Year range:", year_summary["year"].min(), "-", year_summary["year"].max())
print("Total comments:", movie_summary["total_comments"].sum())
print("Total confident comments:", movie_summary["confident_comments"].sum())
print("Duplicate movie IDs:", movie_summary["movie_record_id"].duplicated().sum())

print("\nYEAR-WISE SUMMARY")
display(year_summary)

print("\nLAGAAN VERIFICATION")
display(movie_summary[movie_summary["movie_title"].str.contains("Lagaan", case=False, na=False)])

print("\nSAVED:")
print(movie_path)
print(year_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
OUT = ROOT / "06_outputs/audience_sentiment"
OUT.mkdir(parents=True, exist_ok=True)

# Load corrected reports
movie = pd.read_csv(OUT / "ICMB_130_MOVIE_WISE_SENTIMENT_ANALYSIS_CORRECTED.csv")
year = pd.read_csv(OUT / "ICMB_130_YEAR_WISE_SENTIMENT_TRENDS_CORRECTED.csv")

# 1. Overall sentiment distribution
sentiment_counts = pd.Series({
    "Positive": movie["positive_comments"].sum(),
    "Negative": movie["negative_comments"].sum(),
    "Neutral": movie["neutral_comments"].sum(),
    "Uncertain": movie["uncertain_comments"].sum()
})

plt.figure(figsize=(8, 5))
sentiment_counts.plot(kind="bar")
plt.title("Overall YouTube Comment Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Comments")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUT / "FIG_01_OVERALL_SENTIMENT.png", dpi=300)
plt.show()

# 2. Year-wise sentiment trends (confident labels only)
plt.figure(figsize=(12, 6))
for col, label in [
    ("positive_pct", "Positive"),
    ("negative_pct", "Negative"),
    ("neutral_pct", "Neutral")
]:
    plt.plot(year["year"], year[col], marker="o", label=label)

plt.title("Year-wise Sentiment Trends (Confident Predictions)")
plt.xlabel("Movie Release Year")
plt.ylabel("Share of Confident Sentiment Labels (%)")
plt.xticks(year["year"], rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "FIG_02_YEAR_WISE_SENTIMENT_TRENDS.png", dpi=300)
plt.show()

# 3. Movie-wise sentiment comparison
movie_sorted = movie.sort_values("positive_pct", ascending=False).head(20)

plt.figure(figsize=(12, 8))
plt.barh(movie_sorted["movie_title"], movie_sorted["positive_pct"])
plt.title("Top 20 Movies by Positive Sentiment Share")
plt.xlabel("Positive Share Among Confident Labels (%)")
plt.ylabel("Movie")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(OUT / "FIG_03_MOVIE_WISE_POSITIVE_SENTIMENT.png", dpi=300)
plt.show()

# 4. Year-wise confidence coverage
plt.figure(figsize=(12, 5))
plt.bar(year["year"].astype(str), year["confident_coverage_pct"])
plt.title("Year-wise Sentiment Classification Coverage")
plt.xlabel("Movie Release Year")
plt.ylabel("Confident Predictions (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUT / "FIG_04_YEAR_WISE_CONFIDENCE_COVERAGE.png", dpi=300)
plt.show()

print("\nAll four figures saved successfully:")
for file in sorted(OUT.glob("FIG_0*.png")):
    print(file.name, "|", round(file.stat().st_size / 1024, 2), "KB")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
TOPIC_DIR = ROOT / "06_outputs/topic_modelling"
SENTIMENT_DIR = ROOT / "06_outputs/audience_sentiment"

print("=" * 75)
print("FINAL MULTIMODAL INTEGRATION — INPUT INVENTORY")
print("=" * 75)

print("\n1. TOPIC MODELLING OUTPUTS")
if TOPIC_DIR.exists():
    for path in sorted(TOPIC_DIR.iterdir()):
        if path.is_file():
            print(f"\n{path.name} | {path.stat().st_size / 1024:.1f} KB")
            if path.suffix.lower() == ".csv":
                try:
                    temp = pd.read_csv(path, nrows=3)
                    print("Columns:", temp.columns.tolist())
                    print("Sample shape:", temp.shape)
                except Exception as e:
                    print("Could not preview:", str(e))
else:
    print("Topic modelling directory not found.")

print("\n2. CORRECTED SENTIMENT OUTPUTS")
for filename in [
    "ICMB_130_MOVIE_WISE_SENTIMENT_ANALYSIS_CORRECTED.csv",
    "ICMB_130_YEAR_WISE_SENTIMENT_TRENDS_CORRECTED.csv"
]:
    path = SENTIMENT_DIR / filename
    print("\n", filename, "| Exists:", path.exists())
    if path.exists():
        temp = pd.read_csv(path, nrows=3)
        print("Columns:", temp.columns.tolist())

print("\n3. MASTER DATASET")
master_path = ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"
master = pd.read_csv(master_path)
print("Shape:", master.shape)
print("Unique movie IDs:", master["movie_record_id"].nunique())
print("Columns:", master.columns.tolist())

print("\nINVENTORY COMPLETE — NO FILES MODIFIED.")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
TOPIC_DIR = ROOT / "06_outputs/topic_modelling"
SENTIMENT_DIR = ROOT / "06_outputs/audience_sentiment"

files = {
    "theme_register": TOPIC_DIR / "ICMB_130_FINAL_CURATED_THEME_REGISTER.csv",
    "theme_contributions": TOPIC_DIR / "ICMB_130_THEME_YEAR_THEME_CONTRIBUTIONS.csv",
    "theme_film_contributions": TOPIC_DIR / "ICMB_130_THEME_YEAR_FILM_CONTRIBUTIONS.csv",
    "sentiment": SENTIMENT_DIR / "ICMB_130_MOVIE_WISE_SENTIMENT_ANALYSIS_CORRECTED.csv"
}

# Correct the contribution filename if the inventory's exact filename differs.
if not files["theme_contributions"].exists():
    files["theme_contributions"] = TOPIC_DIR / "ICMB_130_FINAL_ANNUAL_THEME_PREVALENCE.csv"

for name, path in files.items():
    print("\n" + "=" * 75)
    print(name.upper())
    print("=" * 75)
    print("File:", path.name)
    print("Exists:", path.exists())

    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(5))

        if "movie_record_id" in df.columns:
            print("Unique movie IDs:", df["movie_record_id"].nunique())
        if "curated_theme" in df.columns:
            print("Unique themes:", df["curated_theme"].nunique())

print("\nINSPECTION COMPLETE — NO FILES MODIFIED.")

In [ ]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

TOPIC_DIR = ROOT / "06_outputs/topic_modelling"
SENTIMENT_DIR = ROOT / "06_outputs/audience_sentiment"
MASTER_PATH = ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"

REGISTER_PATH = TOPIC_DIR / "ICMB_130_FINAL_CURATED_THEME_REGISTER.csv"
CONTRIBUTIONS_PATH = TOPIC_DIR / "ICMB_130_THEME_YEAR_FILM_CONTRIBUTIONS.csv"
SENTIMENT_PATH = SENTIMENT_DIR / "ICMB_130_MOVIE_WISE_SENTIMENT_ANALYSIS_CORRECTED.csv"

OUTPUT_PATH = ROOT / "06_outputs/ICMB_130_MULTIMODAL_MOVIE_LEVEL_INTEGRATION_V1.csv"

# ------------------------------------------------------------
# 1. LOAD INPUTS
# ------------------------------------------------------------
master = pd.read_csv(MASTER_PATH)
register = pd.read_csv(REGISTER_PATH)
contributions = pd.read_csv(CONTRIBUTIONS_PATH)
sentiment = pd.read_csv(SENTIMENT_PATH)

# ------------------------------------------------------------
# 2. SELECT THE 12 INCLUDED CURATED THEMES
# ------------------------------------------------------------
included = register[
    register["decision"].astype(str).str.strip().str.lower() == "included"
].copy()

theme_names = sorted(contributions["curated_theme"].dropna().unique())

if len(theme_names) != 12:
    raise ValueError(f"Expected 12 contribution themes; found {len(theme_names)}.")

if set(theme_names) - set(included["curated_theme"]):
    raise ValueError("Contribution themes do not match the register's Included themes.")

print("Included themes:", len(theme_names))
print("Contribution records:", len(contributions))

# ------------------------------------------------------------
# 3. CREATE MOVIE-LEVEL THEME FEATURES
# ------------------------------------------------------------
contributions["theme_chunks"] = pd.to_numeric(
    contributions["theme_chunks"], errors="coerce"
).fillna(0)

movie_theme = (
    contributions
    .groupby(["movie_record_id", "curated_theme"], as_index=False)
    ["theme_chunks"].sum()
)

theme_counts = movie_theme.pivot(
    index="movie_record_id",
    columns="curated_theme",
    values="theme_chunks"
).fillna(0)

# Give columns stable, readable names and retain a mapping.
theme_column_map = {}

for i, theme in enumerate(theme_names, start=1):
    slug = re.sub(r"[^a-z0-9]+", "_", theme.lower()).strip("_")
    theme_column_map[theme] = f"theme_{i:02d}_{slug}_chunks"

theme_counts = theme_counts.rename(columns=theme_column_map)

# Calculate each theme's share of that movie's assigned theme chunks.
movie_theme_total = theme_counts.sum(axis=1).replace(0, pd.NA)

theme_shares = theme_counts.div(movie_theme_total, axis=0) * 100
theme_shares = theme_shares.rename(
    columns={col: col.replace("_chunks", "_share_pct")
             for col in theme_shares.columns}
)

theme_features = theme_counts.join(theme_shares)
theme_features = theme_features.reset_index()

# ------------------------------------------------------------
# 4. SELECT SENTIMENT FIELDS
# ------------------------------------------------------------
sentiment_columns = [
    "movie_record_id",
    "total_comments",
    "positive_comments",
    "negative_comments",
    "neutral_comments",
    "confident_comments",
    "uncertain_comments",
    "confident_coverage_pct",
    "positive_pct",
    "negative_pct",
    "neutral_pct"
]

sentiment_features = sentiment[sentiment_columns].copy()

# ------------------------------------------------------------
# 5. VALIDATE KEYS BEFORE MERGING
# ------------------------------------------------------------
for name, df in [
    ("Master", master),
    ("Sentiment", sentiment_features),
    ("Theme features", theme_features)
]:
    if df["movie_record_id"].duplicated().any():
        raise ValueError(f"{name} contains duplicate movie_record_id values.")

master_ids = set(master["movie_record_id"])
sentiment_ids = set(sentiment_features["movie_record_id"])
theme_ids = set(theme_features["movie_record_id"])

print("\nMOVIE ID VALIDATION")
print("Master IDs:", len(master_ids))
print("Sentiment IDs:", len(sentiment_ids))
print("Theme feature IDs:", len(theme_ids))
print("Missing sentiment IDs:", len(master_ids - sentiment_ids))
print("Missing theme IDs:", len(master_ids - theme_ids))
print("Extra sentiment IDs:", len(sentiment_ids - master_ids))
print("Extra theme IDs:", len(theme_ids - master_ids))

if master_ids != sentiment_ids:
    raise ValueError("Master and sentiment movie IDs do not match.")

if not theme_ids.issubset(master_ids):
    raise ValueError("Theme features contain IDs absent from the master dataset.")

# ------------------------------------------------------------
# 6. INTEGRATE AT ONE ROW PER MOVIE
# ------------------------------------------------------------
integrated = master.merge(
    theme_features,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

integrated = integrated.merge(
    sentiment_features,
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

# Movies without a theme contribution receive zero theme counts/shares.
theme_feature_columns = (
    list(theme_counts.columns) +
    list(theme_shares.columns)
)

integrated[theme_feature_columns] = (
    integrated[theme_feature_columns].fillna(0)
)

integrated["has_theme_contribution"] = (
    integrated[list(theme_counts.columns)].sum(axis=1) > 0
)

# ------------------------------------------------------------
# 7. FINAL QUALITY CHECKS
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("MULTIMODAL INTEGRATION QUALITY CHECK")
print("=" * 70)

print("Integrated shape:", integrated.shape)
print("Unique movie IDs:", integrated["movie_record_id"].nunique())
print("Duplicate movie IDs:", integrated["movie_record_id"].duplicated().sum())
print("Movies with theme contributions:",
      integrated["has_theme_contribution"].sum())
print("Movies with sentiment records:",
      integrated["total_comments"].notna().sum())
print("Missing values in theme features:",
      integrated[theme_feature_columns].isna().sum().sum())

assert len(integrated) == len(master) == 130
assert integrated["movie_record_id"].nunique() == 130
assert not integrated["movie_record_id"].duplicated().any()
assert integrated[theme_feature_columns].isna().sum().sum() == 0

# ------------------------------------------------------------
# 8. SAVE WITHOUT OVERWRITING
# ------------------------------------------------------------
if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"Output already exists; no file overwritten: {OUTPUT_PATH}"
    )

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
integrated.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)
print("File exists:", OUTPUT_PATH.exists())
print("Final shape:", integrated.shape)

display(integrated.head(5))

print("\nINTEGRATION COMPLETE.")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")

TOPIC_DIR = ROOT / "06_outputs/topic_modelling"
SENTIMENT_DIR = ROOT / "06_outputs/audience_sentiment"
MASTER_PATH = ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv"

THEME_PATH = TOPIC_DIR / "ICMB_130_FINAL_ANNUAL_THEME_PREVALENCE.csv"
SENTIMENT_PATH = SENTIMENT_DIR / "ICMB_130_YEAR_WISE_SENTIMENT_TRENDS_CORRECTED.csv"

OUTPUT_PATH = ROOT / "06_outputs/ICMB_130_YEAR_WISE_MULTIMODAL_PANEL_V1.csv"

# ------------------------------------------------------------
# 1. LOAD INPUTS
# ------------------------------------------------------------
themes = pd.read_csv(THEME_PATH)
sentiment_year = pd.read_csv(SENTIMENT_PATH)
master = pd.read_csv(MASTER_PATH)

# ------------------------------------------------------------
# 2. CREATE YEAR-WISE THEME PREVALENCE FEATURES
# ------------------------------------------------------------
theme_panel = themes.pivot(
    index="year",
    columns="curated_theme",
    values="dialogue_prevalence_pct"
).reset_index()

theme_panel.columns.name = None

theme_column_map = {
    col: "theme_prevalence_" + (
        "".join(ch.lower() if ch.isalnum() else "_" for ch in col)
        .strip("_")
    ) + "_pct"
    for col in theme_panel.columns
    if col != "year"
}

theme_panel = theme_panel.rename(columns=theme_column_map)

# ------------------------------------------------------------
# 3. PREPARE YEAR-WISE SENTIMENT INDICATORS
# ------------------------------------------------------------
sentiment_columns = [
    "year",
    "total_comments",
    "confident_comments",
    "positive_comments",
    "negative_comments",
    "neutral_comments",
    "uncertain_comments",
    "movies_in_dataset",
    "movies_with_comments",
    "confident_coverage_pct",
    "positive_pct",
    "negative_pct",
    "neutral_pct"
]

sentiment_panel = sentiment_year[sentiment_columns].copy()

# ------------------------------------------------------------
# 4. CREATE YEAR-WISE BOX-OFFICE SUMMARY
# ------------------------------------------------------------
master["india_gross_cr"] = pd.to_numeric(
    master["india_gross_cr"], errors="coerce"
)

box_office_panel = (
    master.groupby("year")
    .agg(
        movies_in_master=("movie_record_id", "nunique"),
        total_india_gross_cr=("india_gross_cr", "sum"),
        mean_india_gross_cr=("india_gross_cr", "mean"),
        median_india_gross_cr=("india_gross_cr", "median")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. MERGE INTO YEAR-WISE MULTIMODAL PANEL
# ------------------------------------------------------------
for name, df in [
    ("Theme panel", theme_panel),
    ("Sentiment panel", sentiment_panel),
    ("Box-office panel", box_office_panel)
]:
    if df["year"].duplicated().any():
        raise ValueError(f"{name} has duplicate years.")

panel = (
    box_office_panel
    .merge(theme_panel, on="year", how="outer", validate="one_to_one")
    .merge(sentiment_panel, on="year", how="left", validate="one_to_one")
    .sort_values("year")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------
print("=" * 70)
print("YEAR-WISE MULTIMODAL PANEL — VALIDATION")
print("=" * 70)

print("Panel shape:", panel.shape)
print("Year range:", panel["year"].min(), "to", panel["year"].max())
print("Unique years:", panel["year"].nunique())
print("Theme prevalence columns:", len(theme_column_map))
print("Duplicate years:", panel["year"].duplicated().sum())
print("\nMissing values by column:")
print(panel.isna().sum().to_string())

assert panel["year"].nunique() == 26
assert panel["year"].min() == 2000
assert panel["year"].max() == 2025
assert not panel["year"].duplicated().any()
assert len(theme_column_map) == 12

# ------------------------------------------------------------
# 7. SAVE WITHOUT OVERWRITING
# ------------------------------------------------------------
if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"Output already exists; no file overwritten: {OUTPUT_PATH}"
    )

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
panel.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)
print("File exists:", OUTPUT_PATH.exists())
print("Final shape:", panel.shape)

display(panel.head(10))

print("\nYEAR-WISE MULTIMODAL PANEL COMPLETE.")

In [ ]:
from pathlib import Path
import pandas as pd
from scipy.stats import spearmanr

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
INPUT_PATH = ROOT / "06_outputs/ICMB_130_YEAR_WISE_MULTIMODAL_PANEL_V1.csv"
OUTPUT_DIR = ROOT / "06_outputs"

CORR_PATH = OUTPUT_DIR / "ICMB_130_EXPLORATORY_THEME_OUTCOME_CORRELATIONS_V1.csv"
PERIOD_PATH = OUTPUT_DIR / "ICMB_130_THREE_PERIOD_MULTIMODAL_COMPARISON_V1.csv"

# ------------------------------------------------------------
# 1. LOAD YEAR-WISE PANEL
# ------------------------------------------------------------
df = pd.read_csv(INPUT_PATH)

theme_cols = [
    col for col in df.columns
    if col.startswith("theme_prevalence_")
]

outcome_cols = [
    "total_india_gross_cr",
    "mean_india_gross_cr",
    "positive_pct",
    "negative_pct",
    "neutral_pct",
    "confident_coverage_pct"
]

# ------------------------------------------------------------
# 2. SPEARMAN CORRELATION: THEMES VS OUTCOMES
# ------------------------------------------------------------
correlation_results = []

for theme in theme_cols:
    for outcome in outcome_cols:
        valid = df[[theme, outcome]].dropna()

        if len(valid) < 3 or valid[theme].nunique() < 2 or valid[outcome].nunique() < 2:
            rho, p_value = float("nan"), float("nan")
        else:
            rho, p_value = spearmanr(valid[theme], valid[outcome])

        correlation_results.append({
            "theme_feature": theme,
            "outcome_variable": outcome,
            "spearman_rho": rho,
            "p_value": p_value,
            "observations": len(valid),
            "interpretation": (
                "Positive association" if pd.notna(rho) and rho > 0
                else "Negative association" if pd.notna(rho) and rho < 0
                else "Not estimable"
            )
        })

correlations = pd.DataFrame(correlation_results)

# ------------------------------------------------------------
# 3. THREE-PERIOD COMPARISON
# ------------------------------------------------------------
df["period"] = pd.cut(
    df["year"],
    bins=[1999, 2008, 2017, 2025],
    labels=["Early (2000–2008)", "Middle (2009–2017)", "Recent (2018–2025)"]
)

period_metrics = (
    theme_cols
    + outcome_cols
)

period_summary = (
    df.groupby("period", observed=False)[period_metrics]
    .mean()
    .reset_index()
)

# Add number of years represented in each period.
period_year_counts = (
    df.groupby("period", observed=False)["year"]
    .nunique()
    .reset_index(name="years_in_period")
)

period_summary = period_summary.merge(
    period_year_counts,
    on="period",
    how="left"
)

# ------------------------------------------------------------
# 4. SAVE WITHOUT OVERWRITING
# ------------------------------------------------------------
for path in [CORR_PATH, PERIOD_PATH]:
    if path.exists():
        raise FileExistsError(f"Existing output protected: {path}")

correlations.to_csv(CORR_PATH, index=False)
period_summary.to_csv(PERIOD_PATH, index=False)

# ------------------------------------------------------------
# 5. DISPLAY RESULTS
# ------------------------------------------------------------
print("=" * 70)
print("EXPLORATORY THEME–OUTCOME CORRELATIONS")
print("=" * 70)

print("Correlation tests:", len(correlations))
print("Years available:", df["year"].nunique())

display(
    correlations.sort_values(
        "spearman_rho",
        key=lambda x: x.abs(),
        ascending=False
    ).head(20)
)

print("\n" + "=" * 70)
print("THREE-PERIOD MULTIMODAL COMPARISON")
print("=" * 70)

display(period_summary)

print("\nSaved correlation report:", CORR_PATH)
print("Saved period comparison:", PERIOD_PATH)
print("\nEXPLORATORY ANALYSIS COMPLETE.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
INPUT_PATH = ROOT / "06_outputs/ICMB_130_EXPLORATORY_THEME_OUTCOME_CORRELATIONS_V1.csv"
OUTPUT_PATH = ROOT / "06_outputs/ICMB_130_EXPLORATORY_THEME_OUTCOME_CORRELATIONS_FDR_V2.csv"

results = pd.read_csv(INPUT_PATH)

# Benjamini-Hochberg FDR correction, excluding non-estimable tests.
valid = results["p_value"].notna()
p_values = results.loc[valid, "p_value"].astype(float)

m = len(p_values)

if m == 0:
    raise ValueError("No valid p-values available for correction.")

order = np.argsort(p_values.to_numpy())
sorted_p = p_values.to_numpy()[order]

adjusted_sorted = sorted_p * m / np.arange(1, m + 1)
adjusted_sorted = np.minimum.accumulate(adjusted_sorted[::-1])[::-1]
adjusted_sorted = np.clip(adjusted_sorted, 0, 1)

adjusted = np.empty(m)
adjusted[order] = adjusted_sorted

results["fdr_adjusted_p_value"] = np.nan
results.loc[valid, "fdr_adjusted_p_value"] = adjusted

results["significant_after_fdr_05"] = (
    results["fdr_adjusted_p_value"] < 0.05
)

if OUTPUT_PATH.exists():
    raise FileExistsError(f"Output already exists: {OUTPUT_PATH}")

results.to_csv(OUTPUT_PATH, index=False)

print("=" * 70)
print("MULTIPLE-TESTING CORRECTION")
print("=" * 70)
print("Total tests:", len(results))
print("Valid tests:", m)
print("Significant before correction (p < 0.05):",
      (results["p_value"] < 0.05).sum())
print("Significant after FDR correction (q < 0.05):",
      results["significant_after_fdr_05"].sum())

print("\nTop associations after correction:")
display(
    results.sort_values("fdr_adjusted_p_value")
    [[
        "theme_feature",
        "outcome_variable",
        "spearman_rho",
        "p_value",
        "fdr_adjusted_p_value",
        "significant_after_fdr_05"
    ]]
    .head(15)
)

print("\nSaved:", OUTPUT_PATH)
print("FDR correction complete.")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
OUTPUT_DIR = ROOT / "06_outputs"

expected_files = [
    "ICMB_130_MULTIMODAL_MOVIE_LEVEL_INTEGRATION_V1.csv",
    "ICMB_130_YEAR_WISE_MULTIMODAL_PANEL_V1.csv",
    "ICMB_130_EXPLORATORY_THEME_OUTCOME_CORRELATIONS_V1.csv",
    "ICMB_130_EXPLORATORY_THEME_OUTCOME_CORRELATIONS_FDR_V2.csv",
    "ICMB_130_THREE_PERIOD_MULTIMODAL_COMPARISON_V1.csv",
    "topic_modelling/ICMB_130_FINAL_CURATED_THEME_REGISTER.csv",
    "topic_modelling/ICMB_130_FINAL_ANNUAL_THEME_PREVALENCE.csv",
    "topic_modelling/ICMB_130_FINAL_LONGITUDINAL_THEME_SUMMARY.csv",
    "topic_modelling/ICMB_130_FINAL_THEME_CONCENTRATION.csv",
    "topic_modelling/ICMB_130_THEME_YEAR_FILM_CONTRIBUTIONS.csv",
    "audience_sentiment/ICMB_130_MOVIE_WISE_SENTIMENT_ANALYSIS_CORRECTED.csv",
    "audience_sentiment/ICMB_130_YEAR_WISE_SENTIMENT_TRENDS_CORRECTED.csv",
    "audience_sentiment/ICMB_130_OVERALL_SENTIMENT_DISTRIBUTION.csv",
    "audience_sentiment/ICMB_130_LANGUAGE_WISE_SENTIMENT.csv",
    "audience_sentiment/ICMB_130_SENTIMENT_RELIABILITY_GROUPS.csv",
    "audience_sentiment/ICMB_130_LANGUAGE_BASELINE_DISTRIBUTION.csv",
    "audience_sentiment/ICMB_130_LANGUAGE_BASELINE_METHOD_DISTRIBUTION.csv"
]

inventory = []

for relative_path in expected_files:
    path = OUTPUT_DIR / relative_path

    record = {
        "file": relative_path,
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None
    }

    if path.exists() and path.suffix.lower() == ".csv":
        try:
            df = pd.read_csv(path)
            record["rows"] = len(df)
            record["columns"] = len(df.columns)
        except Exception as e:
            record["read_status"] = str(e)

    inventory.append(record)

inventory_df = pd.DataFrame(inventory)

print("=" * 75)
print("ICMB-130 FINAL OUTPUT INVENTORY")
print("=" * 75)

display(inventory_df)

print("\nFiles found:", inventory_df["exists"].sum(), "/", len(inventory_df))
print("Missing files:")
display(inventory_df[inventory_df["exists"] == False])

print("\nINVENTORY COMPLETE — NO FILES MODIFIED.")